# Delhi NCR Land Surface Temperature 1990-2050: Day & Night Thermal Dynamics, Explainable AI and Scenario Projections

**Abstract.** This notebook builds a reproducible, GPU-accelerated pipeline for the pre-monsoon (April-June)
land surface temperature (LST) of the Delhi National Capital Region (~55,000 km², 25 districts) from 1990 to
2050, for **two thermal regimes**: the mid-morning **day** surface (~10:30 IST, harmonised Landsat C2 L2 surface
temperature, 1990-2025, corrected to a common 10:45 IST overpass) and the **night** surface (~01:30 IST, MODIS Aqua
in Aqua-equivalent units, bridged to VIIRS for 2025 and to Terra for 2000, hindcast for 1990/1995). Eight observed
frames (3-year windows centred on 1990, 1995, ..., 2025) are explained by 47 drivers (spectral indices, GHSL
built-up and population, GLC_FCS30D land-cover fractions and landscape metrics, terrain, surface water and
ERA5-Land climate, each also as multi-scale neighbourhood means). One pooled XGBoost model per mode is validated
with pooled 20 km-block spatial cross-validation, leave-one-epoch-out transfer, an area-of-applicability analysis
and conformal error bands, and benchmarked against ridge, random-forest, LightGBM and graph-neural-network
baselines. Grouped TreeSHAP attributions (18 driver groups, block-bootstrap intervals) expose non-linear
thresholds and are clustered into four **governance zones** with quantified mitigation levers. The 1 km frames
are brought to **250 m** (Landsat day natively, night by DMS downscaling with block-conserving residual
correction) and **projected to 2030-2050** under SSP1-2.6, SSP2-4.5 and SSP5-8.5 by combining a validated
cellular-automaton land-change model with CMIP6 temperature deltas scaled by empirically fitted dLST/dTair
sensitivities, with Monte-Carlo uncertainty over GCM, sensitivity, land realisation and model error. Projections
are **scenarios, not forecasts**, and SHAP values describe the model, **not causal effects**. Everything is
exported as a web bundle for the companion React dashboard (NCR THERMAL/OPS v2).

## Pipeline overview
| Stage | Section | What happens |
|---|---|---|
| 0 | `00_setup` (this section) | configuration (`CFG` v2, run profiles), dependency check, GPU detection, utilities, stage cache, LSTB binary container |
| 1 | `01_study_area` | 25 districts, 1 km + 250 m UTM 43N grids, ROI masks, canonical cell order, Earth Engine login, data-mode decision |
| 2 | `02_synthetic_world` | synthetic twin 1990-2025 + future inputs with *planted* responses (synthetic mode only) |
| 3 | `03_gee_landsat` | Landsat 5/7/8/9 C2 L2 day LST + spectral composites, path harmonisation, overpass metadata (gee mode) |
| 4 | `04_gee_thermal` | MODIS/VIIRS night LST, Terra/VIIRS → Aqua calibration, MODIS day validation, annual 1990-2025 series (gee mode) |
| 5 | `05_gee_drivers` | GHSL, GLC_FCS30D, SRTM, surface water, ERA5-Land, CMIP6 / SSP urban / population future inputs (gee mode) |
| 6 | `06_features` | 47 features (focal means, distances, landscape metrics), feature table, CV folds, fixed masks |
| 7 | `07_models` | validation suite (block20/block5/district/LOEO/place-time/random), benchmarks, final models, persistent residual, AOA, conformal errors |
| 8 | `08_explain` | grouped TreeSHAP, interactions, bootstrap, dependence/thresholds, governance zones, recommendations, browser models |
| 9 | `09_timeline` | absolute levels, night hindcast, evidence tiers, frame fields, dLST/dTair sensitivities, trends, scope series |
| 10 | `10_downscaling` | 250 m day (Landsat) and night (DMS) fields, scale-invariance validation, EPSG:4326 PNG rasters |
| 11 | `11_projections` | land-change CA (figure-of-merit validated), derived predictors, CMIP6 deltas, Monte-Carlo projections 2030-2050 |
| 12 | `12_export` | web bundle v2 (LSTB cubes, JSON), parquet/geo/models, bundle validation, zip |

## How to run on Kaggle
**Kaggle refuses notebooks larger than 1 MB, and this notebook is ~1.5 MB.** On Kaggle, run the small
`kaggle_launcher.ipynb` (~12 KB) instead: it executes exactly these sections from a code dataset that you build with
`python notebook/make_kaggle_code_bundle.py` and upload once (**+ Create → New Dataset**; **New Version** after code
changes). Import the launcher that script writes next to the zip (**File → Import Notebook**), attach the code
dataset (right sidebar → **Input → + Add Input → Datasets → Your Work**), then follow the steps below; the overrides
of steps 4-6 go into the launcher's run-settings cell. This full notebook remains the readable, self-contained
version for local Jupyter.
1. **Settings → Accelerator → GPU T4 x2**, **Internet → On**.
2. **Add-ons → Secrets** (tick both for *this* notebook):
   * `GEE_SERVICE_ACCOUNT_KEY` — the *entire JSON key file* of a Google Cloud service account registered for
     Earth Engine (Earth Engine API enabled in the project; roles *Earth Engine Resource Viewer* and *Service Usage
     Consumer*; *Earth Engine Resource Writer* in addition if you use `LST_GEE_EXPORT=asset`);
   * `GEE_PROJECT` — the Cloud project id used for Earth Engine quota.
3. **Earth Engine quota: the Contributor tier is required.** Noncommercial projects get 150 EECU-hours/month
   (Community tier) or **1,000 EECU-hours/month (Contributor tier: link a billing account; noncommercial use is not
   charged; switching takes effect at once)**. Measured 2026-10-12: the pilot's Landsat extraction alone used up the
   Community tier's 150 EECU-hours after about 3.5 of its 4 frames (~35-45 EECU-hours per frame); a full run needs
   several hundred. When the quota is used up, Earth Engine puts the project into *Restricted Mode* (reduced
   concurrency) until the 1st of the next month, and the run stops cleanly with that message (everything downloaded
   so far stays cached for the resume). **Read the EECU in Google Cloud Monitoring →
   Earth Engine → "Completed EECU-seconds"**, grouped by `workload_tag` (`lst-v2-s03` Landsat, `lst-v2-s04`
   MODIS/VIIRS, `lst-v2-s05` drivers + CMIP6) and `compute_type`; the Earth Engine task list only shows batch
   exports. Interactive requests use the **standard endpoint** (`LST_GEE_ENDPOINT=highvolume` is opt-in; both
   have the same quota of 40 concurrent requests) with a 420 s request deadline (`LST_GEE_DEADLINE_S`).
4. **Run the pilot first:** `LST_PROFILE=pilot` = frames **1990, 2000, 2020, 2025** (Landsat-5-only TM, the
   Terra-reconstructed night frame and early GHSL/GLC epochs, the dense L7/L8/L9 windows), ridge + XGBoost, no
   projections, plus a **CMIP6 warming-pattern probe** for 2050 (`LST_PILOT_PROBES`); about 1-1.5 h, as an
   interactive *Run All* or a commit. Read the per-frame Landsat cost table at the end of section 03 (with its
   projected full-run time), the PILOT PROBES table of section 05, the EECU per workload tag and
   `metrics.json → timings`. **Extrapolation:** section 03 full ≈ 2 × pilot 03 (8 vs 4 frames; 2020/2025 are the
   densest windows), 04 ≈ 1 × (its frame-independent work runs in full in both), 05 GDDP ≈ 1.5-2 × (the probe
   already fetched the ssp245 blocks of the 2025 and 2050 windows); modelling scales with frames and benchmarks.
5. **Then the full profile as *Save & Run All (Commit)*.** Attach the pilot version's output first (**Input → + Add
   Input → Notebooks → Your Work → the notebook → (+)**; Kaggle attaches the output of the notebook's *newest*
   version) so its Earth Engine tiles and GDDP blocks are reused. **Decision rule:** if one pilot Landsat
   frame took more than about 45 min interactively, or many tiles split, run the full profile with
   `LST_GEE_EXPORT=asset` (needs *Earth Engine Resource Writer*) and `LST_STOP_AFTER=06` in a first commit, then
   attach that version's output and *Save & Run All* again. **Asset mode:** noncommercial projects run about 2
   batch tasks at a time, so the 8 Landsat exports take about 4 rounds (about 1.5-6 h); per-task limits
   `LST_GEE_EXPORT_TIMEOUT_S` (3 h running), `LST_GEE_EXPORT_QUEUE_S` (4 h queued),
   `LST_GEE_EXPORT_TIMEOUT_0405_S` (1 h, sections 04/05); failed or timed-out jobs are cancelled and extracted
   interactively; finished assets persist across sessions and are reused.
6. **Time budget and clean stop.** `LST_TIME_BUDGET_H` (default 11 on Kaggle) stops *starting* new work so the run
   ends cleanly before Kaggle's 12 h limit: the cache is packed, `outputs/RUN_STOPPED.json` says why, every later
   cell prints `skipped: run stopped (...)` (the launcher prints `STOPPED CLEANLY`) and the version completes with
   its output. `LST_STOP_AFTER=NN` stops the same way right after section NN. A version killed at 12 h may have no
   usable output. In interactive sessions enable **Settings → Persistence → Files** so `outputs/` survives a
   restart.
7. **Resuming.** Attach the completed (or cleanly stopped) version's output (**Input → + Add Input → Notebooks →
   Your Work → the notebook → (+)**). Kaggle attaches the output of the notebook's *newest* version, so save no
   other version between the stopped run and the resume (otherwise: open the stopped version → Output → *New
   Dataset* and attach that dataset); keep the code dataset attached. `LST_CACHE_IMPORT=auto` finds its `cache/`
   under `/kaggle/input/` (the run logs which one, and the attached run's `RUN_STOPPED.json`); an explicit value
   is a folder path copied from the Input panel, and a wrong one stops the run at once instead of silently
   recomputing. The cache is packed into `outputs/cache/packed/*.tar` at the end (Kaggle notebook
   outputs are reported to be capped at 500 files) and unpacked on import. Stage-cache keys = configuration + data
   fingerprints + a **per-section code hash** from `build_notebook.py`, so **edit `notebook/sections` and rebuild the
   notebook or code bundle (never hand-edit cells on Kaggle)**; `LST_RESUME=0` recomputes every stage (Earth Engine tiles and GDDP
   blocks stay cached); `LST_INVALIDATE=06,07` recomputes selected stages.

Kaggle has no environment-variable UI: put overrides in a first cell, e.g.
`import os; os.environ["LST_PROFILE"] = "pilot"`, before *Run All*. Missing packages are pip-installed
automatically on Kaggle (never on a local machine unless `LST_ALLOW_PIP=1`). The dashboard needs a browser with
`DecompressionStream` (Chrome/Edge 80+, Firefox 113+, Safari 16.4+).

## Run profiles (`LST_PROFILE`; `LST_FAST_DEV=1` ⇒ `fast`)
| Setting | `full` (Kaggle default) | `fast` (smoke tests) | `pilot` (GEE cost probe) |
|---|---|---|---|
| observed frames | 1990, 1995, …, 2025 (8) | same 8 | 1990, 2000, 2020, 2025 |
| projected frames | 2030 … 2050 (5) × 3 SSPs | same | none (CMIP6 2050 pattern probe instead) |
| land-cover fine grid `LC_FINE_RES_M` | 50 m (gee) / 100 m (synthetic) | 200 m | 50 m |
| bootstrap / Monte-Carlo draws / land realisations | 50 / 500 / 20 | 6 / 120 / 4 | 10 / – / – |
| block CV | 20 km, 10 folds | 20 km, 5 folds | 20 km, 5 folds |
| XGBoost rounds / early stop | 4000 / 100 | 400 / 30 | 1500 / 60 |
| GNN epochs / Moran permutations | 300 / 999 | 30 / 99 | 60 / 199 |
| benchmarks | ridge, random_forest, lightgbm, xgboost, gnn | same | ridge, xgboost |

`LST_FRAMES_OBS` (comma list, any profile) overrides the observed frames: a subset of the 8 SPEC frames with at
least 2 frames, at least 2 night training frames (2005-2025) and 2025; with projections it must contain 2005, 2015
and 2025.

## Environment overrides (all optional)
| Variable | Default | Effect |
|---|---|---|
| `LST_PROFILE` | `full` | run profile (table above) |
| `LST_FRAMES_OBS` | profile | observed frames, e.g. `1990,2005,2025` (validated, see above) |
| `LST_RUN_MODE` | `auto` | `auto` tries Earth Engine and falls back to a synthetic twin with a LOUD banner; `gee` fails hard without Earth Engine; `synthetic` never touches Earth Engine |
| `LST_GRID_RES_M` | 1000 | analysis cell size (m) |
| `LST_FINE_FACTOR` | 4 | fine grid = cell size / factor (250 m at 1 km) |
| `LST_LC_FINE_RES_M` (legacy `LST_FINE_RES_M`) | profile | land-cover resolution for landscape metrics; snapped to a divisor of the cell size |
| `LST_N_BOOTSTRAP`, `LST_N_MC`, `LST_N_LAND_REAL` | profile | SHAP bootstrap replicates, Monte-Carlo draws, land-change realisations |
| `LST_BENCHMARKS` | profile | comma list from `ridge,random_forest,lightgbm,xgboost,gnn` (xgboost is mandatory) |
| `LST_PILOT_PROBES` | `1` for a gee pilot, else `0` | section-05 probe of the CMIP6 2050 warming pattern (results in `FUTURE_INPUTS_DIAG['probes']`) |
| `LST_GEE_ENDPOINT` | `standard` | Earth Engine endpoint: `standard` or `highvolume` (opt-in; same 40-request quota) |
| `LST_GEE_DEADLINE_S` | 420 | client request deadline in seconds (minimum 330) |
| `LST_GEE_EXPORT` | `interactive` | `interactive` (tiled `computePixels`) or `asset` (batch `Export.image.toAsset`) |
| `LST_GEE_ASSET_ROOT` | `projects/<GEE_PROJECT>/assets/ncr_lst_v2` | Earth Engine asset folder for `asset` exports |
| `LST_GEE_EXPORT_TIMEOUT_S` | 10800 | asset mode: per Landsat task, seconds RUNNING before it is cancelled and extracted interactively |
| `LST_GEE_EXPORT_QUEUE_S` | 14400 | asset mode: per Landsat task, seconds queued (not yet RUNNING) before the same fallback |
| `LST_GEE_EXPORT_TIMEOUT_0405_S` | 3600 | asset mode: per running task of sections 04/05 (7200 s queued) |
| `LST_GEE_WORKERS` | 6 | concurrent `computePixels` requests (Earth Engine allows 40 per project) |
| `LST_TIME_BUDGET_H` | 11 on Kaggle, else unlimited | stop starting new work after this many hours (`0`/`none`/`off` = unlimited) |
| `LST_TIME_RESERVE_S` | 1800 | seconds kept free for packing the cache and finishing the version |
| `LST_STOP_AFTER` | – | `01`..`11`: stop cleanly after that section (e.g. `06`) |
| `LST_RESUME` | 1 | reuse stage-cache entries whose key (configuration + data fingerprints + per-section code hash) matches |
| `LST_INVALIDATE` | – | comma list of stage-name prefixes to recompute, e.g. `06,07` (tiles/GDDP blocks stay cached) |
| `LST_CACHE_IMPORT` | `auto` | `auto` = shallowest `/kaggle/input/*/**/cache`; or any folder containing a previous cache (e.g. a path copied from Kaggle's Input panel; a path without a cache stops the run); packed tars are unpacked; `0` disables |
| `LST_PACK_CACHE` | 1 on Kaggle, else 0 | pack `cache/` into `cache/packed/*.tar` at the end and at a clean stop |
| `LST_FRAME_MIN_COVERAGE` | 0.5 | section 06: minimum valid-target share of a training frame for the fixed masks (else weight 0) |
| `LST_DAY_MIN_FRAME_COVERAGE` | 0.2 | section 03: a Landsat day frame below this ROI coverage stops the run (acquisition fault) |
| `LST_GEE_PROJECT` / `LST_GEE_SERVICE_ACCOUNT_KEY` / `GOOGLE_APPLICATION_CREDENTIALS` | – | Earth Engine credentials outside Kaggle (JSON text or file path) |
| `LST_OUTPUT_DIR` | `/kaggle/working/outputs` or `./outputs` | output root |
| `LST_USE_RAPIDS` / `LST_INSTALL_RAPIDS` / `LST_RAPIDS_VERSION` | 0 | opt-in cuDF/cuML paths (not validated; off by default) |
| `LST_BOOT_BLOCK_KM` | auto | SHAP block-bootstrap block size (default: residual-correlogram range) |
| `LST_ALLOW_PIP` | 0 | allow pip installs outside Kaggle |
| `LST_GNN_CPU_BUDGET_S` | 900 | wall-clock budget per GNN fold without CUDA |

## Outputs (`outputs/`)
`web/` (dashboard bundle v2: `manifest.json`, `cells.lstb.gz`, `cube_*.lstb.gz`, `xai_*.lstb.gz`, `raster/*.png`,
JSON summaries), `web_bundle.zip`, `parquet/` (feature table, predictions, SHAP, CV metrics), `geo/` (districts,
zone polygons), `models/` (native XGBoost boosters), `npz/` (interaction tensors), `figures/*.png`, `cache/`
(stage, Earth Engine tile and GDDP block caches, packed into `cache/packed/*.tar` on Kaggle, reusable by later runs)
and, after a clean stop, `RUN_STOPPED.json`.

## Runtime (measured locally, extrapolated to Kaggle 2× T4 — calibrate with the pilot)
**Measured** (synthetic mode, 1 GTX 1650 + 8 CPU threads, CPU-only PyTorch, so the GNN ran on the CPU):

| run | 02–06 | 07 validation | 08 XAI | 09 | 10 | 11 | 12 | total |
|---|---|---|---|---|---|---|---|---|
| `fast`, 2 km (16.6 k cells) | 25 s | 64 min (GNN 50 min) | 5 min | 7 s | 1.7 min | 1.7 min | 12 s | 74 min |
| `fast`, 1 km (66.3 k cells; GNN capped at 100 s/fold) | 56 s | 79 min (GNN 61 min) | 7 min | 9 s | 2.2 min | 6 min | 38 s | 96 min |

Full-profile probes at 1 km: one XGBoost fit with `XGB_MAX_ROUNDS=4000` / early stop 100 takes **42 s (day,
424 k rows) / 33 s (night, 264 k rows)** on the GTX 1650 and, on the synthetic data, never stops early (best
round = 4000; real data stopped at ~2,300 rounds in v1); TreeSHAP of a 4000-tree model costs ~31 s per 60 k rows.
Anchors from the user's real v1 Kaggle run (2× T4, 176 k rows/fold, full profile): XGBoost 13.7 s, LightGBM-GPU
38 s, GNN (300 epochs) 123 s and random forest (4 vCPU) 98 s per fold.

**Extrapolation to the full profile on Kaggle 2× T4 (1 km, 8 frames, ~530 k / 330 k cell-frames per mode)**:
section 07 runs three lanes side by side: XGBoost and the GNN share the GPU lanes (GNN benchmark, 43 folds × 3–5 min
≈ 1.5 h; XGBoost ~175 fits × 20–25 s ≈ 0.5 h), **LightGBM-GPU runs serially in its own lane on GPU 1** (≈ 0.5 h;
its OpenCL learner is not thread safe), and random forest + ridge occupy the CPU lane (≈ 0.5–1.7 h on 4 vCPU);
per-lane busy time is logged after 07. If the LightGBM GPU probe fails, LightGBM joins the CPU lane and adds about
1–2 h (about 3 h without early stopping). XAI (08, 4400-tree TreeSHAP of 8 frames, interactions, 50 bootstrap
replicates ≈ 50 s each per GPU) ≈ 1 h; timeline + downscaling ≈ 5 min; projections (20 land realisations × 3 SSPs,
500 draws) ≈ 45 min; export ≈ 2 min: **modelling ≈ 4.5 h**. Earth Engine extraction ≈ 1.5–3 h on the first run
with interactive `computePixels` (dominated by the 30 m Landsat medians; `asset` mode can add queueing, see item
5), minutes from the tile cache. **Expected total ≈ 6–7.5 h (an estimate; calibrate with the pilot)**, inside the
11 h time budget. If a session is tight: (1) `LST_STOP_AFTER=06` for the extraction in a first commit and resume the
modelling in a second one (attach the first version's output), (2) `LST_BENCHMARKS=ridge,random_forest,lightgbm,
xgboost` drops the GNN benchmark (≈ −1.5 h; headline metrics are XGBoost), (3) `LST_N_BOOTSTRAP=25` (≈ −30 min,
wider CI noise). Pilot (`LST_PROFILE=pilot`, 4 frames, ridge + XGBoost, no projections): ≈ 1–1.5 h including Earth
Engine.

## Loading the results into the dashboard
Download `web_bundle.zip` from the Kaggle *Output* tab and unzip it into `dashboard/public/data/` (so that
`dashboard/public/data/manifest.json` exists), then run `npm install && npm run dev` inside `dashboard/`. The
dashboard opens `public/data` by default (`?bundle=data`), or the synthetic demo in `public/demo` (`?bundle=demo`);
`VITE_DATA_BASE_URL` points it at a hosted bundle. Synthetic bundles always show a permanent "SIMULATED" banner.

## 0.1 Environment detection and dependencies
Installs use `subprocess` (no notebook magics, so the same code runs as a plain script). Packages are installed
only when an import is missing **and** we are on Kaggle (or `LST_ALLOW_PIP=1`); otherwise a warning is logged
and the dependent step falls back gracefully later.

In [ ]:
# Per-section code hashes (generated by build_notebook.py): part of every stage-cache key, so edited sections recompute.
# Edit notebook/sections and rebuild instead of hand-editing cells.
_CODE_SHA = {"00": "7f39b4ccb58a", "01": "e83f22867b77", "02": "02b1a6bff49e", "03": "300d146aeeb9", "04": "0681ba0583d0", "05": "986f285529cf", "06": "2b993579a70f", "07": "bad775697a0c", "08": "7d7ee347b0c1", "09": "409863b7d023", "10": "873241e695d3", "11": "0a7cb1d42a69", "12": "9c3af7691330"}

In [ ]:
import gc
import gzip
import hashlib
import importlib
import importlib.util
import inspect
import io
import json
import math
import os
import pickle
import platform
import queue
import random
import re
import shutil
import struct
import subprocess
import sys
import threading
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import contextmanager
from dataclasses import asdict, dataclass, field, fields, is_dataclass
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Any, Callable, Iterable, Optional, Sequence
import ast
import tarfile

_RUN_T0 = time.time()            # wall-clock start of this run: the time budget (section 0.5.1) counts from here
LST_STOPPED = False              # True once request_stop() was called; later cells are skipped (cell guard)
LST_STOP_REASON = None           # why the run stopped (time budget, LST_STOP_AFTER, ...)
_CURRENT_SECTION = "00"          # set by _lst_enter_section('NN') at the top of every later section

_IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle/working").exists()
_LOG_LOCK = threading.Lock()

# Legacy consoles (e.g. Windows cp1252) cannot encode symbols such as "°" or "→": replace instead of crashing.
for _stream in (sys.stdout, sys.stderr):
    if hasattr(_stream, "reconfigure"):
        try:
            _stream.reconfigure(errors="replace")
        except (ValueError, OSError):
            pass


def log(msg: str, level: str = "INFO") -> None:
    """Print a timestamped, levelled log line (thread-safe, flushed immediately).

    Levels are free-form but conventionally DEBUG / INFO / WARNING / ERROR. Output goes to stdout so that Kaggle's
    cell output and the local runner interleave messages in order.
    """
    stamp = datetime.now().strftime("%H:%M:%S")
    with _LOG_LOCK:
        print(f"[{stamp}] {level.upper():<7} {msg}", flush=True)


def _is_interactive() -> bool:
    """True inside a live notebook kernel that can prompt the user (not a Kaggle batch commit)."""
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "").lower() == "batch":
        return False
    return "ipykernel" in sys.modules


# pip distribution name -> import name (for the "is it already importable?" check)
_PIP_IMPORT_NAMES = {
    "earthengine-api": "ee",
    "pylandstats": "pylandstats",
    "libpysal": "libpysal",
    "esda": "esda",
    "pyarrow": "pyarrow",
    "shapely": "shapely",
    "pyproj": "pyproj",
    "lightgbm": "lightgbm",
    "shap": "shap",
    "pillow": "PIL",
    "cudf-cu12": "cudf",
    "cuml-cu12": "cuml",
}


def _import_name(spec: str) -> str:
    """Map a pip requirement spec (e.g. ``"shap>=0.44"``) to the module name it provides."""
    base = re.split(r"[<>=!~\[;\s]", spec.strip(), maxsplit=1)[0]
    return _PIP_IMPORT_NAMES.get(base.lower(), base.replace("-", "_").lower())


def _is_importable(module: str) -> bool:
    """Check importability without importing (cheap, no side effects)."""
    try:
        return importlib.util.find_spec(module) is not None
    except (ImportError, ValueError):
        return False


def _run_pip(specs: Sequence[str], extra_args: Sequence[str], timeout: int) -> bool:
    """Run one ``pip install`` subprocess; return True on exit code 0."""
    cmd = [sys.executable, "-m", "pip", "install", "-q", *extra_args, *specs]
    log(f"pip install {' '.join(specs)} {' '.join(extra_args)}".rstrip())
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        log(f"pip install timed out after {timeout}s for {list(specs)}", "WARNING")
        return False
    except OSError as exc:
        log(f"pip could not be started: {exc}", "WARNING")
        return False
    if proc.returncode != 0:
        log(f"pip failed (exit {proc.returncode}) for {list(specs)}:\n{proc.stderr[-1500:]}", "WARNING")
        return False
    return True


def pip_install(pkgs, extra_args: Sequence[str] = (), *, timeout: int = 900) -> bool:
    """Install missing packages with ``sys.executable -m pip install -q``.

    Only packages whose import is missing are installed, and only on Kaggle or when ``LST_ALLOW_PIP=1`` (we never
    modify a user's local/global Python silently). A failed group install is retried per package so one broken wheel
    does not block the others. Returns True when every requested package is importable afterwards.
    """
    specs = [pkgs] if isinstance(pkgs, str) else list(pkgs)
    missing = [s for s in specs if not _is_importable(_import_name(s))]
    if not missing:
        return True
    if not (_IS_KAGGLE or os.environ.get("LST_ALLOW_PIP") == "1"):
        log(f"Missing optional packages {missing}; not installing outside Kaggle "
            f"(set LST_ALLOW_PIP=1 to allow). Dependent steps will use fallbacks.", "WARNING")
        return False
    if not _run_pip(missing, extra_args, timeout) and len(missing) > 1:
        for spec in missing:
            _run_pip([spec], extra_args, timeout)
    importlib.invalidate_caches()
    still_missing = [s for s in missing if not _is_importable(_import_name(s))]
    if still_missing:
        log(f"Still missing after pip: {still_missing}", "WARNING")
    return not still_missing


_REQUIRED_OPTIONAL_PKGS = ["earthengine-api", "pylandstats", "libpysal", "esda", "pyarrow",
                           "shapely", "pyproj", "lightgbm", "shap", "pillow"]
pip_install(_REQUIRED_OPTIONAL_PKGS)
log(f"Environment: {'Kaggle' if _IS_KAGGLE else 'local'} | Python {platform.python_version()} | "
    f"{platform.platform()}")

### Optional RAPIDS install (`LST_INSTALL_RAPIDS=1`)
Runs **before** numpy/pandas are imported (section 0.2): a RAPIDS wheel may pull in different numpy/pandas
versions, and swapping them under an interpreter that has already imported the old ones leaves a mixed, broken
environment. The install is constrained to the numpy/pandas versions already installed (pip then picks a
compatible RAPIDS release or fails cleanly); `LST_RAPIDS_VERSION` (e.g. `25.2.*`) pins the release. Kaggle's GPU
image already ships cuDF/cuML, so there the install is a no-op. Using RAPIDS at all is a separate opt-in
(`LST_USE_RAPIDS=1`, section 0.4).

In [ ]:
def _dist_version(dist: str) -> Optional[str]:
    """Installed version of a distribution without importing it (None when absent)."""
    try:
        from importlib.metadata import PackageNotFoundError, version
    except ImportError:  # pragma: no cover - Python < 3.8
        return None
    try:
        return version(dist)
    except PackageNotFoundError:
        return None


def _install_rapids() -> bool:
    """pip-install cuDF/cuML constrained to the installed numpy/pandas; returns True when a restart is needed."""
    pinned = (os.environ.get("LST_RAPIDS_VERSION") or "").strip()
    specs = [f"{name}=={pinned}" if pinned else name for name in ("cudf-cu12", "cuml-cu12")]
    before = {d: _dist_version(d) for d in ("numpy", "pandas")}
    already_imported = [d for d in before if d in sys.modules]
    constraint_path = None
    extra = ["--extra-index-url=https://pypi.nvidia.com"]
    pins = [f"{d}=={v}" for d, v in before.items() if v]
    if pins:
        import tempfile
        with tempfile.NamedTemporaryFile("w", suffix="_lst_constraints.txt", delete=False) as fh:
            fh.write("\n".join(pins) + "\n")
            constraint_path = fh.name
        extra += ["--constraint", constraint_path]
    log(f"LST_INSTALL_RAPIDS=1: installing {specs} constrained to {pins or 'nothing'} (several minutes)", "WARNING")
    try:
        pip_install(specs, extra_args=extra, timeout=2400)
    finally:
        if constraint_path:
            try:
                os.remove(constraint_path)
            except OSError:
                pass
    after = {d: _dist_version(d) for d in before}
    changed = {d: (before[d], after[d]) for d in before if before[d] != after[d]}
    if changed:
        loaded = f" ({', '.join(already_imported)} already imported in this kernel)" if already_imported else ""
        log(f"RAPIDS install changed {changed}{loaded}. RESTART THE KERNEL and run again before continuing.",
            "ERROR")
        return True
    return False


if os.environ.get("LST_INSTALL_RAPIDS", "").strip().lower() in {"1", "true", "yes", "on", "y"}:
    if _install_rapids():
        raise RuntimeError("RAPIDS installation changed numpy/pandas; restart the kernel and run the notebook "
                           "again (the install is skipped next time because cuDF/cuML are then importable).")

## 0.2 Imports
Core scientific stack first; every optional/accelerator library is imported defensively so that a missing package
only disables the feature that needs it. Matplotlib is forced to the non-interactive `Agg` backend when no
notebook kernel is present (headless scripts, CI).

In [ ]:
import numpy as np
import pandas as pd
import scipy
from scipy import ndimage, sparse, stats
import matplotlib

if not os.environ.get("MPLBACKEND") and "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sklearn

# plt.show() under Agg warns "FigureCanvasAgg is non-interactive" - harmless in headless runs.
warnings.filterwarnings("ignore", message=".*non-interactive.*")
warnings.filterwarnings("ignore", message=".*FigureCanvasAgg.*")


def _optional_import(module: str):
    """Import ``module`` or return None (any exception, e.g. CUDA driver errors, counts as missing)."""
    try:
        return importlib.import_module(module)
    except Exception as exc:  # noqa: BLE001 - optional dependency, any failure means "unavailable"
        log(f"optional import '{module}' unavailable: {type(exc).__name__}: {str(exc)[:160]}", "DEBUG")
        return None


torch = _optional_import("torch")
xgb = _optional_import("xgboost")
lgb = _optional_import("lightgbm")
shap = _optional_import("shap")
ee = _optional_import("ee")
libpysal = _optional_import("libpysal")
esda = _optional_import("esda")
pylandstats = _optional_import("pylandstats")
shapely = _optional_import("shapely")
pyproj = _optional_import("pyproj")
PIL = _optional_import("PIL")
if PIL is not None:
    PIL_Image = _optional_import("PIL.Image")   # Pillow image module (PNG rasters in section 10)
else:
    PIL_Image = None

_OPTIONAL_MODULES = {
    "torch": torch, "xgboost": xgb, "lightgbm": lgb, "shap": shap, "earthengine-api": ee,
    "libpysal": libpysal, "esda": esda, "pylandstats": pylandstats, "shapely": shapely, "pyproj": pyproj,
    "pillow": PIL,
}
log("versions: " + ", ".join(
    [f"numpy {np.__version__}", f"pandas {pd.__version__}", f"scipy {scipy.__version__}",
     f"sklearn {sklearn.__version__}", f"matplotlib {matplotlib.__version__}"]
    + [f"{name} {getattr(mod, '__version__', '?')}" if mod is not None else f"{name} MISSING"
       for name, mod in _OPTIONAL_MODULES.items()]))

## 0.3 Configuration (`CFG`, v2)
One dataclass holds every tunable (SPEC §1.6). Defaults come from the **run profile** (`full`, `fast`, `pilot`);
`LST_*` environment variables override them. Study-design constants (frames, windows, scenarios, modes) are fixed
by the SPEC and identical in every profile except `pilot`, which probes Earth Engine cost on four frames (1990,
2000, 2020, 2025) and skips projections (downstream sections must therefore handle an **empty** `FRAMES_PROJ`);
`LST_FRAMES_OBS` overrides the frames of any profile (validated: SPEC frames only, ≥ 2 frames, ≥ 2 night training
frames, 2025, and 2005/2015/2025 when projecting). Run control (time budget, `LST_STOP_AFTER`, cache packing,
`LST_INVALIDATE`) and the Earth Engine session settings (`LST_GEE_ENDPOINT`, `LST_GEE_DEADLINE_S`) are CFG fields
too.

The land-cover resolution for landscape metrics depends on the data mode, which section 1 decides:
`CFG.resolve_mode(mode)` fixes `LC_FINE_RES_M` (snapped to a divisor of `GRID_RES_M`) and `CFG.fine_res()`
returns it (v1 API). The 250 m analysis fine grid is `GRID_RES_M // FINE_FACTOR` (`CFG.grid_fine_res()`).

In [ ]:
def _env_str(name: str, default: Optional[str] = None) -> Optional[str]:
    value = os.environ.get(name)
    return value.strip() if value is not None and value.strip() else default


def _env_int(name: str, default: Optional[int] = None) -> Optional[int]:
    value = _env_str(name)
    if value is None:
        return default
    try:
        return int(float(value))
    except ValueError as exc:
        raise ValueError(f"Environment variable {name}={value!r} is not an integer") from exc


def _env_bool(name: str, default: bool = False) -> bool:
    value = _env_str(name)
    if value is None:
        return default
    low = value.lower()
    if low in {"1", "true", "yes", "on", "y"}:
        return True
    if low in {"0", "false", "no", "off", "n"}:
        return False
    raise ValueError(f"Environment variable {name}={value!r} is not a boolean (use 1/0, true/false)")


def _kaggle_secret(name: str) -> Optional[str]:
    """Read a Kaggle notebook secret; None when not on Kaggle or the secret is not attached."""
    if not _IS_KAGGLE:
        return None
    try:
        from kaggle_secrets import UserSecretsClient  # available only inside Kaggle kernels
        value = UserSecretsClient().get_secret(name)
        return value.strip() if value else None
    except Exception:  # noqa: BLE001 - absent secret raises a generic exception
        return None


def _nearest_divisor(n: int, target: int) -> int:
    """Divisor of ``n`` closest to ``target`` (ties resolved towards the larger, cheaper divisor)."""
    divisors = [d for d in range(1, n + 1) if n % d == 0]
    return min(divisors, key=lambda d: (abs(d - target), -d))


# SPEC §1.4: the final model of each mode. ``device`` and ``seed`` are injected per call (xgb_device_for(gpu_id),
# CFG.SEED + fold offset) so that the same dict can drive parallel folds on different GPUs.
_XGB_DEFAULT_PARAMS = {
    "objective": "reg:squarederror", "tree_method": "hist", "learning_rate": 0.03, "max_depth": 6,
    "min_child_weight": 20, "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 1.0,
    "reg_alpha": 0.0, "max_bin": 256, "eval_metric": "rmse",
}
_VALID_RUN_MODES = ("auto", "gee", "synthetic")
_VALID_PROFILES = ("full", "fast", "pilot")
_VALID_BENCHMARKS = ("ridge", "random_forest", "lightgbm", "xgboost", "gnn")
_VALID_GEE_EXPORT = ("interactive", "asset")
_VALID_GEE_ENDPOINTS = ("standard", "highvolume")
_GEE_DEADLINE_MIN_S = 330.0              # must exceed the ~300 s interactive server limit (else retries never help)
_ALL_BENCHMARKS = ["ridge", "random_forest", "lightgbm", "xgboost", "gnn"]
# Code identity of the shared helpers of sections 00/01/02. Bump it MANUALLY when a helper defined there changes
# results (e.g. fetch_grid, block_mean, GridSpec): every stage-cache key carries it (SPEC §3.1, CFG.key).
SHARED_CODE_TAG = "v2.0"

# SPEC §1.2 - observation frames (Apr-Jun of a multi-year window), projections, scenarios
_FRAMES_OBS_ALL = [1990, 1995, 2000, 2005, 2010, 2015, 2020, 2025]
_WINDOWS_ALL = {1990: (1989, 1992), 1995: (1994, 1996), 2000: (1999, 2001), 2005: (2004, 2006),
                2010: (2009, 2011), 2015: (2014, 2016), 2020: (2019, 2021), 2025: (2024, 2026)}
_FRAMES_PROJ_ALL = [2030, 2035, 2040, 2045, 2050]
_NIGHT_TRAIN_ALL = [2005, 2010, 2015, 2020, 2025]

# SPEC §1.6 - profile-dependent defaults. LC_FINE_RES_M is a {data_mode: metres} map (resolved in section 1).
_PROFILE_DEFAULTS = {
    "full": {"N_BOOTSTRAP": 50, "N_MC": 500, "N_LAND_REAL": 20, "BLOCK_KM": 20, "N_FOLDS_BLOCK": 10,
             "XGB_MAX_ROUNDS": 4000, "XGB_EARLY_STOP": 100, "GNN_EPOCHS": 300, "MORAN_PERMS": 999,
             "BENCHMARKS": list(_ALL_BENCHMARKS), "LC_FINE_RES_M": {"gee": 50, "synthetic": 100},
             "FRAMES_OBS": list(_FRAMES_OBS_ALL), "FRAMES_PROJ": list(_FRAMES_PROJ_ALL)},
    "fast": {"N_BOOTSTRAP": 6, "N_MC": 120, "N_LAND_REAL": 4, "BLOCK_KM": 20, "N_FOLDS_BLOCK": 5,
             "XGB_MAX_ROUNDS": 400, "XGB_EARLY_STOP": 30, "GNN_EPOCHS": 30, "MORAN_PERMS": 99,
             "BENCHMARKS": list(_ALL_BENCHMARKS), "LC_FINE_RES_M": {"gee": 200, "synthetic": 200},
             "FRAMES_OBS": list(_FRAMES_OBS_ALL), "FRAMES_PROJ": list(_FRAMES_PROJ_ALL)},
    # Pilot = Earth Engine cost/runtime probe on the risky paths: TM-only Landsat 5 (1990), the Terra-reconstructed
    # night frame and the early GHSL / GLC five-years-map epochs (2000), the dense L7/L8/L9 windows (2020, 2025);
    # no projections (N_MC / N_LAND_REAL are unused -> 0); section 05 probes the CMIP6 warming pattern instead.
    "pilot": {"N_BOOTSTRAP": 10, "N_MC": 0, "N_LAND_REAL": 0, "BLOCK_KM": 20, "N_FOLDS_BLOCK": 5,
              "XGB_MAX_ROUNDS": 1500, "XGB_EARLY_STOP": 60, "GNN_EPOCHS": 60, "MORAN_PERMS": 199,
              "BENCHMARKS": ["ridge", "xgboost"], "LC_FINE_RES_M": {"gee": 50, "synthetic": 50},
              "FRAMES_OBS": [1990, 2000, 2020, 2025], "FRAMES_PROJ": []},
}


def _projection_window(frame: int) -> tuple:
    """20-year climate window centred on a projection frame (SPEC §3.2: 2025 → 2016-2035, 2050 → 2041-2060)."""
    return (int(frame) - 9, int(frame) + 10)


def _parse_benchmarks(text: str) -> list:
    """Comma/space separated model list → validated, de-duplicated list in canonical order."""
    wanted = [t.strip().lower() for t in re.split(r"[,\s]+", text) if t.strip()]
    unknown = sorted(set(wanted) - set(_VALID_BENCHMARKS))
    if unknown:
        raise ValueError(f"LST_BENCHMARKS: unknown model(s) {unknown}; choose from {list(_VALID_BENCHMARKS)}")
    return [m for m in _VALID_BENCHMARKS if m in wanted]


_CODE_SHA_WARNED = False


def _warn_no_code_sha() -> None:
    """One WARNING per session when the per-section code hashes are missing (cell run outside the built notebook)."""
    global _CODE_SHA_WARNED
    if not _CODE_SHA_WARNED:
        _CODE_SHA_WARNED = True
        log("_CODE_SHA is not defined (notebook not built by build_notebook.py, or its first code cell was not run): "
            "stage-cache keys carry no code identity, so edited code may reuse stale cached stages", "WARNING")


def section_code_hash(*ids) -> Optional[str]:
    """Combined code hash of sections ``ids`` (e.g. ``section_code_hash('07')``) for cross-section stage keys.

    sha1 of the joined ``'NN:<sha>'`` pairs, first 12 hex digits; None when ``_CODE_SHA`` is not defined.
    """
    shas = globals().get("_CODE_SHA")
    if not isinstance(shas, dict):
        _warn_no_code_sha()
        return None
    norm = [f"{int(i):02d}" if str(i).isdigit() else str(i) for i in ids]
    text = ";".join(f"{i}:{shas.get(i)}" for i in norm)
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:12]


def _parse_frames(text: str, name: str = "LST_FRAMES_OBS") -> list:
    """Comma/space separated observation frames -> sorted unique ints (validated against SPEC §1.2 in LSTConfig)."""
    out = []
    for tok in (t.strip() for t in re.split(r"[,\s]+", text) if t.strip()):
        try:
            out.append(int(tok))
        except ValueError as exc:
            raise ValueError(f"{name}: {tok!r} is not a year; choose from {_FRAMES_OBS_ALL}") from exc
    return sorted(set(out))


_OFF_WORDS = {"0", "none", "off", "no", "false", "n", "unlimited", ""}


def _env_hours(name: str, default: Optional[float]) -> Optional[float]:
    """Positive hours from ``name``; '0' / 'none' / 'off' mean unlimited (None)."""
    value = _env_str(name)
    if value is None:
        return default
    if value.lower() in _OFF_WORDS:
        return None
    try:
        hours = float(value)
    except ValueError as exc:
        raise ValueError(f"Environment variable {name}={value!r} is not a number of hours (or 0/none/off)") from exc
    if not (math.isfinite(hours) and hours > 0):
        raise ValueError(f"{name}={value!r} must be > 0 hours (or 0/none/off for no budget)")
    return hours


def _env_float(name: str, default: float) -> float:
    value = _env_str(name)
    if value is None:
        return float(default)
    try:
        return float(value)
    except ValueError as exc:
        raise ValueError(f"Environment variable {name}={value!r} is not a number") from exc


def _parse_section_id(value: Optional[str], name: str = "LST_STOP_AFTER") -> Optional[str]:
    """'6' / '06' -> '06' (sections 01..11); None / '' / 'none' / 'off' -> None."""
    if value is None or str(value).strip().lower() in _OFF_WORDS - {"0"}:
        return None
    text = str(value).strip()
    if not re.fullmatch(r"\d{1,2}", text) or not 1 <= int(text) <= 11:
        raise ValueError(f"{name}={value!r}: give a section number 01..11 (the run stops cleanly after it)")
    return f"{int(text):02d}"


def _parse_invalidate(text: Optional[str]) -> list:
    """LST_INVALIDATE comma tokens (stage-name prefixes such as 06, 07_job, s04)."""
    if not text:
        return []
    tokens = [t.strip() for t in re.split(r"[,\s]+", text) if t.strip()]
    bad = [t for t in tokens if not re.fullmatch(r"[A-Za-z0-9_.\-]+", t)]
    if bad:
        raise ValueError(f"LST_INVALIDATE: invalid token(s) {bad} (use stage-name prefixes such as 06,07_job)")
    return tokens


@dataclass
class LSTConfig:
    """Pipeline configuration v2 (SPEC §1.6 + v1 fields). Construct with :meth:`from_env`."""

    # --- run control -------------------------------------------------------------------------------------------
    PROFILE: str = "full"
    RUN_MODE: str = "auto"
    FAST_DEV: bool = False                    # v1 flag; True iff PROFILE == "fast"
    RESUME: bool = True
    CACHE_IMPORT: str = "auto"                # "auto" | folder path | "off"
    INVALIDATE: list = field(default_factory=list)       # LST_INVALIDATE stage-name prefixes forced to recompute
    PACK_CACHE: bool = False                  # LST_PACK_CACHE: pack the cache into tar files (default on Kaggle)
    TIME_BUDGET_H: Optional[float] = None     # LST_TIME_BUDGET_H (11 on Kaggle): stop starting work after this
    TIME_RESERVE_S: float = 1800.0            # LST_TIME_RESERVE_S: seconds kept free for packing + export
    STOP_AFTER: Optional[str] = None          # LST_STOP_AFTER '01'..'11': stop cleanly after that section
    # --- study design (SPEC §1.1-§1.3) -------------------------------------------------------------------------
    FRAMES_OBS: list = field(default_factory=lambda: list(_FRAMES_OBS_ALL))
    WINDOWS: dict = field(default_factory=lambda: dict(_WINDOWS_ALL))
    FRAMES_PROJ: list = field(default_factory=lambda: list(_FRAMES_PROJ_ALL))
    PROJ_WINDOWS: dict = field(default_factory=dict)       # filled in __post_init__ (20-year GCM windows)
    SCENARIOS: list = field(default_factory=lambda: ["ssp126", "ssp245", "ssp585"])
    NOW: int = 2025
    REFERENCE_PERIOD: tuple = (2000, 2020)
    MODES: list = field(default_factory=lambda: ["day", "night"])
    NIGHT_TRAIN_FRAMES: list = field(default_factory=lambda: list(_NIGHT_TRAIN_ALL))
    SEASON: tuple = ("04-01", "06-30")
    SPRING_SEASON: tuple = ("02-01", "03-31")             # ndvi_spring_f window (SPEC §2.1)
    DAY_REFERENCE_OVERPASS_IST_H: float = 10.75           # 10:45 IST reference overpass (SPEC §4.7)
    CRS: str = "EPSG:32643"
    # --- grids -------------------------------------------------------------------------------------------------
    GRID_RES_M: int = 1000
    FINE_FACTOR: int = 4
    LC_FINE_RES_M: Optional[int] = None       # resolved by resolve_mode(); read via fine_res()
    LC_FINE_RES_DEFAULTS: dict = field(default_factory=lambda: {"gee": 50, "synthetic": 100})
    LC_FINE_RES_REQUEST: Optional[int] = None  # LST_LC_FINE_RES_M (legacy LST_FINE_RES_M); overrides the defaults
    LM_WINDOW_CELLS: int = 1
    TARGET_MODE: str = "anomaly"
    # --- target validity (SPEC §3.3; used by section 06) --------------------------------------------------------
    DAY_MIN_NOBS: float = 2.0                 # 1 km mean of fine clear-scene counts
    FINE_MIN_VALID_FRAC: float = 0.5          # >= 50 % valid fine pixels per 1 km cell
    # Aqua 8-day composites passing QC (mandatory QA <= 1, LST error <= 2 K). Real data (2026-11, Apr-Jun 2019-2021
    # over NCR): median 4.8 per 1 km cell; >= 3 in 77 % of cells, >= 6 in only 30 % (the SPEC's first value, 6).
    NIGHT_MIN_NOBS: int = 3
    # Terra-reconstructed night frame (2 seasons, sparser: >= 3 keeps 10 % of the fixed mask, >= 1 about 70 %); weight 0.
    NIGHT_MIN_NOBS_TERRA: int = 1
    NIGHT_MIN_NOBS_VIIRS: int = 20            # VIIRS daily observations
    LST_MIN_OBS: int = 3                      # v1 meaning (MODIS composites), kept for compatibility
    LST_MIN_COVERAGE: float = 0.5             # v1 meaning
    # --- validation / models -----------------------------------------------------------------------------------
    SEED: int = 42
    N_FOLDS: int = 5
    SPATIAL_BLOCKS: tuple = (5, 5)
    INNER_BLOCK_KM: int = 10
    BLOCK_KM: int = 20
    N_FOLDS_BLOCK: int = 10
    BENCHMARKS: list = field(default_factory=lambda: list(_ALL_BENCHMARKS))
    BOOT_BLOCK_KM: Optional[int] = None       # None = derived from the residual correlogram in section 08
    N_BOOTSTRAP: int = 50
    N_MC: int = 500
    N_LAND_REAL: int = 20
    K_ZONES: int = 4
    DEP_BINS: int = 40
    MORAN_K: int = 8
    MORAN_PERMS: int = 999
    XGB_PARAMS: dict = field(default_factory=lambda: dict(_XGB_DEFAULT_PARAMS))
    XGB_MAX_ROUNDS: int = 4000
    XGB_EARLY_STOP: int = 100
    GNN_EPOCHS: int = 300
    SHAP_BATCH: int = 16384
    SHAP_INTERACTION_MAX_ROWS: Optional[int] = 4000       # None on GPU (set by apply_hardware)
    SHAP_INTERACTION_ROWS_PER_FRAME: int = 5000           # SPEC §4.4 sample for grouped interactions
    # --- Earth Engine --------------------------------------------------------------------------------------------
    GEE_PROJECT: Optional[str] = None
    GEE_ENDPOINT: str = "standard"            # LST_GEE_ENDPOINT: "standard" | "highvolume" (opt-in)
    GEE_DEADLINE_S: float = 420.0             # LST_GEE_DEADLINE_S: client request deadline (>= 330 s)
    GEE_EXPORT: str = "interactive"
    GEE_ASSET_ROOT: Optional[str] = None      # default projects/<GEE_PROJECT>/assets/ncr_lst_v2 (resolved in 01)
    GEE_TILE_PX: int = 128
    GEE_FINE_TILE_PX: int = 2048              # single-band uint8 land-cover tiles
    GEE_STACK_TILE_PX: int = 512              # multi-band float32 fine stacks (48 MB request limit)
    GEE_WORKERS: int = 6
    GEE_MAX_RETRIES: int = 6
    LC_SOURCES: dict = field(default_factory=dict)        # one product for every frame (SPEC §1.4)
    # --- hardware / output ---------------------------------------------------------------------------------------
    INSTALL_RAPIDS: bool = False
    USE_RAPIDS: bool = False                  # cuDF/cuML paths are opt-in (LST_USE_RAPIDS / LST_INSTALL_RAPIDS)
    OUTPUT_DIR: Path = Path("outputs")
    CACHE_DIR: Path = field(init=False)
    STAGE_DIR: Path = field(init=False)
    FIG_DIR: Path = field(init=False)
    PARQUET_DIR: Path = field(init=False)
    GEO_DIR: Path = field(init=False)
    MODEL_DIR: Path = field(init=False)
    NPZ_DIR: Path = field(init=False)
    WEB_DIR: Path = field(init=False)
    DATA_MODE: Optional[str] = None           # "gee" | "synthetic" once resolved by section 1
    ENV_OVERRIDES: dict = field(default_factory=dict)     # LST_* variables that changed a default (for the log)

    def __post_init__(self) -> None:
        if self.PROFILE not in _VALID_PROFILES:
            raise ValueError(f"PROFILE must be one of {_VALID_PROFILES}, got {self.PROFILE!r}")
        if self.RUN_MODE not in _VALID_RUN_MODES:
            raise ValueError(f"RUN_MODE must be one of {_VALID_RUN_MODES}, got {self.RUN_MODE!r}")
        if self.TARGET_MODE not in ("anomaly", "absolute"):
            raise ValueError(f"TARGET_MODE must be 'anomaly' or 'absolute', got {self.TARGET_MODE!r}")
        if self.GEE_EXPORT not in _VALID_GEE_EXPORT:
            raise ValueError(f"GEE_EXPORT must be one of {_VALID_GEE_EXPORT}, got {self.GEE_EXPORT!r}")
        if self.GEE_ENDPOINT not in _VALID_GEE_ENDPOINTS:
            raise ValueError(f"GEE_ENDPOINT (LST_GEE_ENDPOINT) must be one of {_VALID_GEE_ENDPOINTS}, got "
                             f"{self.GEE_ENDPOINT!r}")
        if not math.isfinite(float(self.GEE_DEADLINE_S)) or self.GEE_DEADLINE_S < _GEE_DEADLINE_MIN_S:
            log(f"GEE_DEADLINE_S={self.GEE_DEADLINE_S} is below the minimum {_GEE_DEADLINE_MIN_S:.0f} s (a shorter "
                "client deadline cuts off requests the server would still finish); using "
                f"{_GEE_DEADLINE_MIN_S:.0f} s", "WARNING")
            self.GEE_DEADLINE_S = _GEE_DEADLINE_MIN_S
        if self.TIME_BUDGET_H is not None and not (math.isfinite(self.TIME_BUDGET_H) and self.TIME_BUDGET_H > 0):
            raise ValueError(f"TIME_BUDGET_H must be > 0 hours or None, got {self.TIME_BUDGET_H!r}")
        if not (math.isfinite(float(self.TIME_RESERVE_S)) and self.TIME_RESERVE_S >= 0):
            raise ValueError(f"TIME_RESERVE_S must be >= 0 seconds, got {self.TIME_RESERVE_S!r}")
        self.STOP_AFTER = _parse_section_id(self.STOP_AFTER, "STOP_AFTER")
        if self.GRID_RES_M <= 0:
            raise ValueError(f"GRID_RES_M must be positive, got {self.GRID_RES_M}")
        if self.FINE_FACTOR < 1 or self.GRID_RES_M % self.FINE_FACTOR:
            raise ValueError(f"FINE_FACTOR={self.FINE_FACTOR} must be >= 1 and divide GRID_RES_M={self.GRID_RES_M}")
        if self.LC_FINE_RES_REQUEST is not None and self.LC_FINE_RES_REQUEST <= 0:
            raise ValueError(f"LC_FINE_RES_M must be positive, got {self.LC_FINE_RES_REQUEST}")
        if self.BOOT_BLOCK_KM is not None and self.BOOT_BLOCK_KM <= 0:
            raise ValueError(f"BOOT_BLOCK_KM must be positive or None, got {self.BOOT_BLOCK_KM}")
        for name in ("N_BOOTSTRAP", "N_MC", "N_LAND_REAL"):
            if getattr(self, name) < 0:
                raise ValueError(f"{name} must be >= 0, got {getattr(self, name)}")
        if self.N_FOLDS_BLOCK < 2:
            raise ValueError(f"N_FOLDS_BLOCK must be >= 2, got {self.N_FOLDS_BLOCK}")
        if "xgboost" not in self.BENCHMARKS:
            raise ValueError("BENCHMARKS must include 'xgboost' (it is the final/headline model)")
        self.FRAMES_OBS = sorted({int(y) for y in self.FRAMES_OBS})
        self.FRAMES_PROJ = sorted({int(y) for y in self.FRAMES_PROJ})
        hint = ("set LST_FRAMES_OBS to a comma list of SPEC frames, e.g. 1990,2000,2020,2025 for a pilot or leave it "
                "unset for the profile default")
        unknown = [y for y in self.FRAMES_OBS if y not in _FRAMES_OBS_ALL]
        if unknown:
            raise ValueError(f"FRAMES_OBS {self.FRAMES_OBS}: {unknown} are not SPEC frames {_FRAMES_OBS_ALL}; {hint}")
        missing_windows = [y for y in self.FRAMES_OBS if y not in self.WINDOWS]
        if missing_windows:
            raise ValueError(f"CFG.WINDOWS has no window for observed frame(s) {missing_windows}")
        if len(self.FRAMES_OBS) < 2:
            raise ValueError(f"FRAMES_OBS {self.FRAMES_OBS}: at least 2 observed frames are needed (leave-one-epoch-"
                             f"out validation and frame weights); {hint}")
        # Night training frames are those observed frames that carry an Aqua/VIIRS target (SPEC §3.3); the pilot
        # profile therefore trains night on its subset only.
        self.NIGHT_TRAIN_FRAMES = [y for y in sorted(self.NIGHT_TRAIN_FRAMES) if y in self.FRAMES_OBS]
        if len(self.NIGHT_TRAIN_FRAMES) < 2:
            raise ValueError(f"FRAMES_OBS {self.FRAMES_OBS} leave {len(self.NIGHT_TRAIN_FRAMES)} night training "
                             f"frame(s) {self.NIGHT_TRAIN_FRAMES}; the night model needs at least 2 of "
                             f"{_NIGHT_TRAIN_ALL}; {hint}")
        if self.NOW not in self.FRAMES_OBS:
            raise ValueError(f"FRAMES_OBS {self.FRAMES_OBS} must contain NOW={self.NOW} (the baseline frame); {hint}")
        if self.FRAMES_PROJ:
            need = {2005, 2015, 2025} - set(self.FRAMES_OBS)
            if need:
                raise ValueError(f"FRAMES_OBS {self.FRAMES_OBS} lacks {sorted(need)}: projections (FRAMES_PROJ "
                                 f"{self.FRAMES_PROJ}) need 2005, 2015 and 2025 (land-change calibration and "
                                 f"hindcast); {hint}, or use LST_PROFILE=pilot (no projections)")
        if self.FRAMES_PROJ and (self.N_MC < 1 or self.N_LAND_REAL < 1):
            raise ValueError("projections requested (FRAMES_PROJ non-empty) but N_MC / N_LAND_REAL < 1")
        if self.FRAMES_PROJ and min(self.FRAMES_PROJ) <= self.NOW:
            raise ValueError(f"projection frames must lie after NOW={self.NOW}: {self.FRAMES_PROJ}")
        self.PROJ_WINDOWS = {y: _projection_window(y) for y in [self.NOW] + list(self.FRAMES_PROJ)}
        self.LC_SOURCES = {y: "glc_fcs30d" for y in self.FRAMES_OBS}
        self.OUTPUT_DIR = Path(self.OUTPUT_DIR).expanduser().resolve()
        self.CACHE_DIR = self.OUTPUT_DIR / "cache"
        self.STAGE_DIR = self.CACHE_DIR / "stages"
        self.FIG_DIR = self.OUTPUT_DIR / "figures"
        self.PARQUET_DIR = self.OUTPUT_DIR / "parquet"
        self.GEO_DIR = self.OUTPUT_DIR / "geo"
        self.MODEL_DIR = self.OUTPUT_DIR / "models"
        self.NPZ_DIR = self.OUTPUT_DIR / "npz"
        self.WEB_DIR = self.OUTPUT_DIR / "web"

    # v1 compatibility: v1 code iterates CFG.EPOCHS; in v2 the observed frames play that role.
    @property
    def EPOCHS(self) -> list:
        return self.FRAMES_OBS

    @property
    def FINE_RES_M(self) -> Optional[int]:
        """v1 name of the land-cover fine resolution (= LC_FINE_RES_M)."""
        return self.LC_FINE_RES_M

    @classmethod
    def from_env(cls) -> "LSTConfig":
        """Build the configuration from profile defaults + ``LST_*`` environment variables + Kaggle secrets."""
        overrides: dict = {}

        def _note(name: str) -> None:
            if _env_str(name) is not None:
                overrides[name] = _env_str(name)

        fast_flag = _env_bool("LST_FAST_DEV", False)
        profile_env = (_env_str("LST_PROFILE") or "").lower() or None
        if profile_env is not None and profile_env not in _VALID_PROFILES:
            raise ValueError(f"LST_PROFILE={profile_env!r}; choose from {_VALID_PROFILES}")
        if profile_env is not None:
            profile = profile_env
            if fast_flag and profile != "fast":
                log(f"LST_FAST_DEV=1 ignored because LST_PROFILE={profile} was set explicitly", "WARNING")
        else:
            profile = "fast" if fast_flag else "full"
        d = _PROFILE_DEFAULTS[profile]
        for name in ("LST_PROFILE", "LST_FAST_DEV", "LST_RUN_MODE", "LST_GRID_RES_M", "LST_FINE_FACTOR",
                     "LST_LC_FINE_RES_M", "LST_FINE_RES_M", "LST_N_BOOTSTRAP", "LST_N_MC", "LST_N_LAND_REAL",
                     "LST_BENCHMARKS", "LST_GEE_EXPORT", "LST_GEE_ASSET_ROOT", "LST_RESUME", "LST_CACHE_IMPORT",
                     "LST_OUTPUT_DIR", "LST_GEE_WORKERS", "LST_BOOT_BLOCK_KM", "LST_LST_MIN_OBS",
                     "LST_USE_RAPIDS", "LST_INSTALL_RAPIDS", "LST_TARGET_MODE",
                     # v2 run control (D21 k); some are read by their own section (03-06)
                     "LST_FRAMES_OBS", "LST_PILOT_PROBES", "LST_GEE_ENDPOINT", "LST_GEE_DEADLINE_S",
                     "LST_GEE_EXPORT_TIMEOUT_S", "LST_GEE_EXPORT_QUEUE_S", "LST_GEE_EXPORT_TIMEOUT_0405_S",
                     "LST_TIME_BUDGET_H", "LST_TIME_RESERVE_S", "LST_STOP_AFTER", "LST_INVALIDATE",
                     "LST_PACK_CACHE", "LST_FRAME_MIN_COVERAGE", "LST_DAY_MIN_FRAME_COVERAGE"):
            _note(name)
        frames_obs = (_parse_frames(_env_str("LST_FRAMES_OBS")) if _env_str("LST_FRAMES_OBS")
                      else list(d["FRAMES_OBS"]))

        cache_import = _env_str("LST_CACHE_IMPORT", "auto")
        if cache_import.lower() in {"0", "off", "false", "no", "none", "n"}:
            cache_import = "off"
        elif cache_import.lower() == "auto":
            cache_import = "auto"
        lc_request = _env_int("LST_LC_FINE_RES_M", None)
        if lc_request is None and _env_int("LST_FINE_RES_M", None) is not None:
            lc_request = _env_int("LST_FINE_RES_M", None)
            log("LST_FINE_RES_M is the v1 name of LST_LC_FINE_RES_M (applied)", "DEBUG")
        default_out = Path("/kaggle/working/outputs") if _IS_KAGGLE else Path.cwd() / "outputs"
        cfg = cls(
            PROFILE=profile,
            RUN_MODE=(_env_str("LST_RUN_MODE", "auto") or "auto").lower(),
            FAST_DEV=profile == "fast",
            RESUME=_env_bool("LST_RESUME", True),
            CACHE_IMPORT=cache_import,
            INVALIDATE=_parse_invalidate(_env_str("LST_INVALIDATE")),
            PACK_CACHE=_env_bool("LST_PACK_CACHE", _IS_KAGGLE),
            TIME_BUDGET_H=_env_hours("LST_TIME_BUDGET_H", 11.0 if _IS_KAGGLE else None),
            TIME_RESERVE_S=_env_float("LST_TIME_RESERVE_S", 1800.0),
            STOP_AFTER=_parse_section_id(_env_str("LST_STOP_AFTER"), "LST_STOP_AFTER"),
            FRAMES_OBS=frames_obs,
            FRAMES_PROJ=list(d["FRAMES_PROJ"]),
            GRID_RES_M=_env_int("LST_GRID_RES_M", 1000),
            FINE_FACTOR=_env_int("LST_FINE_FACTOR", 4),
            LC_FINE_RES_DEFAULTS=dict(d["LC_FINE_RES_M"]),
            LC_FINE_RES_REQUEST=lc_request,
            TARGET_MODE=(_env_str("LST_TARGET_MODE", "anomaly") or "anomaly").lower(),
            BLOCK_KM=d["BLOCK_KM"],
            N_FOLDS_BLOCK=d["N_FOLDS_BLOCK"],
            BENCHMARKS=(_parse_benchmarks(_env_str("LST_BENCHMARKS")) if _env_str("LST_BENCHMARKS")
                        else list(d["BENCHMARKS"])),
            N_BOOTSTRAP=_env_int("LST_N_BOOTSTRAP", d["N_BOOTSTRAP"]),
            N_MC=_env_int("LST_N_MC", d["N_MC"]),
            N_LAND_REAL=_env_int("LST_N_LAND_REAL", d["N_LAND_REAL"]),
            XGB_MAX_ROUNDS=d["XGB_MAX_ROUNDS"],
            XGB_EARLY_STOP=d["XGB_EARLY_STOP"],
            GNN_EPOCHS=d["GNN_EPOCHS"],
            MORAN_PERMS=d["MORAN_PERMS"],
            GEE_PROJECT=_env_str("LST_GEE_PROJECT") or _kaggle_secret("GEE_PROJECT"),
            GEE_ENDPOINT=(_env_str("LST_GEE_ENDPOINT", "standard") or "standard").lower(),
            GEE_DEADLINE_S=_env_float("LST_GEE_DEADLINE_S", 420.0),
            GEE_EXPORT=(_env_str("LST_GEE_EXPORT", "interactive") or "interactive").lower(),
            GEE_ASSET_ROOT=_env_str("LST_GEE_ASSET_ROOT"),
            GEE_WORKERS=max(1, min(40, _env_int("LST_GEE_WORKERS", 6))),
            INSTALL_RAPIDS=_env_bool("LST_INSTALL_RAPIDS", False),
            USE_RAPIDS=_env_bool("LST_USE_RAPIDS", False) or _env_bool("LST_INSTALL_RAPIDS", False),
            BOOT_BLOCK_KM=_env_int("LST_BOOT_BLOCK_KM", None),
            LST_MIN_OBS=_env_int("LST_LST_MIN_OBS", 3),
            OUTPUT_DIR=Path(_env_str("LST_OUTPUT_DIR") or default_out),
            ENV_OVERRIDES=overrides,
        )
        if cfg.GEE_ASSET_ROOT is None and cfg.GEE_PROJECT:
            cfg.GEE_ASSET_ROOT = f"projects/{cfg.GEE_PROJECT}/assets/ncr_lst_v2"
        cfg.make_dirs()
        return cfg

    def make_dirs(self) -> None:
        """Create the output directory tree (idempotent)."""
        for path in (self.OUTPUT_DIR, self.CACHE_DIR, self.STAGE_DIR, self.FIG_DIR, self.PARQUET_DIR,
                     self.GEO_DIR, self.MODEL_DIR, self.NPZ_DIR, self.WEB_DIR):
            path.mkdir(parents=True, exist_ok=True)

    def _lc_fine_res_for(self, mode: str) -> int:
        """Requested land-cover resolution for ``mode`` (env > profile default) snapped to a divisor of GRID_RES_M."""
        requested = self.LC_FINE_RES_REQUEST
        if requested is None:
            requested = int(self.LC_FINE_RES_DEFAULTS.get(mode, 100))
        if requested <= 0:
            raise ValueError(f"LC_FINE_RES_M must be positive, got {requested}")
        if self.GRID_RES_M % requested == 0:
            return int(requested)
        snapped = _nearest_divisor(self.GRID_RES_M, requested)
        log(f"LC_FINE_RES_M={requested} does not divide GRID_RES_M={self.GRID_RES_M}; using {snapped} m", "WARNING")
        return int(snapped)

    def resolve_mode(self, mode: str) -> int:
        """Record the data mode decided by section 1 and fix LC_FINE_RES_M accordingly; returns it."""
        if mode not in ("gee", "synthetic"):
            raise ValueError(f"data mode must be 'gee' or 'synthetic', got {mode!r}")
        self.DATA_MODE = mode
        self.LC_FINE_RES_M = self._lc_fine_res_for(mode)
        return self.LC_FINE_RES_M

    def fine_res(self) -> int:
        """Land-cover fine resolution in metres (resolved value, else the RUN_MODE default). v1 API."""
        if self.LC_FINE_RES_M is not None:
            return int(self.LC_FINE_RES_M)
        return self._lc_fine_res_for("synthetic" if self.RUN_MODE == "synthetic" else "gee")

    def grid_fine_res(self) -> int:
        """Resolution (m) of the fine analysis grid ``GRID_FINE`` (250 m at the 1 km grid)."""
        return self.GRID_RES_M // self.FINE_FACTOR

    def window(self, year: int) -> tuple:
        """(first, last) calendar year of a frame: observation window, or the 20-year GCM window of a projection."""
        year = int(year)
        if year in self.WINDOWS:
            return tuple(self.WINDOWS[year])
        if year in self.PROJ_WINDOWS:
            return tuple(self.PROJ_WINDOWS[year])
        raise KeyError(f"no window defined for frame {year}")

    def xgb_params(self, device: str = "cpu", seed: Optional[int] = None, **overrides) -> dict:
        """Copy of ``XGB_PARAMS`` with ``device`` and ``seed`` injected (plus optional overrides)."""
        params = dict(self.XGB_PARAMS)
        params.update({"device": device, "seed": int(self.SEED if seed is None else seed)})
        params.update(overrides)
        return params

    def key(self, *names: str, **extra) -> dict:
        """JSON-able configuration subset for ``stage_cache`` keys.

        Always contains the identity of the run's data (mode, profile, grid, frames, seed) and the identity of the
        code that computes the stage: ``code = {"sec": <current section>, "sha": <its AST hash>}`` (from ``_CODE_SHA``,
        injected by build_notebook.py / run_local.py), ``shared = SHARED_CODE_TAG`` (helpers of 00-02) and, for the
        Earth Engine sections 03-05, ``code01`` (the section-01 fetch helpers). ``names`` adds further CFG fields,
        ``extra`` adds stage-specific values (e.g. a feature-list hash or a cross-section ``section_code_hash``).
        """
        base = {"data_mode": self.DATA_MODE, "profile": self.PROFILE, "grid_res_m": self.GRID_RES_M,
                "fine_factor": self.FINE_FACTOR, "lc_fine_res_m": self.LC_FINE_RES_M,
                "frames_obs": list(self.FRAMES_OBS), "seed": self.SEED}
        for name in names:
            base[name] = getattr(self, name)
        sec = globals().get("_CURRENT_SECTION")
        shas = globals().get("_CODE_SHA")
        if isinstance(shas, dict):
            base["code"] = {"sec": sec, "sha": shas.get(sec)}
        else:
            _warn_no_code_sha()
            base["code"] = None
        base["shared"] = SHARED_CODE_TAG
        if sec in ("03", "04", "05"):
            base["code01"] = shas.get("01") if isinstance(shas, dict) else None
        base.update(extra)
        return base

    def apply_hardware(self, env: dict) -> None:
        """Adjust hardware-dependent settings once ``ENV`` is known."""
        self.SHAP_INTERACTION_MAX_ROWS = None if env.get("xgb_device") == "cuda" else 4000

    def as_dict(self) -> dict:
        """JSON-friendly snapshot (Paths as strings) for logging and the export manifest."""
        out = {}
        for f in fields(self):
            value = getattr(self, f.name)
            out[f.name] = str(value) if isinstance(value, Path) else value
        out["LC_FINE_RES_M"] = self.LC_FINE_RES_M
        out["GRID_FINE_RES_M"] = self.grid_fine_res()
        return out


CFG = LSTConfig.from_env()
log(f"CFG: PROFILE={CFG.PROFILE} RUN_MODE={CFG.RUN_MODE} GRID_RES_M={CFG.GRID_RES_M} "
    f"(fine {CFG.grid_fine_res()} m) FRAMES_OBS={CFG.FRAMES_OBS} FRAMES_PROJ={CFG.FRAMES_PROJ or '[] (none)'} "
    f"OUTPUT_DIR={CFG.OUTPUT_DIR}")
if CFG.ENV_OVERRIDES:
    log("environment overrides: " + ", ".join(f"{k}={v}" for k, v in sorted(CFG.ENV_OVERRIDES.items())))
log(f"run control: time budget {f'{CFG.TIME_BUDGET_H:g} h' if CFG.TIME_BUDGET_H else 'unlimited'} (reserve "
    f"{CFG.TIME_RESERVE_S:.0f} s), stop after section {CFG.STOP_AFTER or '-'}, pack cache {CFG.PACK_CACHE}, "
    f"invalidate {CFG.INVALIDATE or '-'}, Earth Engine endpoint {CFG.GEE_ENDPOINT} (deadline "
    f"{CFG.GEE_DEADLINE_S:.0f} s)")
if CFG.PROFILE == "pilot":
    log(f"PILOT profile: frames {CFG.FRAMES_OBS}, no projections, benchmarks {CFG.BENCHMARKS} (with Earth Engine "
        "plus the CMIP6 warming-pattern probe of section 05, LST_PILOT_PROBES) - an Earth Engine cost/runtime probe, "
        "not a publishable run", "WARNING")

## 0.4 Hardware detection (`ENV`)
The GPU count comes from the **CUDA runtime** (`torch.cuda.device_count()`, else CuPy), which honours
`CUDA_VISIBLE_DEVICES`; `nvidia-smi` (NVML ignores that variable) is only the fallback when neither can be
queried, and its rows are filtered by `CUDA_VISIBLE_DEVICES`. XGBoost trains on CUDA only if GPUs exist **and**
the installed wheel was built with CUDA (`xgboost.build_info()["USE_CUDA"]`, with a tiny training probe as
fallback). RAPIDS (cuDF/cuML) is used only with `LST_USE_RAPIDS=1` (or `LST_INSTALL_RAPIDS=1`).
`ENV["rapids_used"]` records which RAPIDS backends actually ran.

In [ ]:
cp = _optional_import("cupy")
cudf = _optional_import("cudf") if CFG.USE_RAPIDS else None
cuml = _optional_import("cuml") if CFG.USE_RAPIDS else None
if not CFG.USE_RAPIDS and (_is_importable("cudf") or _is_importable("cuml")):
    log("cuDF/cuML are installed but not used (set LST_USE_RAPIDS=1 to enable the RAPIDS paths)")


def _query_nvidia_smi() -> list:
    """Return [(index, name, total_MiB), ...] from nvidia-smi, or [] if unavailable."""
    exe = shutil.which("nvidia-smi")
    if exe is None:
        return []
    try:
        proc = subprocess.run([exe, "--query-gpu=index,name,memory.total", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True, timeout=20)
    except (subprocess.TimeoutExpired, OSError) as exc:
        log(f"nvidia-smi failed: {exc}", "WARNING")
        return []
    gpus = []
    for line in proc.stdout.strip().splitlines():
        parts = [p.strip() for p in line.split(",")]
        if len(parts) >= 3 and parts[1]:
            try:
                index = int(parts[0])
            except ValueError:
                index = len(gpus)
            try:
                gpus.append((index, parts[1], int(float(parts[2]))))
            except ValueError:
                gpus.append((index, parts[1], None))
    return gpus


def _visible_device_filter(smi: list) -> list:
    """Restrict nvidia-smi rows to CUDA_VISIBLE_DEVICES (integer ordinals only; UUID/MIG lists are left as-is)."""
    raw = os.environ.get("CUDA_VISIBLE_DEVICES")
    if raw is None:
        return smi
    tokens = [t.strip() for t in raw.split(",") if t.strip()]
    if not tokens or raw.strip() in ("-1", "none", "NoDevFiles"):
        return []
    try:
        wanted = [int(t) for t in tokens]
    except ValueError:
        return smi
    by_index = {row[0]: row for row in smi}
    out = []
    for w in wanted:          # CUDA stops at the first invalid ordinal
        if w not in by_index:
            break
        out.append(by_index[w])
    return out


def _cupy_device_count() -> int:
    if cp is None:
        return 0
    try:
        return int(cp.cuda.runtime.getDeviceCount())
    except Exception:  # noqa: BLE001 - no driver / no device
        return 0


def _torch_device_count() -> int:
    if torch is None:
        return 0
    try:
        return int(torch.cuda.device_count()) if torch.cuda.is_available() else 0
    except Exception:  # noqa: BLE001 - broken CUDA runtime counts as "no GPU"
        return 0


def _xgb_has_cuda_build() -> bool:
    """Was the installed XGBoost built with CUDA? (build_info, else a 1-round GPU training probe)."""
    if xgb is None:
        return False
    try:
        return bool(xgb.build_info().get("USE_CUDA", False))
    except Exception:  # noqa: BLE001 - very old versions have no build_info
        pass
    try:
        probe = xgb.DMatrix(np.random.default_rng(0).random((64, 3)), label=np.arange(64, dtype=float))
        xgb.train({"device": "cuda", "tree_method": "hist", "verbosity": 0}, probe, num_boost_round=1)
        return True
    except Exception:  # noqa: BLE001
        return False


def _detect_env() -> dict:
    """Collect hardware/software facts into the SPEC ``ENV`` dict."""
    smi_all = _query_nvidia_smi()
    smi = _visible_device_filter(smi_all)
    torch_count = _torch_device_count()
    cuda_count = torch_count or _cupy_device_count()
    n_gpus = cuda_count if cuda_count > 0 else len(smi)
    if cuda_count > 0 and len(smi_all) and len(smi_all) != cuda_count:
        log(f"nvidia-smi lists {len(smi_all)} GPU(s) but the CUDA runtime exposes {cuda_count} "
            f"(CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')!r}); using {cuda_count}", "WARNING")
    gpu_names = [name for _, name, _ in smi][:n_gpus]
    gpu_mem = [mem for _, _, mem in smi][:n_gpus]
    if len(gpu_names) < n_gpus and torch_count:
        gpu_names = [torch.cuda.get_device_name(i) for i in range(torch_count)]
    has_cuda = n_gpus > 0
    xgb_cuda = has_cuda and _xgb_has_cuda_build()
    return {
        "is_kaggle": _IS_KAGGLE,
        "n_gpus": int(n_gpus),
        "gpu_names": gpu_names,
        "gpu_mem_mib": gpu_mem,
        "has_cuda": bool(has_cuda),
        "torch_device_count": int(torch_count),
        "has_cudf": cudf is not None,
        "has_cuml": cuml is not None,
        "has_cupy": cp is not None and has_cuda,
        "rapids_used": set(),   # filled by the stages that actually ran a cuDF/cuML backend
        "xgb_version": getattr(xgb, "__version__", None),
        "xgb_device": "cuda" if xgb_cuda else "cpu",
        "lgbm_device": "cpu",   # resolved by section 07 after a GPU probe
        "torch_version": getattr(torch, "__version__", None),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "cpu_count": os.cpu_count() or 1,
    }


ENV = _detect_env()
CFG.apply_hardware(ENV)
log(f"ENV: {ENV['n_gpus']} GPU(s) {ENV['gpu_names']} | xgb {ENV['xgb_version']} on {ENV['xgb_device']} | "
    f"cupy={ENV['has_cupy']} cudf={ENV['has_cudf']} cuml={ENV['has_cuml']} | {ENV['cpu_count']} CPU threads")
if ENV["has_cuda"] and ENV["xgb_device"] == "cpu":
    log("GPUs found but this XGBoost build has no CUDA support - XGBoost will run on CPU.", "WARNING")

## 0.5 Shared utilities
Timing, memory hygiene (host + CUDA + CuPy pools), retry with exponential backoff and jitter, deterministic
seeding, strict JSON writing (NaN → `null`, numpy/pandas aware, optional gzip), GPU memory reporting and a
multi-GPU work scheduler that pins one task per GPU at a time.

In [ ]:
TIMINGS: dict = {}
_RETRY_RNG = random.Random()   # private RNG so backoff jitter never perturbs seeded global streams


@contextmanager
def timer(name: str):
    """Context manager that logs and accumulates wall-clock seconds for ``name`` into ``TIMINGS``."""
    start = time.perf_counter()
    log(f">> {name}")
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        TIMINGS[name] = round(TIMINGS.get(name, 0.0) + elapsed, 3)
        log(f"<< {name}: {elapsed:.1f}s")


def free_memory() -> None:
    """Release host and GPU memory: Python GC, PyTorch CUDA cache/IPC handles, CuPy memory pools."""
    gc.collect()
    if torch is not None:
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
        except Exception as exc:  # noqa: BLE001 - never fail a pipeline on cleanup
            log(f"torch cache cleanup failed: {exc}", "DEBUG")
    if cp is not None and ENV.get("has_cupy"):
        try:
            # get_default_memory_pool() frees only the CURRENT device's pool; blocks cached on the other GPUs
            # (e.g. CV folds run on cuda:1 by a worker thread) need their own device context.
            for device_id in range(int(cp.cuda.runtime.getDeviceCount())):
                with cp.cuda.Device(device_id):
                    cp.get_default_memory_pool().free_all_blocks()
            cp.get_default_pinned_memory_pool().free_all_blocks()
        except Exception as exc:  # noqa: BLE001
            log(f"cupy pool cleanup failed: {exc}", "DEBUG")


def retry(fn: Callable[[], Any], *, tries: int = 3, base_delay: float = 1.0, max_delay: float = 60.0,
          retry_on: tuple = (Exception,), on_retry: Optional[Callable[[int, BaseException, float], None]] = None):
    """Call ``fn()`` up to ``tries`` times with exponential backoff and jitter.

    Delay before attempt k+1 = min(max_delay, base_delay·2^(k-1)) scaled by a uniform jitter in [0.5, 1.0]
    (decorrelates parallel workers hitting the same quota). Only exceptions in ``retry_on`` are retried;
    ``on_retry(attempt, exc, delay)`` is called before sleeping. The last exception is re-raised.
    """
    if tries < 1:
        raise ValueError("tries must be >= 1")
    for attempt in range(1, tries + 1):
        try:
            return fn()
        except retry_on as exc:
            if attempt == tries:
                raise
            delay = min(max_delay, base_delay * 2 ** (attempt - 1)) * _RETRY_RNG.uniform(0.5, 1.0)
            if on_retry is not None:
                on_retry(attempt, exc, delay)
            time.sleep(delay)
    raise RuntimeError("unreachable")


def set_seeds(seed: int) -> int:
    """Seed Python, NumPy, PyTorch (CPU + all CUDA devices) and CuPy; export PYTHONHASHSEED.

    PYTHONHASHSEED only affects subprocesses (the running interpreter's hash seed is fixed at start-up).
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch is not None:
        torch.manual_seed(seed)
        try:
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
        except Exception:  # noqa: BLE001
            pass
    if cp is not None and ENV.get("has_cupy"):
        try:
            cp.random.seed(seed)
        except Exception:  # noqa: BLE001
            pass
    return seed


def _json_float(value: float, ndigits: Optional[int]):
    if not math.isfinite(value):
        return None
    return round(value, ndigits) if ndigits is not None else value


def _to_jsonable(obj: Any, ndigits: Optional[int]):
    """Recursively convert numpy/pandas/pathlib/dataclass objects into strict-JSON-compatible values."""
    if obj is None or isinstance(obj, (bool, str)):
        return obj
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, (int, np.integer)):
        return int(obj)
    if isinstance(obj, (float, np.floating)):
        return _json_float(float(obj), ndigits)
    if isinstance(obj, dict):
        return {str(_to_jsonable(k, None)) if not isinstance(k, str) else k: _to_jsonable(v, ndigits)
                for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set, frozenset)):
        return [_to_jsonable(v, ndigits) for v in obj]
    if isinstance(obj, np.ndarray):
        if obj.dtype.kind in "fc":
            arr = np.asarray(obj.real if obj.dtype.kind == "c" else obj, dtype=np.float64)
            if ndigits is not None:
                arr = np.round(arr, ndigits)
            return _to_jsonable(arr.tolist(), None)
        return _to_jsonable(obj.tolist(), ndigits)
    if isinstance(obj, pd.DataFrame):
        return _to_jsonable(obj.to_dict(orient="records"), ndigits)
    if isinstance(obj, (pd.Series, pd.Index)):
        return _to_jsonable(obj.tolist(), ndigits)
    if obj is pd.NaT:
        return None
    if isinstance(obj, (pd.Timestamp, datetime, date)):
        return obj.isoformat()
    if isinstance(obj, Path):
        return str(obj)
    if is_dataclass(obj) and not isinstance(obj, type):
        return _to_jsonable(asdict(obj), ndigits)
    if hasattr(obj, "item") and callable(obj.item):     # 0-d arrays, cupy/torch scalars
        return _to_jsonable(obj.item(), ndigits)
    raise TypeError(f"save_json: cannot serialise object of type {type(obj).__name__}")


def _write_bytes_atomic(path: Path, data: bytes) -> Path:
    """Write ``data`` to ``path`` via a temporary sibling + rename (never leaves a truncated file behind)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    try:
        with open(tmp, "wb") as fh:
            fh.write(data)
        os.replace(tmp, path)
    except BaseException:
        try:
            tmp.unlink()
        except OSError:
            pass
        raise
    return path


def _gzip_bytes(data: bytes, level: int = 9) -> bytes:
    """Deterministic gzip (mtime 0, no file name) so identical content gives identical bytes."""
    buf = io.BytesIO()
    with gzip.GzipFile(filename="", mode="wb", compresslevel=level, fileobj=buf, mtime=0) as gz:
        gz.write(data)
    return buf.getvalue()


def save_json(obj: Any, path, *, round_floats: Optional[int] = None) -> Path:
    """Write ``obj`` as strict UTF-8 JSON (NaN/Inf → null, numpy/pandas aware, compact separators).

    ``round_floats`` rounds every float to that many decimals. A path ending in ``.gz`` is gzip-compressed (level 9,
    deterministic header) - the web bundle's ``*.json.gz`` files. The file is written atomically.
    """
    path = Path(path)
    text = json.dumps(_to_jsonable(obj, round_floats), ensure_ascii=False, separators=(",", ":"),
                      allow_nan=False)
    data = text.encode("utf-8")
    if path.suffix == ".gz":
        data = _gzip_bytes(data, 9)
    return _write_bytes_atomic(path, data)


def gpu_mem_report(log_it: bool = True) -> list:
    """Per-GPU memory usage [{gpu, name, used_mib, total_mib, torch_allocated_mib}] (empty without GPUs)."""
    report = []
    if torch is not None and ENV.get("torch_device_count", 0) > 0:
        for i in range(ENV["torch_device_count"]):
            try:
                free_b, total_b = torch.cuda.mem_get_info(i)
                report.append({"gpu": i, "name": torch.cuda.get_device_name(i),
                               "used_mib": round((total_b - free_b) / 2**20),
                               "total_mib": round(total_b / 2**20),
                               "torch_allocated_mib": round(torch.cuda.memory_allocated(i) / 2**20)})
            except Exception as exc:  # noqa: BLE001
                log(f"mem_get_info failed on GPU {i}: {exc}", "DEBUG")
    elif ENV.get("n_gpus", 0) > 0 and shutil.which("nvidia-smi"):
        try:
            proc = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total",
                                   "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=20)
            for line in proc.stdout.strip().splitlines():
                idx, name, used, total = [p.strip() for p in line.split(",")]
                report.append({"gpu": int(idx), "name": name, "used_mib": int(float(used)),
                               "total_mib": int(float(total)), "torch_allocated_mib": None})
        except (subprocess.TimeoutExpired, OSError, ValueError) as exc:
            log(f"nvidia-smi memory query failed: {exc}", "DEBUG")
    if log_it:
        if report:
            log("GPU memory: " + " | ".join(f"GPU{r['gpu']} {r['used_mib']}/{r['total_mib']} MiB" for r in report))
        else:
            log("GPU memory: no CUDA devices")
    return report


def xgb_device_for(gpu_id: int) -> str:
    """XGBoost ``device`` string for a worker: ``"cuda:{gpu_id}"`` when XGBoost can use CUDA, else ``"cpu"``."""
    if ENV.get("xgb_device") != "cuda":
        return "cpu"
    gpu_id = int(gpu_id)
    if not 0 <= gpu_id < max(1, ENV.get("n_gpus", 0)):
        raise ValueError(f"gpu_id {gpu_id} out of range for {ENV.get('n_gpus')} GPU(s)")
    return f"cuda:{gpu_id}"


def _annotate_task_error(exc: BaseException, index: int, item: Any, gpu_id: int) -> None:
    context = f"gpu_pool_map task #{index} (item={str(item)[:120]}) failed on gpu_id={gpu_id}"
    log(f"{context}: {type(exc).__name__}: {str(exc)[:300]}", "ERROR")
    if hasattr(exc, "add_note"):   # Python >= 3.11: context shows up in the traceback
        exc.add_note(context)


def gpu_pool_map(fn: Callable[[Any, int], Any], items: Iterable) -> list:
    """Run ``fn(item, gpu_id)`` for every item, one concurrent task per GPU; results keep input order.

    With ≥2 CUDA GPUs usable by XGBoost, a thread pool with one worker per GPU pulls free GPU ids from a queue, so
    each GPU runs exactly one task at a time (XGBoost/PyTorch release the GIL inside CUDA work, so threads give real
    parallelism). Otherwise tasks run sequentially with ``gpu_id=0`` (``xgb_device_for(0)`` then yields "cpu" on
    CPU-only machines). The first failure cancels pending tasks and is re-raised with its original type, annotated
    with the failing item and GPU.

    **Time budget** (section 0.5.1): before starting every item the run budget is checked with an expected duration
    of 1.25 × the longest item finished so far in this call. When it is exhausted no further item is started, the
    items already running finish (their own stage caches are saved), and ``LSTStop`` is raised in the caller; this
    is a clean stop, not a task failure, so it is not logged as an error.
    """
    items = list(items)
    n_workers = ENV.get("n_gpus", 0) if ENV.get("xgb_device") == "cuda" else 0
    if n_workers <= 1 or len(items) <= 1:
        results = []
        longest = 0.0
        for i, item in enumerate(items):
            check_budget(f"gpu_pool_map item {i + 1}/{len(items)}", 1.25 * longest)
            t0 = time.perf_counter()
            try:
                results.append(fn(item, 0))
            except LSTStop:
                raise
            except BaseException as exc:
                _annotate_task_error(exc, i, item, 0)
                raise
            longest = max(longest, time.perf_counter() - t0)
        return results

    free_gpus: "queue.Queue[int]" = queue.Queue()
    for gpu_id in range(n_workers):
        free_gpus.put(gpu_id)
    stop = threading.Event()
    longest = [0.0]
    longest_lock = threading.Lock()

    def _task(index: int, item: Any):
        gpu_id = free_gpus.get()
        try:
            if stop.is_set():
                raise LSTStop(LST_STOP_REASON or "time budget: gpu_pool_map")
            try:
                check_budget(f"gpu_pool_map item {index + 1}/{len(items)}", 1.25 * longest[0])
            except LSTStop:
                stop.set()
                raise
            t0 = time.perf_counter()
            out = fn(item, gpu_id)
            with longest_lock:
                longest[0] = max(longest[0], time.perf_counter() - t0)
            return out
        except LSTStop:
            stop.set()
            raise
        except BaseException as exc:
            _annotate_task_error(exc, index, item, gpu_id)
            raise
        finally:
            free_gpus.put(gpu_id)

    results: list = [None] * len(items)
    with ThreadPoolExecutor(max_workers=n_workers, thread_name_prefix="gpu") as pool:
        futures = {pool.submit(_task, i, item): i for i, item in enumerate(items)}
        try:
            for future in as_completed(futures):
                results[futures[future]] = future.result()
        except BaseException:
            # Clean stop or failure: nothing new starts; leaving the with-block waits for the running items.
            stop.set()
            for future in futures:
                future.cancel()
            raise
    return results


_OOM_MARKERS = ("out of memory", "cudaerrormemoryallocation", "bad_alloc", "cuda_error_out_of_memory",
                "memoryerror", "failed to allocate", "cusparse_status_alloc_failed",
                "cusparse_status_insufficient_resources", "cublas_status_alloc_failed",
                "cudnn_status_alloc_failed", "cufft_alloc_failed")


def _oom_in_chain(exc: BaseException, *, count_host_memory_error: bool) -> bool:
    torch_oom = getattr(getattr(torch, "cuda", None), "OutOfMemoryError", None) if torch is not None else None
    cupy_oom = None
    if cp is not None:
        cupy_oom = getattr(getattr(getattr(cp, "cuda", None), "memory", None), "OutOfMemoryError", None)
    seen, current = set(), exc
    while current is not None and id(current) not in seen:
        seen.add(id(current))
        if torch_oom is not None and isinstance(current, torch_oom):
            return True
        if cupy_oom is not None and isinstance(current, cupy_oom):
            return True
        if type(current) is MemoryError:
            # a bare MemoryError is the host running out of RAM
            if count_host_memory_error:
                return True
        else:
            message = f"{type(current).__name__}: {current}".lower()
            if any(marker in message for marker in _OOM_MARKERS):
                return True
        current = current.__cause__ or current.__context__
    return False


def is_oom_error(exc: BaseException) -> bool:
    """True if ``exc`` (or its cause/context chain) is a host OR GPU out-of-memory condition.

    Recognises MemoryError, torch.cuda.OutOfMemoryError, cupy OutOfMemoryError and XGBoost/CUDA/RMM/cuSPARSE/cuBLAS
    messages ("out of memory", "cudaErrorMemoryAllocation", "bad_alloc", "CUSPARSE_STATUS_ALLOC_FAILED", ...).
    """
    return _oom_in_chain(exc, count_host_memory_error=True)


def is_gpu_oom_error(exc: BaseException) -> bool:
    """True only for a GPU out-of-memory condition (a bare host ``MemoryError`` is excluded).

    Used to decide GPU -> CPU retries: moving work to the CPU after the *host* ran out of RAM needs more host memory,
    not less, so that case must propagate instead.
    """
    return _oom_in_chain(exc, count_host_memory_error=False)


set_seeds(CFG.SEED)
log("utilities ready")

### 0.5.1 Time budget and clean stop (`LSTStop`)
A Kaggle session is killed at 12 h, and a killed commit may leave **no usable output**. The run therefore watches
its own wall clock (counted from the first code cell, `_RUN_T0`): with `LST_TIME_BUDGET_H` (default 11 h on
Kaggle, unlimited elsewhere) it stops *starting* new work once less than `LST_TIME_RESERVE_S` (default 1800 s) plus
the expected duration of the next unit is left. The checks sit where long work starts: every `gpu_pool_map` item,
every stage-cache MISS (expected = 1.25 × the last compute time of that stage name), every Earth Engine tile, every
Landsat frame, every export poll and every GDDP block. A stop raises `LSTStop` (a `BaseException`, so ordinary
`except Exception` handlers never swallow it), the cell that was running ends normally, `RUN_STOPPED.json` records
why, the cache is packed (section 0.6) and every later cell prints one `skipped: run stopped (...)` line - the
notebook finishes successfully and the Kaggle version keeps its output for a resume.
`LST_STOP_AFTER=NN` stops the same way right after section NN (e.g. `06`: Earth Engine extraction + features in a
first commit, modelling in a second one). API: `time_left_s()`, `budget_exhausted(expected_s)`,
`check_budget(where, expected_s)`, `request_stop(reason)`, `finalize_stop()`.

In [ ]:
class LSTStop(BaseException):
    """Clean stop of the run (time budget or LST_STOP_AFTER); not an error. Handled at cell/section boundaries."""


_LST_STOP_LOCK = threading.RLock()
_LST_FINALIZED = False
_LST_STOP_SECTION = None


def time_left_s() -> float:
    """Seconds left in the run's time budget (``math.inf`` without a budget)."""
    if CFG.TIME_BUDGET_H is None:
        return math.inf
    return float(_RUN_T0 + float(CFG.TIME_BUDGET_H) * 3600.0 - time.time())


def budget_exhausted(expected_s: float = 0.0) -> bool:
    """True when the run was stopped, or less than ``expected_s + CFG.TIME_RESERVE_S`` seconds are left."""
    if LST_STOPPED:
        return True
    left = time_left_s()
    if not math.isfinite(left):
        return False
    return left < max(0.0, float(expected_s or 0.0)) + float(CFG.TIME_RESERVE_S)


def request_stop(reason: str) -> None:
    """Mark the run as stopped (thread-safe, idempotent: the first reason wins). Never packs; see finalize_stop."""
    global LST_STOPPED, LST_STOP_REASON, _LST_STOP_SECTION
    with _LST_STOP_LOCK:
        if LST_STOPPED:
            return
        LST_STOPPED = True
        LST_STOP_REASON = str(reason)
        _LST_STOP_SECTION = _CURRENT_SECTION
    left = time_left_s()
    bar = "#" * 96
    log(f"\n{bar}\n#  RUN STOPPING CLEANLY in section {_CURRENT_SECTION}: {reason}\n#  elapsed "
        f"{(time.time() - _RUN_T0) / 3600:.2f} h, budget "
        f"{f'{CFG.TIME_BUDGET_H:g} h' if CFG.TIME_BUDGET_H else 'unlimited'}"
        f"{f', {left / 60:.0f} min left' if math.isfinite(left) else ''}. Running work finishes; nothing new starts."
        f"\n{bar}", "WARNING")


def check_budget(where: str, expected_s: float = 0.0) -> None:
    """Raise ``LSTStop`` (after ``request_stop``) when ``budget_exhausted(expected_s)``; otherwise do nothing."""
    if budget_exhausted(expected_s):
        request_stop(f"time budget: {where}")
        raise LSTStop(LST_STOP_REASON or f"time budget: {where}")


def finalize_stop() -> None:
    """Write ``RUN_STOPPED.json``, pack the cache and log how to resume (idempotent; cell/section boundaries only)."""
    global _LST_FINALIZED
    with _LST_STOP_LOCK:
        if _LST_FINALIZED:
            return
        _LST_FINALIZED = True
    if not LST_STOPPED:
        request_stop("finalize_stop() called")
    info = {"reason": LST_STOP_REASON, "section": _LST_STOP_SECTION or _CURRENT_SECTION,
            "elapsed_h": round((time.time() - _RUN_T0) / 3600.0, 3), "time_budget_h": CFG.TIME_BUDGET_H,
            "stop_after": CFG.STOP_AFTER, "profile": CFG.PROFILE, "utc": datetime.now(timezone.utc)}
    try:
        save_json(info, CFG.OUTPUT_DIR / "RUN_STOPPED.json")
    except Exception as exc:  # noqa: BLE001 - the stop itself must succeed
        log(f"could not write RUN_STOPPED.json: {type(exc).__name__}: {exc}", "WARNING")
    try:
        pack_cache()
    except Exception as exc:  # noqa: BLE001
        log(f"pack_cache failed during the clean stop: {type(exc).__name__}: {str(exc)[:300]}", "WARNING")
    log(f"run stopped cleanly ({LST_STOP_REASON}); {CFG.OUTPUT_DIR / 'RUN_STOPPED.json'} written. To resume: let "
        "this version finish (Save & Run All / Commit keeps outputs/), then attach its output (Input -> + Add Input "
        "-> Notebooks -> Your Work -> this notebook; that is the newest version's output, so save no other version "
        "in between) and run again: LST_CACHE_IMPORT=auto restores the stage, tile and GDDP caches, so finished "
        "stages are not recomputed.", "WARNING")


def _lst_on_stop(exc: BaseException) -> None:
    """Cell-level handler of ``LSTStop`` (used by the cell guard and run_local.py)."""
    request_stop(str(exc) or "LSTStop")
    finalize_stop()


def _lst_skip_cell() -> None:
    print(f"skipped: run stopped ({LST_STOP_REASON})", flush=True)


def _lst_show_last(value: Any) -> None:
    """Display the value of a guarded cell's last expression as Jupyter would (the guard nests it in a block)."""
    if value is not None:
        sys.displayhook(value)


def _lst_enter_section(nn) -> None:
    """Section marker (first code cell of sections 01-12, generated by build_notebook.py).

    Sets ``_CURRENT_SECTION`` (code identity of stage keys), logs elapsed/remaining time and stops the run cleanly
    before section ``nn`` when ``nn > CFG.STOP_AFTER`` or the time budget is exhausted.
    """
    global _CURRENT_SECTION
    nn = f"{int(nn):02d}"
    _CURRENT_SECTION = nn
    if LST_STOPPED:
        print(f"section {nn} skipped: run stopped ({LST_STOP_REASON})", flush=True)
        finalize_stop()
        return
    left = time_left_s()
    log(f"=== section {nn} | elapsed {(time.time() - _RUN_T0) / 3600:.2f} h | "
        + (f"{left / 3600:.2f} h left of the {CFG.TIME_BUDGET_H:g} h budget" if math.isfinite(left)
           else "no time budget") + " ===")
    if CFG.STOP_AFTER is not None and int(nn) > int(CFG.STOP_AFTER):
        request_stop(f"LST_STOP_AFTER={CFG.STOP_AFTER}: stopping before section {nn}")
        finalize_stop()
    elif budget_exhausted():
        request_stop(f"time budget: less than LST_TIME_RESERVE_S={CFG.TIME_RESERVE_S:.0f} s left before section {nn}")
        finalize_stop()


class _LSTCellGuard(ast.NodeTransformer):
    """IPython AST transformer: every later cell runs as
    ``if not LST_STOPPED: try: <cell> except LSTStop as e: _lst_on_stop(e)  else: _lst_skip_cell()``.

    The last expression of a cell is still displayed. Exempt: the first cell of section 00 (it resets the stop state,
    so a rerun from the top works) and the generated ``_CODE_SHA`` / ``_lst_enter_section`` cells. Never raises (on
    any problem the cell runs unchanged).
    """

    @staticmethod
    def _exempt(body: list) -> bool:
        for node in body:
            if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == "_RUN_T0"
                                                    for t in node.targets):
                return True
        generated = True
        for node in body:
            is_marker = (isinstance(node, ast.Expr) and isinstance(node.value, ast.Call)
                         and isinstance(node.value.func, ast.Name) and node.value.func.id == "_lst_enter_section")
            is_sha = (isinstance(node, ast.Assign) and len(node.targets) == 1
                      and isinstance(node.targets[0], ast.Name) and node.targets[0].id == "_CODE_SHA")
            generated = generated and (is_marker or is_sha)
        return generated

    def visit_Module(self, node):  # noqa: N802 - ast API name
        try:
            body = list(node.body)
            if not body or self._exempt(body):
                return node
            if isinstance(body[-1], ast.Expr):
                last = body[-1]
                body[-1] = ast.copy_location(ast.Expr(value=ast.Call(
                    func=ast.Name(id="_lst_show_last", ctx=ast.Load()), args=[last.value], keywords=[])), last)
            handler = ast.ExceptHandler(
                type=ast.Name(id="LSTStop", ctx=ast.Load()), name="_lst_e",
                body=[ast.Expr(value=ast.Call(func=ast.Name(id="_lst_on_stop", ctx=ast.Load()),
                                              args=[ast.Name(id="_lst_e", ctx=ast.Load())], keywords=[]))])
            guarded = ast.Try(body=body, handlers=[handler], orelse=[], finalbody=[])
            skip = ast.Expr(value=ast.Call(func=ast.Name(id="_lst_skip_cell", ctx=ast.Load()), args=[], keywords=[]))
            wrapper = ast.If(test=ast.UnaryOp(op=ast.Not(), operand=ast.Name(id="LST_STOPPED", ctx=ast.Load())),
                             body=[guarded], orelse=[skip])
            new = ast.Module(body=[wrapper], type_ignores=list(getattr(node, "type_ignores", [])))
            ast.copy_location(wrapper, body[0])
            return ast.fix_missing_locations(new)
        except Exception:  # noqa: BLE001 - the guard must never break a cell
            return node


def _lst_install_cell_guard() -> bool:
    """Register the cell guard in the running IPython shell (idempotent); False outside IPython (run_local.py)."""
    try:
        from IPython import get_ipython
        shell = get_ipython()
    except Exception:  # noqa: BLE001 - no IPython
        return False
    if shell is None or not hasattr(shell, "ast_transformers"):
        return False
    try:
        shell.ast_transformers[:] = [t for t in shell.ast_transformers if type(t).__name__ != "_LSTCellGuard"]
        shell.ast_transformers.append(_LSTCellGuard())
    except Exception as exc:  # noqa: BLE001
        log(f"cell guard not installed ({type(exc).__name__}: {exc}); a time-budget stop then ends the run with an "
            "LSTStop error instead of skipping the remaining cells", "WARNING")
        return False
    log("cell guard installed: after a clean stop every later cell is skipped")
    return True


_LST_CELL_GUARD = _lst_install_cell_guard()
if CFG.TIME_BUDGET_H is not None:
    log(f"time budget {CFG.TIME_BUDGET_H:g} h from {datetime.fromtimestamp(_RUN_T0).strftime('%H:%M:%S')}: new work "
        f"stops when less than {CFG.TIME_RESERVE_S:.0f} s + the expected duration of the next unit is left")

## 0.6 Stage cache, cache import and cache packing (resumable runs)
A Kaggle GPU session stops after 12 h, and Earth Engine extraction or a full validation suite can take hours. Every
heavy stage therefore follows the same pattern:

```python
_sc = stage_cache("07_models_day", CFG.key("BENCHMARKS", "N_FOLDS_BLOCK", features=FEATURES))
RESULT = _sc.load() if _sc.hit else _sc.save(compute())       # or: RESULT = _sc.get_or_compute(compute)
```

The cache file is `CFG.CACHE_DIR/stages/<name>-<sha1(key)[:12]>.pkl`, where the key is the canonical JSON of
**configuration + data fingerprints + code identity**: `CFG.key()` always adds `code = {sec, sha}` (the AST hash of
the current section's code cells, computed by `build_notebook.py` and injected as `_CODE_SHA` in the notebook's first
code cell; comments, docstrings and markdown do not change it), `shared = SHARED_CODE_TAG` (helpers of 00-02) and,
for the Earth Engine sections 03-05, the section-01 hash. Editing a section's code therefore recomputes its stages
on the next run - **edit `notebook/sections` and rebuild the notebook; never hand-edit cells on Kaggle** (a
hand-edited cell keeps the hash it was built with). Cross-section dependencies are added by the owning stage with
`section_code_hash('07', ...)`.

* `LST_RESUME=0` recomputes (and overwrites) every stage; Earth Engine tiles and GDDP blocks stay cached.
* `LST_INVALIDATE=06,07` recomputes only the stages whose name starts with one of the tokens (or with `s` + token,
  e.g. `s04_thermal`); tiles and GDDP blocks are never invalidated by it.
* Before a MISS is computed by `get_or_compute`, the time budget (0.5.1) is checked with 1.25 × the last compute time
  of that stage name (trailing `_YYYY`, `_sspNNN`, `_day`, `_night` stripped); before its first compute in a session
  the `compute_s` of the newest sidecar of that name in `stages/` (an earlier or imported run) is used.
* Writes are atomic, a sidecar `.json` records the full key (checked on load), a corrupt entry is reported and
  recomputed by `get_or_compute`. Classes and functions defined in the notebook itself (e.g. `GridSpec`) are pickled
  **by name** and resolved from the notebook namespace on load (Jupyter and `run_local.py`).

**Cache import** (`CFG.CACHE_IMPORT`): `auto` merges the shallowest `cache` folder found under `/kaggle/input/`
(a previous version's output attached with *Input → + Add Input → Notebooks*); an explicit value may be **any
ancestor folder**, e.g. the attachment's path copied from Kaggle's Input panel (its shallowest `**/cache` holding
`stages/`, `tiles/`, `gddp/` or `packed/` is used); an explicit value that is not a folder or holds no cache stops
the run at once (rather than silently recomputing hours of work); `0` disables. The source and the attached run's
`RUN_STOPPED.json` / `RUN_FAILED.json` are logged. Only `stages/`, `tiles/` and `gddp/` are imported; a file is
copied only when it is missing locally or the imported copy is newer, and packed tar files are unpacked (member
names are checked: no absolute paths, no `..`). Packed tars of this run's own output folder are unpacked at setup
too (resume in place), and a stage/tile that is only packed is restored on demand.

**Packing** (`pack_cache()`, `LST_PACK_CACHE`, default on Kaggle): Kaggle notebook outputs are reported to be capped
at about 500 files, while a full run caches thousands of tiles. At the end of the run (and at a clean stop) the cache
is packed into a few dozen plain tar files `CFG.CACHE_DIR/packed/{stages_<group>,tiles_<group>,gddp}.tar` (atomic,
member counts verified, loose files removed group by group); `web/` and `web_bundle.zip` are never touched.

In [ ]:
_NB_GLOBALS = globals()          # the shared notebook namespace (Jupyter __main__ or run_local's exec dict)
_STAGE_PERSIST_TAG = "lst-notebook-global"
_CACHE_SUBDIRS = ("stages", "tiles", "gddp")          # imported / packed cache content (plus packed/)
_STAGE_COMPUTE_S: dict = {}      # normalised stage name -> seconds of its last compute in this session
_STAGE_REFRESHED: set = set()    # stage files written in this session (always HIT, even with LST_RESUME=0)
_STAGE_NORM_RE = re.compile(r"_(?:\d{4}|ssp\d{3}|day|night)$")


class StageCacheError(RuntimeError):
    """A stage-cache entry exists but cannot be used (corrupt pickle, key mismatch, missing global)."""


class _NotebookPickler(pickle.Pickler):
    """Pickler that stores classes/functions defined in the notebook namespace by name.

    Under ``run_local.py`` the sections run in an ``exec`` namespace whose ``__name__`` is ``"__main__"`` but which is
    *not* ``sys.modules["__main__"]``; plain pickle would fail to find e.g. ``__main__.GridSpec``. Persistent ids make
    the reference explicit and the unpickler resolves it from the namespace.
    """

    def persistent_id(self, obj):
        if isinstance(obj, type) or inspect.isfunction(obj):
            name = getattr(obj, "__qualname__", None)
            if (getattr(obj, "__module__", None) == _NB_GLOBALS.get("__name__", "__main__") and name
                    and "." not in name and _NB_GLOBALS.get(name) is obj):
                return (_STAGE_PERSIST_TAG, name)
        return None


class _NotebookUnpickler(pickle.Unpickler):
    def persistent_load(self, pid):
        if isinstance(pid, tuple) and len(pid) == 2 and pid[0] == _STAGE_PERSIST_TAG:
            obj = _NB_GLOBALS.get(pid[1])
            if obj is None:
                raise pickle.UnpicklingError(f"notebook global {pid[1]!r} is not defined (run earlier sections)")
            return obj
        raise pickle.UnpicklingError(f"unsupported persistent id {pid!r}")


def _canonical_key_json(key_obj: Any) -> str:
    """Canonical JSON text of a cache key (sorted keys, numpy-aware, NaN → null)."""
    return json.dumps(_to_jsonable(key_obj, None), sort_keys=True, separators=(",", ":"), ensure_ascii=True,
                      allow_nan=False)


def _stage_norm_name(name: str) -> str:
    """Stage name without trailing ``_YYYY`` / ``_sspNNN`` / ``_day`` / ``_night`` tokens (budget estimates)."""
    prev = None
    while prev != name:
        prev, name = name, _STAGE_NORM_RE.sub("", name)
    return name


def _stage_compute_estimate(norm: str) -> float:
    """Seconds the stage ``norm`` (normalised name) is expected to compute, for the budget check of a MISS.

    The last compute of this session wins; before the first one, the ``compute_s`` of the most recent stage sidecar
    of the same normalised name in ``CFG.STAGE_DIR`` (an earlier or imported run, possibly with another key) seeds
    the estimate, so a long single-unit stage is not started with only the reserve left. 0.0 when nothing is known.
    """
    if norm in _STAGE_COMPUTE_S:
        return float(_STAGE_COMPUTE_S[norm])
    best_t, best_s = -1.0, 0.0
    try:
        sidecars = list(CFG.STAGE_DIR.glob(f"{norm}*.json"))
    except OSError:
        sidecars = []
    for meta_path in sidecars:
        stem = meta_path.stem.rsplit("-", 1)[0]
        if _stage_norm_name(stem) != norm:
            continue
        try:
            mtime = meta_path.stat().st_mtime
            compute_s = json.loads(meta_path.read_text(encoding="utf-8")).get("compute_s")
        except (OSError, ValueError, AttributeError):
            continue
        if isinstance(compute_s, (int, float)) and math.isfinite(compute_s) and mtime > best_t:
            best_t, best_s = mtime, float(compute_s)
    _STAGE_COMPUTE_S[norm] = best_s
    return best_s


def _stage_invalidated(name: str) -> Optional[str]:
    """The LST_INVALIDATE token that forces ``name`` to recompute, else None."""
    for token in CFG.INVALIDATE:
        if name.startswith(token) or name.startswith("s" + token):
            return token
    return None


# ---- packed-cache index (on-demand restore of entries that only exist inside packed/*.tar) ------------------------
_PACKED_LOCK = threading.RLock()
_PACKED_INDEX: dict = {"sig": None, "members": {}}


def _safe_member_name(name: str) -> bool:
    """A tar member name that stays inside the cache: relative, no '..', no drive letters/backslashes."""
    if not name or name.startswith(("/", "\\")) or re.match(r"^[A-Za-z]:", name) or "\\" in name:
        return False
    parts = name.split("/")
    return ".." not in parts and all(p not in ("",) for p in parts[:-1])


def _packed_members(packed_dir: Optional[Path] = None) -> dict:
    """{member name: (tar path, data offset, size, mtime)} of the regular files in ``packed/*.tar`` (memoised)."""
    packed_dir = Path(packed_dir) if packed_dir is not None else CFG.CACHE_DIR / "packed"
    with _PACKED_LOCK:
        tars = sorted(packed_dir.glob("*.tar")) if packed_dir.is_dir() else []
        try:
            sig = (str(packed_dir), tuple((t.name, t.stat().st_mtime_ns, t.stat().st_size) for t in tars))
        except OSError:
            sig = None
        if sig is not None and sig == _PACKED_INDEX["sig"]:
            return _PACKED_INDEX["members"]
        members = {}
        for tar_path in tars:
            try:
                with tarfile.open(tar_path, "r:") as tf:
                    for m in tf.getmembers():
                        if m.isfile() and _safe_member_name(m.name):
                            members[m.name] = (tar_path, int(m.offset_data), int(m.size), float(m.mtime))
            except (OSError, tarfile.TarError) as exc:
                log(f"packed cache {tar_path.name} is unreadable ({type(exc).__name__}: {exc}); ignored", "WARNING")
        _PACKED_INDEX.update(sig=sig, members=members)
        return members


def _write_member(entry: tuple, dest: Path) -> None:
    """Copy one tar member (by offset) to ``dest`` atomically and restore its mtime."""
    tar_path, offset, size, mtime = entry
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_name(dest.name + ".tmp")
    try:
        with open(tar_path, "rb") as src, open(tmp, "wb") as out:
            src.seek(offset)
            left = size
            while left > 0:
                chunk = src.read(min(left, 1 << 22))
                if not chunk:
                    raise OSError(f"{tar_path.name}: truncated member for {dest.name}")
                out.write(chunk)
                left -= len(chunk)
        os.replace(tmp, dest)
        os.utime(dest, (mtime, mtime))
    except BaseException:
        try:
            tmp.unlink()
        except OSError:
            pass
        raise


def cache_restore(*paths) -> int:
    """Restore cache files that exist only inside this run's ``packed/*.tar``; returns the number restored."""
    if not (CFG.CACHE_DIR / "packed").is_dir():
        return 0
    restored = 0
    with _PACKED_LOCK:
        members = _packed_members()
        for path in paths:
            path = Path(path)
            if path.is_file():
                continue
            try:
                rel = path.resolve().relative_to(CFG.CACHE_DIR.resolve()).as_posix()
            except ValueError:
                continue
            entry = members.get(rel)
            if entry is None:
                continue
            try:
                _write_member(entry, path)
                restored += 1
            except OSError as exc:
                log(f"could not restore {rel} from {entry[0].name}: {exc}", "WARNING")
    return restored


class _StageCache:
    """Handle on one stage-cache entry: ``.hit``, ``.load()``, ``.save(obj)``, ``.get_or_compute(fn)``."""

    def __init__(self, name: str, key_obj: Any):
        if not re.fullmatch(r"[A-Za-z0-9_.\-]+", name):
            raise ValueError(f"stage name {name!r} may contain only letters, digits, '_', '-' and '.'")
        self.name = name
        self.key_json = _canonical_key_json(key_obj)
        self.digest = hashlib.sha1(self.key_json.encode("utf-8")).hexdigest()[:12]
        CFG.STAGE_DIR.mkdir(parents=True, exist_ok=True)
        self.path = CFG.STAGE_DIR / f"{name}-{self.digest}.pkl"
        self.meta_path = self.path.with_suffix(".json")
        if not self.path.is_file():
            cache_restore(self.path, self.meta_path)          # entry packed at the end of an earlier run
        exists = self.path.is_file()
        refreshed = str(self.path) in _STAGE_REFRESHED
        token = _stage_invalidated(name)
        self.forced = bool(exists and token is not None and not refreshed)
        self.hit = exists and (refreshed or (bool(CFG.RESUME) and not self.forced))
        if self.hit:
            log(f"stage cache HIT  {self.path.name} ({self.path.stat().st_size / 2**20:.1f} MiB)")
        elif self.forced:
            log(f"stage cache MISS {self.path.name} (LST_INVALIDATE={token}: recomputing)")
        else:
            reason = "LST_RESUME=0" if (not CFG.RESUME and exists) else "no entry"
            log(f"stage cache MISS {self.path.name} ({reason})", "DEBUG")

    def _check_key(self) -> None:
        if not self.meta_path.is_file():
            return   # imported caches may come without sidecars; the digest in the file name still matches
        try:
            stored = json.loads(self.meta_path.read_text(encoding="utf-8")).get("key")
        except (OSError, json.JSONDecodeError) as exc:
            log(f"stage cache sidecar {self.meta_path.name} unreadable ({exc}); trusting the file-name digest",
                "WARNING")
            return
        if stored is not None and _canonical_key_json(stored) != self.key_json:
            raise StageCacheError(f"{self.path.name}: stored key differs from the requested key (hash collision "
                                  f"or a copied file)")

    def load(self) -> Any:
        """Unpickle the entry; raises ``StageCacheError`` when it is missing or unusable."""
        if not self.path.is_file():
            cache_restore(self.path, self.meta_path)
        if not self.path.is_file():
            raise StageCacheError(f"stage cache entry {self.path} does not exist")
        self._check_key()
        t0 = time.perf_counter()
        try:
            with open(self.path, "rb") as fh:
                obj = _NotebookUnpickler(fh).load()
        except Exception as exc:  # noqa: BLE001 - any unpickling failure makes the entry unusable
            raise StageCacheError(f"stage cache entry {self.path.name} is unreadable: {type(exc).__name__}: "
                                  f"{str(exc)[:200]}") from exc
        log(f"stage cache loaded {self.path.name} in {time.perf_counter() - t0:.1f}s")
        return obj

    def save(self, obj: Any, compute_s: Optional[float] = None) -> Any:
        """Pickle ``obj`` atomically (+ key sidecar) and return it unchanged (so ``X = sc.save(compute())``)."""
        tmp = self.path.with_name(self.path.name + ".tmp")
        t0 = time.perf_counter()
        try:
            with open(tmp, "wb") as fh:
                _NotebookPickler(fh, protocol=pickle.HIGHEST_PROTOCOL).dump(obj)
            os.replace(tmp, self.path)
        except Exception as exc:  # noqa: BLE001 - caching must never break the computation it caches
            try:
                tmp.unlink()
            except OSError:
                pass
            log(f"stage cache could not save {self.path.name}: {type(exc).__name__}: {str(exc)[:200]} "
                "(continuing without cache)", "WARNING")
            return obj
        meta = {"name": self.name, "key": json.loads(self.key_json), "created": datetime.now(timezone.utc),
                "bytes": self.path.stat().st_size,
                "compute_s": None if compute_s is None else round(float(compute_s), 1)}
        try:
            save_json(meta, self.meta_path)
        except Exception as exc:  # noqa: BLE001
            log(f"stage cache sidecar not written ({exc})", "WARNING")
        self.hit = True
        _STAGE_REFRESHED.add(str(self.path))
        log(f"stage cache saved {self.path.name} ({meta['bytes'] / 2**20:.1f} MiB, "
            f"{time.perf_counter() - t0:.1f}s)")
        return obj

    def get_or_compute(self, fn: Callable[[], Any]) -> Any:
        """Load the entry when it is a usable hit, otherwise compute ``fn()``, save and return it.

        Before computing, the time budget is checked (``LSTStop`` when 1.25 × the last compute time of this stage
        name, or of its newest sidecar before the first compute of the session, plus the reserve no longer fits).
        """
        if self.hit:
            try:
                return self.load()
            except StageCacheError as exc:
                log(f"{exc}; recomputing", "WARNING")
                self.hit = False
        norm = _stage_norm_name(self.name)
        check_budget(f"stage {self.name}", 1.25 * _stage_compute_estimate(norm))
        t0 = time.perf_counter()
        obj = fn()
        elapsed = time.perf_counter() - t0
        _STAGE_COMPUTE_S[norm] = elapsed
        return self.save(obj, compute_s=elapsed)

    def __repr__(self) -> str:
        return f"<stage_cache {self.path.name} hit={self.hit}>"


def stage_cache(name: str, key_obj: Any) -> _StageCache:
    """Return the cache handle for stage ``name`` keyed by the JSON-able ``key_obj`` (SPEC §3.1)."""
    return _StageCache(name, key_obj)


# ---- cache import ---------------------------------------------------------------------------------------------------
def _cache_has_content(path: Path) -> bool:
    return any((path / sub).is_dir() for sub in _CACHE_SUBDIRS + ("packed",))


def _search_cache_dir(root: Path) -> Optional[Path]:
    """Shallowest ``root/**/cache`` (excluding this run's own cache) that holds cache content, else None."""
    own = CFG.CACHE_DIR.resolve()
    try:
        candidates = sorted((p for p in root.glob("**/cache") if p.is_dir()), key=lambda p: (len(p.parts), str(p)))
    except OSError as exc:
        log(f"cache import: scanning {root} failed ({exc})", "WARNING")
        return None
    for path in candidates:
        try:
            if path.resolve() == own:
                continue
        except OSError:
            continue
        if _cache_has_content(path):
            return path
    return None


def _import_candidates_hint() -> str:
    """The cache folders found under /kaggle/input, for the error message of a wrong explicit LST_CACHE_IMPORT."""
    root = Path("/kaggle/input")
    if not root.is_dir():
        return ""
    try:
        found = sorted((str(p) for p in root.glob("**/cache") if p.is_dir() and _cache_has_content(p)),
                       key=lambda s: (s.count("/"), s))[:10]
    except OSError:
        found = []
    return ("; cache folders attached now: " + ", ".join(found) if found
            else "; no cache folder is attached under /kaggle/input")


def _find_import_cache() -> Optional[Path]:
    """Source folder for CACHE_IMPORT: explicit folder (itself, or its shallowest ``**/cache`` with content), or the
    shallowest ``/kaggle/input/*/**/cache`` with content. An explicit folder that does not exist or holds no cache
    raises at once: the user asked for a resume, and silently starting fresh would recompute hours of work."""
    setting = CFG.CACHE_IMPORT
    if setting == "off":
        return None
    if setting != "auto":
        path = Path(setting).expanduser()
        if not path.is_dir():
            raise FileNotFoundError(f"LST_CACHE_IMPORT={setting!r} is not a folder. Copy the path of the attached "
                                    "output from Kaggle's Input panel, use 'auto', or remove the setting"
                                    f"{_import_candidates_hint()}")
        try:
            if path.resolve() == CFG.CACHE_DIR.resolve():
                log(f"LST_CACHE_IMPORT={setting!r} is this run's own cache folder; nothing imported", "WARNING")
                return None
        except OSError:
            pass
        if _cache_has_content(path):
            return path
        found = _search_cache_dir(path)
        if found is None:
            raise FileNotFoundError(f"LST_CACHE_IMPORT={setting!r}: no cache folder (with stages/, tiles/, gddp/ or "
                                    f"packed/) inside it{_import_candidates_hint()}")
        return found
    root = Path("/kaggle/input")
    if not root.is_dir():
        return None
    return _search_cache_dir(root)


def _log_attached_run(src: Path) -> None:
    """Log how the run that produced an imported cache ended (its RUN_STOPPED.json / RUN_FAILED.json /
    LAUNCHER_RUN.json next to ``cache/``); warn when it used another profile."""
    for name in ("RUN_STOPPED.json", "RUN_FAILED.json", "LAUNCHER_RUN.json"):
        path = src.parent / name
        if not path.is_file():
            continue
        try:
            info = json.loads(path.read_text(encoding="utf-8"))
        except (OSError, ValueError) as exc:
            log(f"cache import: {path} is unreadable ({exc})", "WARNING")
            continue
        if not isinstance(info, dict):
            continue
        keys = ("status", "reason", "section", "cell", "error", "profile", "bundle_id", "utc")
        log(f"cache import: the attached run's {name}: "
            + ", ".join(f"{k}={str(info[k])[:160]}" for k in keys if info.get(k) is not None))
        if info.get("profile") and info["profile"] != CFG.PROFILE:
            log(f"cache import: the attached run used profile {info['profile']!r}, this run uses {CFG.PROFILE!r}; "
                "only stages with identical keys are reused (Earth Engine tiles and GDDP blocks of shared frames "
                "are)", "WARNING")


def _extract_packed(packed_dir: Path, dst_root: Path, stats_out: dict, label: str) -> None:
    """Unpack ``packed_dir/*.tar`` members under stages/, tiles/, gddp/ into ``dst_root`` (newer-than rule)."""
    margin = 1 << 30
    for tar_path in sorted(packed_dir.glob("*.tar")):
        try:
            with tarfile.open(tar_path, "r:") as tf:
                members = tf.getmembers()
        except (OSError, tarfile.TarError) as exc:
            log(f"cache import: {label} {tar_path.name} is unreadable ({type(exc).__name__}: {exc}); skipped",
                "WARNING")
            continue
        for m in members:
            if not m.isfile():
                if not m.isdir():
                    log(f"cache import: {tar_path.name}: non-file member {m.name!r} rejected", "WARNING")
                continue
            if not _safe_member_name(m.name):
                log(f"cache import: {tar_path.name}: unsafe member name {m.name!r} rejected", "WARNING")
                stats_out["rejected"] += 1
                continue
            top = m.name.split("/", 1)[0]
            if top not in _CACHE_SUBDIRS or m.name.endswith(".tmp"):
                continue
            if top in ("stages", "tiles"):
                stats_out["stage_tile_files"] += 1
            dest = dst_root / m.name
            try:
                if dest.exists() and dest.stat().st_mtime >= m.mtime:
                    stats_out["skipped"] += 1
                    continue
                dest.parent.mkdir(parents=True, exist_ok=True)
                if shutil.disk_usage(dest.parent).free < m.size + margin:
                    log("cache import stopped: free disk space would fall below 1 GiB", "WARNING")
                    stats_out["stopped_low_disk"] = True
                    return
                _write_member((tar_path, int(m.offset_data), int(m.size), float(m.mtime)), dest)
                stats_out["extracted"] += 1
                stats_out["bytes"] += int(m.size)
            except OSError as exc:
                log(f"cache import: could not extract {m.name} from {tar_path.name}: {exc}", "WARNING")


def import_cache() -> dict:
    """Merge a previous run's cache into ``CFG.CACHE_DIR`` without overwriting newer local files.

    Only ``stages/``, ``tiles/`` and ``gddp/`` are imported (loose files copied, ``packed/*.tar`` unpacked with
    member-name checks). A file is taken when it is absent locally or the imported copy has a newer modification
    time; ``*.tmp`` leftovers are skipped. Copying stops (with a warning) when the free disk space would drop below
    1 GiB (Kaggle's /kaggle/working is limited to ~20 GB). This run's own ``packed/*.tar`` (an in-place resume) are
    unpacked first. Returns copy statistics.
    """
    stats_out = {"source": None, "copied": 0, "skipped": 0, "extracted": 0, "rejected": 0, "bytes": 0,
                 "stage_tile_files": 0, "stopped_low_disk": False, "own_packed_extracted": 0}
    own_packed = CFG.CACHE_DIR / "packed"
    if own_packed.is_dir() and any(own_packed.glob("*.tar")):
        own = dict(stats_out)
        _extract_packed(own_packed, CFG.CACHE_DIR, own, "own packed cache")
        stats_out["own_packed_extracted"] = own["extracted"]
        log(f"cache: unpacked {own['extracted']} file(s) from this run's own {own_packed} "
            f"({own['skipped']} already present)")
    src = _find_import_cache()
    if src is None:
        if CFG.CACHE_IMPORT not in ("off",) and _IS_KAGGLE:
            log("cache import: no previous-run cache found under /kaggle/input (fresh run)")
        return stats_out
    stats_out["source"] = str(src)
    dst_root = CFG.CACHE_DIR
    log(f"cache import: merging {src} -> {dst_root}")
    _log_attached_run(src)
    margin = 1 << 30
    for sub in _CACHE_SUBDIRS:
        base = src / sub
        if not base.is_dir():
            continue
        for path in sorted(base.rglob("*")):
            if not path.is_file() or path.name.endswith(".tmp"):
                continue
            rel = path.relative_to(src)
            dest = dst_root / rel
            if sub in ("stages", "tiles"):
                stats_out["stage_tile_files"] += 1
            try:
                st = path.stat()
                if dest.exists() and dest.stat().st_mtime >= st.st_mtime:
                    stats_out["skipped"] += 1
                    continue
                dest.parent.mkdir(parents=True, exist_ok=True)
                if shutil.disk_usage(dest.parent).free < st.st_size + margin:
                    log("cache import stopped: free disk space would fall below 1 GiB", "WARNING")
                    stats_out["stopped_low_disk"] = True
                    break
                tmp = dest.with_name(dest.name + ".tmp")
                shutil.copy2(path, tmp)
                os.replace(tmp, dest)
                stats_out["copied"] += 1
                stats_out["bytes"] += st.st_size
            except OSError as exc:
                log(f"cache import: could not copy {rel}: {exc}", "WARNING")
        if stats_out["stopped_low_disk"]:
            break
    if (src / "packed").is_dir() and not stats_out["stopped_low_disk"]:
        _extract_packed(src / "packed", dst_root, stats_out, "imported packed cache")
    log(f"cache import: {stats_out['copied']} files copied, {stats_out['extracted']} unpacked "
        f"({stats_out['bytes'] / 2**20:,.1f} MiB), {stats_out['skipped']} kept (local copy newer or equal)"
        + (f", {stats_out['rejected']} unsafe tar member(s) rejected" if stats_out["rejected"] else ""))
    if stats_out["stage_tile_files"] == 0:
        log(f"cache import: {src} contains no stage or tile files (stages/, tiles/ or packed/*.tar) - nothing "
            "will be reused; check LST_CACHE_IMPORT and that the attached version's outputs/cache exists",
            "WARNING")
    return stats_out


# ---- cache packing (Kaggle output file cap) ------------------------------------------------------------------------
def _stage_group(stem: str) -> str:
    """Pack group of a stage name: its first two tokens (03_landsat, s04_thermal, 07_job, 08_boot, 11_land, ...);
    three when the second is day/night (09_day_hindcast)."""
    tokens = stem.split("_")
    n = 3 if len(tokens) > 2 and tokens[1] in ("day", "night") else 2
    return re.sub(r"[^A-Za-z0-9_.-]", "_", "_".join(tokens[:n])) or "misc"


def _pack_groups() -> dict:
    """{tar file name: [(arcname, path), ...]} of the loose cache files (``*.tmp`` excluded)."""
    root = CFG.CACHE_DIR
    groups: dict = {}
    stages = root / "stages"
    if stages.is_dir():
        for path in sorted(stages.iterdir()):
            if path.is_file() and not path.name.endswith(".tmp"):
                stem = path.name.rsplit("-", 1)[0] if "-" in path.name else path.stem
                groups.setdefault(f"stages_{_stage_group(stem)}.tar", []).append((f"stages/{path.name}", path))
    tiles = root / "tiles"
    if tiles.is_dir():
        for folder in sorted(tiles.iterdir()):
            if not folder.is_dir():
                continue
            group = re.sub(r"[^A-Za-z0-9_.-]", "_", folder.name.split("_", 1)[0]) or "misc"
            for path in sorted(folder.rglob("*")):
                if path.is_file() and not path.name.endswith(".tmp"):
                    groups.setdefault(f"tiles_{group}.tar", []).append(
                        (f"tiles/{path.relative_to(tiles).as_posix()}", path))
    gddp = root / "gddp"
    if gddp.is_dir():
        for path in sorted(gddp.rglob("*")):
            if path.is_file() and not path.name.endswith(".tmp"):
                groups.setdefault("gddp.tar", []).append((f"gddp/{path.relative_to(gddp).as_posix()}", path))
    return groups


def _pack_one(tar_path: Path, loose: list) -> int:
    """(Re)write ``tar_path`` = its previous members not superseded by ``loose`` + the loose files; verify; atomic."""
    tmp = tar_path.with_name(tar_path.name + ".tmp")
    names_new = {arc for arc, _ in loose}
    expected = []
    try:
        with tarfile.open(tmp, "w", format=tarfile.PAX_FORMAT) as out:
            if tar_path.is_file():
                with tarfile.open(tar_path, "r:") as old:
                    for m in old.getmembers():
                        if m.isfile() and m.name not in names_new and _safe_member_name(m.name):
                            out.addfile(m, old.extractfile(m))
                            expected.append(m.name)
            for arc, path in loose:
                out.add(str(path), arcname=arc, recursive=False)
                expected.append(arc)
        with tarfile.open(tmp, "r:") as check:
            got = [m.name for m in check.getmembers() if m.isfile()]
        if len(got) != len(expected) or set(got) != set(expected):
            raise RuntimeError(f"{tmp.name}: {len(got)} members written, {len(expected)} expected")
        os.replace(tmp, tar_path)
    except BaseException:
        try:
            tmp.unlink()
        except OSError:
            pass
        raise
    return len(expected)


def _count_output_files() -> int:
    try:
        return sum(1 for p in CFG.OUTPUT_DIR.rglob("*") if p.is_file())
    except OSError:
        return -1


def pack_cache(force: bool = False) -> dict:
    """Pack ``CFG.CACHE_DIR/{stages,tiles,gddp}`` into ``CFG.CACHE_DIR/packed/*.tar`` (D5).

    Active when ``CFG.PACK_CACHE`` (``LST_PACK_CACHE``, default on Kaggle) or ``force``. One plain tar per group
    (``stages_<stage prefix>.tar``, ``tiles_<tile-folder prefix>.tar``, ``gddp.tar``), merged with an existing tar of
    the same group, written atomically and verified; the loose files of a group are deleted right after its tar is
    verified (bounded peak disk). ``*.tmp`` files, ``web/`` and ``web_bundle.zip`` are never touched. Entries remain
    usable: stage caches and tiles are restored from the tars on demand. Returns {tar name: member count}.
    """
    if not (force or CFG.PACK_CACHE):
        log("pack_cache: skipped (LST_PACK_CACHE=0)", "DEBUG")
        return {}
    t0 = time.perf_counter()
    packed = CFG.CACHE_DIR / "packed"
    packed.mkdir(parents=True, exist_ok=True)
    out = {}
    with _PACKED_LOCK:
        for tar_name, loose in sorted(_pack_groups().items()):
            if not loose:
                continue
            try:
                out[tar_name] = _pack_one(packed / tar_name, loose)
            except Exception as exc:  # noqa: BLE001 - the loose files stay; packing is an optimisation
                log(f"pack_cache: {tar_name} not written ({type(exc).__name__}: {str(exc)[:200]}); its "
                    f"{len(loose)} loose file(s) are kept", "WARNING")
                continue
            for _, path in loose:
                try:
                    path.unlink()
                except OSError as exc:
                    log(f"pack_cache: could not remove {path.name} ({exc})", "WARNING")
        tiles = CFG.CACHE_DIR / "tiles"
        if tiles.is_dir():
            for folder in sorted(tiles.iterdir(), reverse=True):
                if folder.is_dir():
                    for sub in sorted((p for p in folder.rglob("*") if p.is_dir()), key=lambda p: -len(p.parts)):
                        try:
                            sub.rmdir()
                        except OSError:
                            pass
                    try:
                        folder.rmdir()
                    except OSError:
                        pass
        _PACKED_INDEX.update(sig=None, members={})
    n_files = _count_output_files()
    log(f"pack_cache: {len(out)} tar file(s) updated ({sum(out.values())} members) in "
        f"{time.perf_counter() - t0:.1f}s; {n_files} files under {CFG.OUTPUT_DIR}")
    if n_files > 450:
        log(f"{n_files} files under {CFG.OUTPUT_DIR}: Kaggle notebook outputs are reported to be capped at about 500 "
            "files - files beyond the cap may be missing from the saved version", "WARNING")
    return out


CACHE_IMPORT_STATS = import_cache()

## 0.7 LSTB binary container (SPEC §5.1)
Large per-cell arrays (cells, frame cubes, SHAP matrices) are exported as **LSTB** files: a small JSON header
followed by typed little-endian column blocks, the whole file gzip-compressed. Layout (byte-exact):

```
gzip( "LSTB" | u8 version=1 | u8[3] reserved=0 | u32le header_len | header (UTF-8 JSON, header_len bytes)
      | zero padding to an 8-byte boundary (relative to file start) | data section )
header = {"kind", "n", "meta", "columns": [{"name", "dtype", "shape", "scale", "offset", "nodata",
                                            "byte_offset" (from data start, multiple of 8), "byte_length"}]}
value = raw * scale + offset; raw == nodata → NaN; multi-dimensional shapes are row-major ([frames, n]).
```

Quantisation `raw = round((value − offset) / scale)`; NaN/±Inf → `nodata`; values are clipped to the dtype range
with the nodata code excluded (e.g. int16 with nodata −32768 stores −32767…32767). 8-byte alignment lets the
browser wrap each column as a typed array view without copying. `quant_params` chooses `(scale, offset)` so that the
finite range of a column (plus headroom) spans the integer range, which bounds the rounding error by `scale / 2`.

In [ ]:
LSTB_MAGIC = b"LSTB"
LSTB_VERSION = 1
_LSTB_DTYPES = {"int16": np.dtype("<i2"), "uint8": np.dtype("u1"), "int32": np.dtype("<i4"),
                "float32": np.dtype("<f4")}
_LSTB_DEFAULT_NODATA = {"int16": -32768, "uint8": 255, "int32": -2147483648}


def _lstb_raw_range(dtype: str, nodata: Optional[int]) -> tuple:
    """Integer range available for valid values (the nodata code is excluded when it is an endpoint)."""
    info = np.iinfo(_LSTB_DTYPES[dtype])
    lo, hi = int(info.min), int(info.max)
    if nodata is not None:
        if nodata == lo:
            lo += 1
        elif nodata == hi:
            hi -= 1
    return lo, hi


def quant_params(values, dtype: str = "int16", *, headroom: float = 0.02, nodata: Optional[int] = None) -> tuple:
    """``(scale, offset)`` mapping the finite range of ``values`` (p0-p100, widened by ``headroom`` × span on both
    sides) onto the usable integer range of ``dtype`` (nodata code excluded; default nodata of the dtype).

    float32 returns ``(1.0, 0.0)`` (stored unquantised). A constant or empty column gets a tiny positive scale so
    that its value is still represented exactly at raw 0 (empty → ``(1.0, 0.0)``).
    """
    if dtype not in _LSTB_DTYPES:
        raise ValueError(f"unsupported LSTB dtype {dtype!r}; use one of {list(_LSTB_DTYPES)}")
    if dtype == "float32":
        return 1.0, 0.0
    arr = np.asarray(values, dtype=np.float64).ravel()
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return 1.0, 0.0
    vmin, vmax = float(arr.min()), float(arr.max())
    nodata = _LSTB_DEFAULT_NODATA[dtype] if nodata is None else int(nodata)
    rmin, rmax = _lstb_raw_range(dtype, nodata)
    span = vmax - vmin
    if span <= 0.0:
        # Constant column: raw 0 must decode to the constant exactly (offset = constant). For uint8, raw 0 is the
        # range minimum, which is fine; the scale only has to be positive.
        scale = max(abs(vmin), 1.0) * 1e-6
        offset = vmin if rmin <= 0 <= rmax else vmin - rmin * scale
        return float(scale), float(offset)
    lo = vmin - headroom * span
    hi = vmax + headroom * span
    scale = (hi - lo) / float(rmax - rmin)
    offset = lo - rmin * scale
    return float(scale), float(offset)


def _lstb_kind_from_path(path: Path) -> str:
    """Default ``kind`` from the SPEC §5.2 file names (cells, cube, zones, features, xai)."""
    stem = path.name.split(".")[0]
    if stem == "cells":
        return "cells"
    if stem.startswith("cube_zones"):
        return "zones"
    if stem.startswith("cube_"):
        return "cube"
    if stem.startswith("xai_") and stem.endswith("_features"):
        return "features"
    if stem.startswith("xai_"):
        return "xai"
    return stem


def _lstb_encode_column(col: dict) -> tuple:
    """Validate and quantise one column dict → (header entry without offsets, raw little-endian bytes)."""
    if not isinstance(col, dict) or "name" not in col or "array" not in col:
        raise ValueError(f"LSTB column must be a dict with 'name' and 'array', got {type(col).__name__}")
    name = str(col["name"])
    dtype = str(col.get("dtype", "float32"))
    if dtype not in _LSTB_DTYPES:
        raise ValueError(f"LSTB column {name!r}: unsupported dtype {dtype!r} (use {list(_LSTB_DTYPES)})")
    scale = float(col.get("scale", 1.0) if col.get("scale") is not None else 1.0)
    offset = float(col.get("offset", 0.0) if col.get("offset") is not None else 0.0)
    if not (math.isfinite(scale) and scale > 0.0 and math.isfinite(offset)):
        raise ValueError(f"LSTB column {name!r}: scale must be finite and > 0 and offset finite "
                         f"(got scale={scale}, offset={offset})")
    src = np.asarray(col["array"])
    if src.dtype.kind not in "biuf":
        raise ValueError(f"LSTB column {name!r}: array dtype {src.dtype} is not numeric")
    nodata = col.get("nodata")
    target = _LSTB_DTYPES[dtype]
    if dtype == "float32":
        if nodata is not None:
            raise ValueError(f"LSTB column {name!r}: float32 columns store NaN natively; nodata must be None")
        values = src.astype(np.float64)
        raw = ((values - offset) / scale) if (scale != 1.0 or offset != 0.0) else values
        raw = raw.astype(target)
    else:
        invalid = ~np.isfinite(src) if src.dtype.kind == "f" else np.zeros(src.shape, dtype=bool)
        if nodata is None and invalid.any():
            nodata = _LSTB_DEFAULT_NODATA[dtype]
            log(f"LSTB column {name!r}: {int(invalid.sum())} NaN values and no nodata given; using {nodata}",
                "DEBUG")
        if nodata is not None:
            nodata = int(nodata)
            info = np.iinfo(target)
            if not info.min <= nodata <= info.max:
                raise ValueError(f"LSTB column {name!r}: nodata {nodata} outside the {dtype} range")
        rmin, rmax = _lstb_raw_range(dtype, nodata)
        if src.dtype.kind in "biu" and scale == 1.0 and offset == 0.0:
            exact = src.astype(np.int64)                       # integers pass through exactly
        else:
            with np.errstate(invalid="ignore"):
                exact = np.rint((src.astype(np.float64) - offset) / scale)
        with np.errstate(invalid="ignore"):
            n_clip = int(np.count_nonzero(((exact < rmin) | (exact > rmax)) & ~invalid))
        if n_clip:
            log(f"LSTB column {name!r}: {n_clip} value(s) clipped to the {dtype} range [{rmin}, {rmax}]", "DEBUG")
        exact = np.clip(np.where(invalid, 0, exact), rmin, rmax)
        raw = exact.astype(target)
        if nodata is not None:
            if rmin <= nodata <= rmax:
                collide = int(np.count_nonzero((raw == nodata) & ~invalid))
                if collide:
                    log(f"LSTB column {name!r}: {collide} valid value(s) equal the nodata code {nodata} and will "
                        "decode as NaN (choose an endpoint nodata)", "WARNING")
            raw[invalid] = nodata
    raw = np.ascontiguousarray(raw)
    entry = {"name": name, "dtype": dtype, "shape": [int(s) for s in raw.shape], "scale": scale, "offset": offset,
             "nodata": None if nodata is None else int(nodata)}
    return entry, raw.tobytes(order="C")


def lstb_write(path, columns: Sequence[dict], meta: Optional[dict] = None, *, kind: Optional[str] = None,
               n: Optional[int] = None, compresslevel: int = 9) -> Path:
    """Write an LSTB container (SPEC §5.1); gzip-compressed when ``path`` ends in ``.gz``.

    ``columns``: list of ``{name, array, dtype ("int16"|"uint8"|"int32"|"float32"), scale=1.0, offset=0.0,
    nodata=None}``; shapes ``[n]`` or ``[frames, n]`` (last axis = cells). ``kind`` defaults to ``meta["kind"]`` (removed
    from meta) or is inferred from the SPEC file name; ``n`` defaults to the last dimension of the columns, which must
    agree. The file is written atomically with a deterministic gzip header. Returns the path.
    """
    path = Path(path)
    meta = dict(meta or {})
    if kind is None:
        kind = meta.pop("kind", None) or _lstb_kind_from_path(path)
    if not columns:
        raise ValueError("lstb_write: at least one column is required")
    names = [str(c.get("name")) for c in columns]
    if len(set(names)) != len(names):
        raise ValueError(f"lstb_write: duplicate column names {names}")
    encoded = [_lstb_encode_column(c) for c in columns]
    last_dims = {e["shape"][-1] if e["shape"] else 1 for e, _ in encoded}
    if n is None:
        if len(last_dims) != 1:
            raise ValueError(f"lstb_write: columns disagree on n (last dimension): {sorted(last_dims)}")
        n = last_dims.pop()
    elif any(d != int(n) for d in last_dims):
        raise ValueError(f"lstb_write: n={n} but column last dimensions are {sorted(last_dims)}")
    entries, blobs, cursor = [], [], 0
    for entry, blob in encoded:
        entry["byte_offset"] = cursor
        entry["byte_length"] = len(blob)
        entries.append(entry)
        pad = (-len(blob)) % 8
        blobs.append(blob + b"\0" * pad)
        cursor += len(blob) + pad
    header = {"kind": str(kind), "n": int(n), "meta": _to_jsonable(meta, None), "columns": entries}
    header_bytes = json.dumps(header, ensure_ascii=False, separators=(",", ":"), allow_nan=False).encode("utf-8")
    prefix = LSTB_MAGIC + struct.pack("<B3xI", LSTB_VERSION, len(header_bytes)) + header_bytes
    prefix += b"\0" * ((-len(prefix)) % 8)
    payload = prefix + b"".join(blobs)
    if path.name.endswith(".gz"):
        payload = _gzip_bytes(payload, compresslevel)
    return _write_bytes_atomic(path, payload)


def lstb_parse(data: bytes) -> tuple:
    """Parse LSTB bytes (gzip or inflated) → ``(header, {name: raw typed ndarray})`` without decoding."""
    if data[:2] == b"\x1f\x8b":
        data = gzip.decompress(data)
    if len(data) < 12 or data[:4] != LSTB_MAGIC:
        raise ValueError("not an LSTB container (bad magic)")
    version, header_len = struct.unpack_from("<B3xI", data, 4)
    if version != LSTB_VERSION:
        raise ValueError(f"unsupported LSTB version {version}")
    if 12 + header_len > len(data):
        raise ValueError("truncated LSTB header")
    header = json.loads(data[12:12 + header_len].decode("utf-8"))
    data_start = 12 + header_len + ((-(12 + header_len)) % 8)
    raws = {}
    for col in header["columns"]:
        dt = _LSTB_DTYPES.get(col["dtype"])
        if dt is None:
            raise ValueError(f"LSTB column {col['name']!r}: unknown dtype {col['dtype']!r}")
        off, length = int(col["byte_offset"]), int(col["byte_length"])
        if off % 8:
            raise ValueError(f"LSTB column {col['name']!r}: byte_offset {off} is not 8-byte aligned")
        shape = tuple(int(s) for s in col["shape"])
        count = int(np.prod(shape)) if shape else 1
        if length != count * dt.itemsize or data_start + off + length > len(data):
            raise ValueError(f"LSTB column {col['name']!r}: byte_length {length} inconsistent with shape {shape}")
        raws[col["name"]] = np.frombuffer(data, dtype=dt, count=count, offset=data_start + off).reshape(shape)
    return header, raws


def lstb_read(path) -> dict:
    """Read an LSTB container → ``{"header": {...}, "columns": {name: float32 decoded array}}`` (NaN for nodata).

    Decoding is ``raw * scale + offset`` in float64, then cast to float32 (int32 identifiers above 2^24 lose exact
    integer precision in float32 - use :func:`lstb_parse` for raw integers).
    """
    header, raws = lstb_parse(Path(path).read_bytes())
    cols = {}
    for col in header["columns"]:
        raw = raws[col["name"]]
        values = raw.astype(np.float64) * float(col["scale"]) + float(col["offset"])
        if col.get("nodata") is not None:
            values[raw == col["nodata"]] = np.nan
        cols[col["name"]] = values.astype(np.float32)
    return {"header": header, "columns": cols}


def _lstb_selftest() -> None:
    """Tiny in-memory round trip so a broken container never reaches the export stage unnoticed."""
    import tempfile
    rng = np.random.default_rng(0)
    vals = rng.normal(40, 5, (3, 17)).astype(np.float32)
    vals[1, 4] = np.nan
    scale, offset = quant_params(vals, "int16")
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "selftest.lstb.gz"
        lstb_write(path, [{"name": "v", "array": vals, "dtype": "int16", "scale": scale, "offset": offset,
                           "nodata": -32768},
                          {"name": "id", "array": np.arange(17, dtype=np.int32), "dtype": "int32"}],
                   meta={"kind": "selftest"})
        back = lstb_read(path)
    got = back["columns"]["v"]
    assert back["header"]["kind"] == "selftest" and back["header"]["n"] == 17
    assert np.isnan(got[1, 4]) and np.nanmax(np.abs(got - vals)) <= scale / 2 + 1e-4
    assert np.array_equal(back["columns"]["id"], np.arange(17, dtype=np.float32))


_lstb_selftest()
log("LSTB container self-test passed")

## 0.8 Configuration and hardware summary

In [ ]:
def _show(obj) -> None:
    """Rich display inside notebooks, plain print in scripts."""
    try:
        from IPython.display import display
        if "ipykernel" in sys.modules:
            display(obj)
            return
    except ImportError:
        pass
    print(obj.to_string() if isinstance(obj, pd.DataFrame) else obj)


def _print_table(title: str, rows: Sequence[tuple], width: int = 110) -> None:
    """Two-column key/value table that reads well both in notebooks and in plain logs."""
    key_w = max(len(str(k)) for k, _ in rows) + 2
    print()
    print(title)
    print("-" * min(width, key_w + 60))
    for key, value in rows:
        text = str(value)
        print(f"{str(key):<{key_w}}{text if len(text) <= width - key_w else text[:width - key_w - 3] + '...'}")


_cfg_rows = [(k, json.dumps(v, default=str) if isinstance(v, (dict, list, tuple)) else v)
             for k, v in CFG.as_dict().items()]
_hw_rows = [
    ("Platform", ENV["platform"]), ("Python", ENV["python"]), ("Kaggle", ENV["is_kaggle"]),
    ("GPUs", f"{ENV['n_gpus']} x {', '.join(ENV['gpu_names']) or '-'}"),
    ("GPU memory (MiB)", ", ".join(str(m) for m in ENV["gpu_mem_mib"]) or "-"),
    ("torch CUDA devices", ENV["torch_device_count"]), ("torch", ENV["torch_version"] or "missing"),
    ("XGBoost", f"{ENV['xgb_version'] or 'missing'} (device={ENV['xgb_device']})"),
    ("CuPy / cuDF / cuML", f"{ENV['has_cupy']} / {ENV['has_cudf']} / {ENV['has_cuml']}"),
    ("Cache import", CACHE_IMPORT_STATS.get("source") or "-"),
] + [(f"lib: {name}", "ok" if mod is not None else "MISSING") for name, mod in _OPTIONAL_MODULES.items()]
_print_table("Configuration", _cfg_rows)
_print_table("Hardware / software", _hw_rows)
gpu_mem_report()
del _cfg_rows, _hw_rows

# 1. Study area, grids and data mode

**Study area.** Delhi National Capital Region (NCR): Delhi NCT plus 24 districts of Haryana, Uttar Pradesh and
Rajasthan (~55,000 km²), delineated from geoBoundaries ADM2 (fallback FAO GAUL 2015 level 2, final fallback an
approximate NCR outline partitioned into nearest-headquarters districts). The 25 canonical districts, their aliases
(incl. geoBoundaries' spelling "Samli" for Shamli) and the outline are fixed by `SPEC_v1.md` §1.1-§1.2.

**Grids.** Everything lives on explicit grids in UTM zone 43N (`EPSG:32643`) whose edges are multiples of the cell
size:

| Grid | Resolution | Role |
|---|---|---|
| `GRID` | `CFG.GRID_RES_M` (1 km) | analysis grid: features, models, SHAP, zones, projections |
| `GRID_FINE` | `GRID_RES_M / FINE_FACTOR` (250 m) | Landsat day field, downscaled night field, rasters; fine pixel (r, c) lies in cell (r // 4, c // 4) |
| land-cover grid | `CFG.fine_res()` (50 m gee / 100 m synthetic) | landscape metrics only (built in section 05 / 02) |

**Canonical cell order.** All ROI cells sorted by `cell_id = row × W + col`. `N_CELLS`, `CELL_IDS`, `CELL_ROWS`,
`CELL_COLS`, `CELL_LON`, `CELL_LAT` define the order and length of every exported per-cell array (frames with
partial coverage keep the full length and carry nodata).

**Data mode.** Earth Engine is initialised here (service account → persisted credentials → interactive login).
`auto` falls back to the clearly-labelled synthetic twin with a loud banner when Earth Engine is unavailable; once a
service-account key was supplied, a failure while building the study area stops the run instead (the user clearly
asked for real data). Sections 03-05 run only in gee mode, section 02 only in synthetic mode.

## 1.1 Constants: districts, reference geography, land-cover classes

In [ ]:
_lst_enter_section('01')  # section marker (generated by build_notebook.py)

In [ ]:
import functools
import unicodedata

NCR_DISTRICTS = [
    {"id": 0, "name": "Delhi NCT", "state": "Delhi", "hq": (77.21, 28.61),
     "aliases": ["delhi", "nct of delhi", "national capital territory of delhi"]},
    {"id": 1, "name": "Gurugram", "state": "Haryana", "hq": (77.03, 28.46), "aliases": ["gurgaon", "gurugram"]},
    {"id": 2, "name": "Faridabad", "state": "Haryana", "hq": (77.31, 28.41), "aliases": ["faridabad"]},
    {"id": 3, "name": "Palwal", "state": "Haryana", "hq": (77.33, 28.14), "aliases": ["palwal"]},
    {"id": 4, "name": "Nuh", "state": "Haryana", "hq": (77.00, 28.10), "aliases": ["mewat", "nuh"]},
    {"id": 5, "name": "Rewari", "state": "Haryana", "hq": (76.62, 28.19), "aliases": ["rewari"]},
    {"id": 6, "name": "Jhajjar", "state": "Haryana", "hq": (76.66, 28.61), "aliases": ["jhajjar"]},
    {"id": 7, "name": "Rohtak", "state": "Haryana", "hq": (76.61, 28.90), "aliases": ["rohtak"]},
    {"id": 8, "name": "Sonipat", "state": "Haryana", "hq": (77.02, 28.99), "aliases": ["sonipat", "sonepat"]},
    {"id": 9, "name": "Panipat", "state": "Haryana", "hq": (76.97, 29.39), "aliases": ["panipat"]},
    {"id": 10, "name": "Karnal", "state": "Haryana", "hq": (76.99, 29.69), "aliases": ["karnal"]},
    {"id": 11, "name": "Jind", "state": "Haryana", "hq": (76.32, 29.32), "aliases": ["jind"]},
    {"id": 12, "name": "Bhiwani", "state": "Haryana", "hq": (76.13, 28.79), "aliases": ["bhiwani"]},
    {"id": 13, "name": "Charkhi Dadri", "state": "Haryana", "hq": (76.27, 28.59),
     "aliases": ["charkhi dadri", "dadri"]},
    {"id": 14, "name": "Mahendragarh", "state": "Haryana", "hq": (76.11, 28.05),
     "aliases": ["mahendragarh", "mahendergarh", "narnaul"]},
    {"id": 15, "name": "Meerut", "state": "Uttar Pradesh", "hq": (77.71, 28.98), "aliases": ["meerut"]},
    {"id": 16, "name": "Ghaziabad", "state": "Uttar Pradesh", "hq": (77.44, 28.67), "aliases": ["ghaziabad"]},
    {"id": 17, "name": "Gautam Buddh Nagar", "state": "Uttar Pradesh", "hq": (77.49, 28.47),
     "aliases": ["gautam buddha nagar", "gautam buddh nagar", "gautambudhnagar", "noida"]},
    {"id": 18, "name": "Bulandshahr", "state": "Uttar Pradesh", "hq": (77.85, 28.40),
     "aliases": ["bulandshahr", "bulandshahar"]},
    {"id": 19, "name": "Baghpat", "state": "Uttar Pradesh", "hq": (77.22, 28.94), "aliases": ["baghpat", "bagpat"]},
    {"id": 20, "name": "Hapur", "state": "Uttar Pradesh", "hq": (77.78, 28.73), "aliases": ["hapur", "panchsheel nagar"]},
    # geoBoundaries (v6 IND ADM2) spells the district "Samli".
    {"id": 21, "name": "Shamli", "state": "Uttar Pradesh", "hq": (77.31, 29.45),
     "aliases": ["shamli", "samli", "prabudh nagar", "prabuddh nagar"]},
    {"id": 22, "name": "Muzaffarnagar", "state": "Uttar Pradesh", "hq": (77.70, 29.47), "aliases": ["muzaffarnagar"]},
    {"id": 23, "name": "Alwar", "state": "Rajasthan", "hq": (76.60, 27.55), "aliases": ["alwar"]},
    {"id": 24, "name": "Bharatpur", "state": "Rajasthan", "hq": (77.49, 27.22), "aliases": ["bharatpur"]},
]
DISTRICTS = [{"id": d["id"], "name": d["name"], "state": d["state"]} for d in NCR_DISTRICTS]

# "Dadri" is also a tehsil of Gautam Buddh Nagar (UP): only accept it on the Haryana side.
_ALIAS_LON_MAX = {"dadri": 76.9}
# Delhi's revenue districts (geoBoundaries has no ADM1 attribute, so Delhi is matched by name + bbox).
DELHI_SUBDISTRICTS = ["Central", "East", "New Delhi", "North", "North East", "North West", "Shahdara",
                      "South", "South East", "South West", "West"]
DELHI_BBOX = (76.83, 28.40, 77.35, 28.89)          # lon_min, lat_min, lon_max, lat_max
NCR_SANITY_BBOX = (75.3, 26.6, 78.6, 30.1)         # duplicate district names exist elsewhere in India

# Approximate NCR outline (SPEC_v1 §1.2): synthetic ROI and final GEE fallback.
NCR_OUTLINE = [
    (76.45, 29.95), (76.85, 29.95), (77.20, 29.75), (77.55, 29.75), (78.10, 29.55), (78.20, 29.20),
    (78.45, 28.60), (78.45, 28.20), (78.00, 28.05), (77.55, 27.95), (77.55, 27.50), (77.85, 27.20),
    (77.65, 26.75), (77.20, 26.70), (76.90, 27.05), (76.25, 27.05), (76.10, 27.40), (76.20, 27.90),
    (75.95, 28.00), (75.45, 28.40), (75.55, 28.90), (75.85, 29.05), (75.95, 29.55), (76.20, 29.60),
    (76.45, 29.95),
]
# Reference geography polylines (lon, lat) shared with the synthetic world (section 02).
YAMUNA = [(77.19, 29.95), (77.13, 29.55), (77.12, 29.20), (77.22, 28.85), (77.25, 28.62), (77.30, 28.45),
          (77.45, 28.20), (77.52, 27.90), (77.68, 27.55)]
GANGA = [(78.03, 29.75), (78.10, 29.30), (78.18, 28.95), (78.32, 28.60), (78.45, 28.30)]
ARAVALLI = [(77.17, 28.62), (77.10, 28.40), (76.85, 28.15), (76.60, 27.80), (76.40, 27.35)]

# Harmonised land-cover classes (SPEC_v1 §1) and the GLC_FCS30D remap used for every v2 frame (SPEC §1.4).
LC_NODATA, LC_BUILT, LC_FOREST, LC_WATER, LC_CROPLAND, LC_BARREN, LC_OTHER_VEG = 0, 1, 2, 3, 4, 5, 6
LC_CLASS_NAMES = {0: "nodata", 1: "built/impervious", 2: "tree/forest", 3: "water", 4: "cropland",
                  5: "barren/bare", 6: "other vegetation"}
LC_CLASS_COLORS = {0: "#000000", 1: "#e11d48", 2: "#15803d", 3: "#0ea5e9", 4: "#facc15", 5: "#a8a29e",
                   6: "#86efac"}
LC_REMAPS = {
    "glc_fcs30d": {**{c: 4 for c in (10, 11, 12, 20)},
                   **{c: 2 for c in (51, 52, 61, 62, 71, 72, 81, 82, 91, 92)},
                   **{c: 6 for c in (120, 121, 122, 130, 140, 150, 152, 153, *range(181, 188))},
                   190: 1, 200: 5, 201: 5, 202: 5, 210: 3, 220: 6},
    "esa_worldcover": {10: 2, 20: 6, 30: 6, 40: 4, 50: 1, 60: 5, 70: 6, 80: 3, 90: 6, 95: 2, 100: 6},
    "dynamic_world": {0: 3, 1: 2, 2: 6, 3: 6, 4: 4, 5: 6, 6: 1, 7: 5, 8: 6},
}

NODATA_FLOAT = -9999.0     # sentinel written by .unmask() for continuous bands, converted to NaN locally
NODATA_LC = 0              # land-cover nodata class
NODATA_DISTRICT = -1       # DISTRICT_IDX outside the ROI
# The snapped 1 km grid of the user's real v1 run (geoBoundaries ROI) - logged for comparison only.
_S01_V1_REAL_GRID = {"res": 1000, "x0": 545000.0, "y0": 3321000.0, "width": 297, "height": 366}
log(f"{len(NCR_DISTRICTS)} canonical districts; frames {CFG.FRAMES_OBS}; land cover {sorted(set(CFG.LC_SOURCES.values()))}")

## 1.2 Grid definition (`GridSpec`)
Coordinates are transformed with `pyproj` (`always_xy=True`: lon, lat order). If pyproj is missing, a closed-form
transverse-Mercator series (Snyder 1987, USGS PP 1395, eqs. 8-9 … 8-25) is used for UTM north zones — accurate to
well below a metre inside the zone, far below any pixel size used here. `GridSpec.fine(res)` nests a finer grid on
the same origin, so fine/coarse correspondences are pure integer arithmetic (`r // f`, `c // f`).

In [ ]:
_WGS84_A = 6378137.0
_WGS84_F = 1 / 298.257223563
_UTM_K0 = 0.9996


def _utm_zone(crs: str) -> int:
    """Zone number for a WGS84 / UTM north CRS code such as ``EPSG:32643``."""
    match = re.fullmatch(r"EPSG:326(\d{2})", crs.strip().upper())
    if not match:
        raise ValueError(f"pyproj is unavailable and {crs} is not a WGS84 UTM-north code; install pyproj")
    return int(match.group(1))


def _utm_forward(lon, lat, zone: int):
    """Geographic (deg) -> UTM north (m), Snyder's series."""
    e2 = _WGS84_F * (2 - _WGS84_F)
    ep2 = e2 / (1 - e2)
    phi, lam = np.radians(lat), np.radians(lon)
    lam0 = np.radians(-183.0 + 6.0 * zone)
    n = _WGS84_A / np.sqrt(1 - e2 * np.sin(phi) ** 2)
    t = np.tan(phi) ** 2
    c = ep2 * np.cos(phi) ** 2
    a = np.cos(phi) * (lam - lam0)
    m = _WGS84_A * ((1 - e2 / 4 - 3 * e2**2 / 64 - 5 * e2**3 / 256) * phi
                    - (3 * e2 / 8 + 3 * e2**2 / 32 + 45 * e2**3 / 1024) * np.sin(2 * phi)
                    + (15 * e2**2 / 256 + 45 * e2**3 / 1024) * np.sin(4 * phi)
                    - (35 * e2**3 / 3072) * np.sin(6 * phi))
    x = _UTM_K0 * n * (a + (1 - t + c) * a**3 / 6 + (5 - 18 * t + t**2 + 72 * c - 58 * ep2) * a**5 / 120) + 500000.0
    y = _UTM_K0 * (m + n * np.tan(phi) * (a**2 / 2 + (5 - t + 9 * c + 4 * c**2) * a**4 / 24
                                          + (61 - 58 * t + t**2 + 600 * c - 330 * ep2) * a**6 / 720))
    return x, y


def _utm_inverse(x, y, zone: int):
    """UTM north (m) -> geographic (deg), Snyder's footpoint-latitude series."""
    e2 = _WGS84_F * (2 - _WGS84_F)
    ep2 = e2 / (1 - e2)
    e1 = (1 - np.sqrt(1 - e2)) / (1 + np.sqrt(1 - e2))
    mu = (np.asarray(y, dtype=np.float64) / _UTM_K0) / (_WGS84_A * (1 - e2 / 4 - 3 * e2**2 / 64 - 5 * e2**3 / 256))
    phi1 = (mu + (3 * e1 / 2 - 27 * e1**3 / 32) * np.sin(2 * mu) + (21 * e1**2 / 16 - 55 * e1**4 / 32) * np.sin(4 * mu)
            + (151 * e1**3 / 96) * np.sin(6 * mu) + (1097 * e1**4 / 512) * np.sin(8 * mu))
    n1 = _WGS84_A / np.sqrt(1 - e2 * np.sin(phi1) ** 2)
    t1 = np.tan(phi1) ** 2
    c1 = ep2 * np.cos(phi1) ** 2
    r1 = _WGS84_A * (1 - e2) / (1 - e2 * np.sin(phi1) ** 2) ** 1.5
    d = (np.asarray(x, dtype=np.float64) - 500000.0) / (n1 * _UTM_K0)
    lat = phi1 - (n1 * np.tan(phi1) / r1) * (d**2 / 2 - (5 + 3 * t1 + 10 * c1 - 4 * c1**2 - 9 * ep2) * d**4 / 24
                                             + (61 + 90 * t1 + 298 * c1 + 45 * t1**2 - 252 * ep2 - 3 * c1**2) * d**6 / 720)
    lon = np.radians(-183.0 + 6.0 * zone) + (d - (1 + 2 * t1 + c1) * d**3 / 6
                                              + (5 - 2 * c1 + 28 * t1 - 3 * c1**2 + 8 * ep2 + 24 * t1**2) * d**5 / 120) / np.cos(phi1)
    return np.degrees(lon), np.degrees(lat)


@functools.lru_cache(maxsize=8)
def _grid_transformer(src: str, dst: str):
    return pyproj.Transformer.from_crs(src, dst, always_xy=True)


def _grid_lonlat_to_xy(lon, lat, crs: str):
    """(lon, lat) degrees -> projected metres in ``crs`` (arrays of any shape)."""
    lon, lat = np.asarray(lon, dtype=np.float64), np.asarray(lat, dtype=np.float64)
    if pyproj is not None:
        return _grid_transformer("EPSG:4326", crs).transform(lon, lat)
    return _utm_forward(lon, lat, _utm_zone(crs))


def _grid_xy_to_lonlat(x, y, crs: str):
    """Projected metres in ``crs`` -> (lon, lat) degrees (arrays of any shape)."""
    x, y = np.asarray(x, dtype=np.float64), np.asarray(y, dtype=np.float64)
    if pyproj is not None:
        return _grid_transformer(crs, "EPSG:4326").transform(x, y)
    return _utm_inverse(x, y, _utm_zone(crs))


def _densify_ring(points, n_per_edge: int = 40) -> np.ndarray:
    """Insert points along each edge of a lon/lat ring so its projected extent is exact (edges curve in UTM)."""
    pts = np.asarray(points, dtype=np.float64)
    t = np.linspace(0.0, 1.0, n_per_edge, endpoint=False)[:, None]
    parts = [pts[i] + t * (pts[i + 1] - pts[i]) for i in range(len(pts) - 1)]
    return np.vstack(parts + [pts[-1:]])


@dataclass(frozen=True)
class GridSpec:
    """North-up raster grid: ``x0``/``y0`` are the LEFT/TOP edges (m) in ``crs``; rows run southwards."""

    crs: str
    res: int
    x0: float
    y0: float
    width: int
    height: int

    @property
    def shape(self) -> tuple:
        return (self.height, self.width)

    @property
    def cell_area_km2(self) -> float:
        return (self.res / 1000.0) ** 2

    def x_centers(self) -> np.ndarray:
        return self.x0 + (np.arange(self.width) + 0.5) * self.res

    def y_centers(self) -> np.ndarray:
        return self.y0 - (np.arange(self.height) + 0.5) * self.res

    def cell_centers_xy(self):
        """(X, Y) 2-D arrays (float64, metres) of cell centres, shape (height, width)."""
        return np.meshgrid(self.x_centers(), self.y_centers())

    def lonlat(self):
        """(LON, LAT) 2-D arrays (float64, degrees) of cell centres."""
        x, y = self.cell_centers_xy()
        return _grid_xy_to_lonlat(x, y, self.crs)

    def affine(self) -> dict:
        """``affineTransform`` dict for ``ee.data.computePixels`` (north-up, negative scaleY)."""
        return {"scaleX": float(self.res), "shearX": 0.0, "translateX": float(self.x0),
                "shearY": 0.0, "scaleY": -float(self.res), "translateY": float(self.y0)}

    def transform_list(self) -> list:
        """Six-element affine ``[scaleX, shearX, translateX, shearY, scaleY, translateY]`` (EE crsTransform)."""
        return [float(self.res), 0.0, float(self.x0), 0.0, -float(self.res), float(self.y0)]

    def fine(self, res: int) -> "GridSpec":
        """Nested grid at ``res`` metres sharing the origin; ``res`` must divide ``self.res``."""
        res = int(res)
        if res <= 0 or self.res % res:
            raise ValueError(f"fine resolution {res} m must divide the grid resolution {self.res} m")
        f = self.res // res
        return GridSpec(self.crs, res, self.x0, self.y0, self.width * f, self.height * f)

    def tile(self, row0: int, col0: int, height: int, width: int) -> "GridSpec":
        """Sub-grid starting at (row0, col0) with the given size."""
        return GridSpec(self.crs, self.res, self.x0 + col0 * self.res, self.y0 - row0 * self.res, width, height)

    def bounds_xy(self) -> list:
        """[x_left, y_bottom, x_right, y_top] in metres."""
        return [float(self.x0), float(self.y0 - self.height * self.res),
                float(self.x0 + self.width * self.res), float(self.y0)]

    def bounds_lonlat(self) -> list:
        """[min_lon, min_lat, max_lon, max_lat] of the grid rectangle."""
        ring = [(self.x0, self.y0), (self.x0 + self.width * self.res, self.y0),
                (self.x0 + self.width * self.res, self.y0 - self.height * self.res),
                (self.x0, self.y0 - self.height * self.res), (self.x0, self.y0)]
        pts = _densify_ring(ring, 20)
        lon, lat = _grid_xy_to_lonlat(pts[:, 0], pts[:, 1], self.crs)
        return [float(lon.min()), float(lat.min()), float(lon.max()), float(lat.max())]

    def to_dict(self) -> dict:
        """Manifest form ``{crs, res, x0, y0, width, height}`` (SPEC §5.3 ``study_area.grid``)."""
        return {"crs": self.crs, "res": int(self.res), "x0": float(self.x0), "y0": float(self.y0),
                "width": int(self.width), "height": int(self.height)}

    @classmethod
    def from_lonlat_points(cls, lonlat_points, crs: str, res: int, margin_cells: int = 1) -> "GridSpec":
        """Smallest grid covering the points, edges snapped OUTWARD to multiples of ``res`` (+ margin)."""
        pts = np.asarray(lonlat_points, dtype=np.float64)
        x, y = _grid_lonlat_to_xy(pts[:, 0], pts[:, 1], crs)
        res = int(res)
        x_left = (math.floor(np.min(x) / res) - margin_cells) * res
        x_right = (math.ceil(np.max(x) / res) + margin_cells) * res
        y_bottom = (math.floor(np.min(y) / res) - margin_cells) * res
        y_top = (math.ceil(np.max(y) / res) + margin_cells) * res
        return cls(crs, res, float(x_left), float(y_top), int((x_right - x_left) // res), int((y_top - y_bottom) // res))


def _outline_grid(res: int) -> GridSpec:
    """Grid covering the approximate NCR outline (synthetic mode / final fallback)."""
    return GridSpec.from_lonlat_points(_densify_ring(NCR_OUTLINE), CFG.CRS, res)


def _mask_fine_inplace(lc: np.ndarray, roi_mask: np.ndarray, f: int) -> None:
    """Zero fine pixels whose parent cell is outside the ROI, without materialising a fine-resolution mask."""
    h, w = roi_mask.shape
    if not lc.flags.c_contiguous or lc.shape != (h * f, w * f):
        raise ValueError(f"fine raster must be C-contiguous with shape {(h * f, w * f)}, got {lc.shape}")
    view = lc.reshape(h, f, w, f)
    view *= roi_mask[:, None, :, None].astype(lc.dtype)


def block_mean(fine: np.ndarray, f: int, min_valid_frac: float = 0.0) -> np.ndarray:
    """NaN-aware mean of every ``f × f`` block of a fine raster → coarse raster (float32).

    A block whose share of finite pixels is below ``min_valid_frac`` (or that has none) is NaN. Used for the fine →
    1 km aggregation (e.g. the 4 × 4 block means of SPEC §4.8 and the ≥ 50 % valid rule of §4.1).
    """
    fine = np.asarray(fine, dtype=np.float32)
    h, w = fine.shape[0] // f, fine.shape[1] // f
    if fine.shape != (h * f, w * f):
        raise ValueError(f"fine raster shape {fine.shape} is not a multiple of {f}")
    blocks = fine.reshape(h, f, w, f)
    valid = np.isfinite(blocks)
    n_valid = valid.sum(axis=(1, 3))
    total = np.where(valid, blocks, 0.0).sum(axis=(1, 3), dtype=np.float64)
    with np.errstate(invalid="ignore", divide="ignore"):
        out = (total / n_valid).astype(np.float32)
    out[(n_valid == 0) | (n_valid < min_valid_frac * f * f)] = np.nan
    return out


def upsample_repeat(coarse: np.ndarray, f: int) -> np.ndarray:
    """Nearest-neighbour (block-repeat) upsampling of a coarse raster by an integer factor."""
    return np.repeat(np.repeat(np.asarray(coarse), f, axis=0), f, axis=1)


_probe_grid = _outline_grid(CFG.GRID_RES_M)
log(f"NCR outline grid preview: {_probe_grid.width}x{_probe_grid.height} cells of {_probe_grid.res} m, "
    f"lon/lat bounds {np.round(_probe_grid.bounds_lonlat(), 3).tolist()}")
del _probe_grid

## 1.3 Earth Engine initialisation and data-mode decision
Credential order: (1) Kaggle secret `GEE_SERVICE_ACCOUNT_KEY`, or env `LST_GEE_SERVICE_ACCOUNT_KEY` /
`GOOGLE_APPLICATION_CREDENTIALS` → `ee.ServiceAccountCredentials`; (2) persisted user credentials; (3) interactive
`ee.Authenticate(auth_mode="notebook")` (live notebooks only). Sessions use the **standard endpoint** by default
(`LST_GEE_ENDPOINT=standard`): this pipeline's requests are heavy computations (30 m Landsat medians, multi-year
MODIS stacks), and Google recommends the **high-volume endpoint** only for many small, cacheable requests such as
simple tile reads; `LST_GEE_ENDPOINT=highvolume` is an opt-in. Both endpoints share the same per-project quota of
40 concurrent requests. After a successful login the client request deadline is set to `LST_GEE_DEADLINE_S`
(420 s; at least 330 s, above the ~300 s interactive server limit) with 3 HTTP-level retries. The Earth Engine
work of sections 03, 04 and 05 runs under the **workload tags** `lst-v2-s03`, `lst-v2-s04`, `lst-v2-s05`, so Google
Cloud Monitoring (Earth Engine -> *Completed EECU-seconds*, grouped by `workload_tag`) shows the EECU cost of each
section. A supplied key that fails is reported with the setup step that fixes it (instead of silently falling
through to OAuth, whose errors hide the real cause).

With `LST_GEE_EXPORT=asset` the asset folder `CFG.GEE_ASSET_ROOT` is checked (and created) here, so a missing
*Earth Engine Resource Writer* role is reported before hours of computation; on failure the run continues in
`interactive` mode with a warning.

In [ ]:
_EE_HIGHVOLUME_URL = "https://earthengine-highvolume.googleapis.com"
_S01_KEY_SUPPLIED = False        # set by init_gee: a service-account key was found (user wants real data)
_S01_EE_MAX_RETRIES = 3          # HTTP-level retries of the Earth Engine client (ee.data.setMaxRetries)


def _ee_init_kwargs() -> dict:
    """``ee.Initialize`` endpoint argument: none for the standard endpoint, ``opt_url`` for high-volume (D1)."""
    if CFG.GEE_ENDPOINT == "highvolume":
        return {"opt_url": _EE_HIGHVOLUME_URL}
    if CFG.GEE_ENDPOINT != "standard":
        raise ValueError(f"LST_GEE_ENDPOINT={CFG.GEE_ENDPOINT!r}; choose 'standard' or 'highvolume'")
    return {}


def _ee_configure_session() -> None:
    """Request deadline + client retries after a successful Initialize (guarded: older clients lack the setters)."""
    deadline_ms = int(float(CFG.GEE_DEADLINE_S) * 1000)
    done = []
    if hasattr(ee.data, "setDeadline"):
        ee.data.setDeadline(deadline_ms)
        done.append(f"deadline {deadline_ms / 1000:.0f} s")
    if hasattr(ee.data, "setMaxRetries"):
        ee.data.setMaxRetries(_S01_EE_MAX_RETRIES)
        done.append(f"max retries {_S01_EE_MAX_RETRIES}")
    url = _EE_HIGHVOLUME_URL if CFG.GEE_ENDPOINT == "highvolume" else "standard (earthengine.googleapis.com)"
    log(f"Earth Engine session: endpoint {CFG.GEE_ENDPOINT} [{url}], {', '.join(done) or 'client defaults'}")


@contextmanager
def gee_workload_tag(tag: str):
    """Run the enclosed Earth Engine work under a workload tag (EECU per section in Cloud Monitoring).

    Uses ``ee.data.workloadTagContext`` when available, else ``setWorkloadTag`` / ``resetWorkloadTag``, else does
    nothing. The tag is reset on any exit (also on exceptions and LSTStop). The Earth Engine client keeps the tag in
    process-wide state, so worker threads started inside the block inherit it.
    """
    data = getattr(ee, "data", None) if ee is not None else None
    ctx = getattr(data, "workloadTagContext", None) if data is not None else None
    if callable(ctx):
        with ctx(tag):
            yield
        return
    setter = getattr(data, "setWorkloadTag", None) if data is not None else None
    resetter = getattr(data, "resetWorkloadTag", None) if data is not None else None
    if callable(setter):
        setter(tag)
        try:
            yield
        finally:
            if callable(resetter):
                resetter()
            else:
                setter("")
        return
    yield


def _service_account_key_text(verbose: bool = True) -> Optional[str]:
    """Service-account JSON text from Kaggle secrets / env var (JSON or path) / GOOGLE_APPLICATION_CREDENTIALS.

    Logs which source was used (never the key itself) and warns loudly about a value that is set but unusable,
    because silently ignoring it would push the run into interactive OAuth login instead.
    """
    say = log if verbose else (lambda *a, **k: None)
    candidates = [("Kaggle secret GEE_SERVICE_ACCOUNT_KEY", _kaggle_secret("GEE_SERVICE_ACCOUNT_KEY")),
                  ("env LST_GEE_SERVICE_ACCOUNT_KEY", _env_str("LST_GEE_SERVICE_ACCOUNT_KEY")),
                  ("env GOOGLE_APPLICATION_CREDENTIALS", _env_str("GOOGLE_APPLICATION_CREDENTIALS"))]
    for source, value in candidates:
        if not value:
            continue
        text = value.strip().strip("'\"").strip()  # tolerate a key pasted inside quotes
        if text.startswith("{"):
            say(f"Earth Engine service-account key found in {source}")
            return text
        path = Path(text).expanduser()
        if path.is_file():
            try:
                file_text = path.read_text(encoding="utf-8")
                if json.loads(file_text).get("type") == "service_account":
                    say(f"Earth Engine service-account key file found via {source}: {path}")
                    return file_text
                say(f"{source} points to {path}, which is not a service-account key; ignoring", "WARNING")
            except (OSError, json.JSONDecodeError, AttributeError) as exc:
                say(f"credential file {path} ({source}) is not a readable JSON key ({type(exc).__name__}); "
                    "ignoring", "WARNING")
            continue
        say(f"{source} is set but is neither JSON nor an existing file. Paste the ENTIRE key file contents "
            "(it starts with '{' and contains \"type\": \"service_account\").", "WARNING")
    if _IS_KAGGLE:
        say("No Earth Engine service-account key found. On Kaggle: Add-ons -> Secrets -> add "
            "GEE_SERVICE_ACCOUNT_KEY and make sure its checkbox is ticked for THIS notebook.", "WARNING")
    return None


def _service_account_hint(message: str) -> str:
    """Map common service-account failures to the setup step that fixes them."""
    low = message.lower()
    if "not signed up" in low or "not registered" in low or "register" in low:
        return ("the Cloud project is not registered for Earth Engine: register it at "
                "https://code.earthengine.google.com/register (noncommercial; the Contributor tier gives "
                "1,000 EECU-h/month)")
    if "serviceusage" in low or "user_project_denied" in low or "caller does not have required permission" in low:
        return ("grant the service account the roles 'Service Usage Consumer' and 'Earth Engine Resource Viewer' "
                "in the Cloud project, and enable the Earth Engine API there")
    if "has not been used in project" in low or "is disabled" in low:
        return "enable the 'Google Earth Engine API' in the Cloud project (APIs & Services -> Library)"
    if "invalid_grant" in low or "invalid jwt" in low or "account not found" in low:
        return "the key was deleted/revoked or belongs to a deleted account: create a new JSON key"
    if "project" in low and ("not found" in low or "no project" in low or "required" in low):
        return "set the Kaggle secret GEE_PROJECT (or LST_GEE_PROJECT) to the Cloud project id"
    if "keyerror" in low or "client_email" in low or "jsondecodeerror" in low:
        return "the secret is not a complete service-account key: paste the whole JSON key file"
    return "check the key, the project id and the service account's roles"


def _ee_smoke_test() -> None:
    """Round-trip a trivial computation so we fail fast on bad credentials / unregistered projects."""
    if ee.Number(1).add(1).getInfo() != 2:
        raise RuntimeError("Earth Engine smoke test returned an unexpected value")


def init_gee(project: Optional[str] = None) -> tuple:
    """Initialise Earth Engine. Returns ``(ok, message)`` and never raises.

    The endpoint is ``CFG.GEE_ENDPOINT``: ``standard`` (default; no ``opt_url``) or ``highvolume`` (``opt_url`` =
    the high-volume URL). After every successful ``ee.Initialize`` + smoke test the request deadline is set to
    ``CFG.GEE_DEADLINE_S`` and the client retries to 3. On success ``CFG.GEE_PROJECT`` holds the project actually
    used (the key's ``project_id`` when no project was configured) and ``CFG.GEE_ASSET_ROOT`` defaults to
    ``projects/<project>/assets/ncr_lst_v2``.
    """
    global _S01_KEY_SUPPLIED
    if ee is None:
        return False, "earthengine-api is not installed"
    project = project or CFG.GEE_PROJECT
    _ee_init_kwargs()                # ValueError for an invalid LST_GEE_ENDPOINT (never silently another endpoint)
    log(f"Earth Engine project: {project or '(not set - will use the key file project_id)'}, endpoint "
        f"{CFG.GEE_ENDPOINT}")
    errors = []
    key_text = _service_account_key_text()
    _S01_KEY_SUPPLIED = bool(key_text)
    used_project = None
    if key_text:
        email = "?"
        try:
            info = json.loads(key_text)
            email = info["client_email"]
            credentials = ee.ServiceAccountCredentials(email, key_data=key_text)
            used_project = project or info.get("project_id")
            ee.Initialize(credentials, project=used_project, **_ee_init_kwargs())
            _ee_smoke_test()
            _ee_configure_session()
            message = f"service account {email} (project {used_project})"
        except Exception as exc:  # noqa: BLE001 - reported with a remedy below
            detail = f"{type(exc).__name__}: {str(exc)[:400]}"
            hint = _service_account_hint(detail)
            log(f"Service-account login failed for {email}: {detail}\n    Fix: {hint}", "ERROR")
            return False, f"service account {email}: {detail} | fix: {hint}"
    else:
        message = None
        try:
            ee.Initialize(project=project, **_ee_init_kwargs())
            _ee_smoke_test()
            _ee_configure_session()
            used_project, message = project, f"persisted user credentials (project {project})"
        except Exception as exc:  # noqa: BLE001
            errors.append(f"persisted credentials: {type(exc).__name__}: {str(exc)[:300]}")
        if message is None and _is_interactive():
            log("Falling back to interactive Earth Engine login (no service-account key). If the login page says "
                "'Project has an incompatible OAuth2 Client configuration', pick (or create) a Cloud project with "
                "NO OAuth 2.0 client IDs for authentication - or, better, use the GEE_SERVICE_ACCOUNT_KEY secret, "
                "which needs no browser login.", "WARNING")
            try:
                ee.Authenticate(auth_mode="notebook")
                ee.Initialize(project=project, **_ee_init_kwargs())
                _ee_smoke_test()
                _ee_configure_session()
                used_project, message = project, f"interactive notebook authentication (project {project})"
            except Exception as exc:  # noqa: BLE001
                errors.append(f"interactive auth: {type(exc).__name__}: {str(exc)[:300]}")
        elif message is None:
            errors.append("interactive auth skipped (not a live notebook session)")
        if message is None:
            return False, " | ".join(errors)
    if used_project and not CFG.GEE_PROJECT:
        CFG.GEE_PROJECT = used_project
    if CFG.GEE_ASSET_ROOT is None and CFG.GEE_PROJECT:
        CFG.GEE_ASSET_ROOT = f"projects/{CFG.GEE_PROJECT}/assets/ncr_lst_v2"
    return True, message


def _s01_prepare_asset_root() -> tuple:
    """Ensure ``CFG.GEE_ASSET_ROOT`` exists for ``GEE_EXPORT=asset`` (creating the folder); ``(ok, message)``."""
    root = CFG.GEE_ASSET_ROOT
    if not root:
        return False, "no asset root: GEE_PROJECT is unknown (set the GEE_PROJECT secret or LST_GEE_ASSET_ROOT)"
    try:
        retry(lambda: ee.data.getAsset(root), tries=2, base_delay=2.0, max_delay=10.0)
        return True, f"asset folder {root} exists"
    except Exception as exc:  # noqa: BLE001 - "not found" is expected on the first run
        missing = re.search(r"not found|does not exist|404", str(exc), re.IGNORECASE)
        if not missing:
            return False, (f"cannot read asset folder {root}: {type(exc).__name__}: {str(exc)[:200]} | fix: grant "
                           "the service account the IAM role 'Earth Engine Resource Writer'")
    try:
        ee.data.createAsset({"type": "FOLDER"}, root)
        return True, f"asset folder {root} created"
    except Exception as exc:  # noqa: BLE001
        return False, (f"cannot create asset folder {root}: {type(exc).__name__}: {str(exc)[:200]} | fix: grant the "
                       "service account the IAM role 'Earth Engine Resource Writer' (or set LST_GEE_EXPORT="
                       "interactive)")


def _print_synthetic_banner(reason: str) -> None:
    """Impossible-to-miss notice that every number downstream comes from the synthetic twin."""
    bar = "!" * 96
    lines = [
        "SYNTHETIC MODE - RESULTS ARE NOT REAL OBSERVATIONS",
        "Earth Engine data were NOT used. All rasters, models, SHAP values, thresholds, zones, timelines and "
        "projections below come from a synthetic twin of Delhi NCR with PLANTED responses (SYNTHETIC_TRUTH). "
        "Use them only to test the pipeline and the dashboard. Every export is flagged data_mode=synthetic.",
        f"Reason: {reason[:300]}",
        "To use real data: attach the Kaggle secrets GEE_SERVICE_ACCOUNT_KEY + GEE_PROJECT (or authenticate "
        "interactively) and re-run with LST_RUN_MODE=gee.",
    ]
    import textwrap

    print("\n" + bar + "\n" + bar)
    for line in lines:
        for piece in textwrap.wrap(line, 88) or [""]:
            print(f"!!!  {piece:<88}!!!")
    print(bar + "\n" + bar + "\n", flush=True)


_S01_ASSET_CAVEAT = None
if CFG.RUN_MODE == "synthetic":
    DATA_MODE = "synthetic"
    _MODE_REASON = "LST_RUN_MODE=synthetic"
else:
    _gee_ok, _gee_msg = init_gee()
    if _gee_ok:
        DATA_MODE = "gee"
        _MODE_REASON = f"Earth Engine initialised via {_gee_msg}"
        log(_MODE_REASON)
        if CFG.GEE_EXPORT == "asset":
            _asset_ok, _asset_msg = _s01_prepare_asset_root()
            if _asset_ok:
                log(f"GEE_EXPORT=asset: {_asset_msg}")
            else:
                log(f"GEE_EXPORT=asset unavailable - continuing with interactive computePixels: {_asset_msg}",
                    "WARNING")
                CFG.GEE_EXPORT = "interactive"
                _S01_ASSET_CAVEAT = ("Batch asset export was requested but unavailable (" + _asset_msg[:200]
                                     + "); extraction used interactive computePixels.")
            del _asset_ok, _asset_msg
    elif CFG.RUN_MODE == "gee":
        raise RuntimeError(
            "LST_RUN_MODE=gee but Earth Engine could not be initialised.\n"
            f"Details: {_gee_msg}\n"
            "Fix: on Kaggle add Secrets GEE_SERVICE_ACCOUNT_KEY (full service-account JSON key; the account "
            "must be registered for Earth Engine) and GEE_PROJECT (Cloud project with the Earth Engine API "
            "enabled), enable Internet, or run interactively to authenticate; elsewhere set "
            "LST_GEE_SERVICE_ACCOUNT_KEY / GOOGLE_APPLICATION_CREDENTIALS and LST_GEE_PROJECT. "
            "Use LST_RUN_MODE=auto to fall back to synthetic data instead.")
    else:
        DATA_MODE = "synthetic"
        _MODE_REASON = f"Earth Engine unavailable ({_gee_msg})"
    del _gee_ok, _gee_msg
CFG.resolve_mode(DATA_MODE)
if DATA_MODE == "synthetic":
    _print_synthetic_banner(_MODE_REASON)
log(f"DATA_MODE={DATA_MODE} | GRID_RES_M={CFG.GRID_RES_M} | fine grid {CFG.grid_fine_res()} m | "
    f"land-cover grid {CFG.fine_res()} m | GEE_EXPORT={CFG.GEE_EXPORT}")

## 1.4 Earth Engine: ROI and district matching
Candidate ADM2 polygons are pulled for the NCR bounding box (attributes + centroid + area only), matched to the 25
canonical districts client-side by alias (case/space/diacritic-insensitive) with a bounding-box sanity filter, and
the matched polygons are tagged with the canonical `did`. Delhi's revenue districts are merged into "Delhi NCT".

**Unmatched districts are located, not assumed merged.** For every canonical district still unmatched, the layer is
queried for the polygon containing its headquarters point: an *unmatched* polygon there (a spelling the alias table
does not know) is matched by location with a loud warning; a polygon already matched to another district means the
district is genuinely merged into that parent in this layer (acceptable); no polygon at all is reported as an error.

In [ ]:
_GEOBOUNDARIES_ADM2 = "WM/geoLab/geoBoundaries/600/ADM2"
_GAUL_ADM2 = "FAO/GAUL/2015/level2"


def _norm_name(text) -> str:
    """Lower-case ASCII alphanumerics only ("Gautam Buddha Nagar" -> "gautambuddhanagar")."""
    ascii_text = unicodedata.normalize("NFKD", str(text or "")).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", ascii_text.lower())


def _strip_admin_words(norm: str) -> str:
    for word in ("district", "delhi", "nctof", "nct"):
        norm = norm.replace(word, "")
    return norm


_ALIAS_TO_ID = {_norm_name(a): d["id"] for d in NCR_DISTRICTS[1:] for a in d["aliases"]}
_DELHI_NAME_KEYS = {_strip_admin_words(_norm_name(s)) for s in DELHI_SUBDISTRICTS} | {""}
_DELHI_ADM1_KEYS = {_norm_name(a) for a in NCR_DISTRICTS[0]["aliases"]}


def _in_bbox(lon: float, lat: float, bbox) -> bool:
    return bbox[0] <= lon <= bbox[2] and bbox[1] <= lat <= bbox[3]


def _match_district(record: dict) -> Optional[int]:
    """Canonical district id for one candidate ADM2 record, or None if it is not an NCR district."""
    lon, lat = record.get("lon"), record.get("lat")
    if lon is None or lat is None or not _in_bbox(lon, lat, NCR_SANITY_BBOX):
        return None
    name = _norm_name(record.get("name"))
    adm1 = _norm_name(record.get("adm1")) if record.get("adm1") else None
    stripped = _strip_admin_words(name)
    for key in (name, stripped):
        did = _ALIAS_TO_ID.get(key)
        if did is None:
            continue
        if key in _ALIAS_LON_MAX and lon > _ALIAS_LON_MAX[key]:
            return None
        state_norm = _norm_name(NCR_DISTRICTS[did]["state"])
        if adm1 is not None and state_norm not in adm1 and adm1 not in state_norm:
            continue
        return did
    in_delhi_box = _in_bbox(lon, lat, DELHI_BBOX)
    if adm1 is not None:
        return 0 if (adm1 in _DELHI_ADM1_KEYS and in_delhi_box) else None
    return 0 if (stripped in _DELHI_NAME_KEYS and in_delhi_box) else None


def _gee_candidates(source: str):
    """(server FeatureCollection, client records) of ADM2 units intersecting the NCR sanity box."""
    rect = ee.Geometry.Rectangle(list(NCR_SANITY_BBOX), None, False)
    if source == "geoboundaries":
        fc = ee.FeatureCollection(_GEOBOUNDARIES_ADM2).filter(ee.Filter.eq("shapeGroup", "IND")).filterBounds(rect)
        name_prop, adm1_prop = "shapeName", None
    else:
        fc = ee.FeatureCollection(_GAUL_ADM2).filter(ee.Filter.eq("ADM0_NAME", "India")).filterBounds(rect)
        name_prop, adm1_prop = "ADM2_NAME", "ADM1_NAME"

    def _summarise(feature):
        geom = feature.geometry()
        centroid = geom.centroid(500).coordinates()
        props = {"name": feature.get(name_prop), "lon": centroid.get(0), "lat": centroid.get(1),
                 "area_km2": geom.area(500).divide(1e6)}
        if adm1_prop:
            props["adm1"] = feature.get(adm1_prop)
        return ee.Feature(None, props)

    info = retry(lambda: _gee_getinfo_guarded(fc.map(_summarise)), tries=4, base_delay=3.0, max_delay=60.0)
    records = [{"key": feat["id"], **feat["properties"]} for feat in info.get("features", [])]
    return fc, records


def _gee_hq_hits(fc, dids) -> dict:
    """{did: [system:index of every candidate polygon containing the district's HQ point]} (one server call)."""
    if not dids:
        return {}
    hits = ee.Dictionary({str(did): fc.filterBounds(ee.Geometry.Point(list(NCR_DISTRICTS[did]["hq"])))
                          .aggregate_array("system:index") for did in dids})
    info = retry(lambda: _gee_getinfo_guarded(hits), tries=4, base_delay=3.0, max_delay=60.0)
    return {int(k): [str(v) for v in (vals or [])] for k, vals in (info or {}).items()}


def _resolve_unmatched(missing_ids, hq_hits: dict, key_to_did: dict, records_by_key: dict):
    """Classify canonical districts left unmatched by name, using the polygons that contain their HQ point.

    Returns (new_matches {candidate key: did}, messages [(level, text)]). An unmatched polygon at the HQ is matched by
    location (an alias the table does not know); a polygon already matched to another district means the district is
    merged into that parent in this layer; no polygon at all is an error.
    """
    new, messages = {}, []
    for did in missing_ids:
        name = NCR_DISTRICTS[did]["name"]
        keys = list(hq_hits.get(did, []))
        free = [k for k in keys if k not in key_to_did and k not in new and k in records_by_key]
        taken = [k for k in keys if k in key_to_did or k in new]
        if free:
            key = free[0]
            new[key] = did
            messages.append(("WARNING", f"{did:2d} {name}: matched BY LOCATION to polygon "
                                        f"'{records_by_key[key].get('name')}' containing its HQ (name not in the "
                                        f"alias table - add it to SPEC_v1 §1.1)"))
        elif taken:
            parent = key_to_did.get(taken[0], new.get(taken[0]))
            messages.append(("INFO", f"{did:2d} {name}: merged into parent district "
                                     f"{NCR_DISTRICTS[parent]['name']} in this layer (HQ inside its polygon)"))
        else:
            messages.append(("ERROR", f"{did:2d} {name}: no polygon of this layer contains its HQ - the district's "
                                      "territory may be missing from the ROI"))
    return new, messages


def _gee_match_roi(source: str):
    """Match one boundary source; returns (roi_fc, matched {did: [names]}, total area) or None if unusable."""
    fc, records = _gee_candidates(source)
    records_by_key = {rec["key"]: rec for rec in records}
    key_to_did, matched, area = {}, {}, 0.0
    for rec in records:
        did = _match_district(rec)
        if did is None:
            continue
        key_to_did[rec["key"]] = did
    missing_ids = [d["id"] for d in NCR_DISTRICTS if d["id"] not in set(key_to_did.values())]
    if missing_ids:
        try:
            located, messages = _resolve_unmatched(missing_ids, _gee_hq_hits(fc, missing_ids), key_to_did,
                                                   records_by_key)
        except Exception as exc:  # noqa: BLE001 - diagnostics must not lose an otherwise usable layer
            located, messages = {}, [("WARNING", f"HQ location check failed ({type(exc).__name__}: "
                                                 f"{str(exc)[:160]}); unmatched: "
                                                 f"{[NCR_DISTRICTS[d]['name'] for d in missing_ids]}")]
        key_to_did.update(located)
        for level, text in messages:
            log(f"[{source}] {text}", level)
    for key, did in key_to_did.items():
        matched.setdefault(did, []).append(str(records_by_key[key].get("name")))
        area += float(records_by_key[key].get("area_km2") or 0.0)
    log(f"[{source}] {len(records)} candidates, matched {len(matched)}/25 districts, area {area:,.0f} km2")
    for did in sorted(matched):
        log(f"    {did:2d} {NCR_DISTRICTS[did]['name']:<20} <- {', '.join(sorted(set(matched[did])))}")
    if 0 not in matched or len(matched) < 18:
        log(f"[{source}] too few matches (Delhi matched: {0 in matched}); trying the next boundary source", "WARNING")
        return None
    if not 45_000 <= area <= 65_000:
        log(f"[{source}] matched area {area:,.0f} km2 is outside the expected 45,000-65,000 km2", "WARNING")
    mapping = ee.Dictionary({k: v for k, v in key_to_did.items()})
    roi_fc = (fc.filter(ee.Filter.inList("system:index", list(key_to_did)))
                .map(lambda f: f.set("did", mapping.get(f.get("system:index")))))
    return roi_fc, matched, area


def _round_coords(obj, ndigits: int = 5):
    """Round nested GeoJSON coordinate lists."""
    if isinstance(obj, (list, tuple)):
        if obj and isinstance(obj[0], (int, float)):
            return [round(float(v), ndigits) for v in obj]
        return [_round_coords(v, ndigits) for v in obj]
    return obj


def _gee_boundary_geojson(roi_fc, dids) -> dict:
    """Dissolved, 200 m-simplified district polygons (EPSG:4326) with properties id/name/state."""
    features = []
    for did in sorted(dids):
        meta = NCR_DISTRICTS[did]
        geom = (roi_fc.filter(ee.Filter.eq("did", did)).geometry(100)
                .dissolve(100).simplify(200))
        features.append(ee.Feature(geom, {"id": did, "name": meta["name"], "state": meta["state"]}))
    info = retry(lambda: _gee_getinfo_guarded(ee.FeatureCollection(features)), tries=4, base_delay=3.0, max_delay=60.0)
    out = []
    for feat in info["features"]:
        props = feat["properties"]
        geometry = feat["geometry"]
        geometry = {"type": geometry["type"], "coordinates": _round_coords(geometry["coordinates"])}
        out.append({"type": "Feature", "geometry": geometry,
                    "properties": {"id": int(props["id"]), "name": props["name"], "state": props["state"]}})
    return {"type": "FeatureCollection", "features": out}


def _gee_district_collection(roi_fc, dids):
    """One server-side feature per matched district (dissolved at 100 m) with ``did``, ``name``, ``state``.

    Delhi's revenue districts become a single feature, so ``reduceRegions`` over ``EE_DISTRICTS`` yields exactly one
    row per canonical district. The geometry stays server-side (no coordinates in every request).
    """
    feats = []
    for did in sorted(dids):
        meta = NCR_DISTRICTS[did]
        geom = roi_fc.filter(ee.Filter.eq("did", did)).geometry(100).dissolve(100)
        feats.append(ee.Feature(geom, {"did": int(did), "name": meta["name"], "state": meta["state"]}))
    return ee.FeatureCollection(feats)


def _geojson_lonlat_points(geojson: dict) -> np.ndarray:
    """All vertex (lon, lat) pairs of a FeatureCollection of (Multi)Polygons."""
    pts = []

    def _walk(coords):
        if coords and isinstance(coords[0], (int, float)):
            pts.append(coords[:2])
        else:
            for c in coords:
                _walk(c)

    for feat in geojson["features"]:
        _walk(feat["geometry"]["coordinates"])
    return np.asarray(pts, dtype=np.float64)

## 1.5 Earth Engine: tiled `computePixels` extraction with retry, subdivision and caching
Shared by sections 01 and 03-05. Each tile is an explicit grid request (`crsCode` + `affineTransform` +
`dimensions`) returning a structured NumPy array. Transient errors (HTTP 429/5xx, "Too many concurrent
aggregations", "Earth Engine (memory) capacity exceeded", timeouts) are retried with exponential backoff; size/memory/
time-out errors split the tile into quarters (up to depth 3); once a tile cannot (or, with `allow_split=False`, may
not) be split, such an error is retried at most twice and then **permanent** (a time-out at the maximum depth of a
splittable request keeps the full `GEE_MAX_RETRIES` budget). On the first failure the remaining
tiles are cancelled and in-flight tiles stop retrying; on a clean time-budget stop (`LSTStop`, checked before every
tile) queued tiles are cancelled while the tiles in flight finish and are cached. Per-call statistics (tiles,
cached tiles, splits, seconds) are kept in `_GEE_FETCH_STATS[cache_key]` and shown in the progress log. Every
top-level tile is cached as `.npy` under `CFG.CACHE_DIR/tiles/<cache_key>/`, so interrupted runs resume (also across
Kaggle sessions through the cache import); the cache key contains a hash of the serialised Earth Engine expression, so
any change of dates, QA rules, scale factors or other server-side parameters gets a fresh cache folder.
`gee_tile_px(n_bands, dtype)` returns the largest square tile that respects the 48 MB per-request limit.

In [ ]:
_GEE_CACHE_VERSION = "v2"
_GEE_MAX_SPLIT_DEPTH = 3
_GEE_REQUEST_BYTES_MAX = 50_331_648          # 48 MB uncompressed per computePixels request
_GEE_SPLIT_RE = re.compile(r"user memory limit exceeded|computation timed out|too large|request size|"
                           r"response size|out of memory|memory limit", re.IGNORECASE)
_GEE_TIMEOUT_RE = re.compile(r"timed out|deadline", re.IGNORECASE)
_GEE_TRANSIENT_RE = re.compile(r"\b(429|500|502|503|504)\b|too many requests|too many concurrent aggregations|"
                               r"capacity exceeded|"
                               r"quota|rate limit|internal error|service unavailable|backend error|deadline|"
                               r"temporar|connection|reset by peer|timed out|remote end closed|broken pipe|ssl",
                               re.IGNORECASE)


_GEE_RESTRICTED_RE = re.compile(r"restricted mode", re.IGNORECASE)
GEE_RESTRICTED_REASON = (
    "Earth Engine quota used up: the project is in Restricted Mode (its monthly EECU-hour quota is exhausted; the "
    "free Community tier has 150 EECU-hours per month, and a full run needs several hundred). Switch the project to "
    "the Contributor tier (1,000 EECU-hours per month; needs a linked billing account, noncommercial use is not "
    "charged; takes effect at once) at https://console.cloud.google.com/earth-engine/configuration/manage-tier , or "
    "wait until the quota resets on the 1st of the month (00:00 Pacific time). Everything downloaded so far is "
    "cached: run again to resume.")


def _gee_raise_if_restricted(exc: BaseException) -> None:
    """Stop the run cleanly (``request_stop`` + ``LSTStop``) when ``exc`` says the project is in Earth Engine
    Restricted Mode: the monthly quota is gone, so retrying cannot help, and a clean stop keeps (and packs) every
    tile downloaded so far for the resume."""
    if not _GEE_RESTRICTED_RE.search(str(exc)):
        return
    stop, stop_cls = globals().get("request_stop"), globals().get("LSTStop")
    if callable(stop):
        stop(GEE_RESTRICTED_REASON)
    if isinstance(stop_cls, type) and issubclass(stop_cls, BaseException):
        raise stop_cls(GEE_RESTRICTED_REASON) from exc
    raise RuntimeError(GEE_RESTRICTED_REASON) from exc


def _gee_getinfo_guarded(obj):
    """``obj.getInfo()``, turning Restricted Mode into a clean stop (for the plain ``retry`` calls of this section)."""
    try:
        return obj.getInfo()
    except Exception as exc:  # noqa: BLE001 - re-raised unless it is Restricted Mode
        _gee_raise_if_restricted(exc)
        raise


class _GEETransientError(RuntimeError):
    """Retryable Earth Engine failure (quota, 5xx, transient timeouts)."""


class _GEESplitError(RuntimeError):
    """Request too large/slow for one call: subdivide the tile."""


class _GEEAborted(RuntimeError):
    """Another tile of the same request failed permanently: stop retrying this one."""


def gee_tile_px(n_bands: int, dtype=np.float32, cap: int = 2048, safety: float = 0.8) -> int:
    """Largest square tile side (multiple of 16 px) whose ``n_bands × dtype`` payload stays under the 48 MB limit."""
    bytes_per_px = max(1, int(n_bands)) * np.dtype(dtype).itemsize
    side = int(math.sqrt(_GEE_REQUEST_BYTES_MAX * safety / bytes_per_px))
    return int(max(16, min(int(cap), side // 16 * 16)))


def _gee_expression_digest(image) -> str:
    """SHA-1 of the serialised Earth Engine expression (captures every server-side parameter).

    Falls back to a random token (cache effectively disabled for that request) if serialisation fails, so a stale tile
    can never be reused under an incomplete key.
    """
    import uuid

    try:
        text = image.serialize()
    except Exception as exc:  # noqa: BLE001
        log(f"could not serialise the Earth Engine expression ({type(exc).__name__}); tile cache disabled for "
            "this request", "WARNING")
        text = uuid.uuid4().hex
    return hashlib.sha1(str(text).encode("utf-8")).hexdigest()


def _gee_cache_key(name: str, grid: GridSpec, extra: Any = None, image=None) -> str:
    """Stable cache folder name tied to the grid geometry, the request description and (when ``image`` is given) the
    hash of the serialised expression."""
    expr = _gee_expression_digest(image) if image is not None else None
    payload = json.dumps([_GEE_CACHE_VERSION, asdict(grid), extra, expr], sort_keys=True, default=str)
    digest = hashlib.sha1(payload.encode()).hexdigest()[:10]
    return f"{name}_{grid.res}m_{digest}"


gee_cache_key = _gee_cache_key     # public name for sections 03-05


def _structured_to_stack(raw, bands: Sequence[str], tile: GridSpec, dtype) -> np.ndarray:
    """computePixels NUMPY_NDARRAY result (structured, one field per band) -> (bands, h, w) array."""
    raw = np.asarray(raw)
    if raw.dtype.names:
        missing = [b for b in bands if b not in raw.dtype.names]
        if missing:
            raise RuntimeError(f"computePixels result lacks bands {missing}; got {raw.dtype.names}")
        stack = np.stack([np.asarray(raw[b]) for b in bands])
    elif raw.ndim == 3:
        stack = np.moveaxis(raw, -1, 0)
    else:
        stack = raw[None]
    if stack.shape[1:] != (tile.height, tile.width):
        raise RuntimeError(f"computePixels returned shape {stack.shape[1:]}, expected {(tile.height, tile.width)}")
    return stack.astype(dtype, copy=False)


def _classify_gee_error(exc: BaseException, allow_split: bool) -> BaseException:
    """Map an Earth Engine exception to split / transient / permanent (returns the exception to raise).

    * size/memory errors split the tile while it can be split; at the maximum depth (``_fetch_tile``) they get two
      tries and are then permanent (an identical request cannot suddenly fit the per-user memory limit) - except
      timeouts, which stay retryable with the full ``GEE_MAX_RETRIES`` budget; with ``allow_split=False`` both get
      the two-try limit;
    * load-related errors (429/5xx, concurrent aggregations, capacity exceeded, timeouts, connection resets) are
      transient and retried with backoff;
    * everything else (bad asset, auth, invalid argument) is permanent;
    * Restricted Mode (monthly quota used up) raises a clean stop at once (``_gee_raise_if_restricted``).
    """
    _gee_raise_if_restricted(exc)
    message = str(exc)
    if _GEE_SPLIT_RE.search(message):
        if allow_split:
            return _GEESplitError(message)
        if _GEE_TIMEOUT_RE.search(message):
            return _GEETransientError(message)
        return RuntimeError(f"Earth Engine request still too large at the maximum split depth: {message}")
    if _GEE_TRANSIENT_RE.search(message):
        return _GEETransientError(message)
    return exc


def _compute_pixels(image, tile: GridSpec, bands: Sequence[str], dtype, allow_split: bool) -> np.ndarray:
    """One computePixels call, translating EE errors into split / transient / permanent failures."""
    request = {
        "expression": image,
        "fileFormat": "NUMPY_NDARRAY",
        "bandIds": list(bands),
        "grid": {"dimensions": {"width": int(tile.width), "height": int(tile.height)},
                 "affineTransform": tile.affine(), "crsCode": tile.crs},
    }
    try:
        raw = ee.data.computePixels(request)
    except Exception as exc:  # noqa: BLE001 - classified below
        raise _classify_gee_error(exc, allow_split) from exc
    return _structured_to_stack(raw, bands, tile, dtype)


class _GEENoSplitError(_GEETransientError):
    """Size/time-out error of a request that may not be split: retried at most ``_GEE_NOSPLIT_TRIES`` times."""


_GEE_NOSPLIT_TRIES = 2
_GEE_FETCH_STATS: dict = {}       # cache_key -> {tiles, cached, splits, seconds, fetched}
_GEE_STATS_LOCK = threading.Lock()


def _fetch_tile(image, tile: GridSpec, bands: Sequence[str], dtype, depth: int = 0,
                abort: Optional[threading.Event] = None, allow_split: bool = True,
                stats: Optional[dict] = None) -> np.ndarray:
    """Fetch one tile with backoff; quarter it recursively on size/memory errors; stop when ``abort`` is set.

    With ``allow_split=False`` a size or time-out error is retried at most twice in total and then raised as a
    permanent error, instead of being retried ``GEE_MAX_RETRIES`` times. At the maximum split depth (with
    ``allow_split=True``) a size/memory error gets the same two tries, while a time-out stays a load-related transient
    error and keeps the full ``GEE_MAX_RETRIES`` budget (an 8 km tile that times out under load usually succeeds later).
    ``stats`` (optional dict) counts the splits.
    """
    can_split = bool(allow_split) and depth < _GEE_MAX_SPLIT_DEPTH and min(tile.width, tile.height) >= 2
    size_failures = [0]

    def _once():
        try:
            return _compute_pixels(image, tile, bands, dtype, True)
        except _GEESplitError as exc:
            if can_split:
                raise
            if allow_split and _GEE_TIMEOUT_RE.search(str(exc)):
                raise _GEETransientError(str(exc)) from exc   # max depth: time-outs keep GEE_MAX_RETRIES
            raise _GEENoSplitError(str(exc)) from exc

    def _on_retry(attempt, exc, delay):
        if abort is not None and abort.is_set():
            raise _GEEAborted("another tile failed; not retrying") from exc
        if isinstance(exc, _GEENoSplitError):
            size_failures[0] += 1
            if size_failures[0] >= _GEE_NOSPLIT_TRIES:
                raise RuntimeError(f"Earth Engine request for a {tile.width}x{tile.height} tile failed "
                                   f"{size_failures[0]} times with a size/time-out error and may not be split "
                                   f"further: {str(exc)[:300]}") from exc
        log(f"computePixels retry {attempt} for {tile.width}x{tile.height} tile in {delay:.1f}s: {str(exc)[:160]}",
            "WARNING")

    if abort is not None and abort.is_set():
        raise _GEEAborted("another tile failed; not fetching")
    try:
        return retry(_once, tries=CFG.GEE_MAX_RETRIES, base_delay=2.0, max_delay=120.0,
                     retry_on=(_GEETransientError,), on_retry=_on_retry)
    except _GEESplitError as exc:
        log(f"splitting {tile.width}x{tile.height} tile (depth {depth + 1}): {str(exc)[:160]}", "WARNING")
        if stats is not None:
            with _GEE_STATS_LOCK:
                stats["splits"] = stats.get("splits", 0) + 1
    h1, w1 = tile.height // 2, tile.width // 2
    out = np.empty((len(bands), tile.height, tile.width), dtype=dtype)
    for r0, h in ((0, h1), (h1, tile.height - h1)):
        for c0, w in ((0, w1), (w1, tile.width - w1)):
            out[:, r0:r0 + h, c0:c0 + w] = _fetch_tile(image, tile.tile(r0, c0, h, w), bands, dtype, depth + 1,
                                                       abort, allow_split, stats)
    return out


def _save_npy_atomic(path: Path, array: np.ndarray) -> None:
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "wb") as fh:
        np.save(fh, array)
    os.replace(tmp, path)


def fetch_grid(image, grid: GridSpec, bands: Sequence[str], tile_px: int, cache_key: str,
               dtype=np.float32, allow_split: bool = True, stats: Optional[dict] = None) -> np.ndarray:
    """Download ``bands`` of an ee.Image on ``grid`` as a (len(bands), H, W) array.

    Tiles of ``tile_px`` × ``tile_px`` are fetched concurrently (``CFG.GEE_WORKERS`` threads) with retry, recursive
    subdivision (``allow_split``) and an on-disk cache (also restored from ``cache/packed/*.tar``). For float dtypes
    the ``NODATA_FLOAT`` sentinel is turned into NaN; integer outputs are returned as delivered.

    Per-call statistics {tiles, cached, splits, fetched, seconds} are kept in ``_GEE_FETCH_STATS[cache_key]`` and
    copied into ``stats`` when given. Before every tile the run's time budget is checked (expected duration 1.25 ×
    the slowest tile so far); on a clean stop (``LSTStop``) queued tiles are cancelled, the tiles in flight finish
    and are cached, and ``LSTStop`` is re-raised. On any other failure the remaining tiles are cancelled and
    in-flight tiles stop retrying.
    """
    if ee is None:
        raise RuntimeError("earthengine-api is not available")
    bands, dtype = list(bands), np.dtype(dtype)
    limit = gee_tile_px(len(bands), dtype, cap=1 << 15, safety=1.0)
    if tile_px > limit:
        log(f"{cache_key}: tile_px {tile_px} exceeds the 48 MB request limit for {len(bands)} {dtype} band(s); "
            f"using {limit}", "WARNING")
        tile_px = limit
    abort = threading.Event()
    cache_dir = CFG.CACHE_DIR / "tiles" / cache_key
    cache_dir.mkdir(parents=True, exist_ok=True)
    tiles = [(r0, c0, min(tile_px, grid.height - r0), min(tile_px, grid.width - c0))
             for r0 in range(0, grid.height, tile_px) for c0 in range(0, grid.width, tile_px)]
    out = np.empty((len(bands), grid.height, grid.width), dtype=dtype)
    call_stats = {"tiles": len(tiles), "cached": 0, "splits": 0, "fetched": 0, "seconds": 0.0, "tile_px": tile_px}
    with _GEE_STATS_LOCK:
        _GEE_FETCH_STATS[cache_key] = call_stats
    slowest = [0.0]
    t_start = time.perf_counter()

    def _job(spec):
        r0, c0, h, w = spec
        path = cache_dir / f"r{r0:06d}_c{c0:06d}_{h}x{w}.npy"
        if not path.exists():
            cache_restore(path)
        if path.exists():
            try:
                cached = np.load(path)
                if cached.shape == (len(bands), h, w):
                    return spec, cached.astype(dtype, copy=False), True
            except (OSError, ValueError) as exc:
                log(f"ignoring unreadable cache tile {path.name}: {exc}", "WARNING")
        check_budget(f"Earth Engine tile {cache_key}", 1.25 * slowest[0])
        t0 = time.perf_counter()
        arr = _fetch_tile(image, grid.tile(r0, c0, h, w), bands, dtype, abort=abort, allow_split=allow_split,
                          stats=call_stats)
        _save_npy_atomic(path, arr)
        with _GEE_STATS_LOCK:
            slowest[0] = max(slowest[0], time.perf_counter() - t0)
            call_stats["fetched"] += 1
        return spec, arr, False

    n_done, step = 0, max(1, len(tiles) // 10)
    pool = ThreadPoolExecutor(max_workers=max(1, min(CFG.GEE_WORKERS, len(tiles))), thread_name_prefix="ee")
    futures = [pool.submit(_job, t) for t in tiles]
    try:
        for future in as_completed(futures):
            (r0, c0, h, w), arr, cached = future.result()
            out[:, r0:r0 + h, c0:c0 + w] = arr
            n_done += 1
            with _GEE_STATS_LOCK:
                call_stats["cached"] += int(cached)
                call_stats["seconds"] = round(time.perf_counter() - t_start, 1)
            if n_done % step == 0 or n_done == len(tiles):
                log(f"  {cache_key}: {n_done}/{len(tiles)} tiles ({call_stats['cached']} from cache, "
                    f"{call_stats['splits']} split(s), {call_stats['seconds']:.0f}s)")
    except LSTStop:
        # Clean stop: no new tile starts and in-flight tiles stop retrying; a request already in flight finishes and
        # is cached for the resumed run.
        abort.set()
        pool.shutdown(wait=True, cancel_futures=True)
        raise
    except BaseException:
        # Fail fast: cancel queued tiles, tell in-flight ones to stop retrying, and do not wait for them.
        abort.set()
        pool.shutdown(wait=False, cancel_futures=True)
        raise
    pool.shutdown(wait=True)
    call_stats["seconds"] = round(time.perf_counter() - t_start, 1)
    if stats is not None:
        stats.update(call_stats)
    if dtype.kind == "f":
        out[out == NODATA_FLOAT] = np.nan
    return out


def _grid_ee_geom(grid: GridSpec):
    """Bounding polygon of a GridSpec in EPSG:4326 as an ee.Geometry (planar edges, densified)."""
    ring_xy = [(grid.x0, grid.y0), (grid.x0 + grid.width * grid.res, grid.y0),
               (grid.x0 + grid.width * grid.res, grid.y0 - grid.height * grid.res),
               (grid.x0, grid.y0 - grid.height * grid.res), (grid.x0, grid.y0)]
    pts = _densify_ring(ring_xy, 8)
    lons, lats = _grid_xy_to_lonlat(pts[:, 0], pts[:, 1], grid.crs)
    ring = [[float(lo), float(la)] for lo, la in zip(lons, lats)]
    return ee.Geometry.Polygon([ring], None, False)


grid_ee_geom = _grid_ee_geom       # public name for sections 03-05

## 1.6 ROI, districts and the analysis grid
* **Earth Engine:** geoBoundaries ADM2 → FAO GAUL level 2 → approximate outline. The grid is snapped to the
  dissolved district polygons (+1 cell margin) and the district raster is painted server-side (`paint` of `did`)
  on exactly that grid, so it is pixel-identical to everything extracted later. `EE_DISTRICTS` holds one dissolved
  feature per district (`did`, `name`, `state`) for `reduceRegions`.
* **Synthetic / last resort:** the approximate NCR outline (SPEC_v1 §1.2), partitioned into the 25 districts by
  nearest headquarters — a Voronoi partition in UTM metres, identical for the raster (cell-centre nearest HQ) and the
  GeoJSON (Voronoi polygons clipped to the outline, simplified to 200 m).

In [ ]:
def _lonlat_array_to_xy(points) -> np.ndarray:
    pts = np.asarray(points, dtype=np.float64)
    x, y = _grid_lonlat_to_xy(pts[:, 0], pts[:, 1], CFG.CRS)
    return np.column_stack([x, y])


def _convex_hull(points: np.ndarray) -> np.ndarray:
    """Andrew's monotone chain convex hull (closed ring) - fallback when shapely is unavailable."""
    pts = np.unique(points, axis=0)
    if len(pts) < 3:
        return np.vstack([pts, pts[:1]])

    def _turn(o, a, b):  # z-component of (a - o) x (b - o): > 0 for a counter-clockwise turn
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    def _half(seq):
        hull = []
        for p in seq:
            while len(hull) >= 2 and _turn(hull[-2], hull[-1], p) <= 0:
                hull.pop()
            hull.append(p)
        return hull

    lower, upper = _half(pts), _half(pts[::-1])
    ring = np.array(lower[:-1] + upper[:-1])
    return np.vstack([ring, ring[:1]])


def _xy_ring_to_lonlat(ring_xy: np.ndarray) -> list:
    lon, lat = _grid_xy_to_lonlat(ring_xy[:, 0], ring_xy[:, 1], CFG.CRS)
    return [[round(float(a), 5), round(float(b), 5)] for a, b in zip(lon, lat)]


def _voronoi_boundary_geojson(hq_xy: np.ndarray, outline_xy: np.ndarray, district_idx, grid: GridSpec) -> dict:
    """District polygons = Voronoi cells of the HQs clipped to the outline, simplified to 200 m (EPSG:4326)."""
    features = []
    if shapely is not None:
        from shapely.geometry import MultiPoint, Point, Polygon
        from shapely.ops import voronoi_diagram

        outline = Polygon(outline_xy).buffer(0)
        cells = voronoi_diagram(MultiPoint([tuple(p) for p in hq_xy]), envelope=outline.buffer(100_000))
        for d, hq in zip(NCR_DISTRICTS, hq_xy):
            region = next(c for c in cells.geoms if c.covers(Point(hq)))
            geom = region.intersection(outline).simplify(200, preserve_topology=True)
            polys = [geom] if geom.geom_type == "Polygon" else [g for g in getattr(geom, "geoms", []) if g.geom_type == "Polygon"]
            coords = [[_xy_ring_to_lonlat(np.asarray(p.exterior.coords))]
                      + [_xy_ring_to_lonlat(np.asarray(r.coords)) for r in p.interiors] for p in polys]
            geometry = ({"type": "Polygon", "coordinates": coords[0]} if len(coords) == 1
                        else {"type": "MultiPolygon", "coordinates": coords})
            features.append({"type": "Feature", "geometry": geometry,
                             "properties": {"id": d["id"], "name": d["name"], "state": d["state"]}})
        return {"type": "FeatureCollection", "features": features}
    log("shapely unavailable: district outlines approximated by convex hulls of their cells", "WARNING")
    x, y = grid.cell_centers_xy()
    for d in NCR_DISTRICTS:
        sel = district_idx == d["id"]
        ring = _convex_hull(np.column_stack([x[sel], y[sel]])) if sel.any() else np.repeat(hq_xy[d["id"]][None], 4, 0)
        features.append({"type": "Feature", "geometry": {"type": "Polygon", "coordinates": [_xy_ring_to_lonlat(ring)]},
                         "properties": {"id": d["id"], "name": d["name"], "state": d["state"]}})
    return {"type": "FeatureCollection", "features": features}


def _outline_districts(grid: GridSpec):
    """ROI mask, nearest-HQ district raster and matching Voronoi GeoJSON for the approximate NCR outline."""
    from matplotlib.path import Path as _MplPath

    x, y = grid.cell_centers_xy()
    outline_xy = _lonlat_array_to_xy(_densify_ring(NCR_OUTLINE, 10))
    roi_mask = _MplPath(outline_xy).contains_points(np.column_stack([x.ravel(), y.ravel()])).reshape(grid.shape)
    hq_xy = _lonlat_array_to_xy([d["hq"] for d in NCR_DISTRICTS])
    district_idx = np.full(grid.shape, NODATA_DISTRICT, dtype=np.int16)
    # nearest HQ per cell, row-chunked so the (rows, W, 25) distance block stays small at fine resolutions
    for r0 in range(0, grid.height, 256):
        xs, ys = x[r0:r0 + 256, :, None], y[r0:r0 + 256, :, None]
        d2 = (xs - hq_xy[:, 0]) ** 2 + (ys - hq_xy[:, 1]) ** 2
        district_idx[r0:r0 + 256] = np.argmin(d2, axis=-1).astype(np.int16)
    district_idx[~roi_mask] = NODATA_DISTRICT
    boundary = _voronoi_boundary_geojson(hq_xy, outline_xy, district_idx, grid)
    return roi_mask, district_idx, boundary


def _geojson_to_ee_districts(geojson: dict):
    """ee.FeatureCollection (``did``, ``name``, ``state``) from a district GeoJSON (outline fallback in gee mode)."""
    feats = []
    for feat in geojson["features"]:
        props = feat["properties"]
        geom = ee.Geometry(feat["geometry"], None, False)
        feats.append(ee.Feature(geom, {"did": int(props["id"]), "name": props["name"], "state": props["state"]}))
    return ee.FeatureCollection(feats)


def _gee_build_roi(res: int) -> dict:
    """Try geoBoundaries, then GAUL, then the NCR outline. Returns a dict describing the ROI."""
    for source in ("geoboundaries", "gaul"):
        try:
            matched = _gee_match_roi(source)
        except Exception as exc:  # noqa: BLE001 - try the next source
            log(f"[{source}] boundary query failed: {type(exc).__name__}: {str(exc)[:200]}", "WARNING")
            continue
        if matched is None:
            continue
        roi_fc, districts, area = matched
        boundary = _gee_boundary_geojson(roi_fc, districts.keys())
        grid = GridSpec.from_lonlat_points(_geojson_lonlat_points(boundary), CFG.CRS, res)
        asset = _GEOBOUNDARIES_ADM2 if source == "geoboundaries" else _GAUL_ADM2
        return {"source": asset, "fc": roi_fc, "geom": roi_fc.geometry(), "boundary": boundary,
                "grid": grid, "area_km2": area, "local_districts": None,
                "ee_districts": _gee_district_collection(roi_fc, districts.keys())}
    log("No administrative boundary source usable; falling back to the approximate NCR outline + "
        "nearest-HQ (Voronoi) districts", "WARNING")
    grid = _outline_grid(res)
    roi_mask, district_idx, boundary = _outline_districts(grid)
    geom = ee.Geometry.Polygon([[list(p) for p in NCR_OUTLINE]], None, False)
    return {"source": "approximate NCR outline (SPEC_v1 §1.2) with Voronoi districts", "fc": None, "geom": geom,
            "boundary": boundary, "grid": grid, "area_km2": float(roi_mask.sum() * grid.cell_area_km2),
            "local_districts": (roi_mask, district_idx), "ee_districts": _geojson_to_ee_districts(boundary)}


def acquire_gee_study_area() -> dict:
    """Earth Engine study area: grid, ROI mask, district raster, boundary GeoJSON and EE handles."""
    res = CFG.GRID_RES_M
    roi = _gee_build_roi(res)
    grid = roi["grid"]
    log(f"GEE grid: {grid.width}x{grid.height} cells @ {res} m, origin ({grid.x0:.0f}, {grid.y0:.0f}) {grid.crs}")
    ref = _S01_V1_REAL_GRID
    if res == ref["res"] and roi["local_districts"] is None and (
            grid.width, grid.height, grid.x0, grid.y0) != (ref["width"], ref["height"], ref["x0"], ref["y0"]):
        log(f"grid differs from the v1 real-data grid ({ref['width']}x{ref['height']} at x0 {ref['x0']:.0f}, "
            f"y_top {ref['y0']:.0f}); the boundary layer may have changed", "WARNING")
    if roi["local_districts"] is None:
        paint = ee.Image.constant(NODATA_DISTRICT).toInt16().paint(roi["fc"], "did").toInt16().rename("did")
        district_idx = fetch_grid(paint, grid, ["did"], gee_tile_px(1, np.int16, cap=1024),
                                  _gee_cache_key("districts", grid, roi["source"], image=paint), np.int16)[0]
        district_idx = district_idx.astype(np.int16)
        district_idx[(district_idx < 0) | (district_idx >= len(NCR_DISTRICTS))] = NODATA_DISTRICT
        roi_mask = district_idx >= 0
    else:
        roi_mask, district_idx = roi["local_districts"]
    log(f"ROI: {int(roi_mask.sum()):,} cells = {roi_mask.sum() * grid.cell_area_km2:,.0f} km2 ({roi['source']})")
    return {"GRID": grid, "ROI_MASK": roi_mask, "DISTRICT_IDX": district_idx.astype(np.int16),
            "BOUNDARY_GEOJSON": roi["boundary"], "ROI_SOURCE": roi["source"],
            "BOUNDARY_APPROXIMATE": roi["local_districts"] is not None,
            "EE_DISTRICTS": roi["ee_districts"], "EE_ROI": roi["geom"]}


def synthetic_study_area() -> dict:
    """Synthetic study area: outline grid, nearest-HQ districts (no Earth Engine)."""
    grid = _outline_grid(CFG.GRID_RES_M)
    roi_mask, district_idx, boundary = _outline_districts(grid)
    return {"GRID": grid, "ROI_MASK": roi_mask, "DISTRICT_IDX": district_idx, "BOUNDARY_GEOJSON": boundary,
            "ROI_SOURCE": "approximate NCR outline (SPEC_v1 §1.2) with Voronoi districts [synthetic]",
            "BOUNDARY_APPROXIMATE": True, "EE_DISTRICTS": None, "EE_ROI": None}


_S01_AREA = None
if DATA_MODE == "gee":
    try:
        with timer("01 GEE study area"):
            _S01_AREA = acquire_gee_study_area()
    except Exception as _exc:  # noqa: BLE001 - auto mode without a key must not crash
        if CFG.RUN_MODE == "gee" or _S01_KEY_SUPPLIED:
            if CFG.RUN_MODE != "gee":
                log("Earth Engine study-area construction failed although a service-account key was supplied; "
                    "stopping instead of falling back to synthetic data (set LST_RUN_MODE=synthetic to force the "
                    "synthetic twin)", "ERROR")
            raise
        log(f"Earth Engine study-area construction failed: {type(_exc).__name__}: {_exc}", "ERROR")
        DATA_MODE = "synthetic"
        CFG.resolve_mode(DATA_MODE)
        _MODE_REASON = f"Earth Engine study area failed: {type(_exc).__name__}: {str(_exc)[:300]}"
        _print_synthetic_banner(_MODE_REASON)
        _S01_AREA = None
if DATA_MODE == "synthetic":
    with timer("01 synthetic study area"):
        _S01_AREA = synthetic_study_area()

## 1.7 Interface globals: grids, masks, canonical cell order, provenance and caveats
`ROI_MASK_FINE` marks a fine pixel as inside the ROI iff its parent 1 km cell is (so fine aggregates and 1 km cells
always describe the same territory). The canonical order is row-major over ROI cells, i.e. ascending `cell_id`.

In [ ]:
GRID = _S01_AREA["GRID"]
ROI_MASK = np.ascontiguousarray(_S01_AREA["ROI_MASK"], dtype=bool)
DISTRICT_IDX = np.ascontiguousarray(_S01_AREA["DISTRICT_IDX"], dtype=np.int16)
DISTRICT_IDX[~ROI_MASK] = NODATA_DISTRICT
BOUNDARY_GEOJSON = _S01_AREA["BOUNDARY_GEOJSON"]
BOUNDARY_APPROXIMATE = bool(_S01_AREA["BOUNDARY_APPROXIMATE"])
EE_DISTRICTS = _S01_AREA["EE_DISTRICTS"] if DATA_MODE == "gee" else None
# Do not use EE_ROI: fc.geometry() of this collection evaluates to an empty MultiPoint on real data (2026-11), so any
# filterBounds or reduction over it silently selects nothing. Use gee_roi_geometry() (section 03) or EE_DISTRICTS.
EE_ROI = _S01_AREA["EE_ROI"] if DATA_MODE == "gee" else None
DATA_MODE_REASON = str(_MODE_REASON)
_S01_ROI_SOURCE = _S01_AREA["ROI_SOURCE"]

FINE_FACTOR = int(CFG.FINE_FACTOR)
GRID_FINE = GRID.fine(CFG.grid_fine_res())
ROI_MASK_FINE = np.ascontiguousarray(upsample_repeat(ROI_MASK, FINE_FACTOR), dtype=bool)

_s01_flat = np.flatnonzero(ROI_MASK.ravel())                 # ascending = canonical order (cell_id = row*W + col)
if _s01_flat.size and _s01_flat.max() > np.iinfo(np.int32).max:
    raise OverflowError("cell ids exceed int32 - grid too large")
N_CELLS = int(_s01_flat.size)
CELL_IDS = _s01_flat.astype(np.int32)
CELL_ROWS = (_s01_flat // GRID.width).astype(np.int32)
CELL_COLS = (_s01_flat % GRID.width).astype(np.int32)
_s01_lon, _s01_lat = GRID.lonlat()
CELL_LON = np.asarray(_s01_lon, dtype=np.float64)[CELL_ROWS, CELL_COLS]
CELL_LAT = np.asarray(_s01_lat, dtype=np.float64)[CELL_ROWS, CELL_COLS]
del _s01_flat, _s01_lon, _s01_lat

# Provenance: variable -> {year: text}. Later sections add one entry per variable and frame.
DATA_SOURCES = {"roi": {int(y): _S01_ROI_SOURCE for y in CFG.FRAMES_OBS}}

# Static caveats for the manifest notes (SPEC §2: acquisition sections append frame-specific ones).
DATA_CAVEATS = [
    "Day LST of the 1990 and 1995 frames comes from Landsat 5 alone: some WRS-2 path/rows have fewer than 6 clear "
    "pre-monsoon scenes in the window, so these frames are tiered 'observed-sparse' and are noisier.",
    "The 2005 day frame includes Landsat 7 ETM+ scenes after the 2003 scan-line-corrector failure (SLC-off): "
    "striped gaps are filled only by other scenes of the window.",
    "Day LST is a ~10:30 IST snapshot; frames are corrected to a common 10:45 IST overpass with an ERA5-Land "
    "skin-temperature heating rate. The correction (typically a few tenths of a degree, more for early Landsat 5 "
    "scenes) carries its own uncertainty, included in the level intervals.",
    "Night LST is in Aqua-equivalent units: the 2025 frame is VIIRS and the 2000 frame Terra (22:30), both "
    "calibrated to Aqua on overlap years; 1990 and 1995 night frames are model hindcasts, not observations.",
    "ERA5-Land climate covariates have ~9-11 km native resolution (bilinearly resampled to the grid) and known "
    "warm/dry biases over irrigated Indo-Gangetic cropland.",
    "Land cover is GLC_FCS30D for every frame (five-yearly maps before 2000, ~81 % global accuracy); the 2025 frame "
    "uses its 2022 map.",
    "GHSL built-up and population for 2025 and 2030 are model extrapolations (GHSL P2023A), and GHSL accuracy "
    "before 2000 is lower (Landsat back-casting).",
    "Projections 2030-2050 are scenario projections, not forecasts: they combine CMIP6 temperature deltas scaled "
    "by an empirical dLST/dTair sensitivity with simulated land change, with Monte-Carlo uncertainty.",
    "SHAP values explain the model's predictions, not causal effects; thresholds and recommendations are model-based "
    "what-if estimates.",
]
if BOUNDARY_APPROXIMATE:
    DATA_CAVEATS.append(
        "District boundaries are APPROXIMATE (nearest-HQ Voronoi partition of the SPEC_v1 §1.2 outline): district "
        "names and per-district statistics describe approximate areas.")
if _S01_ASSET_CAVEAT:
    DATA_CAVEATS.append(_S01_ASSET_CAVEAT)
if CFG.PROFILE == "pilot":
    DATA_CAVEATS.append(f"PILOT RUN: only the {', '.join(str(y) for y in CFG.FRAMES_OBS)} frames and no projections "
                        "(Earth Engine cost probe); not a complete analysis.")
if DATA_MODE == "synthetic":
    DATA_CAVEATS.append("SYNTHETIC DATA: every variable is simulated from planted responses; see SYNTHETIC_TRUTH. "
                        "No result describes the real Delhi NCR.")
for _caveat in DATA_CAVEATS:
    log(f"caveat: {_caveat}")
del _S01_AREA
log(f"GRID {GRID.width}x{GRID.height} @ {GRID.res} m | GRID_FINE {GRID_FINE.width}x{GRID_FINE.height} @ "
    f"{GRID_FINE.res} m | ROI {N_CELLS:,} cells ({N_CELLS * GRID.cell_area_km2:,.0f} km2) | "
    f"land cover {CFG.fine_res()} m | mode {DATA_MODE}")

## 1.8 Contract check
Verifies the section-1 interface (SPEC §3.1) so downstream sections can rely on shapes, dtypes and order.

In [ ]:
def _validate_study_area() -> None:
    """Raise AssertionError with a precise message if any section-1 global violates the contract."""
    h, w = GRID.shape
    f = FINE_FACTOR
    assert DATA_MODE in ("gee", "synthetic"), DATA_MODE
    assert CFG.DATA_MODE == DATA_MODE, (CFG.DATA_MODE, DATA_MODE)
    assert GRID.res == CFG.GRID_RES_M and GRID.crs == CFG.CRS, (GRID.res, GRID.crs)
    assert GRID.x0 % GRID.res == 0 and GRID.y0 % GRID.res == 0, "grid edges must be multiples of the cell size"
    assert (GRID_FINE.width, GRID_FINE.height) == (w * f, h * f) and GRID_FINE.x0 == GRID.x0 \
        and GRID_FINE.y0 == GRID.y0, GRID_FINE
    assert GRID.res % CFG.fine_res() == 0, "LC_FINE_RES_M must divide GRID_RES_M"
    assert ROI_MASK.shape == (h, w) and ROI_MASK.dtype == bool, (ROI_MASK.shape, ROI_MASK.dtype)
    assert ROI_MASK_FINE.shape == (h * f, w * f) and ROI_MASK_FINE.dtype == bool
    assert int(ROI_MASK_FINE.sum()) == N_CELLS * f * f, "ROI_MASK_FINE must cover exactly the ROI cells"
    assert DISTRICT_IDX.shape == (h, w) and DISTRICT_IDX.dtype == np.int16, (DISTRICT_IDX.shape, DISTRICT_IDX.dtype)
    assert np.all(DISTRICT_IDX[~ROI_MASK] == NODATA_DISTRICT), "DISTRICT_IDX must be -1 outside the ROI"
    assert np.all(DISTRICT_IDX[ROI_MASK] >= 0) and DISTRICT_IDX.max() < len(DISTRICTS), "district id out of range"
    assert len(DISTRICTS) == 25 and all(set(d) == {"id", "name", "state"} for d in DISTRICTS)
    assert [d["id"] for d in DISTRICTS] == list(range(25))
    assert BOUNDARY_GEOJSON["type"] == "FeatureCollection" and BOUNDARY_GEOJSON["features"]
    for feat in BOUNDARY_GEOJSON["features"]:
        assert set(feat["properties"]) == {"id", "name", "state"}, feat["properties"]
        assert feat["geometry"]["type"] in ("Polygon", "MultiPolygon"), feat["geometry"]["type"]
    assert N_CELLS > 0 and N_CELLS == int(ROI_MASK.sum())
    for name, arr, dt in (("CELL_IDS", CELL_IDS, np.int32), ("CELL_ROWS", CELL_ROWS, np.int32),
                          ("CELL_COLS", CELL_COLS, np.int32), ("CELL_LON", CELL_LON, np.float64),
                          ("CELL_LAT", CELL_LAT, np.float64)):
        assert arr.shape == (N_CELLS,) and arr.dtype == dt, (name, arr.shape, arr.dtype)
    assert np.all(np.diff(CELL_IDS) > 0), "CELL_IDS must be strictly ascending (canonical order)"
    assert np.array_equal(CELL_IDS, CELL_ROWS * GRID.width + CELL_COLS)
    assert np.all(ROI_MASK[CELL_ROWS, CELL_COLS])
    lon0, lat0, lon1, lat1 = NCR_SANITY_BBOX
    assert np.all((CELL_LON > lon0) & (CELL_LON < lon1) & (CELL_LAT > lat0) & (CELL_LAT < lat1)), \
        "cell centres outside the NCR sanity box"
    assert (EE_DISTRICTS is None) == (DATA_MODE == "synthetic")
    assert isinstance(DATA_SOURCES, dict) and isinstance(DATA_CAVEATS, list)
    assert isinstance(DATA_MODE_REASON, str) and DATA_MODE_REASON


_validate_study_area()
log("section-1 contract check passed")

## 1.9 Study-area figure and district table

In [ ]:
def _s01_save_show(fig, name: str) -> None:
    fig.savefig(CFG.FIG_DIR / name, dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig)


_s01_tag = " [SYNTHETIC]" if DATA_MODE == "synthetic" else ""
_s01_extent = [GRID.x0 / 1000, (GRID.x0 + GRID.width * GRID.res) / 1000,
               (GRID.y0 - GRID.height * GRID.res) / 1000, GRID.y0 / 1000]
_s01_counts = np.bincount(DISTRICT_IDX[ROI_MASK].astype(np.int64), minlength=len(DISTRICTS))
_s01_img = np.where(ROI_MASK, DISTRICT_IDX, np.nan).astype(np.float32)
from matplotlib.colors import BoundaryNorm as _S01Norm, ListedColormap as _S01Cmap

# 25 distinct colours (tab20 + 5 of Dark2), interleaved with a stride coprime to 25 so consecutive ids differ in hue
_s01_palette = list(plt.get_cmap("tab20").colors) + list(plt.get_cmap("Dark2").colors[:5])
_s01_cmap = _S01Cmap([_s01_palette[(7 * i) % 25] for i in range(25)])
fig, ax = plt.subplots(figsize=(8.5, 9.5), constrained_layout=True)
ax.imshow(np.ma.masked_invalid(_s01_img), cmap=_s01_cmap, norm=_S01Norm(np.arange(-0.5, 25.5), 25),
          extent=_s01_extent, interpolation="nearest")
for _d in DISTRICTS:
    _sel = (CELL_ROWS[DISTRICT_IDX[CELL_ROWS, CELL_COLS] == _d["id"]],
            CELL_COLS[DISTRICT_IDX[CELL_ROWS, CELL_COLS] == _d["id"]])
    if _sel[0].size:
        _xc = GRID.x0 / 1000 + (np.median(_sel[1]) + 0.5) * GRID.res / 1000
        _yc = GRID.y0 / 1000 - (np.median(_sel[0]) + 0.5) * GRID.res / 1000
        ax.text(_xc, _yc, f"{_d['id']}", ha="center", va="center", fontsize=8, color="white",
                bbox={"boxstyle": "round,pad=0.15", "fc": "black", "alpha": 0.55, "lw": 0})
ax.set_title(f"Delhi NCR study area: {len(DISTRICTS)} districts, {N_CELLS:,} cells @ {GRID.res} m "
             f"(fine grid {GRID_FINE.res} m){_s01_tag}", fontsize=10)
ax.set_xlabel("UTM 43N easting (km)")
ax.set_ylabel("UTM 43N northing (km)")
_s01_save_show(fig, "01_study_area.png")

_S01_DISTRICT_TABLE = pd.DataFrame({
    "id": [d["id"] for d in DISTRICTS], "district": [d["name"] for d in DISTRICTS],
    "state": [d["state"] for d in DISTRICTS], "cells": _s01_counts.astype(int),
    "area_km2": np.round(_s01_counts * GRID.cell_area_km2, 0)}).set_index("id")
print(f"Districts on the analysis grid{_s01_tag} (ROI source: {_S01_ROI_SOURCE})")
_show(_S01_DISTRICT_TABLE)
if (_s01_counts == 0).any():
    log(f"districts without cells (merged or absent in the boundary layer): "
        f"{[DISTRICTS[i]['name'] for i in np.flatnonzero(_s01_counts == 0)]}", "WARNING")
del _s01_img, _s01_counts, _s01_extent, _s01_tag, _s01_palette, _s01_cmap, _d, _sel
free_memory()

# 2. Synthetic world 1990–2025 and future inputs (synthetic mode only)

When `DATA_MODE == "synthetic"` this section builds a **synthetic twin of Delhi NCR** that has the same
globals, keys, shapes, dtypes and units as the Earth Engine acquisition layer (sections 03–05, SPEC §3.2):
`RAW_STATIC`, `RAW_OBS`, `LC_FINE`, `FRAME_META`, `NIGHT_CALIBRATION`, `ANNUAL_SERIES`, `FUTURE_INPUTS` and
`SYNTHETIC_TRUTH`. Sections 06–12 therefore run unchanged on it. In gee mode the section does nothing.

**What is simulated and why** (every number is *planted* and recorded in `SYNTHETIC_TRUTH`):

* **Geography** (v1 generator, ported): a N→SE sloping plain with the Aravalli ridge (relief growing from the
  Delhi ridge towards Alwar), slightly incised Yamuna/Ganga floodplains, rivers, jheels and ponds, rocky foothills,
  the arid south-west and an irrigated north-east (Karnal–Shamli–Muzaffarnagar sugarcane belt).
* **Urban growth 1990 → 2030** from a 1990 state: Delhi first (logistic radius growth centred in the 1990s),
  Gurugram, Noida, Ghaziabad and Faridabad accelerating after 2000, Greater Noida / Manesar after 2010.
  Built-up land is irreversible and is taken mainly from cropland; forest and water are stable.
  Products: GHSL-like `built_f` (fine), `built_v` and `pop` (1 km); a GLC_FCS30D-like land-cover map with its
  own classification noise (`LC_FINE`, `glc_*`; the 2025 frame uses the 2022 map exactly like the real product).
* **Landsat-like spectra** consistent with land cover: pre-monsoon harvested cropland has a low NDVI but a high
  February–March NDVI (`ndvi_spring_f`), the irrigated north-east stays green, the bright barren south-west has a
  high albedo.
* **ERA5-Land-like climate** per frame (1 km, smooth): a south-west → north-east gradient (NE cooler, wetter),
  warming of ~0.2 K/decade and inter-annual anomalies that are identical to the annual series (`ANNUAL_SERIES`).
  Wind (not a planted driver) follows its own synoptic/terrain pattern more than the SW→NE gradient, so it is not
  a near-copy of the planted soil-moisture field (|r| ≈ 0.9 before; the night model then credited wind).
* **Identifiability by design:** albedo has a within-class brightness component and building height a typology
  component, so every planted driver has independent variation that a model can, in principle, attribute.
* **Planted LST physics** at the fine grid (so every 1 km aggregate is consistent):
  *day* (~10:45 IST) — saturating NDVI cooling, park cooling via neighbourhood NDVI, harvest heating via
  `ndvi_drop`, albedo cooling, a **negative daytime surface UHI** (built cooler than barren/fallow land),
  barren heating, saturating water cooling, regional moisture (swvl1 / VPD), elevation and TPI;
  *night* (~01:30 IST) — saturating built-up warming plus building volume (**positive night SUHI**), water
  warmer than land, cold-air pooling (TPI), soil-moisture thermal inertia, weak NDVI cooling.
* **Absolute levels** `L(t)`: day ≈ 41 °C + trend + β_day·ΔTmax(t) (β_day = 1.3), night ≈ 26 °C + trend +
  β_night·ΔTmin(t) (β_night = 0.9), both with a soil-moisture term, so section 09 can recover β from the annual
  satellite series.
* **Observation artefacts** that sections 06–10 must undo: the Landsat overpass-time bias
  (`lst_day_f` is shifted by −(10.75 − overpass_h)·heating_rate), per-path offsets removed by the same overlap
  harmonisation as section 03, few scenes in the 1990s, Landsat 7 SLC-off stripes, gaps, MODIS Terra drift after
  2022, Terra-22:30 and VIIRS night products that must be calibrated to Aqua (`NIGHT_CALIBRATION` is fitted on
  simulated overlap years exactly as section 04 fits it on real ones).
* **Future inputs** (illustrative): 34 pseudo-GCMs for ssp245/ssp585, a warming pattern, SSP urban-growth ratios
  per district, GHSL-2030-like built-up/population, SSP urban-population factors and AR6-like GSAT increments.

Everything is vectorised NumPy/SciPy, seeded and deterministic (independent random stream per purpose).

In [ ]:
_lst_enter_section('02')  # section marker (generated by build_notebook.py)

In [ ]:
import json as _s02_json
import math as _s02_math
import time as _s02_time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as _s02_ndi
from scipy import sparse as _s02_sparse

_S02_ACTIVE = globals().get("DATA_MODE") == "synthetic"
_S02_GENERATOR = "delhi-ncr-synthetic-v2"
_S02_REF_OVERPASS_IST_H = 10.75          # SPEC §4.7 reference overpass (10:45 IST)

# Harmonised land-cover classes (SPEC_v1 §1): 0 nodata, 1 built, 2 forest, 3 water, 4 cropland, 5 barren, 6 other veg
_S02_BUILT, _S02_FOREST, _S02_WATER, _S02_CROP, _S02_BARREN, _S02_OTHER = 1, 2, 3, 4, 5, 6
_S02_GLC_KEYS = ["glc_built", "glc_forest", "glc_water", "glc_cropland", "glc_barren", "glc_otherveg"]
_S02_FINE_KEYS = ["lst_day_f", "lst_day_nobs_f", "ndvi_f", "ndbi_f", "mndwi_f", "albedo_f", "ndvi_spring_f", "built_f"]
_S02_ERA5_KEYS = ["era5_t2m", "era5_rh", "era5_vpd", "era5_swvl1", "era5_evap", "era5_ssr", "era5_wind",
                  "era5_tmax", "era5_tmin", "era5_precip_jfm"]
_S02_COARSE_KEYS = ["lst_night", "lst_night_nobs", "lst_day_modis", "built_v", "pop"] + _S02_GLC_KEYS + _S02_ERA5_KEYS
_S02_PATHROWS = ["146/040", "146/041", "147/039", "147/040", "147/041"]

# Reference geography (lon, lat) - identical to the v1 generator.
_S02_YAMUNA = [(77.19, 29.95), (77.13, 29.55), (77.12, 29.20), (77.22, 28.85), (77.25, 28.62), (77.30, 28.45),
               (77.45, 28.20), (77.52, 27.90), (77.68, 27.55)]
_S02_GANGA = [(78.03, 29.75), (78.10, 29.30), (78.18, 28.95), (78.32, 28.60), (78.45, 28.30)]
_S02_ARAVALLI = [(77.17, 28.62), (77.10, 28.40), (76.85, 28.15), (76.60, 27.80), (76.40, 27.35)]

# Urban centres: name, lon, lat, amplitude, radius 1990 (km), radius 2025 (km), logistic mid-year, time scale (yr).
# r(y) follows a logistic between the two anchor radii, so 1990 and 2025 are exact and 2030 extrapolates smoothly.
# Delhi grows first (mid-point 1996); the satellite cities accelerate after 2000; Greater Noida/Manesar after 2010.
_S02_URBAN_CENTRES = [
    ("Delhi", 77.20, 28.63, 1.00, 8.5, 14.0, 1996, 8), ("Rohini", 77.09, 28.73, 0.85, 1.0, 4.6, 2000, 5),
    ("Dwarka", 77.04, 28.59, 0.80, 0.5, 4.0, 2005, 5), ("Gurugram", 77.03, 28.46, 0.92, 1.4, 7.6, 2008, 5),
    ("Noida", 77.36, 28.57, 0.92, 1.1, 6.6, 2006, 5), ("Greater Noida", 77.50, 28.47, 0.75, 0.4, 4.8, 2012, 5),
    ("Ghaziabad", 77.44, 28.67, 0.88, 2.4, 6.4, 2004, 6), ("Faridabad", 77.31, 28.40, 0.88, 2.6, 6.2, 2005, 6),
    ("Manesar", 76.93, 28.36, 0.62, 0.3, 3.2, 2011, 5), ("Bahadurgarh", 76.92, 28.69, 0.60, 0.8, 2.8, 2008, 6),
    ("Meerut", 77.71, 28.98, 0.80, 2.8, 4.4, 2002, 10), ("Sonipat", 77.02, 28.99, 0.65, 1.3, 2.9, 2008, 8),
    ("Panipat", 76.97, 29.39, 0.70, 1.8, 3.2, 2004, 10), ("Karnal", 76.99, 29.69, 0.65, 1.7, 3.0, 2004, 10),
    ("Rohtak", 76.61, 28.90, 0.70, 1.9, 3.1, 2003, 10), ("Rewari", 76.62, 28.19, 0.60, 1.2, 2.3, 2005, 10),
    ("Alwar", 76.60, 27.55, 0.70, 1.8, 3.1, 2004, 10), ("Bharatpur", 77.49, 27.22, 0.65, 1.5, 2.6, 2004, 10),
    ("Bulandshahr", 77.85, 28.40, 0.60, 1.3, 2.3, 2004, 10), ("Muzaffarnagar", 77.70, 29.47, 0.65, 1.5, 2.6, 2004, 10),
    ("Hapur", 77.78, 28.73, 0.60, 1.2, 2.3, 2005, 10), ("Bhiwani", 76.13, 28.79, 0.60, 1.5, 2.6, 2004, 10),
    ("Palwal", 77.33, 28.14, 0.55, 0.7, 1.9, 2008, 8), ("Jhajjar", 76.66, 28.61, 0.50, 0.7, 1.7, 2005, 10),
    ("Jind", 76.32, 29.32, 0.55, 1.0, 2.0, 2004, 10), ("Shamli", 77.31, 29.45, 0.50, 0.7, 1.5, 2006, 10),
    ("Baghpat", 77.22, 28.94, 0.50, 0.5, 1.4, 2006, 10), ("Nuh", 77.00, 28.10, 0.50, 0.4, 1.2, 2008, 10),
    ("Narnaul", 76.11, 28.05, 0.55, 0.9, 1.7, 2005, 10), ("Charkhi Dadri", 76.27, 28.59, 0.50, 0.6, 1.4, 2006, 10),
]

_S02_DEFAULT_WINDOWS = {1990: (1989, 1992), 1995: (1994, 1996), 2000: (1999, 2001), 2005: (2004, 2006),
                        2010: (2009, 2011), 2015: (2014, 2016), 2020: (2019, 2021), 2025: (2024, 2026)}

# Landsat acquisition per frame: real-archive sensors, scene totals and per-path/row counts (USGS STAC inventory,
# docs/research/gee_datasets_v2.md §3), mean overpass (IST hours), heating rate (ERA5-Land 09:30→11:30 IST skin
# temperature slope, K/h), the planted path-146 minus path-147 offset (°C) and the share of SLC-off scenes.
_S02_DAY_ACQ = {
    1990: {"sensors": ["Landsat 5 TM"], "scenes": 69, "per_pr": (3, 3, 8, 10, 10), "overpass": 10.18,
           "heating": 3.30, "path_delta": 0.90, "slc_share": 0.0},
    1995: {"sensors": ["Landsat 5 TM"], "scenes": 89, "per_pr": (5, 6, 4, 4, 4), "overpass": 9.92,
           "heating": 3.20, "path_delta": -0.60, "slc_share": 0.0},
    2000: {"sensors": ["Landsat 5 TM", "Landsat 7 ETM+"], "scenes": 110, "per_pr": (8, 10, 8, 6, 7),
           "overpass": 10.50, "heating": 3.15, "path_delta": 0.50, "slc_share": 0.0},
    2005: {"sensors": ["Landsat 7 ETM+ (SLC-off)"], "scenes": 155, "per_pr": (9, 12, 13, 13, 13), "overpass": 10.70,
           "heating": 3.25, "path_delta": 0.40, "slc_share": 1.0},
    2010: {"sensors": ["Landsat 5 TM", "Landsat 7 ETM+ (SLC-off)"], "scenes": 332, "per_pr": (20, 21, 26, 30, 29),
           "overpass": 10.62, "heating": 3.10, "path_delta": -0.30, "slc_share": 155 / 332},
    2015: {"sensors": ["Landsat 8 OLI/TIRS", "Landsat 7 ETM+ (SLC-off)"], "scenes": 354,
           "per_pr": (21, 26, 27, 26, 27), "overpass": 10.78, "heating": 3.35, "path_delta": 0.25,
           "slc_share": 174 / 354},
    2020: {"sensors": ["Landsat 8 OLI/TIRS", "Landsat 7 ETM+ (SLC-off)"], "scenes": 302,
           "per_pr": (20, 22, 22, 27, 26), "overpass": 10.80, "heating": 3.40, "path_delta": -0.20,
           "slc_share": 124 / 302},
    2025: {"sensors": ["Landsat 8 OLI/TIRS", "Landsat 9 OLI-2/TIRS-2"], "scenes": 367,
           "per_pr": (23, 25, 29, 29, 29), "overpass": 10.80, "heating": 3.30, "path_delta": 0.15,
           "slc_share": 0.0},
}
# WRS-2 footprints over the ROI (descending orbit: the swath edge drifts east by ~0.24° lon per degree of latitude).
_S02_WRS = {"tilt_deg_lon_per_deg_lat": 0.24, "lat_ref": 28.5, "centre": {"146": 78.07, "147": 76.52},
            "east_edge_147": 77.50, "west_edge_146": 77.12, "half_swath_km": 92.5,
            "row_split": {"147": (29.60, 28.15), "146": (None, 28.15)}}

# Night products per frame (SPEC §1.2/§2.2): source, sensor, overpass (IST h), composites in the window.
_S02_NIGHT_ACQ = {
    1990: (None, None, None, 0), 1995: (None, None, None, 0),
    2000: ("terra_cal", "MODIS Terra MOD11A2 night (22:30 IST), calibrated to Aqua", 22.45, 24),
    2005: ("aqua", "MODIS Aqua MYD11A2 night", 2.20, 36), 2010: ("aqua", "MODIS Aqua MYD11A2 night", 2.20, 36),
    2015: ("aqua", "MODIS Aqua MYD11A2 night", 2.20, 36), 2020: ("aqua", "MODIS Aqua MYD11A2 night", 2.20, 36),
    2025: ("viirs_cal", "VIIRS S-NPP VNP21A1N, calibrated to Aqua", 2.10, 273),
}

# --- Planted LST physics (anomaly terms, °C; evaluated per fine pixel) ---------------------------------------------
_S02_DAY = {
    "ndvi_cooling_c": -4.5, "ndvi_saturation": 0.30,                      # -4.5 * min(ndvi, 0.30) / 0.30
    "park_cooling_c": -1.2, "park_onset": 0.20, "park_full": 0.35, "park_window_m": 3000,
    "harvest_heating_c": 2.2, "harvest_onset": 0.12, "harvest_full": 0.40,  # ndvi_drop = spring - pre-monsoon
    "albedo_per_unit": -9.0, "albedo_ref": 0.18,
    "built_per_unit": -2.2,                                                # morning urban cool island
    "barren_per_unit": 2.8,
    "water_cooling_c": -7.0, "water_scale": 0.08, "water_window_m": 1000,  # on the 1 km neighbourhood water share
    "water_reach_c": -0.8, "water_reach_m": 600.0,
    "swvl1_per_unit": -12.0, "vpd_per_kpa": 0.6,
    "lapse_per_m": -0.0045, "elev_ref_m": 220.0, "tpi_per_m": 0.006, "tpi_window_m": 7000,
}
_S02_NIGHT = {
    "built_c": 3.2, "built_scale": 0.30,                                   # 3.2 * (1-exp(-b/0.3)) / (1-exp(-1/0.3))
    "built_v_per_log1p": 0.6,                                              # building volume (m3/m2), log1p
    "water_warming_c": 2.5, "water_scale": 0.08, "water_window_m": 1000,
    "ndvi_per_unit": -0.8, "barren_per_unit": -1.2,
    "tpi_per_m": 0.012, "tpi_window_m": 7000, "lapse_per_m": -0.003, "elev_ref_m": 220.0,
    "swvl1_per_unit": 6.0, "vpd_per_kpa": -0.4,
}
_S02_NOISE = {"day_corr_sd": 0.45, "day_corr_km": 8.0, "day_meso_sd": 0.25, "day_meso_km": 2.0, "day_white_sd": 0.30,
              "night_corr_sd": 0.35, "night_corr_km": 8.0, "night_white_sd": 0.25, "persistent_share": 0.75,
              "landsat_white_base": 0.25, "landsat_white_per_sqrt_n": 0.8, "landsat_scene_corr": 0.35,
              "modis_day_corr": 0.25, "modis_day_white": 0.35, "aqua_frame_white": 0.15,
              "terra_night_annual": 0.25, "viirs_night_annual": 0.18, "aqua_night_annual": 0.15,
              "annual_cell_sd": 0.30, "annual_scope_sd_day": 0.20, "annual_scope_sd_night": 0.15}
# Absolute levels: L(y) = base + trend*(y-2010)/10 + beta*(T(y) - T_ref) + gamma*(swvl1(y) - swvl1_ref)
_S02_LEVEL = {"day": {"base_c": 41.0, "trend_per_decade": 0.15, "beta": 1.3, "gamma_swvl1": -12.0,
                      "temperature_var": "era5_tmax"},
              "night": {"base_c": 26.0, "trend_per_decade": 0.15, "beta": 0.9, "gamma_swvl1": 6.0,
                        "temperature_var": "era5_tmin"}}
# Satellite-specific behaviour (annual series and 1 km day product).
_S02_TERRA_DAY_OFFSET_C = -0.8        # MODIS 1 km day LST minus the Landsat 10:45 IST level (algorithm + time)
_S02_TERRA_DAY_IST = {2022: 11.03, 2023: 10.70, 2024: 10.53, 2025: 10.12, 2026: 9.37}   # else 11.12 (11:07 IST)
_S02_TERRA_DAY_RATE = 3.0             # K/h around 11:00 IST
_S02_AQUA_NIGHT_IST = {2023: 2.45, 2024: 2.87, 2025: 3.33, 2026: 4.03}                # else 2.20 (02:12 IST)
_S02_NIGHT_COOLING_RATE = 0.55        # K/h after midnight
_S02_DRIFT_FROM = 2022                # Terra/Aqua flagged "drift" from this year (SPEC §2.2)
# Planted calibration maps aqua = a + b * src per dominant class (1..6).
_S02_CAL_PLANTED = {
    "terra_to_aqua": {1: (-0.5, 0.95), 2: (-0.8, 0.95), 3: (0.2, 0.96), 4: (-0.9, 0.94), 5: (-1.2, 0.93),
                      6: (-0.9, 0.94)},
    "viirs_to_aqua": {1: (-0.6, 0.99), 2: (-0.3, 1.00), 3: (0.2, 0.99), 4: (-0.7, 0.985), 5: (-1.0, 0.96),
                      6: (-0.6, 0.985)},
}
_S02_CAL_FIT_YEARS = {"terra_to_aqua": (2003, 2019), "viirs_to_aqua": (2013, 2021)}

# ERA5-Land-like Apr-Jun climatology: value at the gradient centre, change per unit of the SW->NE coordinate g
# (g = -1 south-west ... +1 north-east), amplitude of the smooth static pattern. Units as SPEC §2.3.
_S02_CLIM = {"era5_tmax": (39.6, -1.6, 0.25), "era5_tmin": (25.4, -0.5, 0.15), "era5_t2m": (32.6, -1.1, 0.20),
             "era5_rh": (31.0, 9.0, 2.0), "era5_swvl1": (0.18, 0.07, 0.010), "era5_evap": (1.6, 1.1, 0.15),
             "era5_ssr": (23.2, -0.6, 0.20), "era5_wind": (2.6, -0.2, 0.30), "era5_precip_jfm": (45.0, 25.0, 5.0),
             "era5_skin_day": (40.2, -2.0, 0.30), "era5_skin_night": (24.8, -0.4, 0.20)}
_S02_LAPSE_VARS = ("era5_tmax", "era5_tmin", "era5_t2m", "era5_skin_day", "era5_skin_night")

# Spectral class signatures (pre-monsoon Apr-Jun; spring = Feb-Mar), order built, forest, water, crop, barren, other.
_S02_SIG = {"ndvi": (0.11, 0.46, -0.08, 0.17, 0.08, 0.26),
            "ndvi_spring": (0.13, 0.50, -0.06, 0.56, 0.09, 0.32),
            "ndbi": (0.10, -0.20, -0.36, 0.03, 0.13, -0.06),
            "mndwi": (-0.22, -0.36, 0.48, -0.26, -0.30, -0.30),
            "albedo": (0.17, 0.12, 0.06, 0.19, 0.27, 0.16)}

# Future (illustrative stand-ins for GDDP-CMIP6, Chen et al. 2020, IIASA SSP v3 and AR6 WG1 Table 4.5).
_S02_CMIP6 = {"n_models": 34, "delta_2050": {"ssp245": {"tasmax": 1.1, "tasmin": 1.2},
                                              "ssp585": {"tasmax": 1.7, "tasmin": 1.85}},
              "shape_exponent": {"ssp245": 1.05, "ssp585": 1.20}, "model_sd": 0.30,
              "idio_sd": {"ssp245": 0.20, "ssp585": 0.08}, "bias_sd_k": 0.5, "window_noise_k": 0.06}
# Annual urban-land growth rate per district under SSP2 (%/yr, decaying with an e-folding time of 60 yr).
_S02_URBAN_RATE = {0: 0.9, 1: 3.2, 2: 2.2, 3: 2.4, 4: 1.8, 5: 2.2, 6: 2.4, 7: 1.8, 8: 2.8, 9: 2.0, 10: 1.8, 11: 1.5,
                   12: 1.4, 13: 1.4, 14: 1.3, 15: 2.0, 16: 2.6, 17: 3.4, 18: 1.7, 19: 2.3, 20: 2.3, 21: 1.6,
                   22: 1.6, 23: 1.6, 24: 1.4}
_S02_SSP_URBAN_MULT = {"ssp126": 0.75, "ssp245": 1.00, "ssp585": 1.35}
_S02_POP_FACTOR_2050 = {"ssp126": 1.55, "ssp245": 1.48, "ssp585": 1.62}
# AR6 WG1 Table 4.5 GSAT change vs 1995-2014 (central): near term 2021-2040, mid term 2041-2060.
_S02_AR6_GSAT = {"ssp126": (0.6, 0.9), "ssp245": (0.7, 1.1), "ssp585": (0.8, 1.5)}
_S02_POP_TOTAL = {"1990": 31.5e6, "2025": 66.0e6}

if _S02_ACTIVE:
    log(f"section 02: synthetic world generator {_S02_GENERATOR} is active")
else:
    log("section 02 skipped: data mode is not 'synthetic' (sections 03-05 provide the acquisition globals)")
    # SPEC §3.2: SYNTHETIC_TRUTH is None in gee mode; define it early so later sections can always reference it.
    if "SYNTHETIC_TRUTH" not in globals():
        SYNTHETIC_TRUTH = None

## 2.1 Numerical helpers
Random streams are derived from `(CFG.SEED, purpose tags)` with `SeedSequence`, so every field is reproducible and
independent of call order. Spatially-correlated Gaussian fields are synthesised on a coarser lattice, smoothed and
bilinearly upsampled (same statistics at a fraction of the cost). Resampling between the land-cover grid
(`LC_FINE_RES_M`, e.g. 100 m) and the fine grid (250 m) uses exact **area-overlap matrices** (sparse, separable),
because 100 m does not divide 250 m.

In [ ]:
def _s02_cfg(name: str, default):
    """CFG attribute with a fallback (keeps the section usable with older configuration objects)."""
    value = getattr(CFG, name, None)
    return default if value is None else value


def _s02_lc_res() -> int:
    """Land-cover grid resolution (m): the resolved ``CFG.fine_res()`` (SPEC §1.1 LC_FINE_RES_M)."""
    if hasattr(CFG, "fine_res"):
        return int(CFG.fine_res())
    return int(_s02_cfg("LC_FINE_RES_M", 100))


def _s02_rng(*tags) -> np.random.Generator:
    """Independent reproducible random stream for (CFG.SEED, section 02, tags)."""
    return np.random.default_rng(np.random.SeedSequence([int(_s02_cfg("SEED", 42)), 2002, *[int(t) for t in tags]]))


def _s02_interp_axis_int(a: np.ndarray, f: int, axis: int) -> np.ndarray:
    """Linear interpolation of cell-centred samples onto an f-times finer grid along one axis (edges clamped)."""
    n = a.shape[axis]
    pos = np.clip((np.arange(n * f) + 0.5) / f - 0.5, 0.0, n - 1)
    i0 = np.floor(pos).astype(np.intp)
    i1 = np.minimum(i0 + 1, n - 1)
    shape = [1] * a.ndim
    shape[axis] = -1
    weight = (pos - i0).astype(np.float32).reshape(shape)
    lo = np.take(a, i0, axis=axis)
    return lo + (np.take(a, i1, axis=axis) - lo) * weight


def _s02_upsample_int(field: np.ndarray, f: int) -> np.ndarray:
    """Separable bilinear upsampling by an integer factor (pixel-edge aligned, float32)."""
    field = field.astype(np.float32, copy=f == 1)
    if f == 1:
        return field
    return _s02_interp_axis_int(_s02_interp_axis_int(field, f, 0), f, 1)


def _s02_corr_noise(shape, sigma_px: float, rng: np.random.Generator) -> np.ndarray:
    """Zero-mean, unit-variance Gaussian random field with correlation length ~``sigma_px`` pixels (float32)."""
    h, w = int(shape[0]), int(shape[1])
    if sigma_px < 0.5:
        return rng.standard_normal((h, w), dtype=np.float32)
    k = max(1, int(sigma_px // 3))
    small = rng.standard_normal((-(-h // k) + 1, -(-w // k) + 1), dtype=np.float32)
    small = _s02_ndi.gaussian_filter(small, sigma_px / k, mode="reflect")
    field = _s02_upsample_int(small, k)[:h, :w]
    field -= field.mean()
    field /= field.std() + 1e-12
    return np.ascontiguousarray(field, dtype=np.float32)


def _s02_noise_m(shape, res_m: float, corr_m: float, *tags) -> np.ndarray:
    """Correlated unit noise with a correlation length given in metres on a grid of ``res_m`` metres."""
    return _s02_corr_noise(shape, corr_m / float(res_m), _s02_rng(*tags))


def _s02_interp_axis_ratio(a: np.ndarray, src_res: int, dst_res: int, axis: int) -> np.ndarray:
    """Linear interpolation of cell-centred samples from ``src_res`` to ``dst_res`` cells along one axis."""
    n = a.shape[axis]
    extent = n * int(src_res)
    if extent % int(dst_res):
        raise ValueError(f"extent {extent} m is not a multiple of {dst_res} m")
    n_dst = extent // int(dst_res)
    pos = np.clip((np.arange(n_dst) + 0.5) * (dst_res / src_res) - 0.5, 0.0, n - 1)
    i0 = np.floor(pos).astype(np.intp)
    i1 = np.minimum(i0 + 1, n - 1)
    shape = [1] * a.ndim
    shape[axis] = -1
    weight = (pos - i0).astype(np.float32).reshape(shape)
    lo = np.take(a, i0, axis=axis)
    return lo + (np.take(a, i1, axis=axis) - lo) * weight


def _s02_bilinear(field: np.ndarray, src_res: int, dst_res: int) -> np.ndarray:
    """Bilinear resampling of a smooth field between two nested grids sharing the origin (any resolution ratio)."""
    field = np.asarray(field, dtype=np.float32)
    if int(src_res) == int(dst_res):
        return field.copy()
    out = _s02_interp_axis_ratio(_s02_interp_axis_ratio(field, src_res, dst_res, 0), src_res, dst_res, 1)
    return np.ascontiguousarray(out, dtype=np.float32)


def _s02_area_matrix(n_src: int, src_res: int, n_dst: int, dst_res: int):
    """Sparse (n_dst x n_src) matrix of 1-D area-overlap weights (rows sum to 1) between aligned grids."""
    src_res, dst_res = int(src_res), int(dst_res)
    if n_src * src_res != n_dst * dst_res:
        raise ValueError(f"grids do not share an extent: {n_src}x{src_res} m vs {n_dst}x{dst_res} m")
    max_span = -(-dst_res // src_res) + 1
    j = np.repeat(np.arange(n_dst), max_span)
    lo = j * dst_res
    first = lo // src_res
    i = first + np.tile(np.arange(max_span), n_dst)
    overlap = (np.minimum((i + 1) * src_res, lo + dst_res) - np.maximum(i * src_res, lo)).astype(np.float64)
    keep = (i < n_src) & (overlap > 0)
    mat = _s02_sparse.csr_matrix((overlap[keep] / dst_res, (j[keep], i[keep])), shape=(n_dst, n_src))
    return mat.astype(np.float32)


def _s02_area_resample(a: np.ndarray, mat_r, mat_c) -> np.ndarray:
    """Area-weighted resampling ``mat_r @ a @ mat_c.T`` of a 2-D float array (exact for indicator fractions)."""
    tmp = mat_r @ np.asarray(a, dtype=np.float32)
    return np.ascontiguousarray((mat_c @ tmp.T).T, dtype=np.float32)


def _s02_block_mean(fine: np.ndarray, f: int, valid: np.ndarray = None):
    """f x f block means of the finite (and ``valid``) fine pixels -> (mean float32 with NaN, count int)."""
    h, w = fine.shape[0] // f, fine.shape[1] // f
    ok = np.isfinite(fine) if valid is None else (valid & np.isfinite(fine))
    total = np.where(ok, fine, 0.0).reshape(h, f, w, f).sum(axis=(1, 3), dtype=np.float64)
    count = ok.reshape(h, f, w, f).sum(axis=(1, 3))
    out = np.full((h, w), np.nan, np.float32)
    np.divide(total, count, out=out, where=count > 0, casting="unsafe")
    return out, count


def _s02_box_mean(x: np.ndarray, size: int, valid: np.ndarray = None) -> np.ndarray:
    """NaN-aware moving-window mean ``uniform(x*m)/uniform(m)`` with a square window of ``size`` pixels."""
    size = max(1, int(size))
    ok = np.isfinite(x) if valid is None else (valid & np.isfinite(x))
    num = _s02_ndi.uniform_filter(np.where(ok, x, 0.0).astype(np.float32), size, mode="nearest")
    den = _s02_ndi.uniform_filter(ok.astype(np.float32), size, mode="nearest")
    out = np.full(x.shape, np.nan, np.float32)
    np.divide(num, den, out=out, where=den > 1e-6)
    return out


def _s02_px(metres: float, res_m: int) -> int:
    """Window size in pixels for a physical length (at least one pixel)."""
    return max(1, int(round(float(metres) / float(res_m))))


def _s02_lonlat_to_xy(points) -> np.ndarray:
    """(lon, lat) pairs -> projected metres in the grid CRS (pyproj, falling back to section 01's converter)."""
    pts = np.asarray(points, dtype=np.float64)
    try:
        import pyproj as _s02_pyproj
        transformer = _s02_pyproj.Transformer.from_crs("EPSG:4326", GRID.crs, always_xy=True)
        x, y = transformer.transform(pts[:, 0], pts[:, 1])
    except ImportError:
        converter = globals().get("_grid_lonlat_to_xy")
        if converter is None:
            raise RuntimeError("neither pyproj nor section 01's _grid_lonlat_to_xy is available") from None
        x, y = converter(pts[:, 0], pts[:, 1], GRID.crs)
    return np.column_stack([np.asarray(x, np.float64), np.asarray(y, np.float64)])


def _s02_polyline_distance_position(xs: np.ndarray, ys: np.ndarray, line_xy: np.ndarray, chunk_rows: int = 256):
    """Distance (m) from every grid point (xs[j], ys[i]) to a polyline and the along-line position (0..1)."""
    ox, oy = float(xs[0]), float(ys[0])
    xr = (xs - ox).astype(np.float32)[None, :]
    a = (line_xy[:-1] - (ox, oy)).astype(np.float32)
    b = (line_xy[1:] - (ox, oy)).astype(np.float32)
    seg_len = np.hypot(*(b - a).T)
    cum = np.concatenate([[0.0], np.cumsum(seg_len)]) / max(float(seg_len.sum()), 1e-9)
    dist = np.empty((len(ys), len(xs)), np.float32)
    pos = np.empty_like(dist)
    for r0 in range(0, len(ys), chunk_rows):
        yr = (ys[r0:r0 + chunk_rows] - oy).astype(np.float32)[:, None]
        best = np.full((len(yr), len(xs)), np.inf, np.float32)
        best_pos = np.zeros_like(best)
        for i in range(len(a)):
            (ax, ay), (dx, dy) = a[i], b[i] - a[i]
            t = np.clip(((xr - ax) * dx + (yr - ay) * dy) / max(float(dx * dx + dy * dy), 1e-9), 0.0, 1.0)
            d2 = (xr - ax - t * dx) ** 2 + (yr - ay - t * dy) ** 2
            closer = d2 < best
            best = np.where(closer, d2, best)
            best_pos = np.where(closer, cum[i] + t * (cum[i + 1] - cum[i]), best_pos)
        dist[r0:r0 + chunk_rows] = np.sqrt(best)
        pos[r0:r0 + chunk_rows] = best_pos
    return dist, pos


def _s02_polyline_distance_capped(xs: np.ndarray, ys: np.ndarray, line_xy: np.ndarray, cap_m: float) -> np.ndarray:
    """Distance (m) to a polyline, exact up to ``cap_m`` (windowed per segment; xs increasing, ys decreasing)."""
    dist = np.full((len(ys), len(xs)), cap_m, np.float32)
    neg_ys = -ys
    for (ax, ay), (bx, by) in zip(line_xy[:-1], line_xy[1:]):
        c0 = int(np.searchsorted(xs, min(ax, bx) - cap_m))
        c1 = int(np.searchsorted(xs, max(ax, bx) + cap_m, side="right"))
        r0 = int(np.searchsorted(neg_ys, -(max(ay, by) + cap_m)))
        r1 = int(np.searchsorted(neg_ys, -(min(ay, by) - cap_m), side="right"))
        if c0 >= c1 or r0 >= r1:
            continue
        xw = (xs[c0:c1] - ax).astype(np.float32)[None, :]
        yw = (ys[r0:r1] - ay).astype(np.float32)[:, None]
        dx, dy = float(bx - ax), float(by - ay)
        t = np.clip((xw * dx + yw * dy) / max(dx * dx + dy * dy, 1e-9), 0.0, 1.0)
        window = dist[r0:r1, c0:c1]
        np.minimum(window, np.sqrt((xw - t * dx) ** 2 + (yw - t * dy) ** 2), out=window)
    return dist


def _s02_logistic(x):
    return 1.0 / (1.0 + np.exp(-x))


def _s02_magnus_vpd(t_c: np.ndarray, rh_pct: np.ndarray) -> np.ndarray:
    """Vapour-pressure deficit (kPa) from air temperature (°C) and relative humidity (%), Magnus-Tetens."""
    es = 0.6108 * np.exp(17.27 * t_c / (t_c + 237.3))
    return (es * (1.0 - np.clip(rh_pct, 0.0, 100.0) / 100.0)).astype(np.float32)


def _s02_py(obj):
    """Recursively convert numpy scalars/arrays and tuples to plain JSON-able Python objects."""
    if isinstance(obj, dict):
        return {(int(k) if isinstance(k, (np.integer,)) else k): _s02_py(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_s02_py(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return [_s02_py(v) for v in obj.tolist()]
    if isinstance(obj, (np.floating, float)):
        value = float(obj)
        return None if not np.isfinite(value) else round(value, 6)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj

## 2.2 Static world: grids, geography, water, Landsat footprints, climatology
* Elevation is built at the fine grid (250 m): plain + Aravalli ridge (Gaussian cross-section, relief and width
  growing along the ridge) + floodplain incision + hillocks; slope/aspect are computed at the fine grid and
  averaged to 1 km (aspect as sin/cos *before* averaging, as the GEE recipe does).
* The rural land-cover template (cropland, other vegetation, barren, forest, water) is built at the land-cover
  resolution with the v1 rules; rivers are narrow channels with sand bars, jheels are random ellipses whose
  occurrence (JRC-like `water_occ`) is drawn per lake (seasonal ones are dry, i.e. not water, in Apr–Jun).
* WRS-2 path 146 (east) and 147 (west) overlap in a ~35 km strip; rows 039/040/041 split by latitude.

In [ ]:
def _s02_context() -> dict:
    """Grid geometry shared by every generator step (fine grid, land-cover grid, masks, coordinates)."""
    res = int(GRID.res)
    res_f = int(GRID_FINE.res)
    if res % res_f:
        raise ValueError(f"fine grid {res_f} m must divide the analysis grid {res} m")
    f = res // res_f
    r_lc = _s02_lc_res()
    if r_lc <= 0 or res % r_lc:
        raise ValueError(f"CFG.LC_FINE_RES_M={r_lc} must be a positive divisor of GRID_RES_M={res}")
    h, w = GRID.shape
    hf, wf = GRID_FINE.shape
    if (hf, wf) != (h * f, w * f):
        raise ValueError(f"GRID_FINE shape {(hf, wf)} is not {f} x GRID shape {(h, w)}")
    roi = np.asarray(ROI_MASK, bool)
    roi_f = globals().get("ROI_MASK_FINE")
    roi_f = np.repeat(np.repeat(roi, f, 0), f, 1) if roi_f is None else np.asarray(roi_f, bool)
    if roi_f.shape != (hf, wf):
        raise ValueError(f"ROI_MASK_FINE shape {roi_f.shape} != fine grid {(hf, wf)}")
    f_lc = res // r_lc
    lc_grid = GRID.fine(r_lc)
    lon, lat = GRID.lonlat()
    lon_f, lat_f = GRID_FINE.lonlat()
    cells = np.flatnonzero(roi.ravel())
    district = np.asarray(DISTRICT_IDX).ravel()[cells].astype(np.int64)
    ctx = {
        "res": res, "res_f": res_f, "f": f, "r_lc": r_lc, "f_lc": f_lc, "H": h, "W": w, "Hf": hf, "Wf": wf,
        "Hl": h * f_lc, "Wl": w * f_lc, "roi": roi, "roi_f": roi_f, "lc_grid": lc_grid,
        "lon": np.asarray(lon, np.float64), "lat": np.asarray(lat, np.float64),
        "lon_f": np.asarray(lon_f, np.float32), "lat_f": np.asarray(lat_f, np.float32),
        "xs_f": GRID_FINE.x_centers(), "ys_f": GRID_FINE.y_centers(),
        "cells": cells, "cell_district": district, "n_districts": len(DISTRICTS),
        "mat_r": _s02_area_matrix(h * f_lc, r_lc, hf, res_f), "mat_c": _s02_area_matrix(w * f_lc, r_lc, wf, res_f),
    }
    # SW -> NE climate coordinate g in (-1, 1) at 1 km and at the fine grid (NE positive).
    ctx["g"] = np.tanh(((ctx["lon"] - 77.0) * 0.7 + (ctx["lat"] - 28.4) * 0.7) / 0.8).astype(np.float32)
    # Irrigated north-east (Karnal/Panipat belt + western-UP sugarcane belt) and the arid south-west.
    lon_f, lat_f = ctx["lon_f"], ctx["lat_f"]
    haryana_ne = _s02_logistic((lat_f - 29.05) / 0.12) * _s02_logistic((lon_f - 76.75) / 0.15)
    west_up = _s02_logistic((lon_f - 77.25) / 0.12) * _s02_logistic((lat_f - 28.75) / 0.12)
    ctx["irrig_f"] = np.maximum(haryana_ne, west_up).astype(np.float32)
    ctx["arid_f"] = _s02_logistic(((76.5 - lon_f) + (28.3 - lat_f)) / 0.35).astype(np.float32)
    return ctx


def _s02_geography(ctx: dict) -> dict:
    """Fine elevation (m), slope (deg), aspect sin/cos, 7 km TPI and ridge proximity (fine float32 fields)."""
    res_f, shape_f = ctx["res_f"], (ctx["Hf"], ctx["Wf"])
    xs, ys = ctx["xs_f"], ctx["ys_f"]
    ridge_d, ridge_t = _s02_polyline_distance_position(xs, ys, _s02_lonlat_to_xy(_S02_ARAVALLI))
    relief = 80.0 + 470.0 * ridge_t ** 1.3                    # ~300 m a.s.l. at the Delhi ridge, ~650 m near Alwar
    width = 3000.0 + 7000.0 * ridge_t
    undulation = 0.75 + 0.25 * np.tanh(_s02_noise_m(shape_f, res_f, 6000.0, 1))
    ridge = relief * undulation * np.exp(-0.5 * (ridge_d / width) ** 2)
    lon_f, lat_f = ctx["lon_f"], ctx["lat_f"]
    plain = np.clip(205.0 + 22.0 * (lat_f - 28.3) - 12.0 * (lon_f - 77.2), 180.0, 260.0)
    d_river = np.minimum(_s02_polyline_distance_capped(xs, ys, _s02_lonlat_to_xy(_S02_YAMUNA), 12000.0),
                         _s02_polyline_distance_capped(xs, ys, _s02_lonlat_to_xy(_S02_GANGA), 12000.0))
    incision = 7.0 * np.exp(-d_river / 2500.0)                 # floodplains a few metres below the plain
    hillocks = 4.0 * _s02_noise_m(shape_f, res_f, 1500.0, 2) + 1.2 * _s02_noise_m(shape_f, res_f, 500.0, 3)
    elevation = (plain + ridge - incision + hillocks).astype(np.float32)
    dz_row, dz_col = np.gradient(elevation, float(res_f))
    gx, gy_north = dz_col, -dz_row                            # rows run southwards
    slope = np.degrees(np.arctan(np.hypot(gx, gy_north))).astype(np.float32)
    aspect = np.arctan2(-gx, -gy_north)                        # direction the slope faces, clockwise from north
    ridge_prox = (np.exp(-0.5 * (ridge_d / (1.3 * width)) ** 2) * np.clip(relief / 300.0, 0.3, 1.0)).astype(np.float32)
    tpi = elevation - _s02_box_mean(elevation, _s02_px(_S02_DAY["tpi_window_m"], res_f))
    return {"elevation_f": elevation, "slope_f": slope, "aspect_sin_f": np.sin(aspect).astype(np.float32),
            "aspect_cos_f": np.cos(aspect).astype(np.float32), "tpi_f": tpi.astype(np.float32),
            "ridge_prox_f": ridge_prox}


def _s02_lakes(ctx: dict, rng: np.random.Generator):
    """Elliptical ponds/jheels (~1 per 150 km² of ROI, 150-700 m semi-axes) on the land-cover grid.

    Returns the lake mask and the per-pixel JRC-like occurrence (0..1) drawn per lake: perennial lakes have an
    occurrence above 0.5, seasonal jheels below (they are dry - not water - in the pre-monsoon land cover).
    """
    f, r_lc = ctx["f_lc"], ctx["r_lc"]
    hl, wl = ctx["Hl"], ctx["Wl"]
    lakes = np.zeros((hl, wl), bool)
    occ = np.zeros((hl, wl), np.float32)
    roi_cells = np.flatnonzero(ctx["roi"])
    cell_km2 = (ctx["res"] / 1000.0) ** 2
    n_lakes = max(8, int(roi_cells.size * cell_km2 / 150.0))
    for cell in rng.choice(roi_cells, size=min(n_lakes, roi_cells.size), replace=False):
        r, c = divmod(int(cell), ctx["W"])
        cy, cx = (r + rng.random()) * f, (c + rng.random()) * f
        ry, rx = rng.uniform(150, 700) / r_lc, rng.uniform(150, 700) / r_lc
        lake_occ = float(rng.uniform(0.25, 0.98))
        r0, r1 = max(0, int(cy - ry) - 1), min(hl, int(cy + ry) + 2)
        c0, c1 = max(0, int(cx - rx) - 1), min(wl, int(cx + rx) + 2)
        yy, xx = np.mgrid[r0:r1, c0:c1]
        inside = ((yy + 0.5 - cy) / max(ry, 0.5)) ** 2 + ((xx + 0.5 - cx) / max(rx, 0.5)) ** 2 <= 1.0
        lakes[r0:r1, c0:c1] |= inside
        occ[r0:r1, c0:c1] = np.where(inside, np.maximum(occ[r0:r1, c0:c1], lake_occ), occ[r0:r1, c0:c1])
    return lakes, occ


def _s02_static_landcover(ctx: dict, geo: dict) -> dict:
    """Epoch-independent land-cover-grid layers (shared noise => temporally coherent land cover; v1 rules)."""
    r = ctx["r_lc"]
    shape = (ctx["Hl"], ctx["Wl"])
    lc_grid = ctx["lc_grid"]
    xs, ys = lc_grid.x_centers(), lc_grid.y_centers()
    cap = 4000.0
    meander = 120.0 * _s02_noise_m(shape, r, 800.0, 10)
    d_yam = _s02_polyline_distance_capped(xs, ys, _s02_lonlat_to_xy(_S02_YAMUNA), cap)
    d_gan = _s02_polyline_distance_capped(xs, ys, _s02_lonlat_to_xy(_S02_GANGA), cap)
    river_rel = np.minimum((d_yam + meander) / 170.0, (d_gan + meander) / 320.0)   # in channel half-widths
    river_m = np.minimum(d_yam, d_gan)
    del d_yam, d_gan, meander

    tex = _s02_noise_m(shape, r, 250.0, 12)        # field-scale texture
    patch = _s02_noise_m(shape, r, 1500.0, 13)     # landscape-scale patches
    ridge = _s02_bilinear(geo["ridge_prox_f"], ctx["res_f"], r)
    arid = _s02_bilinear(ctx["arid_f"], ctx["res_f"], r)
    base = np.full(shape, _S02_CROP, np.uint8)
    base[0.6 * patch + 0.4 * tex > 0.72] = _S02_OTHER                                   # grass / fallow / shrub
    base[(river_m < 3000) & (0.5 * patch + 0.5 * tex > 0.6)] = _S02_OTHER              # floodplain grassland
    base[arid + 0.45 * patch + 0.2 * tex > 1.25] = _S02_BARREN                         # SW Haryana / Rajasthan
    base[(ridge > 0.25) & (ridge + 0.35 * patch > 0.85)] = _S02_BARREN                 # rocky foothills
    base[(river_rel > 1.0) & (river_rel < 1.0 + 700.0 / 170.0) & (tex > 0.3)] = _S02_BARREN   # sand bars
    base[ridge + 0.3 * tex > 0.75] = _S02_FOREST                                         # Aravalli scrub forest
    base[(river_m < 1500) & (tex > 1.0)] = _S02_FOREST                                   # riparian trees
    base[tex > 2.1] = _S02_FOREST                                                         # groves / farm forestry
    del ridge, arid

    lakes, lake_occ = _s02_lakes(ctx, _s02_rng(11))
    channel = river_rel < 1.0
    occurrence = np.zeros(shape, np.float32)
    occurrence[channel] = np.clip(1.02 - 0.55 * river_rel[channel], 0.45, 1.0)
    margin = (river_rel >= 1.0) & (river_rel < 2.0)
    occurrence[margin] = 0.12 * (2.0 - river_rel[margin])                               # seasonally flooded margins
    occurrence = np.maximum(occurrence, lake_occ)
    water = channel | (lakes & (lake_occ >= 0.5))
    base[lakes & ~water] = _S02_OTHER                                                   # dry seasonal jheel beds
    park = _s02_noise_m(shape, r, 150.0, 14)
    static = {
        "base": base,
        "built_noise": (0.22 * tex + 0.10 * patch).astype(np.float32),
        "village": _s02_noise_m(shape, r, 120.0, 15),
        "park": park > 1.9,
        "lawn": (park > 1.5) & (park <= 1.9),
        "water": water,
        "occurrence": occurrence,
        "ghsl_noise": _s02_noise_m(shape, r, 300.0, 16),
        "glc_confusion": _s02_noise_m(shape, r, 300.0, 17),
    }
    del tex, patch, park, river_rel, river_m, lakes, lake_occ
    return static


def _s02_landsat_geometry(ctx: dict) -> dict:
    """WRS-2 path coverage masks, row splits and the SLC-off gap pattern on the fine grid."""
    lon, lat = ctx["lon_f"], ctx["lat_f"]
    tilt = _S02_WRS["tilt_deg_lon_per_deg_lat"] * (lat - _S02_WRS["lat_ref"])
    cover = {"147": lon <= _S02_WRS["east_edge_147"] + tilt, "146": lon >= _S02_WRS["west_edge_146"] + tilt}
    split = _S02_WRS["row_split"]
    rows = {"147": np.where(lat > split["147"][0], 39, np.where(lat > split["147"][1], 40, 41)).astype(np.int8),
            "146": np.where(lat > split["146"][1], 40, 41).astype(np.int8)}
    slc_gap = {}
    km_per_deg_lon = 111.32 * np.cos(np.radians(lat))
    for path in ("146", "147"):
        across_km = (lon - (_S02_WRS["centre"][path] + tilt)) * km_per_deg_lon
        along_km = lat * 110.6
        phase = 0.7 if path == "146" else 0.0
        stripes = 0.5 + 0.5 * np.sin(2.0 * np.pi * (along_km / 1.0) + phase)
        # Gaps widen from zero at the scene centre to ~25 % of the area at the swath edge (Landsat 7 SLC-off).
        slc_gap[path] = (0.25 * np.clip(np.abs(across_km) / _S02_WRS["half_swath_km"], 0.0, 1.0) * stripes
                         ).astype(np.float32)
    return {"cover": cover, "rows": rows, "slc_gap": slc_gap}

## 2.3 Climate: ERA5-Land-like annual anomalies, frame fields and planted levels
NCR-mean Apr–Jun anomalies are generated for 1985–2030 from AR(1) innovations with a common hot-dry mode (hot
years have lower soil moisture and humidity), warming of 0.20 K/decade (Tmax) and 0.25 K/decade (Tmin) and a
slow solar dimming. A frame field is the static climatology + the **mean anomaly over the frame window** + a tiny
frame pattern, so the ROI mean of every frame field equals the mean of the NCR annual series over that window.
Levels: `L(y) = base + trend·(y−2010)/10 + β·(T(y) − T_ref) + γ·(swvl1(y) − swvl1_ref)` with reference period
2000–2020; a frame level is the mean of `L(y)` over its window.

In [ ]:
def _s02_annual_climate(years: np.ndarray) -> dict:
    """NCR-mean annual anomalies (Apr-Jun; precipitation Jan-Mar as a multiplicative factor) for ``years``."""
    rng = _s02_rng(40)
    n = len(years)
    z = np.zeros((8, n))
    innov = rng.standard_normal((8, n))
    for t in range(n):
        z[:, t] = (0.25 * z[:, t - 1] if t else 0.0) + np.sqrt(1 - 0.25 ** 2) * innov[:, t]
    dy = years - 2005.0
    d = {}
    d["era5_tmax"] = 0.020 * dy + 0.70 * z[0]
    d["era5_tmin"] = 0.025 * dy + 0.55 * (0.75 * z[0] + 0.66 * z[1])
    d["era5_t2m"] = 0.5 * (d["era5_tmax"] + d["era5_tmin"]) + 0.05 * z[7]
    d["era5_swvl1"] = -0.010 * z[0] + 0.014 * z[2] + 0.0002 * dy
    d["era5_rh"] = -2.5 * z[0] + 3.0 * z[2]
    d["era5_evap"] = 25.0 * d["era5_swvl1"] + 0.05 * z[3]
    d["era5_ssr"] = 0.35 * z[0] - 0.2 * z[2] - 0.025 * dy
    d["era5_wind"] = 0.12 * z[3]
    d["era5_precip_jfm"] = np.exp(0.35 * z[4] - 0.06)          # multiplicative factor
    d["era5_skin_day"] = 1.25 * d["era5_tmax"] - 15.0 * d["era5_swvl1"] + 0.12 * z[5]
    d["era5_skin_night"] = 1.0 * d["era5_tmin"] + 6.0 * d["era5_swvl1"] + 0.10 * z[6]
    return {k: v.astype(np.float64) for k, v in d.items()}


def _s02_levels(years: np.ndarray, anom: dict, ref_period) -> dict:
    """Planted annual absolute levels (°C) for day (10:45 IST, Landsat units) and night (Aqua-equivalent)."""
    ref = (years >= ref_period[0]) & (years <= ref_period[1])
    out = {}
    for mode, p in _S02_LEVEL.items():
        temp = anom[p["temperature_var"]]
        sw = anom["era5_swvl1"]
        out[mode] = (p["base_c"] + p["trend_per_decade"] * (years - 2010.0) / 10.0
                     + p["beta"] * (temp - temp[ref].mean()) + p["gamma_swvl1"] * (sw - sw[ref].mean()))
    return out


def _s02_climatology(ctx: dict, geo: dict) -> dict:
    """Static 1 km climatology fields (unmasked) incl. ERA5-scale elevation lapse for temperatures."""
    shape, res = (ctx["H"], ctx["W"]), ctx["res"]
    g = ctx["g"]
    elev_1km, _ = _s02_block_mean(geo["elevation_f"], ctx["f"])
    elev_smooth = _s02_ndi.gaussian_filter(np.nan_to_num(elev_1km, nan=220.0), 9000.0 / res, mode="nearest")
    clim = {}
    for i, (var, (centre, slope, amp)) in enumerate(_S02_CLIM.items()):
        pattern = _s02_noise_m(shape, res, 40000.0, 50, i)
        field = centre + slope * g + amp * pattern
        if var in _S02_LAPSE_VARS:
            field = field - 0.0065 * (elev_smooth - 220.0)
        clim[var] = field.astype(np.float32)
    clim["era5_swvl1"] = np.clip(clim["era5_swvl1"], 0.05, 0.45)
    clim["era5_precip_jfm"] = np.maximum(clim["era5_precip_jfm"], 5.0)
    return clim


def _s02_frame_climate(ctx: dict, clim: dict, anom: dict, years: np.ndarray, year: int, window) -> dict:
    """ERA5-like 1 km fields of one frame (unmasked, float32); VPD from the frame t2m and RH (Magnus)."""
    sel = (years >= window[0]) & (years <= window[1])
    shape, res = (ctx["H"], ctx["W"]), ctx["res"]
    out = {}
    for i, var in enumerate(_S02_ERA5_KEYS):
        if var == "era5_vpd":
            continue
        frame_pattern = 0.08 * _S02_CLIM[var][2] * _s02_noise_m(shape, res, 60000.0, 51, year, i)
        if var == "era5_precip_jfm":
            field = clim[var] * float(anom[var][sel].mean()) + frame_pattern
            field = np.maximum(field, 1.0)
        else:
            field = clim[var] + float(anom[var][sel].mean()) + frame_pattern
        out[var] = field.astype(np.float32)
    out["era5_rh"] = np.clip(out["era5_rh"], 5.0, 95.0).astype(np.float32)
    out["era5_swvl1"] = np.clip(out["era5_swvl1"], 0.04, 0.45).astype(np.float32)
    out["era5_evap"] = np.maximum(out["era5_evap"], 0.05).astype(np.float32)
    out["era5_wind"] = np.maximum(out["era5_wind"], 0.3).astype(np.float32)
    out["era5_vpd"] = _s02_magnus_vpd(out["era5_t2m"], out["era5_rh"])
    return {k: out[k] for k in _S02_ERA5_KEYS}

## 2.4 Urban growth and land cover (true surface, GHSL-like and GLC-like products)
Urban intensity is a probabilistic union of Gaussian kernels whose radii follow the per-centre logistic
trajectories. Land cover per year follows the v1 rules (urban core + noise + villages ⇒ built; parks/lawns inside
new urban cores; water last) with **irreversible** built-up. The GLC-like product adds a static class confusion
(cropland ↔ other vegetation, barren → other vegetation) and a small year-specific confusion, so `glc_*` agree
with GHSL `built_f` without being identical — exactly the cross-product situation of the real data.

In [ ]:
def _s02_centre_radius_km(r1990: float, r2025: float, mid: float, tau: float, year: float) -> float:
    """Logistic radius trajectory passing exactly through r1990 (1990) and r2025 (2025)."""
    s = lambda y: 1.0 / (1.0 + _s02_math.exp(-(y - mid) / tau))  # noqa: E731
    return r1990 + (r2025 - r1990) * (s(year) - s(1990)) / (s(2025) - s(1990))


def _s02_urban_intensity(ctx: dict, year: float, centres_xy: np.ndarray) -> np.ndarray:
    """Fine-grid urban intensity (0..1): windowed probabilistic union of Gaussian urban kernels."""
    xs, ys = ctx["xs_f"], ctx["ys_f"]
    neg_ys = -ys
    not_urban = np.ones((ctx["Hf"], ctx["Wf"]), np.float32)
    for (_name, _, _, amp, r90, r25, mid, tau), (cx, cy) in zip(_S02_URBAN_CENTRES, centres_xy):
        r_m = 1000.0 * _s02_centre_radius_km(r90, r25, mid, tau, year)
        reach = 4.5 * r_m
        c0, c1 = int(np.searchsorted(xs, cx - reach)), int(np.searchsorted(xs, cx + reach, side="right"))
        r0, r1 = int(np.searchsorted(neg_ys, -(cy + reach))), int(np.searchsorted(neg_ys, -(cy - reach), side="right"))
        if c0 >= c1 or r0 >= r1:
            continue
        dx = (xs[c0:c1] - cx).astype(np.float32)[None, :]
        dy = (ys[r0:r1] - cy).astype(np.float32)[:, None]
        kernel = amp * np.exp(-0.5 * (dx * dx + dy * dy) / (r_m * r_m))
        not_urban[r0:r1, c0:c1] *= 1.0 - kernel
    return 1.0 - not_urban


def _s02_mask_lc_inplace(lc: np.ndarray, roi: np.ndarray, f: int) -> None:
    """Zero land-cover pixels whose parent 1 km cell is outside the ROI (no fine mask materialised)."""
    h, w = roi.shape
    view = lc.reshape(h, f, w, f)
    view *= roi[:, None, :, None].astype(lc.dtype)


def _s02_true_landcover(static: dict, urban_lc: np.ndarray, ctx: dict, previous_built, year: float) -> np.ndarray:
    """True land cover of one year (uint8 1..6, 0 outside the ROI): rural base + irreversible urbanisation."""
    step = (year - 2010.0) / 5.0
    urban_core = urban_lc > 0.35
    if previous_built is not None:
        urban_core &= ~previous_built
    parks = urban_core & static["park"]
    lawns = urban_core & static["lawn"]
    built = (urban_lc + static["built_noise"] > 0.45) | (static["village"] > 2.10 - 0.06 * step)
    built &= ~(parks | lawns)
    built &= ~static["water"]
    if previous_built is not None:
        built |= previous_built
    lc = static["base"].copy()
    lc[built] = _S02_BUILT
    lc[lawns & ~built] = _S02_OTHER
    lc[parks & ~built] = _S02_FOREST
    lc[static["water"]] = _S02_WATER
    _s02_mask_lc_inplace(lc, ctx["roi"], ctx["f_lc"])
    return lc


def _s02_glc_product(lc_true: np.ndarray, static: dict, ctx: dict, year: int) -> np.ndarray:
    """GLC_FCS30D-like classification of the true land cover (static + year-specific confusion; uint8)."""
    lc = lc_true.copy()
    conf = static["glc_confusion"] > 1.88                                       # ~3 % persistent confusion
    lc[conf & (lc_true == _S02_CROP)] = _S02_OTHER
    lc[conf & (lc_true == _S02_OTHER)] = _S02_CROP
    lc[conf & (lc_true == _S02_BARREN)] = _S02_OTHER
    yearly = _s02_noise_m(lc.shape, ctx["r_lc"], 200.0, 60, year) > 2.33        # ~1 % year-specific confusion
    swap = yearly & ((lc == _S02_CROP) | (lc == _S02_OTHER))
    lc[swap] = np.where(lc[swap] == _S02_CROP, _S02_OTHER, _S02_CROP).astype(np.uint8)
    _s02_mask_lc_inplace(lc, ctx["roi"], ctx["f_lc"])
    return lc


def _s02_lc_block_fractions(lc: np.ndarray, f: int) -> np.ndarray:
    """(H, W, 6) fractions of classes 1..6 inside each f x f land-cover block (bincount, no one-hot copy)."""
    h, w = lc.shape[0] // f, lc.shape[1] // f
    block_of_row = (np.arange(lc.shape[0]) // f) * w
    block_of_col = np.arange(lc.shape[1]) // f
    key = (block_of_row[:, None] + block_of_col[None, :]) * 7 + lc
    counts = np.bincount(key.ravel(), minlength=h * w * 7).reshape(h, w, 7)
    return (counts[..., 1:] / float(f * f)).astype(np.float32)


def _s02_fine_fractions(lc: np.ndarray, ctx: dict) -> np.ndarray:
    """(6, Hf, Wf) area fractions of classes 1..6 on the fine grid (exact area overlap from the LC grid)."""
    out = np.empty((6, ctx["Hf"], ctx["Wf"]), np.float32)
    for k in range(6):
        out[k] = _s02_area_resample((lc == k + 1).astype(np.float32), ctx["mat_r"], ctx["mat_c"])
    np.clip(out, 0.0, 1.0, out=out)
    return out


def _s02_ghsl_built(lc_true: np.ndarray, urban_lc: np.ndarray, static: dict, ctx: dict) -> np.ndarray:
    """GHSL-like built-up surface fraction at the fine grid (from a 100 m-like built density)."""
    noise = static["ghsl_noise"]
    is_built = lc_true == _S02_BUILT
    dens = np.where(is_built, np.clip(0.42 + 0.45 * urban_lc + 0.10 * noise, 0.15, 0.95),
                    np.clip(0.03 * urban_lc + 0.015 * np.maximum(noise, 0.0), 0.0, 0.08)).astype(np.float32)
    dens[lc_true == 0] = 0.0
    dens[lc_true == _S02_WATER] = 0.0
    return np.clip(_s02_area_resample(dens, ctx["mat_r"], ctx["mat_c"]), 0.0, 1.0)


def _s02_population(ctx: dict, urban_1km: np.ndarray, built_1km: np.ndarray, water_1km: np.ndarray,
                    year: float) -> np.ndarray:
    """GHSL-POP-like density (persons/km², 1 km) renormalised to a geometric NCR total trajectory."""
    total = _S02_POP_TOTAL["1990"] * (_S02_POP_TOTAL["2025"] / _S02_POP_TOTAL["1990"]) ** ((year - 1990.0) / 35.0)
    shape, res = (ctx["H"], ctx["W"]), ctx["res"]
    rural = 420.0 * np.exp(0.3 * _s02_noise_m(shape, res, 12000.0, 70))
    raw = (rural * (1.0 + 0.2 * ctx["g"]) + 30000.0 * np.clip(urban_1km, 0, 1) ** 2.2 + 2500.0 * built_1km)
    raw = raw * (1.0 - np.clip(water_1km, 0.0, 1.0))
    roi = ctx["roi"]
    area_km2 = (res / 1000.0) ** 2
    scale = total / max(float(raw[roi].sum() * area_km2), 1.0)
    return np.maximum(raw * scale, 0.0).astype(np.float32)

## 2.5 Spectral fields and planted LST
Spectral indices are class-signature mixtures of the fine land-cover fractions with irrigation / aridity
modifiers, field-scale texture and sensor noise. The planted anomaly terms are evaluated on the fine grid; the
1 km truth is their block mean, and the frame anomaly is centred to zero mean over the ROI at 1 km.

In [ ]:
def _s02_spectral(fr: np.ndarray, built_f: np.ndarray, ctx: dict, clim_f: dict, year: int) -> dict:
    """True (surface) Landsat-like indices at the fine grid from land-cover fractions (6, Hf, Wf)."""
    shape, res_f = (ctx["Hf"], ctx["Wf"]), ctx["res_f"]
    irrig, arid = ctx["irrig_f"], ctx["arid_f"]
    crop, barren, other = fr[3], fr[4], fr[5]
    rng = _s02_rng(80, year)
    mix = {k: np.tensordot(np.asarray(v, np.float32), fr, axes=([0], [0])) for k, v in _S02_SIG.items()}
    crop_shift = float(rng.normal(0.0, 0.008))          # year-specific harvest timing
    spring_shift = float(rng.normal(0.0, 0.02))
    greening = 0.010 * (year - 2007.5) / 10.0           # slow irrigation intensification
    tex_static = _s02_noise_m(shape, res_f, 1000.0, 81)
    tex_frame = _s02_noise_m(shape, res_f, 5000.0, 82, year)
    white = lambda sd, tag: sd * _s02_rng(83, year, tag).standard_normal(shape, dtype=np.float32)  # noqa: E731

    ndvi = (mix["ndvi"] + crop * (0.20 * irrig - 0.04 * arid + crop_shift + greening * irrig) + other * 0.06 * irrig
            - 0.03 * barren * arid + 0.03 * tex_static + 0.012 * tex_frame + white(0.012, 1))
    spring = (mix["ndvi_spring"] + crop * (-0.06 * irrig - 0.15 * arid + spring_shift) + other * 0.04 * irrig
              + 0.03 * tex_static + 0.02 * _s02_noise_m(shape, res_f, 5000.0, 84, year) + white(0.012, 2))
    green_dev = ndvi - mix["ndvi"]
    ndbi = mix["ndbi"] - 0.25 * green_dev + 0.03 * built_f + 0.02 * _s02_noise_m(shape, res_f, 2000.0, 85, year) \
        + white(0.010, 3)
    mndwi = mix["mndwi"] + 0.25 * (clim_f["era5_swvl1"] - 0.18) + 0.025 * _s02_noise_m(shape, res_f, 2000.0, 86, year) \
        + white(0.010, 4)
    # Within-class surface brightness (soil colour / roof material, static, ~2.5 km patches, sd 0.018): real
    # albedo varies strongly inside every land-cover class. Without it albedo would be an almost exact linear
    # mixture of the class fractions, so the planted albedo cooling could not be separated from the barren /
    # cropland terms by any model (integrator finding: the recovered albedo effect then had the wrong sign).
    albedo = (mix["albedo"] + barren * 0.05 * arid + crop * (0.02 * (1.0 - irrig) - 0.03 * irrig)
              + 0.018 * _s02_noise_m(shape, res_f, 2500.0, 19) + 0.008 * tex_static + 0.004 * tex_frame
              + white(0.005, 5))
    return {"ndvi": np.clip(ndvi, -0.3, 0.9).astype(np.float32),
            "ndvi_spring": np.clip(spring, -0.3, 0.9).astype(np.float32),
            "ndbi": np.clip(ndbi, -0.8, 0.8).astype(np.float32),
            "mndwi": np.clip(mndwi, -0.8, 0.9).astype(np.float32),
            "albedo": np.clip(albedo, 0.03, 0.45).astype(np.float32)}


def _s02_planted_terms(mode: str, spec: dict, fr: np.ndarray, built_f: np.ndarray, bv_f: np.ndarray, geo: dict,
                       clim_f: dict, dist_water_m: np.ndarray, ctx: dict) -> dict:
    """Planted anomaly terms (°C, fine float32) keyed by SHAP group (SPEC §4.4 names)."""
    roi_f, res_f = ctx["roi_f"], ctx["res_f"]
    ndvi = spec["ndvi"]
    water_local = _s02_box_mean(fr[2], _s02_px(1000.0, res_f), valid=roi_f)
    sw = clim_f["era5_swvl1"] - float(np.nanmean(clim_f["era5_swvl1"][roi_f]))
    vpd = clim_f["era5_vpd"] - float(np.nanmean(clim_f["era5_vpd"][roi_f]))
    terms = {}
    if mode == "day":
        p = _S02_DAY
        ndvi_focal = _s02_box_mean(ndvi, _s02_px(p["park_window_m"], res_f), valid=roi_f)
        drop = spec["ndvi_spring"] - ndvi
        terms["ndvi"] = (p["ndvi_cooling_c"] * np.clip(ndvi, 0.0, p["ndvi_saturation"]) / p["ndvi_saturation"]
                         + p["park_cooling_c"] * np.clip((ndvi_focal - p["park_onset"])
                                                         / (p["park_full"] - p["park_onset"]), 0.0, 1.0))
        terms["phenology"] = p["harvest_heating_c"] * np.clip((drop - p["harvest_onset"])
                                                              / (p["harvest_full"] - p["harvest_onset"]), 0.0, 1.0)
        terms["albedo"] = p["albedo_per_unit"] * (spec["albedo"] - p["albedo_ref"])
        terms["built"] = p["built_per_unit"] * built_f
        terms["barren"] = p["barren_per_unit"] * fr[4]
        terms["water"] = (p["water_cooling_c"] * (1.0 - np.exp(-water_local / p["water_scale"]))
                          + p["water_reach_c"] * np.exp(-dist_water_m / p["water_reach_m"]))
        terms["climate_moisture"] = p["swvl1_per_unit"] * sw + p["vpd_per_kpa"] * vpd
        terms["terrain"] = p["lapse_per_m"] * (geo["elevation_f"] - p["elev_ref_m"]) + p["tpi_per_m"] * geo["tpi_f"]
    else:
        p = _S02_NIGHT
        norm = 1.0 - _s02_math.exp(-1.0 / p["built_scale"])
        terms["built"] = p["built_c"] * (1.0 - np.exp(-built_f / p["built_scale"])) / norm
        terms["built_volume"] = p["built_v_per_log1p"] * np.log1p(np.maximum(bv_f, 0.0))
        terms["water"] = p["water_warming_c"] * (1.0 - np.exp(-water_local / p["water_scale"]))
        terms["ndvi"] = p["ndvi_per_unit"] * ndvi
        terms["barren"] = p["barren_per_unit"] * fr[4]
        terms["terrain"] = p["lapse_per_m"] * (geo["elevation_f"] - p["elev_ref_m"]) + p["tpi_per_m"] * geo["tpi_f"]
        terms["climate_moisture"] = p["swvl1_per_unit"] * sw + p["vpd_per_kpa"] * vpd
    return {k: np.nan_to_num(v.astype(np.float32), nan=0.0) for k, v in terms.items()}


def _s02_planted_noise(mode: str, ctx: dict, year: int, persistent: np.ndarray) -> np.ndarray:
    """Spatially-correlated (persistent + frame) and white noise of the planted surface field (fine)."""
    shape, res_f = (ctx["Hf"], ctx["Wf"]), ctx["res_f"]
    n = _S02_NOISE
    share = n["persistent_share"]
    tag = 90 if mode == "day" else 91
    corr_sd = n["day_corr_sd"] if mode == "day" else n["night_corr_sd"]
    corr_km = n["day_corr_km"] if mode == "day" else n["night_corr_km"]
    frame = _s02_noise_m(shape, res_f, corr_km * 1000.0, tag, year)
    out = corr_sd * (np.sqrt(share) * persistent + np.sqrt(1.0 - share) * frame)
    if mode == "day":
        out += n["day_meso_sd"] * _s02_noise_m(shape, res_f, n["day_meso_km"] * 1000.0, 92, year)
    white_sd = n["day_white_sd"] if mode == "day" else n["night_white_sd"]
    out += white_sd * _s02_rng(93 if mode == "day" else 94, year).standard_normal(shape, dtype=np.float32)
    return out.astype(np.float32)

## 2.6 Observation simulation (Landsat day composites, MODIS day, night products)
**Landsat:** per WRS-2 path, the clear-scene count of a fine pixel is Binomial(scenes of its path/row, clear
probability), reduced by SLC-off gaps for the Landsat 7 share; the per-path median composite is the raw surface
temperature (truth shifted by the overpass bias) + the planted path offset + composite noise that shrinks with
the number of scenes. Then the **section-03 harmonisation** runs: the path offset is estimated as the median
146 − 147 difference over the overlap, split ±δ/2, removed, and the paths are mosaicked (mean where both).

In [ ]:
def _s02_landsat_day(year: int, raw_surface: np.ndarray, ctx: dict, wrs: dict) -> dict:
    """Simulated harmonised Landsat day composite (°C), clear-scene count and the estimated path offsets."""
    acq = _S02_DAY_ACQ[year]
    shape, res_f, roi_f = (ctx["Hf"], ctx["Wf"]), ctx["res_f"], ctx["roi_f"]
    counts = dict(zip(_S02_PATHROWS, acq["per_pr"]))
    rng = _s02_rng(100, year)
    clear = np.clip(0.86 + 0.05 * _s02_noise_m(shape, res_f, 15000.0, 101, year), 0.5, 0.97)
    blocked = _s02_noise_m(shape, res_f, 1500.0, 102, year) > 2.9            # persistent haze / QA holes (~0.2 %)
    planted = {"146": 0.5 * acq["path_delta"], "147": -0.5 * acq["path_delta"]}
    n = _S02_NOISE
    comps, nobs = {}, {}
    for path in ("146", "147"):
        rows = wrs["rows"][path]
        n_scenes = np.zeros(shape, np.int32)
        for row in (39, 40, 41):
            key = f"{path}/{row:03d}"
            if key in counts:
                n_scenes[rows == row] = counts[key]
        p_clear = clear * (1.0 - acq["slc_share"] * wrs["slc_gap"][path])
        covered = wrs["cover"][path] & roi_f & ~blocked
        n_obs = np.where(covered, rng.binomial(n_scenes, p_clear), 0).astype(np.int32)
        sd = n["landsat_white_base"] + n["landsat_white_per_sqrt_n"] / np.sqrt(np.maximum(n_obs, 1))
        scene_corr = n["landsat_scene_corr"] / np.sqrt(np.maximum(n_scenes, 1))
        err = (sd * _s02_rng(103, year, int(path)).standard_normal(shape, dtype=np.float32)
               + scene_corr * _s02_noise_m(shape, res_f, 6000.0, 104, year, int(path)))
        comps[path] = np.where(n_obs > 0, raw_surface + planted[path] + err, np.nan).astype(np.float32)
        nobs[path] = n_obs
    both = np.isfinite(comps["146"]) & np.isfinite(comps["147"])
    if both.sum() >= 50:
        delta = float(np.median((comps["146"] - comps["147"])[both]))
    else:
        log(f"02: frame {year}: path overlap has only {int(both.sum())} pixels; path offset set to 0", "WARNING")
        delta = 0.0
    h146, h147 = comps["146"] - 0.5 * delta, comps["147"] + 0.5 * delta
    mosaic = np.where(both, 0.5 * (h146 + h147), np.where(np.isfinite(h146), h146, h147)).astype(np.float32)
    count = (nobs["146"] + nobs["147"]).astype(np.float32)
    mosaic[count <= 0] = np.nan
    return {"lst": mosaic, "nobs": count, "path_offsets": {"146": round(0.5 * delta, 4), "147": round(-0.5 * delta, 4)},
            "planted_offsets": planted, "overlap_pixels": int(both.sum())}


def _s02_terra_drift(year: int) -> float:
    """MODIS Terra day bias (°C) from the earlier drifted overpass (0 before 2022)."""
    h = _S02_TERRA_DAY_IST.get(int(year), 11.12)
    return -(11.12 - h) * _S02_TERRA_DAY_RATE


def _s02_aqua_drift(year: int) -> float:
    """MODIS Aqua night bias (°C) from the later drifted overpass (0 up to 2022)."""
    h = _S02_AQUA_NIGHT_IST.get(int(year), 2.20)
    return -(h - 2.20) * _S02_NIGHT_COOLING_RATE


def _s02_ols(x: np.ndarray, y: np.ndarray):
    """Simple least squares y = a + b x -> (a, b)."""
    xm, ym = float(x.mean()), float(y.mean())
    var = float(((x - xm) ** 2).sum())
    b = float(((x - xm) * (y - ym)).sum() / var) if var > 0 else 1.0
    return ym - b * xm, b


def _s02_fit_calibration(src: np.ndarray, aqua: np.ndarray, cls: np.ndarray):
    """Per-class linear maps aqua = a + b*src, overall fit statistics and shrunk per-cell mean residuals.

    ``src``/``aqua``/``cls`` are (years, cells). Classes with < 30 points use the pooled map (recorded as such).
    """
    ok = np.isfinite(src) & np.isfinite(aqua)
    pooled = _s02_ols(src[ok], aqua[ok])
    by_class, pred = [], np.full(src.shape, np.nan)
    for c in range(1, 7):
        sel = ok & (cls == c)
        n_c = int(sel.sum())
        a, b = _s02_ols(src[sel], aqua[sel]) if n_c >= 30 else pooled
        by_class.append({"class": c, "a": round(float(a), 4), "b": round(float(b), 4), "n": n_c})
        pred[sel] = a + b * src[sel]
    resid = np.where(ok, aqua - pred, np.nan)
    sse = float(np.nansum(resid ** 2))
    sst = float(((aqua[ok] - aqua[ok].mean()) ** 2).sum())
    n_years = np.isfinite(resid).sum(axis=0)
    mean_resid = np.where(n_years > 0, np.nansum(resid, axis=0) / np.maximum(n_years, 1), 0.0)
    shrunk = mean_resid * n_years / (n_years + 3.0)
    summary = {"r2": round(1.0 - sse / sst, 4) if sst > 0 else None, "rmse": round(float(np.sqrt(sse / ok.sum())), 4),
               "n": int(ok.sum()), "by_class": by_class}
    coef = {d["class"]: (d["a"], d["b"]) for d in by_class}
    return summary, coef, shrunk.astype(np.float32)


def _s02_apply_calibration(src: np.ndarray, cls: np.ndarray, coef: dict, shrunk: np.ndarray) -> np.ndarray:
    """Aqua-equivalent values from source values with the fitted class maps + shrunk cell residuals."""
    a = np.zeros(src.shape, np.float64)
    b = np.ones(src.shape, np.float64)
    for c, (ac, bc) in coef.items():
        a = np.where(cls == c, ac, a)
        b = np.where(cls == c, bc, b)
    return (a + b * src + shrunk).astype(np.float32)


def _s02_planted_src(true_aqua: np.ndarray, cls: np.ndarray, kind: str) -> np.ndarray:
    """Invert the planted map aqua = a + b*src to obtain the source sensor's value (noise-free)."""
    out = np.array(true_aqua, np.float64, copy=True)
    for c, (a, b) in _S02_CAL_PLANTED[kind].items():
        sel = cls == c
        out[sel] = (true_aqua[sel] - a) / b
    return out

## 2.7 Future inputs (illustrative in synthetic mode)
* `cmip6`: 34 pseudo-GCMs (`pGCM-01…34`, deliberately not real model names) with an ROI-mean Apr–Jun tasmax /
  tasmin for the 2025 baseline window and 2030–2050; deltas follow `k_g · Δ̄_s(y)` with a model sensitivity
  `k_g ~ N(1, 0.3)` shared across scenarios plus scenario-specific spread (2050 vs 2025: ssp245 ≈ 1.1 ± 0.4 K,
  ssp585 ≈ 1.7 ± 0.5 K; tasmin slightly larger).
* `urban_growth`: Chen-2020-like decadal urban-area ratios vs 2020 per district (peri-urban districts fastest,
  SSP1 slower, SSP5 faster), geometrically interpolated to 5-year frames.
* `ghsl_2030`: the generator's own 2030 extrapolation; `pop_growth`: SSP urban-population factors vs 2025;
  `gsat_increment`: AR6 WG1 Table 4.5 central values (vs 1995–2014) interpolated to the frame windows and
  expressed relative to the 2016–2035 window.

In [ ]:
def _s02_future_cmip6(clim_tmax_mean: float, clim_tmin_mean: float, anom: dict, years: np.ndarray) -> pd.DataFrame:
    """Pseudo-GCM ROI-mean Apr-Jun tasmax/tasmin (K) for the 2025 baseline and the projection frames."""
    rng = _s02_rng(120)
    p = _S02_CMIP6
    frames = [int(_s02_cfg("NOW", 2025))] + [int(y) for y in _s02_cfg("FRAMES_PROJ", [2030, 2035, 2040, 2045, 2050])]
    base_year = frames[0]
    k = rng.normal(1.0, p["model_sd"], p["n_models"])
    k = 1.0 + (k - k.mean()) / max(k.std(), 1e-9) * p["model_sd"]
    sel = (years >= base_year - 9) & (years <= base_year + 10)
    base = {"tasmax": clim_tmax_mean + float(anom["era5_tmax"][sel].mean()) + 273.15,
            "tasmin": clim_tmin_mean + float(anom["era5_tmin"][sel].mean()) + 273.15}
    rows = []
    for m in range(p["n_models"]):
        model = f"pGCM-{m + 1:02d}"
        bias = {v: float(rng.normal(0.0, p["bias_sd_k"])) for v in ("tasmax", "tasmin")}
        for scenario in ("ssp245", "ssp585"):
            k_s = k[m] + float(rng.normal(0.0, p["idio_sd"][scenario]))
            for var in ("tasmax", "tasmin"):
                for frame in frames:
                    frac = max(0.0, (frame - base_year) / 25.0) ** p["shape_exponent"][scenario]
                    delta = k_s * p["delta_2050"][scenario][var] * frac
                    noise = float(rng.normal(0.0, p["window_noise_k"])) if frame != base_year else 0.0
                    scen_offset = 0.03 if (scenario == "ssp585" and frame == base_year) else 0.0
                    rows.append((scenario, model, int(frame), var,
                                 round(base[var] + bias[var] + scen_offset + delta + noise, 4)))
    return pd.DataFrame(rows, columns=["scenario", "model", "frame", "var", "value_k"])


def _s02_future_pattern(ctx: dict) -> dict:
    """Smooth ensemble-median 2050 warming pattern / ROI mean (≈ 1 ± 0.05), NaN outside the ROI."""
    shape, res, roi, g = (ctx["H"], ctx["W"]), ctx["res"], ctx["roi"], ctx["g"]
    out = {}
    for i, (var, slope) in enumerate((("tasmax", -0.04), ("tasmin", 0.03))):
        field = 1.0 + slope * g + 0.015 * _s02_noise_m(shape, res, 50000.0, 121, i)
        field = field / float(field[roi].mean())
        out[var] = np.where(roi, field, np.nan).astype(np.float32)
    return out


def _s02_future_urban(ctx: dict) -> pd.DataFrame:
    """Chen-2020-like urban-area ratio vs 2020 per (scenario, district, frame), decades interpolated geometrically."""
    frames = sorted({2020, int(_s02_cfg("NOW", 2025))} | {int(y) for y in _s02_cfg("FRAMES_PROJ", [2030, 2035, 2040,
                                                                                                    2045, 2050])})
    decades = list(range(2020, max(frames) + 10, 10))
    rows = []
    present = set(np.unique(ctx["cell_district"]).tolist())
    for scenario, mult in _S02_SSP_URBAN_MULT.items():
        for d in DISTRICTS:
            did = int(d["id"])
            if did not in present:
                continue
            r0 = _S02_URBAN_RATE.get(did, 1.5) / 100.0 * mult
            # ratio(t) = exp(∫ r0 exp(-(s-2020)/60) ds) - saturating growth
            dec = {t: _s02_math.exp(r0 * 60.0 * (1.0 - _s02_math.exp(-(t - 2020) / 60.0))) for t in decades}
            for frame in frames:
                if frame in dec:
                    ratio = dec[frame]
                else:
                    lo = max(t for t in decades if t <= frame)
                    hi = min(t for t in decades if t >= frame)
                    w = (frame - lo) / (hi - lo)
                    ratio = dec[lo] ** (1.0 - w) * dec[hi] ** w
                rows.append((scenario, did, int(frame), round(float(ratio), 5)))
    return pd.DataFrame(rows, columns=["scenario", "district", "frame", "ratio"])


def _s02_future_pop_growth() -> pd.DataFrame:
    """NCR urban-population factor vs 2025 per SSP (compound growth reaching the 2050 factor)."""
    now = int(_s02_cfg("NOW", 2025))
    frames = [now] + [int(y) for y in _s02_cfg("FRAMES_PROJ", [2030, 2035, 2040, 2045, 2050])]
    rows = []
    for scenario, f2050 in _S02_POP_FACTOR_2050.items():
        rate = _s02_math.log(f2050) / 25.0
        for frame in frames:
            rows.append((scenario, int(frame), round(_s02_math.exp(rate * (frame - now)), 5)))
    return pd.DataFrame(rows, columns=["scenario", "frame", "factor"])


def _s02_future_gsat() -> dict:
    """GSAT increment (K) of each frame window over the 2016-2035 window, from AR6 Table 4.5 central values."""
    now = int(_s02_cfg("NOW", 2025))
    frames = [now] + [int(y) for y in _s02_cfg("FRAMES_PROJ", [2030, 2035, 2040, 2045, 2050])]

    def vs_1995_2014(near: float, mid: float, centre: float) -> float:
        # piecewise-linear in window centre: 0 at 2004.5 (1995-2014), near at 2030.5, mid at 2050.5
        if centre <= 2030.5:
            return near * (centre - 2004.5) / 26.0
        return near + (mid - near) * (centre - 2030.5) / 20.0

    out = {}
    for scenario, (near, mid) in _S02_AR6_GSAT.items():
        base = vs_1995_2014(near, mid, now + 0.5)
        out[scenario] = {int(fr): round(vs_1995_2014(near, mid, fr + 0.5) - base, 4) for fr in frames}
    return out

## 2.8 Generator orchestration
One chronological pass over the land-cover years (1990 … 2025, plus 2022 for the GLC 2025 band and 2030 for the
GHSL-2030 extrapolation). Each observation frame is processed completely inside the pass and only its RAW
arrays are kept. Night calibration and annual series follow, as they need every frame's anomaly maps.

In [ ]:
def _s02_generate() -> dict:
    """Build the synthetic world; returns every SPEC §3.2 global plus QA payloads (private)."""
    t_start = _s02_time.perf_counter()
    ctx = _s02_context()
    frames = sorted(int(y) for y in _s02_cfg("FRAMES_OBS", list(_S02_DEFAULT_WINDOWS)))
    unknown = [y for y in frames if y not in _S02_DAY_ACQ]
    if unknown:
        raise ValueError(f"section 02 has no synthetic acquisition table for frames {unknown}")
    windows = {int(k): tuple(int(v) for v in vals)
               for k, vals in dict(_s02_cfg("WINDOWS", _S02_DEFAULT_WINDOWS)).items()}
    for y in frames:
        windows.setdefault(y, _S02_DEFAULT_WINDOWS[y])
    ref_period = tuple(int(v) for v in _s02_cfg("REFERENCE_PERIOD", (2000, 2020)))
    roi, roi_f, f = ctx["roi"], ctx["roi_f"], ctx["f"]
    h, w, hf, wf = ctx["H"], ctx["W"], ctx["Hf"], ctx["Wf"]
    log(f"02: grid {w}x{h} @ {ctx['res']} m | fine {wf}x{hf} @ {ctx['res_f']} m | land cover "
        f"{ctx['Wl']}x{ctx['Hl']} @ {ctx['r_lc']} m | ROI {int(roi.sum()):,} cells | frames {frames}")

    # ---- static world ---------------------------------------------------------------------------------------------
    geo = _s02_geography(ctx)
    static = _s02_static_landcover(ctx, geo)
    wrs = _s02_landsat_geometry(ctx)
    occ_f = np.clip(_s02_area_resample(static["occurrence"], ctx["mat_r"], ctx["mat_c"]), 0.0, 1.0)
    clim = _s02_climatology(ctx, geo)
    centres_xy = _s02_lonlat_to_xy([(c[1], c[2]) for c in _S02_URBAN_CENTRES])
    persistent = {m: _s02_noise_m((hf, wf), ctx["res_f"], 1000.0 * _S02_NOISE[f"{m}_corr_km"], 95, i)
                  for i, m in enumerate(("day", "night"))}
    # Building typology (static, ~2.5 km patches, log-sd 0.45): towers vs dense low-rise at equal built-up share,
    # as in GHSL built_volume vs built_surface. Without it built_v was a monotone function of built_f (Spearman
    # 1.00 at 1 km), so the planted built / built_volume night terms were not separately identifiable.
    height_typology = np.exp(0.45 * _s02_noise_m((hf, wf), ctx["res_f"], 2500.0, 18)).astype(np.float32)
    log(f"02: static world ready in {_s02_time.perf_counter() - t_start:.1f}s")

    def fine_out(a):
        return np.where(roi_f, a, np.nan).astype(np.float32)

    def coarse_out(a):
        return np.where(roi, a, np.nan).astype(np.float32)

    raw_static = {}
    for key in ("elevation", "slope", "aspect_sin", "aspect_cos"):
        raw_static[key] = coarse_out(_s02_block_mean(geo[f"{key}_f"], f)[0])
    raw_static["water_occ"] = coarse_out(_s02_block_mean(occ_f, f)[0])
    raw_static["elevation_f"] = fine_out(geo["elevation_f"])
    raw_static["water_occ_f"] = fine_out(occ_f)

    # ---- climate and levels ---------------------------------------------------------------------------------------
    years = np.arange(1985, 2031)
    anom = _s02_annual_climate(years)
    lev_ann = _s02_levels(years, anom, ref_period)

    def window_mean(series, window):
        sel = (years >= window[0]) & (years <= window[1])
        return float(series[sel].mean())

    levels = {m: {y: window_mean(lev_ann[m], windows[y]) for y in frames} for m in ("day", "night")}

    # ---- chronological land-cover pass ----------------------------------------------------------------------------
    lc_years = sorted(set(range(1990, 2026, 5)) | {2022, 2030} | set(frames))
    lc_years = [y for y in lc_years if y <= max(max(frames), 2030)]
    raw_obs, lc_fine, frame_meta = {}, {}, {}
    day_anom, night_anom, dominant = {}, {}, {}
    planted_effects = {"day": {}, "night": {}}
    qa = {"night_truth_fine": {}, "built_share": {}, "uhi": {"day": {}, "night": {}}, "ndvi_support": {}}
    previous_built, previous_ghsl, glc_by_year = None, None, {}
    ghsl_2030 = None
    dist_water_m = None
    for year in lc_years:
        t0 = _s02_time.perf_counter()
        urban_f = _s02_urban_intensity(ctx, float(year), centres_xy)
        urban_lc = _s02_bilinear(urban_f, ctx["res_f"], ctx["r_lc"])
        lc_true = _s02_true_landcover(static, urban_lc, ctx, previous_built, float(year))
        previous_built = lc_true == _S02_BUILT
        glc_by_year[year] = _s02_glc_product(lc_true, static, ctx, year)
        built_f = _s02_ghsl_built(lc_true, urban_lc, static, ctx)
        if previous_ghsl is not None:
            np.maximum(built_f, previous_ghsl, out=built_f)        # GHSL built-up surface never decreases
        previous_ghsl = built_f
        vert = 1.0 / (1.0 + _s02_math.exp(-(year - 2008.0) / 5.0))
        height_f = (3.0 + 4.5 * np.clip(urban_f, 0.0, 1.0) ** 1.6 * (1.0 + 0.6 * vert)) * height_typology
        bv_f = (built_f * height_f).astype(np.float32)               # m3 of building per m2 of ground
        del urban_lc
        if year == 2030:
            urban_1km = _s02_block_mean(urban_f, f)[0]
            built_1km = _s02_block_mean(built_f, f)[0]
            water_1km = _s02_lc_block_fractions(glc_by_year[2022 if 2022 in glc_by_year else year], ctx["f_lc"])[..., 2]
            ghsl_2030 = {"built": coarse_out(built_1km),
                         "pop": coarse_out(_s02_population(ctx, urban_1km, built_1km, water_1km, 2030.0))}
            del lc_true, urban_f, built_f, bv_f, height_f
            continue
        if year not in frames:
            del lc_true, urban_f, bv_f, height_f
            continue

        # ---- one observation frame --------------------------------------------------------------------------------
        glc_year = 2022 if (year == 2025 and 2022 in glc_by_year) else year
        glc_map = glc_by_year[glc_year]
        lc_fine[year] = glc_map
        glc_frac = _s02_lc_block_fractions(glc_map, ctx["f_lc"])
        fr = _s02_fine_fractions(lc_true, ctx)
        if dist_water_m is None:                                     # water is static: compute once
            water_mask = (fr[2] >= 0.3) | (occ_f >= 0.5)
            dist_water_m = (_s02_ndi.distance_transform_edt(~water_mask) * ctx["res_f"]).astype(np.float32) \
                if water_mask.any() else np.full((hf, wf), 1e6, np.float32)
        clim_1km = _s02_frame_climate(ctx, clim, anom, years, year, windows[year])
        clim_f = {k: _s02_bilinear(clim_1km[k], ctx["res"], ctx["res_f"]) for k in ("era5_swvl1", "era5_vpd")}
        spec = _s02_spectral(fr, built_f, ctx, clim_f, year)

        for mode in ("day", "night"):
            terms = _s02_planted_terms(mode, spec, fr, built_f, bv_f, geo, clim_f, dist_water_m, ctx)
            total = sum(terms.values()) + _s02_planted_noise(mode, ctx, year, persistent[mode])
            block, _ = _s02_block_mean(total, f, valid=roi_f)
            mu = float(np.nanmean(block[roi]))
            anom_1km = (block - mu).astype(np.float32)
            (day_anom if mode == "day" else night_anom)[year] = np.where(roi, anom_1km, np.nan).astype(np.float32)
            planted_effects[mode][year] = {}
            for name, term in terms.items():
                tb = _s02_block_mean(term, f, valid=roi_f)[0][roi]
                planted_effects[mode][year][name] = float(np.nanmean(np.abs(tb - np.nanmean(tb))))
            if mode == "day":
                truth_day_f = (levels["day"][year] + total - mu).astype(np.float32)
            elif year in (frames[0], frames[-1]):
                qa["night_truth_fine"][year] = fine_out(levels["night"][year] + total - mu)
            del terms, total, block

        # Landsat composite: the truth seen at the scene overpass instead of 10:45 IST (cooler for early scenes).
        acq = _S02_DAY_ACQ[year]
        overpass_bias = -(_S02_REF_OVERPASS_IST_H - acq["overpass"]) * acq["heating"]
        ls = _s02_landsat_day(year, truth_day_f + overpass_bias, ctx, wrs)
        del truth_day_f
        valid_f = np.isfinite(ls["lst"])

        def sensor(a, sd, tag, _year=year):
            """Field + white sensor noise of a fixed stream (110, year, tag)."""
            return a + sd * _s02_rng(110, _year, tag).standard_normal(a.shape, dtype=np.float32)

        obs = {"lst_day_f": fine_out(ls["lst"]), "lst_day_nobs_f": fine_out(ls["nobs"])}
        for tag, (key, src, sd) in enumerate((("ndvi_f", "ndvi", 0.006), ("ndbi_f", "ndbi", 0.006),
                                              ("mndwi_f", "mndwi", 0.006), ("albedo_f", "albedo", 0.003))):
            obs[key] = fine_out(np.where(valid_f, sensor(spec[src], sd, tag), np.nan))
        # Feb-Mar composite: its own small gaps, and half of the pixels without any clear Apr-Jun scene.
        spring_gap = ((_s02_noise_m((hf, wf), ctx["res_f"], 1000.0, 111, year) > 3.0)
                      | ((ls["nobs"] <= 0) & (_s02_rng(112, year).random((hf, wf)) < 0.5)))
        obs["ndvi_spring_f"] = fine_out(np.where(spring_gap, np.nan, sensor(spec["ndvi_spring"], 0.006, 9)))
        obs["built_f"] = fine_out(built_f)

        # 1 km products
        built_1km = _s02_block_mean(built_f, f)[0]
        urban_1km = _s02_block_mean(urban_f, f)[0]
        bv_1km = _s02_block_mean(bv_f, f)[0]
        pop = _s02_population(ctx, urban_1km, built_1km, glc_frac[..., 2], float(year))
        coarse = {"lst_night": np.full((h, w), np.nan, np.float32), "lst_night_nobs": coarse_out(np.zeros((h, w))),
                  "built_v": coarse_out(bv_1km), "pop": coarse_out(pop)}
        if year >= 2000:
            drift_years = [y for y in range(windows[year][0], windows[year][1] + 1) if y >= 2000]
            drift = float(np.mean([_s02_terra_drift(y) for y in drift_years]))
            modis = (levels["day"][year] + _S02_TERRA_DAY_OFFSET_C + drift + 0.9 * np.nan_to_num(day_anom[year])
                     + _S02_NOISE["modis_day_corr"] * _s02_noise_m((h, w), ctx["res"], 3000.0, 113, year)
                     + _S02_NOISE["modis_day_white"] * _s02_rng(114, year).standard_normal((h, w)))
            modis[_s02_rng(115, year).random((h, w)) < 0.002] = np.nan
            coarse["lst_day_modis"] = coarse_out(modis)
        else:
            coarse["lst_day_modis"] = np.full((h, w), np.nan, np.float32)
        for k, key in enumerate(_S02_GLC_KEYS):
            coarse[key] = coarse_out(glc_frac[..., k])
        for key in _S02_ERA5_KEYS:
            coarse[key] = coarse_out(clim_1km[key])
        raw_obs[year] = {**obs, **coarse}
        dominant[year] = (np.argmax(glc_frac, axis=-1) + 1).astype(np.uint8)

        # Section-06-style 1 km fractions (for the realised UHI and records).
        glc_b = glc_frac[..., 0]
        denom = np.where(glc_b >= 0.999, np.inf, 1.0 - glc_b)
        frac_water = glc_frac[..., 2] / denom * (1.0 - built_1km)
        urban_cells = roi & (built_1km >= 0.5)
        rural_cells = roi & (built_1km <= 0.05) & (frac_water < 0.2)
        for mode, amap in (("day", day_anom[year]), ("night", night_anom[year])):
            qa["uhi"][mode][year] = (round(float(np.nanmean(amap[urban_cells]) - np.nanmean(amap[rural_cells])), 3)
                                     if urban_cells.sum() >= 3 and rural_cells.sum() >= 3 else None)
        ndvi_1km = _s02_block_mean(obs["ndvi_f"], f)[0][roi]
        qa["ndvi_support"][year] = [float(v) for v in np.nanpercentile(ndvi_1km, [5, 25, 50, 75, 95])]
        qa["built_share"][year] = float(np.nanmean(built_1km[roi]))
        frame_meta[year] = {
            "window": tuple(windows[year]),
            "day": {"sensors": list(acq["sensors"]), "scenes": int(acq["scenes"]),
                    "scenes_per_pathrow": {pr: int(n) for pr, n in zip(_S02_PATHROWS, acq["per_pr"])},
                    "mean_overpass_ist_h": float(acq["overpass"]),
                    "coverage": round(float(np.isfinite(obs["lst_day_f"][roi_f]).mean()), 4),
                    "heating_rate_k_per_h": float(acq["heating"]), "path_offsets": dict(ls["path_offsets"])},
            "night": None,                                            # filled after the calibration step
        }
        qa.setdefault("path_offsets", {})[year] = {"planted": ls["planted_offsets"], "estimated": ls["path_offsets"],
                                                   "overlap_pixels": ls["overlap_pixels"]}
        log(f"02: frame {year} done in {_s02_time.perf_counter() - t0:.1f}s (built share "
            f"{qa['built_share'][year]:.3f}, day UHI {qa['uhi']['day'][year]}, night UHI {qa['uhi']['night'][year]})")
        del lc_true, fr, spec, ls, obs, coarse, urban_f, built_f, bv_f, height_f, clim_f, valid_f, spring_gap
    del static, previous_built, previous_ghsl, glc_by_year, persistent, dist_water_m, height_typology
    free_memory()

    # ---- night products, calibration and annual series ------------------------------------------------------------
    night = _s02_night_products(ctx, frames, levels, lev_ann, years, night_anom, day_anom, dominant, anom, clim)
    for year in frames:
        raw_obs[year]["lst_night"] = night["fields"][year]
        raw_obs[year]["lst_night_nobs"] = night["nobs"][year]
        source, sensor_name, overpass, composites = _S02_NIGHT_ACQ[year]
        frame_meta[year]["night"] = {
            "source": source, "sensor": sensor_name, "overpass_ist_h": overpass, "composites": int(composites),
            "coverage": round(float(np.isfinite(night["fields"][year][roi]).mean()), 4)}

    future = {"cmip6": _s02_future_cmip6(float(np.mean(clim["era5_tmax"][roi])), float(np.mean(clim["era5_tmin"][roi])),
                                         anom, years),
              "cmip6_pattern": _s02_future_pattern(ctx), "urban_growth": _s02_future_urban(ctx),
              "ghsl_2030": ghsl_2030, "pop_growth": _s02_future_pop_growth(), "gsat_increment": _s02_future_gsat()}

    truth = _s02_truth(frames, windows, levels, lev_ann, years, anom, ref_period, qa, planted_effects, night, future)
    qa["levels"] = levels
    qa["lev_ann"] = {m: dict(zip(years.tolist(), v.tolist())) for m, v in lev_ann.items()}
    log(f"02: synthetic world generated in {_s02_time.perf_counter() - t_start:.1f}s")
    return {"RAW_STATIC": raw_static, "RAW_OBS": raw_obs, "LC_FINE": lc_fine, "FRAME_META": frame_meta,
            "NIGHT_CALIBRATION": night["calibration"], "ANNUAL_SERIES": night["annual"], "FUTURE_INPUTS": future,
            "SYNTHETIC_TRUTH": truth, "_qa": qa}

## 2.9 Night products and the annual series
Per 1 km cell and year (2000–2025) the true Aqua-equivalent night LST is the annual level + the time-interpolated
frame anomaly + annual noise. Aqua is observed with noise; Terra (22:30) and VIIRS values are generated by
inverting the planted class maps (+ sensor noise) and **calibrated back** with maps fitted on the overlap years
(Terra↔Aqua 2003–2019, VIIRS↔Aqua 2013–2021), per dominant GLC class, plus the per-cell mean residual shrunk
with n/(n+3) — exactly the section-04 procedure. Frames: 2000 = calibrated Terra, 2005–2020 = Aqua,
2025 = calibrated VIIRS, 1990/1995 = no product.

In [ ]:
def _s02_interp_anomaly(frame_maps: dict, frames: list, year: int) -> np.ndarray:
    """Linear interpolation in time between frame anomaly maps (clamped outside the frame range)."""
    if year <= frames[0]:
        return frame_maps[frames[0]]
    if year >= frames[-1]:
        return frame_maps[frames[-1]]
    hi = min(y for y in frames if y >= year)
    lo = max(y for y in frames if y <= year)
    if hi == lo:
        return frame_maps[lo]
    wgt = (year - lo) / (hi - lo)
    return (1.0 - wgt) * frame_maps[lo] + wgt * frame_maps[hi]


def _s02_scope_means(values: np.ndarray, district: np.ndarray, n_districts: int) -> dict:
    """{"ncr": mean, "d:<id>": mean} of a per-cell vector (NaN-aware; districts without cells omitted)."""
    ok = np.isfinite(values)
    out = {"ncr": float(values[ok].mean()) if ok.any() else np.nan}
    sums = np.bincount(district[ok], weights=values[ok], minlength=n_districts)
    counts = np.bincount(district[ok], minlength=n_districts)
    for d in range(n_districts):
        if counts[d] > 0:
            out[f"d:{d}"] = float(sums[d] / counts[d])
    return out


def _s02_night_products(ctx, frames, levels, lev_ann, years, night_anom, day_anom, dominant, anom, clim) -> dict:
    """Night frame fields/counts, fitted NIGHT_CALIBRATION and the ANNUAL_SERIES DataFrame."""
    roi, cells, district = ctx["roi"], ctx["cells"], ctx["cell_district"]
    n_d = ctx["n_districts"]
    h, w = ctx["H"], ctx["W"]
    nc = cells.size
    noise = _S02_NOISE
    rng = _s02_rng(130)
    ann_years = np.arange(2000, 2026)
    yidx = {int(y): i for i, y in enumerate(years)}
    night_maps = {y: night_anom[y].ravel()[cells] for y in frames}
    day_maps = {y: day_anom[y].ravel()[cells] for y in frames}
    dom_maps = {y: dominant[y].ravel()[cells] for y in frames}

    def nearest_frame(y):
        return min(frames, key=lambda fy: (abs(fy - y), -fy))

    true_n = np.empty((ann_years.size, nc))
    cls = np.empty((ann_years.size, nc), np.uint8)
    for i, y in enumerate(ann_years):
        true_n[i] = (lev_ann["night"][yidx[int(y)]] + _s02_interp_anomaly(night_maps, frames, int(y))
                     + noise["annual_cell_sd"] * rng.standard_normal(nc))
        cls[i] = dom_maps[nearest_frame(int(y))]
    aqua_obs = true_n + noise["aqua_night_annual"] * rng.standard_normal(true_n.shape)
    calibration, applied = {}, {}
    expected = {"terra_to_aqua": {}, "viirs_to_aqua": {}}
    for kind, sd in (("terra_to_aqua", noise["terra_night_annual"]), ("viirs_to_aqua", noise["viirs_night_annual"])):
        src = _s02_planted_src(true_n, cls, kind) + sd * rng.standard_normal(true_n.shape)
        y0, y1 = _S02_CAL_FIT_YEARS[kind]
        fit = (ann_years >= y0) & (ann_years <= y1)
        summary, coef, shrunk = _s02_fit_calibration(src[fit], aqua_obs[fit], cls[fit])
        calibration[kind] = summary
        # Errors-in-variables: OLS slopes are attenuated by the reliability var(true src)/var(observed src).
        src_true = _s02_planted_src(true_n[fit], cls[fit], kind)
        for c, (_, b_planted) in _S02_CAL_PLANTED[kind].items():
            sel = cls[fit] == c
            if sel.sum() >= 30:
                var_true = float(np.var(src_true[sel]))
                expected[kind][c] = round(b_planted * var_true / (var_true + sd * sd), 4)
        applied[kind] = {"coef": coef, "shrunk": shrunk, "src": src}

    fields, nobs = {}, {}
    for year in frames:
        source = _S02_NIGHT_ACQ[year][0]
        field = np.full(h * w, np.nan, np.float32)
        count = np.zeros(h * w, np.float32)
        truth = levels["night"][year] + night_maps[year]
        frng = _s02_rng(131, year)
        if source == "aqua":
            vals = truth + noise["aqua_frame_white"] * frng.standard_normal(nc)
            n = frng.binomial(36, np.clip(0.9 + 0.04 * frng.standard_normal(nc), 0.5, 1.0))
        elif source in ("terra_cal", "viirs_cal"):
            kind = "terra_to_aqua" if source == "terra_cal" else "viirs_to_aqua"
            n_years = 2 if source == "terra_cal" else 3
            sd = noise["terra_night_annual" if source == "terra_cal" else "viirs_night_annual"] / np.sqrt(n_years)
            src = _s02_planted_src(truth, dom_maps[year], kind) + sd * frng.standard_normal(nc)
            vals = _s02_apply_calibration(src, dom_maps[year], applied[kind]["coef"], applied[kind]["shrunk"])
            n = (frng.binomial(24, 0.88, nc) if source == "terra_cal"
                 else frng.binomial(273, np.clip(0.55 + 0.05 * frng.standard_normal(nc), 0.2, 0.9)))
        else:
            vals, n = None, np.zeros(nc)
        if vals is not None:
            gap = frng.random(nc) < 0.002                       # persistent cloud / QA-masked cells
            low = frng.random(nc) < 0.003                       # cells with too few valid composites
            n = np.where(low, np.minimum(n, 4 if source != "viirs_cal" else 12), n)
            n = np.where(gap, 0, n)
            vals = np.where(n > 0, vals, np.nan)
            field[cells] = vals
        count[cells] = n
        fields[year] = np.where(roi.ravel(), field, np.nan).reshape(h, w).astype(np.float32)
        nobs[year] = np.where(roi.ravel(), count, np.nan).reshape(h, w).astype(np.float32)

    # ---- annual series (SPEC §2.2) -------------------------------------------------------------------------------
    rows = []
    clim_cells = {v: clim[v].ravel()[cells].astype(np.float64) for v in
                  ("era5_skin_day", "era5_skin_night", "era5_tmax", "era5_tmin", "era5_swvl1")}
    clim_scope = {v: _s02_scope_means(vals, district, n_d) for v, vals in clim_cells.items()}
    scopes = list(clim_scope["era5_tmax"].keys())
    drng = _s02_rng(132)
    scope_sd = {"era5_skin_day": 0.12, "era5_skin_night": 0.08, "era5_tmax": 0.10, "era5_tmin": 0.08,
                "era5_swvl1": 0.004}
    for y in range(1990, 2026):
        i = yidx[y]
        for v in clim_cells:
            for s in scopes:
                dev = 0.0 if s == "ncr" else float(drng.normal(0.0, scope_sd[v]))
                rows.append((y, s, v, clim_scope[v][s] + float(anom[v][i]) + dev, None))
    sat_rng = _s02_rng(133)
    for i, y in enumerate(ann_years):
        y = int(y)
        flag_drift = "drift" if y >= _S02_DRIFT_FROM else None
        # MODIS Terra day (QC + view-time filtered): Landsat-like level + MODIS offset + drift, anomaly x 0.9.
        day_cells = (lev_ann["day"][yidx[y]] + _S02_TERRA_DAY_OFFSET_C + _s02_terra_drift(y)
                     + 0.9 * _s02_interp_anomaly(day_maps, frames, y))
        for s, val in _s02_scope_means(day_cells, district, n_d).items():
            rows.append((y, s, "modis_terra_day", val + float(sat_rng.normal(0.0, noise["annual_scope_sd_day"])),
                         flag_drift))
        if y >= 2003:
            aqua_cells = aqua_obs[i] + _s02_aqua_drift(y)
            for s, val in _s02_scope_means(aqua_cells, district, n_d).items():
                rows.append((y, s, "modis_aqua_night",
                             val + float(sat_rng.normal(0.0, noise["annual_scope_sd_night"])), flag_drift))
        if y >= 2012:
            ap = applied["viirs_to_aqua"]
            viirs_cells = _s02_apply_calibration(ap["src"][i], cls[i], ap["coef"], ap["shrunk"]).astype(np.float64)
            for s, val in _s02_scope_means(viirs_cells, district, n_d).items():
                rows.append((y, s, "viirs_night_cal",
                             val + float(sat_rng.normal(0.0, noise["annual_scope_sd_night"])), None))
    annual = pd.DataFrame({
        "year": pd.Series([r[0] for r in rows], dtype="int64"),
        "scope": pd.Series([r[1] for r in rows], dtype=object),
        "variable": pd.Series([r[2] for r in rows], dtype=object),
        "value": pd.Series([r[3] for r in rows], dtype="float64"),
        "flag": pd.Series([r[4] for r in rows], dtype=object),
    })
    annual = annual.sort_values(["variable", "scope", "year"], kind="stable").reset_index(drop=True)
    return {"fields": fields, "nobs": nobs, "calibration": calibration, "annual": annual,
            "expected_ols_b": expected}

## 2.10 `SYNTHETIC_TRUTH` — every planted fact, JSON-ready
Thresholds are given per SHAP group in the definitions used by section 08 (primary feature, kind, value,
direction); "saturation" for an exponential term is where the slope falls below 10 % of its maximum
(`scale · ln 10`). `planted_mean_abs_effect_c` (1 km mean |term − mean|) is the expected importance ranking.

In [ ]:
def _s02_truth(frames, windows, levels, lev_ann, years, anom, ref_period, qa, planted_effects, night, future) -> dict:
    d, n = _S02_DAY, _S02_NIGHT
    ln10 = _s02_math.log(10.0)
    thresholds = {
        "day": {
            "ndvi": {"primary": "ndvi", "kind": "saturation", "value": d["ndvi_saturation"], "direction": "cooling",
                     "effect_c": d["ndvi_cooling_c"],
                     "focal": {"feature": "ndvi_m3", "onset": d["park_onset"], "full": d["park_full"],
                               "effect_c": d["park_cooling_c"], "window_m": d["park_window_m"]}},
            "phenology": {"primary": "ndvi_drop", "kind": "onset", "value": d["harvest_onset"],
                          "saturation": d["harvest_full"], "direction": "warming", "effect_c": d["harvest_heating_c"]},
            "albedo": {"primary": "albedo", "kind": "linear", "slope_c_per_unit": d["albedo_per_unit"],
                       "direction": "cooling"},
            "built": {"primary": "frac_built", "kind": "linear", "slope_c_per_unit": d["built_per_unit"],
                      "direction": "cooling"},
            "barren": {"primary": "frac_barren", "kind": "linear", "slope_c_per_unit": d["barren_per_unit"],
                       "direction": "warming"},
            "water": {"primary": "frac_water", "kind": "saturation", "value": round(d["water_scale"] * ln10, 4),
                      "e_folding": d["water_scale"], "direction": "cooling", "effect_c": d["water_cooling_c"],
                      "reach": {"feature": "dist_water", "effect_c": d["water_reach_c"],
                                "e_folding_km": d["water_reach_m"] / 1000.0}},
            "climate_moisture": {"primary": "era5_swvl1", "kind": "linear", "slope_c_per_unit": d["swvl1_per_unit"],
                                 "direction": "cooling", "vpd_slope_c_per_kpa": d["vpd_per_kpa"]},
            "terrain": {"primary": "elevation", "kind": "linear", "slope_c_per_m": d["lapse_per_m"],
                        "direction": "cooling", "tpi_slope_c_per_m": d["tpi_per_m"],
                        "tpi_window_km": d["tpi_window_m"] / 1000.0},
        },
        "night": {
            "built": {"primary": "frac_built", "kind": "saturation", "value": round(n["built_scale"] * ln10, 4),
                      "e_folding": n["built_scale"], "direction": "warming", "effect_c": n["built_c"]},
            "built_volume": {"primary": "built_v", "kind": "linear-in-log1p",
                             "slope_c_per_log1p": n["built_v_per_log1p"], "direction": "warming"},
            "water": {"primary": "frac_water", "kind": "saturation", "value": round(n["water_scale"] * ln10, 4),
                      "e_folding": n["water_scale"], "direction": "warming", "effect_c": n["water_warming_c"]},
            "ndvi": {"primary": "ndvi", "kind": "linear", "slope_c_per_unit": n["ndvi_per_unit"],
                     "direction": "cooling"},
            "barren": {"primary": "frac_barren", "kind": "linear", "slope_c_per_unit": n["barren_per_unit"],
                       "direction": "cooling"},
            "terrain": {"primary": "elevation", "kind": "linear", "slope_c_per_m": n["lapse_per_m"],
                        "direction": "cooling", "tpi_slope_c_per_m": n["tpi_per_m"],
                        "tpi_window_km": n["tpi_window_m"] / 1000.0, "note": "cold-air pooling in hollows"},
            "climate_moisture": {"primary": "era5_swvl1", "kind": "linear", "slope_c_per_unit": n["swvl1_per_unit"],
                                 "direction": "warming", "vpd_slope_c_per_kpa": n["vpd_per_kpa"],
                                 "note": "soil-moisture thermal inertia"},
        },
    }
    all_groups = ["ndvi", "phenology", "ndbi", "moisture_index", "albedo", "built", "built_volume", "population",
                  "forest", "water", "cropland", "barren", "urban_proximity", "landscape", "terrain", "climate_heat",
                  "climate_moisture", "climate_wind"]
    mean_effects = {m: {g: round(float(np.mean([planted_effects[m][y][g] for y in frames])), 4)
                        for g in planted_effects[m][frames[0]]} for m in ("day", "night")}
    ref_sel = (years >= ref_period[0]) & (years <= ref_period[1])
    acq = {y: _S02_DAY_ACQ[y] for y in frames}
    # Noise-free expectation of the SPEC §4.7 β regression: OLS of the planted annual level on the ERA5 temperature
    # (+ swvl1) over the regression years. It differs slightly from the planted β because the land-driven level
    # trend is partly collinear with the warming trend of the regressor (a real-world confound, kept on purpose).
    expected_beta = {}
    for mode, (y0, y1) in (("day", (2003, 2020)), ("night", (2003, 2025))):
        sel = (years >= y0) & (years <= y1)
        temp = anom[_S02_LEVEL[mode]["temperature_var"]][sel]
        design = np.column_stack([np.ones(sel.sum()), temp, anom["era5_swvl1"][sel]])
        coef, *_ = np.linalg.lstsq(design, lev_ann[mode][sel], rcond=None)
        expected_beta[mode] = {"years": [y0, y1], "beta": round(float(coef[1]), 4),
                               "gamma_swvl1": round(float(coef[2]), 3)}
    truth = {
        "generator": _S02_GENERATOR, "seed": int(_s02_cfg("SEED", 42)), "data_mode": "synthetic",
        "grid_res_m": int(GRID.res), "fine_res_m": int(GRID_FINE.res), "lc_fine_res_m": _s02_lc_res(),
        "frames": list(frames), "windows": {y: list(windows[y]) for y in frames},
        "formula": {
            "day": ("anom_day = -4.5*min(ndvi,0.30)/0.30 - 1.2*clip((box3km(ndvi)-0.20)/0.15,0,1) "
                    "+ 2.2*clip((ndvi_drop-0.12)/0.28,0,1) - 9.0*(albedo-0.18) - 2.2*built_f + 2.8*frac_barren "
                    "- 7.0*(1-exp(-box1km(frac_water)/0.08)) - 0.8*exp(-dist_water/0.6km) "
                    "- 12*(swvl1-mean) + 0.6*(vpd-mean) - 0.0045*(elev-220) + 0.006*tpi7km + noise; "
                    "centred to 0 over the ROI at 1 km"),
            "night": ("anom_night = 3.2*(1-exp(-built_f/0.30))/(1-exp(-1/0.30)) + 0.6*log1p(built_v) "
                      "+ 2.5*(1-exp(-box1km(frac_water)/0.08)) - 0.8*ndvi - 1.2*frac_barren + 0.012*tpi7km "
                      "- 0.003*(elev-220) + 6*(swvl1-mean) - 0.4*(vpd-mean) + noise; centred to 0 over the ROI"),
            "level": "L(y) = base + trend*(y-2010)/10 + beta*(T(y)-T_ref) + gamma*(swvl1(y)-swvl1_ref); "
                     "frame level = mean of L(y) over the frame window; T = era5_tmax (day) / era5_tmin (night)",
        },
        "level_definition": "ROI mean (1 km cells) of the true absolute field; day at the 10:45 IST reference "
                            "overpass in Landsat units, night in Aqua-equivalent units",
        "levels": {m: {y: round(levels[m][y], 4) for y in frames} for m in ("day", "night")},
        "levels_annual": {m: {int(y): round(float(v), 4) for y, v in zip(years, lev_ann[m]) if 1990 <= y <= 2025}
                          for m in ("day", "night")},
        "level_model": {m: {**p, "ref_period": list(ref_period),
                            "temperature_ref": round(float(anom[p["temperature_var"]][ref_sel].mean()), 4)}
                        for m, p in _S02_LEVEL.items()},
        "beta_day": _S02_LEVEL["day"]["beta"], "beta_night": _S02_LEVEL["night"]["beta"],
        "beta": {"day": {"value": _S02_LEVEL["day"]["beta"], "temperature_var": "era5_tmax",
                         "co_regressor": {"era5_swvl1": _S02_LEVEL["day"]["gamma_swvl1"]},
                         "expected_ols_noise_free": expected_beta["day"]},
                 "night": {"value": _S02_LEVEL["night"]["beta"], "temperature_var": "era5_tmin",
                           "co_regressor": {"era5_swvl1": _S02_LEVEL["night"]["gamma_swvl1"]},
                           "expected_ols_noise_free": expected_beta["night"]}},
        "planted_thresholds": thresholds,
        "planted_parameters": {"day": dict(_S02_DAY), "night": dict(_S02_NIGHT)},
        "planted_mean_abs_effect_c": mean_effects,
        "irrelevant_by_construction": {m: [g for g in all_groups if g not in thresholds[m]] for m in ("day", "night")},
        "monotone_expectations": {"day": {"ndvi": -1, "forest": -1, "water": -1, "moisture_index": -1},
                                  "night": {"built": 1, "built_volume": 1}},
        "uhi": {"definition": "mean anomaly over cells with frac_built >= 0.5 minus cells with frac_built <= 0.05 "
                              "and frac_water < 0.2 (SPEC §4.7)",
                "day": {"sign": "negative", "target_range_c": [-2.5, -1.5], "realised": qa["uhi"]["day"]},
                "night": {"sign": "positive", "target_range_c": [2.0, 5.0], "realised": qa["uhi"]["night"]}},
        "overpass": {"reference_ist_h": _S02_REF_OVERPASS_IST_H,
                     "mean_overpass_ist_h": {y: acq[y]["overpass"] for y in frames},
                     "heating_rate_k_per_h": {y: acq[y]["heating"] for y in frames},
                     "raw_day_level_bias_c": {
                         y: round(-(_S02_REF_OVERPASS_IST_H - acq[y]["overpass"]) * acq[y]["heating"], 4)
                         for y in frames},
                     "note": "RAW lst_day_f = truth - (10.75 - overpass_h) * heating_rate; SPEC §4.7 correction "
                             "recovers the 10:45 IST truth"},
        "heating_rate_k_per_h": {y: acq[y]["heating"] for y in frames},
        "path_offsets": {"convention": "per-path bias relative to the harmonised mosaic (value removed from that "
                                       "path's composite); planted symmetric +-delta/2 with delta = 146 - 147",
                         "planted": {y: qa["path_offsets"][y]["planted"] for y in frames},
                         "estimated": {y: qa["path_offsets"][y]["estimated"] for y in frames},
                         "overlap_pixels": {y: qa["path_offsets"][y]["overlap_pixels"] for y in frames}},
        "calibration": {k: {"planted_by_class": [{"class": c, "a": a, "b": b,
                                                  "expected_ols_b": night["expected_ols_b"][k].get(c)}
                                                 for c, (a, b) in v.items()],
                            "note": "OLS on noisy source values attenuates b by var(true)/var(observed); "
                                    "expected_ols_b is the planted slope times that reliability",
                            "fit_years": list(_S02_CAL_FIT_YEARS[k]),
                            "source_noise_sd_c": _S02_NOISE["terra_night_annual" if k.startswith("terra")
                                                            else "viirs_night_annual"],
                            "recovered": night["calibration"][k]}
                        for k, v in _S02_CAL_PLANTED.items()},
        "night_availability": {y: _S02_NIGHT_ACQ[y][0] for y in frames},
        "satellites": {"terra_day_minus_landsat_level_c": _S02_TERRA_DAY_OFFSET_C,
                       "terra_day_anomaly_factor": 0.9,
                       "terra_day_drift_c": {y: round(_s02_terra_drift(y), 3) for y in range(2022, 2027)},
                       "aqua_night_drift_c": {y: round(_s02_aqua_drift(y), 3) for y in range(2022, 2027)},
                       "drift_flag_from": _S02_DRIFT_FROM},
        "climate": {"warming_per_decade_k": {"era5_tmax": 0.20, "era5_tmin": 0.25},
                    "gradient": "SW -> NE: NE cooler by day, wetter (swvl1 0.11 -> 0.25), smaller diurnal range",
                    "climatology": {k: {"centre": c, "per_unit_g": s} for k, (c, s, _) in _S02_CLIM.items()}},
        "urban_growth": {"centres": [{"name": c[0], "lon": c[1], "lat": c[2], "amplitude": c[3], "r1990_km": c[4],
                                      "r2025_km": c[5], "mid_year": c[6], "tau_years": c[7]}
                                     for c in _S02_URBAN_CENTRES],
                         "built_share": {y: round(v, 4) for y, v in qa["built_share"].items()},
                         "irreversible": True, "glc_year_for_frame": {y: (2022 if y == 2025 else y) for y in frames},
                         "pop_total_persons": {"1990": _S02_POP_TOTAL["1990"], "2025": _S02_POP_TOTAL["2025"]}},
        "ndvi_support_1km_p05_p25_p50_p75_p95": qa["ndvi_support"],
        "noise": dict(_S02_NOISE),
        "future": {"illustrative": True,
                   "note": "synthetic stand-ins: pseudo-GCMs (not real CMIP6 output), Chen-2020-like urban ratios, "
                           "IIASA-SSP-like population factors, AR6 Table 4.5 central GSAT values",
                   "cmip6_delta_2050": _s02_cmip6_summary(future["cmip6"]),
                   "urban_rate_pct_per_year_ssp2": dict(_S02_URBAN_RATE),
                   "ssp_urban_multiplier": dict(_S02_SSP_URBAN_MULT),
                   "pop_factor_2050": dict(_S02_POP_FACTOR_2050),
                   "ar6_gsat_vs_1995_2014": {k: {"2021-2040": v[0], "2041-2060": v[1]}
                                             for k, v in _S02_AR6_GSAT.items()}},
    }
    truth = _s02_py(truth)
    _s02_json.dumps(truth)                       # fail fast if anything is not JSON-serialisable
    return truth


def _s02_cmip6_summary(cmip6: pd.DataFrame) -> dict:
    """Ensemble mean/sd of the 2050-minus-2025 delta per scenario and variable."""
    out = {}
    now = int(_s02_cfg("NOW", 2025))
    last = int(max(cmip6["frame"]))
    for (scenario, var), grp in cmip6.groupby(["scenario", "var"]):
        piv = grp.pivot_table(index="model", columns="frame", values="value_k")
        delta = piv[last] - piv[now]
        out.setdefault(scenario, {})[var] = {"mean": round(float(delta.mean()), 3), "sd": round(float(delta.std()), 3)}
    return out

## 2.11 Run the generator and publish the acquisition globals

In [ ]:
if _S02_ACTIVE:
    with timer("02 synthetic world"):
        _S02_OUT = _s02_generate()
    RAW_STATIC = _S02_OUT["RAW_STATIC"]
    RAW_OBS = _S02_OUT["RAW_OBS"]
    LC_FINE = _S02_OUT["LC_FINE"]
    FRAME_META = _S02_OUT["FRAME_META"]
    NIGHT_CALIBRATION = _S02_OUT["NIGHT_CALIBRATION"]
    ANNUAL_SERIES = _S02_OUT["ANNUAL_SERIES"]
    FUTURE_INPUTS = _S02_OUT["FUTURE_INPUTS"]
    SYNTHETIC_TRUTH = _S02_OUT["SYNTHETIC_TRUTH"]
    _S02_QA = _S02_OUT["_qa"]
    del _S02_OUT

    # Provenance (DATA_SOURCES[var][year]) and caveats, mirroring what sections 03-05 record in gee mode.
    if not isinstance(globals().get("DATA_SOURCES"), dict):
        DATA_SOURCES = {}
    if not isinstance(globals().get("DATA_CAVEATS"), list):
        DATA_CAVEATS = []
    _s02_desc = f"SYNTHETIC ({_S02_GENERATOR}, seed {int(_s02_cfg('SEED', 42))})"
    _s02_src_text = {
        "lst_day_f": "planted day LST at the scene overpass (overpass bias + path offsets + composite noise), "
                     "harmonised like Landsat C2 L2 ST",
        "lst_day_nobs_f": "simulated clear-scene count per WRS-2 path/row (Binomial, SLC-off gaps)",
        "ndvi_f": "class-signature mixture (Landsat-like, Apr-Jun)", "ndbi_f": "class-signature mixture",
        "mndwi_f": "class-signature mixture", "albedo_f": "class-signature mixture (Liang-like)",
        "ndvi_spring_f": "class-signature mixture (Feb-Mar)", "built_f": "GHSL-like built-up surface fraction",
        "lst_night": "planted night LST (Aqua-equivalent; Terra/VIIRS calibrated)",
        "lst_night_nobs": "simulated valid composites / daily observations",
        "lst_day_modis": "MODIS-Terra-like 1 km day LST (offset, drift, noise)",
        "built_v": "GHSL-like building volume (m3/m2)", "pop": "GHSL-like population density (persons/km2)",
        "land_cover": "GLC_FCS30D-like classification (2025 frame uses the 2022 map)",
    }
    for _s02_var in _S02_FINE_KEYS + _S02_COARSE_KEYS + ["land_cover"]:
        _s02_text = _s02_src_text.get(_s02_var, "ERA5-Land-like Apr-Jun field" if _s02_var.startswith("era5")
                                      else "GLC_FCS30D-like class fraction")
        DATA_SOURCES.setdefault(_s02_var, {}).update({int(y): f"{_s02_desc}: {_s02_text}" for y in RAW_OBS})
    for _s02_var in ("elevation", "slope", "aspect_sin", "aspect_cos", "water_occ", "elevation_f", "water_occ_f"):
        DATA_SOURCES.setdefault(_s02_var, {}).update({int(y): f"{_s02_desc}: static {_s02_var}" for y in RAW_OBS})
    DATA_SOURCES.setdefault("heating_rate", {}).update(
        {int(y): f"{_s02_desc}: planted 09:30-11:30 IST skin-temperature heating rate" for y in RAW_OBS})
    DATA_SOURCES.setdefault("annual_series", {}).update(
        {"1990-2025": f"{_s02_desc}: ERA5-like and satellite-like annual Apr-Jun means (NCR + districts)"})
    DATA_SOURCES.setdefault("night_calibration", {}).update(
        {"2000-2025": f"{_s02_desc}: planted Terra/VIIRS -> Aqua per-class offsets, recovered like section 04"})
    # the same keys as section 05 records in gee mode: future_inputs plus one future_<input> entry per input
    _s02_now = int(_s02_cfg("NOW", 2025))
    _s02_span = f"{_s02_now}-{max([_s02_now] + [int(y) for y in (_s02_cfg('FRAMES_PROJ', []) or [])])}"
    DATA_SOURCES.setdefault("future_inputs", {}).update(
        {_s02_span: f"{_s02_desc}: illustrative pseudo-GCM / SSP stand-ins (not NASA GDDP-CMIP6, Chen 2020, "
                    "IIASA SSP v3 or AR6 data)"})
    for _s02_var in sorted(FUTURE_INPUTS):
        DATA_SOURCES.setdefault(f"future_{_s02_var}", {}).update(
            {_s02_span: f"{_s02_desc}: synthetic stand-in for {_s02_var}"})
    for _s02_caveat in (
            "SYNTHETIC DATA: every variable is simulated from a planted response; see SYNTHETIC_TRUTH.",
            "SYNTHETIC future inputs (34 pseudo-GCMs, SSP urban growth, population factors, GSAT increments) are "
            "illustrative stand-ins, not real scenario data."):
        # Section 01 already states the general synthetic caveat; never add a second wording of it.
        if _s02_caveat.startswith("SYNTHETIC DATA") and any(str(c).startswith("SYNTHETIC DATA") for c in DATA_CAVEATS):
            continue
        if _s02_caveat not in DATA_CAVEATS:
            DATA_CAVEATS.append(_s02_caveat)
    del _s02_desc, _s02_src_text, _s02_var, _s02_text, _s02_caveat, _s02_now, _s02_span
    free_memory()

## 2.12 Contract check (SPEC §3.2)
The same assertions guard the GEE path in section 05; here they prove the synthetic twin is interchangeable.

In [ ]:
def _s02_validate() -> None:
    """Raise AssertionError with a precise message if a §3.2 global violates the contract."""
    h, w = GRID.shape
    hf, wf = GRID_FINE.shape
    roi = np.asarray(ROI_MASK, bool)
    roi_f = np.asarray(globals().get("ROI_MASK_FINE", np.repeat(np.repeat(roi, hf // h, 0), wf // w, 1)), bool)
    assert set(RAW_STATIC) == {"elevation", "slope", "aspect_sin", "aspect_cos", "water_occ", "elevation_f",
                               "water_occ_f"}, sorted(RAW_STATIC)
    for key, arr in RAW_STATIC.items():
        shape = (hf, wf) if key.endswith("_f") else (h, w)
        mask = roi_f if key.endswith("_f") else roi
        assert arr.shape == shape and arr.dtype == np.float32, (key, arr.shape, arr.dtype)
        assert np.isnan(arr[~mask]).all() and np.isfinite(arr[mask]).all(), f"RAW_STATIC[{key}] NaN pattern"
    frames = [int(y) for y in _s02_cfg("FRAMES_OBS", list(_S02_DEFAULT_WINDOWS))]
    assert sorted(RAW_OBS) == sorted(frames), sorted(RAW_OBS)
    r_lc = _s02_lc_res()
    lc_shape = (h * GRID.res // r_lc, w * GRID.res // r_lc)
    for year in frames:
        obs = RAW_OBS[year]
        expected_keys = set(_S02_FINE_KEYS) | set(_S02_COARSE_KEYS)
        assert set(obs) == expected_keys, (year, sorted(set(obs) ^ expected_keys))
        for key, arr in obs.items():
            fine = key in _S02_FINE_KEYS
            assert arr.shape == ((hf, wf) if fine else (h, w)) and arr.dtype == np.float32, (
                year, key, arr.shape, arr.dtype)
            assert np.isnan(arr[~(roi_f if fine else roi)]).all(), f"RAW_OBS[{year}][{key}] must be NaN outside the ROI"
        glc = np.stack([obs[k][roi] for k in _S02_GLC_KEYS])
        assert np.allclose(glc.sum(0), 1.0, atol=1e-4), f"glc fractions of {year} do not sum to 1"
        assert np.nanmin(obs["built_f"]) >= 0 and np.nanmax(obs["built_f"]) <= 1
        assert LC_FINE[year].shape == lc_shape and LC_FINE[year].dtype == np.uint8, (year, LC_FINE[year].shape)
        assert int(LC_FINE[year].max()) <= 6
        meta = FRAME_META[year]
        assert set(meta) == {"window", "day", "night"}
        assert set(meta["day"]) == {"sensors", "scenes", "scenes_per_pathrow", "mean_overpass_ist_h", "coverage",
                                    "heating_rate_k_per_h", "path_offsets"}, sorted(meta["day"])
        assert set(meta["day"]["scenes_per_pathrow"]) == set(_S02_PATHROWS)
        assert set(meta["night"]) == {"source", "sensor", "overpass_ist_h", "composites", "coverage"}
        assert meta["night"]["source"] in ("aqua", "terra_cal", "viirs_cal", None)
        has_night = np.isfinite(obs["lst_night"][roi]).any()
        assert has_night == (meta["night"]["source"] is not None), f"night availability of {year}"
    for kind in ("terra_to_aqua", "viirs_to_aqua"):
        cal = NIGHT_CALIBRATION[kind]
        assert set(cal) == {"r2", "rmse", "n", "by_class"} and all(set(c) == {"class", "a", "b", "n"}
                                                                   for c in cal["by_class"]), cal
    assert list(ANNUAL_SERIES.columns) == ["year", "scope", "variable", "value", "flag"], list(ANNUAL_SERIES.columns)
    assert set(ANNUAL_SERIES["variable"]) == {"era5_skin_day", "era5_skin_night", "era5_tmax", "era5_tmin",
                                              "era5_swvl1", "modis_terra_day", "modis_aqua_night", "viirs_night_cal"}
    assert ANNUAL_SERIES["year"].between(1990, 2025).all() and "ncr" in set(ANNUAL_SERIES["scope"])
    assert set(FUTURE_INPUTS) == {"cmip6", "cmip6_pattern", "urban_growth", "ghsl_2030", "pop_growth",
                                  "gsat_increment"}, sorted(FUTURE_INPUTS)
    assert list(FUTURE_INPUTS["cmip6"].columns) == ["scenario", "model", "frame", "var", "value_k"]
    assert FUTURE_INPUTS["cmip6"]["model"].nunique() == _S02_CMIP6["n_models"]
    for var in ("tasmax", "tasmin"):
        assert FUTURE_INPUTS["cmip6_pattern"][var].shape == (h, w)
    assert list(FUTURE_INPUTS["urban_growth"].columns) == ["scenario", "district", "frame", "ratio"]
    assert list(FUTURE_INPUTS["pop_growth"].columns) == ["scenario", "frame", "factor"]
    for var in ("built", "pop"):
        assert FUTURE_INPUTS["ghsl_2030"][var].shape == (h, w) and FUTURE_INPUTS["ghsl_2030"][var].dtype == np.float32
    assert set(FUTURE_INPUTS["gsat_increment"]) == {"ssp126", "ssp245", "ssp585"}
    assert isinstance(SYNTHETIC_TRUTH, dict) and SYNTHETIC_TRUTH["generator"] == _S02_GENERATOR
    _s02_json.dumps(SYNTHETIC_TRUTH)


if _S02_ACTIVE:
    _s02_validate()
    log("section-02 contract check passed (SPEC §3.2 keys, shapes, dtypes, NaN outside the ROI)")

## 2.13 Quality-assurance figures and the planted-world summary
Fine day LST (raw composite + the SPEC §4.7 overpass correction) and the planted fine night field for the first
and last frame, built-up growth, and the annual series with the planted levels.

In [ ]:
def _s02_save_show(fig, name: str) -> None:
    try:
        fig.savefig(Path(CFG.FIG_DIR) / name, dpi=110, bbox_inches="tight")
    except Exception as exc:  # noqa: BLE001 - a figure must never stop the pipeline
        log(f"02: could not save figure {name}: {exc}", "WARNING")
    plt.show()
    plt.close(fig)


if _S02_ACTIVE:
    _s02_frames = sorted(RAW_OBS)
    _s02_first, _s02_last = _s02_frames[0], _s02_frames[-1]
    _s02_stride = max(1, max(GRID_FINE.shape) // 900)
    _s02_ext = [0, GRID.width * GRID.res / 1000, 0, GRID.height * GRID.res / 1000]

    def _s02_day_fine(year):
        meta = FRAME_META[year]["day"]
        corr = (_S02_REF_OVERPASS_IST_H - meta["mean_overpass_ist_h"]) * meta["heating_rate_k_per_h"]
        return RAW_OBS[year]["lst_day_f"] + corr

    fig, axes = plt.subplots(2, 2, figsize=(11, 11), constrained_layout=True)
    _s02_day = [_s02_day_fine(y) for y in (_s02_first, _s02_last)]
    _s02_vmin, _s02_vmax = np.nanpercentile(np.concatenate([a[::7, ::7].ravel() for a in _s02_day]), [2, 98])
    for ax, year, arr in zip(axes[0], (_s02_first, _s02_last), _s02_day):
        im = ax.imshow(arr[::_s02_stride, ::_s02_stride], cmap="inferno", vmin=_s02_vmin, vmax=_s02_vmax,
                       extent=_s02_ext, interpolation="nearest")
        ax.set_title(f"Day LST {year} (fine, overpass-corrected) [SYNTHETIC]", fontsize=9)
    fig.colorbar(im, ax=axes[0], shrink=0.7, label="°C")
    _s02_night = [_S02_QA["night_truth_fine"].get(y) for y in (_s02_first, _s02_last)]
    _s02_night = [a if a is not None else np.full(GRID_FINE.shape, np.nan, np.float32) for a in _s02_night]
    _s02_nmin, _s02_nmax = np.nanpercentile(np.concatenate([a[::7, ::7].ravel() for a in _s02_night]), [2, 98])
    for ax, year, arr in zip(axes[1], (_s02_first, _s02_last), _s02_night):
        im = ax.imshow(arr[::_s02_stride, ::_s02_stride], cmap="magma", vmin=_s02_nmin, vmax=_s02_nmax,
                       extent=_s02_ext, interpolation="nearest")
        ax.set_title(f"Night LST {year} (planted fine truth; source {FRAME_META[year]['night']['source']}) "
                     "[SYNTHETIC]", fontsize=9)
    fig.colorbar(im, ax=axes[1], shrink=0.7, label="°C")
    _s02_save_show(fig, "02_synthetic_lst_fine.png")

    fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), constrained_layout=True)
    for ax, year in zip(axes[:2], (_s02_first, _s02_last)):
        im = ax.imshow(RAW_OBS[year]["built_f"][::_s02_stride, ::_s02_stride], cmap="viridis", vmin=0, vmax=1,
                       extent=_s02_ext, interpolation="nearest")
        ax.set_title(f"GHSL-like built_f {year} [SYNTHETIC]", fontsize=9)
    fig.colorbar(im, ax=axes[:2], shrink=0.8, label="built-up fraction")
    _s02_share = [float(np.nanmean(RAW_OBS[y]["built_f"])) for y in _s02_frames]
    _s02_glcb = [float(np.nanmean(RAW_OBS[y]["glc_built"])) for y in _s02_frames]
    axes[2].plot(_s02_frames, _s02_share, "o-", label="GHSL-like built_f (ROI mean)")
    axes[2].plot(_s02_frames, _s02_glcb, "s--", label="GLC-like glc_built (ROI mean)")
    _s02_d = np.asarray(DISTRICT_IDX)
    for did, name in ((0, "Delhi NCT"), (1, "Gurugram"), (17, "Gautam Buddh Nagar"), (16, "Ghaziabad")):
        sel = _s02_d == did
        if sel.any():
            axes[2].plot(_s02_frames, [float(np.nanmean(_s02_block_mean(RAW_OBS[y]["built_f"],
                                                                         GRID_FINE.shape[0] // GRID.shape[0])[0][sel]))
                                       for y in _s02_frames], ".:", label=name)
    axes[2].set_ylabel("built-up fraction")
    axes[2].set_title("Urban growth [SYNTHETIC]", fontsize=9)
    axes[2].legend(fontsize=7)
    _s02_save_show(fig, "02_synthetic_built_growth.png")

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), constrained_layout=True)
    _s02_ncr = ANNUAL_SERIES[ANNUAL_SERIES["scope"] == "ncr"]
    for ax, mode, variables in ((axes[0], "day", ["era5_tmax", "era5_skin_day", "modis_terra_day"]),
                                (axes[1], "night", ["era5_tmin", "era5_skin_night", "modis_aqua_night",
                                                    "viirs_night_cal"])):
        for var in variables:
            sub = _s02_ncr[_s02_ncr["variable"] == var]
            ax.plot(sub["year"], sub["value"], ".-", label=var)
            drift = sub[sub["flag"] == "drift"]
            if len(drift):
                ax.scatter(drift["year"], drift["value"], marker="x", color="k", zorder=5)
        lv = _S02_QA["lev_ann"][mode]
        ax.plot(list(range(1990, 2026)), [lv[y] for y in range(1990, 2026)], "k--", lw=1, label=f"planted {mode} level")
        ax.plot(_s02_frames, [SYNTHETIC_TRUTH["levels"][mode][y] for y in _s02_frames], "ko", ms=4,
                label="frame level (truth)")
        ax.set_title(f"NCR annual series, {mode} (x = drift flag) [SYNTHETIC]", fontsize=9)
        ax.legend(fontsize=7)
        ax.set_ylabel("°C")
    _s02_save_show(fig, "02_synthetic_annual_series.png")

    _s02_rows = []
    for y in _s02_frames:
        obs = RAW_OBS[y]
        _s02_rows.append({
            "year": y, "day_tier_scenes_min": min(FRAME_META[y]["day"]["scenes_per_pathrow"].values()),
            "day_cov": FRAME_META[y]["day"]["coverage"], "raw_day_mean": round(float(np.nanmean(obs["lst_day_f"])), 2),
            "true_day_level": SYNTHETIC_TRUTH["levels"]["day"][y],
            "overpass_bias": SYNTHETIC_TRUTH["overpass"]["raw_day_level_bias_c"][y],
            "night_src": FRAME_META[y]["night"]["source"],
            "night_mean": (round(float(np.nanmean(obs["lst_night"])), 2) if np.isfinite(obs["lst_night"]).any()
                           else None),
            "true_night_level": SYNTHETIC_TRUTH["levels"]["night"][y],
            "uhi_day": SYNTHETIC_TRUTH["uhi"]["day"]["realised"][y],
            "uhi_night": SYNTHETIC_TRUTH["uhi"]["night"]["realised"][y],
            "built": round(float(np.nanmean(obs["built_f"])), 4),
            "path_off_est": FRAME_META[y]["day"]["path_offsets"]["146"],
            "path_off_true": SYNTHETIC_TRUTH["path_offsets"]["planted"][y]["146"],
        })
    print("Synthetic world per frame [SYNTHETIC]")
    print(pd.DataFrame(_s02_rows).set_index("year").to_string())
    print("Night calibration (recovered vs planted):")
    for _s02_kind in ("terra_to_aqua", "viirs_to_aqua"):
        _s02_cal = NIGHT_CALIBRATION[_s02_kind]
        print(f"  {_s02_kind}: r2={_s02_cal['r2']} rmse={_s02_cal['rmse']} n={_s02_cal['n']} | " + ", ".join(
            f"c{c['class']}: b={c['b']:.3f} (planted {_S02_CAL_PLANTED[_s02_kind][c['class']][1]:.3f})"
            for c in _s02_cal["by_class"]))
    print("Planted mean |effect| per group (°C, 1 km):", _s02_json.dumps(SYNTHETIC_TRUTH["planted_mean_abs_effect_c"]))
    del (_s02_frames, _s02_first, _s02_last, _s02_stride, _s02_ext, _s02_day, _s02_night, _s02_vmin, _s02_vmax,
         _s02_nmin, _s02_nmax, _s02_share, _s02_glcb, _s02_ncr, _s02_rows, _s02_d, _s02_kind, _s02_cal, fig, axes, im)
    _S02_QA = {k: v for k, v in _S02_QA.items() if k != "night_truth_fine"}   # drop the large QA rasters
    free_memory()

# 3. Earth Engine: Landsat C2 L2 day LST and spectral composites (gee mode only)

This section builds the **day target** of v2 and the Landsat spectral predictors for every observed frame
(SPEC §2.1). It only runs in gee mode; in synthetic mode section 02 already provides the same globals and nothing
here runs (the functions are still defined but none of them is called).

**Why Landsat?** Collection-2 Level-2 surface temperature (ST) is the only day-LST record that runs unchanged from
1990 to 2025 (docs/research/gee_datasets_v2.md). The overpass is ~10:10–10:50 IST, at 30 m. The 1990s archive over
Delhi NCR is thin, so every frame is the Apr–Jun season of a **multi-year window** (`CFG.WINDOWS`, e.g.
1990 = 1989–1992).

**Recipe** (verified against the Earth Engine catalog and the earthengine-api source):
1. **Collections:** `LANDSAT/LT05|LE07|LC08|LC09/C02/T1_L2`, restricted to the five WRS-2 path/rows that cover
   the ROI (146/040, 146/041, 147/039, 147/040, 147/041). Landsat 7 is used only before **2021-01-01**: its orbit
   drifted and by 2022 it passed at 09:17 IST.
2. **Masks:** QA_PIXEL bits 0–4 clear (fill, dilated cloud, cirrus, cloud, shadow); QA_RADSAT == 0;
   ST_QA × 0.01 ≤ 3 K; ST_CDIST × 0.01 ≥ 1 km (ST errors are known to grow near clouds); scaled SR in [0, 1];
   ST DN > 0 (ST is not retrieved where ASTER GED emissivity is missing).
3. **Scaling:** SR × 2.75e-05 − 0.2; ST × 0.00341802 + 149 K → °C.
4. **Harmonisation:** TM/ETM+ reflectance is mapped to OLI with the Roy et al. (2016) ETM+→OLI OLS coefficients.
   These are applied unscaled, because C2 reflectance is already 0–1. TM is treated like ETM+.
5. **Per-path composites:** scenes of adjacent rows on one path come from the same orbit pass, so the scenes of each
   path are first **mosaicked per acquisition date**. Overlapping rows of the same pass then count once, as one
   observation. The per-date mosaics are reduced with one combined `median + count` reducer: one pass over the data
   gives both the composite and the clear-observation count. NDVI, NDBI, MNDWI (green/SWIR1) and the Liang (2001)
   broadband albedo are computed from that same median composite, at 30 m. A separate Feb–Mar composite (red/NIR
   only) gives `ndvi_spring_f`.
6. **Aggregation:** each 30 m band is aggregated to the fine grid (`GRID_FINE`, 250 m at the 1 km grid) with a
   mask-weighted, area-weighted `reduceResolution(mean)`. A fine pixel is kept only where ≥ 50 % of its area has
   valid 30 m pixels, so a small sliver cannot stand for the whole pixel. Aggregating **per path, before the
   mosaic**, keeps the server-side work at the 30 m composite and moves the mosaic to the cheap 250 m level.
7. **Path-offset harmonisation** (client side, exact SPEC semantics): δ = median(LST₁₄₆ − LST₁₄₇) over the
   overlap of the two paths at ≥ 250 m. The offset is split symmetrically: δ/2 is removed from path 146 and
   added to path 147. The two paths are then mosaicked (mean where both exist). Spectral bands are mosaicked
   without an offset. `path_offsets = {"146": +δ/2, "147": −δ/2}` is the value removed from each path, the same
   convention as section 02.
8. **Metadata** (`FRAME_META[y]["day"]`): the scene lists come from `aggregate_array` of WRS_PATH, WRS_ROW,
   SCENE_CENTER_TIME, CLOUD_COVER(_LAND) and `system:time_start`. Earth Engine does not catalogue DATE_ACQUIRED,
   so the acquisition date comes from `system:time_start`. From these lists the section derives: the sensors; the
   number of *clear* scenes (land cloud ≤ 30 %, the definition of the USGS inventory in the research notes) per
   path/row; the mean overpass in IST (UTC + 5.5 h); and the coverage of the ROI.
9. **Heating rate** for the 10:45 IST overpass correction of section 09: ERA5-Land hourly `skin_temperature`,
   Apr–Jun of the window, ROI mean, `(T(06 UTC) − T(04 UTC)) / 2` in K h⁻¹. 04 and 06 UTC are 09:30 and
   11:30 IST.

**Extraction** (`CFG.GEE_EXPORT`):
* `interactive`: tiled `computePixels` (section 01's `fetch_grid`: retry/backoff, quartering on memory and timeout
  errors, an on-disk tile cache keyed by the hash of the serialised expression). Tiles cover about 64 km × 64 km
  (256 px at 250 m), so each request's 30 m medians stay within interactive limits.
* `asset`: one `Export.image.toAsset` batch task per frame on the exact `GRID_FINE` (crs + crsTransform +
  dimensions, pyramiding mean). All frames are submitted at once, but **noncommercial projects run about 2 batch
  tasks at a time**, so the 8 frame exports take about 4 rounds (about 1.5-6 h). Every task has its own limits:
  `LST_GEE_EXPORT_TIMEOUT_S` (3 h) counted from the first poll that saw it RUNNING and `LST_GEE_EXPORT_QUEUE_S`
  (4 h) while it waits in the queue. A FAILED, CANCELLED or timed-out frame does not stop the others: a still
  active task is cancelled and that frame is extracted interactively. Each poll logs the number of RUNNING tasks
  (the maximum is kept in `_S03_QA['export_concurrency']`). Completed exports are read with cheap `computePixels`.
  The asset name contains the expression hash, so a rerun (or a new Kaggle session) reuses finished assets and
  attaches to tasks that are still running instead of paying the EECU again.

A frame whose ROI fine-pixel coverage is below `LST_DAY_MIN_FRAME_COVERAGE` (0.2) stops the section with a
diagnosis (an acquisition fault, e.g. a QA or ST-band problem of one sensor) before it is stage-cached; its tiles
stay cached, so a rerun is cheap. Every frame records its extraction cost (mode, wall time, tiles, splits) in
`_S03_QA`; the table at the end of the section (and, for the pilot, a projected full-run time) calibrates the full
run.

Missing data is never silently zero. A path/row without scenes is logged as a warning and recorded in
`DATA_CAVEATS`. Its pixels stay NaN, and its clear-scene count stays 0, which is the true count.

In [ ]:
_lst_enter_section('03')  # section marker (generated by build_notebook.py)

In [ ]:
_S03_ACTIVE = globals().get("DATA_MODE") == "gee"
_S03_RECIPE_VERSION = "s03-landsat-v2.2"   # v2.2: 30 m working grid on the native 15 mod 30 m lattice

# Landsat Collection 2 Level-2 Tier 1 collections (Earth Engine catalog ids) and their operating periods in the
# catalog (start inclusive; end exclusive or None = still acquiring). Landsat 7 is cut at 2021-01-01 (orbit drift).
_S03_SENSORS = {
    "L5": {"asset": "LANDSAT/LT05/C02/T1_L2", "start": "1984-03-16", "end": "2012-05-06", "family": "tm",
           "label": "Landsat 5 TM"},
    "L7": {"asset": "LANDSAT/LE07/C02/T1_L2", "start": "1999-05-28", "end": "2021-01-01", "family": "tm",
           "label": "Landsat 7 ETM+"},
    "L8": {"asset": "LANDSAT/LC08/C02/T1_L2", "start": "2013-03-18", "end": None, "family": "oli",
           "label": "Landsat 8 OLI/TIRS"},
    "L9": {"asset": "LANDSAT/LC09/C02/T1_L2", "start": "2021-10-31", "end": None, "family": "oli",
           "label": "Landsat 9 OLI-2/TIRS-2"},
}
_S03_L7_SLC_OFF_DATE = "2003-05-31"            # scan-line-corrector failure (ETM+ striping afterwards)
_S03_L7_END = "2021-01-01"                     # SPEC §2.1: Landsat 7 only before this date

# Band layouts (C2 L2): TM/ETM+ SR_B1..B5,B7 = blue..SWIR2, ST_B6; OLI SR_B2..B7 = blue..SWIR2, ST_B10.
_S03_SR_NAMES = ["blue", "green", "red", "nir", "swir1", "swir2"]
_S03_SR_BANDS = {"tm": ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"],
                 "oli": ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]}
_S03_ST_BAND = {"tm": "ST_B6", "oli": "ST_B10"}
_S03_SR_SCALE, _S03_SR_OFFSET = 2.75e-05, -0.2
_S03_ST_SCALE, _S03_ST_OFFSET_K = 0.00341802, 149.0
_S03_QA_CLEAR_BITS = 0b11111                   # QA_PIXEL bits 0-4: fill, dilated cloud, cirrus, cloud, shadow
_S03_ST_QA_MAX_K = 3.0                         # ST_QA x 0.01 (K uncertainty)
_S03_ST_CDIST_MIN_KM = 1.0                     # ST_CDIST x 0.01 (km to nearest cloud)
# Roy et al. 2016 (RSE 185, Table 2) ETM+ -> OLI OLS: OLI = intercept + slope x ETM+, blue..SWIR2 (0-1 reflectance).
_S03_ROY_SLOPES = [0.8474, 0.8483, 0.9047, 0.8462, 0.8937, 0.9071]
_S03_ROY_INTERCEPTS = [0.0003, 0.0088, 0.0061, 0.0412, 0.0254, 0.0172]
# Liang (2001) shortwave broadband albedo (TM/ETM+ bands 1,3,4,5,7 = OLI 2,4,5,6,7 after harmonisation).
_S03_LIANG = {"blue": 0.356, "red": 0.130, "nir": 0.373, "swir1": 0.085, "swir2": 0.072, "const": -0.0018}

# WRS-2 path/rows covering the ROI (gee_datasets_v2.md F1); 145, 148 and row 042 add < 2 % and are excluded.
_S03_PATHROWS = ["146/040", "146/041", "147/039", "147/040", "147/041"]
_S03_PATHS = ["146", "147"]
_S03_CLEAR_SCENE_MAX_CLOUD = 30.0              # "clear scene" = land cloud cover <= 30 % (USGS inventory definition)
_S03_SPARSE_SCENES = 6                         # SPEC §1.2 tier rule: < 6 clear scenes on any path/row
_S03_IST_OFFSET_H = 5.5
_S03_PIXEL_MIN_VALID_FRAC = 0.5                # >= 50 % valid 30 m area per fine pixel
_S03_OFFSET_MIN_RES_M = 250                    # SPEC: path offset estimated at >= 250 m
_S03_MIN_OVERLAP_PX = 200                      # minimum overlap pixels for an offset estimate
_S03_ERA5_HOURLY = "ECMWF/ERA5_LAND/HOURLY"
_S03_ERA5_SCALE_M = 11132.0                    # ERA5-Land native pixel size (catalog)
_S03_HEATING_UTC_HOURS = (4, 6)                # 09:30 and 11:30 IST
_S03_COMPOSITE_SCALE_M = 30.0
_S03_PATH_BANDS = ["lst", "nobs", "ndvi", "ndbi", "mndwi", "albedo", "ndvi_spring"]
_S03_FINE_KEYS = ["lst_day_f", "lst_day_nobs_f", "ndvi_f", "ndbi_f", "mndwi_f", "albedo_f", "ndvi_spring_f"]
_S03_BAND_TO_KEY = dict(zip(_S03_PATH_BANDS, _S03_FINE_KEYS))
_S03_DAY_META_KEYS = ("sensors", "scenes", "scenes_per_pathrow", "mean_overpass_ist_h", "coverage",
                      "heating_rate_k_per_h", "path_offsets")
_S03_INTERACTIVE_TILE_GROUND_M = 64_000        # ~64 km tiles: 256 px at 250 m (interactive compute limits)
_S03_EXPORT_TIMEOUT_S = int(_env_int("LST_GEE_EXPORT_TIMEOUT_S", 3 * 3600))    # per task, from first RUNNING poll
_S03_EXPORT_QUEUE_S = int(_env_int("LST_GEE_EXPORT_QUEUE_S", 4 * 3600))        # per task, waiting before RUNNING
_S03_EXPORT_POLL_S = float(_env_int("LST_GEE_EXPORT_POLL_S", 30))
_S03_EXPORT_POLL_MAX_S = 300.0
_S03_ASSET_PROPERTY = "lst_expr_sha1"          # expression hash stored on every exported asset
# D9: below this ROI fine-pixel coverage a day frame is an acquisition fault (RuntimeError before stage caching)
_S03_DAY_MIN_FRAME_COVERAGE = float(_env_str("LST_DAY_MIN_FRAME_COVERAGE") or 0.2)
_S03_FULL_FRAMES = 8                           # frames of the full profile (pilot cost extrapolation)

# Section-3 QA record (private): per-frame scene tables, all-scene counts, offset diagnostics, task records.
_S03_QA: dict = {}

if _S03_ACTIVE:
    # The acquisition layer (03-05) owns these globals in gee mode; 04 and 05 add to them.
    if not isinstance(globals().get("RAW_STATIC"), dict):
        RAW_STATIC = {}
    if not isinstance(globals().get("RAW_OBS"), dict):
        RAW_OBS = {}
    if not isinstance(globals().get("FRAME_META"), dict):
        FRAME_META = {}
    if "SYNTHETIC_TRUTH" not in globals():
        SYNTHETIC_TRUTH = None
    log(f"03: Landsat day LST for frames {CFG.FRAMES_OBS} (windows "
        f"{ {y: CFG.WINDOWS[y] for y in CFG.FRAMES_OBS} }) via GEE_EXPORT={CFG.GEE_EXPORT}")
else:
    log(f"03: DATA_MODE={globals().get('DATA_MODE')} - Earth Engine Landsat section skipped (section 02 provides "
        "the day LST)")

## 3.1 Shared Earth Engine fetch / export helpers (public, reused by sections 04–05)
* `gee_fetch_grid(image, grid, bands, name, ...)` downloads bands of an `ee.Image` onto a `GridSpec` as a
  `(len(bands), H, W)` array. Masked pixels become NaN for float outputs. It wraps section 01's `fetch_grid`
  (tiled, retried, split on memory or timeout errors, tile cache keyed by the expression hash) and picks a tile
  size that respects the 48 MB request limit.
* `gee_export_to_asset(image, asset_id, grid, ...)` runs one batch `Export.image.toAsset` on the exact grid. It
  reuses an existing asset whose stored expression hash matches, attaches to a task with the same description
  that is still running, and otherwise starts a new one. With `wait=True` it polls until done.
  `gee_wait_for_exports(handles, raise_on_failure=False)` waits for several handles at once with per-task running
  and queue limits and records each terminal state; `gee_cancel_export(handle)` cancels a still-active task whose
  job is then extracted interactively.
* `gee_fetch_asset(asset_id, grid, bands, name)` reads a finished asset with `computePixels`, using large tiles
  because no computation is involved.

In [ ]:
def _s03_call(fn, what: str, tries: Optional[int] = None):
    """Run a small Earth Engine call with backoff on transient errors (quota, 5xx, timeouts); others raise."""

    def _once():
        try:
            return fn()
        except Exception as exc:  # noqa: BLE001 - classified into transient vs permanent
            err = _classify_gee_error(exc, allow_split=False)
            if err is exc:
                raise
            raise err from exc

    def _on_retry(attempt, exc, delay):
        log(f"{what}: transient Earth Engine error (attempt {attempt}), retrying in {delay:.0f}s: "
            f"{str(exc)[:160]}", "WARNING")

    return retry(_once, tries=int(tries or CFG.GEE_MAX_RETRIES), base_delay=3.0, max_delay=90.0,
                 retry_on=(_GEETransientError,), on_retry=_on_retry)


def _s03_getinfo(obj, what: str):
    """``obj.getInfo()`` with transient-error backoff."""
    return _s03_call(lambda: obj.getInfo(), what)


def _s03_default_tile_px(grid: GridSpec, n_bands: int, dtype=np.float32, heavy: bool = True) -> int:
    """Tile side for ``computePixels``.

    Heavy (computed) images use ~64 km tiles, so a request's 30 m medians fit interactive limits. Asset reads use
    the largest tile under the 48 MB limit, capped at ``CFG.GEE_STACK_TILE_PX``.
    """
    cap = int(CFG.GEE_STACK_TILE_PX)
    if heavy:
        cap = min(cap, max(32, int(round(_S03_INTERACTIVE_TILE_GROUND_M / grid.res / 16.0)) * 16))
    return gee_tile_px(n_bands, dtype, cap=cap)


def gee_fetch_grid(image, grid: GridSpec, bands: Sequence[str], name: str, *, tile_px: Optional[int] = None,
                   dtype=np.float32, extra: Any = None, heavy: bool = True, unmask: bool = True,
                   split: bool = True, stats: Optional[dict] = None) -> np.ndarray:
    """Download ``bands`` of ``image`` on ``grid`` as a ``(len(bands), H, W)`` array (shared by sections 03–05).

    * ``name`` is the cache-folder prefix; the folder name also hashes ``grid``, ``extra`` and the serialised
      expression, so any change of server-side parameters gets fresh tiles.
    * For float dtypes the image is ``unmask(NODATA_FLOAT, sameFootprint=False)``-ed before download, so masked
      pixels (also those outside a clip footprint) arrive as the sentinel, and ``fetch_grid`` turns them into NaN.
      Integer outputs are delivered as computed (choose an explicit nodata value server-side).
    * ``heavy=True`` (computed composites) uses ~64 km tiles; ``heavy=False`` (asset reads) uses large tiles.
    * ``split=False`` never subdivides a tile (size/time-out errors then get at most two tries); ``stats`` (dict)
      receives the fetch statistics (tiles, cached, splits, fetched, seconds).
    """
    if ee is None:
        raise RuntimeError("earthengine-api is not available")
    bands = [str(b) for b in bands]
    if not bands:
        raise ValueError("gee_fetch_grid: no bands requested")
    dtype = np.dtype(dtype)
    img = ee.Image(image).select(bands)
    if unmask and dtype.kind == "f":
        img = img.unmask(NODATA_FLOAT, False).toFloat()
    tile = int(tile_px) if tile_px else _s03_default_tile_px(grid, len(bands), dtype, heavy=heavy)
    key = gee_cache_key(name, grid, extra, image=img)
    return fetch_grid(img, grid, bands, tile, key, dtype, allow_split=split, stats=stats)


_S03_ASSET_ROOT_RE = re.compile(r"^projects/[^/]+/assets$")


def _s03_is_not_found(exc: BaseException) -> bool:
    return bool(re.search(r"not found|does not exist|\b404\b", str(exc), re.IGNORECASE))


def _s03_get_asset(asset_id: str) -> Optional[dict]:
    """Asset metadata dict, or None if the asset does not exist (other errors raise)."""
    try:
        return _s03_call(lambda: ee.data.getAsset(asset_id), f"getAsset {asset_id}", tries=3)
    except Exception as exc:  # noqa: BLE001
        if _s03_is_not_found(exc):
            return None
        raise


def _s03_ensure_folder(folder: str) -> None:
    """Create ``folder`` (and missing parents below ``projects/<p>/assets``) as Earth Engine FOLDER assets."""
    folder = folder.rstrip("/")
    if not folder or _S03_ASSET_ROOT_RE.match(folder) or "/" not in folder:
        return
    info = _s03_get_asset(folder)
    if info is not None:
        kind = str(info.get("type", "")).upper()
        if kind and kind not in ("FOLDER", "IMAGE_COLLECTION"):
            raise RuntimeError(f"asset folder path {folder} exists but is a {kind}, not a folder")
        return
    _s03_ensure_folder(folder.rsplit("/", 1)[0])
    try:
        _s03_call(lambda: ee.data.createAsset({"type": "FOLDER"}, folder), f"createAsset {folder}", tries=3)
        log(f"created Earth Engine asset folder {folder}")
    except Exception as exc:  # noqa: BLE001
        if re.search(r"already exists", str(exc), re.IGNORECASE):
            return
        raise RuntimeError(f"cannot create asset folder {folder}: {type(exc).__name__}: {str(exc)[:200]} | fix: "
                           "grant the service account 'Earth Engine Resource Writer' or set "
                           "LST_GEE_EXPORT=interactive") from exc


def _s03_task_description(text: str) -> str:
    """Batch task description: letters, digits, '_' and '-' only, at most 100 characters."""
    return re.sub(r"[^A-Za-z0-9_-]", "_", str(text))[:100]


def _s03_active_task(description: str) -> Optional[dict]:
    """A READY/RUNNING task of this project with exactly ``description`` (e.g. from an interrupted session)."""
    try:
        tasks = _s03_call(lambda: ee.data.getTaskList(), "getTaskList", tries=3) or []
    except Exception as exc:  # noqa: BLE001 - listing is an optimisation; never fatal
        log(f"could not list Earth Engine tasks ({type(exc).__name__}: {str(exc)[:120]}); starting a new task",
            "WARNING")
        return None
    for task in tasks:
        if task.get("description") == description and str(task.get("state")) in ("READY", "RUNNING"):
            return task
    return None


def gee_export_to_asset(image, asset_id: str, grid: GridSpec, *, description: Optional[str] = None,
                        pyramiding: str = "mean", max_pixels: float = 1e13, wait: bool = True,
                        timeout_s: Optional[float] = None, poll_s: Optional[float] = None,
                        reuse: bool = True) -> dict:
    """Batch-export ``image`` to the Earth Engine asset ``asset_id`` on exactly ``grid`` (shared by 03–05).

    The export uses ``crs`` + ``crsTransform`` + ``dimensions="WxH"`` (no region/scale), which the API turns into
    the exact pixel rectangle of ``grid``, with pyramiding policy ``pyramiding`` for every band. The SHA-1 of the
    serialised expression is stored as the asset property ``lst_expr_sha1``:

    * an existing asset with the same hash is **reused** (no EECU spent); one with a different hash is deleted and
      re-exported;
    * a READY/RUNNING task with the same description (an interrupted session) is attached to, not duplicated;
    * the parent folder is created if missing.

    Returns a handle ``{asset_id, description, task, task_id, state, reused, digest, started, eecu_s}``. With
    ``wait=True`` this blocks until the task is done (see ``gee_wait_for_exports``) and raises on FAILED,
    CANCELLED or timeout.
    """
    if ee is None:
        raise RuntimeError("earthengine-api is not available")
    if not asset_id or "/" not in asset_id:
        raise ValueError(f"invalid asset id {asset_id!r}")
    description = _s03_task_description(description or asset_id.rsplit("/", 1)[-1])
    digest = _gee_expression_digest(image)
    handle = {"asset_id": asset_id, "description": description, "task": None, "task_id": None, "state": None,
              "reused": False, "digest": digest, "started": time.time(), "eecu_s": None}
    _s03_ensure_folder(asset_id.rsplit("/", 1)[0])
    existing = _s03_get_asset(asset_id)
    if existing is not None:
        stored = (existing.get("properties") or {}).get(_S03_ASSET_PROPERTY)
        if reuse and stored == digest:
            log(f"asset {asset_id} already exported with the same expression - reused (no EECU spent)")
            handle.update(state="COMPLETED", reused=True)
            return handle
        log(f"asset {asset_id} exists with a different expression hash ({stored}); deleting and re-exporting",
            "WARNING")
        _s03_call(lambda: ee.data.deleteAsset(asset_id), f"deleteAsset {asset_id}", tries=3)
    running = _s03_active_task(description)
    if running is not None:
        log(f"attaching to running Earth Engine task {running.get('id')} ({description}, state "
            f"{running.get('state')}) instead of starting a duplicate")
        handle.update(task_id=running.get("id"), state=str(running.get("state")))
    else:
        export_image = ee.Image(image).set(_S03_ASSET_PROPERTY, digest)
        task = ee.batch.Export.image.toAsset(
            image=export_image, description=description, assetId=asset_id,
            pyramidingPolicy={".default": str(pyramiding)}, dimensions=f"{int(grid.width)}x{int(grid.height)}",
            crs=grid.crs, crsTransform=grid.transform_list(), maxPixels=int(max_pixels))
        _s03_call(task.start, f"start export {description}", tries=4)
        handle.update(task=task, task_id=getattr(task, "id", None), state="READY")
        log(f"started Earth Engine export {description} -> {asset_id} (task {handle['task_id']}, "
            f"{grid.width}x{grid.height} px @ {grid.res} m)")
    if wait:
        gee_wait_for_exports([handle], timeout_s=timeout_s, poll_s=poll_s)
    return handle


def _s03_task_status(handle: dict) -> dict:
    """Current status dict of an export handle (own Task object or an attached task id)."""
    if handle.get("task") is not None:
        return _s03_call(handle["task"].status, f"status {handle['description']}", tries=4)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")   # getTaskStatus is deprecated but works on every API version
        statuses = _s03_call(lambda: ee.data.getTaskStatus(handle["task_id"]),
                             f"status {handle['description']}", tries=4)
    return statuses[0] if statuses else {"state": "UNKNOWN"}


def _s03_failure_hint(message: str) -> str:
    low = message.lower()
    if "permission" in low or "denied" in low or "not authorized" in low:
        return "grant the service account 'Earth Engine Resource Writer' on the project (or LST_GEE_EXPORT=interactive)"
    if "maxpixels" in low or "too many pixels" in low:
        return "the export exceeds maxPixels: raise max_pixels"
    if "memory" in low or "computation timed out" in low:
        return ("the composite exceeded batch memory/time: rerun (tasks are retried server side) or use "
                "LST_GEE_EXPORT=interactive, whose tiles split automatically")
    if "quota" in low or "eecu" in low:
        return "the project's EECU quota is exhausted: wait for the monthly reset or move to the Contributor tier"
    return "inspect the task in the Earth Engine Code Editor Tasks tab"


_S03_EXPORT_TERMINAL = ("COMPLETED", "FAILED", "CANCELLED", "TIMEOUT_RUNNING", "TIMEOUT_QUEUED")


def gee_wait_for_exports(handles: Sequence[dict], *, timeout_s: Optional[float] = None,
                         poll_s: Optional[float] = None, queue_s: Optional[float] = None,
                         raise_on_failure: bool = True, stats: Optional[dict] = None) -> list:
    """Poll export handles until every one is terminal (backoff ``poll_s`` × 1.5 up to 5 min).

    Per-task limits: a task times out ``timeout_s`` (default ``LST_GEE_EXPORT_TIMEOUT_S`` = 3 h) after the first poll
    that saw it RUNNING (``h["running_since"]``), and ``queue_s`` (default ``LST_GEE_EXPORT_QUEUE_S`` = 4 h) after
    the wait began while it is not yet RUNNING (noncommercial projects run about 2 batch tasks at a time, so queueing
    is normal).

    * ``raise_on_failure=True``: FAILED (with the server's error and a fix hint), CANCELLED/CANCEL_REQUESTED or a
      vanished task raise ``RuntimeError``; a per-task timeout raises ``TimeoutError`` (the task keeps running on the
      server; rerun to attach to it).
    * ``raise_on_failure=False``: each handle ends with ``h["state"]`` in ``COMPLETED | FAILED | CANCELLED |
      TIMEOUT_RUNNING | TIMEOUT_QUEUED`` (plus ``h["error"]``); the others keep being polled. The caller cancels
      still-active tasks (``gee_cancel_export``) and extracts those jobs interactively.

    Every poll logs the number of RUNNING tasks; the maximum observed concurrency is logged at the end and written to
    ``stats["max_running"]`` (with ``polls`` and ``seconds``). Every poll checks the run's time budget and raises
    ``LSTStop`` when it is exhausted (tasks keep running server side; a later session attaches to them by
    description). Completed handles get ``eecu_s`` (batch EECU-seconds, when the server reports it).
    """
    timeout_s = float(_S03_EXPORT_TIMEOUT_S if timeout_s is None else timeout_s)
    queue_s = float(_S03_EXPORT_QUEUE_S if queue_s is None else queue_s)
    delay = float(_S03_EXPORT_POLL_S if poll_s is None else poll_s)
    t0 = time.time()
    pending = [h for h in handles if h.get("state") not in _S03_EXPORT_TERMINAL]
    for h in pending:
        h.setdefault("running_since", None)
        h.setdefault("error", None)
    unknown_since: dict = {}
    max_running, polls = 0, 0
    _exhausted = globals().get("budget_exhausted")
    _check = globals().get("check_budget")

    def _fail(h: dict, state: str, message: str, exc_type=RuntimeError) -> None:
        if raise_on_failure:
            raise exc_type(message)
        h["state"], h["error"] = state, message
        log(f"export {h['description']} ended {state}: {message[:300]}", "WARNING")

    while pending:
        if _exhausted is not None and _exhausted():
            if _check is not None:
                _check(f"waiting for Earth Engine exports ({len(pending)} pending)")
        still, n_running = [], 0
        now = time.time()
        for h in pending:
            status = _s03_task_status(h)
            state = str(status.get("state", "UNKNOWN"))
            h["state"] = state
            if state == "COMPLETED":
                h["eecu_s"] = status.get("batch_eecu_usage_seconds")
                if _s03_get_asset(h["asset_id"]) is None:
                    _fail(h, "FAILED", f"export {h['description']} reports COMPLETED but asset {h['asset_id']} "
                                       "does not exist")
                    continue
                log(f"export {h['description']} COMPLETED in {time.time() - h['started']:.0f}s"
                    + (f" ({float(h['eecu_s']) / 3600:.2f} EECU-h)" if h["eecu_s"] else ""))
                continue
            if state == "FAILED":
                message = str(status.get("error_message") or "no error message")
                _fail(h, "FAILED", f"Earth Engine export {h['description']} FAILED: {message[:400]} | fix: "
                                   f"{_s03_failure_hint(message)}")
                continue
            if state in ("CANCELLED", "CANCEL_REQUESTED"):
                _fail(h, "CANCELLED", f"Earth Engine export {h['description']} was {state} (by a user or the "
                                      "server); rerun the section to start it again")
                continue
            if state in ("UNKNOWN", "UNSUBMITTED"):
                first = unknown_since.setdefault(h["description"], now)
                if now - first > 600:
                    _fail(h, "FAILED", f"Earth Engine export {h['description']} has state {state} for > 10 min "
                                       "(task not found)")
                    continue
            if state == "RUNNING":
                n_running += 1
                if h.get("running_since") is None:
                    h["running_since"] = now
            if h.get("running_since") is not None and now - h["running_since"] > timeout_s:
                _fail(h, "TIMEOUT_RUNNING",
                      f"Earth Engine export {h['description']} still running after "
                      f"{(now - h['running_since']) / 60:.0f} min (limit LST_GEE_EXPORT_TIMEOUT_S="
                      f"{timeout_s:.0f} s); the task keeps running on the server - rerun to attach to it",
                      TimeoutError)
                continue
            if h.get("running_since") is None and now - t0 > queue_s:
                _fail(h, "TIMEOUT_QUEUED",
                      f"Earth Engine export {h['description']} still {state} (not started) after "
                      f"{(now - t0) / 60:.0f} min (limit LST_GEE_EXPORT_QUEUE_S={queue_s:.0f} s)", TimeoutError)
                continue
            still.append(h)
        polls += 1
        max_running = max(max_running, n_running)
        pending = still
        if not pending:
            break
        elapsed = time.time() - t0
        log(f"export poll {polls}: {n_running} RUNNING, {len(pending)} not finished ("
            + ", ".join(f"{h['description'][-24:]}={h['state']}" for h in pending)
            + f"), {elapsed / 60:.1f} min elapsed")
        time.sleep(max(0.01, delay))
        delay = min(_S03_EXPORT_POLL_MAX_S, delay * 1.5)
    seconds = time.time() - t0
    if handles:
        log(f"Earth Engine exports: {sum(h.get('state') == 'COMPLETED' for h in handles)}/{len(handles)} COMPLETED "
            f"in {seconds / 60:.1f} min; maximum observed concurrency {max_running} RUNNING task(s)")
    if stats is not None:
        stats.update({"max_running": int(max_running), "polls": int(polls), "seconds": round(seconds, 1),
                      "states": {h["description"]: h.get("state") for h in handles}})
    return list(handles)


def gee_cancel_export(handle: dict) -> bool:
    """Cancel a still READY/RUNNING export task of ``handle`` (guarded, logged); True when a cancel was sent."""
    task_id = handle.get("task_id")
    if not task_id or handle.get("state") not in ("TIMEOUT_RUNNING", "TIMEOUT_QUEUED", "READY", "RUNNING"):
        return False
    cancel = getattr(getattr(ee, "data", None), "cancelTask", None)
    if not callable(cancel):
        log(f"cannot cancel export {handle.get('description')} (this earthengine-api has no ee.data.cancelTask); it "
            "keeps running on the server", "WARNING")
        return False
    try:
        _s03_call(lambda: cancel(task_id), f"cancelTask {task_id}", tries=3)
    except Exception as exc:  # noqa: BLE001 - a failed cancel only costs EECU
        log(f"could not cancel export {handle.get('description')} ({task_id}): {type(exc).__name__}: "
            f"{str(exc)[:160]}", "WARNING")
        return False
    log(f"cancelled Earth Engine export {handle.get('description')} ({task_id}, {handle.get('state')}); the job is "
        "extracted interactively instead", "WARNING")
    return True


def gee_fetch_asset(asset_id: str, grid: GridSpec, bands: Sequence[str], name: str,
                    dtype=np.float32, stats: Optional[dict] = None) -> np.ndarray:
    """Read a finished asset on ``grid`` with ``computePixels`` (large tiles, no server-side computation).

    The asset's ``updateTime`` is part of the tile-cache key, so tiles of an older asset with the same id are not
    reused.
    """
    info = _s03_get_asset(asset_id)
    if info is None:
        raise RuntimeError(f"asset {asset_id} does not exist")
    return gee_fetch_grid(ee.Image(asset_id), grid, bands, name, dtype=dtype, heavy=False,
                          extra={"asset": asset_id, "updated": info.get("updateTime")}, stats=stats)

## 3.2 Server-side Landsat recipe
Per-sensor preparation (masks, scaling, Roy harmonisation), the filters for season, window and path/row,
per-date path mosaics, the combined median+count reduction and the aggregation to the fine grid. Everything here
only *builds* Earth Engine expressions. Nothing is computed until a `getInfo`, `computePixels` or export.

In [ ]:
def _s03_season_ranges(years: Sequence[int], season: Sequence[str], start: Optional[str] = None,
                       end: Optional[str] = None) -> list:
    """``[(start, end_exclusive), ...]`` ISO dates of the [MM-DD start, MM-DD end] (inclusive) season of each year,
    clipped to an optional ``[start, end)`` period (e.g. a sensor's operating period); empty ranges are dropped."""
    lo = pd.Timestamp(start) if start else None
    hi = pd.Timestamp(end) if end else None
    out = []
    for y in years:
        a = pd.Timestamp(f"{int(y)}-{season[0]}")
        b = pd.Timestamp(f"{int(y)}-{season[1]}") + pd.Timedelta(days=1)
        if lo is not None:
            a = max(a, lo)
        if hi is not None:
            b = min(b, hi)
        if a < b:
            out.append((a.strftime("%Y-%m-%d"), b.strftime("%Y-%m-%d")))
    return out


def _s03_season_collection(col, years: Sequence[int], season: Sequence[str], start: Optional[str] = None,
                           end: Optional[str] = None):
    """``col`` restricted to the season of each year (``_s03_season_ranges``): one ``filterDate`` per year, merged.

    Do not use an ``ee.Filter.Or`` of date ranges on large collections instead: on ERA5-Land hourly it made the
    first real pilot fail with "User memory limit exceeded" (reproduced locally; still failing after a ``filterDate``
    on the whole window), while the per-year ``filterDate`` merge of the same images took ~10 s. Returns an empty
    collection when no range is left.
    """
    ranges = _s03_season_ranges(years, season, start, end)
    if not ranges:
        return col.filterDate("1900-01-01", "1900-01-02")
    out = col.filterDate(*ranges[0])
    for a, b in ranges[1:]:
        out = out.merge(col.filterDate(a, b))
    return out


def _s03_season_filter(years: Sequence[int], season: Sequence[str]):
    """``ee.Filter`` passing images inside [MM-DD start, MM-DD end] (inclusive) of any of ``years``.

    For small, already filtered collections only; on a large collection use ``_s03_season_collection``.
    """
    filters = [ee.Filter.date(a, b) for a, b in _s03_season_ranges(years, season)]
    return filters[0] if len(filters) == 1 else ee.Filter.Or(*filters)


def _s03_pathrow_filter(pathrows: Sequence[str]):
    """``ee.Filter`` for an explicit list of WRS-2 path/rows ("146/040")."""
    parts = []
    for pr in pathrows:
        p, r = (int(v) for v in pr.split("/"))
        parts.append(ee.Filter.And(ee.Filter.eq("WRS_PATH", p), ee.Filter.eq("WRS_ROW", r)))
    return parts[0] if len(parts) == 1 else ee.Filter.Or(*parts)


def _s03_sensors_for(y0: int, y1: int) -> list:
    """Sensors whose catalogue period overlaps the window [y0-01-01, (y1+1)-01-01) (client-side pruning)."""
    lo, hi = pd.Timestamp(f"{y0}-01-01"), pd.Timestamp(f"{y1 + 1}-01-01")
    keep = []
    for key, s in _S03_SENSORS.items():
        start = pd.Timestamp(s["start"])
        end = pd.Timestamp(s["end"]) if s["end"] else pd.Timestamp.max
        if start < hi and lo < end:
            keep.append(key)
    return keep


def _s03_raw_collection(sensor: str, years: Sequence[int], season: Sequence[str], pathrows: Sequence[str],
                        roi_geom):
    """Filtered (unmapped) C2 L2 collection of one sensor: season of the window years (within the sensor's operating
    period), path/rows, ROI."""
    s = _S03_SENSORS[sensor]
    col = _s03_season_collection(ee.ImageCollection(s["asset"]), years, season, s["start"], s["end"] or "2100-01-01")
    return col.filterBounds(roi_geom).filter(_s03_pathrow_filter(pathrows))


def _s03_prep_fn(family: str):
    """Server-side per-scene preparation for a sensor family ("tm" = TM/ETM+, "oli" = OLI/OLI-2).

    Returns bands blue, green, red, nir, swir1, swir2 (OLI-equivalent reflectance 0-1) and lst (°C), masked by the
    SPEC §2.1 rules, plus ``s03_date`` (YYYY-MM-dd of ``system:time_start``) for the per-date path mosaics.
    """
    sr_bands, st_band = _S03_SR_BANDS[family], _S03_ST_BAND[family]

    def _prep(img):
        img = ee.Image(img)
        qa = img.select("QA_PIXEL")
        ok = (qa.bitwiseAnd(_S03_QA_CLEAR_BITS).eq(0)
              .And(img.select("QA_RADSAT").eq(0))
              .And(img.select("ST_QA").multiply(0.01).lte(_S03_ST_QA_MAX_K))
              .And(img.select("ST_CDIST").multiply(0.01).gte(_S03_ST_CDIST_MIN_KM)))
        sr = img.select(sr_bands, _S03_SR_NAMES).multiply(_S03_SR_SCALE).add(_S03_SR_OFFSET)
        in_range = sr.reduce(ee.Reducer.min()).gte(0).And(sr.reduce(ee.Reducer.max()).lte(1))
        if family == "tm":
            sr = sr.multiply(ee.Image.constant(_S03_ROY_SLOPES)).add(ee.Image.constant(_S03_ROY_INTERCEPTS))
        st_raw = img.select(st_band)
        lst = (st_raw.multiply(_S03_ST_SCALE).add(_S03_ST_OFFSET_K).subtract(273.15).rename("lst")
               .updateMask(st_raw.gt(0)))
        out = sr.rename(_S03_SR_NAMES).addBands(lst).updateMask(ok).updateMask(in_range).toFloat()
        out = ee.Image(out.copyProperties(img, ["system:time_start"]))
        return out.set("s03_date", ee.Date(img.get("system:time_start")).format("YYYY-MM-dd"))

    return _prep


def _s03_path_collection(year: int, path: str, season: Sequence[str], roi_geom, bands: Sequence[str]):
    """Prepared scenes of one WRS-2 path in the window/season, mosaicked per acquisition date (one pass = one obs)."""
    y0, y1 = CFG.WINDOWS[year]
    years = list(range(int(y0), int(y1) + 1))
    pathrows = [pr for pr in _S03_PATHROWS if pr.startswith(path + "/")]
    merged = None
    for sensor in _s03_sensors_for(y0, y1):
        col = (_s03_raw_collection(sensor, years, season, pathrows, roi_geom)
               .map(_s03_prep_fn(_S03_SENSORS[sensor]["family"])).select(list(bands)))
        merged = col if merged is None else merged.merge(col)
    if merged is None:
        return None
    dates = merged.aggregate_array("s03_date").distinct()

    def _per_date(d):
        return merged.filter(ee.Filter.eq("s03_date", d)).mosaic().set("s03_date", d)

    return ee.ImageCollection.fromImages(dates.map(_per_date))


def _s03_projection_30m():
    """30 m working projection on the native Landsat lattice of the grid CRS.

    USGS Collection-2 products in UTM have pixel corners at 15 m mod 30 m (the translations of their STAC
    ``proj:transform`` end in ...15 / ...85 m), so the composite grid uses translations of 15 m: the 30 m pixels
    then coincide with the scenes' own pixels and no half-pixel resampling happens before reduceResolution.
    """
    return ee.Projection(CFG.CRS, [_S03_COMPOSITE_SCALE_M, 0, 15, 0, -_S03_COMPOSITE_SCALE_M, 15])


def _s03_to_fine(image, fine: GridSpec):
    """Mask- and area-weighted mean of a 30 m image onto ``fine`` (exact grid transform)."""
    ratio = (fine.res / _S03_COMPOSITE_SCALE_M) ** 2
    max_pixels = int(min(65535, max(1024, math.ceil(ratio * 2.5))))
    return (image.setDefaultProjection(_s03_projection_30m())
            .reduceResolution(reducer=ee.Reducer.mean(), bestEffort=False, maxPixels=max_pixels)
            .reproject(crs=fine.crs, crsTransform=fine.transform_list()))


def _s03_valid_fraction(image, fine: GridSpec):
    """Fraction (0-1) of each fine pixel covered by valid (unmasked) 30 m pixels of a single-band image."""
    return _s03_to_fine(image.mask().gt(0).unmask(0, False).toFloat(), fine)


def _s03_masked_bands(names: Sequence[str]):
    """Fully masked float image with the given band names (stand-in for an empty composite)."""
    return ee.Image.constant([0.0] * len(names)).rename(list(names)).toFloat().updateMask(0)


def _s03_path_image(year: int, path: str, fine: GridSpec, roi_geom, has_main: bool, has_spring: bool):
    """Fine-grid bands ``<band>_<path>`` (``_S03_PATH_BANDS``) of one WRS-2 path for one frame.

    One combined median+count reduction over the per-date mosaics; indices and albedo from the same median;
    a red/NIR-only Feb-Mar median for the spring NDVI. Empty composites are replaced by fully masked bands (and
    a zero count), so the image always has the same bands. Missing data stays masked; it is never zero-filled.
    """
    proj = _s03_projection_30m()
    vmin = float(_S03_PIXEL_MIN_VALID_FRAC)
    if has_main:
        daily = _s03_path_collection(year, path, CFG.SEASON, roi_geom, _S03_SR_NAMES + ["lst"])
        red = daily.reduce(ee.Reducer.median().combine(reducer2=ee.Reducer.count(), sharedInputs=True))
        comp = red.setDefaultProjection(proj)
        med = comp.select([f"{b}_median" for b in _S03_SR_NAMES + ["lst"]], _S03_SR_NAMES + ["lst"])
        lst = med.select("lst")
        count = comp.select("lst_count").unmask(0, False).toFloat().rename("nobs")
        ndvi = med.normalizedDifference(["nir", "red"]).rename("ndvi")
        ndbi = med.normalizedDifference(["swir1", "nir"]).rename("ndbi")
        mndwi = med.normalizedDifference(["green", "swir1"]).rename("mndwi")
        albedo = med.expression(
            f"{_S03_LIANG['blue']} * b + {_S03_LIANG['red']} * r + {_S03_LIANG['nir']} * n + "
            f"{_S03_LIANG['swir1']} * s1 + {_S03_LIANG['swir2']} * s2 + ({_S03_LIANG['const']})",
            {"b": med.select("blue"), "r": med.select("red"), "n": med.select("nir"), "s1": med.select("swir1"),
             "s2": med.select("swir2")}).rename("albedo")
        vf_lst = _s03_valid_fraction(lst, fine)
        vf_sr = _s03_valid_fraction(ndvi, fine)
        lst_f = _s03_to_fine(lst, fine).updateMask(vf_lst.gte(vmin))
        spec_f = _s03_to_fine(ee.Image.cat([ndvi, ndbi, mndwi, albedo]), fine).updateMask(vf_sr.gte(vmin))
        nobs_f = _s03_to_fine(count, fine)
    else:
        lst_f = _s03_masked_bands(["lst"])
        spec_f = _s03_masked_bands(["ndvi", "ndbi", "mndwi", "albedo"])
        nobs_f = ee.Image.constant(0.0).rename("nobs").toFloat()
    if has_spring:
        spring = _s03_path_collection(year, path, CFG.SPRING_SEASON, roi_geom, ["red", "nir"]).median()
        spring = spring.setDefaultProjection(proj)
        ndvi_s = spring.normalizedDifference(["nir", "red"]).rename("ndvi_spring")
        spring_f = _s03_to_fine(ndvi_s, fine).updateMask(_s03_valid_fraction(ndvi_s, fine).gte(vmin))
    else:
        spring_f = _s03_masked_bands(["ndvi_spring"])
    stack = ee.Image.cat([lst_f, nobs_f, spec_f, spring_f]).select(_S03_PATH_BANDS)
    return stack.rename([f"{b}_{path}" for b in _S03_PATH_BANDS]).toFloat()


def _s03_frame_image(year: int, fine: GridSpec, roi_geom, clip_geom, availability: dict):
    """14-band fine-grid image of one frame (7 bands × paths 146/147), clipped to the buffered ROI.

    ``availability[path] = (has_main_scenes, has_spring_scenes)``.
    """
    parts = [_s03_path_image(year, p, fine, roi_geom, *availability[p]) for p in _S03_PATHS]
    return ee.Image.cat(parts).clip(clip_geom).toFloat()


def _s03_frame_bands() -> list:
    return [f"{b}_{p}" for p in _S03_PATHS for b in _S03_PATH_BANDS]


def _s03_roi_geometry(dilate_cells: int = 0):
    """ROI as an ``ee.Geometry`` in the grid CRS: union of the ROI cells (optionally dilated by whole cells).

    Built client-side from ``ROI_MASK``, so it is exactly the analysis-grid ROI, with a vertex count of a few
    thousand. Far cheaper per request than the detailed administrative polygons. Without shapely the grid
    rectangle is used, which is correct but computes more pixels.
    """
    mask = ROI_MASK
    if dilate_cells > 0:
        mask = ndimage.binary_dilation(mask, iterations=int(dilate_cells))
    if shapely is None:
        log("shapely unavailable: Earth Engine requests use the grid rectangle instead of the ROI polygon", "WARNING")
        return grid_ee_geom(GRID)
    from shapely.geometry import box as _box, mapping as _mapping
    from shapely.ops import unary_union as _union

    boxes = []
    res, x0, y0 = float(GRID.res), float(GRID.x0), float(GRID.y0)
    for r in range(mask.shape[0]):
        row = mask[r].astype(np.int8)
        edges = np.flatnonzero(np.diff(np.concatenate([[0], row, [0]])))
        for c0, c1 in zip(edges[::2], edges[1::2]):
            boxes.append(_box(x0 + c0 * res, y0 - (r + 1) * res, x0 + c1 * res, y0 - r * res))
    if not boxes:
        raise RuntimeError("ROI_MASK is empty; cannot build the Earth Engine ROI geometry")
    geom = _union(boxes).buffer(0)
    gj = _mapping(geom)
    gj = {"type": gj["type"], "coordinates": json.loads(json.dumps(gj["coordinates"]))}
    return ee.Geometry(gj, CFG.CRS, False)


# Public names for sections 04-05 (same behaviour as the private helpers above).
gee_call = _s03_call                    # gee_call(fn, what, tries=None): small EE call with transient backoff
gee_getinfo = _s03_getinfo              # gee_getinfo(obj, what): obj.getInfo() with transient backoff
gee_season_filter = _s03_season_filter  # gee_season_filter(years, ("04-01", "06-30")) -> ee.Filter (inclusive end)
gee_season_collection = _s03_season_collection  # (col, years, season, start=None, end=None) -> merged per-year filterDate
gee_roi_geometry = _s03_roi_geometry    # gee_roi_geometry(dilate_cells=0) -> ROI cell-union ee.Geometry (grid CRS)

## 3.3 Scene metadata, overpass time and ERA5-Land heating rate
One `getInfo` per frame returns everything client-side needs: per sensor, the scene lists (aggregate_array of path,
row, `system:time_start`, SCENE_CENTER_TIME, CLOUD_COVER_LAND, CLOUD_COVER) for the Apr–Jun and Feb–Mar
composites, plus the ERA5-Land mean skin temperature at 04 and 06 UTC over the ROI. `aggregate_array` skips
null values, so a list shorter than the collection would mis-align scenes. When that happens the scenes are
re-read as one feature per image, which keeps the nulls.

In [ ]:
_S03_SCENE_PROPS = {"path": "WRS_PATH", "row": "WRS_ROW", "time": "system:time_start", "sct": "SCENE_CENTER_TIME",
                    "ccl": "CLOUD_COVER_LAND", "cc": "CLOUD_COVER"}
_S03_SCT_RE = re.compile(r"(\d{1,2}):(\d{2}):(\d{2}(?:\.\d+)?)")


def _s03_parse_utc_hours(sct: Any, time_ms: Any) -> Optional[float]:
    """UTC decimal hours of a scene from SCENE_CENTER_TIME ("05:18:31.447Z"), else from ``system:time_start``."""
    if isinstance(sct, str):
        m = _S03_SCT_RE.search(sct)
        if m:
            h, mi, s = int(m.group(1)), int(m.group(2)), float(m.group(3))
            if 0 <= h < 24 and 0 <= mi < 60 and 0 <= s < 61:
                return h + mi / 60.0 + s / 3600.0
    if time_ms is not None:
        try:
            t = pd.Timestamp(int(time_ms), unit="ms", tz="UTC")
            return t.hour + t.minute / 60.0 + (t.second + t.microsecond / 1e6) / 3600.0
        except (ValueError, OverflowError, TypeError):
            return None
    return None


def _s03_scene_records(sensor: str, block: dict) -> Optional[list]:
    """Scene dicts from one sensor's aggregate_array block, or None if the lists are mis-aligned."""
    n = int(block.get("n") or 0)
    lists = {k: list(block.get(k) or []) for k in _S03_SCENE_PROPS}
    if n == 0:
        return []
    if any(len(lists[k]) != n for k in lists):
        return None
    return [{"sensor": sensor, **{k: lists[k][i] for k in lists}} for i in range(n)]


def _s03_fallback_records(sensor: str, col) -> list:
    """Scene dicts via one Feature per image (keeps null properties aligned)."""
    def _feat(img):
        return ee.Feature(None, {k: ee.Image(img).get(v) for k, v in _S03_SCENE_PROPS.items()})

    info = _s03_getinfo(ee.FeatureCollection(col.map(_feat)), f"scene features {sensor}")
    return [{"sensor": sensor, **{k: (f.get("properties") or {}).get(k) for k in _S03_SCENE_PROPS}}
            for f in (info or {}).get("features", [])]


def _s03_scene_table(records: list) -> pd.DataFrame:
    """Normalise scene records: path/row strings, UTC date, IST hour, land cloud cover and the clear flag."""
    cols = ["sensor", "pathrow", "path", "date", "utc_h", "ist_h", "cloud_land", "clear"]
    rows = []
    for rec in records:
        try:
            path, row = int(rec["path"]), int(rec["row"])
        except (TypeError, ValueError, KeyError):
            continue
        utc_h = _s03_parse_utc_hours(rec.get("sct"), rec.get("time"))
        date = (pd.Timestamp(int(rec["time"]), unit="ms", tz="UTC").strftime("%Y-%m-%d")
                if rec.get("time") is not None else None)
        cloud = rec.get("ccl")
        if cloud is None or not np.isfinite(float(cloud)) or float(cloud) < 0:
            cloud = rec.get("cc")
        cloud = None if cloud is None or not np.isfinite(float(cloud)) or float(cloud) < 0 else float(cloud)
        rows.append({"sensor": rec["sensor"], "pathrow": f"{path}/{row:03d}", "path": str(path), "date": date,
                     "utc_h": utc_h, "ist_h": None if utc_h is None else (utc_h + _S03_IST_OFFSET_H) % 24.0,
                     "cloud_land": cloud, "clear": cloud is None or cloud <= _S03_CLEAR_SCENE_MAX_CLOUD})
    return pd.DataFrame(rows, columns=cols)


def _s03_sensor_labels(table: pd.DataFrame) -> list:
    """Sensor labels in canonical order; Landsat 7 marked "(SLC-off)" when any of its scenes postdates 2003-05-31."""
    labels = []
    for key in _S03_SENSORS:
        sub = table[table["sensor"] == key]
        if sub.empty:
            continue
        label = _S03_SENSORS[key]["label"]
        if key == "L7" and (sub["date"].dropna() >= _S03_L7_SLC_OFF_DATE).any():
            label += " (SLC-off)"
        labels.append(label)
    return labels


def _s03_summarise_scenes(table: pd.DataFrame) -> dict:
    """FRAME_META day scene fields from a scene table (clear scenes; all-scene counts for the QA record)."""
    clear = table[table["clear"]]
    per_pr = {pr: int((clear["pathrow"] == pr).sum()) for pr in _S03_PATHROWS}
    per_pr_all = {pr: int((table["pathrow"] == pr).sum()) for pr in _S03_PATHROWS}
    ist = clear["ist_h"].dropna()
    if ist.empty:
        ist = table["ist_h"].dropna()
    return {"sensors": _s03_sensor_labels(table), "scenes": int(sum(per_pr.values())), "scenes_per_pathrow": per_pr,
            "scenes_all": int(sum(per_pr_all.values())), "scenes_per_pathrow_all": per_pr_all,
            "mean_overpass_ist_h": float(ist.mean()) if not ist.empty else float("nan"),
            "overpass_ist_range": [float(ist.min()), float(ist.max())] if not ist.empty else None,
            "by_sensor": {k: int((table["sensor"] == k).sum()) for k in _S03_SENSORS if (table["sensor"] == k).any()}}


def _s03_heating_rate(era5: Optional[dict]) -> float:
    """``(T06 − T04) / 2`` in K h⁻¹ from the ERA5-Land ROI means (raises if ERA5 returned nothing)."""
    if not era5 or era5.get("t04") is None or era5.get("t06") is None:
        raise RuntimeError(f"ERA5-Land hourly skin temperature returned no ROI mean ({era5}); cannot derive the "
                           "overpass heating rate")
    t04, t06 = float(era5["t04"]), float(era5["t06"])
    if not (200.0 < t04 < 360.0 and 200.0 < t06 < 360.0):
        raise RuntimeError(f"ERA5-Land skin temperatures {t04:.2f} / {t06:.2f} K are implausible")
    return (t06 - t04) / float(_S03_HEATING_UTC_HOURS[1] - _S03_HEATING_UTC_HOURS[0])


def _s03_era5_stats(years: Sequence[int], roi_geom):
    """Server-side ROI means of ERA5-Land skin temperature at 04 and 06 UTC over the Apr-Jun season of ``years``.

    The hour filter is applied to each season-year ``filterDate`` part before merging (as in
    ``_s03_season_collection``): the same images selected through an ``ee.Filter.Or`` of the seasons exceeded Earth
    Engine's per-request memory.
    """
    base = ee.ImageCollection(_S03_ERA5_HOURLY).select("skin_temperature")
    h0, h1 = _S03_HEATING_UTC_HOURS

    def _hour_mean(hour: int, name: str):
        ranges = _s03_season_ranges(years, CFG.SEASON)
        parts = [base.filterDate(a, b).filter(ee.Filter.calendarRange(hour, hour, "hour")) for a, b in ranges]
        col = parts[0]
        for p in parts[1:]:
            col = col.merge(p)
        return col.mean().rename(name)

    t04 = _hour_mean(h0, "t04")
    t06 = _hour_mean(h1, "t06")
    return t04.addBands(t06).reduceRegion(reducer=ee.Reducer.mean(), geometry=roi_geom, scale=_S03_ERA5_SCALE_M,
                                          maxPixels=int(1e9), bestEffort=True)


def _s03_frame_metadata(year: int, roi_geom) -> dict:
    """Scene tables (Apr-Jun and Feb-Mar) and the ERA5-Land heating rate of one frame (one getInfo + fallbacks)."""
    y0, y1 = CFG.WINDOWS[year]
    years = list(range(int(y0), int(y1) + 1))
    sensors = _s03_sensors_for(y0, y1)
    cols = {season: {s: _s03_raw_collection(s, years, CFG.SEASON if season == "main" else CFG.SPRING_SEASON,
                                            _S03_PATHROWS, roi_geom) for s in sensors}
            for season in ("main", "spring")}
    request = {season: {s: ee.Dictionary({"n": col.size(), **{k: col.aggregate_array(v)
                                                               for k, v in _S03_SCENE_PROPS.items()}})
                        for s, col in by_sensor.items()} for season, by_sensor in cols.items()}
    request["era5"] = _s03_era5_stats(years, roi_geom)
    info = _s03_getinfo(ee.Dictionary(request), f"03 metadata {year}") or {}
    tables = {}
    for season in ("main", "spring"):
        records = []
        for s in sensors:
            block = (info.get(season) or {}).get(s) or {}
            recs = _s03_scene_records(s, block)
            if recs is None:
                log(f"03: {year} {season} {s}: aggregate_array lists are mis-aligned (null properties); re-reading "
                    "one feature per scene", "WARNING")
                recs = _s03_fallback_records(s, cols[season][s])
            records += recs
        tables[season] = _s03_scene_table(records)
    return {"main": tables["main"], "spring": tables["spring"], "era5": info.get("era5"), "sensors_queried": sensors}

## 3.4 Path-offset harmonisation and mosaic (client side)
The per-path fine composites are combined in NumPy, so the offset is computed exactly as SPEC §2.1 defines it. The
overlap is compared at ≥ 250 m: at finer grids both paths are first block-averaged. A path without scenes, or an
overlap smaller than 200 pixels, gives δ = 0. That case is reported and recorded, never hidden. Outside the ROI
every band is NaN. Inside it, the clear-observation count is finite, and 0 where no scene was clear.

In [ ]:
def _s03_coarsen_pair(a: np.ndarray, b: np.ndarray, k: int):
    """Block means (factor ``k``) of two fine rasters, trimming any remainder rows/cols."""
    if k <= 1:
        return a, b
    h, w = (a.shape[0] // k) * k, (a.shape[1] // k) * k
    return block_mean(a[:h, :w], k), block_mean(b[:h, :w], k)


def _s03_harmonise_paths(paths: dict, roi_fine: np.ndarray, res_m: int) -> tuple:
    """Harmonise and mosaic per-path fine composites.

    ``paths[p][band]`` (p in "146", "147"; band in ``_S03_PATH_BANDS``) are float arrays with NaN = nodata.
    Returns ``(fine, info)``: ``fine`` maps the RAW_OBS keys to float32 arrays; ``info`` has ``delta`` (median
    146 − 147 over the overlap, °C), ``path_offsets`` ``{"146": +δ/2, "147": −δ/2}`` (value removed from each
    path), ``overlap_px`` and ``method``.
    """
    lst = {p: np.asarray(paths[p]["lst"], dtype=np.float32) for p in _S03_PATHS}
    k = max(1, int(math.ceil(_S03_OFFSET_MIN_RES_M / float(res_m))))
    a, b = _s03_coarsen_pair(lst["146"], lst["147"], k)
    both = np.isfinite(a) & np.isfinite(b)
    n_overlap = int(both.sum())
    has = {p: bool(np.isfinite(lst[p]).any()) for p in _S03_PATHS}
    if n_overlap >= _S03_MIN_OVERLAP_PX:
        delta, method = float(np.median((a - b)[both])), "overlap-median"
    else:
        delta = 0.0
        method = "single-path" if not all(has.values()) else "insufficient-overlap"
    harmonised = {"146": lst["146"] - 0.5 * delta, "147": lst["147"] + 0.5 * delta}

    def _mean_where_both(x, y):
        fx, fy = np.isfinite(x), np.isfinite(y)
        out = np.where(fx & fy, 0.5 * (x + y), np.where(fx, x, y))
        return out.astype(np.float32)

    fine = {"lst_day_f": _mean_where_both(harmonised["146"], harmonised["147"])}
    nobs = np.zeros(roi_fine.shape, dtype=np.float32)
    for p in _S03_PATHS:
        n = np.asarray(paths[p]["nobs"], dtype=np.float32)
        nobs += np.where(np.isfinite(n) & (n > 0), n, 0.0).astype(np.float32)
    fine["lst_day_nobs_f"] = nobs
    for band in ("ndvi", "ndbi", "mndwi", "albedo", "ndvi_spring"):
        fine[_S03_BAND_TO_KEY[band]] = _mean_where_both(np.asarray(paths["146"][band], dtype=np.float32),
                                                        np.asarray(paths["147"][band], dtype=np.float32))
    # Physical plausibility (guards against a wrong scale factor or fill values leaking through).
    bounds = {"lst_day_f": (-20.0, 85.0), "ndvi_f": (-1.0, 1.0), "ndbi_f": (-1.0, 1.0), "mndwi_f": (-1.0, 1.0),
              "albedo_f": (-0.05, 1.05), "ndvi_spring_f": (-1.0, 1.0)}
    rejected = {}
    for key, (lo, hi) in bounds.items():
        bad = np.isfinite(fine[key]) & ((fine[key] < lo) | (fine[key] > hi))
        if bad.any():
            rejected[key] = int(bad.sum())
            fine[key][bad] = np.nan
    for key in fine:
        fine[key][~roi_fine] = np.nan
    info = {"delta": round(delta, 4), "path_offsets": {"146": round(0.5 * delta, 4), "147": round(-0.5 * delta, 4)},
            "overlap_px": n_overlap, "overlap_res_m": int(res_m * k), "method": method, "paths_with_data": has,
            "rejected_implausible": rejected}
    return fine, info

## 3.5 Run: metadata, extraction (interactive or asset), harmonisation, provenance
Each finished frame is stored in the stage cache (`03_landsat_<year>`). The key holds the run identity, the
window, the seasons, the fine grid, the ROI and the recipe version, so a resumed run (or one with an imported
cache) skips the frame without contacting Earth Engine. In asset mode, all missing frames are exported in
parallel first and fetched afterwards.

In [ ]:
def _s03_stage_key(year: int) -> dict:
    roi_digest = hashlib.sha1(np.packbits(ROI_MASK).tobytes()).hexdigest()[:16]
    return CFG.key(year=int(year), window=list(CFG.WINDOWS[year]), season=list(CFG.SEASON),
                   spring=list(CFG.SPRING_SEASON), fine_grid=asdict(GRID_FINE), roi=roi_digest,
                   recipe=_S03_RECIPE_VERSION)


def _s03_availability(main: pd.DataFrame, spring: pd.DataFrame) -> dict:
    """{path: (has Apr-Jun scenes, has Feb-Mar scenes)} from the scene tables (all scenes, clear or not)."""
    return {p: (bool((main["path"] == p).any()), bool((spring["path"] == p).any())) for p in _S03_PATHS}


def _s03_split_paths(stack: np.ndarray) -> dict:
    """(14, Hf, Wf) frame stack -> {path: {band: array}} in ``_s03_frame_bands()`` order."""
    out, i = {}, 0
    for p in _S03_PATHS:
        out[p] = {}
        for b in _S03_PATH_BANDS:
            out[p][b] = stack[i]
            i += 1
    return out


def _s03_check_frame_coverage(year: int, coverage: float, summary: dict, min_coverage: float) -> None:
    """D9: a day frame below ``min_coverage`` of the ROI fine pixels is an acquisition fault -> RuntimeError.

    Raised before the frame is stage-cached (its Earth Engine tiles stay cached, so a rerun after a fix is cheap).
    """
    if coverage >= float(min_coverage):
        return
    y0, y1 = CFG.WINDOWS[year]
    per_pr = ", ".join(f"{pr}: {summary['scenes_per_pathrow'].get(pr, 0)} clear / "
                       f"{summary.get('scenes_per_pathrow_all', {}).get(pr, 0)} all" for pr in _S03_PATHROWS)
    raise RuntimeError(
        f"03: day frame {year} ({y0}-{y1}) has Landsat LST on only {100 * coverage:.1f} % of the ROI fine pixels "
        f"(minimum {100 * float(min_coverage):.0f} %, LST_DAY_MIN_FRAME_COVERAGE). Scenes per path/row: {per_pr}; "
        f"sensors: {', '.join(summary.get('sensors') or []) or 'none'}. Hint: check the QA_PIXEL / ST-band handling "
        "for this sensor (ST_QA, ST_CDIST and the ST DN > 0 mask remove pixels where the ST product is missing); the "
        "Earth Engine tiles are cached, so a rerun is cheap; lower LST_DAY_MIN_FRAME_COVERAGE only if the gap is "
        "real (day frames are always observed or observed-sparse, never modelled).")


def _s03_finalise_frame(year: int, meta: dict, stack: np.ndarray, extraction: str,
                        cost: Optional[dict] = None) -> dict:
    """Harmonise a fetched frame stack and assemble the stage-cache record (fine arrays, day meta, QA, notes).

    Raises RuntimeError when the frame's ROI coverage is below ``_S03_DAY_MIN_FRAME_COVERAGE`` (D9).
    """
    fine, harm = _s03_harmonise_paths(_s03_split_paths(stack), ROI_MASK_FINE, GRID_FINE.res)
    summary = _s03_summarise_scenes(meta["main"])
    spring_summary = _s03_summarise_scenes(meta["spring"])
    roi_f = ROI_MASK_FINE
    coverage = float(np.isfinite(fine["lst_day_f"][roi_f]).mean()) if roi_f.any() else 0.0
    _s03_check_frame_coverage(year, coverage, summary, _S03_DAY_MIN_FRAME_COVERAGE)
    heating = _s03_heating_rate(meta["era5"])
    day = {"sensors": summary["sensors"], "scenes": summary["scenes"],
           "scenes_per_pathrow": dict(summary["scenes_per_pathrow"]),
           "mean_overpass_ist_h": round(float(summary["mean_overpass_ist_h"]), 4),
           "coverage": round(coverage, 4), "heating_rate_k_per_h": round(float(heating), 4),
           "path_offsets": dict(harm["path_offsets"])}
    notes, warnings_ = [], []
    window = CFG.WINDOWS[year]
    missing = [pr for pr, n in summary["scenes_per_pathrow_all"].items() if n == 0]
    if missing:
        warnings_.append(f"day frame {year} ({window[0]}-{window[1]}): NO Landsat scenes on WRS-2 path/row(s) "
                         f"{', '.join(missing)} - their footprint is NaN unless another path covers it")
    sparse_prs = {pr: n for pr, n in summary["scenes_per_pathrow"].items() if n < _S03_SPARSE_SCENES}
    if sparse_prs:
        notes.append(f"Day frame {year} ({window[0]}-{window[1]}): fewer than {_S03_SPARSE_SCENES} clear Landsat "
                     f"scenes on path/row(s) " + ", ".join(f"{pr} ({n})" for pr, n in sparse_prs.items())
                     + " - tiered observed-sparse.")
    if harm["method"] != "overlap-median":
        warnings_.append(f"day frame {year}: path offset not estimated ({harm['method']}, {harm['overlap_px']} "
                         "overlap pixels) - paths mosaicked without offset correction")
    elif abs(harm["delta"]) > 2.0:
        warnings_.append(f"day frame {year}: large path 146-147 offset {harm['delta']:+.2f} °C (different dates or "
                         "weather on the two paths) - removed symmetrically")
    if coverage < 0.95:
        warnings_.append(f"day frame {year}: Landsat LST covers only {100 * coverage:.1f} % of the ROI fine pixels")
    if not 0.5 <= heating <= 6.0:
        warnings_.append(f"day frame {year}: ERA5-Land heating rate {heating:.2f} K/h is outside the expected "
                         "0.5-6 K/h range")
    ist = day["mean_overpass_ist_h"]
    if not np.isfinite(ist) or not 8.5 <= ist <= 11.5:
        warnings_.append(f"day frame {year}: mean overpass {ist} IST h is outside 08:30-11:30")
    if harm["rejected_implausible"]:
        warnings_.append(f"day frame {year}: implausible values set to NaN {harm['rejected_implausible']}")
    sensors_txt = " + ".join(summary["sensors"]) or "none"
    assets = ", ".join(_S03_SENSORS[s]["asset"] for s in meta["sensors_queried"])
    y0, y1 = window
    base = (f"Landsat C2 L2 T1 ({assets}; {sensors_txt}; Landsat 7 only < {_S03_L7_END}), WRS-2 "
            f"{'/'.join(_S03_PATHROWS)}")
    mask_txt = (f"QA_PIXEL bits0-4 clear, QA_RADSAT=0, ST_QA<={_S03_ST_QA_MAX_K:g} K, "
                f"ST_CDIST>={_S03_ST_CDIST_MIN_KM:g} km, SR in [0,1]")
    agg_txt = (f"per-date path mosaics, per-path median, 30 m -> {GRID_FINE.res} m mean (>= "
               f"{100 * _S03_PIXEL_MIN_VALID_FRAC:.0f} % valid area), mosaic of paths 146/147; extraction {extraction}")
    season_txt = f"{CFG.SEASON[0]}..{CFG.SEASON[1]} of {y0}-{y1}"
    sources = {
        "lst_day_f": (f"{base}: ST {_S03_ST_SCALE}*DN+{_S03_ST_OFFSET_K:g} K -> °C, {season_txt}, {mask_txt}, "
                      f"{agg_txt}; path offsets 146 {harm['path_offsets']['146']:+.3f} / 147 "
                      f"{harm['path_offsets']['147']:+.3f} °C removed ({harm['method']}); {summary['scenes']} clear "
                      f"scenes (land cloud <= {_S03_CLEAR_SCENE_MAX_CLOUD:g} %), mean overpass {ist:.2f} h IST; "
                      "not overpass-corrected"),
        "lst_day_nobs_f": (f"{base}: number of clear acquisition dates per pixel (per-path count of per-date mosaics, "
                           f"summed over paths), {season_txt}, {mask_txt}, mean to {GRID_FINE.res} m"),
    }
    spec = (f"{base}: SR*{_S03_SR_SCALE:g}{_S03_SR_OFFSET:+g}, TM/ETM+ -> OLI Roy et al. 2016 OLS, {season_txt}, "
            f"{mask_txt}, from the same per-path median composite as lst_day_f, {agg_txt}")
    sources["ndvi_f"] = f"{spec}; NDVI (nir-red)/(nir+red)"
    sources["ndbi_f"] = f"{spec}; NDBI (swir1-nir)/(swir1+nir)"
    sources["mndwi_f"] = f"{spec}; MNDWI (green-swir1)/(green+swir1)"
    sources["albedo_f"] = (f"{spec}; Liang 2001 albedo 0.356 blue + 0.130 red + 0.373 nir + 0.085 swir1 + "
                           "0.072 swir2 - 0.0018")
    sources["ndvi_spring_f"] = (f"{base}: NDVI of the red/NIR median composite, {CFG.SPRING_SEASON[0]}.."
                                f"{CFG.SPRING_SEASON[1]} of {y0}-{y1} ({spring_summary['scenes_all']} scenes), "
                                f"{mask_txt}, {agg_txt}")
    sources["heating_rate"] = (f"{_S03_ERA5_HOURLY} skin_temperature, {season_txt}, ROI mean at "
                               f"{_S03_HEATING_UTC_HOURS[0]:02d} and {_S03_HEATING_UTC_HOURS[1]:02d} UTC "
                               f"(09:30 / 11:30 IST): (T06 - T04)/2 = {heating:.3f} K/h")
    nobs_roi = fine["lst_day_nobs_f"][roi_f]
    cost = dict(cost or {})
    cost.update({"coverage": round(coverage, 4), "scenes": int(summary["scenes"]),
                 "scenes_all": int(summary["scenes_all"]),
                 "median_nobs": round(float(np.median(nobs_roi)), 2) if nobs_roi.size else None})
    qa = {"summary": summary, "spring": spring_summary, "harmonisation": harm, "era5_k": meta["era5"],
          "scene_table": meta["main"], "extraction": extraction, "cost": cost}
    return {"fine": fine, "day": day, "sources": sources, "notes": notes, "warnings": warnings_, "qa": qa}


def _s03_asset_id(year: int, digest: str) -> str:
    root = (CFG.GEE_ASSET_ROOT or "").rstrip("/")
    if not root:
        raise RuntimeError("GEE_EXPORT=asset needs CFG.GEE_ASSET_ROOT (LST_GEE_ASSET_ROOT or GEE_PROJECT)")
    return f"{root}/landsat_day_{int(year)}_{int(GRID_FINE.res)}m_{digest[:12]}"


def _s03_export_frames(prepared: dict) -> dict:
    """Asset mode: start every frame export, wait with per-task limits, return {year: handle} (terminal states).

    A frame whose export does not end COMPLETED has its task cancelled when still active and is extracted
    interactively by the caller. The maximum observed task concurrency goes to ``_S03_QA['export_concurrency']``.
    """
    handles = {}
    for year, item in prepared.items():
        digest = _gee_expression_digest(item["image"])
        asset_id = _s03_asset_id(year, digest)
        try:
            handles[year] = gee_export_to_asset(item["image"], asset_id, GRID_FINE,
                                                description=f"ncr_lst_v2_landsat_day_{year}_{digest[:10]}",
                                                wait=False)
        except Exception as exc:  # noqa: BLE001 - this frame is extracted interactively instead
            log(f"03: asset export of frame {year} could not start ({type(exc).__name__}: {str(exc)[:200]}); the "
                "frame is extracted interactively", "WARNING")
            handles[year] = {"asset_id": asset_id, "description": f"landsat_day_{year}", "task_id": None,
                             "state": "FAILED", "error": f"export not started: {type(exc).__name__}: {exc}",
                             "reused": False, "eecu_s": None, "started": time.time()}
    stats: dict = {}
    with timer("03 asset exports"):
        gee_wait_for_exports(list(handles.values()), raise_on_failure=False, stats=stats)
    _S03_QA["export_concurrency"] = int(stats.get("max_running", 0))
    for year, h in handles.items():
        cancelled = h.get("state") != "COMPLETED" and gee_cancel_export(h)
        _S03_QA.setdefault(year, {})["export"] = {
            **{k: h.get(k) for k in ("asset_id", "task_id", "state", "error", "reused", "eecu_s", "description")},
            "cancelled": bool(cancelled), "fallback_interactive": h.get("state") != "COMPLETED"}
    return handles


def _s03_run() -> dict:
    """Acquire every observed frame; returns {year: stage record}."""
    roi_geom = _s03_roi_geometry(0)
    clip_geom = _s03_roi_geometry(1)
    bands = _s03_frame_bands()
    records, todo = {}, []
    caches = {y: stage_cache(f"03_landsat_{y}", _s03_stage_key(y)) for y in CFG.FRAMES_OBS}
    for year in CFG.FRAMES_OBS:
        sc = caches[year]
        if sc.hit:
            try:
                records[year] = sc.load()
                _S03_QA.setdefault(year, {})["from_stage_cache"] = True
                continue
            except StageCacheError as exc:
                log(f"{exc}; re-acquiring frame {year}", "WARNING")
        todo.append(year)
    prepared = {}
    for year in todo:
        with timer(f"03 metadata {year}"):
            meta = _s03_frame_metadata(year, roi_geom)
        if meta["main"].empty:
            y0, y1 = CFG.WINDOWS[year]
            raise RuntimeError(f"03: no Landsat C2 L2 T1 scenes found for frame {year} (Apr-Jun {y0}-{y1}, path/rows "
                               f"{_S03_PATHROWS}); check the collections/filters or the Earth Engine catalogue")
        avail = _s03_availability(meta["main"], meta["spring"])
        for p, (has_main, has_spring) in avail.items():
            if not has_main:
                log(f"03: frame {year}: WRS-2 path {p} has NO Apr-Jun scenes - its bands stay masked", "WARNING")
            if not has_spring:
                log(f"03: frame {year}: WRS-2 path {p} has NO Feb-Mar scenes - ndvi_spring stays masked there",
                    "WARNING")
        image = _s03_frame_image(year, GRID_FINE, roi_geom, clip_geom, avail)
        prepared[year] = {"meta": meta, "image": image}
        s = _s03_summarise_scenes(meta["main"])
        log(f"03: frame {year}: {s['scenes']} clear / {s['scenes_all']} scenes {s['scenes_per_pathrow']}, sensors "
            f"{s['sensors']}, mean overpass {s['mean_overpass_ist_h']:.2f} h IST")
    handles = _s03_export_frames(prepared) if (CFG.GEE_EXPORT == "asset" and prepared) else {}
    longest = 0.0
    for year, item in prepared.items():
        # D3: do not start a frame that would not finish inside the time budget (1.25 x the slowest frame so far)
        check_budget(f"03 frame {year}", 1.25 * longest)
        t0 = time.perf_counter()
        fetch_stats: dict = {}
        with timer(f"03 extract {year}"):
            h = handles.get(year)
            if h is not None and h.get("state") == "COMPLETED":
                stack = gee_fetch_asset(h["asset_id"], GRID_FINE, bands, f"s03_asset_{year}", stats=fetch_stats)
                extraction, mode = f"batch asset {h['asset_id']}", "asset"
            else:
                stack = gee_fetch_grid(item["image"], GRID_FINE, bands, f"s03_landsat_{year}", heavy=True,
                                       stats=fetch_stats)
                mode = "interactive" if h is None else f"interactive (asset export {h.get('state')})"
                extraction = f"{mode} computePixels"
            wall = time.perf_counter() - t0
            cost = {"mode": mode, "seconds": round(wall, 1), "tiles": int(fetch_stats.get("tiles", 0)),
                    "tiles_cached": int(fetch_stats.get("cached", 0)), "splits": int(fetch_stats.get("splits", 0)),
                    "tile_px": fetch_stats.get("tile_px")}
            record = _s03_finalise_frame(year, item["meta"], stack, extraction, cost=cost)
            del stack
        longest = max(longest, time.perf_counter() - t0)
        records[year] = caches[year].save(record)
        free_memory()
    return records


def _s03_cost_table() -> pd.DataFrame:
    """Per-frame extraction cost (D2/T-F1-08): mode, wall seconds, tiles, splits, coverage, median nobs, scenes."""
    rows = []
    for year in CFG.FRAMES_OBS:
        cost = (_S03_QA.get(year) or {}).get("cost") or {}
        rows.append({"year": year, "mode": cost.get("mode", "?"), "seconds": cost.get("seconds"),
                     "tiles": cost.get("tiles"), "cached_tiles": cost.get("tiles_cached"),
                     "splits": cost.get("splits"), "coverage": cost.get("coverage"),
                     "median_nobs": cost.get("median_nobs"), "scenes": cost.get("scenes"),
                     "from_stage_cache": bool((_S03_QA.get(year) or {}).get("from_stage_cache"))})
    return pd.DataFrame(rows).set_index("year")


if _S03_ACTIVE:
    with gee_workload_tag("lst-v2-s03"), timer("03 Landsat day LST"):
        _S03_RECORDS = _s03_run()
    for _s03_year in CFG.FRAMES_OBS:
        _s03_rec = _S03_RECORDS[_s03_year]
        RAW_OBS.setdefault(int(_s03_year), {}).update({k: np.ascontiguousarray(v, dtype=np.float32)
                                                       for k, v in _s03_rec["fine"].items()})
        _s03_fm = FRAME_META.setdefault(int(_s03_year), {})
        _s03_fm["window"] = tuple(int(v) for v in CFG.WINDOWS[_s03_year])
        _s03_fm["day"] = dict(_s03_rec["day"])
        for _s03_var, _s03_text in _s03_rec["sources"].items():
            DATA_SOURCES.setdefault(_s03_var, {})[int(_s03_year)] = _s03_text
        for _s03_w in _s03_rec["warnings"]:
            log(f"03: {_s03_w}", "WARNING")
        for _s03_c in _s03_rec["notes"] + [f"Landsat {w}." for w in _s03_rec["warnings"]]:
            if _s03_c not in DATA_CAVEATS:
                DATA_CAVEATS.append(_s03_c)
        _S03_QA.setdefault(int(_s03_year), {}).update(_s03_rec["qa"])
    del _S03_RECORDS, _s03_rec, _s03_fm, _s03_year, _s03_var, _s03_text
    free_memory()

## 3.6 Contract check and quality report
These checks verify the section-3 part of the SPEC §3.2 interface: the fine arrays have shape `(Hf, Wf)`, are
float32 and are NaN outside the ROI, and `FRAME_META[y]["day"]` has exactly the SPEC keys and the five path/rows.
The table and figure show, per frame, the evidence behind the day tier (scenes per path/row), the overpass, the
heating rate, the path offset and the coverage.

In [ ]:
def _s03_validate() -> None:
    hf, wf = GRID_FINE.height, GRID_FINE.width
    for year in CFG.FRAMES_OBS:
        obs = RAW_OBS[year]
        for key in _S03_FINE_KEYS:
            arr = obs[key]
            assert arr.shape == (hf, wf) and arr.dtype == np.float32, (year, key, arr.shape, arr.dtype)
            assert np.isnan(arr[~ROI_MASK_FINE]).all(), f"RAW_OBS[{year}][{key}] must be NaN outside the ROI"
        assert np.isfinite(obs["lst_day_nobs_f"][ROI_MASK_FINE]).all(), "lst_day_nobs_f must be finite in the ROI"
        assert (obs["lst_day_nobs_f"][ROI_MASK_FINE] >= 0).all()
        day = FRAME_META[year]["day"]
        assert set(day) == set(_S03_DAY_META_KEYS), sorted(day)
        assert set(day["scenes_per_pathrow"]) == set(_S03_PATHROWS), day["scenes_per_pathrow"]
        assert day["scenes"] == sum(day["scenes_per_pathrow"].values())
        assert set(day["path_offsets"]) == set(_S03_PATHS)
        assert abs(day["path_offsets"]["146"] + day["path_offsets"]["147"]) < 1e-3, "offsets must be symmetric"
        assert 0.0 <= day["coverage"] <= 1.0 and isinstance(day["sensors"], list) and day["sensors"]
        assert np.isfinite(day["mean_overpass_ist_h"]) and np.isfinite(day["heating_rate_k_per_h"])
        assert tuple(FRAME_META[year]["window"]) == tuple(CFG.WINDOWS[year])
        for key in _S03_FINE_KEYS:
            assert year in DATA_SOURCES.get(key, {}), (key, year)


if _S03_ACTIVE:
    _s03_validate()
    log("section-3 contract check passed")
    _s03_rows = []
    for _s03_y in CFG.FRAMES_OBS:
        _s03_d = FRAME_META[_s03_y]["day"]
        _s03_lst = RAW_OBS[_s03_y]["lst_day_f"]
        _s03_rows.append({
            "year": _s03_y, "window": "-".join(map(str, CFG.WINDOWS[_s03_y])), "sensors": ", ".join(_s03_d["sensors"]),
            "clear_scenes": _s03_d["scenes"], "min_per_pathrow": min(_s03_d["scenes_per_pathrow"].values()),
            "overpass_ist": "{:d}:{:02d}".format(*divmod(int(round(60 * _s03_d["mean_overpass_ist_h"])), 60)),
            "heating_K_h": _s03_d["heating_rate_k_per_h"], "offset_146": _s03_d["path_offsets"]["146"],
            "coverage": _s03_d["coverage"], "lst_mean_c": round(float(np.nanmean(_s03_lst)), 2),
            "corr_to_10:45_c": round((CFG.DAY_REFERENCE_OVERPASS_IST_H - _s03_d["mean_overpass_ist_h"])
                                     * _s03_d["heating_rate_k_per_h"], 2)})
    print("Landsat day frames (raw, harmonised; corr_to_10:45 is applied in section 09)")
    _show(pd.DataFrame(_s03_rows).set_index("year"))
    _s03_cost = _s03_cost_table()
    print("Landsat extraction cost per frame (seconds/tiles of the run that extracted the frame; from_stage_cache = "
          "reused now)" + (f"; max export concurrency {_S03_QA['export_concurrency']}"
                           if "export_concurrency" in _S03_QA else ""))
    _show(_s03_cost)
    # Full-frame equivalent seconds of the frames whose tiles were downloaded (not all read from the tile cache):
    # seconds / fetched tiles x tiles; frames rebuilt from cached tiles say nothing about Earth Engine speed.
    _s03_num = _s03_cost[["seconds", "tiles", "cached_tiles"]].apply(pd.to_numeric, errors="coerce")
    _s03_fetched = _s03_num["tiles"] - _s03_num["cached_tiles"].fillna(0)
    _s03_ok = (_s03_fetched > 0) & _s03_num["seconds"].notna() & _s03_cost["mode"].astype(str).ne("asset")
    _s03_secs = (_s03_num["seconds"] / _s03_fetched * _s03_num["tiles"])[_s03_ok]
    if CFG.PROFILE == "pilot" and len(_s03_secs):
        log(f"03 PILOT: ~{_s03_secs.mean() / 60:.1f} min per frame of downloads ({len(_s03_secs)} frame(s) with "
            f"downloaded tiles) -> projected full-run section 03: ~{_s03_secs.mean() * _S03_FULL_FRAMES / 3600:.1f} h "
            f"fresh ({_S03_FULL_FRAMES} frames), or ~{_s03_secs.mean() * 4 / 3600:.1f} h with this pilot's output "
            "attached (its tiles are reused; 4 new frames). Rough: the 1990-2005 windows are lighter than the dense "
            "2010-2025 ones. Decide one commit vs the LST_STOP_AFTER=06 split with README step 7.", "WARNING")
    elif CFG.PROFILE == "pilot":
        log("03 PILOT: no Landsat tiles were downloaded in this session (all read from the cache), so there is no "
            "Earth Engine speed to project from; see the run that downloaded them.")
    del _s03_cost, _s03_secs, _s03_num, _s03_fetched, _s03_ok
    _s03_n = len(CFG.FRAMES_OBS)
    _s03_ncol = min(4, _s03_n)
    _s03_nrow = int(math.ceil(_s03_n / _s03_ncol))
    _s03_stride = max(1, GRID_FINE.width // 600)
    _s03_all = np.concatenate([RAW_OBS[y]["lst_day_f"][::_s03_stride, ::_s03_stride].ravel() for y in CFG.FRAMES_OBS])
    _s03_lo, _s03_hi = (np.nanpercentile(_s03_all, [1, 99]) if np.isfinite(_s03_all).any() else (20.0, 50.0))
    fig, axes = plt.subplots(_s03_nrow, _s03_ncol, figsize=(4.2 * _s03_ncol, 4.6 * _s03_nrow), squeeze=False,
                             constrained_layout=True)
    for _s03_i, _s03_y in enumerate(CFG.FRAMES_OBS):
        _s03_ax = axes[_s03_i // _s03_ncol][_s03_i % _s03_ncol]
        _s03_im = _s03_ax.imshow(RAW_OBS[_s03_y]["lst_day_f"][::_s03_stride, ::_s03_stride], cmap="inferno",
                                 vmin=_s03_lo, vmax=_s03_hi, interpolation="nearest")
        _s03_ax.set_title(f"{_s03_y} ({FRAME_META[_s03_y]['day']['scenes']} scenes, "
                          f"cov {FRAME_META[_s03_y]['day']['coverage']:.2f})", fontsize=9)
        _s03_ax.set_axis_off()
    for _s03_j in range(_s03_n, _s03_nrow * _s03_ncol):
        axes[_s03_j // _s03_ncol][_s03_j % _s03_ncol].set_axis_off()
    fig.colorbar(_s03_im, ax=axes, shrink=0.6, label="Landsat LST (°C, harmonised, raw overpass)")
    fig.suptitle(f"Landsat C2 L2 day LST on the {GRID_FINE.res} m grid", fontsize=11)
    fig.savefig(CFG.FIG_DIR / "03_landsat_day_lst.png", dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    del _s03_rows, _s03_y, _s03_d, _s03_lst, _s03_all, _s03_lo, _s03_hi, _s03_im, _s03_ax, _s03_i, _s03_n
    del _s03_ncol, _s03_nrow, _s03_stride
    free_memory()

# 4. Night LST, MODIS day validation, night calibrations and the annual series (gee mode only)

This section runs only when `DATA_MODE == "gee"`; in synthetic mode section 02 already produced every global
listed below and this section does nothing (SPEC §0 mode gating).

**What is produced** (SPEC §2.1 last bullet, §2.2, §3.2):

| Global | Content |
|---|---|
| `RAW_OBS[y]["lst_night"]` | Apr–Jun window median night LST at 1 km, °C in **Aqua-equivalent** units |
| `RAW_OBS[y]["lst_night_nobs"]` | number of valid composites (Aqua/Terra 8-day) or daily observations (VIIRS) |
| `RAW_OBS[y]["lst_day_modis"]` | MODIS Terra day LST (QC + view time 10.0–11.5 h), window median at 1 km (validation) |
| `FRAME_META[y]["night"]` | source, sensor, overpass (IST h, measured from the view-time bands), composites, coverage |
| `NIGHT_CALIBRATION` | Terra→Aqua and VIIRS→Aqua per-class linear maps (+ shrunk per-cell residuals) with R², RMSE, n |
| `ANNUAL_SERIES` | 1990–2025 annual Apr–Jun series for NCR + 25 districts (ERA5-Land, MODIS, calibrated VIIRS) |

**Why these products** (`docs/research/gee_datasets_v2.md` §0, §1B):

* *Night ≈ 01:30–02:00 IST.* Aqua MYD11A2 (night pass 02:12 IST, stable 2003–2022) is the reference sensor. By
  2025 Aqua drifted to ~03:20 IST, so the 2025 frame uses VIIRS S-NPP VNP21A1N (02:06 IST, stable 2013–2026).
  For 2000 only Terra exists, whose night pass is ~22:27 IST — 3.5 h earlier, i.e. a warmer, less cooled surface.
  Both substitutes are mapped to Aqua-equivalent values with an **overlap calibration**:
  per dominant GLC_FCS30D class a linear map `aqua = a + b·src` fitted on annual Apr–Jun medians of every cell in
  the overlap years (VIIRS↔Aqua 2013–2021, Terra↔Aqua 2003–2019), plus a per-cell mean residual shrunk towards 0
  with weight `n/(n+3)` (n = overlap years of the cell). The shrinkage is a James–Stein-style compromise: a cell
  with many overlap years keeps most of its persistent local offset (emissivity/algorithm differences between the
  MYD11 split-window and VNP21 TES retrievals are surface-specific, 1–3 K over bare/arid land), a cell with one or
  two years is mostly explained by its class map. The quality of the transfer is checked by leave-one-year-out.
* *QC.* MOD11/MYD11: mandatory QA (bits 0–1) ≤ 1 and LST error (bits 6–7) ≤ 1 (≤ 2 K). VNP21: mandatory QA ≤ 1 and
  LST accuracy (bits 14–15) ≥ 2 (≤ 1.5 K). The catalog does not state the VNP21 `LST_1KM` scale, so it is checked
  at run time: values > 1000 are digital numbers and get × 0.02 (a wrong scale would corrupt LST by 50×).
* *MODIS Terra day* (11:07 IST until 2020) is kept as an **independent 1 km validation** of the Landsat day level and
  for the downscaling demonstration; composites are restricted to a view time of 10.0–11.5 h local solar time so
  the 2022+ orbit drift (10:07 IST in 2025) cannot leak in silently — drifted years are flagged `"drift"`.
* *Annual series 1990–2025.* ERA5-Land hourly skin temperature at 05 UTC (10:30 IST) and 20 UTC (01:30 IST),
  daily-aggregate Tmax/Tmin and monthly soil moisture over NCR and every district (server-side `reduceRegions`, one
  batched request per variable — never a per-district `getInfo` loop); satellite series (Terra day, Aqua night,
  VIIRS night calibrated to Aqua) are scope means of the 1 km annual fields on the analysis grid — exactly the cells
  of `DISTRICT_IDX`, and the only way to apply the per-cell VIIRS calibration. They feed the β (dLST/dTair)
  regressions and the 1990/1995 night-level hindcast of section 09.

Every Earth Engine expression is built lazily and pulled with v1's tiled `fetch_grid` (section 01: retry/backoff,
quartering, on-disk tile cache keyed by the serialised expression) or, with `LST_GEE_EXPORT=asset`, batch-exported
to an asset first (`Export.image.toAsset` on the exact grid transform) for the multi-year stacks. Small reductions
use one batched `getInfo` per logical request with recursive halving on memory/time-out errors.

In [ ]:
_lst_enter_section('04')  # section marker (generated by build_notebook.py)

In [ ]:
import math as _s04_math
import time as _s04_time
from dataclasses import dataclass as _s04_dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as _s04_ndi

_S04_ACTIVE = globals().get("DATA_MODE") == "gee"
_S04_VERSION = "s04-v2.1"   # v2.1: class calibration needs >= 20 cells; 64 km heavy tiles for GLC fractions

# Earth Engine assets (verified on the catalog pages, docs/research/gee_datasets_v2.md §1B/§1C)
_S04_TERRA = "MODIS/061/MOD11A2"           # 8-day, 2000-02-18 -> ; LST_*_1km x0.02 K, *_view_time x0.1 h
_S04_AQUA = "MODIS/061/MYD11A2"            # 8-day, 2002-07-04 -> (first full Apr-Jun season: 2003)
_S04_VIIRS_N = "NASA/VIIRS/002/VNP21A1N"   # daily S-NPP night, 2012-01-19 -> ; LST_1KM (K), QC, View_Time (h)
_S04_ERA5_HOURLY = "ECMWF/ERA5_LAND/HOURLY"
_S04_ERA5_DAILY = "ECMWF/ERA5_LAND/DAILY_AGGR"
_S04_ERA5_MONTHLY = "ECMWF/ERA5_LAND/MONTHLY_AGGR"
_S04_GLC_ANNUAL = "projects/sat-io/open-datasets/GLC-FCS30D/annual"           # b1 = 2000 ... b23 = 2022
_S04_GLC_FIVE = "projects/sat-io/open-datasets/GLC-FCS30D/five-years-map"     # b1 = 1985, b2 = 1990, b3 = 1995
_S04_GLC_FIVE_BANDS = {1985: "b1", 1990: "b2", 1995: "b3"}
_S04_GLC_LAST_YEAR = 2022

_S04_MODIS_SCALE_M = 926.625433055833      # MODIS/VIIRS 1 km sinusoidal nominal pixel
_S04_ERA5_SCALE_M = 11132.0                # ERA5-Land 0.1 deg
_S04_GLC_SCALE_M = 30.0

_S04_TERRA_FIRST, _S04_AQUA_FIRST, _S04_VIIRS_FIRST = 2000, 2003, 2012
_S04_DRIFT_FROM = 2022                     # Terra (earlier) and Aqua (later) orbits drifted from 2022 (SPEC §2.2)
_S04_DAY_VIEW_WINDOW_H = (10.0, 11.5)      # MOD11A2 Day_view_time, local solar time (SPEC §2.1)
_S04_CAL_YEARS = {"terra_to_aqua": (2003, 2019), "viirs_to_aqua": (2013, 2021)}
_S04_CAL_MIN_POINTS = 30                   # class maps with fewer (cell, year) pairs fall back to the pooled map
_S04_CAL_MIN_CELLS = 20                    # ... or with fewer distinct cells (a few cells x many years is no fit)
# Asset-mode limits of the 04/05 batch exports (D6): per running task, and while queued
_S04_EXPORT_TIMEOUT_S = int(_env_int("LST_GEE_EXPORT_TIMEOUT_0405_S", 3600))
_S04_EXPORT_QUEUE_S = 7200
_S04_EXPORT_DIAG: list = []                # per batch: {jobs, max_running, states, seconds}
_S04_CAL_SHRINK_K = 3.0                    # residual shrinkage weight n / (n + k)
# Valid composites / days for an annual Apr-Jun median. Night MODIS accepts one composite (itself a multi-night mean):
# real Aqua data (2026-11) give >= 3 on only 13-50 % of the NCR cells per year, which failed the 50 % scope gate in
# 18 of 23 years, while >= 1 covers 75-94 %. Too few years left the level regression and beta on 8 points.
_S04_ANNUAL_MIN_N = {"modis": 3, "modis_night": 1, "viirs": 10}
_S04_SCOPE_MIN_COVERAGE = 0.5              # a scope-year satellite mean needs >= 50 % of the scope's cells valid
_S04_GLC_ANCHORS = (2000, 2005, 2010, 2015, 2020, 2022)   # land-cover years used for the "dominant class"
_S04_NOMINAL_IST_H = {"aqua": 2.20, "terra": 22.45, "viirs": 2.10}   # docs §0 F4 (granule mid-times, May)
_S04_ERA5_HOURS_UTC = {"era5_skin_day": 5, "era5_skin_night": 20}   # 10:30 IST and 01:30 IST
_S04_ERA5_SERIES = ("era5_skin_day", "era5_skin_night", "era5_tmax", "era5_tmin", "era5_swvl1")
_S04_SAT_SERIES = ("modis_terra_day", "modis_aqua_night", "viirs_night_cal")
_S04_SERIES_YEARS = (1990, 2025)
_S04_GLC_KEYS = ["glc_built", "glc_forest", "glc_water", "glc_cropland", "glc_barren", "glc_otherveg"]
_S04_NIGHT_SENSOR = {
    "aqua": "MODIS Aqua MYD11A2 night (~02:12 IST)",
    "terra_cal": "MODIS Terra MOD11A2 night (~22:30 IST), calibrated to Aqua",
    "viirs_cal": "VIIRS S-NPP VNP21A1N night (~02:06 IST), calibrated to Aqua",
}

if _S04_ACTIVE:
    log(f"section 04 ({_S04_VERSION}): MODIS/VIIRS thermal products for frames {list(CFG.FRAMES_OBS)}")
else:
    log("section 04 skipped: data mode is not 'gee' (section 02 provided the synthetic thermal products)")

## 4.1 Earth Engine helpers shared by sections 04 and 05
* `_s04_season_filter` / `_s04_season_dates` — the Apr–Jun season of `CFG.SEASON` as a month filter (or a
  day-of-year filter if the season is not month-aligned) and as explicit per-year dates.
* `_s04_to_grid` — v1's aggregation recipe: area-weighted `reduceResolution(mean)` from the native projection,
  then `reproject` onto the exact grid transform (nearest when the source is not coarser).
* `_s04_getinfo` — one `getInfo` with the section-01 error classification (transient → backoff retry);
  `_s04_batched` evaluates a request over a list of items and halves the list on memory/time-out errors, so a
  batch degrades gracefully instead of failing or falling back to per-item loops.
* `_s04_fetch` / `_s04_fetch_jobs` — grid extraction through section 03's public `gee_fetch_grid` (section 01's
  `fetch_grid` if 03 is absent); with `CFG.GEE_EXPORT == "asset"` the listed heavy stacks are first exported with
  03's `gee_export_to_asset` (all tasks started, then `gee_wait_for_exports`; identical assets reused by their
  stored expression hash) and read back with `gee_fetch_asset`. Without section 03 a self-contained exporter
  is used (asset names carry the expression hash). Failed exports fall back to interactive extraction.

In [ ]:
def _s04_season_months() -> Optional[tuple]:
    """(first_month, last_month) if ``CFG.SEASON`` is month-aligned (e.g. 04-01 .. 06-30), else None."""
    (m0, d0), (m1, d1) = [tuple(int(v) for v in s.split("-")) for s in CFG.SEASON]
    last_day = (pd.Timestamp(2001, m1, 1) + pd.offsets.MonthEnd(0)).day
    return (m0, m1) if d0 == 1 and d1 == last_day else None


def _s04_season_filter():
    """Server-side filter selecting the pre-monsoon season of any year."""
    months = _s04_season_months()
    if months is not None:
        return ee.Filter.calendarRange(months[0], months[1], "month")
    d0 = pd.Timestamp(f"2001-{CFG.SEASON[0]}").dayofyear
    d1 = pd.Timestamp(f"2001-{CFG.SEASON[1]}").dayofyear
    return ee.Filter.dayOfYear(int(d0), int(d1))


def _s04_season_dates(year: int) -> tuple:
    """[start, end) ISO dates of the season of ``year`` (end exclusive)."""
    start = f"{int(year)}-{CFG.SEASON[0]}"
    end = (pd.Timestamp(f"{int(year)}-{CFG.SEASON[1]}") + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    return start, end


def _s04_season_days() -> int:
    """Number of days in the season (non-leap year), e.g. 91 for Apr-Jun."""
    return int((pd.Timestamp(f"2001-{CFG.SEASON[1]}") - pd.Timestamp(f"2001-{CFG.SEASON[0]}")).days + 1)


def _s04_years_dates(y0: int, y1: int) -> tuple:
    """[start, end) dates covering the calendar years y0..y1 (the season filter is applied separately)."""
    return f"{int(y0)}-01-01", f"{int(y1) + 1}-01-01"


def _s04_to_grid(image, native_scale_m: float, grid: GridSpec, reducer=None):
    """Aggregate ``image`` (with a valid default projection) onto ``grid`` (v1 recipe).

    Coarsening uses an area-weighted ``reduceResolution`` (mask-aware mean by default) with ``maxPixels`` sized to the
    scale ratio; refining or equal scales use a plain ``reproject`` (nearest neighbour).
    """
    if native_scale_m < grid.res * 0.999:
        ratio = (grid.res / native_scale_m) ** 2
        max_pixels = int(min(65535, max(1024, _s04_math.ceil(ratio * 2.5))))
        image = image.reduceResolution(reducer=reducer or ee.Reducer.mean(), bestEffort=False, maxPixels=max_pixels)
    return image.reproject(crs=grid.crs, crsTransform=grid.transform_list())


def _s04_native(collection_id: str, band: str):
    """Native projection of ``band`` in an image collection (from its first image)."""
    return ee.Image(ee.ImageCollection(collection_id).first()).select(band).projection()


def _s04_masked_dummy(band: str = "lst"):
    """A fully masked one-band image: merged into a collection it guarantees reducers return ``band`` even for an
    empty collection (median/count of an empty collection would otherwise have zero bands), without changing any
    statistic (masked pixels are ignored by every reducer)."""
    return ee.Image.constant(0).toFloat().rename(band).updateMask(ee.Image.constant(0))


def _s04_getinfo(obj, label: str, allow_split: bool = False):
    """``obj.getInfo()`` with backoff on transient errors; memory/time-out errors become ``_GEESplitError`` when
    ``allow_split`` (the caller then halves its batch)."""
    def _call():
        try:
            return obj.getInfo()
        except Exception as exc:  # noqa: BLE001 - classified below
            raise _classify_gee_error(exc, allow_split) from exc

    def _on_retry(attempt, exc, delay):
        log(f"{label}: getInfo retry {attempt} in {delay:.1f}s: {str(exc)[:160]}", "WARNING")

    return retry(_call, tries=CFG.GEE_MAX_RETRIES, base_delay=2.0, max_delay=120.0,
                 retry_on=(_GEETransientError,), on_retry=_on_retry)


def _s04_batched(items: Sequence, build: Callable, parse: Callable, label: str, isolate: bool = False) -> list:
    """Evaluate ``build(items)`` with ONE getInfo; on memory/time-out errors split ``items`` in halves recursively.

    ``parse(info, items)`` turns the getInfo payload into a list of records. Returns the concatenated records.
    With ``isolate=True`` a *permanent* error (e.g. one GCM whose images lack a band) also splits the batch, so the
    failing item is isolated, logged and skipped while every other item is still evaluated in batches; a single item
    that still fails with a transient error after its retries is skipped the same way (the caller's completeness
    filter then drops it). "Too many concurrent aggregations" that persists after the retries halves the batch like
    a memory/time-out error.
    """
    items = list(items)
    if not items:
        return []
    try:
        info = _s04_getinfo(build(items), label, allow_split=len(items) > 1)
    except _GEESplitError as exc:
        half = len(items) // 2
        log(f"{label}: request too heavy for {len(items)} items ({str(exc)[:120]}); splitting {half}+"
            f"{len(items) - half}", "WARNING")
        return (_s04_batched(items[:half], build, parse, f"{label}[a]", isolate)
                + _s04_batched(items[half:], build, parse, f"{label}[b]", isolate))
    except _GEETransientError as exc:
        if len(items) > 1 and re.search(r"too many concurrent aggregations", str(exc), re.IGNORECASE):
            half = len(items) // 2
            log(f"{label}: 'too many concurrent aggregations' for {len(items)} items after retries; splitting "
                f"{half}+{len(items) - half}", "WARNING")
            return (_s04_batched(items[:half], build, parse, f"{label}[a]", isolate)
                    + _s04_batched(items[half:], build, parse, f"{label}[b]", isolate))
        if isolate and len(items) == 1:
            log(f"{label}: item {items[0]} still fails after its retries and is skipped: {str(exc)[:200]}", "WARNING")
            return []
        if not isolate:
            raise
        half = len(items) // 2
        return (_s04_batched(items[:half], build, parse, f"{label}[a]", isolate)
                + _s04_batched(items[half:], build, parse, f"{label}[b]", isolate))
    except Exception as exc:  # noqa: BLE001 - permanent server error
        if not isolate:
            raise
        if len(items) == 1:
            log(f"{label}: item {items[0]} failed permanently and is skipped: {type(exc).__name__}: "
                f"{str(exc)[:200]}", "WARNING")
            return []
        half = len(items) // 2
        return (_s04_batched(items[:half], build, parse, f"{label}[a]", isolate)
                + _s04_batched(items[half:], build, parse, f"{label}[b]", isolate))
    return parse(info, items)


def _s04_fc_records(info: dict) -> list:
    """Property dicts of a FeatureCollection getInfo payload."""
    if not isinstance(info, dict) or "features" not in info:
        raise RuntimeError(f"unexpected FeatureCollection payload: {str(info)[:200]}")
    return [dict(f.get("properties") or {}) for f in info["features"]]


def _s04_asset_name(name: str, image, grid: GridSpec) -> str:
    """Asset id under ``CFG.GEE_ASSET_ROOT`` that encodes the request and the expression hash."""
    digest = _gee_expression_digest(image)[:12]
    safe = re.sub(r"[^A-Za-z0-9_-]", "_", name)[:60]
    return f"{CFG.GEE_ASSET_ROOT.rstrip('/')}/{safe}_{int(grid.res)}m_{digest}"


def _s04_asset_exists(asset_id: str) -> bool:
    try:
        ee.data.getAsset(asset_id)
        return True
    except Exception:  # noqa: BLE001 - "not found" and permission errors look alike: treat as absent
        return False


def _s04_cancel_task(task, name: str) -> None:
    """Cancel a still-active batch task (guarded; a failed cancel only costs EECU)."""
    try:
        if hasattr(task, "cancel"):
            task.cancel()
        elif getattr(task, "id", None) and hasattr(ee.data, "cancelTask"):
            ee.data.cancelTask(task.id)
        else:
            return
        log(f"  asset export of {name} cancelled; extracted with computePixels instead", "WARNING")
    except Exception as exc:  # noqa: BLE001
        log(f"  could not cancel the asset export of {name}: {str(exc)[:160]}", "WARNING")


def _s04_export_assets(jobs: list, timeout_s: Optional[float] = None, queue_s: Optional[float] = None,
                       poll_s: float = 30.0) -> dict:
    """Batch-export every job to an asset (submitted together, polled together; used only without section 03).

    Per-task limits: ``timeout_s`` (LST_GEE_EXPORT_TIMEOUT_0405_S, 1 h) after the first poll that saw it RUNNING and
    ``queue_s`` (2 h) while not yet RUNNING. A task that fails or times out is cancelled when still active and left
    out, so its job is extracted interactively exactly once. Returns {job name: asset id} of the ready assets.
    """
    timeout_s = float(_S04_EXPORT_TIMEOUT_S if timeout_s is None else timeout_s)
    queue_s = float(_S04_EXPORT_QUEUE_S if queue_s is None else queue_s)
    ready, pending = {}, {}
    for job in jobs:
        asset_id = _s04_asset_name(job["name"], job["image"], job["grid"])
        if _s04_asset_exists(asset_id):
            ready[job["name"]] = asset_id
            continue
        grid = job["grid"]
        policy = {".default": "mode"} if np.dtype(job["dtype"]).kind in "iu" else {".default": "mean"}
        try:
            task = ee.batch.Export.image.toAsset(
                image=job["image"], description=re.sub(r"[^A-Za-z0-9_-]", "_", f"ncr_{job['name']}")[:100],
                assetId=asset_id, crs=grid.crs, crsTransform=grid.transform_list(),
                dimensions=f"{int(grid.width)}x{int(grid.height)}", maxPixels=int(1e13), pyramidingPolicy=policy)
            task.start()
            pending[job["name"]] = {"task": task, "asset_id": asset_id, "running_since": None}
            log(f"  asset export started: {asset_id}")
        except Exception as exc:  # noqa: BLE001 - fall back to interactive extraction for this job
            log(f"asset export of {job['name']} could not start ({str(exc)[:160]}); using computePixels", "WARNING")
    t0 = _s04_time.time()
    max_running = 0
    while pending:
        _f = globals().get("check_budget")
        if _f is not None:
            _f(f"waiting for {len(pending)} section-04/05 asset export(s)")
        now = _s04_time.time()
        n_running = 0
        for name, rec in list(pending.items()):
            task = rec["task"]
            try:
                status = task.status() or {}
            except Exception as exc:  # noqa: BLE001 - transient status failure: poll again
                log(f"task status of {name} unavailable: {str(exc)[:120]}", "WARNING")
                continue
            state = status.get("state", "UNKNOWN")
            if state == "COMPLETED":
                ready[name] = rec["asset_id"]
                pending.pop(name)
                log(f"  asset export finished: {rec['asset_id']}")
                continue
            if state in ("FAILED", "CANCELLED", "CANCEL_REQUESTED"):
                log(f"asset export of {name} {state}: {str(status.get('error_message', state))[:200]}; using "
                    "computePixels", "WARNING")
                pending.pop(name)
                continue
            if state == "RUNNING":
                n_running += 1
                rec["running_since"] = rec["running_since"] or now
            too_long = rec["running_since"] is not None and now - rec["running_since"] > timeout_s
            too_queued = rec["running_since"] is None and now - t0 > queue_s
            if too_long or too_queued:
                log(f"asset export of {name} {'running' if too_long else 'queued'} beyond its limit "
                    f"({timeout_s if too_long else queue_s:.0f} s)", "WARNING")
                _s04_cancel_task(task, name)
                pending.pop(name)
        max_running = max(max_running, n_running)
        if not pending:
            break
        log(f"  section-04/05 exports: {n_running} RUNNING, {len(pending)} not finished")
        _s04_time.sleep(poll_s)
    _S04_EXPORT_DIAG.append({"jobs": [j["name"] for j in jobs], "max_running": int(max_running),
                             "ready": sorted(ready), "seconds": round(_s04_time.time() - t0, 1)})
    return ready


def _s04_have_s03_helpers() -> bool:
    """True when section 03's public fetch/export helpers are defined (the normal gee-mode notebook)."""
    return all(callable(globals().get(n)) for n in ("gee_fetch_grid", "gee_export_to_asset", "gee_wait_for_exports",
                                                    "gee_fetch_asset"))


def _s04_fetch(image, grid: GridSpec, bands: Sequence[str], name: str, dtype=np.float32,
               tile_px: Optional[int] = None, heavy: bool = False) -> np.ndarray:
    """Interactive tiled extraction of ``bands`` on ``grid`` (cached by expression hash) -> (B, H, W).

    Uses section 03's ``gee_fetch_grid`` when available (same tiling/caching/unmask conventions as the Landsat
    stacks), else section 01's ``fetch_grid`` directly. ``heavy=True`` marks a *computed* image built from 30 m
    inputs (GLC fractions, SRTM/GSW, GHSL): without an explicit ``tile_px`` it then uses section 03's ~64 km tiles
    (64 px at 1 km, 256 px at 250 m), so each request stays within interactive compute limits; light images (MODIS,
    VIIRS, ERA5, asset reads) use large tiles.
    """
    bands = list(bands)
    if tile_px:
        tile = int(tile_px)
    elif heavy and callable(globals().get("_s03_default_tile_px")):
        tile = int(_s03_default_tile_px(grid, len(bands), dtype, heavy=True))
    else:
        tile = int(gee_tile_px(len(bands), dtype, cap=1024))
    if _s04_have_s03_helpers():
        return gee_fetch_grid(image, grid, bands, name, tile_px=tile, dtype=dtype, heavy=heavy)
    return fetch_grid(image, grid, bands, tile, gee_cache_key(name, grid, bands, image=image), dtype)


def _s04_fetch_jobs_s03(jobs: list) -> dict:
    """Asset mode through section 03's helpers: start every export (no wait), wait for all, read the assets.

    Assets get stable ids ``<GEE_ASSET_ROOT>/<job name>``; section 03 stores the expression hash on each asset, reuses
    an identical one, re-exports a changed one and attaches to a still-running task of an interrupted session. The
    wait uses per-task limits (LST_GEE_EXPORT_TIMEOUT_0405_S = 1 h running, 2 h queued) and never aborts on the first
    failure: COMPLETED jobs are read from their asset; a job that FAILED, was CANCELLED or timed out has its task
    cancelled (when still active) and is extracted interactively, once.
    """
    handles, out = {}, {}
    for job in jobs:
        asset_id = f"{CFG.GEE_ASSET_ROOT.rstrip('/')}/{re.sub(r'[^A-Za-z0-9_-]', '_', job['name'])[:90]}"
        pyramiding = "mode" if np.dtype(job["dtype"]).kind in "iu" else "mean"
        try:
            handles[job["name"]] = gee_export_to_asset(job["image"], asset_id, job["grid"],
                                                       description=f"ncr_{job['name']}", pyramiding=pyramiding,
                                                       wait=False)
        except Exception as exc:  # noqa: BLE001 - interactive extraction still works
            log(f"asset export of {job['name']} could not start ({type(exc).__name__}: {str(exc)[:160]}); "
                "using computePixels", "WARNING")
    stats: dict = {}
    if handles:
        gee_wait_for_exports(list(handles.values()), timeout_s=_S04_EXPORT_TIMEOUT_S, queue_s=_S04_EXPORT_QUEUE_S,
                             raise_on_failure=False, stats=stats)
        _cancel = globals().get("gee_cancel_export")
        for h in handles.values():
            if h.get("state") != "COMPLETED" and callable(_cancel):
                _cancel(h)
    _S04_EXPORT_DIAG.append({"jobs": [j["name"] for j in jobs], "max_running": int(stats.get("max_running", 0)),
                             "states": {n: h.get("state") for n, h in handles.items()},
                             "seconds": stats.get("seconds")})
    for job in jobs:
        h = handles.get(job["name"])
        with timer(f"04 fetch {job['name']}"):
            if h is not None and h.get("state") == "COMPLETED":
                out[job["name"]] = gee_fetch_asset(h["asset_id"], job["grid"], list(job["bands"]), job["name"],
                                                   dtype=job["dtype"])
            else:
                out[job["name"]] = _s04_fetch(job["image"], job["grid"], job["bands"], job["name"], job["dtype"],
                                              job.get("tile_px"), heavy=bool(job.get("heavy", False)))
    return out


def _s04_fetch_jobs(jobs: list, use_assets: bool = True) -> dict:
    """Fetch several grid jobs ``{name, image, grid, bands, dtype}``; with ``CFG.GEE_EXPORT == "asset"`` (and
    ``use_assets``) they are batch-exported first and read back from their assets. Returns {name: (B, H, W)}."""
    if use_assets and CFG.GEE_EXPORT == "asset" and jobs and _s04_have_s03_helpers():
        return _s04_fetch_jobs_s03(jobs)
    assets = _s04_export_assets(jobs) if (use_assets and CFG.GEE_EXPORT == "asset" and jobs) else {}
    out = {}
    for job in jobs:
        image = ee.Image(assets[job["name"]]).select(list(job["bands"])) if job["name"] in assets else job["image"]
        with timer(f"04 fetch {job['name']}"):
            out[job["name"]] = _s04_fetch(image, job["grid"], job["bands"], job["name"], job["dtype"],
                                          job.get("tile_px"), heavy=bool(job.get("heavy", False))
                                          and job["name"] not in assets)
    return out


def _s04_cells(field2d: np.ndarray) -> np.ndarray:
    """(H, W) grid -> canonical ROI cell vector (float64)."""
    return np.asarray(field2d, dtype=np.float64)[CELL_ROWS, CELL_COLS]


def _s04_grid_from_cells(values: np.ndarray, fill=np.nan, dtype=np.float32) -> np.ndarray:
    """Canonical ROI cell vector -> (H, W) grid (``fill`` outside the ROI)."""
    out = np.full(GRID.shape, fill, dtype=dtype)
    out[CELL_ROWS, CELL_COLS] = np.asarray(values).astype(dtype, copy=False)
    return out


def _s04_fill_roi_nearest(arr: np.ndarray, roi: np.ndarray, label: str) -> np.ndarray:
    """Fill non-finite ROI pixels with the nearest finite value (Euclidean, pixel units); NaN outside the ROI.

    Used for static/driver fields that must be complete inside the ROI (contract §3.2). Logs the share filled;
    raises if the field has no finite value at all.
    """
    out = np.array(arr, dtype=np.float32, copy=True)
    finite = np.isfinite(out)
    holes = roi & ~finite
    if holes.any():
        if not finite.any():
            raise RuntimeError(f"{label}: no valid pixel to fill from")
        _, (ir, ic) = _s04_ndi.distance_transform_edt(~finite, return_indices=True)
        out[holes] = out[ir[holes], ic[holes]]
        share = holes.sum() / max(1, int(roi.sum()))
        log(f"  {label}: {int(holes.sum()):,} ROI pixels ({100 * share:.2f} %) gap-filled by nearest neighbour",
            "WARNING" if share > 0.02 else "INFO")
    out[~roi] = np.nan
    return out


def _s04_roi_digest() -> str:
    """Short hash of the ROI/district rasters (part of every stage-cache key of 04/05)."""
    h = hashlib.sha1()
    h.update(np.ascontiguousarray(ROI_MASK).tobytes())
    h.update(np.ascontiguousarray(DISTRICT_IDX).tobytes())
    return h.hexdigest()[:16]

## 4.2 Calibration and scope-mean mathematics (pure NumPy, unit-tested)
**Calibration model.** For cell *i*, year *t* with dominant class *c(i,t)*:
`aqua(i,t) = a_c + b_c · src(i,t) + r_i + ε`. The class map (a_c, b_c) is ordinary least squares over all valid
(cell, year) pairs of the class (pooled fit when a class has < 30 pairs, fewer than 20 distinct cells or no spread).
The persistent residual `r_i` is the cell's mean residual after the class map, shrunk with weight `n_i/(n_i + 3)`,
where n_i is the number of overlap years of the cell — the posterior mean of a random cell effect when the residual
variance is roughly 3× the between-cell variance, so short records are pulled towards the class map.
`NIGHT_CALIBRATION` reports R²/RMSE of the class maps (as SPEC requires); the residual term's honest gain is measured
by **leave-one-year-out** (fit on all other years, predict the held-out year) and logged/recorded in
`NIGHT_CALIBRATION_DIAG`.

In [ ]:
def _s04_ols(x: np.ndarray, y: np.ndarray) -> tuple:
    """Least squares y = a + b x -> (a, b); a degenerate x (no spread) gives slope 1 and the mean offset."""
    x, y = np.asarray(x, np.float64), np.asarray(y, np.float64)
    xm, ym = float(x.mean()), float(y.mean())
    var = float(((x - xm) ** 2).sum())
    if var <= 1e-12 * max(1.0, x.size):
        return ym - xm, 1.0
    b = float(((x - xm) * (y - ym)).sum() / var)
    return ym - b * xm, b


def _s04_fit_calibration(src: np.ndarray, aqua: np.ndarray, cls: np.ndarray, *,
                         min_points: int = _S04_CAL_MIN_POINTS, shrink_k: float = _S04_CAL_SHRINK_K,
                         min_cells: int = _S04_CAL_MIN_CELLS) -> tuple:
    """Fit per-class linear maps ``aqua = a + b·src`` and shrunk per-cell residuals.

    ``src``, ``aqua``: (years, cells) float arrays (NaN = missing); ``cls``: (years, cells) dominant class 1..6 (0 =
    unknown, handled by the pooled map). Returns ``(summary, coef, shrunk, info)``:

    * ``summary`` = ``{"r2", "rmse", "n", "by_class": [{"class", "a", "b", "n"}]}`` (SPEC §3.2; class-map fit);
    * ``coef`` = {class: (a, b)} including key 0 = pooled map (used for unknown classes);
    * ``shrunk`` = (cells,) float32 shrunk mean residual (0 where a cell has no overlap year);
    * ``info`` = {"pooled_classes": [...], "cells_per_class": {class: int}, "n_years_per_cell": (cells,) int,
      "r2_with_residual": float}.

    A class gets its own map only with >= ``min_points`` valid (cell, year) pairs AND >= ``min_cells`` distinct cells
    (a handful of cells observed in many years would otherwise give an unstable slope) and a non-degenerate spread.
    """
    src = np.asarray(src, np.float64)
    aqua = np.asarray(aqua, np.float64)
    cls = np.asarray(cls)
    if not (src.shape == aqua.shape == cls.shape) or src.ndim != 2:
        raise ValueError(f"calibration inputs must share a (years, cells) shape: {src.shape} {aqua.shape} {cls.shape}")
    ok = np.isfinite(src) & np.isfinite(aqua)
    n_ok = int(ok.sum())
    if n_ok < max(10, min_points):
        raise RuntimeError(f"night calibration has only {n_ok} valid (cell, year) pairs")
    pooled = _s04_ols(src[ok], aqua[ok])
    coef, by_class, pooled_classes, cells_per_class = {0: pooled}, [], [], {}
    pred = np.full(src.shape, np.nan)
    unknown = ok & ((cls < 1) | (cls > 6))
    pred[unknown] = pooled[0] + pooled[1] * src[unknown]
    for c in range(1, 7):
        sel = ok & (cls == c)
        n_c = int(sel.sum())
        cells_c = int(sel.any(axis=0).sum())
        cells_per_class[int(c)] = cells_c
        spread = float(np.ptp(src[sel])) if n_c else 0.0
        if n_c >= min_points and cells_c >= int(min_cells) and spread > 1e-6:
            a, b = _s04_ols(src[sel], aqua[sel])
        else:
            a, b = pooled
            pooled_classes.append(c)
        coef[c] = (float(a), float(b))
        by_class.append({"class": int(c), "a": round(float(a), 4), "b": round(float(b), 4), "n": n_c})
        pred[sel] = a + b * src[sel]
    resid = np.where(ok, aqua - pred, np.nan)
    sse = float(np.nansum(resid ** 2))
    sst = float(((aqua[ok] - aqua[ok].mean()) ** 2).sum())
    n_years = ok.sum(axis=0)
    mean_resid = np.where(n_years > 0, np.nansum(resid, axis=0) / np.maximum(n_years, 1), 0.0)
    shrunk = (mean_resid * n_years / (n_years + float(shrink_k))).astype(np.float32)
    resid2 = np.where(ok, resid - shrunk[None, :], np.nan)
    summary = {"r2": round(1.0 - sse / sst, 4) if sst > 0 else None,
               "rmse": round(float(np.sqrt(sse / n_ok)), 4), "n": n_ok, "by_class": by_class}
    info = {"pooled_classes": pooled_classes, "cells_per_class": cells_per_class,
            "n_years_per_cell": n_years.astype(np.int32),
            "r2_with_residual": round(1.0 - float(np.nansum(resid2 ** 2)) / sst, 4) if sst > 0 else None}
    return summary, coef, shrunk, info


def _s04_apply_calibration(src: np.ndarray, cls: np.ndarray, coef: dict, shrunk: Optional[np.ndarray]) -> np.ndarray:
    """Aqua-equivalent values ``a_c + b_c·src (+ shrunk residual)``; classes without a map use the pooled map."""
    src = np.asarray(src, np.float64)
    cls = np.asarray(cls)
    a0, b0 = coef[0]
    a = np.full(src.shape, a0, np.float64)
    b = np.full(src.shape, b0, np.float64)
    for c, (ac, bc) in coef.items():
        if c == 0:
            continue
        a = np.where(cls == c, ac, a)
        b = np.where(cls == c, bc, b)
    out = a + b * src
    if shrunk is not None:
        out = out + np.asarray(shrunk, np.float64)
    return out.astype(np.float32)


def _s04_loyo_rmse(src: np.ndarray, aqua: np.ndarray, cls: np.ndarray) -> dict:
    """Leave-one-year-out transfer error of the calibration (class map only, and class map + shrunk residual)."""
    years = src.shape[0]
    err_cls, err_full = [], []
    for t in range(years):
        keep = np.arange(years) != t
        ok_t = np.isfinite(src[t]) & np.isfinite(aqua[t])
        if ok_t.sum() < 10 or keep.sum() < 2:
            continue
        try:
            _, coef, shrunk, _ = _s04_fit_calibration(src[keep], aqua[keep], cls[keep])
        except RuntimeError:
            continue
        p_cls = _s04_apply_calibration(src[t], cls[t], coef, None)
        p_full = _s04_apply_calibration(src[t], cls[t], coef, shrunk)
        err_cls.append((p_cls - aqua[t])[ok_t])
        err_full.append((p_full - aqua[t])[ok_t])
    if not err_cls:
        return {"loyo_rmse_class_map": None, "loyo_rmse_with_residual": None, "loyo_years": 0}
    e1, e2 = np.concatenate(err_cls), np.concatenate(err_full)
    return {"loyo_rmse_class_map": round(float(np.sqrt(np.mean(e1 ** 2))), 4),
            "loyo_rmse_with_residual": round(float(np.sqrt(np.mean(e2 ** 2))), 4),
            "loyo_bias_with_residual": round(float(np.mean(e2)), 4), "loyo_years": len(err_cls)}


def _s04_scope_means(values: np.ndarray, district: np.ndarray, n_districts: int,
                     min_coverage: float = _S04_SCOPE_MIN_COVERAGE) -> dict:
    """{"ncr": mean, "d:<id>": mean} of a canonical cell vector.

    A scope is reported only if at least ``min_coverage`` of its cells are finite (a heavily clouded year would
    otherwise give a biased mean of the remaining cells). Districts without cells are omitted.
    """
    values = np.asarray(values, np.float64)
    district = np.asarray(district, np.int64)
    ok = np.isfinite(values)
    out = {}
    if ok.mean() >= min_coverage:
        out["ncr"] = float(values[ok].mean())
    sums = np.bincount(district[ok], weights=values[ok], minlength=n_districts)
    counts = np.bincount(district[ok], minlength=n_districts)
    totals = np.bincount(district, minlength=n_districts)
    for d in range(n_districts):
        if totals[d] > 0 and counts[d] >= min_coverage * totals[d] and counts[d] > 0:
            out[f"d:{d}"] = float(sums[d] / counts[d])
    return out


def _s04_nearest_anchor(year: int, anchors: Sequence[int]) -> int:
    """Nearest land-cover anchor year (ties -> the later anchor); years after the last map use the last map."""
    year = min(int(year), max(anchors))
    return min(anchors, key=lambda a: (abs(a - year), -a))


def _s04_view_time_to_ist(t_h: Optional[float], expected: str, lon_c: float) -> tuple:
    """Interpret a mean view time (h) and convert it to IST hours.

    MODIS ``*_view_time`` is local solar time; for VIIRS the catalog only says "h", so both interpretations (local
    solar time, UTC) are tried and the one inside the plausible window of the pass is kept:
    night passes after midnight (Aqua/VIIRS) 0-6 h IST, Terra's evening pass 19-24 h IST. Returns
    ``(ist_hours or None, how)``.
    """
    if t_h is None or not np.isfinite(t_h):
        return None, "no view time"
    windows = {"after_midnight": (0.0, 6.0), "evening": (19.0, 24.0)}
    lo, hi = windows[expected]
    candidates = [("local solar time", (t_h + 5.5 - lon_c / 15.0) % 24.0), ("UTC", (t_h + 5.5) % 24.0)]
    for how, ist in candidates:
        if lo <= ist <= hi:
            return round(float(ist), 3), how
    return None, f"implausible view time {t_h:.2f} h"

## 4.3 Server-side product builders (MODIS 8-day, VIIRS daily, ERA5-Land, GLC_FCS30D)
Each builder returns a lazy `ee.Image`/`ee.ImageCollection`; nothing is computed until a fetch or a batched
`getInfo`. MODIS/VIIRS images are masked with QC (and the view-time window for the Terra day product), composited
per pixel (median, plus the valid count) in their native sinusoidal projection and aggregated to the 1 km grid with
the mask-aware mean. The GLC_FCS30D class fractions use the **two-step** `reduceResolution` of SPEC §2.3:
30 m class indicators → fine grid (250 m) → analysis grid (1 km); each step stays far below the `maxPixels` limit
and keeps the area weighting exact on the nested grids.

In [ ]:
def _s04_modis_col(collection_id: str, which: str, y0: int, y1: int, view_window: Optional[tuple] = None):
    """QC-masked MOD11A2/MYD11A2 8-day LST (°C, band ``lst``) and view time (h local solar, band ``vtime``)."""
    lst_band, qc_band, vt_band = f"LST_{which}_1km", f"QC_{which}", f"{which}_view_time"

    def _prep(img):
        qc = img.select(qc_band)
        good = qc.bitwiseAnd(3).lte(1).And(qc.rightShift(6).bitwiseAnd(3).lte(1))   # mandatory QA <= 1, err <= 2 K
        vtime = img.select(vt_band).multiply(0.1)
        if view_window is not None:
            good = good.And(vtime.gte(float(view_window[0]))).And(vtime.lte(float(view_window[1])))
        lst = img.select(lst_band).multiply(0.02).subtract(273.15)
        # Plain float32 bands: the scaled integer bands carry a bounded type (Float<-273.15, 1037.55>), and Earth
        # Engine refuses to reduce a collection mixing them with the plain-float masked dummy of _s04_composite.
        return (lst.rename("lst").addBands(vtime.rename("vtime")).toFloat().updateMask(good)
                .set("system:time_start", img.get("system:time_start")))

    start, end = _s04_years_dates(y0, y1)
    return ee.ImageCollection(collection_id).filterDate(start, end).filter(_s04_season_filter()).map(_prep)


def _s04_viirs_col(y0: int, y1: int, lst_scale: float, vt_scale: float):
    """QC-masked VNP21A1N daily night LST (°C, ``lst``) and view time (h, ``vtime``) with run-time scales."""
    def _prep(img):
        qc = img.select("QC")
        good = qc.bitwiseAnd(3).lte(1).And(qc.rightShift(14).bitwiseAnd(3).gte(2))   # LST accuracy <= 1.5 K
        lst = img.select("LST_1KM").multiply(float(lst_scale)).subtract(273.15)
        vtime = img.select("View_Time").multiply(float(vt_scale))
        return (lst.rename("lst").addBands(vtime.rename("vtime")).toFloat().updateMask(good)
                .set("system:time_start", img.get("system:time_start")))

    start, end = _s04_years_dates(y0, y1)
    return ee.ImageCollection(_S04_VIIRS_N).filterDate(start, end).filter(_s04_season_filter()).map(_prep)


def _s04_composite(col, native, grid: GridSpec, min_n: int = 1) -> tuple:
    """Per-pixel median (masked where fewer than ``min_n`` valid values) and valid count, both on ``grid``."""
    # Every image float32 (as the dummy): a collection mixing band types cannot be reduced.
    lst = (col.select("lst").map(lambda im: ee.Image(im).toFloat())
           .merge(ee.ImageCollection([_s04_masked_dummy("lst")])))
    count = lst.count()
    median = lst.median().updateMask(count.gte(int(min_n))).setDefaultProjection(native)
    count = count.unmask(0).toFloat().setDefaultProjection(native)
    return (_s04_to_grid(median, _S04_MODIS_SCALE_M, grid), _s04_to_grid(count, _S04_MODIS_SCALE_M, grid))


def _s04_viirs_scales() -> dict:
    """Run-time check of the VNP21A1N ``LST_1KM`` and ``View_Time`` scales (SPEC §2.2; catalog leaves them open).

    One batched request: the ROI median of the Apr-Jun 2020 per-pixel median of the raw bands. LST values > 1000 are
    digital numbers (× 0.02 K); View_Time > 24 is in 0.1 h. The scaled LST must be a plausible night temperature.
    """
    start, end = _s04_season_dates(2020)
    raw = (ee.ImageCollection(_S04_VIIRS_N).filterDate(start, end).select(["LST_1KM", "View_Time"]).median())
    probe = raw.reduceRegion(reducer=ee.Reducer.median(), geometry=grid_ee_geom(GRID), scale=5000,
                             bestEffort=True, maxPixels=int(1e9))
    info = _s04_getinfo(probe, "VIIRS scale probe") or {}
    lst_raw, vt_raw = info.get("LST_1KM"), info.get("View_Time")
    if lst_raw is None:
        raise RuntimeError("VIIRS VNP21A1N scale probe returned no LST over the ROI (Apr-Jun 2020)")
    lst_scale = 0.02 if float(lst_raw) > 1000 else 1.0
    kelvin = float(lst_raw) * lst_scale
    if not 250.0 <= kelvin <= 340.0:
        raise RuntimeError(f"VIIRS LST_1KM scale check failed: raw {lst_raw} x {lst_scale} = {kelvin:.1f} K")
    vt_scale = 0.1 if (vt_raw is not None and float(vt_raw) > 24.0) else 1.0
    log(f"VIIRS VNP21A1N run-time scales: LST_1KM raw median {float(lst_raw):.2f} -> x{lst_scale} "
        f"({kelvin - 273.15:.1f} °C); View_Time raw {vt_raw} -> x{vt_scale}")
    return {"lst_scale": lst_scale, "vt_scale": vt_scale, "lst_raw_median": float(lst_raw),
            "vt_raw_median": None if vt_raw is None else float(vt_raw)}


@_s04_dataclass(frozen=True)
class _S04NightPlan:
    """Night product chosen for one observation frame."""

    year: int
    source: Optional[str]       # "aqua" | "terra_cal" | "viirs_cal" | None
    y0: int
    y1: int


def _s04_night_plan(year: int) -> _S04NightPlan:
    """Choose the night product of a frame from its window (SPEC §1.2/§2.2).

    * windows reaching the drift years (≥ 2022) use VIIRS (calibrated); Aqua drifted to ~03:20 IST by 2025;
    * windows inside 2003–2021 use Aqua (years before 2003 dropped: first full Aqua season is 2003);
    * windows ending 2000–2002 use Terra night (calibrated) from 2000;
    * earlier windows have no night product (modelled by section 09).
    """
    y0, y1 = (int(v) for v in CFG.window(year))
    if y1 >= _S04_DRIFT_FROM:
        return _S04NightPlan(int(year), "viirs_cal", max(y0, _S04_VIIRS_FIRST), y1)
    if y1 >= _S04_AQUA_FIRST:
        return _S04NightPlan(int(year), "aqua", max(y0, _S04_AQUA_FIRST), y1)
    if y1 >= _S04_TERRA_FIRST:
        return _S04NightPlan(int(year), "terra_cal", max(y0, _S04_TERRA_FIRST), y1)
    return _S04NightPlan(int(year), None, y0, y1)


def _s04_night_col(plan: _S04NightPlan, viirs: dict):
    """Masked night collection of a plan (``None`` source -> None)."""
    if plan.source == "aqua":
        return _s04_modis_col(_S04_AQUA, "Night", plan.y0, plan.y1)
    if plan.source == "terra_cal":
        return _s04_modis_col(_S04_TERRA, "Night", plan.y0, plan.y1)
    if plan.source == "viirs_cal":
        return _s04_viirs_col(plan.y0, plan.y1, viirs["lst_scale"], viirs["vt_scale"])
    return None


def _s04_native_of(source: str):
    return {"aqua": lambda: _s04_native(_S04_AQUA, "LST_Night_1km"),
            "terra_cal": lambda: _s04_native(_S04_TERRA, "LST_Night_1km"),
            "terra": lambda: _s04_native(_S04_TERRA, "LST_Night_1km"),
            "terra_day": lambda: _s04_native(_S04_TERRA, "LST_Day_1km"),
            "viirs_cal": lambda: _s04_native(_S04_VIIRS_N, "LST_1KM"),
            "viirs": lambda: _s04_native(_S04_VIIRS_N, "LST_1KM")}[source]()


def _s04_annual_stack(kind: str, years: Sequence[int], viirs: dict):
    """One band per year (``y<year>``): Apr-Jun per-pixel median of a product at 1 km on GRID.

    ``kind``: "aqua_night" | "terra_night" | "viirs_night" | "terra_day" (QC + view-time window).
    """
    bands = []
    for y in years:
        if kind == "aqua_night":
            col = _s04_modis_col(_S04_AQUA, "Night", y, y)
            native, min_n = _s04_native_of("aqua"), _S04_ANNUAL_MIN_N["modis_night"]
        elif kind == "terra_night":
            col = _s04_modis_col(_S04_TERRA, "Night", y, y)
            native, min_n = _s04_native_of("terra"), _S04_ANNUAL_MIN_N["modis_night"]
        elif kind == "terra_day":
            col = _s04_modis_col(_S04_TERRA, "Day", y, y, _S04_DAY_VIEW_WINDOW_H)
            native, min_n = _s04_native_of("terra_day"), _S04_ANNUAL_MIN_N["modis"]
        elif kind == "viirs_night":
            col = _s04_viirs_col(y, y, viirs["lst_scale"], viirs["vt_scale"])
            native, min_n = _s04_native_of("viirs"), _S04_ANNUAL_MIN_N["viirs"]
        else:
            raise ValueError(f"unknown annual stack kind {kind!r}")
        median, _ = _s04_composite(col, native, GRID, min_n=min_n)
        bands.append(median.rename(f"y{int(y)}"))
    names = [f"y{int(y)}" for y in years]
    return ee.Image.cat(bands).unmask(NODATA_FLOAT).toFloat(), names


def _s04_era5_annual_image(variable: str, years: Sequence[int]):
    """ERA5-Land annual Apr-Jun means (one band ``y<year>`` per year), bilinear at the reduction scale.

    * ``era5_skin_day`` / ``era5_skin_night``: HOURLY ``skin_temperature`` at 05 / 20 UTC (10:30 / 01:30 IST), °C;
    * ``era5_tmax`` / ``era5_tmin``: DAILY_AGGR ``temperature_2m_max`` / ``_min``, mean of the daily values, °C;
    * ``era5_swvl1``: MONTHLY_AGGR ``volumetric_soil_water_layer_1`` (m³ m⁻³), mean of the Apr-Jun months.
    """
    if variable in _S04_ERA5_HOURS_UTC:
        coll_id, band, offset = _S04_ERA5_HOURLY, "skin_temperature", -273.15
        hour = _S04_ERA5_HOURS_UTC[variable]
    elif variable in ("era5_tmax", "era5_tmin"):
        coll_id, band, offset, hour = _S04_ERA5_DAILY, f"temperature_2m_{variable[-3:]}", -273.15, None
    elif variable == "era5_swvl1":
        coll_id, band, offset, hour = _S04_ERA5_MONTHLY, "volumetric_soil_water_layer_1", 0.0, None
    else:
        raise ValueError(f"unknown ERA5 annual variable {variable!r}")
    native = _s04_native(coll_id, band)
    bands = []
    for y in years:
        start, end = _s04_season_dates(y)
        col = ee.ImageCollection(coll_id).filterDate(start, end).select(band)
        if hour is not None:
            col = col.filter(ee.Filter.calendarRange(int(hour), int(hour), "hour"))
        img = col.mean().add(float(offset)).setDefaultProjection(native).resample("bilinear")
        bands.append(img.rename(f"y{int(y)}"))
    return ee.Image.cat(bands)


def _s04_scopes_fc():
    """NCR (did = -1) + one dissolved feature per district (``did``) as an ee.FeatureCollection.

    The NCR feature is the union of the ROI cells (``gee_roi_geometry``), not ``EE_ROI``: that one evaluates to an empty
    geometry on real data, so its reduction returned null and the annual series had no NCR row for any ERA5 variable.
    """
    ncr = ee.FeatureCollection([ee.Feature(gee_roi_geometry(0), {"did": -1})])
    return ee.FeatureCollection(EE_DISTRICTS).merge(ncr)


def _s04_glc_band_lists() -> dict:
    """Band names of the GLC_FCS30D collections (one batched request, memoised)."""
    memo = globals().get("_S04_GLC_BANDLISTS")
    if memo:
        return memo
    geom = grid_ee_geom(GRID)
    req = ee.Dictionary({
        "annual": ee.Image(ee.ImageCollection(_S04_GLC_ANNUAL).filterBounds(geom).first()).bandNames(),
        "five": ee.Image(ee.ImageCollection(_S04_GLC_FIVE).filterBounds(geom).first()).bandNames(),
    })
    info = _s04_getinfo(req, "GLC_FCS30D band lists") or {}
    out = {"annual": list(info.get("annual") or []), "five": list(info.get("five") or [])}
    if not out["annual"]:
        raise RuntimeError(f"GLC_FCS30D annual collection has no tiles over the ROI ({_S04_GLC_ANNUAL})")
    if len(out["annual"]) != 23:
        log(f"GLC_FCS30D annual tiles expose {len(out['annual'])} bands (expected 23 for 2000-2022)", "WARNING")
    globals()["_S04_GLC_BANDLISTS"] = out
    return out


def _s04_glc_band(year: int) -> tuple:
    """(collection id, band, map year) of GLC_FCS30D for a frame/anchor year (SPEC §2.3).

    1990 -> five-years-map b2, 1995 -> b3 (1985 b1); 2000..2022 -> annual b<year-1999>; later years -> the 2022 map
    (b23). Years before 2000 that are not 1985/1990/1995 use the nearest five-year map.
    """
    lists = _s04_glc_band_lists()
    year = int(year)
    if year < 2000:
        map_year = min(_S04_GLC_FIVE_BANDS, key=lambda y: (abs(y - year), -y))
        band = _S04_GLC_FIVE_BANDS[map_year]
        if band not in lists["five"]:
            raise RuntimeError(f"GLC_FCS30D five-years-map lacks band {band} (has {lists['five']})")
        return _S04_GLC_FIVE, band, map_year
    map_year = min(year, _S04_GLC_LAST_YEAR, 1999 + len(lists["annual"]))
    band = f"b{map_year - 1999}"
    if band not in lists["annual"]:
        band = lists["annual"][min(map_year - 2000, len(lists["annual"]) - 1)]
    return _S04_GLC_ANNUAL, band, map_year


def _s04_glc_classes(collection_id: str, band: str):
    """Harmonised classes (0 nodata, 1..6) of one GLC_FCS30D band, in the tiles' native 30 m projection."""
    geom = grid_ee_geom(GRID)
    col = ee.ImageCollection(collection_id).filterBounds(geom)
    native = ee.Image(col.first()).select(band).projection()
    table = LC_REMAPS["glc_fcs30d"]
    classes = (col.select(band).mosaic().remap(list(table.keys()), list(table.values()), 0)
               .rename("lc").setDefaultProjection(native))
    return classes, native


def _s04_glc_fraction_image(collection_id: str, band: str):
    """Six class fractions + valid share on GRID via the two-step reduceResolution (30 m -> fine -> 1 km)."""
    classes, native = _s04_glc_classes(collection_id, band)
    valid = classes.gt(0)
    indicators = (ee.Image.cat([classes.eq(c).rename(k) for c, k in zip(range(1, 7), _S04_GLC_KEYS)])
                  .toFloat().updateMask(valid).setDefaultProjection(native))
    valid_f = valid.unmask(0).toFloat().rename("glc_valid").setDefaultProjection(native)
    stack = ee.Image.cat([indicators, valid_f])
    step1 = _s04_to_grid(stack, _S04_GLC_SCALE_M, GRID_FINE)          # 30 m -> 250 m (fine grid)
    step2 = _s04_to_grid(step1, float(GRID_FINE.res), GRID)            # 250 m -> 1 km
    return step2.unmask(NODATA_FLOAT).toFloat()


def _s04_glc_normalise(raw: np.ndarray, label: str) -> dict:
    """(7, H, W) fetched fractions + valid share -> {glc_key: (H, W) float32}, summing to 1 in the ROI.

    Fractions are renormalised over the six classes (the mean over valid 30 m pixels already excludes nodata);
    ROI cells without any classified pixel take the fractions of the nearest classified cell.
    """
    fr = np.asarray(raw[:6], np.float64)
    total = np.nansum(np.where(np.isfinite(fr), np.clip(fr, 0.0, None), 0.0), axis=0)
    good = (total > 1e-6) & ROI_MASK
    fr = np.where(good[None], np.clip(np.nan_to_num(fr, nan=0.0), 0.0, None) / np.where(good, total, 1.0)[None],
                  np.nan)
    out = {}
    if (ROI_MASK & ~good).any():
        if not good.any():
            raise RuntimeError(f"{label}: no classified GLC_FCS30D pixel inside the ROI")
        _, (ir, ic) = _s04_ndi.distance_transform_edt(~good, return_indices=True)
        holes = ROI_MASK & ~good
        fr[:, holes] = fr[:, ir[holes], ic[holes]]
        log(f"  {label}: {int(holes.sum()):,} ROI cells without GLC classes filled from the nearest cell", "WARNING")
    for i, key in enumerate(_S04_GLC_KEYS):
        arr = fr[i].astype(np.float32)
        arr[~ROI_MASK] = np.nan
        out[key] = arr
    return out


def _s04_glc_fractions(year: int) -> dict:
    """Normalised GLC_FCS30D fractions on GRID for a frame/anchor year (memoised; tile cache shared with 05).

    Returns ``{"fractions": {glc_key: (H,W)}, "dominant": (H,W) uint8, "band": str, "collection": str,
    "map_year": int}``.
    """
    coll, band, map_year = _s04_glc_band(year)
    memo = globals().setdefault("_S04_GLC_MEMO", {})
    key = (coll, band, GRID.res, _s04_roi_digest())
    if key not in memo:
        image = _s04_glc_fraction_image(coll, band)
        names = _S04_GLC_KEYS + ["glc_valid"]
        with timer(f"04/05 GLC fractions {band} ({map_year})"):
            raw = _s04_fetch(image, GRID, names, f"glc_frac_{coll.rsplit('/', 1)[-1]}_{band}", heavy=True)
        fractions = _s04_glc_normalise(raw, f"GLC {map_year}")
        stack = np.stack([np.nan_to_num(fractions[k], nan=-1.0) for k in _S04_GLC_KEYS])
        dominant = (np.argmax(stack, axis=0) + 1).astype(np.uint8)
        dominant[~ROI_MASK] = 0
        memo[key] = {"fractions": fractions, "dominant": dominant, "band": band, "collection": coll,
                     "map_year": int(map_year)}
        del raw, stack
    return memo[key]

## 4.4 Night frames, MODIS day validation and the overlap calibrations
Order of work (all requests batched):
1. VIIRS scale probe (one request) → VIIRS collections with the right scale.
2. Frame composites: one 1 km fetch per frame with `night_raw`, `night_nobs`, `day_modis`; frame metadata
   (composite counts, ROI mean view time per frame) in **one** `getInfo`.
3. Annual Apr–Jun medians at 1 km (Aqua night 2003–2025, Terra night 2003–2019, VIIRS night 2012–2025, Terra day
   2000–2025 with the view-time filter) — four multi-band fetches (asset-exported in `asset` mode).
4. Dominant GLC class at the anchor years 2000/2005/2010/2015/2020/2022; each overlap year uses its nearest anchor.
5. Fit Terra→Aqua and VIIRS→Aqua, apply them to the 2000/2025 frame composites and the VIIRS annual fields.

In [ ]:
def _s04_frame_requests(plans: dict, viirs: Optional[dict]) -> tuple:
    """Per-frame fetch jobs and the batched metadata request."""
    jobs, meta_req = [], {}
    for year, plan in plans.items():
        bands, names = [], []
        y0, y1 = (int(v) for v in CFG.window(year))
        col = _s04_night_col(plan, viirs) if plan.source else None
        if col is not None:
            median, count = _s04_composite(col, _s04_native_of(plan.source), GRID, min_n=1)
            bands += [median.rename("night_raw"), count.rename("night_nobs")]
            names += ["night_raw", "night_nobs"]
            vt = col.select("vtime").mean().reduceRegion(
                reducer=ee.Reducer.mean(), geometry=grid_ee_geom(GRID), scale=5000, bestEffort=True,
                maxPixels=int(1e9)).get("vtime")
            meta_req[str(year)] = ee.Dictionary({"n": col.size(), "vtime": vt})
        if y1 >= _S04_TERRA_FIRST:
            day = _s04_modis_col(_S04_TERRA, "Day", max(y0, _S04_TERRA_FIRST), y1, _S04_DAY_VIEW_WINDOW_H)
            median, _ = _s04_composite(day, _s04_native_of("terra_day"), GRID, min_n=1)
            bands.append(median.rename("day_modis"))
            names.append("day_modis")
        if bands:
            jobs.append({"name": f"s04_frame{year}", "image": ee.Image.cat(bands).unmask(NODATA_FLOAT).toFloat(),
                         "grid": GRID, "bands": names, "dtype": np.float32, "year": year})
    return jobs, meta_req


def _s04_annual_requests(viirs: dict) -> tuple:
    """Fetch jobs of the four annual stacks and their year lists."""
    last = int(_S04_SERIES_YEARS[1])
    years = {
        "aqua_night": list(range(_S04_AQUA_FIRST, last + 1)),
        "terra_night": list(range(_S04_CAL_YEARS["terra_to_aqua"][0], _S04_CAL_YEARS["terra_to_aqua"][1] + 1)),
        "viirs_night": list(range(_S04_VIIRS_FIRST, last + 1)),
        "terra_day": list(range(_S04_TERRA_FIRST, last + 1)),
    }
    jobs = []
    for kind, ys in years.items():
        image, names = _s04_annual_stack(kind, ys, viirs)
        jobs.append({"name": f"s04_annual_{kind}", "image": image, "grid": GRID, "bands": names,
                     "dtype": np.float32})
    return jobs, years


def _s04_class_matrix(years: Sequence[int], dominant: dict) -> np.ndarray:
    """(years, cells) dominant class using the nearest land-cover anchor of each year."""
    anchors = sorted(dominant)
    return np.stack([dominant[_s04_nearest_anchor(y, anchors)][CELL_ROWS, CELL_COLS] for y in years])


def _s04_calibrate(kind: str, src_ann: dict, aqua_ann: dict, dominant: dict) -> dict:
    """Fit one calibration on the overlap years present in both annual stacks."""
    y0, y1 = _S04_CAL_YEARS[kind]
    years = [y for y in range(y0, y1 + 1) if y in src_ann and y in aqua_ann]
    if len(years) < 3:
        raise RuntimeError(f"{kind}: only {len(years)} overlap years available ({years})")
    src = np.stack([_s04_cells(src_ann[y]) for y in years])
    aqua = np.stack([_s04_cells(aqua_ann[y]) for y in years])
    cls = _s04_class_matrix(years, dominant)
    summary, coef, shrunk, info = _s04_fit_calibration(src, aqua, cls)
    loyo = _s04_loyo_rmse(src, aqua, cls)
    diag = {"years": [int(years[0]), int(years[-1])], "n_years": len(years),
            "pooled_classes": info["pooled_classes"], "cells_per_class": dict(info["cells_per_class"]),
            "min_cells_per_class": int(_S04_CAL_MIN_CELLS), "r2_with_residual": info["r2_with_residual"],
            "cells_with_residual": int((info["n_years_per_cell"] > 0).sum()),
            "mean_abs_shrunk_residual": round(float(np.mean(np.abs(shrunk))), 4), **loyo}
    log(f"{kind}: R2 {summary['r2']} RMSE {summary['rmse']} °C n {summary['n']:,} (years {years[0]}-{years[-1]}); "
        f"LOYO RMSE class map {loyo['loyo_rmse_class_map']} -> with shrunk residual "
        f"{loyo['loyo_rmse_with_residual']} °C; pooled classes {info['pooled_classes'] or 'none'}")
    for row in summary["by_class"]:
        log(f"    class {row['class']} ({LC_CLASS_NAMES[row['class']]}): aqua = {row['a']:+.3f} + "
            f"{row['b']:.4f} x src  (n {row['n']:,}, {info['cells_per_class'].get(row['class'], 0):,} cells"
            f"{', pooled map' if row['class'] in info['pooled_classes'] else ''})")
    return {"summary": summary, "coef": coef, "shrunk": shrunk, "diag": diag}


def _s04_overpass(plan: _S04NightPlan, meta: dict, lon_c: float) -> tuple:
    """Measured mean overpass (IST h) of a frame's night product, nominal value as fallback."""
    entry = meta.get(str(plan.year)) or {}
    expected = "evening" if plan.source == "terra_cal" else "after_midnight"
    ist, how = _s04_view_time_to_ist(entry.get("vtime"), expected, lon_c)
    nominal = _S04_NOMINAL_IST_H["terra" if plan.source == "terra_cal" else
                                 "viirs" if plan.source == "viirs_cal" else "aqua"]
    if ist is None:
        log(f"night {plan.year}: overpass from view time unavailable ({how}); using nominal {nominal:.2f} h IST",
            "WARNING")
        return nominal, f"nominal ({how})"
    return ist, f"ROI mean view time ({how})"


def _s04_build_thermal() -> dict:
    """Compute every product of section 04 (the stage-cached unit)."""
    frames = [int(y) for y in CFG.FRAMES_OBS]
    plans = {y: _s04_night_plan(y) for y in frames}
    bounds = GRID.bounds_lonlat()
    lon_c = 0.5 * (bounds[0] + bounds[2])
    caveats, sources = [], {"lst_night": {}, "lst_night_nobs": {}, "lst_day_modis": {}}
    log("night plan: " + ", ".join(f"{y}: {p.source or 'none'} {p.y0}-{p.y1}" if p.source else f"{y}: none"
                                   for y, p in plans.items()))

    with timer("04 VIIRS scale probe"):
        viirs = _s04_viirs_scales()

    # ---- frame composites + metadata ------------------------------------------------------------------------
    jobs, meta_req = _s04_frame_requests(plans, viirs)
    meta = _s04_getinfo(ee.Dictionary(meta_req), "night frame metadata") if meta_req else {}
    fetched = _s04_fetch_jobs(jobs, use_assets=False)
    raw = {job["year"]: {b: fetched[job["name"]][i] for i, b in enumerate(job["bands"])} for job in jobs}
    del fetched

    # ---- annual stacks for calibration and series -------------------------------------------------------------
    ann_jobs, ann_years = _s04_annual_requests(viirs)
    ann_fetched = _s04_fetch_jobs(ann_jobs, use_assets=True)
    annual = {}
    for job in ann_jobs:
        kind = job["name"].replace("s04_annual_", "")
        stack = ann_fetched.pop(job["name"])
        annual[kind] = {int(b[1:]): np.where(ROI_MASK, stack[i], np.nan).astype(np.float32)
                        for i, b in enumerate(job["bands"])}
        del stack
    free_memory()

    # ---- dominant land-cover class at the anchor years ----------------------------------------------------------
    dominant = {}
    for anchor in _S04_GLC_ANCHORS:
        glc = _s04_glc_fractions(anchor)
        dominant[int(anchor)] = glc["dominant"]
    log("dominant GLC class shares at the anchors: " + "; ".join(
        f"{a}: " + "/".join(f"{100 * np.mean(dominant[a][ROI_MASK] == c):.0f}" for c in range(1, 7))
        for a in sorted(dominant)) + " (% built/forest/water/crop/barren/otherveg)")

    # ---- calibrations ----------------------------------------------------------------------------------------
    calib = {"terra_to_aqua": _s04_calibrate("terra_to_aqua", annual["terra_night"], annual["aqua_night"], dominant),
             "viirs_to_aqua": _s04_calibrate("viirs_to_aqua", annual["viirs_night"], annual["aqua_night"], dominant)}

    # ---- frame products ------------------------------------------------------------------------------------
    out_frames, night_meta = {}, {}
    roi_n = int(ROI_MASK.sum())
    for year in frames:
        plan = plans[year]
        r = raw.get(year, {})
        night = np.full(GRID.shape, np.nan, np.float32)
        nobs = np.where(ROI_MASK, 0.0, np.nan).astype(np.float32)
        if plan.source is not None and "night_raw" in r:
            src = np.where(ROI_MASK, r["night_raw"], np.nan)
            nobs = np.where(ROI_MASK, np.nan_to_num(r["night_nobs"], nan=0.0), np.nan).astype(np.float32)
            if plan.source == "aqua":
                night = src.astype(np.float32)
            else:
                kind = "terra_to_aqua" if plan.source == "terra_cal" else "viirs_to_aqua"
                cls = dominant[_s04_nearest_anchor(year, sorted(dominant))][CELL_ROWS, CELL_COLS]
                c = calib[kind]
                vals = _s04_apply_calibration(_s04_cells(src), cls, c["coef"], c["shrunk"])
                night = _s04_grid_from_cells(vals)
                night[~np.isfinite(src)] = np.nan
            night[~ROI_MASK] = np.nan
        day_modis = (np.where(ROI_MASK, r["day_modis"], np.nan).astype(np.float32) if "day_modis" in r
                     else np.full(GRID.shape, np.nan, np.float32))
        coverage = float(np.isfinite(night[ROI_MASK]).sum() / max(1, roi_n))
        if plan.source is not None and coverage == 0.0:
            log(f"night {year}: {plan.source} composite is empty over the ROI - treated as no product", "WARNING")
            caveats.append(f"The {year} night frame has no valid {plan.source} composite; it is modelled instead.")
            plan = _S04NightPlan(year, None, plan.y0, plan.y1)
            plans[year] = plan
            nobs = np.where(ROI_MASK, 0.0, np.nan).astype(np.float32)
        if plan.source is not None:
            ist, how = _s04_overpass(plan, meta, lon_c)
            n_comp = int((meta.get(str(year)) or {}).get("n") or 0)
            night_meta[year] = {"source": plan.source, "sensor": _S04_NIGHT_SENSOR[plan.source],
                                "overpass_ist_h": float(ist), "composites": n_comp, "coverage": round(coverage, 4)}
            unit = "daily observations" if plan.source == "viirs_cal" else "8-day composites"
            cal_txt = ""
            if plan.source != "aqua":
                kind = "terra_to_aqua" if plan.source == "terra_cal" else "viirs_to_aqua"
                cal_txt = (f"; calibrated to Aqua ({kind}: per-class OLS on annual Apr-Jun medians "
                           f"{_S04_CAL_YEARS[kind][0]}-{_S04_CAL_YEARS[kind][1]} + per-cell residual x n/(n+3); "
                           f"LOYO RMSE {calib[kind]['diag']['loyo_rmse_with_residual']} °C)")
            coll = {"aqua": _S04_AQUA, "terra_cal": _S04_TERRA, "viirs_cal": _S04_VIIRS_N}[plan.source]
            qc = ("QC mandatory<=1 & LST accuracy>=2 (<=1.5 K), LST_1KM x" + str(viirs["lst_scale"])
                  if plan.source == "viirs_cal" else "QC_Night bits0-1<=1 & bits6-7<=1, x0.02 K")
            sources["lst_night"][year] = (f"{coll} {plan.y0}-{plan.y1} {CFG.SEASON[0]}..{CFG.SEASON[1]}, {qc}, "
                                          f"per-pixel median, mean-aggregated to {GRID.res} m{cal_txt}; overpass "
                                          f"{ist:.2f} h IST ({how})")
            sources["lst_night_nobs"][year] = f"{coll} count of QC-valid {unit} per pixel, mean to {GRID.res} m"
            if n_comp < 1:
                log(f"night {year}: composite count unavailable from the metadata request", "WARNING")
        else:
            night_meta[year] = {"source": None, "sensor": None, "overpass_ist_h": None, "composites": 0,
                                "coverage": 0.0}
            sources["lst_night"][year] = "no night satellite product for this window (modelled by section 09)"
            sources["lst_night_nobs"][year] = "0 (no night product)"
        y0, y1 = (int(v) for v in CFG.window(year))
        if "day_modis" in r:
            drift = " [drift years included: flagged]" if y1 >= _S04_DRIFT_FROM else ""
            sources["lst_day_modis"][year] = (f"{_S04_TERRA} LST_Day_1km {max(y0, _S04_TERRA_FIRST)}-{y1} Apr-Jun, "
                                              f"QC_Day bits0-1<=1 & bits6-7<=1, Day_view_time x0.1 in "
                                              f"{_S04_DAY_VIEW_WINDOW_H[0]}-{_S04_DAY_VIEW_WINDOW_H[1]} h local "
                                              f"solar time, per-pixel median, mean to {GRID.res} m{drift}")
            if y1 >= _S04_DRIFT_FROM:
                cov_day = float(np.isfinite(day_modis[ROI_MASK]).mean())
                caveats.append(f"MODIS Terra day LST of the {year} frame comes from drifted orbits (~10:07 IST in "
                               f"2025); after the 10.0-11.5 h view-time filter {100 * cov_day:.0f} % of the ROI "
                               "keeps a value. It is a validation layer only.")
        else:
            sources["lst_day_modis"][year] = "no MODIS Terra day product before 2000 (NaN)"
        out_frames[year] = {"lst_night": night.astype(np.float32), "lst_night_nobs": nobs.astype(np.float32),
                            "lst_day_modis": day_modis}
        log(f"frame {year}: night {night_meta[year]['source'] or '-'} coverage {coverage:.3f}"
            + (f", mean {np.nanmean(night):.2f} °C, overpass {night_meta[year]['overpass_ist_h']:.2f} h IST"
               if plan.source else "")
            + (f"; MODIS day mean {np.nanmean(day_modis):.2f} °C" if np.isfinite(day_modis).any() else ""))
    del raw

    # ---- annual series ---------------------------------------------------------------------------------------
    annual_df, series_caveats = _s04_annual_series(annual, calib, dominant)
    caveats += series_caveats
    calibration = {k: calib[k]["summary"] for k in ("terra_to_aqua", "viirs_to_aqua")}
    diag = {k: calib[k]["diag"] for k in calib}
    diag["viirs_scales"] = viirs
    if _S04_EXPORT_DIAG:
        diag["exports"] = list(_S04_EXPORT_DIAG)
    cal_state = {k: {"coef": {int(c): tuple(v) for c, v in calib[k]["coef"].items()},
                     "shrunk": calib[k]["shrunk"]} for k in calib}
    del annual
    free_memory()
    return {"frames": out_frames, "night_meta": night_meta, "calibration": calibration, "diag": diag,
            "cal_state": cal_state, "annual": annual_df, "sources": sources, "caveats": caveats,
            "dominant_anchors": dominant}

## 4.5 Annual series 1990–2025 (`ANNUAL_SERIES`)
* **ERA5-Land** (`era5_skin_day`, `era5_skin_night`, `era5_tmax`, `era5_tmin`, `era5_swvl1`): one image per variable
  with one band per year, reduced with `reduceRegions(mean)` over the 25 district features + the NCR feature at
  1 km in `EPSG:32643` (bilinear input — area-weighted, no empty small districts); one batched `getInfo` per
  variable, halved automatically if Earth Engine reports memory/time-outs.
* **Satellites** (`modis_terra_day` 2000–, `modis_aqua_night` 2003–, `viirs_night_cal` 2012–): scope means of the
  annual 1 km fields (≥ 50 % of the scope's cells valid); VIIRS uses the VIIRS→Aqua calibration per cell.
  `flag = "drift"` for Terra and Aqua from 2022 (their orbits drifted ~1 h); else None.

In [ ]:
def _s04_era5_series(years: Sequence[int]) -> list:
    """Rows (year, scope, variable, value, flag) of the ERA5-Land annual series via batched reduceRegions."""
    rows = []
    scopes = _s04_scopes_fc()

    def _parse(info, ys):
        recs = []
        for props in _s04_fc_records(info):
            did = props.get("did")
            if did is None:
                continue
            scope = "ncr" if int(did) < 0 else f"d:{int(did)}"
            for y in ys:
                value = props.get(f"y{int(y)}")
                if value is not None and np.isfinite(float(value)):
                    recs.append((int(y), scope, float(value)))
        return recs

    for variable in _S04_ERA5_SERIES:
        def _build(ys, variable=variable):
            image = _s04_era5_annual_image(variable, ys)
            names = [f"y{int(y)}" for y in ys]
            # forEach names the outputs: with one band (a batch split down to one year) they would be called "mean"
            fc = image.reduceRegions(collection=scopes, reducer=ee.Reducer.mean().forEach(names), scale=1000,
                                     crs=CFG.CRS, tileScale=4)
            return fc.select(["did"] + names, None, False)

        with timer(f"04 annual series {variable}"):
            recs = _s04_batched(list(years), _build, _parse, f"ERA5 {variable}")
        rows += [(y, s, variable, v, None) for (y, s, v) in recs]
        n_scopes = len({s for _, s, _ in recs})
        log(f"  {variable}: {len(recs)} values over {n_scopes} scopes")
    return rows


def _s04_annual_series(annual: dict, calib: dict, dominant: dict) -> tuple:
    """Assemble ANNUAL_SERIES (ERA5 + satellite) as the SPEC §3.2 DataFrame."""
    caveats = []
    y_first, y_last = _S04_SERIES_YEARS
    rows = _s04_era5_series(range(y_first, y_last + 1))
    district = DISTRICT_IDX[CELL_ROWS, CELL_COLS].astype(np.int64)
    n_d = len(DISTRICTS)
    vc = calib["viirs_to_aqua"]
    sat_inputs = (("modis_terra_day", annual["terra_day"], None, True),
                  ("modis_aqua_night", annual["aqua_night"], None, True),
                  ("viirs_night_cal", annual["viirs_night"], vc, False))
    for variable, fields, cal, drifts in sat_inputs:
        n_rows, skipped = 0, []
        for year in sorted(fields):
            vals = _s04_cells(fields[year])
            if cal is not None:
                cls = dominant[_s04_nearest_anchor(year, sorted(dominant))][CELL_ROWS, CELL_COLS]
                finite = np.isfinite(vals)
                vals = np.where(finite, _s04_apply_calibration(np.nan_to_num(vals), cls, cal["coef"], cal["shrunk"]),
                                np.nan)
            means = _s04_scope_means(vals, district, n_d)
            if "ncr" not in means:
                skipped.append(year)
            flag = "drift" if (drifts and year >= _S04_DRIFT_FROM) else None
            for scope, value in means.items():
                rows.append((int(year), scope, variable, float(value), flag))
                n_rows += 1
        log(f"  {variable}: {n_rows} values" + (f"; NCR mean skipped (coverage < 50 %) in {skipped}" if skipped
                                                else ""))
        if skipped and variable == "modis_terra_day":
            caveats.append("MODIS Terra day annual means are missing where fewer than 50 % of the cells keep a "
                           f"value after the QC and 10.0-11.5 h view-time filters (years {skipped}).")
    rows = [r for r in rows if y_first <= r[0] <= y_last]
    df = pd.DataFrame({
        "year": pd.Series([r[0] for r in rows], dtype="int64"),
        "scope": pd.Series([r[1] for r in rows], dtype=object),
        "variable": pd.Series([r[2] for r in rows], dtype=object),
        "value": pd.Series([r[3] for r in rows], dtype="float64"),
        "flag": pd.Series([r[4] for r in rows], dtype=object),
    })
    df = df.sort_values(["variable", "scope", "year"], kind="stable").reset_index(drop=True)
    missing = sorted(set(_S04_ERA5_SERIES + _S04_SAT_SERIES) - set(df["variable"]))
    if missing:
        raise RuntimeError(f"ANNUAL_SERIES lacks variables {missing}")
    return df, caveats

## 4.6 Run (gee mode) and publish the section-04 globals
The whole section is one stage-cache unit keyed by the grid, ROI, frames, windows, season and code version, so a
resumed Kaggle session (or an imported cache) skips every Earth Engine call. `RAW_OBS` / `FRAME_META` are merged
into (section 03 created them with the day products); `NIGHT_CALIBRATION_DIAG` keeps the leave-one-year-out
transfer errors, the VIIRS run-time scales and the per-class fallbacks for the export's `metrics.night_calibration`.

In [ ]:
def _s04_publish(result: dict) -> None:
    """Merge the section-04 products into the shared acquisition globals."""
    global RAW_OBS, FRAME_META, NIGHT_CALIBRATION, NIGHT_CALIBRATION_DIAG, ANNUAL_SERIES, DATA_SOURCES, DATA_CAVEATS
    if not isinstance(globals().get("RAW_OBS"), dict):
        RAW_OBS = {}
    if not isinstance(globals().get("FRAME_META"), dict):
        FRAME_META = {}
    for year, fields in result["frames"].items():
        RAW_OBS.setdefault(int(year), {}).update(fields)
        entry = FRAME_META.setdefault(int(year), {"window": tuple(int(v) for v in CFG.window(year))})
        entry.setdefault("window", tuple(int(v) for v in CFG.window(year)))
        entry["night"] = dict(result["night_meta"][year])
    NIGHT_CALIBRATION = result["calibration"]
    NIGHT_CALIBRATION_DIAG = result["diag"]
    ANNUAL_SERIES = result["annual"]
    if not isinstance(globals().get("DATA_SOURCES"), dict):
        DATA_SOURCES = {}
    for var, by_year in result["sources"].items():
        DATA_SOURCES.setdefault(var, {}).update({int(y): t for y, t in by_year.items()})
    DATA_SOURCES.setdefault("annual_series", {})["1990-2025"] = (
        f"ERA5-Land ({_S04_ERA5_HOURLY} skin_temperature 05/20 UTC; {_S04_ERA5_DAILY} temperature_2m_max/min; "
        f"{_S04_ERA5_MONTHLY} volumetric_soil_water_layer_1) Apr-Jun means, reduceRegions(mean) at 1 km EPSG:32643 "
        "over NCR + districts; "
        f"{_S04_TERRA} day (QC, view time 10.0-11.5 h), {_S04_AQUA} night (QC), {_S04_VIIRS_N} night calibrated to "
        "Aqua: annual Apr-Jun per-pixel medians at 1 km, scope means of cells (>= 50 % valid); Terra/Aqua >= 2022 "
        "flagged 'drift'")
    DATA_SOURCES.setdefault("night_calibration", {})["2000-2025"] = (
        "per dominant GLC_FCS30D class OLS aqua = a + b*src on annual Apr-Jun medians (Terra night 2003-2019, VIIRS "
        "night 2013-2021) + per-cell mean residual shrunk n/(n+3); dominant class from the nearest of the "
        f"{list(_S04_GLC_ANCHORS)} GLC maps")
    if not isinstance(globals().get("DATA_CAVEATS"), list):
        DATA_CAVEATS = []
    for text in result["caveats"] + [
            "Night LST of the 2025 frame is VIIRS S-NPP (TES algorithm) and of the 2000 frame MODIS Terra (22:30 IST); "
            "both are mapped to Aqua (split-window, ~02:12 IST) by per-class overlap calibrations whose "
            "leave-one-year-out errors are reported in metrics.night_calibration."]:
        if text not in DATA_CAVEATS:
            DATA_CAVEATS.append(text)


if _S04_ACTIVE:
    if ee is None or EE_DISTRICTS is None:
        raise RuntimeError("section 04 needs an initialised Earth Engine session and EE_DISTRICTS (section 01)")
    # code03: section 04 fetches through section-03 helpers (gee_fetch_grid unmask/NaN conventions,
    # gee_fetch_asset, the asset-export helpers), so a change there must invalidate this stage (D4)
    _s04_key = CFG.key("WINDOWS", "SEASON", version=_S04_VERSION, grid=GRID.to_dict(), roi=_s04_roi_digest(),
                       code03=section_code_hash("03"), cal_years={k: list(v) for k, v in _S04_CAL_YEARS.items()},
                       anchors=list(_S04_GLC_ANCHORS), series=list(_S04_SERIES_YEARS))
    _s04_sc = stage_cache("s04_thermal", _s04_key)
    _S04_EXPORT_DIAG.clear()
    with gee_workload_tag("lst-v2-s04"), timer("04 MODIS/VIIRS thermal products"):
        _S04_RESULT = _s04_sc.get_or_compute(_s04_build_thermal)
    _s04_publish(_S04_RESULT)
    # kept for diagnostics: dominant GLC classes at the anchors and the per-cell calibration state
    _S04_DOMINANT = _S04_RESULT["dominant_anchors"]
    _S04_CAL_STATE = _S04_RESULT["cal_state"]
    for _s04_k, _s04_v in NIGHT_CALIBRATION.items():
        log(f"NIGHT_CALIBRATION[{_s04_k}]: R2 {_s04_v['r2']} RMSE {_s04_v['rmse']} n {_s04_v['n']:,}; LOYO "
            f"{NIGHT_CALIBRATION_DIAG[_s04_k]['loyo_rmse_with_residual']} °C")
    log(f"ANNUAL_SERIES: {len(ANNUAL_SERIES):,} rows, variables {sorted(ANNUAL_SERIES['variable'].unique())}")
    del _S04_RESULT, _s04_key, _s04_sc
    free_memory()

## 4.7 Quality-assurance figure
(a) Night frames (Aqua-equivalent) side by side; (b) the NCR annual series — ERA5-Land skin temperature at the two
overpass hours against the satellite series (drift years hollow); (c) the two night calibrations: in-sample R² and
RMSE of the class maps and the leave-one-year-out RMSE with the shrunk residual (°C).

In [ ]:
def _s04_save_show(fig, name: str) -> None:
    path = CFG.FIG_DIR / name
    try:
        fig.savefig(path, dpi=110, bbox_inches="tight")
        log(f"figure -> {path}")
    except OSError as exc:
        log(f"could not save {path}: {exc}", "WARNING")
    plt.show()
    plt.close(fig)


def _s04_qa_figure() -> None:
    frames = [y for y in CFG.FRAMES_OBS if FRAME_META[y]["night"]["source"]]
    n = max(1, len(frames))
    fig = plt.figure(figsize=(3.2 * max(n, 3), 7.2))
    gs = fig.add_gridspec(2, max(n, 3))
    vals = np.concatenate([RAW_OBS[y]["lst_night"][ROI_MASK] for y in frames]) if frames else np.array([np.nan])
    lo, hi = (np.nanpercentile(vals, [2, 98]) if np.isfinite(vals).any() else (0, 1))
    for i, year in enumerate(frames):
        ax = fig.add_subplot(gs[0, i])
        im = ax.imshow(RAW_OBS[year]["lst_night"], cmap="magma", vmin=lo, vmax=hi)
        ax.set_title(f"night {year} ({FRAME_META[year]['night']['source']})", fontsize=9)
        ax.set_axis_off()
    if frames:
        fig.colorbar(im, ax=fig.axes[: len(frames)], shrink=0.6, label="°C (Aqua-equivalent)")
    ax = fig.add_subplot(gs[1, : max(1, max(n, 3) - 1)])
    ncr = ANNUAL_SERIES[ANNUAL_SERIES["scope"] == "ncr"]
    styles = {"era5_skin_day": ("tab:orange", "-"), "era5_skin_night": ("tab:blue", "-"),
              "modis_terra_day": ("tab:red", "o"), "modis_aqua_night": ("navy", "s"),
              "viirs_night_cal": ("tab:purple", "^")}
    for var, (color, marker) in styles.items():
        sub = ncr[ncr["variable"] == var].sort_values("year")
        if sub.empty:
            continue
        if marker == "-":
            ax.plot(sub["year"], sub["value"], color=color, label=var)
        else:
            ok = sub["flag"].isna()
            ax.plot(sub["year"][ok], sub["value"][ok], marker, color=color, label=var, ms=4)
            ax.plot(sub["year"][~ok], sub["value"][~ok], marker, mfc="none", color=color, ms=4)
    ax.set_xlabel("year")
    ax.set_ylabel("°C (NCR Apr-Jun mean)")
    ax.legend(fontsize=7, ncol=3)
    ax.set_title("annual series (hollow = drifted orbit)", fontsize=9)
    ax = fig.add_subplot(gs[1, -1])
    diag = NIGHT_CALIBRATION_DIAG
    rows = [(k, NIGHT_CALIBRATION[k]["r2"], NIGHT_CALIBRATION[k]["rmse"], diag[k]["loyo_rmse_with_residual"])
            for k in ("terra_to_aqua", "viirs_to_aqua")]
    ax.axis("off")
    table = ax.table(cellText=[[k.replace("_to_", "->"), f"{r2}", f"{rmse}", f"{loyo}"]
                               for k, r2, rmse, loyo in rows],
                     colLabels=["map", "R2", "RMSE", "LOYO"], loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.0, 1.6)
    ax.set_title("night calibrations (°C)", fontsize=9)
    _s04_save_show(fig, "04_night_thermal_qa.png")


if _S04_ACTIVE:
    try:
        _s04_qa_figure()
    except Exception as exc:  # noqa: BLE001 - a QA figure must never abort the pipeline
        log(f"section-04 QA figure failed: {type(exc).__name__}: {exc}", "WARNING")

# 5. Drivers 1990–2025 and future inputs 2025–2050 (gee mode only)

Runs only when `DATA_MODE == "gee"` (section 02 provides the same globals in synthetic mode). It completes the
acquisition layer of SPEC §3.2:

| Global | Content (SPEC §2.3) |
|---|---|
| `RAW_STATIC` | SRTM `elevation`, `slope`, `aspect_sin`, `aspect_cos` and JRC `water_occ` at 1 km; `elevation_f`, `water_occ_f` on the fine grid |
| `RAW_OBS[y]` | GHSL `built_f` (fine), `built_v`, `pop`; GLC_FCS30D `glc_*` fractions; ERA5-Land `era5_*` window climatology |
| `LC_FINE[y]` | harmonised GLC_FCS30D classes (0–6) at `CFG.LC_FINE_RES_M` for the landscape metrics |
| `FUTURE_INPUTS` | GDDP-CMIP6 ROI-mean tasmax/tasmin per model × scenario × frame window, the ssp245 2050 warming pattern, Chen-2020 SSP urban-growth ratios per district, GHSL 2030, IIASA SSP urban-population factors, AR6 GSAT increments |

**Design choices and their reasons**

* **One land-cover product for all frames** (GLC_FCS30D: five-yearly maps for 1990/1995, annual maps 2000–2022,
  2025 → the 2022 map): the real-data experiments showed ~1 °C offsets when products are mixed. Fractions use the
  two-step `reduceResolution` (30 m indicators → 250 m → 1 km) defined in section 04 (shared tile cache).
* **GHSL P2023A** is the only consistent 5-yearly built-up/population record 1975–2030 (2025/2030 are GHSL's own
  extrapolations; noted in the caveats). Units: `built_f` = built-up surface fraction (m² / 10⁴ m² per 100 m cell),
  `built_v` = building volume per ground area (m³ m⁻²), `pop` = persons km⁻² (count × 100 at 100 m), all as
  area-weighted means.
* **ERA5-Land** window climatologies (Apr–Jun of every window year): 2 m temperature, RH and VPD from T and Td
  (Magnus–Tetens), soil water layer 1, evaporation (mm day⁻¹, positive = evaporation), net solar radiation
  (MJ m⁻² day⁻¹), wind speed √(u²+v²), daily Tmax/Tmin and Jan–Mar precipitation (mm). Monthly `_sum` bands are
  monthly totals, so a seasonal daily rate is `mean(monthly total) × months / season days` (exact for whole months).
  Fields are bilinearly resampled from 0.1° to 1 km.
* **Future climate** comes from NASA/GDDP-CMIP6 (Earth Engine copy: historical, ssp245, ssp585): for every model with
  tasmax and tasmin, the Apr–Jun mean of the daily values over the 20-year window centred on each frame (2025:
  2016–2035 … 2050: 2041–2060), ROI mean. It is built from small, cached per-(scenario, model, 5-year block)
  requests on the native 0.25° grid (section 5.3); the warming pattern reuses the same blocks. GFDL-CM4 is published
  on two grids (gr1/gr2); only gr1 is used so its days are not double-counted. ssp126 is pattern-scaled later
  (section 11) with the AR6 GSAT increments defined here.
* **Urban growth** per district from Chen et al. (2020) SSP urban land (1 km, value 2 = urban, bands `SSP1…SSP5`,
  decadal 2020–2100), as ratios to the district's own 2020 urban area (ratios avoid the baseline mismatch with GHSL),
  geometrically interpolated to 5-year frames. The SSP bands are resolved by name (`ssp1`/`ssp2`/`ssp5` after
  normalisation) or, for exactly 5 unnamed bands, by position with a warning; anything else, or three identical
  layers, stops the section before any GDDP request.
* **Population growth** factors from the IIASA SSP Scenario Explorer (SSP v3 series): national population of model
  "IIASA-WiC POP 2025" × urban share of "Urbanization Model (World Bank, 2022)" (SSP-extensions platform), India.
  No NCR-specific SSP urban projection exists in that database, so the national urban-population trajectory is the
  documented proxy (caveat recorded).

In [ ]:
_lst_enter_section('05')  # section marker (generated by build_notebook.py)

In [ ]:
import math as _s05_math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_S05_ACTIVE = globals().get("DATA_MODE") == "gee"
_S05_VERSION = "s05-v2.1"   # v2.1: GDDP blocks on the native grid, Chen band resolution, 64 km heavy tiles

_S05_GHSL = "JRC/GHSL/P2023A"                 # GHS_BUILT_S / GHS_BUILT_V / GHS_POP, images "<collection>/<epoch>"
_S05_GHSL_EPOCHS = (1975, 2030)               # every 5 years; 2025 and 2030 are GHSL extrapolations
_S05_GHSL_SCALE_M = 100.0
_S05_SRTM = "USGS/SRTMGL1_003"
_S05_SRTM_SCALE_M = 30.0
_S05_GSW = "JRC/GSW1_4/GlobalSurfaceWater"
_S05_GSW_SCALE_M = 30.0
_S05_GDDP = "NASA/GDDP-CMIP6"
_S05_GDDP_SCALE_M = 27830.0
_S05_GDDP_HIST_LAST = 2014
_S05_GDDP_BLOCK_YEARS = 5                     # one request per (scenario, model, 5-year block): <= ~460 daily images
_S05_GDDP_MEMO: dict = {}                     # (scenario, model, b0, b1, band_filter) -> block data or None (failed)
_S05_GDDP_STATS: dict = {"requests": 0, "cache_hits": 0, "seconds": 0.0, "failed_items": []}
# D2: pilot probes (CMIP6 2050 warming pattern through the block cache); default on for a gee pilot
_S05_PILOT_PROBES = _env_bool("LST_PILOT_PROBES",
                              CFG.PROFILE == "pilot" and globals().get("DATA_MODE") == "gee")
_S05_CHEN = "projects/sat-io/open-datasets/FUTURE-URBAN-LAND/CHEN_2020_2100"
_S05_CHEN_URBAN = 2
_S05_CHEN_BANDS = {"ssp126": "SSP1", "ssp245": "SSP2", "ssp585": "SSP5"}   # expected names (resolved at run time)
_S05_STATIC_KEYS = ["elevation", "slope", "aspect_sin", "aspect_cos", "water_occ"]
_S05_STATIC_FINE_KEYS = ["elevation_f", "water_occ_f"]
_S05_ERA5_KEYS = ["era5_t2m", "era5_rh", "era5_vpd", "era5_swvl1", "era5_evap", "era5_ssr", "era5_wind",
                  "era5_tmax", "era5_tmin", "era5_precip_jfm"]
_S05_GLC_KEYS = list(_S04_GLC_KEYS)
_S05_COARSE_KEYS = ["built_v", "pop"] + _S05_GLC_KEYS + _S05_ERA5_KEYS
# SPEC §3.2 RAW_OBS keys (fine keys except built_f come from section 03, night/MODIS keys from section 04)
_S05_RAW_FINE_KEYS = ["lst_day_f", "lst_day_nobs_f", "ndvi_f", "ndbi_f", "mndwi_f", "albedo_f", "ndvi_spring_f",
                      "built_f"]
_S05_RAW_COARSE_KEYS = ["lst_night", "lst_night_nobs", "lst_day_modis"] + _S05_COARSE_KEYS

# The 34 GDDP-CMIP6 models of the Earth Engine catalog (checked 2026-10-02; re-checked at run time).
_S05_GDDP_MODELS = [
    "ACCESS-CM2", "ACCESS-ESM1-5", "BCC-CSM2-MR", "CESM2", "CESM2-WACCM", "CMCC-CM2-SR5", "CMCC-ESM2", "CNRM-CM6-1",
    "CNRM-ESM2-1", "CanESM5", "EC-Earth3", "EC-Earth3-Veg-LR", "FGOALS-g3", "GFDL-CM4", "GFDL-ESM4", "GISS-E2-1-G",
    "HadGEM3-GC31-LL", "HadGEM3-GC31-MM", "IITM-ESM", "INM-CM4-8", "INM-CM5-0", "IPSL-CM6A-LR", "KACE-1-0-G",
    "KIOST-ESM", "MIROC-ES2L", "MIROC6", "MPI-ESM1-2-HR", "MPI-ESM1-2-LR", "MRI-ESM2-0", "NESM3", "NorESM2-LM",
    "NorESM2-MM", "TaiESM1", "UKESM1-0-LL",
]

# IIASA SSP Scenario Explorer, India (retrieved 2026-10-02 from the public ixmp4 API):
#   population: platform "ssp" (https://data.ece.iiasa.ac.at/ssp), model "IIASA-WiC POP 2025", variable "Population"
#   (million), run version 1 (SSP v3 series, 2025 rebase; Zenodo record "SSP Population and Human-Capital
#   Projections, 2025-Rebased", zen2025V01);
#   urban share: platform "ssp-extensions", model "Urbanization Model (World Bank, 2022)", variable
#   "Population|Urban [Share]" (%), run version 1.
# Years 2025, 2030, ..., 2050. Urban population = population x share / 100.
_S05_SSP_INDIA = {
    "SSP1": {"pop_million": (1445.483, 1492.563, 1525.461, 1545.104, 1554.934, 1554.792),
             "urban_pct": (38.830, 41.501, 44.115, 46.657, 49.218, 51.915)},
    "SSP2": {"pop_million": (1445.483, 1501.738, 1548.090, 1583.720, 1607.738, 1620.410),
             "urban_pct": (35.900, 37.649, 39.393, 41.047, 42.626, 44.212)},
    "SSP5": {"pop_million": (1445.483, 1492.043, 1524.346, 1543.366, 1552.587, 1551.873),
             "urban_pct": (41.479, 45.175, 48.880, 52.582, 56.226, 59.654)},
}
_S05_SSP_YEARS = (2025, 2030, 2035, 2040, 2045, 2050)
_S05_SSP_OF = {"ssp126": "SSP1", "ssp245": "SSP2", "ssp585": "SSP5"}
_S05_SSP_CITATION = ("IIASA SSP Scenario Explorer (SSP v3 series): India 'Population' of model 'IIASA-WiC POP 2025' "
                     "(platform ssp, run v1, 2025-rebased SSP population) x 'Population|Urban [Share]' of "
                     "'Urbanization Model (World Bank, 2022)' (platform ssp-extensions, run v1); retrieved "
                     "2026-10-02 via the public ixmp4 API (https://data.ece.iiasa.ac.at/ssp)")

# IPCC AR6 WG1 Chapter 4, Table 4.5: assessed 20-year GSAT change relative to 1995-2014 (central estimates, °C):
# near term 2021-2040, mid-term 2041-2060, long term 2081-2100 (verified against the published table).
_S05_AR6_GSAT = {"ssp126": {"2021-2040": 0.6, "2041-2060": 0.9, "2081-2100": 0.9},
                 "ssp245": {"2021-2040": 0.7, "2041-2060": 1.1, "2081-2100": 1.8},
                 "ssp585": {"2021-2040": 0.8, "2041-2060": 1.5, "2081-2100": 3.5}}
_S05_AR6_CITATION = ("IPCC AR6 WG1 Ch. 4 (Lee et al. 2021) Table 4.5, assessed GSAT change vs 1995-2014 "
                     "(central values), piecewise-linear in window centre between 1995-2014 (0), 2021-2040, "
                     "2041-2060 and 2081-2100")

if _S05_ACTIVE:
    log(f"section 05 ({_S05_VERSION}): drivers for frames {list(CFG.FRAMES_OBS)}, future inputs for "
        f"{[CFG.NOW] + list(CFG.FRAMES_PROJ)}")
else:
    log("section 05 skipped: data mode is not 'gee' (section 02 provided the synthetic drivers and future inputs)")

## 5.1 Static terrain and surface water (`RAW_STATIC`)
SRTM 30 m elevation and `ee.Terrain.products` slope/aspect; aspect enters as sin/cos **at 30 m before averaging**
(averaging angles directly is meaningless across north). JRC GSW 1.4 `occurrence` (% of valid observations with
water 1984–2021) is unmasked to 0 where water was never seen and scaled to 0–1. Area-weighted means to 1 km and to
the fine grid; isolated gaps inside the ROI (SRTM voids) are filled from the nearest valid pixel.

In [ ]:
def _s05_static_images() -> tuple:
    """(1 km image, fine image) of the static terrain/water bands."""
    dem = ee.Image(_S05_SRTM).select("elevation")
    proj = dem.projection()
    terrain = ee.Terrain.products(dem)
    aspect = terrain.select("aspect").multiply(_s05_math.pi / 180.0)
    gsw = ee.Image(_S05_GSW).select("occurrence")
    water = gsw.unmask(0).divide(100.0).toFloat().rename("water_occ").setDefaultProjection(gsw.projection())
    stack = ee.Image.cat([dem.toFloat().rename("elevation"), terrain.select("slope").toFloat().rename("slope"),
                          aspect.sin().toFloat().rename("aspect_sin"),
                          aspect.cos().toFloat().rename("aspect_cos")]).setDefaultProjection(proj)
    coarse = ee.Image.cat([_s04_to_grid(stack, _S05_SRTM_SCALE_M, GRID),
                           _s04_to_grid(water, _S05_GSW_SCALE_M, GRID)]).unmask(NODATA_FLOAT).toFloat()
    fine = ee.Image.cat([_s04_to_grid(dem.toFloat().rename("elevation_f").setDefaultProjection(proj),
                                      _S05_SRTM_SCALE_M, GRID_FINE),
                         _s04_to_grid(water.rename("water_occ_f"), _S05_GSW_SCALE_M, GRID_FINE)]
                        ).unmask(NODATA_FLOAT).toFloat()
    return coarse, fine


def _s05_build_static() -> tuple:
    """RAW_STATIC dict and provenance."""
    coarse_img, fine_img = _s05_static_images()
    with timer("05 static terrain/water"):
        coarse = _s04_fetch(coarse_img, GRID, _S05_STATIC_KEYS, "s05_static", heavy=True)
        fine = _s04_fetch(fine_img, GRID_FINE, _S05_STATIC_FINE_KEYS, "s05_static_fine", heavy=True)
    static = {}
    for i, key in enumerate(_S05_STATIC_KEYS):
        static[key] = _s04_fill_roi_nearest(coarse[i], ROI_MASK, f"RAW_STATIC[{key}]")
    for i, key in enumerate(_S05_STATIC_FINE_KEYS):
        static[key] = _s04_fill_roi_nearest(fine[i], ROI_MASK_FINE, f"RAW_STATIC[{key}]")
    for key in ("water_occ", "water_occ_f"):
        static[key] = np.where(np.isfinite(static[key]), np.clip(static[key], 0.0, 1.0), np.nan).astype(np.float32)
    terr = (f"{_S05_SRTM} elevation, ee.Terrain.products slope/aspect (aspect sin/cos at 30 m before averaging), "
            f"area-weighted mean to {GRID.res} m")
    sources = {k: terr for k in ("elevation", "slope", "aspect_sin", "aspect_cos")}
    sources["elevation_f"] = terr.replace(f"to {GRID.res} m", f"to {GRID_FINE.res} m")
    sources["water_occ"] = f"{_S05_GSW} occurrence/100 (unmasked 0), area-weighted mean to {GRID.res} m"
    sources["water_occ_f"] = f"{_S05_GSW} occurrence/100 (unmasked 0), area-weighted mean to {GRID_FINE.res} m"
    del coarse, fine
    return static, sources

## 5.2 Per-frame drivers: GHSL, GLC_FCS30D, ERA5-Land, `LC_FINE`
* GHSL epoch = the frame year (5-yearly 1975–2030; clamped to the record).
* ERA5-Land: `MONTHLY_AGGR` months of the season in every window year (mean of monthly means / totals) and
  `DAILY_AGGR` daily Tmax/Tmin; Jan–Mar precipitation = mean monthly total × 3 months (mm per JFM season, averaged
  over the window years). Magnus–Tetens: `e_s(T) = 0.6108·exp(17.27·T/(T+237.3))` kPa, RH = 100·e_s(Td)/e_s(T),
  VPD = e_s(T) − e_s(Td) — computed from monthly mean T and Td (a mild underestimate of the mean of hourly VPD,
  consistent across frames). Wind = |mean vector| of monthly u/v (also slightly below the mean speed).
* `LC_FINE`: the same GLC band, majority (`mode`) aggregation 30 m → `LC_FINE_RES_M`, nodata 0, masked outside the
  ROI; asset-exported first in `asset` mode (largest raster of the section).

In [ ]:
def _s05_ghsl_epoch(year: int) -> int:
    lo, hi = _S05_GHSL_EPOCHS
    return int(min(hi, max(lo, 5 * round(int(year) / 5))))


def _s05_ghsl(kind: str, epoch: int, band: str):
    image = ee.Image(f"{_S05_GHSL}/{kind}/{int(epoch)}").select(band)
    return image, image.projection()


def _s05_ghsl_fine_built(epoch: int):
    built, proj = _s05_ghsl("GHS_BUILT_S", epoch, "built_surface")
    frac = built.toFloat().divide(10000.0).rename("built_f").setDefaultProjection(proj)
    return _s04_to_grid(frac, _S05_GHSL_SCALE_M, GRID_FINE).unmask(NODATA_FLOAT).toFloat()


def _s05_ghsl_coarse(epoch: int, built_band: str = "built_v"):
    """1 km built volume (m³ m⁻²) and population density (persons km⁻²)."""
    vol, vproj = _s05_ghsl("GHS_BUILT_V", epoch, "built_volume_total")
    pop, pproj = _s05_ghsl("GHS_POP", epoch, "population_count")
    vol = vol.toFloat().divide(10000.0).rename(built_band).setDefaultProjection(vproj)
    pop = pop.toFloat().multiply(100.0).rename("pop").setDefaultProjection(pproj)
    return [_s04_to_grid(vol, _S05_GHSL_SCALE_M, GRID), _s04_to_grid(pop, _S05_GHSL_SCALE_M, GRID)]


def _s05_magnus_es(t_c):
    """Saturation vapour pressure (kPa) of an ee.Image of temperature in °C (Magnus-Tetens)."""
    return t_c.multiply(17.27).divide(t_c.add(237.3)).exp().multiply(0.6108)


def _s05_era5_frame_image(year: int):
    """The ten ERA5-Land window-climatology bands of SPEC §2.3 (bilinear, on GRID)."""
    months = _s04_season_months()
    if months is None:
        raise ValueError(f"ERA5-Land monthly climatology needs a month-aligned season, got {CFG.SEASON}")
    n_months = months[1] - months[0] + 1
    days = _s04_season_days()
    y0, y1 = (int(v) for v in CFG.window(year))
    start, end = _s04_years_dates(y0, y1)
    native = _s04_native(_S04_ERA5_MONTHLY, "temperature_2m")
    mon = ee.ImageCollection(_S04_ERA5_MONTHLY).filterDate(start, end).filter(_s04_season_filter())
    m = mon.mean()
    t = m.select("temperature_2m").subtract(273.15)
    td = m.select("dewpoint_temperature_2m").subtract(273.15)
    es, ea = _s05_magnus_es(t), _s05_magnus_es(td)
    rh = ea.divide(es).multiply(100.0).min(100.0)
    vpd = es.subtract(ea).max(0.0)
    rate = float(n_months) / float(days)          # monthly totals -> per-day rate over the season
    evap = m.select("total_evaporation_sum").multiply(-1000.0 * rate)        # m (negative = evaporation) -> mm/day
    ssr = m.select("surface_net_solar_radiation_sum").multiply(rate / 1e6)    # J m-2 -> MJ m-2 day-1
    u, v = m.select("u_component_of_wind_10m"), m.select("v_component_of_wind_10m")
    wind = u.pow(2).add(v.pow(2)).sqrt()
    daily = (ee.ImageCollection(_S04_ERA5_DAILY).filterDate(start, end).filter(_s04_season_filter())
             .select(["temperature_2m_max", "temperature_2m_min"]).mean().subtract(273.15))
    jfm = (ee.ImageCollection(_S04_ERA5_MONTHLY).filterDate(start, end)
           .filter(ee.Filter.calendarRange(1, 3, "month")).select("total_precipitation_sum").mean()
           .multiply(3.0 * 1000.0))                                            # mm per Jan-Mar season
    stack = ee.Image.cat([
        t.rename("era5_t2m"), rh.rename("era5_rh"), vpd.rename("era5_vpd"),
        m.select("volumetric_soil_water_layer_1").rename("era5_swvl1"), evap.rename("era5_evap"),
        ssr.rename("era5_ssr"), wind.rename("era5_wind"), daily.select("temperature_2m_max").rename("era5_tmax"),
        daily.select("temperature_2m_min").rename("era5_tmin"), jfm.rename("era5_precip_jfm"),
    ]).toFloat()
    return stack.setDefaultProjection(native).resample("bilinear").reproject(crs=GRID.crs,
                                                                            crsTransform=GRID.transform_list())


def _s05_frame_coarse_image(year: int):
    """built_v, pop and the ERA5 bands of one frame as one 1 km image."""
    epoch = _s05_ghsl_epoch(year)
    bands = _s05_ghsl_coarse(epoch) + [_s05_era5_frame_image(year)]
    return ee.Image.cat(bands).unmask(NODATA_FLOAT).toFloat(), ["built_v", "pop"] + _S05_ERA5_KEYS


def _s05_lc_fine_image(year: int, lc_grid: GridSpec):
    """Harmonised GLC classes on the land-cover grid (uint8, 0 = nodata)."""
    coll, band, map_year = _s04_glc_band(year)
    classes, _ = _s04_glc_classes(coll, band)
    if _S04_GLC_SCALE_M < lc_grid.res * 0.999:
        ratio = (lc_grid.res / _S04_GLC_SCALE_M) ** 2
        classes = classes.reduceResolution(reducer=ee.Reducer.mode(), bestEffort=False,
                                           maxPixels=int(max(1024, _s05_math.ceil(ratio * 2.5))))
    image = classes.reproject(crs=lc_grid.crs, crsTransform=lc_grid.transform_list()).unmask(0).toUint8()
    return image.rename("lc"), coll, band, map_year


def _s05_build_drivers() -> dict:
    """Per-frame driver fields, LC_FINE and provenance (the stage-cached unit)."""
    frames = [int(y) for y in CFG.FRAMES_OBS]
    lc_res = int(CFG.fine_res())
    lc_grid = GRID.fine(lc_res)
    f_lc = GRID.res // lc_res
    static, sources = _s05_build_static()
    sources = {k: {y: v for y in frames} for k, v in sources.items()}
    for key in ["built_f", "land_cover"] + _S05_COARSE_KEYS:
        sources.setdefault(key, {})
    obs, lc_fine, caveats = {}, {}, []

    # GHSL fine built-up and the 1 km stack per frame (interactive: light requests)
    for year in frames:
        epoch = _s05_ghsl_epoch(year)
        with timer(f"05 drivers {year}"):
            built_f = _s04_fetch(_s05_ghsl_fine_built(epoch), GRID_FINE, ["built_f"], f"s05_built_f_{epoch}",
                                 tile_px=gee_tile_px(1, np.float32, cap=1024))[0]
            image, names = _s05_frame_coarse_image(year)
            coarse = _s04_fetch(image, GRID, names, f"s05_coarse_{year}", heavy=True)
            glc = _s04_glc_fractions(year)
        rec = {"built_f": np.clip(_s04_fill_roi_nearest(built_f, ROI_MASK_FINE, f"built_f {year}"), 0.0, 1.0)}
        rec["built_f"][~ROI_MASK_FINE] = np.nan
        for i, key in enumerate(names):
            rec[key] = _s04_fill_roi_nearest(coarse[i], ROI_MASK, f"{key} {year}")
        rec["built_v"] = np.where(ROI_MASK, np.clip(rec["built_v"], 0.0, None), np.nan).astype(np.float32)
        rec["pop"] = np.where(ROI_MASK, np.clip(rec["pop"], 0.0, None), np.nan).astype(np.float32)
        rec.update({k: glc["fractions"][k].astype(np.float32) for k in _S05_GLC_KEYS})
        obs[year] = {k: np.ascontiguousarray(v, dtype=np.float32) for k, v in rec.items()}
        del built_f, coarse
        tag = " (GHSL extrapolation)" if epoch >= 2025 else ""
        sources["built_f"][year] = (f"{_S05_GHSL}/GHS_BUILT_S/{epoch} built_surface/1e4 (fraction) at 100 m, "
                                    f"area-weighted mean to {GRID_FINE.res} m{tag}")
        sources["built_v"][year] = (f"{_S05_GHSL}/GHS_BUILT_V/{epoch} built_volume_total/1e4 (m3 per m2 ground), "
                                    f"area-weighted mean to {GRID.res} m{tag}")
        sources["pop"][year] = (f"{_S05_GHSL}/GHS_POP/{epoch} population_count x100 (persons/km2 at 100 m), "
                                f"area-weighted mean to {GRID.res} m{tag}")
        glc_txt = (f"{glc['collection']} band {glc['band']} (= {glc['map_year']}), v1 class remap, class-indicator "
                   f"means 30 m -> {GRID_FINE.res} m -> {GRID.res} m (two-step reduceResolution), renormalised")
        if glc["map_year"] != year:
            glc_txt += f" [{glc['map_year']} map used for the {year} frame]"
        for k in _S05_GLC_KEYS:
            sources[k][year] = glc_txt
        y0, y1 = (int(v) for v in CFG.window(year))
        era = (f"{_S04_ERA5_MONTHLY} {y0}-{y1} {CFG.SEASON[0]}..{CFG.SEASON[1]} monthly mean, bilinear to "
               f"{GRID.res} m")
        sources["era5_t2m"][year] = f"{era}: temperature_2m - 273.15 (°C)"
        sources["era5_rh"][year] = f"{era}: Magnus RH from temperature_2m and dewpoint_temperature_2m (%)"
        sources["era5_vpd"][year] = f"{era}: Magnus es(T) - es(Td) (kPa)"
        sources["era5_swvl1"][year] = f"{era}: volumetric_soil_water_layer_1 (m3 m-3)"
        sources["era5_evap"][year] = (f"{era}: -total_evaporation_sum x1000 x months/season days (mm/day, positive "
                                      "= evaporation)")
        sources["era5_ssr"][year] = f"{era}: surface_net_solar_radiation_sum / 1e6 x months/season days (MJ m-2 d-1)"
        sources["era5_wind"][year] = f"{era}: sqrt(u10^2 + v10^2) of the monthly mean components (m/s)"
        sources["era5_tmax"][year] = (f"{_S04_ERA5_DAILY} {y0}-{y1} Apr-Jun mean of temperature_2m_max - 273.15 "
                                      f"(°C), bilinear to {GRID.res} m")
        sources["era5_tmin"][year] = sources["era5_tmax"][year].replace("temperature_2m_max", "temperature_2m_min")
        sources["era5_precip_jfm"][year] = (f"{_S04_ERA5_MONTHLY} {y0}-{y1} Jan-Mar total_precipitation_sum: mean "
                                            f"monthly total x 3 x 1000 (mm per JFM season), bilinear to {GRID.res} m")
        log(f"frame {year}: built_f {np.nanmean(obs[year]['built_f']):.3f}, built_v "
            f"{np.nanmean(obs[year]['built_v']):.2f} m3/m2, pop {np.nanmean(obs[year]['pop']):,.0f}/km2, glc_built "
            f"{np.nanmean(obs[year]['glc_built']):.3f}, era5_tmax {np.nanmean(obs[year]['era5_tmax']):.2f} °C, "
            f"swvl1 {np.nanmean(obs[year]['era5_swvl1']):.3f}")

    # LC_FINE (largest rasters: asset-exported first in asset mode)
    lc_jobs = []
    for year in frames:
        image, coll, band, map_year = _s05_lc_fine_image(year, lc_grid)
        lc_jobs.append({"name": f"s05_lc_{band}_{lc_res}m", "image": image, "grid": lc_grid, "bands": ["lc"],
                        "dtype": np.uint8, "tile_px": int(CFG.GEE_FINE_TILE_PX), "year": year,
                        "src": (coll, band, map_year)})
    # identical bands (e.g. a frame reusing the 2022 map) are fetched once
    unique = {}
    for job in lc_jobs:
        unique.setdefault(job["name"], job)
    fetched = _s04_fetch_jobs(list(unique.values()), use_assets=True)
    for job in lc_jobs:
        lc = np.ascontiguousarray(fetched[job["name"]][0], dtype=np.uint8).copy()
        lc[lc > 6] = 0
        _mask_fine_inplace(lc, ROI_MASK, f_lc)
        lc_fine[job["year"]] = lc
        coll, band, map_year = job["src"]
        sources["land_cover"][job["year"]] = (f"{coll} band {band} (= {map_year}), v1 class remap, mode-aggregated "
                                              f"to {lc_res} m")
        shares = np.bincount(lc.ravel(), minlength=7)[1:] / max(1, int((lc > 0).sum()))
        log(f"  LC_FINE {job['year']} ({band}, {lc_res} m): " + ", ".join(
            f"{LC_CLASS_NAMES[c].split('/')[0]} {100 * s:.1f}%" for c, s in zip(range(1, 7), shares)))
    del fetched
    if any(_s05_ghsl_epoch(y) >= 2025 for y in frames):
        caveats.append("GHSL built-up, volume and population of the 2025 frame are GHSL P2023A extrapolations.")
    if any(_s04_glc_band(y)[2] != y for y in frames):
        caveats.append("Land cover of the 2025 frame (and of any frame after 2022) is the GLC_FCS30D 2022 map.")
    free_memory()
    return {"static": static, "obs": obs, "lc_fine": lc_fine, "sources": sources, "caveats": caveats}

## 5.3 Future inputs (`FUTURE_INPUTS`, SPEC §3.2 / §4.9–4.10)
**GDDP-CMIP6 blocks (D7).** There are no 20-year interactive reductions. Every value comes from per-(scenario,
model, 5-year block) means: blocks 2016–2020, 2021–2025, …, 2056–2060 (a window starting in 2014 or earlier joins the
`historical` scenario). Each block is **one** `ee.data.computePixels` call of `[mean(tasmax, tasmin), count]` over
the Apr–Jun days of that model and block (≤ ~460 daily images) on a small EPSG:4326 grid aligned to the native
0.25° GDDP lattice (grid bounds + 1 cell, ~16 × 18 px). Transient errors are retried, a failing item is isolated
(the model is dropped by the completeness filter), nothing is split. Results are cached as
`CACHE_DIR/gddp/<scenario>_<model>_<b0>_<b1>_<expression digest>.npz`, so a failure later in the section (or a
new session with an imported cache) costs no further GDDP request. Blocks run in a thread pool of
`CFG.GEE_WORKERS`, with the time budget checked before every request.

**CMIP6 table.** For each frame F (2025 baseline + projections) the window W(F) = `CFG.PROJ_WINDOWS[F]` (20 years
centred on F) is covered by 4 blocks; its field is the per-pixel day-count-weighted mean of the blocks, bilinearly
interpolated to the 1 km cell centres, and `value_k` is the mean of that field over the ROI cells, per scenario
s ∈ {ssp245, ssp585}, model and variable. Day counts (at the ROI-centre pixel) are checked against the season length,
models incomplete in any frame are dropped and values must lie in 270–330 K.

**Warming pattern.** Median over the complete ssp245 models of (field W(2050) − field W(2025)) on the grid, divided
by its ROI mean (≈ 1 everywhere; > 1 where the GCMs warm faster than the NCR average) — from the same cached blocks.
The pilot (no projection frames) keeps a uniform pattern but **probes** this computation for 2050
(`LST_PILOT_PROBES`, default on for a gee pilot): its time, requests and ROI-mean warming are recorded in
`FUTURE_INPUTS_DIAG['probes']` and printed as the PILOT PROBES table; a probe failure is a loud warning, not an
error.

**AR6 GSAT increments.** `G_s(c)` = assessed GSAT change vs 1995–2014 at window centre c, piecewise linear through
(2004.5, 0), (2030.5, near term), (2050.5, mid-term), (2090.5, long term); `gsat_increment[s][F] = G_s(c(W(F))) −
G_s(c(W(2025)))`. Section 11 pattern-scales ssp126 as `ΔT_245 × inc_126 / inc_245`.

In [ ]:
def _s05_gcm_window(frame: int) -> tuple:
    """20-year GCM window of a projection frame or of NOW (``CFG.PROJ_WINDOWS``: 2025 -> 2016-2035, ...).

    Not ``CFG.window``: for NOW that returns the 2024-2026 *observation* window of the 2025 frame.
    """
    frame = int(frame)
    if frame not in CFG.PROJ_WINDOWS:
        raise KeyError(f"no GCM window for frame {frame} (CFG.PROJ_WINDOWS has {sorted(CFG.PROJ_WINDOWS)})")
    y0, y1 = CFG.PROJ_WINDOWS[frame]
    return int(y0), int(y1)


def _s05_window_centre(frame: int) -> float:
    y0, y1 = _s05_gcm_window(frame)
    return 0.5 * (y0 + y1)


def _s05_ar6_gsat_at(scenario: str, centre: float) -> float:
    """AR6 assessed GSAT change (vs 1995-2014) at a 20-year window centre, piecewise linear (extrapolated flat
    beyond the last assessed window)."""
    table = _S05_AR6_GSAT[scenario]
    xs = [2004.5, 2030.5, 2050.5, 2090.5]
    ys = [0.0, table["2021-2040"], table["2041-2060"], table["2081-2100"]]
    return float(np.interp(float(centre), xs, ys))


def _s05_gsat_increment(frames: Sequence[int]) -> dict:
    """{scenario: {frame: K}} GSAT increment of each frame window over the NOW window (SPEC §3.2)."""
    base_centre = _s05_window_centre(CFG.NOW)
    return {s: {int(f): round(_s05_ar6_gsat_at(s, _s05_window_centre(f)) - _s05_ar6_gsat_at(s, base_centre), 4)
                for f in frames} for s in _S05_AR6_GSAT}


def _s05_pop_growth(frames: Sequence[int]) -> tuple:
    """DataFrame [scenario, frame, factor]: India urban population vs NOW per SSP (geometric between 5-year values,
    last 5-year growth rate extrapolated beyond 2050), plus caveats."""
    caveats, rows = [], []
    years = np.asarray(_S05_SSP_YEARS, np.float64)
    for scenario, ssp in _S05_SSP_OF.items():
        t = _S05_SSP_INDIA[ssp]
        urban = np.asarray(t["pop_million"]) * np.asarray(t["urban_pct"]) / 100.0
        log_u = np.log(urban)

        def at(year: float, log_u=log_u) -> float:
            if year <= years[-1]:
                return float(np.exp(np.interp(year, years, log_u)))
            rate = (log_u[-1] - log_u[-2]) / (years[-1] - years[-2])
            return float(np.exp(log_u[-1] + rate * (year - years[-1])))

        base = at(float(CFG.NOW))
        for frame in frames:
            rows.append((scenario, int(frame), round(at(float(frame)) / base, 5)))
    if max(frames) > _S05_SSP_YEARS[-1]:
        caveats.append(f"SSP urban-population factors beyond {_S05_SSP_YEARS[-1]} extrapolate the last 5-year "
                       "growth rate.")
    return pd.DataFrame(rows, columns=["scenario", "frame", "factor"]), caveats


def _s05_gddp_models() -> list:
    """Models present in the Earth Engine GDDP-CMIP6 copy (one cheap batched request), else the documented 34."""
    def _day(scenario, day):
        return (ee.ImageCollection(_S05_GDDP).filterDate(day, (pd.Timestamp(day) + pd.Timedelta(days=1))
                                                         .strftime("%Y-%m-%d"))
                .filter(ee.Filter.eq("scenario", scenario)).aggregate_array("model").distinct())
    try:
        info = _s04_getinfo(ee.Dictionary({"historical": _day("historical", "2010-05-01"),
                                           "ssp245": _day("ssp245", "2050-05-01"),
                                           "ssp585": _day("ssp585", "2050-05-01")}), "GDDP-CMIP6 model list")
        found = sorted(set(info.get("ssp245") or []) | set(info.get("ssp585") or []))
    except Exception as exc:  # noqa: BLE001 - the documented list is a safe fallback
        log(f"GDDP-CMIP6 model discovery failed ({str(exc)[:160]}); using the documented 34 models", "WARNING")
        return list(_S05_GDDP_MODELS)
    if not found:
        log("GDDP-CMIP6 model discovery returned nothing; using the documented 34 models", "WARNING")
        return list(_S05_GDDP_MODELS)
    extra, missing = sorted(set(found) - set(_S05_GDDP_MODELS)), sorted(set(_S05_GDDP_MODELS) - set(found))
    if extra or missing:
        log(f"GDDP-CMIP6 models differ from the documented list: +{extra} -{missing}", "WARNING")
    return found


def _s05_gddp_window(scenario: str, y0: int, y1: int, band_filter: bool = True):
    """Daily Apr-Jun GDDP images of a window (historical <= 2014 joined with the scenario), tasmax+tasmin only.

    ``band_filter`` keeps only images whose ``system:band_names`` list both variables (models without them drop
    out cleanly); without it a model lacking a band fails on ``select`` and is isolated by the batched request.
    """
    def _base(start, end):
        col = ee.ImageCollection(_S05_GDDP).filterDate(start, end).filter(_s04_season_filter())
        if band_filter:
            col = (col.filter(ee.Filter.listContains("system:band_names", "tasmax"))
                   .filter(ee.Filter.listContains("system:band_names", "tasmin")))
        return col.filter(ee.Filter.Or(ee.Filter.neq("model", "GFDL-CM4"), ee.Filter.eq("grid_label", "gr1")))

    parts = []
    if y0 <= _S05_GDDP_HIST_LAST:
        start, end = _s04_years_dates(y0, min(y1, _S05_GDDP_HIST_LAST))
        parts.append(_base(start, end).filter(ee.Filter.eq("scenario", "historical")))
    if y1 > _S05_GDDP_HIST_LAST:
        start, end = _s04_years_dates(max(y0, _S05_GDDP_HIST_LAST + 1), y1)
        parts.append(_base(start, end).filter(ee.Filter.eq("scenario", scenario)))
    col = parts[0]
    for extra in parts[1:]:
        col = col.merge(extra)
    return col.select(["tasmax", "tasmin"])


def _s05_window_for(frame: int) -> tuple:
    """GCM window of a frame: ``CFG.PROJ_WINDOWS`` when defined, else the SPEC rule ``_projection_window`` (the
    pilot probe needs the 2050 window although the pilot has no projection frames)."""
    frame = int(frame)
    if frame in CFG.PROJ_WINDOWS:
        return _s05_gcm_window(frame)
    y0, y1 = _projection_window(frame)
    return int(y0), int(y1)


def _s05_blocks(y0: int, y1: int) -> list:
    """5-year blocks covering [y0, y1]: (y0, y0+4), (y0+5, y0+9), ... (2016-2020, ..., 2056-2060 for the SPEC
    windows); a window starting <= 2014 gets blocks that join the historical scenario via _s05_gddp_window."""
    return [(int(b0), int(min(b0 + _S05_GDDP_BLOCK_YEARS - 1, y1)))
            for b0 in range(int(y0), int(y1) + 1, _S05_GDDP_BLOCK_YEARS)]


def _s05_gddp_lattice() -> dict:
    """EPSG:4326 request grid aligned to the native 0.25° GDDP lattice, covering the grid bounds + 1 cell (memoised).

    The native transform is read once from the catalog (``projection().getInfo()``); if it cannot be parsed, a 0.25°
    lattice anchored at lon 0 / lat 90 is used (the GDDP grid has its pixel edges on multiples of 0.25°).
    """
    memo = globals().get("_S05_GDDP_LATTICE")
    if memo:
        return memo
    sx, ox, sy, oy = 0.25, 0.0, -0.25, 90.0
    how = "fallback 0.25 deg lattice anchored at lon 0 / lat 90"
    try:
        info = _s04_getinfo(_s04_native(_S05_GDDP, "tasmax"), "GDDP-CMIP6 native projection") or {}
        tr, crs = info.get("transform"), info.get("crs")
        if (isinstance(tr, (list, tuple)) and len(tr) == 6 and crs in (None, "EPSG:4326")
                and abs(float(tr[1])) < 1e-12 and abs(float(tr[3])) < 1e-12
                and 0.01 < float(tr[0]) < 2.0 and -2.0 < float(tr[4]) < -0.01):
            sx, ox, sy, oy = float(tr[0]), float(tr[2]), float(tr[4]), float(tr[5])
            how = f"native {crs or 'EPSG:4326'} transform {list(tr)}"
        else:
            log(f"GDDP-CMIP6 native projection not usable ({str(info)[:160]}); using the {how}", "WARNING")
    except Exception as exc:  # noqa: BLE001 - the fallback lattice is the documented 0.25 deg grid
        log(f"GDDP-CMIP6 native projection unavailable ({type(exc).__name__}: {str(exc)[:160]}); using the {how}",
            "WARNING")
    lon0, lat0, lon1, lat1 = GRID.bounds_lonlat()
    i0 = int(math.floor((lon0 - ox) / sx)) - 1
    i1 = int(math.ceil((lon1 - ox) / sx)) + 1
    j0 = int(math.floor((lat1 - oy) / sy)) - 1          # sy < 0: row index grows southwards
    j1 = int(math.ceil((lat0 - oy) / sy)) + 1
    x0, y0 = ox + i0 * sx, oy + j0 * sy
    width, height = i1 - i0, j1 - j0
    lattice = {"transform": [sx, 0.0, x0, 0.0, sy, y0], "width": int(width), "height": int(height), "how": how,
               "affine": {"scaleX": sx, "shearX": 0.0, "translateX": x0, "shearY": 0.0, "scaleY": sy,
                          "translateY": y0}}
    log(f"GDDP-CMIP6 request grid: {width}x{height} px of {sx:g} deg from ({x0:g}, {y0:g}) [{how}]")
    globals()["_S05_GDDP_LATTICE"] = lattice
    return lattice


def _s05_gddp_cache_path(scen_eff: str, model: str, b0: int, b1: int, digest: str) -> Path:
    safe = re.sub(r"[^A-Za-z0-9_.-]", "_", f"{scen_eff}_{model}_{int(b0)}_{int(b1)}_{digest[:10]}")
    return CFG.CACHE_DIR / "gddp" / f"{safe}.npz"


def _s05_gddp_block_image(scenario: str, model: str, b0: int, b1: int, band_filter: bool):
    """Mean daily tasmax/tasmin and the day count ``n`` of one model over the Apr-Jun days of one 5-year block."""
    col = _s05_gddp_window(scenario, b0, b1, band_filter).filter(ee.Filter.eq("model", model))
    img = ee.Image.cat([col.mean(), col.select("tasmax").count().rename("n")]).toFloat()
    return img.unmask(NODATA_FLOAT, False)


def _s05_gddp_block(scenario: str, model: str, b0: int, b1: int, band_filter: bool, lattice: dict,
                    slowest: list) -> tuple:
    """One block: ``(data or None, from_cache, error)``; data = {tasmax, tasmin (K), n} on the request grid.

    One ``ee.data.computePixels`` call (NUMPY_NDARRAY) on the aligned EPSG:4326 grid, never split; transient errors
    are retried by ``_s03_call``, a permanent error isolates the item (returned as an error, the completeness filter
    then drops the model). The result is cached atomically under ``CACHE_DIR/gddp`` (expression digest in the name;
    the grid transform is checked on load).
    """
    scen_eff = "historical" if int(b1) <= _S05_GDDP_HIST_LAST else scenario
    img = _s05_gddp_block_image(scenario, model, b0, b1, band_filter)
    digest = _gee_expression_digest(img)
    path = _s05_gddp_cache_path(scen_eff, model, b0, b1, digest)
    shape = (int(lattice["height"]), int(lattice["width"]))
    if not path.exists():
        cache_restore(path)
    if path.exists():
        try:
            with np.load(path) as z:
                if (tuple(z["tasmax"].shape) == shape
                        and np.allclose(np.asarray(z["transform"], np.float64), lattice["transform"], atol=1e-9)):
                    return {k: np.asarray(z[k], np.float32) for k in ("tasmax", "tasmin", "n")}, True, None
            log(f"GDDP block cache {path.name} was made on another request grid; refetching", "WARNING")
        except (OSError, ValueError, KeyError) as exc:
            log(f"ignoring unreadable GDDP block cache {path.name}: {exc}", "WARNING")
    check_budget(f"GDDP block {scen_eff} {model} {b0}-{b1}", 1.25 * slowest[0])
    request = {"expression": img, "fileFormat": "NUMPY_NDARRAY", "bandIds": ["tasmax", "tasmin", "n"],
               "grid": {"dimensions": {"width": shape[1], "height": shape[0]}, "affineTransform": lattice["affine"],
                        "crsCode": "EPSG:4326"}}
    t0 = time.perf_counter()
    try:
        raw = _s03_call(lambda: ee.data.computePixels(request), f"GDDP {scen_eff} {model} {b0}-{b1}")
        raw = np.asarray(raw)
        if raw.dtype.names:
            stack = np.stack([np.asarray(raw[b], np.float32) for b in ("tasmax", "tasmin", "n")])
        else:
            stack = np.moveaxis(np.asarray(raw, np.float32), -1, 0)
        if stack.shape != (3,) + shape:
            raise RuntimeError(f"computePixels returned shape {stack.shape}, expected {(3,) + shape}")
    except Exception as exc:  # noqa: BLE001 - permanent error: isolate this item
        return None, False, f"{type(exc).__name__}: {str(exc)[:240]}"
    slowest[0] = max(slowest[0], time.perf_counter() - t0)
    stack = stack.astype(np.float32)
    stack[stack == NODATA_FLOAT] = np.nan
    n = np.nan_to_num(stack[2], nan=0.0)
    data = {"tasmax": np.where(n > 0, stack[0], np.nan).astype(np.float32),
            "tasmin": np.where(n > 0, stack[1], np.nan).astype(np.float32), "n": n.astype(np.float32)}
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    try:
        with open(tmp, "wb") as fh:
            np.savez(fh, **data, transform=np.asarray(lattice["transform"], np.float64))
        os.replace(tmp, path)
    except OSError as exc:
        log(f"GDDP block cache {path.name} not written ({exc})", "WARNING")
    return data, False, None


def _s05_gddp_key(scenario: str, model: str, b0: int, b1: int, band_filter: bool) -> tuple:
    scen_eff = "historical" if int(b1) <= _S05_GDDP_HIST_LAST else scenario
    return scen_eff, str(model), int(b0), int(b1), bool(band_filter)


def _s05_gddp_fetch_blocks(items: Sequence[tuple], band_filter: bool, label: str) -> dict:
    """Fetch (scenario, model, b0, b1) blocks in a thread pool of ``CFG.GEE_WORKERS`` (memo + disk cache).

    Updates ``_S05_GDDP_MEMO`` ({key: data or None}) and ``_S05_GDDP_STATS`` (requests, cache_hits, failed_items,
    seconds); returns the statistics of this call. The time budget is checked before every request.
    """
    lattice = _s05_gddp_lattice()
    memo = _S05_GDDP_MEMO
    todo, seen = [], set()
    for s, m, b0, b1 in items:
        key = _s05_gddp_key(s, m, b0, b1, band_filter)
        if key not in memo and key not in seen:
            seen.add(key)
            todo.append((s, m, b0, b1, key))
    call = {"items": len(todo), "requests": 0, "cache_hits": 0, "failed_items": [], "seconds": 0.0}
    if not todo:
        return call
    t0 = time.perf_counter()
    slowest = [0.0]
    step = max(1, len(todo) // 10)

    def _job(spec):
        s, m, b0, b1, key = spec
        return key, _s05_gddp_block(s, m, b0, b1, band_filter, lattice, slowest)

    pool = ThreadPoolExecutor(max_workers=max(1, min(CFG.GEE_WORKERS, len(todo))), thread_name_prefix="gddp")
    futures = [pool.submit(_job, spec) for spec in todo]
    done = 0
    try:
        for fut in as_completed(futures):
            key, (data, cached, error) = fut.result()
            memo[key] = data
            done += 1
            if cached:
                call["cache_hits"] += 1
            else:
                call["requests"] += 1
            if error is not None:
                call["failed_items"].append({"item": list(key[:4]), "error": error})
                log(f"GDDP block {key[0]} {key[1]} {key[2]}-{key[3]} failed and is skipped (the model is dropped "
                    f"if incomplete): {error}", "WARNING")
            if done % step == 0 or done == len(todo):
                log(f"  {label}: {done}/{len(todo)} GDDP blocks ({call['cache_hits']} from cache, "
                    f"{len(call['failed_items'])} failed, {time.perf_counter() - t0:.0f}s)")
    except LSTStop:
        pool.shutdown(wait=True, cancel_futures=True)     # in-flight blocks finish and are cached
        raise
    except BaseException:
        pool.shutdown(wait=False, cancel_futures=True)
        raise
    pool.shutdown(wait=True)
    call["seconds"] = round(time.perf_counter() - t0, 1)
    st = _S05_GDDP_STATS
    st["requests"] += call["requests"]
    st["cache_hits"] += call["cache_hits"]
    st["seconds"] = round(st["seconds"] + call["seconds"], 1)
    st["failed_items"] += call["failed_items"]
    return call


def _s05_window_native(scenario: str, model: str, window: tuple, band_filter: bool) -> Optional[dict]:
    """Per-pixel n-weighted mean of the blocks of ``window`` on the request grid ({tasmax, tasmin, n}); None when a
    block is missing (failed item)."""
    datas = []
    for b0, b1 in _s05_blocks(*window):
        data = _S05_GDDP_MEMO.get(_s05_gddp_key(scenario, model, b0, b1, band_filter))
        if data is None:
            return None
        datas.append(data)
    n_tot = np.zeros_like(datas[0]["n"], dtype=np.float64)
    out = {}
    for var in ("tasmax", "tasmin"):
        num = np.zeros_like(n_tot)
        den = np.zeros_like(n_tot)
        for d in datas:
            v = np.asarray(d[var], np.float64)
            w = np.where(np.isfinite(v), np.asarray(d["n"], np.float64), 0.0)
            num += w * np.nan_to_num(v, nan=0.0)
            den += w
        with np.errstate(invalid="ignore", divide="ignore"):
            out[var] = np.where(den > 0, num / np.where(den > 0, den, 1.0), np.nan)
    for d in datas:
        n_tot += np.asarray(d["n"], np.float64)
    out["n"] = n_tot
    return out


def _s05_grid_points() -> tuple:
    """(lat, lon) of the GRID cell centres, flattened (memoised)."""
    memo = globals().get("_S05_GRID_LATLON")
    if memo is None:
        lon, lat = GRID.lonlat()
        memo = (np.asarray(lat, np.float64).ravel(), np.asarray(lon, np.float64).ravel())
        globals()["_S05_GRID_LATLON"] = memo
    return memo


def _s05_to_grid(field: np.ndarray, lattice: dict) -> np.ndarray:
    """Bilinear interpolation of a request-grid field to the GRID cell centres (NaN-safe normalised weights)."""
    from scipy.interpolate import RegularGridInterpolator

    sx, _, x0, _, sy, y0 = lattice["transform"]
    h, w = field.shape
    lons = x0 + (np.arange(w) + 0.5) * sx
    lats = y0 + (np.arange(h) + 0.5) * sy
    order = np.argsort(lats)                     # ascending axes for the interpolator
    lats, vals = lats[order], np.asarray(field, np.float64)[order]
    valid = np.isfinite(vals)
    pts = np.column_stack(_s05_grid_points())
    num = RegularGridInterpolator((lats, lons), np.where(valid, vals, 0.0), method="linear", bounds_error=False,
                                  fill_value=np.nan)(pts)
    den = RegularGridInterpolator((lats, lons), valid.astype(np.float64), method="linear", bounds_error=False,
                                  fill_value=np.nan)(pts)
    with np.errstate(invalid="ignore", divide="ignore"):
        out = np.where(den > 1e-9, num / np.where(den > 1e-9, den, 1.0), np.nan)
    return out.reshape(GRID.shape)


def _s05_centre_index(lattice: dict) -> tuple:
    """(row, col) of the request-grid pixel containing the ROI centre."""
    sx, _, x0, _, sy, y0 = lattice["transform"]
    lon_c, lat_c = float(np.mean(CELL_LON)), float(np.mean(CELL_LAT))
    col = int(np.clip(math.floor((lon_c - x0) / sx), 0, lattice["width"] - 1))
    row = int(np.clip(math.floor((lat_c - y0) / sy), 0, lattice["height"] - 1))
    return row, col


def _s05_cmip6_table(frames: Sequence[int], models: Sequence[str], band_filter: bool = True) -> tuple:
    """GDDP-CMIP6 ROI-mean Apr-Jun tasmax/tasmin per (scenario, model, frame) -> DataFrame, diagnostics (D7).

    Every value comes from cached per-(scenario, model, 5-year block) means: the window field is the per-pixel
    n-weighted mean of its blocks, bilinear to the grid cell centres, and ``value_k`` its mean over ``ROI_MASK``.
    """
    scenarios = ("ssp245", "ssp585")
    windows = {int(f): _s05_window_for(f) for f in frames}
    items = [(s, m, b0, b1) for f in frames for s in scenarios for m in models for b0, b1 in _s05_blocks(*windows[f])]
    with timer(f"05 GDDP-CMIP6 blocks for windows {sorted(set(windows.values()))} ({len(models)} models)"):
        call = _s05_gddp_fetch_blocks(items, band_filter, "GDDP table")
    lattice = _s05_gddp_lattice()
    rc = _s05_centre_index(lattice)
    roi = np.asarray(ROI_MASK, bool)
    season_days = _s04_season_days()
    rows, counts = [], []
    for frame in frames:
        y0, y1 = windows[int(frame)]
        expected = season_days * (y1 - y0 + 1)
        for s in scenarios:
            for m in models:
                nat = _s05_window_native(s, m, (y0, y1), band_filter)
                n = int(round(float(nat["n"][rc]))) if nat is not None else 0
                counts.append({"scenario": s, "model": m, "frame": int(frame), "n": n})
                if nat is None or n == 0:
                    continue
                if not 0.9 * expected <= n <= 1.05 * expected:
                    log(f"GDDP {m} {s} {y0}-{y1}: {n} daily values at the ROI centre (expected ~{expected}); "
                        "possible duplicated/missing days", "WARNING")
                vals = {}
                for var in ("tasmax", "tasmin"):
                    field = _s05_to_grid(nat[var], lattice)
                    vals[var] = float(np.nanmean(field[roi])) if np.isfinite(field[roi]).any() else float("nan")
                if not all(np.isfinite(v) for v in vals.values()):
                    continue
                for var in ("tasmax", "tasmin"):
                    rows.append((s, m, int(frame), var, vals[var]))
    df = pd.DataFrame(rows, columns=["scenario", "model", "frame", "var", "value_k"])
    # keep only models complete in every frame and both variables of a scenario (deltas must be defined)
    need = len(frames) * 2
    complete = df.groupby(["scenario", "model"])["value_k"].transform("size") == need
    dropped = sorted(set(map(tuple, df.loc[~complete, ["scenario", "model"]].drop_duplicates().values)))
    missing = sorted({(s, m) for s in scenarios for m in models}
                     - set(map(tuple, df[["scenario", "model"]].drop_duplicates().values)))
    if dropped or missing:
        log(f"GDDP-CMIP6: dropping incomplete model-scenarios {dropped}; without data: {missing}", "WARNING")
    df = df[complete].sort_values(["scenario", "model", "var", "frame"], kind="stable").reset_index(drop=True)
    diag = {"image_counts": counts, "dropped": [list(d) for d in dropped], "no_data": [list(d) for d in missing],
            "band_filter": bool(band_filter), "requests": call["requests"], "cache_hits": call["cache_hits"],
            "seconds": call["seconds"], "failed_items": call["failed_items"], "blocks": call["items"],
            "request_grid": {k: lattice[k] for k in ("transform", "width", "height", "how")}}
    if df.empty:
        raise RuntimeError("GDDP-CMIP6 returned no complete model over the ROI")
    plausible = df["value_k"].between(270.0, 330.0)
    if not plausible.all():
        raise RuntimeError(f"GDDP-CMIP6 ROI means outside 270-330 K: {df[~plausible].head().to_dict('records')}")
    return df, diag


def _s05_cmip6_pattern(models: Sequence[str], target: int, band_filter: bool = True) -> tuple:
    """Ensemble-median ssp245 delta field (W(target) - W(NOW)) on GRID, divided by its ROI mean, for tasmax/tasmin.

    Uses only the cached blocks of the two windows (shared with the CMIP6 table; no other request). Returns
    ``(pattern {var: (H, W) float32}, info {var: {roi_mean_delta_k, min, max}, n_models, requests, cache_hits})``.
    """
    w_now, w_tgt = _s05_window_for(CFG.NOW), _s05_window_for(target)
    items = [("ssp245", m, b0, b1) for m in models for w in (w_now, w_tgt) for b0, b1 in _s05_blocks(*w)]
    with timer(f"05 GDDP-CMIP6 ssp245 pattern {w_tgt[0]}-{w_tgt[1]} minus {w_now[0]}-{w_now[1]}"):
        call = _s05_gddp_fetch_blocks(items, band_filter, f"GDDP pattern {target}")
    lattice = _s05_gddp_lattice()
    deltas = {"tasmax": [], "tasmin": []}
    used = []
    for m in models:
        a = _s05_window_native("ssp245", m, w_now, band_filter)
        b = _s05_window_native("ssp245", m, w_tgt, band_filter)
        if a is None or b is None:
            continue
        used.append(m)
        for var in deltas:
            deltas[var].append(_s05_to_grid(b[var] - a[var], lattice))
    if not used:
        raise RuntimeError(f"no ssp245 model has the blocks of both windows {w_now} and {w_tgt}")
    out, info = {}, {"n_models": len(used), "models": used, "requests": call["requests"],
                     "cache_hits": call["cache_hits"], "seconds": call["seconds"],
                     "windows": [list(w_now), list(w_tgt)]}
    for var in ("tasmax", "tasmin"):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)          # all-NaN cells outside the ROI
            median = np.nanmedian(np.stack(deltas[var]), axis=0).astype(np.float32)
        field = _s04_fill_roi_nearest(median, ROI_MASK, f"cmip6_pattern {var}")
        mean = float(np.nanmean(field[ROI_MASK]))
        if not np.isfinite(mean) or mean <= 0.05:
            log(f"cmip6_pattern {var}: ROI-mean delta {mean:.3f} K is too small to normalise; using 1", "WARNING")
            field = np.where(ROI_MASK, 1.0, np.nan).astype(np.float32)
        else:
            field = (field / mean).astype(np.float32)
        out[var] = field
        info[var] = {"roi_mean_delta_k": round(mean, 4), "min": round(float(np.nanmin(field)), 4),
                     "max": round(float(np.nanmax(field)), 4)}
        log(f"  cmip6_pattern {var}: ensemble-median ({len(used)} models) ROI-mean delta {mean:.2f} K, normalised "
            f"range {info[var]['min']:.3f}-{info[var]['max']:.3f}")
    return out, info


def _s05_chen_resolve(bands: Sequence[str]) -> dict:
    """{scenario: band} of the Chen-2020 SSP urban layers (deterministic; raises when ambiguous).

    A band name normalised to lower-case alphanumerics that equals ``ssp1`` / ``ssp2`` / ``ssp5`` maps to ssp126 /
    ssp245 / ssp585. Otherwise, only for exactly 5 bands, the bands are taken positionally as SSP1..SSP5 (1st, 2nd,
    5th) with a WARNING. Any other layout, or three resolved bands that are not distinct, raises RuntimeError.
    """
    bands = [str(b) for b in bands]
    norm = {b: re.sub(r"[^a-z0-9]", "", b.lower()) for b in bands}
    resolved = {}
    for scenario, want in (("ssp126", "ssp1"), ("ssp245", "ssp2"), ("ssp585", "ssp5")):
        match = [b for b in bands if norm[b] == want]
        if len(match) == 1:
            resolved[scenario] = match[0]
    if len(resolved) < 3:
        if len(bands) != 5:
            raise RuntimeError(f"Chen-2020 band layout {bands} cannot be mapped to SSP1/SSP2/SSP5 (no ssp1/ssp2/ssp5 "
                               "names and not exactly 5 bands)")
        log(f"Chen-2020 bands {bands} carry no SSP names: assuming SSP1..SSP5 band order (ssp126 = {bands[0]}, "
            f"ssp245 = {bands[1]}, ssp585 = {bands[4]})", "WARNING")
        resolved = {"ssp126": bands[0], "ssp245": bands[1], "ssp585": bands[4]}
    if len(set(resolved.values())) != 3:
        raise RuntimeError(f"Chen-2020 bands resolve to non-distinct layers {resolved} (bands {bands})")
    return resolved


def _s05_chen_decade(index: str, t0: Any, t1: Any) -> Optional[int]:
    """Nominal year of a Chen-2020 layer: the 4-digit year ending its ``system:index`` ("global_SSP_2020"), else the
    midpoint of its [time_start, time_end] window rounded to a decade. Never time_start alone: the 2020 layer is
    stamped 2015-01-01 (start of its 2015-2024 window)."""
    m = re.search(r"(\d{4})\D*$", str(index))
    if m:
        return int(m.group(1))
    try:
        a, b = pd.Timestamp(int(t0), unit="ms"), pd.Timestamp(int(t1), unit="ms")
        mid = a + (b - a) / 2
        return int(10 * round(mid.year / 10))
    except (TypeError, ValueError):
        return None


def _s05_chen_bands() -> tuple:
    """Resolve the Chen-2020 band per scenario and the layers of each decade (one batched request).

    Returns ``(resolved, decades, layers)`` with ``layers = {decade: [system:index, ...]}``.
    """
    col = ee.ImageCollection(_S05_CHEN)
    info = _s04_getinfo(ee.Dictionary({"bands": ee.Image(col.first()).bandNames(),
                                       "ids": col.aggregate_array("system:index"),
                                       "t0": col.aggregate_array("system:time_start"),
                                       "t1": col.aggregate_array("system:time_end")}), "Chen-2020 layout")
    bands = list(info.get("bands") or [])
    ids = [str(i) for i in (info.get("ids") or [])]
    t0, t1 = list(info.get("t0") or []), list(info.get("t1") or [])
    aligned = len(t0) == len(ids) and len(t1) == len(ids)       # aggregate_array skips missing values
    layers: dict = {}
    for k, idx in enumerate(ids):
        decade = _s05_chen_decade(idx, t0[k] if aligned else None, t1[k] if aligned else None)
        if decade is not None:
            layers.setdefault(decade, []).append(idx)
    years = sorted(layers)
    resolved = _s05_chen_resolve(bands)
    if 2020 not in years:
        raise RuntimeError(f"Chen-2020 collection lacks the 2020 layer (decades {years}, layers {layers})")
    log(f"Chen-2020 layout: bands {bands}, decades {years}; using {resolved}")
    return resolved, years, layers


def _s05_geo_interp(dec: dict, need_decades: Sequence[int], f: int) -> float:
    """Geometric interpolation of decadal ratios at frame ``f``."""
    if f in dec:
        return float(dec[f])
    lo = max(d for d in need_decades if d <= f)
    hi = min(d for d in need_decades if d >= f)
    w = (f - lo) / (hi - lo)
    return float(dec[lo] ** (1.0 - w) * dec[hi] ** w)


def _s05_urban_growth(frames: Sequence[int], chen: Optional[tuple] = None) -> tuple:
    """DataFrame [scenario, district, frame, ratio] of Chen-2020 urban area vs 2020 per district."""
    bands, years, layers = chen if chen is not None else _s05_chen_bands()
    need_decades = sorted({2020} | {int(10 * _s05_math.floor(f / 10)) for f in frames}
                          | {int(10 * _s05_math.ceil(f / 10)) for f in frames})
    missing = [d for d in need_decades if d not in years]
    if missing:
        raise RuntimeError(f"Chen-2020 lacks decades {missing} needed for frames {list(frames)}")
    col = ee.ImageCollection(_S05_CHEN)
    items = [(s, d) for s in bands for d in need_decades]

    def _build(sub):
        imgs = []
        for s, d in sub:
            # the decade's layers by name, mosaicked (one global image per decade, so no filterBounds is needed;
            # filterBounds(EE_ROI) selected nothing because EE_ROI evaluates to an empty geometry)
            layer = ee.Image(col.filter(ee.Filter.inList("system:index", layers[d])).mosaic()).select(bands[s])
            imgs.append(layer.eq(_S05_CHEN_URBAN).multiply(ee.Image.pixelArea()).divide(1e6).rename(f"{s}_{d}"))
        names = [f"{s}_{d}" for s, d in sub]    # forEach names the outputs: one band would be called "sum"
        fc = ee.Image.cat(imgs).reduceRegions(collection=EE_DISTRICTS, reducer=ee.Reducer.sum().forEach(names),
                                              scale=1000, crs=CFG.CRS, tileScale=4)
        return fc.select(["did"] + names, None, False)

    def _parse(info, sub):
        out = []
        for props in _s04_fc_records(info):
            for s, d in sub:
                value = props.get(f"{s}_{d}")
                if props.get("did") is not None and value is not None:
                    out.append((s, int(props["did"]), int(d), float(value)))
        return out

    with timer("05 Chen-2020 urban area per district"):
        recs = _s04_batched(items, _build, _parse, "Chen-2020 urban area")
    area = {(s, did, d): v for s, did, d, v in recs}
    dids = sorted({did for _, did, _ in area})
    rows, caveats, zero, last_ratio = [], [], [], {}
    last = int(max(frames))
    for s in bands:
        ncr = {d: sum(area.get((s, did, d), 0.0) for did in dids) for d in need_decades}
        if not ncr[2020] > 0:
            raise RuntimeError(f"Chen-2020 band {bands[s]} ({s}) has no urban land (value {_S05_CHEN_URBAN}) in the "
                               "NCR in 2020; the band layout or the urban code is wrong")
        ncr_dec = {d: max(ncr[d], 1e-9) / ncr[2020] for d in need_decades}
        last_ratio[s] = _s05_geo_interp(ncr_dec, need_decades, last)
        for did in dids:
            base = area.get((s, did, 2020), 0.0)
            if base > 0.5:            # km2: at least ~half a 1 km cell of urban land in 2020
                dec = {d: max(area.get((s, did, d), base), 1e-9) / base for d in need_decades}
            else:
                zero.append(did)
                dec = ncr_dec
            for f in frames:
                rows.append((s, int(did), int(f), round(_s05_geo_interp(dec, need_decades, int(f)), 5)))
    if len(last_ratio) > 1 and max(last_ratio.values()) - min(last_ratio.values()) <= 1e-6:
        raise RuntimeError(f"Chen-2020 bands resolve to identical layers: the NCR urban ratio at {last} is "
                           f"{last_ratio} for every scenario (bands {bands})")
    zero = sorted(set(zero))
    if zero:
        names = [DISTRICTS[d]["name"] for d in zero if 0 <= d < len(DISTRICTS)]
        caveats.append(f"Chen-2020 shows (almost) no 2020 urban land in {names}; their urban-growth ratios use the "
                       "NCR-wide ratio.")
    df = pd.DataFrame(rows, columns=["scenario", "district", "frame", "ratio"])
    df = df.sort_values(["scenario", "district", "frame"], kind="stable").reset_index(drop=True)
    return df, caveats


def _s05_ghsl_2030() -> dict:
    """GHSL 2030 built-up fraction and population density on GRID."""
    built, proj = _s05_ghsl("GHS_BUILT_S", 2030, "built_surface")
    frac = built.toFloat().divide(10000.0).rename("built").setDefaultProjection(proj)
    pop = _s05_ghsl_coarse(2030)[1]
    image = ee.Image.cat([_s04_to_grid(frac, _S05_GHSL_SCALE_M, GRID), pop]).unmask(NODATA_FLOAT).toFloat()
    raw = _s04_fetch(image, GRID, ["built", "pop"], "s05_ghsl_2030", heavy=True)
    return {"built": np.clip(_s04_fill_roi_nearest(raw[0], ROI_MASK, "ghsl_2030 built"), 0.0, 1.0),
            "pop": np.clip(_s04_fill_roi_nearest(raw[1], ROI_MASK, "ghsl_2030 pop"), 0.0, None)}


def _s05_run_probes(pattern_models: Sequence[str], band_filter: bool, caveats: list) -> dict:
    """Pilot probes (D2): the ssp245 CMIP6 warming pattern for 2050 through the block cache; never raises."""
    t0 = time.perf_counter()
    before = (int(_S05_GDDP_STATS["requests"]), int(_S05_GDDP_STATS["cache_hits"]))
    rec = {"seconds": None, "requests": 0, "cache_hits": 0, "roi_mean_delta_k": None, "error": None,
           "window": list(_projection_window(2050)), "n_models": 0}
    try:
        _, info = _s05_cmip6_pattern(pattern_models, 2050, band_filter=band_filter)
        rec["roi_mean_delta_k"] = {v: info[v]["roi_mean_delta_k"] for v in ("tasmax", "tasmin")}
        rec["n_models"] = int(info["n_models"])
    except Exception as exc:  # noqa: BLE001 - a probe failure is reported, never fatal (LSTStop still propagates)
        rec["error"] = f"{type(exc).__name__}: {str(exc)[:300]}"
        log(f"PILOT PROBE cmip6_pattern_2050 FAILED: {rec['error']} - the full run would fail the same way in "
            "section 05; fix this before the full run", "WARNING")
        caveats.append(f"Pilot probe of the CMIP6 2050 warming pattern failed ({rec['error'][:160]}).")
    rec["seconds"] = round(time.perf_counter() - t0, 1)
    rec["requests"] = int(_S05_GDDP_STATS["requests"]) - before[0]
    rec["cache_hits"] = int(_S05_GDDP_STATS["cache_hits"]) - before[1]
    return {"cmip6_pattern_2050": rec}


def _s05_build_future() -> dict:
    """FUTURE_INPUTS (the second stage-cached unit of section 05)."""
    frames = [int(CFG.NOW)] + [int(y) for y in CFG.FRAMES_PROJ]
    caveats, diag = [], {}
    _S05_GDDP_MEMO.clear()
    _S05_GDDP_STATS.update(requests=0, cache_hits=0, seconds=0.0, failed_items=[])
    # cheap, fail-fast inputs first: a wrong Chen layout or a GHSL problem must not cost the GDDP requests
    chen = _s05_chen_bands()
    ghsl_2030 = _s05_ghsl_2030()
    models = _s05_gddp_models()
    try:
        cmip6, diag["cmip6"] = _s05_cmip6_table(frames, models)
    except RuntimeError as exc:
        if "no complete model" not in str(exc):
            raise
        # 'system:band_names' filtering removed every image: retry without it (failing models are isolated)
        log(f"GDDP-CMIP6 with the band-name filter returned nothing ({exc}); retrying without it", "WARNING")
        cmip6, diag["cmip6"] = _s05_cmip6_table(frames, models, band_filter=False)
    band_filter = diag["cmip6"]["band_filter"]
    for s, grp in cmip6.groupby("scenario"):
        if len(frames) > 1:
            d = grp.pivot_table(index="model", columns=["var", "frame"], values="value_k")
            last = frames[-1]
            dt = (d[("tasmax", last)] - d[("tasmax", frames[0])])
            log(f"  CMIP6 {s}: {grp['model'].nunique()} models, tasmax delta {frames[0]}->{last} median "
                f"{dt.median():.2f} K (P10 {dt.quantile(0.1):.2f}, P90 {dt.quantile(0.9):.2f})")
    pattern_models = sorted(cmip6.loc[cmip6["scenario"] == "ssp245", "model"].unique())
    if CFG.FRAMES_PROJ:
        target = 2050 if 2050 in CFG.FRAMES_PROJ else int(max(CFG.FRAMES_PROJ))
        pattern, diag["cmip6_pattern"] = _s05_cmip6_pattern(pattern_models, target, band_filter=band_filter)
        if target != 2050:
            caveats.append(f"The CMIP6 warming pattern uses the {target} window (2050 is not a projection frame).")
    else:
        pattern = {v: np.where(ROI_MASK, 1.0, np.nan).astype(np.float32) for v in ("tasmax", "tasmin")}
        diag["cmip6_pattern"] = "uniform (no projection frames in this profile)"
        log("no projection frames: cmip6_pattern set to 1 inside the ROI")
        if _S05_PILOT_PROBES:
            diag["probes"] = _s05_run_probes(pattern_models, band_filter, caveats)
    diag["gddp"] = {k: (list(v) if isinstance(v, list) else v) for k, v in _S05_GDDP_STATS.items()}
    urban, urban_cav = _s05_urban_growth(sorted({2020} | set(frames)), chen=chen)
    caveats += urban_cav
    pop_growth, pop_cav = _s05_pop_growth(frames)
    caveats += pop_cav
    gsat = _s05_gsat_increment(frames)
    log("  pop_growth factors at " + str(frames[-1]) + ": " + ", ".join(
        f"{s} {pop_growth[(pop_growth.scenario == s) & (pop_growth.frame == frames[-1])].factor.iloc[0]:.3f}"
        for s in _S05_SSP_OF))
    log("  gsat_increment at " + str(frames[-1]) + ": " + ", ".join(f"{s} {gsat[s][frames[-1]]:.3f} K" for s in gsat))
    caveats.append("SSP population growth is the IIASA SSP v3 national urban-population trajectory of India (no "
                   "NCR-specific SSP projection exists in the SSP database); it scales the NCR population total.")
    caveats.append("Chen et al. (2020) SSP urban land is binary at 1 km; it enters only as district growth ratios "
                   "relative to its own 2020 layer.")
    future = {"cmip6": cmip6, "cmip6_pattern": pattern, "urban_growth": urban, "ghsl_2030": ghsl_2030,
              "pop_growth": pop_growth, "gsat_increment": gsat}
    w = {f: tuple(int(v) for v in _s05_gcm_window(f)) for f in frames}
    lattice = _s05_gddp_lattice()
    grid_txt = (f"requested per (scenario, model, {_S05_GDDP_BLOCK_YEARS}-year block) on the native "
                f"{lattice['transform'][0]:g} deg grid ({lattice['width']}x{lattice['height']} px over the ROI) and "
                "cached; window field = day-count-weighted mean of its blocks, bilinear to the "
                f"{GRID.res} m cell centres")
    sources = {
        "cmip6": (f"{_S05_GDDP} (Earth Engine; historical <= 2014 + ssp245/ssp585), {cmip6['model'].nunique()} of "
                  f"{len(models)} models complete with tasmax/tasmin (GFDL-CM4 grid gr1 only), Apr-Jun mean of daily "
                  f"values over the 20-year windows {w}, {grid_txt}; value = ROI mean of the window field"),
        "cmip6_pattern": (f"{_S05_GDDP} ssp245 ensemble-median delta field of the 2050 window minus the {CFG.NOW} "
                          f"window from the same cached blocks ({grid_txt}), divided by its ROI mean"
                          if CFG.FRAMES_PROJ else "uniform 1 (no projection frames in this profile)"),
        "urban_growth": (f"{_S05_CHEN} (Chen et al. 2020, value {_S05_CHEN_URBAN} = urban; bands "
                         f"{dict(chen[0])}, decadal tiles mosaicked over the ROI) urban area per district / 2020, "
                         "decades interpolated geometrically"),
        "ghsl_2030": f"{_S05_GHSL}/GHS_BUILT_S/2030 built_surface/1e4 and GHS_POP/2030 x100, mean to {GRID.res} m",
        "pop_growth": _S05_SSP_CITATION + "; factor = urban population(frame) / urban population(2025)",
        "gsat_increment": _S05_AR6_CITATION,
    }
    return {"future": future, "sources": sources, "caveats": caveats, "diag": diag}

## 5.4 Run (gee mode), publish and check the acquisition contract (SPEC §3.2)
Two stage-cache units (drivers; future inputs) so a failure in one does not recompute the other. The contract
check is the gee-mode twin of section 02's: every `RAW_OBS` key of every frame (including the Landsat products of
section 03 and the thermal products of section 04), shapes, dtypes, NaN outside the ROI, glc fractions summing to
1, `LC_FINE` shapes, `FRAME_META` structure, `NIGHT_CALIBRATION`, `ANNUAL_SERIES` and `FUTURE_INPUTS` columns.

In [ ]:
def _s05_publish(drivers: dict, future: dict) -> None:
    global RAW_STATIC, RAW_OBS, LC_FINE, FUTURE_INPUTS, FUTURE_INPUTS_DIAG, DATA_SOURCES, DATA_CAVEATS
    if not isinstance(globals().get("RAW_OBS"), dict):
        RAW_OBS = {}
    if not isinstance(globals().get("RAW_STATIC"), dict):
        RAW_STATIC = {}
    RAW_STATIC.update(drivers["static"])     # section 03 created the (empty) dict; 05 owns its keys
    for year, fields in drivers["obs"].items():
        RAW_OBS.setdefault(int(year), {}).update(fields)
    LC_FINE = {int(y): v for y, v in drivers["lc_fine"].items()}
    FUTURE_INPUTS = future["future"]
    FUTURE_INPUTS_DIAG = future["diag"]
    if not isinstance(globals().get("DATA_SOURCES"), dict):
        DATA_SOURCES = {}
    for var, by_year in drivers["sources"].items():
        DATA_SOURCES.setdefault(var, {}).update({int(y): t for y, t in by_year.items()})
    span = f"{CFG.NOW}-{max([CFG.NOW] + list(CFG.FRAMES_PROJ))}"
    DATA_SOURCES.setdefault("future_inputs", {})[span] = "; ".join(f"{k}: {v}" for k, v in future["sources"].items())
    for key, text in future["sources"].items():
        DATA_SOURCES.setdefault(f"future_{key}", {})[span] = text
    if not isinstance(globals().get("DATA_CAVEATS"), list):
        DATA_CAVEATS = []
    for text in drivers["caveats"] + future["caveats"]:
        if text not in DATA_CAVEATS:
            DATA_CAVEATS.append(text)


def _s05_validate() -> None:
    """Raise AssertionError with a precise message if a §3.2 global violates the contract (gee mode)."""
    h, w = GRID.shape
    hf, wf = GRID_FINE.shape
    roi, roi_f = np.asarray(ROI_MASK, bool), np.asarray(ROI_MASK_FINE, bool)
    assert set(RAW_STATIC) == set(_S05_STATIC_KEYS) | set(_S05_STATIC_FINE_KEYS), sorted(RAW_STATIC)
    for key, arr in RAW_STATIC.items():
        fine = key.endswith("_f")
        mask = roi_f if fine else roi
        assert arr.shape == ((hf, wf) if fine else (h, w)) and arr.dtype == np.float32, (key, arr.shape, arr.dtype)
        assert np.isnan(arr[~mask]).all() and np.isfinite(arr[mask]).all(), f"RAW_STATIC[{key}] NaN pattern"
    frames = [int(y) for y in CFG.FRAMES_OBS]
    assert sorted(RAW_OBS) == sorted(frames), f"RAW_OBS frames {sorted(RAW_OBS)} != {sorted(frames)}"
    lc_res = int(CFG.fine_res())
    lc_shape = (h * GRID.res // lc_res, w * GRID.res // lc_res)
    expected = set(_S05_RAW_FINE_KEYS) | set(_S05_RAW_COARSE_KEYS)
    for year in frames:
        obs = RAW_OBS[year]
        missing, extra = sorted(expected - set(obs)), sorted(set(obs) - expected)
        assert not missing, f"RAW_OBS[{year}] lacks {missing} (section 03 day/spectral keys or section 04/05 keys)"
        assert not extra, f"RAW_OBS[{year}] has unexpected keys {extra}"
        for key, arr in obs.items():
            fine = key in _S05_RAW_FINE_KEYS
            assert arr.shape == ((hf, wf) if fine else (h, w)) and arr.dtype == np.float32, (
                year, key, arr.shape, arr.dtype)
            assert np.isnan(arr[~(roi_f if fine else roi)]).all(), f"RAW_OBS[{year}][{key}] must be NaN outside ROI"
        glc = np.stack([obs[k][roi] for k in _S05_GLC_KEYS])
        assert np.allclose(glc.sum(0), 1.0, atol=1e-4), f"glc fractions of {year} do not sum to 1"
        assert np.nanmin(obs["built_f"]) >= 0 and np.nanmax(obs["built_f"]) <= 1, f"built_f range {year}"
        for key in _S05_COARSE_KEYS:
            assert np.isfinite(obs[key][roi]).all(), f"RAW_OBS[{year}][{key}] has gaps inside the ROI"
        assert LC_FINE[year].shape == lc_shape and LC_FINE[year].dtype == np.uint8, (year, LC_FINE[year].shape)
        assert int(LC_FINE[year].max()) <= 6
        meta = FRAME_META[year]
        assert {"window", "day", "night"} <= set(meta), f"FRAME_META[{year}] keys {sorted(meta)}"
        assert set(meta["night"]) == {"source", "sensor", "overpass_ist_h", "composites", "coverage"}
        assert meta["night"]["source"] in ("aqua", "terra_cal", "viirs_cal", None)
        has_night = bool(np.isfinite(obs["lst_night"][roi]).any())
        assert has_night == (meta["night"]["source"] is not None), f"night availability of {year}"
    for kind in ("terra_to_aqua", "viirs_to_aqua"):
        cal = NIGHT_CALIBRATION[kind]
        assert set(cal) == {"r2", "rmse", "n", "by_class"} and all(set(c) == {"class", "a", "b", "n"}
                                                                   for c in cal["by_class"]), cal
    assert list(ANNUAL_SERIES.columns) == ["year", "scope", "variable", "value", "flag"], list(ANNUAL_SERIES.columns)
    assert set(ANNUAL_SERIES["variable"]) == set(_S04_ERA5_SERIES + _S04_SAT_SERIES), sorted(
        set(ANNUAL_SERIES["variable"]))
    assert ANNUAL_SERIES["year"].between(1990, 2025).all() and "ncr" in set(ANNUAL_SERIES["scope"])
    assert set(FUTURE_INPUTS) == {"cmip6", "cmip6_pattern", "urban_growth", "ghsl_2030", "pop_growth",
                                  "gsat_increment"}, sorted(FUTURE_INPUTS)
    cm = FUTURE_INPUTS["cmip6"]
    assert list(cm.columns) == ["scenario", "model", "frame", "var", "value_k"], list(cm.columns)
    assert set(cm["scenario"]) <= {"ssp245", "ssp585"} and set(cm["var"]) == {"tasmax", "tasmin"}
    assert set(cm["frame"]) == {int(CFG.NOW)} | {int(y) for y in CFG.FRAMES_PROJ}, sorted(set(cm["frame"]))
    for var in ("tasmax", "tasmin"):
        pat = FUTURE_INPUTS["cmip6_pattern"][var]
        assert pat.shape == (h, w) and pat.dtype == np.float32 and np.isfinite(pat[roi]).all()
        assert abs(float(np.mean(pat[roi])) - 1.0) < 1e-3, f"cmip6_pattern {var} ROI mean != 1"
    ug = FUTURE_INPUTS["urban_growth"]
    assert list(ug.columns) == ["scenario", "district", "frame", "ratio"]
    assert set(ug["scenario"]) == {"ssp126", "ssp245", "ssp585"} and (ug["ratio"] > 0).all()
    assert list(FUTURE_INPUTS["pop_growth"].columns) == ["scenario", "frame", "factor"]
    for var in ("built", "pop"):
        g = FUTURE_INPUTS["ghsl_2030"][var]
        assert g.shape == (h, w) and g.dtype == np.float32 and np.isfinite(g[roi]).all()
    assert set(FUTURE_INPUTS["gsat_increment"]) == {"ssp126", "ssp245", "ssp585"}
    assert all(abs(v[int(CFG.NOW)]) < 1e-9 for v in FUTURE_INPUTS["gsat_increment"].values())


if _S05_ACTIVE:
    if ee is None or EE_DISTRICTS is None:
        raise RuntimeError("section 05 needs an initialised Earth Engine session and EE_DISTRICTS (section 01)")
    # code04: section 05 computes with section-04 helpers (GLC fractions, ROI fill, regridding, fetch); code03: and
    # fetches through section-03 helpers (gee_fetch_grid, gee_fetch_asset, asset exports), D4
    _s05_base_key = dict(version=_S05_VERSION, grid=GRID.to_dict(), roi=_s04_roi_digest(),
                         code03=section_code_hash("03"), code04=section_code_hash("04"))
    _S04_EXPORT_DIAG.clear()
    _s05_sc_drv = stage_cache("s05_drivers", CFG.key("WINDOWS", "SEASON", **_s05_base_key,
                                                     lc_fine_res=int(CFG.fine_res())))
    with gee_workload_tag("lst-v2-s05"), timer("05 drivers"):
        _S05_DRIVERS = _s05_sc_drv.get_or_compute(_s05_build_drivers)
    _s05_sc_fut = stage_cache("s05_future", CFG.key("SEASON", "NOW", "FRAMES_PROJ", **_s05_base_key,
                                                    proj_windows={str(k): list(v) for k, v in
                                                                  CFG.PROJ_WINDOWS.items()},
                                                    probes=bool(_S05_PILOT_PROBES and not CFG.FRAMES_PROJ)))
    with gee_workload_tag("lst-v2-s05"), timer("05 future inputs"):
        _S05_FUTURE = _s05_sc_fut.get_or_compute(_s05_build_future)
    _s05_publish(_S05_DRIVERS, _S05_FUTURE)
    del _S05_DRIVERS, _S05_FUTURE, _s05_sc_drv, _s05_sc_fut, _s05_base_key
    globals().pop("_S04_GLC_MEMO", None)     # GLC fraction cache shared by 04/05 is no longer needed
    free_memory()
    _s05_validate()
    log("section-05 contract check passed (SPEC §3.2: RAW_STATIC, RAW_OBS, LC_FINE, FRAME_META, NIGHT_CALIBRATION, "
        "ANNUAL_SERIES, FUTURE_INPUTS)")
    if _S04_EXPORT_DIAG:
        FUTURE_INPUTS_DIAG["exports"] = list(_S04_EXPORT_DIAG)
    _s05_g = FUTURE_INPUTS_DIAG.get("gddp") or {}
    log(f"GDDP-CMIP6 blocks: {_s05_g.get('requests', 0)} requests, {_s05_g.get('cache_hits', 0)} from cache, "
        f"{len(_s05_g.get('failed_items') or [])} failed, {_s05_g.get('seconds', 0)}s")
    del _s05_g

## 5.5 Quality-assurance figure
Built-up fraction (latest frame, fine grid), dominant GLC class, ERA5-Land Tmax climatology, and the CMIP6 ROI-mean
tasmax change per model (frame vs 2025) for both scenarios.

In [ ]:
def _s05_qa_figure() -> None:
    last = int(max(CFG.FRAMES_OBS))
    obs = RAW_OBS[last]
    fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))
    im = axes[0].imshow(obs["built_f"], cmap="inferno", vmin=0, vmax=1)
    axes[0].set_title(f"built_f {last} ({GRID_FINE.res} m)", fontsize=9)
    fig.colorbar(im, ax=axes[0], shrink=0.7)
    stack = np.stack([np.nan_to_num(obs[k], nan=-1) for k in _S05_GLC_KEYS])
    dom = np.where(ROI_MASK, np.argmax(stack, 0) + 1, 0)
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap([LC_CLASS_COLORS[c] for c in range(7)])
    axes[1].imshow(dom, cmap=cmap, vmin=0, vmax=6, interpolation="nearest")
    axes[1].set_title(f"dominant GLC class {last}", fontsize=9)
    im = axes[2].imshow(obs["era5_tmax"], cmap="coolwarm")
    axes[2].set_title(f"ERA5-Land Apr-Jun Tmax {last} (°C)", fontsize=9)
    fig.colorbar(im, ax=axes[2], shrink=0.7)
    for ax in axes[:3]:
        ax.set_axis_off()
    cm = FUTURE_INPUTS["cmip6"]
    frames = sorted(cm["frame"].unique())
    if len(frames) > 1:
        tx = cm[cm["var"] == "tasmax"].pivot_table(index=["scenario", "model"], columns="frame", values="value_k")
        for s, color in (("ssp245", "tab:blue"), ("ssp585", "tab:red")):
            if s not in tx.index.get_level_values(0):
                continue
            d = tx.loc[s].sub(tx.loc[s][frames[0]], axis=0)
            axes[3].plot(frames, d.T.values, color=color, alpha=0.15, lw=0.8)
            axes[3].plot(frames, d.median().values, color=color, lw=2.2, label=f"{s} median")
        axes[3].set_xlabel("frame (20-year window centre)")
        axes[3].set_ylabel(f"Apr-Jun tasmax change vs {frames[0]} (K)")
        axes[3].legend(fontsize=8)
    axes[3].set_title("GDDP-CMIP6 ROI-mean (one line per model)", fontsize=9)
    fig.tight_layout()
    path = CFG.FIG_DIR / "05_drivers_future_qa.png"
    try:
        fig.savefig(path, dpi=110, bbox_inches="tight")
        log(f"figure -> {path}")
    except OSError as exc:
        log(f"could not save {path}: {exc}", "WARNING")
    plt.show()
    plt.close(fig)


if _S05_ACTIVE:
    try:
        _s05_qa_figure()
    except Exception as exc:  # noqa: BLE001 - a QA figure must never abort the pipeline
        log(f"section-05 QA figure failed: {type(exc).__name__}: {exc}", "WARNING")

## 5.6 Pilot probes
With `LST_PILOT_PROBES` (default on for a gee pilot) the CMIP6 2050 warming pattern was computed through the block
cache although the pilot has no projection frames; the full run reuses the fetched blocks.

In [ ]:
if _S05_ACTIVE and isinstance(globals().get("FUTURE_INPUTS_DIAG"), dict) and FUTURE_INPUTS_DIAG.get("probes"):
    _s05_rows = []
    for _s05_name, _s05_p in FUTURE_INPUTS_DIAG["probes"].items():
        _s05_d = _s05_p.get("roi_mean_delta_k") or {}
        _s05_rows.append({"probe": _s05_name, "ok": _s05_p.get("error") is None, "seconds": _s05_p.get("seconds"),
                          "requests": _s05_p.get("requests"), "cache_hits": _s05_p.get("cache_hits"),
                          "models": _s05_p.get("n_models"), "tasmax_delta_k": _s05_d.get("tasmax"),
                          "tasmin_delta_k": _s05_d.get("tasmin"), "error": (_s05_p.get("error") or "")[:80]})
    print("PILOT PROBES (ssp245 ensemble-median ROI warming of the 2050 window vs the 2025 window)")
    _show(pd.DataFrame(_s05_rows).set_index("probe"))
    del _s05_rows, _s05_name, _s05_p, _s05_d

# 6. Features, targets, masks and cross-validation folds

One row of the modelling table `DF` is one **ROI cell × observed frame** (all ROI cells, also those without a
valid target; the target columns are NaN and the sample weight 0 there). The 47 predictors are fixed in number
and **order** (SPEC §4.1): the order of `FEATURES` is the column order of every model, SHAP matrix and web file.

| Family | Base features (33) | Physical role in pre-monsoon LST |
|---|---|---|
| spectral | `ndvi ndbi mndwi albedo ndvi_spring ndvi_drop` | transpiration/shading (NDVI), built/bare signal (NDBI), surface wetness (MNDWI, green/SWIR1), absorbed shortwave (albedo); `ndvi_drop = ndvi_spring − ndvi` marks rabi wheat harvested before April (bare, hot) versus irrigated/sugarcane fields that stay green |
| landcover | `frac_built frac_forest frac_water frac_cropland frac_barren` | composition of the 1 km cell (GHSL built-up + GLC_FCS30D classes on a common simplex; *other vegetation* is the implicit reference) |
| urban | `built_v log_pop dist_urban dist_builtup lm_pd lm_ed lm_contag` | building volume (heat storage, night UHI), population (anthropogenic heat), proximity to dense urban cores and to any built-up land, landscape configuration (fragmentation, edges, aggregation) |
| water | `dist_water` | advective/evaporative cooling reach of rivers, canals and jheels |
| terrain | `elevation slope aspect_sin aspect_cos elev_tpi7 elev_tpi21 elev_std7` | lapse rate, illumination, cold-air pooling (topographic position), ridge relief |
| climate | `era5_t2m era5_vpd era5_swvl1 era5_evap era5_ssr era5_wind era5_precip_jfm` | the regional gradient (ERA5-Land, Apr–Jun of the frame window) that v1 lacked; replaces raw coordinates (SPEC §1.4) |

**Focal (14):** NaN-aware box means at 3 and 15 cells of `ndvi ndbi albedo frac_built frac_forest frac_water
frac_barren` (`<b>_m3`, `<b>_m15`). The real-data experiments (`docs/research/real_data_accuracy_experiments.md` §b)
showed that v1's weak spatial CV came from missing neighbourhood context — a park cools its surroundings, a city
warms its fringe: 3 and 15 km window means, together with distance and relative-elevation features, raised 5×5
block CV R² from 0.63 → 0.83 (the windows alone ≈ 0.79). In v2 the distance and TPI features are part of the 33
base features, so the base-only ablation of section 07 isolates the 14 focal means only. The windows are
defined in **cells**, so at the 2 km smoke-test grid they span 6 and 30 km (documented, not rescaled).

**Definitions** (all on the 1 km analysis grid `GRID`, cells outside the ROI are NaN):
* spectral / built-up 1 km values = mean of the valid fine (250 m) pixels when ≥ 50 % of the 16 are valid;
* `frac_built` = 1 km mean of GHSL `built_f`; the other four fractions put the GLC classes on the GHSL simplex:
  $f_k = g_k / (1 - g_{built}) \cdot (1 - f_{built})$ (0 when $g_{built} \ge 0.999$), so
  $f_{built} + \sum_k f_k + f_{otherveg} = 1$ and one land-cover *definition* of "built" is used everywhere;
* `built_v` = log1p(m³ m⁻²), `log_pop` = log1p(persons km⁻²);
* focal box mean $\bar x_w = \mathrm{U}_w(x\,m) / \mathrm{U}_w(m)$ with $\mathrm{U}_w$ the `uniform_filter` of
  width $w$ and $m$ the valid-cell indicator (ROI ∧ finite), zero padding beyond the grid edge; |values| < 10⁻¹²
  (running-sum residues of all-zero windows) are set to exactly 0, so "no water within 15 cells" is exactly 0;
* `elev_tpi7 = elevation − box7(elevation)`, `elev_tpi21` likewise, `elev_std7` = NaN-aware box standard deviation;
* distances (km) by the Euclidean distance transform × cell size: `dist_water` to cells with
  `frac_water ≥ 0.3 or water_occ ≥ 0.5`, `dist_urban` to cells whose box7 mean `frac_built ≥ 0.3` (urban core; 0.5 matched no real cell), `dist_builtup` to
  `frac_built ≥ 0.3` (cells outside the ROI are never targets);
* landscape metrics PD/ED/CONTAG from `LC_FINE` exactly as v1 (vectorised, validated against `pylandstats`).

**Targets** (SPEC §3.3). *Day*: the 1 km mean of the harmonised Landsat fine composite (≥ 50 % valid fine pixels)
with a 1 km mean clear-scene count ≥ 2, weight `w_day = clip(nobs/5, 0.4, 1)`, all observed frames train.
*Night*: Aqua-equivalent 1 km LST with ≥ 3 valid 8-day composites (`NIGHT_MIN_NOBS`; real Apr–Jun data give a
median of ~5 per cell over a 3-year window; the calibrated Terra frame ≥ 1, it has weight 0) (VIIRS: ≥ 20 daily
observations), weight
`w_night = clip(nobs/12, 0.5, 1)`; only `CFG.NIGHT_TRAIN_FRAMES` (2005–2025) train — the calibrated Terra 2000 frame
keeps its observed anomaly (weight 0) as the hindcast check, 1990/1995 have no night product.

**Fixed masks and anomalies.** Per mode, the *mask frames* are the training frames whose valid-target share of the
ROI cells is ≥ `LST_FRAME_MIN_COVERAGE` (default 0.5). `FIXED_MASK[mode]` = ROI cells with a valid target in
**every** mask frame (strict intersection; a share < 80 % of the ROI logs a warning and a `DATA_CAVEATS` entry with
each frame's missing share; fewer than two mask frames, or a mask keeping < 1 % of the ROI — the frames' valid sets
barely overlap, so levels and anomalies would rest on a handful of cells — stop the run with the per-frame
coverages). Training frames below the threshold keep
their observations but get weight 0 (they would otherwise shrink the mask for every frame). The anomaly target
$a(i,t) = LST(i,t) - \overline{LST}_{FIXED \wedge valid}(t)$ uses this one cell set for every frame, so a changing
valid-cell composition (clouds, SLC-off gaps) cannot masquerade as warming; `RAW_LEVELS[mode][t]` stores the
subtracted level (day: before the overpass-time correction of §4.7). `FIXED_MASK_COVERAGE[mode][t]` = share of
the FIXED_MASK cells with a valid target at frame $t$ (1.0 for mask frames); section 09 corrects the level of a
frame that does not cover the whole mask with the model-predicted anomaly of its missing cells (§4.7).
A day frame without a single valid target is an acquisition fault and stops the section.

**Cross-validation groupings** (columns of `DF`, so every later stage reuses identical folds; a cell keeps its
fold in every frame):
* `fold_block20` — 20 km square blocks (`block20_id`) greedily size-balanced into `CFG.N_FOLDS_BLOCK` folds
  (largest block first → currently smallest fold), the headline spatial scheme (≈ the residual correlation range);
* `fold_block5` — v1's 5 × 5 tiles over the grid extent, greedy into 5 folds (comparable with v1);
* `fold_district` — the district id (leave-one-district-out in section 07);
* `fold_random` — seeded random rows, 5 folds (the optimistic upper bound);
* `block10_id` — 10 km blocks for the inner early-stopping split (always drawn from training rows only).

**Missing predictors** stay NaN (float32): tree models route them, linear models / RF / GNN impute with the
training-fold median inside each fold.

In [ ]:
_lst_enter_section('06')  # section marker (generated by build_notebook.py)

In [ ]:
import hashlib as _s06_hashlib
import math as _s06_math
import os as _s06_os
import time as _s06_time
import warnings as _s06_warnings
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view as _s06_sliding_window_view
from scipy import ndimage as _s06_ndi

# ---- feature list (SPEC §4.1, order binding) ---------------------------------------------------------------------
BASE_FEATURES = [
    "ndvi", "ndbi", "mndwi", "albedo", "ndvi_spring", "ndvi_drop",
    "frac_built", "frac_forest", "frac_water", "frac_cropland", "frac_barren",
    "built_v", "log_pop", "dist_urban", "dist_builtup", "lm_pd", "lm_ed", "lm_contag",
    "dist_water",
    "elevation", "slope", "aspect_sin", "aspect_cos", "elev_tpi7", "elev_tpi21", "elev_std7",
    "era5_t2m", "era5_vpd", "era5_swvl1", "era5_evap", "era5_ssr", "era5_wind", "era5_precip_jfm",
]
FOCAL_SPEC = {"bases": ["ndvi", "ndbi", "albedo", "frac_built", "frac_forest", "frac_water", "frac_barren"],
              "windows": [3, 15]}
_S06_FOCAL_FEATURES = [f"{b}_m{w}" for b in FOCAL_SPEC["bases"] for w in FOCAL_SPEC["windows"]]
FEATURES = BASE_FEATURES + _S06_FOCAL_FEATURES
if len(BASE_FEATURES) != 33 or len(FEATURES) != 47 or len(set(FEATURES)) != 47:
    raise AssertionError(f"feature list must be 33 base + 14 focal unique names, "
                         f"got {len(BASE_FEATURES)}/{len(FEATURES)}")

_S06_FAMILY = {}
for _fam, _names in (("spectral", ["ndvi", "ndbi", "mndwi", "albedo", "ndvi_spring", "ndvi_drop"]),
                     ("landcover", ["frac_built", "frac_forest", "frac_water", "frac_cropland", "frac_barren"]),
                     ("urban", ["built_v", "log_pop", "dist_urban", "dist_builtup", "lm_pd", "lm_ed", "lm_contag"]),
                     ("water", ["dist_water"]),
                     ("terrain", ["elevation", "slope", "aspect_sin", "aspect_cos", "elev_tpi7", "elev_tpi21",
                                  "elev_std7"]),
                     ("climate", ["era5_t2m", "era5_vpd", "era5_swvl1", "era5_evap", "era5_ssr", "era5_wind",
                                  "era5_precip_jfm"])):
    for _n in _names:
        _S06_FAMILY[_n] = _fam
for _b in FOCAL_SPEC["bases"]:
    for _w in FOCAL_SPEC["windows"]:
        _S06_FAMILY[f"{_b}_m{_w}"] = _S06_FAMILY[_b]       # a focal mean belongs to its base feature's family

# ---- SHAP groups (SPEC §4.4, order binding): id -> (members, primary, label, family, actionable) ---------------------
_S06_GROUP_ROWS = [
    ("ndvi", ["ndvi", "ndvi_m3", "ndvi_m15"], "ndvi", "Vegetation (NDVI)", "spectral", True),
    ("phenology", ["ndvi_spring", "ndvi_drop"], "ndvi_drop", "Spring green-up", "spectral", False),
    ("ndbi", ["ndbi", "ndbi_m3", "ndbi_m15"], "ndbi", "Built-up index (NDBI)", "spectral", False),
    ("moisture_index", ["mndwi"], "mndwi", "Surface wetness (MNDWI)", "spectral", False),
    ("albedo", ["albedo", "albedo_m3", "albedo_m15"], "albedo", "Albedo", "spectral", False),
    ("built", ["frac_built", "frac_built_m3", "frac_built_m15"], "frac_built", "Built-up fraction", "landcover", True),
    ("built_volume", ["built_v"], "built_v", "Built volume", "urban", False),
    ("population", ["log_pop"], "log_pop", "Population density", "urban", False),
    ("forest", ["frac_forest", "frac_forest_m3", "frac_forest_m15"], "frac_forest", "Tree cover", "landcover", True),
    ("water", ["frac_water", "frac_water_m3", "frac_water_m15", "dist_water"], "frac_water", "Water", "water", True),
    ("cropland", ["frac_cropland"], "frac_cropland", "Cropland", "landcover", True),
    ("barren", ["frac_barren", "frac_barren_m3", "frac_barren_m15"], "frac_barren", "Barren land", "landcover", True),
    ("urban_proximity", ["dist_urban", "dist_builtup"], "dist_urban", "Distance to urban core", "urban", False),
    ("landscape", ["lm_pd", "lm_ed", "lm_contag"], "lm_ed", "Landscape pattern", "landcover", True),
    ("terrain", ["elevation", "slope", "aspect_sin", "aspect_cos", "elev_tpi7", "elev_tpi21", "elev_std7"],
     "elevation", "Terrain", "terrain", False),
    ("climate_heat", ["era5_t2m", "era5_ssr"], "era5_t2m", "Air temperature & radiation", "climate", False),
    ("climate_moisture", ["era5_vpd", "era5_swvl1", "era5_evap", "era5_precip_jfm"], "era5_swvl1",
     "Soil & air moisture", "climate", False),
    ("climate_wind", ["era5_wind"], "era5_wind", "Wind", "climate", False),
]
SHAP_GROUPS = OrderedDict((gid, list(members)) for gid, members, *_ in _S06_GROUP_ROWS)
GROUP_META = OrderedDict(
    (gid, {"label": label, "primary": primary, "family": family, "actionable": bool(actionable),
           "members": list(members)})
    for gid, members, primary, label, family, actionable in _S06_GROUP_ROWS)
_S06_GROUP_OF = {f: gid for gid, members in SHAP_GROUPS.items() for f in members}
if len(SHAP_GROUPS) != 18 or sorted(_S06_GROUP_OF) != sorted(FEATURES) \
        or sum(len(m) for m in SHAP_GROUPS.values()) != len(FEATURES):
    raise AssertionError("SHAP_GROUPS must partition the 47 FEATURES into 18 groups")
for _gid, _meta in GROUP_META.items():
    if _meta["primary"] not in SHAP_GROUPS[_gid]:
        raise AssertionError(f"primary feature of group {_gid} is not a member")

# Monotone-constraint and target-validity constants used by sections 06/07
_S06_MIN_VALID_SHARE = 0.5      # landscape-metric windows need >= 50 % valid fine land-cover pixels
_S06_N_LC_CLASSES = 6
_S06_LM_CHUNK_PIXELS = 1 << 23  # ~8.4 M fine pixels per landscape-metric chunk -> peak RAM well below 1 GB
_S06_LM_STRUCTURE = np.zeros((3, 3, 3), dtype=bool)   # 8-connectivity in a window, none along the stack axis
_S06_LM_STRUCTURE[1] = True
# Training frames whose valid-target share of the ROI is below this value are left out of the fixed mask and get
# weight 0 (D8; part of the stage key).
_S06_FRAME_MIN_COVERAGE = float(_s06_os.environ.get("LST_FRAME_MIN_COVERAGE", "0.5"))
if not 0.0 < _S06_FRAME_MIN_COVERAGE <= 1.0:
    raise ValueError(f"LST_FRAME_MIN_COVERAGE must be in (0, 1], got {_S06_FRAME_MIN_COVERAGE}")
_S06_FIXED_MASK_WARN_SHARE = 0.8
_S06_VERSION = "06-v2.1"
log(f"06: {len(FEATURES)} features ({len(BASE_FEATURES)} base + {len(_S06_FOCAL_FEATURES)} focal), "
    f"{len(SHAP_GROUPS)} SHAP groups")

## 6.1 Grid operators: NaN-aware box statistics and distances
`uniform_filter` computes running window sums in O(n) per axis, independent of the window size, so the 15-cell
windows cost the same as the 3-cell ones. Means are formed as the ratio of filtered value·mask and filtered mask
(zero padding), which is exactly the mean of the valid cells inside the window — at the ROI edge the window
shrinks to the cells that exist instead of mixing in fake zeros. The box standard deviation is evaluated on values
centred on the ROI mean to avoid cancellation in $E[x^2] - E[x]^2$.

In [ ]:
def _s06_box_mean(x, size, valid=None):
    """NaN-aware box mean of a (H, W) grid over valid cells (float64 in, float64 out; NaN where no valid cell)."""
    x = np.asarray(x, dtype=np.float64)
    m = np.isfinite(x) if valid is None else (np.asarray(valid, bool) & np.isfinite(x))
    size = int(size)
    num = _s06_ndi.uniform_filter(np.where(m, x, 0.0), size=size, mode="constant", cval=0.0)
    den = _s06_ndi.uniform_filter(m.astype(np.float64), size=size, mode="constant", cval=0.0)
    with np.errstate(invalid="ignore", divide="ignore"):
        out = num / den
    out[den < 0.5 / (size * size)] = np.nan          # no valid cell in the window (den is k / size^2, k >= 1)
    # uniform_filter's running sums leave residues of ~1e-17 in windows whose values are all exactly 0 (e.g. no
    # water within 15 cells); they would let trees split on "0 vs 1e-17" and make untouched cells look changed in
    # counterfactuals. NaN is untouched (abs(NaN) < tol is False).
    out[np.abs(out) < 1e-12] = 0.0
    return out


def _s06_box_std(x, size, valid=None):
    """NaN-aware box standard deviation (population form) over valid cells (exactly 0 in constant windows)."""
    x = np.asarray(x, dtype=np.float64)
    m = np.isfinite(x) if valid is None else (np.asarray(valid, bool) & np.isfinite(x))
    if not m.any():
        return np.full(x.shape, np.nan)
    centre = float(x[m].mean())
    xc = np.where(m, x - centre, np.nan)
    mean = _s06_box_mean(xc, size, m)
    mean_sq = _s06_box_mean(xc * xc, size, m)
    var = mean_sq - mean * mean
    # E[x^2] - E[x]^2 of a constant window is a cancellation residue of the order eps * E[x^2]: snap it to 0
    var[var <= 1e-12 * np.abs(mean_sq)] = 0.0
    return np.sqrt(np.maximum(var, 0.0))


def _s06_tpi(elev, size, roi):
    """Topographic position elevation − box mean (m); |TPI| < 1e-9 m (filter residue on flat ground) set to 0."""
    tpi = elev - _s06_box_mean(elev, size, roi)
    tpi[np.abs(tpi) < 1e-9] = 0.0
    return tpi


# Urban core = cells whose 7 x 7 km mean built fraction reaches this value. Real data (2026-11): the 1 km built
# fraction saturates near 0.69 and its 7 km mean peaks at 0.45-0.49 over NCR, so the first value (0.5, tuned on the
# synthetic world) selected no cell; 0.3 gives cores of ~240 (1990) to ~560 km2 (2025).
_S06_URBAN_CORE_BUILT = 0.3
_S06_URBAN_CORE_TOP_SHARE = 0.005       # fallback when no cell reaches it: the top 0.5 % of the ROI by 7 km mean


def _s06_urban_core(box7_built, roi):
    """Cells of the urban core (box-7 mean built fraction >= ``_S06_URBAN_CORE_BUILT``); when none reaches it, the
    top ``_S06_URBAN_CORE_TOP_SHARE`` of the ROI by that mean (with a warning), so ``dist_urban`` never collapses to
    an uninformative constant."""
    core = roi & (box7_built >= _S06_URBAN_CORE_BUILT)
    vals = box7_built[roi]
    if core.any() or not vals.size or float(vals.max()) <= 0.0:
        return core
    thr = float(np.quantile(vals, 1.0 - _S06_URBAN_CORE_TOP_SHARE))
    log(f"06: no cell has a 7 km mean built fraction >= {_S06_URBAN_CORE_BUILT} (max {float(vals.max()):.3f}); the "
        f"urban core is the top {_S06_URBAN_CORE_TOP_SHARE:.1%} of the ROI (>= {thr:.3f})", "WARNING")
    return roi & (box7_built >= thr) & (box7_built > 0.0)


def _s06_distance_km(target, res_m, label=""):
    """Euclidean distance (km) from every cell to the nearest ``target`` cell (0 on targets).

    With no target at all the distance is undefined; the grid diagonal (the largest distance the grid can express)
    is used and a warning is logged, so the feature stays finite and constant (uninformative) instead of NaN.
    """
    target = np.asarray(target, bool)
    if not target.any():
        log(f"06: no target cells for {label or 'distance'}; using the grid diagonal as a constant", "WARNING")
        return np.full(target.shape, _s06_math.hypot(*target.shape) * res_m / 1000.0, dtype=np.float64)
    return _s06_ndi.distance_transform_edt(~target) * (float(res_m) / 1000.0)


def _s06_fine_to_cell(fine, min_valid_frac):
    """1 km mean of the fine (250 m) pixels of each cell; NaN when < ``min_valid_frac`` of them are valid."""
    return block_mean(np.asarray(fine, np.float32), int(CFG.FINE_FACTOR), float(min_valid_frac)).astype(np.float64)

## 6.2 Vectorised landscape metrics (ported unchanged from v1)
A window is the block of fine land-cover pixels under `CFG.LM_WINDOW_CELLS` × `CFG.LM_WINDOW_CELLS` grid cells
centred on the cell. With $A$ the valid area, $N_p$ the number of 8-connected patches (summed over classes 1–6),
$g_{ik}$ the 4-neighbour adjacencies counted in both directions and $m$ the number of classes present:

$$\mathrm{PD} = \frac{N_p}{A}\,10^{4}\times 100,\qquad \mathrm{ED} = \frac{r\sum_{i<k} g_{ik}}{A}\,10^{4},\qquad
  \mathrm{CONTAG} = \Big[1 + \frac{\sum_{i,k} p_{ik}\ln p_{ik}}{2\ln m}\Big]\times 100,\quad p_{ik} = g_{ik}/\textstyle\sum g .$$

Sides on the landscape border or touching nodata are not edges (pylandstats `count_boundary=False`); a single-class
window gets CONTAG = 100. All windows of a chunk are stacked into one `(n, b, b)` array: patches come from one 3-D
`ndimage.label` per class (no connectivity along the stack axis) and adjacencies from one `bincount`. With
`ENV["has_cupy"]` the same array code runs on the GPU (`cupyx.scipy.ndimage.label`), falling back to NumPy.

In [ ]:
def _s06_lm_backend():
    """Return (array module, label function, backend name) for landscape metrics."""
    if ENV.get("has_cupy"):
        try:
            import cupy as _cp
            import cupyx.scipy.ndimage as _cndi

            return _cp, _cndi.label, "cupy"
        except Exception as exc:  # noqa: BLE001 - broken CUDA install -> CPU
            log(f"cupy unavailable for landscape metrics ({exc!r}); using NumPy", "WARNING")
    return np, _s06_ndi.label, "numpy"


def _s06_window_stack_view(lc_fine, f, window_cells):
    """Strided view (H, W, b, b) of the fine pixels under each cell's window (no copy)."""
    if window_cells < 1 or (window_cells > 1 and window_cells % 2 == 0):
        raise ValueError(f"LM_WINDOW_CELLS must be 1 or an odd number >= 3, got {window_cells}")
    half_px = (window_cells - 1) // 2 * f
    padded = np.pad(lc_fine, half_px, mode="constant", constant_values=0) if half_px else lc_fine
    size = window_cells * f
    return _s06_sliding_window_view(padded, (size, size))[::f, ::f]


def _s06_metrics_from_counts(counts, valid, n_patches, adjacency, fine_res, px_per_win):
    """Turn per-window counts into PD, ED and CONTAG (NumPy, float64)."""
    area_ha = valid * fine_res * fine_res / 1e4
    total_adj = adjacency.sum(axis=(1, 2)).astype(np.float64)
    like_adj = np.einsum("nii->n", adjacency).astype(np.float64)
    edge_len_m = (total_adj - like_adj) / 2.0 * fine_res  # each unlike pair is counted twice
    n_present = (counts[:, 1:] > 0).sum(axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        pd_ = n_patches / area_ha * 100.0
        ed = edge_len_m / area_ha
        prob = adjacency / total_adj[:, None, None]
        plogp = np.where(prob > 0, prob * np.log(np.where(prob > 0, prob, 1.0)), 0.0)
        joint_entropy = -plogp.sum(axis=(1, 2))
        contag = (1.0 - joint_entropy / (2.0 * np.log(np.maximum(n_present, 2)))) * 100.0
    contag = np.where(n_present == 1, 100.0, contag)
    contag = np.where((n_present >= 2) & (total_adj == 0), np.nan, contag)
    insufficient = valid < _S06_MIN_VALID_SHARE * px_per_win
    for arr in (pd_, ed, contag):
        arr[insufficient] = np.nan
    return {"counts": counts, "pd": pd_, "ed": ed, "contag": contag}


def _s06_stack_metrics(stack, fine_res, xp=np, label_fn=_s06_ndi.label):
    """Class counts, PD, ED and CONTAG for a stack of windows ``(n, b, b)`` uint8 (0 = nodata)."""
    n_win, size, _ = stack.shape
    n_cls = _S06_N_LC_CLASSES
    stack = xp.asarray(stack)
    win_id = xp.arange(n_win, dtype=xp.int64)[:, None, None]
    counts = xp.bincount((win_id * (n_cls + 1) + stack).ravel(), minlength=n_win * (n_cls + 1))
    counts = counts.reshape(n_win, n_cls + 1)
    valid = counts[:, 1:].sum(axis=1)

    n_patches = xp.zeros(n_win, dtype=xp.int64)
    px_per_win = size * size
    for cls in range(1, n_cls + 1):
        if int(counts[:, cls].sum()) == 0:
            continue
        labels, n_labels = label_fn(stack == cls, structure=xp.asarray(_S06_LM_STRUCTURE))
        n_labels = int(n_labels)
        if n_labels == 0:
            continue
        flat = labels.ravel()
        pix = xp.flatnonzero(flat)
        owner = xp.zeros(n_labels + 1, dtype=xp.int64)
        owner[flat[pix]] = pix // px_per_win  # every pixel of a label lies in the same window
        n_patches += xp.bincount(owner[1:], minlength=n_win)
        del labels, flat, pix, owner

    adjacency = xp.zeros(n_win * n_cls * n_cls, dtype=xp.int64)
    for a, b in ((stack[:, :, :-1], stack[:, :, 1:]), (stack[:, :-1, :], stack[:, 1:, :])):
        ok = (a > 0) & (b > 0)
        wid = xp.broadcast_to(win_id, a.shape)[ok]
        ca = a[ok].astype(xp.int64) - 1
        cb = b[ok].astype(xp.int64) - 1
        base = wid * (n_cls * n_cls)
        adjacency += xp.bincount(base + ca * n_cls + cb, minlength=n_win * n_cls * n_cls)
        adjacency += xp.bincount(base + cb * n_cls + ca, minlength=n_win * n_cls * n_cls)
        del ok, wid, ca, cb, base
    adjacency = adjacency.reshape(n_win, n_cls, n_cls)

    to_np = (lambda arr: arr) if xp is np else xp.asnumpy
    counts, valid, n_patches, adjacency = map(to_np, (counts, valid, n_patches, adjacency))
    return _s06_metrics_from_counts(counts, valid, n_patches, adjacency, fine_res, px_per_win)


def _s06_landscape_metrics(lc_fine, f, fine_res, window_cells, cell_mask):
    """PD / ED / CONTAG rasters (H, W) float32 for the cells in ``cell_mask`` (others NaN) + the backend used."""
    n_rows, n_cols = cell_mask.shape
    view = _s06_window_stack_view(lc_fine, f, window_cells)
    size = view.shape[-1]
    rows, cols = np.nonzero(cell_mask)
    out = {k: np.full((n_rows, n_cols), np.nan, dtype=np.float32) for k in ("lm_pd", "lm_ed", "lm_contag")}
    xp, label_fn, backend = _s06_lm_backend()
    per_chunk = max(1, _S06_LM_CHUNK_PIXELS // (size * size))
    for start in range(0, rows.size, per_chunk):
        r_idx, c_idx = rows[start:start + per_chunk], cols[start:start + per_chunk]
        stack = np.ascontiguousarray(view[r_idx, c_idx])
        try:
            res = _s06_stack_metrics(stack, fine_res, xp, label_fn)
        except Exception as exc:  # noqa: BLE001 - GPU failure -> CPU for the remaining chunks
            if xp is np:
                raise
            log(f"GPU landscape metrics failed ({exc!r}); continuing on CPU", "WARNING")
            xp, label_fn, backend = np, _s06_ndi.label, "numpy"
            free_memory()
            res = _s06_stack_metrics(stack, fine_res, xp, label_fn)
        out["lm_pd"][r_idx, c_idx] = res["pd"]
        out["lm_ed"][r_idx, c_idx] = res["ed"]
        out["lm_contag"][r_idx, c_idx] = res["contag"]
        del stack, res
    out["backend"] = backend
    return out


def _s06_lc_factor(year):
    """Fine land-cover pixels per grid cell (f) and their size (m) for a frame's ``LC_FINE`` raster."""
    lc = LC_FINE[year]
    n_rows, n_cols = ROI_MASK.shape
    f = lc.shape[0] // n_rows
    if f < 1 or lc.shape != (n_rows * f, n_cols * f):
        raise ValueError(f"LC_FINE[{year}] shape {lc.shape} incompatible with grid {ROI_MASK.shape}")
    return f, float(GRID.res) / f

### 6.2.1 Validation against pylandstats
120 random windows (pooled over frames) are re-computed with `pylandstats.Landscape(array, res=(r, r), nodata=0,
neighborhood_rule="8")`; pylandstats 3.x uses the same definitions (PD per 100 ha of valid area, ED without the
boundary, CONTAG from the double-counted adjacency matrix), except that it returns NaN for a single-class window,
which is excluded from the CONTAG comparison. A mismatch warns but never stops the pipeline.

In [ ]:
def _s06_validate_landscape_metrics(n=120, tol=1e-6):
    """Compare the vectorised metrics with pylandstats on ``n`` random valid windows → ``LM_VALIDATION`` dict."""
    try:
        import pylandstats as pls
    except ImportError:
        log("pylandstats not installed - landscape metric validation skipped", "WARNING")
        return {"n": 0, "status": "skipped", "reason": "pylandstats not installed"}
    try:
        rng = np.random.default_rng(CFG.SEED)
        years = sorted(LC_FINE)
        per_year = np.bincount(rng.integers(0, len(years), size=n), minlength=len(years))
        ours, ref = {"pd": [], "ed": [], "contag": []}, {"pd": [], "ed": [], "contag": []}
        n_single_class = 0
        for year, n_year in zip(years, per_year):
            if n_year == 0:
                continue
            f, fine_res = _s06_lc_factor(year)
            view = _s06_window_stack_view(LC_FINE[year], f, int(CFG.LM_WINDOW_CELLS))
            cand_r, cand_c = np.nonzero(ROI_MASK)
            pick = rng.choice(cand_r.size, size=min(int(n_year) * 3, cand_r.size), replace=False)
            stack = np.ascontiguousarray(view[cand_r[pick], cand_c[pick]])
            res = _s06_stack_metrics(stack, fine_res)
            keep = np.flatnonzero(np.isfinite(res["pd"]))[: int(n_year)]
            for i in keep:
                with _s06_warnings.catch_warnings():
                    _s06_warnings.simplefilter("ignore")
                    ls = pls.Landscape(stack[i], res=(fine_res, fine_res), nodata=0, neighborhood_rule="8")
                    ref_vals = (ls.patch_density(), ls.edge_density(count_boundary=False), ls.contagion())
                ours["pd"].append(res["pd"][i])
                ref["pd"].append(ref_vals[0])
                ours["ed"].append(res["ed"][i])
                ref["ed"].append(ref_vals[1])
                if np.isfinite(ref_vals[2]):
                    ours["contag"].append(res["contag"][i])
                    ref["contag"].append(ref_vals[2])
                else:
                    n_single_class += 1
            del view, stack, res
    except Exception as exc:  # noqa: BLE001 - validation must never stop the pipeline
        log(f"landscape metric validation failed: {exc!r}", "WARNING")
        return {"n": 0, "status": "error", "reason": repr(exc)}

    out = {"n": len(ours["pd"]), "window_cells": int(CFG.LM_WINDOW_CELLS), "n_single_class": n_single_class,
           "pylandstats_version": getattr(pls, "__version__", "unknown")}
    worst = 0.0
    for metric in ("pd", "ed", "contag"):
        a, b = np.asarray(ours[metric], float), np.asarray(ref[metric], float)
        diff = np.abs(a - b) if a.size else np.array([np.nan])
        out[f"{metric}_max_abs_diff"] = float(np.nanmax(diff)) if a.size else None
        out[f"{metric}_mean_abs_diff"] = float(np.nanmean(diff)) if a.size else None
        out[f"{metric}_n"] = int(a.size)
        if a.size:
            worst = max(worst, float(np.nanmax(diff / np.maximum(np.abs(b), 1.0))))
    out["max_rel_diff"] = worst
    out["status"] = "ok" if worst <= tol else "warn"
    out["note"] = ("CONTAG compared only on windows with >= 2 classes (pylandstats returns NaN otherwise; "
                   "this pipeline uses 100). Relative tolerance %.0e." % tol)
    if out["status"] != "ok":
        log(f"landscape metrics differ from pylandstats (max rel diff {worst:.3g}): {out}", "WARNING")
    else:
        log(f"landscape metrics match pylandstats on {out['n']} windows (max rel diff {worst:.2e})")
    return out

## 6.3 Feature grids per frame
`_s06_base_grids` turns the acquisition globals of one frame into the 33 base grids except the focal-dependent
ones; `FOCAL_RECOMPUTE` then adds `dist_*` and the 14 focal means. The split is deliberate: section 08
(recommendations: change base features → recompute focal features, distances held fixed) and section 11
(projected land cover → recompute focal features **and** distances) call `FOCAL_RECOMPUTE` on modified grids, so
observed and counterfactual/projected feature fields come from one implementation.

In [ ]:
def FOCAL_RECOMPUTE(grids, recompute_distances=True, roi_mask=None, water_occ=None):
    """Return a copy of ``grids`` with the 14 focal features (and, optionally, the 3 distances) recomputed.

    Parameters
    ----------
    grids : dict feature -> (H, W) array. Must contain the focal bases (``FOCAL_SPEC["bases"]``) and, when
        ``recompute_distances``, ``frac_built`` and ``frac_water``.
    recompute_distances : recompute ``dist_water``, ``dist_urban`` and ``dist_builtup`` (True for new land cover,
        False for small counterfactual edits where distances are held fixed, SPEC §4.4).
    roi_mask : (H, W) bool, default ``ROI_MASK``; cells outside are NaN and never act as targets or neighbours.
    water_occ : (H, W) JRC water occurrence, default ``RAW_STATIC["water_occ"]`` (static).

    Every output grid is float32 with NaN outside the ROI.
    """
    roi = ROI_MASK if roi_mask is None else np.asarray(roi_mask, bool)
    out = dict(grids)
    for base in FOCAL_SPEC["bases"]:
        if base not in grids:
            raise KeyError(f"FOCAL_RECOMPUTE needs the base grid {base!r}")
        for w in FOCAL_SPEC["windows"]:
            out[f"{base}_m{w}"] = np.where(roi, _s06_box_mean(grids[base], w, roi), np.nan).astype(np.float32)
    if recompute_distances:
        res_m = float(GRID.res)
        occ = RAW_STATIC["water_occ"] if water_occ is None else water_occ
        fw = np.nan_to_num(np.asarray(grids["frac_water"], np.float64), nan=0.0)
        fb = np.nan_to_num(np.asarray(grids["frac_built"], np.float64), nan=0.0)
        water_t = roi & ((fw >= 0.3) | (np.nan_to_num(np.asarray(occ, np.float64), nan=0.0) >= 0.5))
        box7_built = np.nan_to_num(_s06_box_mean(np.where(roi, grids["frac_built"], np.nan), 7, roi), nan=0.0)
        out["dist_water"] = np.where(roi, _s06_distance_km(water_t, res_m, "dist_water"), np.nan).astype(np.float32)
        out["dist_urban"] = np.where(roi, _s06_distance_km(_s06_urban_core(box7_built, roi), res_m, "dist_urban"),
                                     np.nan).astype(np.float32)
        out["dist_builtup"] = np.where(roi, _s06_distance_km(roi & (fb >= 0.3), res_m, "dist_builtup"),
                                       np.nan).astype(np.float32)
    return out


def _s06_static_grids():
    """Static terrain grids (1 km): elevation, slope, aspect, topographic position and relief."""
    roi = ROI_MASK
    elev = np.where(roi, RAW_STATIC["elevation"], np.nan).astype(np.float64)
    out = {"elevation": elev, "slope": RAW_STATIC["slope"], "aspect_sin": RAW_STATIC["aspect_sin"],
           "aspect_cos": RAW_STATIC["aspect_cos"],
           "elev_tpi7": _s06_tpi(elev, 7, roi),
           "elev_tpi21": _s06_tpi(elev, 21, roi),
           "elev_std7": _s06_box_std(elev, 7, roi)}
    return {k: np.where(roi, v, np.nan).astype(np.float32) for k, v in out.items()}


def _s06_base_grids(year, static):
    """Base feature grids of one frame (everything except focal means and distances) + LM backend name."""
    raw = RAW_OBS[year]
    roi = ROI_MASK
    min_frac = float(CFG.FINE_MIN_VALID_FRAC)
    g = {}
    for name, key in (("ndvi", "ndvi_f"), ("ndbi", "ndbi_f"), ("mndwi", "mndwi_f"), ("albedo", "albedo_f"),
                      ("ndvi_spring", "ndvi_spring_f")):
        g[name] = _s06_fine_to_cell(raw[key], min_frac)
    g["ndvi_drop"] = g["ndvi_spring"] - g["ndvi"]

    # Land-cover simplex: GHSL built-up fraction + GLC classes rescaled to the non-built share (SPEC §4.1).
    frac_built = np.clip(_s06_fine_to_cell(raw["built_f"], min_frac), 0.0, 1.0)
    glc_b = np.asarray(raw["glc_built"], np.float64)
    with np.errstate(invalid="ignore", divide="ignore"):
        scale = np.where(glc_b >= 0.999, 0.0, (1.0 - frac_built) / (1.0 - glc_b))
    g["frac_built"] = frac_built
    for name, key in (("frac_forest", "glc_forest"), ("frac_water", "glc_water"),
                      ("frac_cropland", "glc_cropland"), ("frac_barren", "glc_barren")):
        g[name] = np.clip(np.asarray(raw[key], np.float64) * scale, 0.0, 1.0)

    g["built_v"] = np.log1p(np.clip(np.asarray(raw["built_v"], np.float64), 0.0, None))
    g["log_pop"] = np.log1p(np.clip(np.asarray(raw["pop"], np.float64), 0.0, None))
    for name in ("era5_t2m", "era5_vpd", "era5_swvl1", "era5_evap", "era5_ssr", "era5_wind", "era5_precip_jfm"):
        g[name] = np.asarray(raw[name], np.float64)
    g.update(static)

    backend = "none"
    if year in LC_FINE:
        f, fine_res = _s06_lc_factor(year)
        lm = _s06_landscape_metrics(LC_FINE[year], f, fine_res, int(CFG.LM_WINDOW_CELLS), roi)
        backend = lm.pop("backend")
        g.update(lm)
    else:
        log(f"06: LC_FINE has no {year} raster - landscape metrics set to NaN", "WARNING")
        for name in ("lm_pd", "lm_ed", "lm_contag"):
            g[name] = np.full(roi.shape, np.nan)
    return {k: np.where(roi, v, np.nan).astype(np.float32) for k, v in g.items()}, backend


def _s06_frame_grids(year, static):
    """All 47 feature grids (H, W) float32 of one observed frame, in ``FEATURES`` order."""
    base, backend = _s06_base_grids(year, static)
    full = FOCAL_RECOMPUTE(base, recompute_distances=True)
    missing = [f for f in FEATURES if f not in full]
    if missing:
        raise RuntimeError(f"06: frame {year} lacks feature grids {missing}")
    return {f: np.ascontiguousarray(full[f], dtype=np.float32) for f in FEATURES}, backend

## 6.4 Targets, fixed masks and raw levels
The day target is aggregated from the fine Landsat composite here (not taken from a 1 km product) so the 1 km
value is the plain mean of the same 250 m pixels the dashboard shows. Validity rules and weights follow SPEC §3.3;
the weight down-weights cells whose composite rests on few clear scenes (day) or few composites (night) in both
fitting and the weighted CV metrics.

In [ ]:
def _s06_targets(year):
    """{mode: {"lst", "nobs", "valid", "weight"}} (H, W) grids of one frame (weights 0 outside training frames)."""
    raw = RAW_OBS[year]
    roi = ROI_MASK
    out = {}
    # ---- day: harmonised Landsat (before the overpass correction) ------------------------------------------------
    lst_day = _s06_fine_to_cell(raw["lst_day_f"], float(CFG.FINE_MIN_VALID_FRAC))
    nobs_f = np.where(ROI_MASK_FINE, np.nan_to_num(np.asarray(raw["lst_day_nobs_f"], np.float32), nan=0.0), np.nan)
    nobs_day = _s06_fine_to_cell(nobs_f, 0.0)
    valid_day = roi & np.isfinite(lst_day) & (np.nan_to_num(nobs_day, nan=0.0) >= float(CFG.DAY_MIN_NOBS))
    w_day = np.where(valid_day, np.clip(np.nan_to_num(nobs_day) / 5.0, 0.4, 1.0), 0.0)
    out["day"] = {"lst": np.where(valid_day, lst_day, np.nan), "nobs": nobs_day, "valid": valid_day,
                  "weight": w_day}
    # ---- night: Aqua-equivalent 1 km (VIIRS counts daily observations) -------------------------------------------
    meta = (FRAME_META.get(year) or {}).get("night") or {}
    source = meta.get("source")
    min_n = float({"viirs_cal": CFG.NIGHT_MIN_NOBS_VIIRS, "terra_cal": CFG.NIGHT_MIN_NOBS_TERRA}.get(
        source, CFG.NIGHT_MIN_NOBS))
    lst_n = np.asarray(raw["lst_night"], np.float64)
    nobs_n = np.nan_to_num(np.asarray(raw["lst_night_nobs"], np.float64), nan=0.0)
    valid_n = roi & np.isfinite(lst_n) & (nobs_n >= min_n)
    train_n = year in set(CFG.NIGHT_TRAIN_FRAMES)
    w_n = np.where(valid_n & train_n, np.clip(nobs_n / 12.0, 0.5, 1.0), 0.0)
    out["night"] = {"lst": np.where(valid_n, lst_n, np.nan), "nobs": nobs_n, "valid": valid_n, "weight": w_n,
                    "source": source, "min_nobs": min_n}
    return out


def _s06_target_coverage(targets, mode):
    """{year: share of the ROI cells with a valid ``mode`` target} for every frame of ``targets``."""
    n_roi = max(int(ROI_MASK.sum()), 1)
    return {int(y): float((targets[y][mode]["valid"] & ROI_MASK).sum()) / n_roi for y in sorted(targets)}


def _s06_check_day_frames(targets):
    """Stop when a day frame has no valid target at all (acquisition fault, SPEC §1.2 / D9)."""
    if "day" not in CFG.MODES:
        return
    cov = _s06_target_coverage(targets, "day")
    empty = [y for y, c in cov.items() if c <= 0.0]
    if empty:
        raise RuntimeError(
            f"06: day frame(s) {empty} have no valid Landsat target in any ROI cell (an acquisition fault: every "
            f"observed day frame must be observed or observed-sparse). Valid-target share of the ROI per frame: "
            + ", ".join(f"{y}: {c:.1%}" for y, c in cov.items())
            + ". Check section 03 for that frame (scenes per path/row, QA / ST band filtering).")


def _s06_fixed_masks(targets, train_frames):
    """FIXED_MASK per mode (strict intersection over the well-covered training frames, D8) + info, caveats.

    Returns (masks, info, coverage, caveats): ``coverage[mode][year]`` = share of the FIXED_MASK cells with a valid
    target at that frame (every frame; 1.0 for the mask frames); ``info[mode]`` holds the rule, the mask frames,
    each frame's valid share of the ROI (``coverage``), the training frames left out (``excluded_frames``, weight 0),
    the cell count and share of the ROI.
    """
    masks, info, coverage, caveats = {}, {}, {}, []
    n_roi = int(ROI_MASK.sum())
    thr = float(_S06_FRAME_MIN_COVERAGE)
    for mode in CFG.MODES:
        cov = _s06_target_coverage(targets, mode)
        frames = [y for y in train_frames[mode] if y in targets]
        mask_frames = [y for y in frames if cov[y] >= thr]
        excluded = [y for y in frames if cov[y] < thr]
        if len(mask_frames) < 2:
            raise RuntimeError(
                f"06: mode {mode} has only {len(mask_frames)} training frame(s) with a valid-target share of the ROI "
                f">= {thr:.0%} (LST_FRAME_MIN_COVERAGE); at least 2 are needed for a fixed mask. Coverage of the "
                f"training frames: " + ", ".join(f"{y}: {cov[y]:.1%}" for y in frames))
        mask = ROI_MASK.copy()
        for y in mask_frames:
            mask &= targets[y][mode]["valid"]
        share = float(mask.sum()) / max(n_roi, 1)
        if share < 0.01:
            raise RuntimeError(
                f"06: FIXED_MASK[{mode}] (cells valid in every mask frame {mask_frames}) keeps only {int(mask.sum())} "
                f"cells ({share:.2%} of the ROI); the valid-target sets of the frames barely overlap. Coverage: "
                + ", ".join(f"{y}: {cov[y]:.1%}" for y in frames))
        if share < _S06_FIXED_MASK_WARN_SHARE:
            text = (f"The {mode} fixed mask (cells with a valid target in every training frame with >= {thr:.0%} "
                    f"coverage) keeps only {share:.1%} of the ROI cells; missing share of the ROI per frame: "
                    + ", ".join(f"{y} {1.0 - cov[y]:.1%}" for y in mask_frames)
                    + ". Frame levels and anomalies refer to this cell set.")
            log("06: " + text, "WARNING")
            caveats.append(text)
        if excluded:
            text = (f"{mode} training frame(s) " + ", ".join(f"{y} ({cov[y]:.1%} of the ROI valid)" for y in excluded)
                    + f" are below the {thr:.0%} coverage threshold (LST_FRAME_MIN_COVERAGE): they keep their "
                    "observations but are left out of the fixed mask and of model training (weight 0); their "
                    "level gets a model-based composition correction (section 09).")
            log("06: " + text, "WARNING")
            caveats.append(text)
        n_fixed = max(int(mask.sum()), 1)
        coverage[mode] = {int(y): (1.0 if y in mask_frames
                                   else float((mask & targets[y][mode]["valid"]).sum()) / n_fixed)
                          for y in sorted(targets)}
        masks[mode] = mask
        info[mode] = {"rule": f"valid in every training frame with >= {thr:g} valid-target coverage of the ROI",
                      "frames": list(mask_frames), "training_frames": list(frames), "excluded_frames": list(excluded),
                      "coverage": {int(y): round(cov[y], 4) for y in cov}, "min_coverage": thr,
                      "n_cells": int(mask.sum()), "share": round(share, 4)}
        log(f"06: FIXED_MASK[{mode}]: {int(mask.sum()):,} cells ({share:.1%} of the ROI; strict intersection of mask "
            f"frames {mask_frames}" + (f"; excluded {excluded}" if excluded else "") + ")")
    return masks, info, coverage, caveats

## 6.5 Cross-validation blocks and greedy fold assignment
Blocks are numbered in grid row-major order from the grid's top-left corner (`block_id = (row // b) ·
ceil(W / b) + col // b`), the convention of the dashboard fold map. Fold sizes are balanced on **ROI cell
counts**, which are identical in every frame, so a cell (and its whole block) belongs to one fold in every frame
and mode — the condition that keeps a cell's own past and future out of its training set.

In [ ]:
def _s06_block_grid(block_cells):
    """(H, W) int32 block ids of square blocks of ``block_cells`` cells (row-major from the top-left)."""
    h, w = ROI_MASK.shape
    b = max(1, int(block_cells))
    rows, cols = np.indices((h, w))
    return ((rows // b) * _s06_math.ceil(w / b) + cols // b).astype(np.int32)


def _s06_greedy_block_folds(block_counts, n_folds):
    """Deterministic size-balanced assignment: largest block first → currently smallest fold (ties: lowest id)."""
    order = sorted((b for b, c in block_counts.items() if c > 0), key=lambda b: (-block_counts[b], b))
    if len(order) < n_folds:
        log(f"only {len(order)} non-empty blocks for {n_folds} folds - some folds will be empty", "WARNING")
    load = np.zeros(n_folds, dtype=np.int64)
    assignment = {}
    for block in order:
        fold = int(np.argmin(load))
        assignment[block] = fold
        load[fold] += block_counts[block]
    return assignment, load


def _s06_fold_grid(block_grid, n_folds):
    """(H, W) int16 fold of every ROI cell (−1 outside) from greedy size balancing of ``block_grid`` blocks."""
    ids = block_grid[ROI_MASK]
    uniq, counts = np.unique(ids, return_counts=True)
    assignment, load = _s06_greedy_block_folds(dict(zip(uniq.tolist(), counts.tolist())), int(n_folds))
    lut = np.full(int(block_grid.max()) + 1, -1, dtype=np.int16)
    for block, fold in assignment.items():
        lut[block] = fold
    fold = np.where(ROI_MASK, lut[block_grid], -1).astype(np.int16)
    return fold, load


def _s06_cv_grids():
    """Block-id and fold grids for all schemes + a summary dict."""
    h, w = ROI_MASK.shape
    res = float(GRID.res)
    b20 = max(1, int(round(float(CFG.BLOCK_KM) * 1000.0 / res)))
    b10 = max(1, int(round(float(CFG.INNER_BLOCK_KM) * 1000.0 / res)))
    block20 = _s06_block_grid(b20)
    block10 = _s06_block_grid(b10)
    nby, nbx = (int(v) for v in CFG.SPATIAL_BLOCKS)
    rows, cols = np.indices((h, w))
    tiles5 = ((rows * nby // h) * nbx + cols * nbx // w).astype(np.int32)
    fold20, load20 = _s06_fold_grid(block20, int(CFG.N_FOLDS_BLOCK))
    fold5, load5 = _s06_fold_grid(tiles5, int(CFG.N_FOLDS))
    info = {"block20_cells": b20, "block20_km_actual": b20 * res / 1000.0, "block10_cells": b10,
            "n_block20": int(np.unique(block20[ROI_MASK]).size), "n_block10": int(np.unique(block10[ROI_MASK]).size),
            "fold_block20_cells": load20.tolist(), "fold_block5_cells": load5.tolist(),
            "block5_tiles": [nby, nbx]}
    log(f"06: blocks20 = {b20}x{b20} cells ({info['block20_km_actual']:g} km; {info['n_block20']} non-empty) -> "
        f"{CFG.N_FOLDS_BLOCK} folds of {load20.tolist()} cells; v1 5x5 tiles -> folds {load5.tolist()}; "
        f"inner blocks {b10}x{b10} cells")
    return {"block20": block20, "block10": block10, "tiles5": tiles5, "fold20": fold20, "fold5": fold5}, info

## 6.6 Assemble `DF`, `FEATURE_GRIDS`, `FIXED_MASK`, `RAW_LEVELS` (stage-cached)
Frames are stacked in `CFG.FRAMES_OBS` order, cells in canonical order inside every frame, so `DF` is sorted by
(`year`, `cell_id`) by construction and frame `j` occupies rows `[j·N_CELLS, (j+1)·N_CELLS)` — later sections can
slice frames without a search. The stage-cache key contains a fingerprint of the acquisition globals (sums and
valid counts of every raster), so a changed input layer can never reuse stale features.

In [ ]:
_S06_DF_DTYPES = {
    "cell_id": "int32", "row": "int32", "col": "int32", "x": "float64", "y": "float64",
    "lon": "float64", "lat": "float64", "district": "int16", "year": "int16",
    **{name: "float32" for name in FEATURES},
    "lst_day": "float32", "lst_night": "float32", "w_day": "float32", "w_night": "float32",
    "anom_day": "float32", "anom_night": "float32",
    "fold_block5": "int8", "fold_block20": "int8", "fold_random": "int8",
    "block10_id": "int32", "block20_id": "int32", "fold_district": "int16",
}


def _s06_fingerprint():
    """SHA-1 over cheap summaries (nansum, finite count) of every acquisition raster used by this section."""
    h = _s06_hashlib.sha1()

    def add(tag, arr):
        a = np.asarray(arr)
        if a.dtype.kind in "fc":
            fin = np.isfinite(a)
            h.update(f"{tag}:{a.shape}:{float(np.sum(a, where=fin, dtype=np.float64)):.9e}:{int(fin.sum())};"
                     .encode())
        else:
            h.update(f"{tag}:{a.shape}:{int(a.astype(np.int64).sum())}:{int(np.count_nonzero(a))};".encode())

    for key in sorted(RAW_STATIC):
        add(f"static/{key}", RAW_STATIC[key])
    for year in sorted(RAW_OBS):
        for key in sorted(RAW_OBS[year]):
            add(f"{year}/{key}", RAW_OBS[year][key])
    for year in sorted(LC_FINE):
        add(f"lc/{year}", np.bincount(np.asarray(LC_FINE[year]).ravel(), minlength=8))
    add("roi", ROI_MASK)
    add("district", DISTRICT_IDX)
    h.update(repr(sorted((int(y), (((m or {}).get("night")) or {}).get("source"))
                         for y, m in FRAME_META.items())).encode())
    return h.hexdigest()[:16]


def _s06_build():
    """Compute every product of section 06; returns a dict (the stage-cache payload)."""
    t_start = _s06_time.perf_counter()
    frames = [int(y) for y in CFG.FRAMES_OBS if int(y) in RAW_OBS]
    missing = sorted(set(int(y) for y in CFG.FRAMES_OBS) - set(frames))
    if missing:
        raise RuntimeError(f"06: RAW_OBS lacks observed frames {missing}")
    if GRID_FINE.shape != (GRID.height * CFG.FINE_FACTOR, GRID.width * CFG.FINE_FACTOR):
        raise RuntimeError(f"06: GRID_FINE {GRID_FINE.shape} is not GRID x FINE_FACTOR")

    with timer("06_static_grids"):
        static = _s06_static_grids()
    grids, targets, backends = {}, {}, {}
    for year in frames:
        t0 = _s06_time.perf_counter()
        with timer(f"06_feature_grids_{year}"):
            grids[year], backends[year] = _s06_frame_grids(year, static)
            targets[year] = _s06_targets(year)
        log(f"06: frame {year}: features ready in {_s06_time.perf_counter() - t0:.1f}s (landscape metrics on "
            f"{backends[year]}); day targets {int(targets[year]['day']['valid'].sum()):,}, night targets "
            f"{int(targets[year]['night']['valid'].sum()):,} "
            f"(night source {targets[year]['night']['source']}, min nobs {targets[year]['night']['min_nobs']:g})")
    _s06_check_day_frames(targets)
    train_frames = {"day": list(frames), "night": [y for y in CFG.NIGHT_TRAIN_FRAMES if y in frames]}
    fixed, fixed_info, fixed_cov, caveats = _s06_fixed_masks(targets, train_frames)
    # under-covered training frames keep their observations (anomalies, levels) but do not train (D8)
    for mode in CFG.MODES:
        for y in fixed_info[mode]["excluded_frames"]:
            targets[y][mode]["weight"] = np.zeros_like(targets[y][mode]["weight"])
        train_frames[mode] = [y for y in train_frames[mode] if y not in fixed_info[mode]["excluded_frames"]]

    # ---- anomalies over the fixed mask + raw levels ---------------------------------------------------------------
    raw_levels = {m: {} for m in CFG.MODES}
    anoms = {m: {} for m in CFG.MODES}
    for year in frames:
        for mode in CFG.MODES:
            t = targets[year][mode]
            sel = fixed[mode] & t["valid"]
            level = float(np.mean(t["lst"][sel])) if sel.any() else float("nan")
            raw_levels[mode][year] = level
            anoms[mode][year] = (t["lst"] - level) if np.isfinite(level) else np.full(ROI_MASK.shape, np.nan)
            if t["valid"].any() and not sel.any():
                log(f"06: {mode} {year}: valid targets but no fixed-mask cell - anomaly undefined", "WARNING")

    # ---- CV grids and the long table ----------------------------------------------------------------------------
    cv, cv_info = _s06_cv_grids()
    r, c = CELL_ROWS, CELL_COLS
    n_cells, n_frames = int(N_CELLS), len(frames)
    X, Y = GRID.cell_centers_xy()
    tile = lambda a: np.tile(np.asarray(a), n_frames)  # noqa: E731 - canonical cell order repeated per frame
    cols = {
        "cell_id": tile(CELL_IDS), "row": tile(r), "col": tile(c),
        "x": tile(np.asarray(X, np.float64)[r, c]), "y": tile(np.asarray(Y, np.float64)[r, c]),
        "lon": tile(CELL_LON), "lat": tile(CELL_LAT), "district": tile(DISTRICT_IDX[r, c]),
        "year": np.repeat(np.asarray(frames, np.int16), n_cells),
    }
    for name in FEATURES:
        cols[name] = np.concatenate([grids[y][name][r, c] for y in frames])
    for mode in CFG.MODES:
        cols[f"lst_{mode}"] = np.concatenate([targets[y][mode]["lst"][r, c] for y in frames])
        cols[f"w_{mode}"] = np.concatenate([targets[y][mode]["weight"][r, c] for y in frames])
        cols[f"anom_{mode}"] = np.concatenate([anoms[mode][y][r, c] for y in frames])
    cols["fold_block5"] = tile(cv["fold5"][r, c])
    cols["fold_block20"] = tile(cv["fold20"][r, c])
    n_rows = n_cells * n_frames
    perm = np.random.default_rng(CFG.SEED).permutation(n_rows)
    fold_random = np.empty(n_rows, dtype=np.int8)
    fold_random[perm] = np.arange(n_rows) % int(CFG.N_FOLDS)
    cols["fold_random"] = fold_random
    cols["block10_id"] = tile(cv["block10"][r, c])
    cols["block20_id"] = tile(cv["block20"][r, c])
    cols["fold_district"] = tile(DISTRICT_IDX[r, c])
    frame = pd.DataFrame({k: cols[k] for k in _S06_DF_DTYPES}).astype(_S06_DF_DTYPES)
    frame.index = pd.RangeIndex(len(frame))
    del cols

    frame_info = {
        "frames": frames, "train_frames": train_frames, "fixed_mask": fixed_info, "cv": cv_info,
        "lm_backend": backends, "caveats": caveats,
        "targets": {int(y): {m: {"n_valid": int(targets[y][m]["valid"].sum()),
                                 "n_train": int((targets[y][m]["weight"] > 0).sum()),
                                 "mean_weight": (float(targets[y][m]["weight"][targets[y][m]["weight"] > 0].mean())
                                                 if (targets[y][m]["weight"] > 0).any() else None)}
                         for m in CFG.MODES} for y in frames},
        "seconds": round(_s06_time.perf_counter() - t_start, 1),
    }
    with timer("06_lm_validation"):
        lm_validation = _s06_validate_landscape_metrics(n=120)
    return {"FEATURE_GRIDS": grids, "DF": frame, "FIXED_MASK": fixed, "FIXED_MASK_COVERAGE": fixed_cov,
            "RAW_LEVELS": raw_levels, "LM_VALIDATION": lm_validation, "info": frame_info}


# Cross-section code: _s06_fine_to_cell uses block_mean of section 01, a shared helper covered by SHARED_CODE_TAG.
_s06_key = CFG.key("LM_WINDOW_CELLS", "BLOCK_KM", "N_FOLDS_BLOCK", "INNER_BLOCK_KM", "N_FOLDS", "SPATIAL_BLOCKS",
                   "DAY_MIN_NOBS", "FINE_MIN_VALID_FRAC", "NIGHT_MIN_NOBS", "NIGHT_MIN_NOBS_VIIRS", "NIGHT_MIN_NOBS_TERRA",
                   "NIGHT_TRAIN_FRAMES", "MODES", features=FEATURES, version=_S06_VERSION,
                   frame_min_coverage=_S06_FRAME_MIN_COVERAGE, fingerprint=_s06_fingerprint())
with timer("06_features"):
    _s06_payload = stage_cache("06_features", _s06_key).get_or_compute(_s06_build)
FEATURE_GRIDS = _s06_payload["FEATURE_GRIDS"]
DF = _s06_payload["DF"]
FIXED_MASK = _s06_payload["FIXED_MASK"]
FIXED_MASK_COVERAGE = {m: {int(y): float(v) for y, v in c.items()}
                       for m, c in _s06_payload["FIXED_MASK_COVERAGE"].items()}
RAW_LEVELS = _s06_payload["RAW_LEVELS"]
LM_VALIDATION = _s06_payload["LM_VALIDATION"]
_S06_INFO = _s06_payload["info"]
del _s06_payload
free_memory()
# caveats are part of the cached payload, so a resumed run reports them too
if not isinstance(globals().get("DATA_CAVEATS"), list):
    DATA_CAVEATS = []
for _s06_c in _S06_INFO.get("caveats", []):
    if _s06_c not in DATA_CAVEATS:
        DATA_CAVEATS.append(_s06_c)
        log("06 caveat: " + _s06_c, "WARNING")
log(f"06: DF {len(DF):,} rows x {DF.shape[1]} columns ({DF.memory_usage(deep=False).sum() / 2**20:.0f} MiB); "
    f"frames {_S06_INFO['frames']}; training rows day {int((DF['w_day'] > 0).sum()):,}, night "
    f"{int((DF['w_night'] > 0).sum()):,}")
log("06: RAW_LEVELS (°C, fixed-mask mean, day before overpass correction): "
    + "; ".join(f"{m}: " + ", ".join(f"{y}={v:.2f}" for y, v in RAW_LEVELS[m].items() if np.isfinite(v))
                for m in CFG.MODES))


def _s06_coverage_table():
    """Per frame and mode: valid share of the ROI, share of FIXED_MASK valid, training rows, mask frame or not."""
    years = DF["year"].to_numpy()
    rows = []
    for y in _S06_INFO["frames"]:
        row = {"year": int(y)}
        for m in CFG.MODES:
            fm = _S06_INFO["fixed_mask"][m]
            row[f"{m}_valid_share_roi"] = fm["coverage"].get(int(y))
            row[f"{m}_fixed_mask_valid"] = FIXED_MASK_COVERAGE[m].get(int(y))
            row[f"{m}_train_rows"] = int(((years == y) & (DF[f"w_{m}"].to_numpy() > 0)).sum())
            row[f"{m}_role"] = ("mask" if int(y) in fm["frames"] else
                                ("excluded" if int(y) in fm["excluded_frames"] else "-"))
        rows.append(row)
    return pd.DataFrame(rows)


print(f"Target coverage per frame (mask frames need >= {_S06_FRAME_MIN_COVERAGE:.0%} of the ROI valid; "
      "FIXED_MASK = cells valid in every mask frame):")
print(_s06_coverage_table().round(4).to_string(index=False))

## 6.7 Feature metadata (`FEATURE_META`) and statistics (`FEATURE_STATS`)
`FEATURE_META` drives every label in the dashboard. Keys: v1's `label, unit, group, description, source,
actionable, display` plus `family` (dominant-driver colouring) and `base` (False for the 14 focal means);
`group` is the SHAP group (§4.4). `actionable` marks the levers a planner can move directly: the five land-cover
fractions, NDVI (greening within existing covers) and the three landscape-configuration metrics. Focal means,
distances, NDBI/MNDWI/albedo, terrain, population and climate respond to those levers or are context.

In [ ]:
_S06_SRC_LANDSAT = "Landsat C2 L2 SR (30 m, TM/ETM+ → OLI harmonised), Apr–Jun median, 1 km mean of fine pixels"
_S06_SRC_ERA5 = "ERA5-Land monthly (Apr–Jun of the frame window), bilinear to 1 km"
_S06_META_ROWS = [
    # name, label, unit, display, source, description
    ("ndvi", "NDVI", "", "index", _S06_SRC_LANDSAT,
     "Normalised Difference Vegetation Index; denser, greener vegetation cools the surface by transpiration and "
     "shading."),
    ("ndbi", "NDBI", "", "index", _S06_SRC_LANDSAT,
     "Normalised Difference Built-up Index (SWIR1 vs NIR); high for built-up and bare, dry surfaces."),
    ("mndwi", "MNDWI", "", "index", _S06_SRC_LANDSAT,
     "Modified Normalised Difference Water Index (green vs SWIR1); high for open water and wet surfaces."),
    ("albedo", "Albedo", "", "index", _S06_SRC_LANDSAT + " (Liang 2001 shortwave albedo)",
     "Broadband shortwave albedo; bright surfaces absorb less solar radiation."),
    ("ndvi_spring", "NDVI (Feb–Mar)", "", "index", "Landsat C2 L2 SR, Feb–Mar median of the frame window",
     "Late-winter NDVI, when the rabi crop (wheat, mustard) is green."),
    ("ndvi_drop", "NDVI drop (spring − season)", "", "index", "ndvi_spring − ndvi",
     "Loss of greenness from Feb–Mar to Apr–Jun; large where crops were harvested and fields lie bare and hot."),
    ("frac_built", "Built-up fraction", "fraction", "percent", "JRC GHSL P2023A GHS_BUILT_S, 1 km mean",
     "Share of the cell covered by buildings and other built-up surfaces."),
    ("frac_forest", "Tree cover", "fraction", "percent", "GLC_FCS30D forest classes on the GHSL simplex",
     "Share of the cell under trees / forest."),
    ("frac_water", "Water", "fraction", "percent", "GLC_FCS30D water on the GHSL simplex",
     "Share of the cell covered by open water (rivers, canals, lakes, jheels)."),
    ("frac_cropland", "Cropland", "fraction", "percent", "GLC_FCS30D cropland on the GHSL simplex",
     "Share of the cell under cultivation."),
    ("frac_barren", "Barren land", "fraction", "percent", "GLC_FCS30D bare areas on the GHSL simplex",
     "Share of the cell that is bare soil, sand or rock."),
    ("built_v", "Built volume (log1p m³/m²)", "log1p(m³/m²)", "number", "JRC GHSL P2023A GHS_BUILT_V, 1 km mean",
     "Building volume per unit ground area (log); stores daytime heat and releases it at night."),
    ("log_pop", "Population density (log)", "log1p(persons/km²)", "number", "JRC GHSL P2023A GHS_POP, 1 km mean",
     "Residents per km² (log); proxy for anthropogenic heat and building density."),
    ("dist_urban", "Distance to urban core", "km", "number", "derived: distance to cells with 7-cell mean built ≥ 0.3",
     "Euclidean distance to the nearest dense urban core."),
    ("dist_builtup", "Distance to built-up", "km", "number", "derived: distance to cells with built fraction ≥ 0.3",
     "Euclidean distance to the nearest substantially built-up cell."),
    ("lm_pd", "Patch density", "patches/100 ha", "number", "FRAGSTATS PD on GLC_FCS30D fine land cover",
     "Number of land-cover patches (8-connected) per 100 ha; higher = more fragmented mosaic."),
    ("lm_ed", "Edge density", "m/ha", "number", "FRAGSTATS ED on GLC_FCS30D fine land cover",
     "Length of boundaries between different land-cover classes per hectare."),
    ("lm_contag", "Contagion", "%", "number", "FRAGSTATS CONTAG on GLC_FCS30D fine land cover",
     "Aggregation of land-cover classes (0 = maximally interspersed, 100 = one contiguous class)."),
    ("dist_water", "Distance to water", "km", "number",
     "derived: distance to cells with water fraction ≥ 0.3 or JRC water occurrence ≥ 50 %",
     "Euclidean distance to the nearest river, canal or lake cell."),
    ("elevation", "Elevation", "m", "number", "SRTM GL1 (30 m), cell mean",
     "Mean terrain height; air and surface temperature fall with altitude."),
    ("slope", "Slope", "°", "number", "SRTM GL1 (30 m), cell mean",
     "Mean terrain slope; steep Aravalli ridges change solar exposure and land use."),
    ("aspect_sin", "Aspect (east-west)", "", "index", "SRTM GL1, sin(aspect) averaged at 30 m",
     "Sine of terrain aspect: +1 east-facing, −1 west-facing slopes."),
    ("aspect_cos", "Aspect (north-south)", "", "index", "SRTM GL1, cos(aspect) averaged at 30 m",
     "Cosine of terrain aspect: +1 north-facing, −1 south-facing (sun-exposed) slopes."),
    ("elev_tpi7", "Topographic position (7 km)", "m", "number", "derived: elevation − 7-cell mean",
     "Height above (+) or below (−) the surrounding 7 × 7 cells; valleys collect cold air at night."),
    ("elev_tpi21", "Topographic position (21 km)", "m", "number", "derived: elevation − 21-cell mean",
     "Height relative to the wider 21 × 21-cell landscape (ridges vs basins)."),
    ("elev_std7", "Relief (7 km std)", "m", "number", "derived: 7-cell standard deviation of elevation",
     "Local relief; high on the Aravalli ridges, ~0 on the Indo-Gangetic plain."),
    ("era5_t2m", "Air temperature (ERA5-Land)", "°C", "number", _S06_SRC_ERA5,
     "Mean 2 m air temperature; carries the regional climate gradient."),
    ("era5_vpd", "Vapour-pressure deficit", "kPa", "number", _S06_SRC_ERA5 + " (Magnus formula)",
     "Atmospheric drying power; high VPD favours dry, hot surfaces."),
    ("era5_swvl1", "Soil moisture (0–7 cm)", "m³/m³", "number", _S06_SRC_ERA5,
     "Top-soil volumetric water content; wet soils evaporate and warm slowly."),
    ("era5_evap", "Evaporation", "mm/day", "number", _S06_SRC_ERA5 + " (positive = evaporation)",
     "Total evaporation; latent heat flux that cools the surface."),
    ("era5_ssr", "Net solar radiation", "MJ/m²/day", "number", _S06_SRC_ERA5,
     "Surface net shortwave radiation; the energy that heats the surface by day."),
    ("era5_wind", "Wind speed", "m/s", "number", _S06_SRC_ERA5 + " (√(u² + v²) at 10 m)",
     "10 m wind speed; mixing reduces surface–air temperature contrasts."),
    ("era5_precip_jfm", "Winter precipitation (Jan–Mar)", "mm", "number",
     "ERA5-Land monthly total precipitation, Jan–Mar of the window years",
     "Antecedent winter rain; recharges soil moisture before the pre-monsoon heat."),
]
_S06_ACTIONABLE = {"frac_built", "frac_forest", "frac_water", "frac_cropland", "frac_barren", "ndvi",
                   "lm_pd", "lm_ed", "lm_contag"}


def _s06_feature_meta():
    """FEATURE_META in FEATURES order (base rows + focal rows derived from their base)."""
    rows = {name: (label, unit, display, source, desc) for name, label, unit, display, source, desc in _S06_META_ROWS}
    if list(rows) != BASE_FEATURES:
        raise AssertionError("FEATURE_META base rows must follow BASE_FEATURES order")
    meta = OrderedDict()
    for name in FEATURES:
        if name in rows:
            label, unit, display, source, desc = rows[name]
            base = True
        else:
            base_name, w = name.rsplit("_m", 1)
            b_label, unit, display, b_source, b_desc = rows[base_name]
            label = f"{b_label} ({w}-cell mean)"
            source = f"derived: NaN-aware {w} × {w}-cell box mean of {base_name}"
            desc = (f"Neighbourhood mean of {b_label} over {w} × {w} cells ({int(w) * GRID.res / 1000:g} km); "
                    f"captures the cooling or warming reach of the surrounding land.")
            base = False
        meta[name] = {"label": label, "unit": unit, "group": _S06_GROUP_OF[name], "description": desc,
                      "source": source, "actionable": name in _S06_ACTIONABLE, "display": display,
                      "family": _S06_FAMILY[name], "base": base}
    return meta


FEATURE_META = _s06_feature_meta()
if list(FEATURE_META) != FEATURES:
    raise AssertionError("FEATURE_META order must match FEATURES")


def _s06_feature_stats(frame):
    """Robust summary per feature over all DF rows (NaN-aware; NaN when a feature is entirely missing)."""
    summary = {}
    for name in FEATURES:
        v = frame[name].to_numpy(np.float64)
        v = v[np.isfinite(v)]
        if v.size == 0:
            summary[name] = {k: float("nan") for k in ("min", "max", "p01", "p99", "median", "mean", "std")}
            summary[name]["missing"] = 1.0
            continue
        p01, p99, med = np.percentile(v, [1, 99, 50])
        summary[name] = {"min": float(v.min()), "max": float(v.max()), "p01": float(p01), "p99": float(p99),
                         "median": float(med), "mean": float(v.mean()),
                         "std": float(v.std(ddof=1)) if v.size > 1 else 0.0,
                         "missing": round(1.0 - v.size / max(len(frame), 1), 5)}
    return summary


FEATURE_STATS = _s06_feature_stats(DF)

## 6.8 Diagnostics
* rows, targets and weights per frame; feature means per frame (shifts in the land-surface mix);
* Spearman rank correlations on a seeded sample of rows (pairs with |ρ| > 0.9 are flagged: they split SHAP credit
  arbitrarily, which is why SHAP is reported per **group** — focal means and their base share a group);
* variance inflation factors $\mathrm{VIF}_j = 1/(1-R_j^2)$ on standardised complete cases (sample);
* missingness per feature and frame.

In [ ]:
def _s06_show(obj):
    """display() inside a notebook kernel, full-width print() in plain scripts."""
    import sys

    if "ipykernel" in sys.modules:
        try:
            from IPython.display import display

            display(obj)
            return
        except Exception:  # noqa: BLE001
            pass
    print(obj.to_string() if isinstance(obj, pd.DataFrame) else obj)


def _s06_vif(data):
    """VIF per column of a complete-case float64 matrix via least squares on standardised data."""
    out = pd.Series(np.nan, index=FEATURES, name="VIF")
    if data.shape[0] < len(FEATURES) + 2:
        log(f"VIF skipped: only {data.shape[0]} complete rows", "WARNING")
        return out
    std = data.std(axis=0)
    usable = std > 0
    z = (data[:, usable] - data[:, usable].mean(axis=0)) / std[usable]
    names = [n for n, u in zip(FEATURES, usable) if u]
    for j, name in enumerate(names):
        others = np.delete(z, j, axis=1)
        design = np.column_stack([np.ones(len(z)), others])
        coef, *_ = np.linalg.lstsq(design, z[:, j], rcond=None)
        resid = z[:, j] - design @ coef
        r2 = 1.0 - resid @ resid / (z[:, j] @ z[:, j])
        out[name] = np.inf if r2 >= 1.0 - 1e-12 else 1.0 / (1.0 - r2)
    return out


def _s06_plot_spearman(corr):
    """Spearman heatmap of the 47 predictors (diverging, fixed [-1, 1]; family-coloured tick labels)."""
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap

    cmap = LinearSegmentedColormap.from_list("div", ["#2a78d6", "#f0efec", "#e34948"])
    fam_colors = {"spectral": "#15803d", "landcover": "#b45309", "urban": "#be123c", "water": "#0369a1",
                  "terrain": "#57534e", "climate": "#7c3aed"}
    fig, ax = plt.subplots(figsize=(13, 11.5))
    im = ax.imshow(corr.to_numpy(), cmap=cmap, vmin=-1, vmax=1)
    ax.set_xticks(range(len(FEATURES)))
    ax.set_xticklabels(FEATURES, rotation=90, fontsize=6.5)
    ax.set_yticks(range(len(FEATURES)))
    ax.set_yticklabels(FEATURES, fontsize=6.5)
    for lbl in list(ax.get_xticklabels()) + list(ax.get_yticklabels()):
        lbl.set_color(fam_colors.get(_S06_FAMILY.get(lbl.get_text()), "#000000"))
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Spearman ρ")
    ax.set_title("Spearman rank correlation of the 47 predictors (tick colour = family)")
    fig.tight_layout()
    fig.savefig(Path(CFG.FIG_DIR) / "06_spearman_correlation.png", dpi=140)
    plt.show()
    plt.close(fig)


def _s06_plot_maps():
    """Selected feature grids and the fixed masks of the last observed frame."""
    import matplotlib.pyplot as plt

    year = _S06_INFO["frames"][-1]
    g = FEATURE_GRIDS[year]
    panels = [("ndvi", "viridis"), ("ndvi_m15", "viridis"), ("frac_built", "magma"), ("frac_built_m15", "magma"),
              ("dist_water", "cividis"), ("elev_tpi21", "BrBG"), ("lm_ed", "plasma"), ("era5_t2m", "inferno")]
    fig, axes = plt.subplots(2, 5, figsize=(19, 7.6))
    for ax, (name, cmap) in zip(axes.ravel(), panels):
        arr = g[name]
        lo, hi = np.nanpercentile(arr, [1, 99]) if np.isfinite(arr).any() else (0, 1)
        im = ax.imshow(arr, cmap=cmap, vmin=lo, vmax=hi)
        ax.set_title(f"{name} {year}", fontsize=9)
        ax.set_axis_off()
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    for ax, mode in zip(axes.ravel()[len(panels):], CFG.MODES):
        show = np.where(ROI_MASK, FIXED_MASK[mode].astype(float), np.nan)
        ax.imshow(show, cmap="Greys", vmin=-0.3, vmax=1.0)
        ax.set_title(f"FIXED_MASK[{mode}] ({_S06_INFO['fixed_mask'][mode]['share']:.1%} of ROI)", fontsize=9)
        ax.set_axis_off()
    fig.suptitle(f"Feature grids ({GRID.res} m grid; focal windows in cells)", fontsize=11)
    fig.tight_layout()
    fig.savefig(Path(CFG.FIG_DIR) / "06_feature_maps.png", dpi=120)
    plt.show()
    plt.close(fig)


def _s06_run_diagnostics(frame, max_rows=60000):
    """Print per-frame tables, correlations, VIF and missingness; returns the tables."""
    Path(CFG.FIG_DIR).mkdir(parents=True, exist_ok=True)
    per_frame = pd.DataFrame([
        {"year": y, **{f"{m}_{k}": v for m in CFG.MODES for k, v in _S06_INFO["targets"][y][m].items()},
         "level_day_raw": RAW_LEVELS["day"].get(y), "level_night_raw": RAW_LEVELS["night"].get(y)}
        for y in _S06_INFO["frames"]])
    print("Targets per frame (valid / training cells, mean weight, raw fixed-mask level °C):")
    _s06_show(per_frame.round(3))
    means = frame.groupby("year")[BASE_FEATURES].mean().T
    print("Base-feature means per frame:")
    _s06_show(means.round(3))

    rng = np.random.default_rng(CFG.SEED)
    sample = frame.iloc[np.sort(rng.choice(len(frame), size=min(max_rows, len(frame)), replace=False))]
    corr = sample[FEATURES].corr(method="spearman")
    high = [(a, b, float(corr.at[a, b])) for i, a in enumerate(FEATURES) for b in FEATURES[i + 1:]
            if np.isfinite(corr.at[a, b]) and abs(corr.at[a, b]) > 0.9]
    for a, b, rho in high:
        same = _S06_GROUP_OF[a] == _S06_GROUP_OF[b]
        log(f"06: highly correlated pair {a} ~ {b} (Spearman rho = {rho:+.2f}; "
            f"{'same SHAP group' if same else 'DIFFERENT SHAP groups'})", "INFO" if same else "WARNING")
    if not high:
        log("06: no predictor pair exceeds |Spearman rho| = 0.9")
    _s06_plot_spearman(corr)

    data = sample[FEATURES].to_numpy(np.float64)
    data = data[np.isfinite(data).all(axis=1)][:30000]
    vif = _s06_vif(data)
    missing = frame.groupby("year")[FEATURES].apply(lambda g: g.isna().mean()).T
    missing["overall"] = frame[FEATURES].isna().mean()
    summary = pd.DataFrame({"family": [_S06_FAMILY[f] for f in FEATURES], "group": [_S06_GROUP_OF[f] for f in FEATURES],
                            "VIF": vif, "missing_overall": missing["overall"]}, index=FEATURES)
    print("VIF (complete-case sample) and missing share per feature:")
    _s06_show(summary.round(3))
    high_vif = [n for n, v in vif.items() if np.isfinite(v) and v > 10]
    if high_vif:
        log(f"06: VIF > 10 for {high_vif} (expected for focal means of smooth fields; SHAP is grouped)")
    try:
        _s06_plot_maps()
    except Exception as exc:  # noqa: BLE001 - a figure must never stop the pipeline
        log(f"06: feature-map figure failed: {exc!r}", "WARNING")
    return {"per_frame": per_frame, "spearman": corr, "high_corr_pairs": high, "vif": vif, "missing": missing}


with timer("06_diagnostics"):
    _S06_DIAGNOSTICS = _s06_run_diagnostics(DF)

## 6.9 Contract check (SPEC §3.3) and hand-off
The check enforces what later sections rely on: exact feature order, dtypes, canonical row order, one fold per
cell in every frame, weights > 0 only on valid targets of training frames, zero-mean anomalies over the fixed mask
of every training frame, and focal/distance grids consistent with `FOCAL_RECOMPUTE`. The table is also written to
`features_long.parquet` (zstd) as the reproducible hand-off.

In [ ]:
def _s06_contract_check():
    """Assert the SPEC §3.3 contract of the section-06 globals (raises AssertionError on a violation)."""
    frames = [int(y) for y in CFG.FRAMES_OBS]
    assert FEATURES == BASE_FEATURES + _S06_FOCAL_FEATURES and len(FEATURES) == 47
    assert list(FEATURE_META) == FEATURES and all({"family", "group", "base", "actionable"} <= set(v)
                                                  for v in FEATURE_META.values())
    assert sum(v["actionable"] for v in FEATURE_META.values()) == 9
    assert list(SHAP_GROUPS) == [g[0] for g in _S06_GROUP_ROWS] and list(GROUP_META) == list(SHAP_GROUPS)
    assert sorted(FEATURE_GRIDS) == frames
    for y in frames:
        assert list(FEATURE_GRIDS[y]) == FEATURES
        for name, arr in FEATURE_GRIDS[y].items():
            assert arr.shape == ROI_MASK.shape and arr.dtype == np.float32, (y, name, arr.shape, arr.dtype)
            assert not np.isfinite(arr[~ROI_MASK]).any(), f"{name} {y} finite outside the ROI"
    expected_cols = list(_S06_DF_DTYPES)
    assert list(DF.columns) == expected_cols, "DF columns differ from the contract"
    assert isinstance(DF.index, pd.RangeIndex) and DF.index.start == 0
    assert len(DF) == N_CELLS * len(frames)
    for col, dt in _S06_DF_DTYPES.items():
        assert str(DF[col].dtype) == dt, (col, DF[col].dtype, dt)
    year = DF["year"].to_numpy()
    cell = DF["cell_id"].to_numpy()
    assert np.all(np.diff(year) >= 0) and np.array_equal(year, np.repeat(frames, N_CELLS))
    assert np.array_equal(cell, np.tile(CELL_IDS, len(frames))), "cells must follow the canonical order per frame"
    by_cell = DF.groupby("cell_id")[["fold_block5", "fold_block20", "block10_id", "block20_id", "fold_district"]]
    assert int(by_cell.nunique().max().max()) == 1, "a cell must keep its folds/blocks in every frame"
    assert DF["fold_block20"].between(0, CFG.N_FOLDS_BLOCK - 1).all() and DF["fold_block5"].between(0, 4).all()
    assert DF["fold_random"].between(0, CFG.N_FOLDS - 1).all()
    b20 = DF.groupby("block20_id")["fold_block20"].nunique()
    assert int(b20.max()) == 1, "every 20 km block must map to exactly one fold"
    night_train = set(CFG.NIGHT_TRAIN_FRAMES)
    for mode in CFG.MODES:
        w = DF[f"w_{mode}"].to_numpy()
        a = DF[f"anom_{mode}"].to_numpy()
        lst = DF[f"lst_{mode}"].to_numpy()
        assert np.all(w >= 0) and np.all(np.isfinite(lst[w > 0])) and np.all(np.isfinite(a[w > 0]))
        assert np.array_equal(np.isfinite(a), np.isfinite(lst) & np.isin(year, [y for y in frames
                                                                               if np.isfinite(RAW_LEVELS[mode][y])]))
        if mode == "night":
            assert not np.any(w[~np.isin(year, list(night_train))] > 0), "night weights outside NIGHT_TRAIN_FRAMES"
        fixed_row = FIXED_MASK[mode][DF["row"].to_numpy(), DF["col"].to_numpy()]
        for y in frames:
            sel = (year == y) & fixed_row & np.isfinite(a)
            if sel.any():
                assert abs(float(np.mean(a[sel].astype(np.float64)))) < 1e-3, f"{mode} {y} anomaly mean != 0"
        assert FIXED_MASK[mode].dtype == bool and not FIXED_MASK[mode][~ROI_MASK].any()
        assert sorted(RAW_LEVELS[mode]) == frames
        # D8: strict intersection over >= 2 mask frames; excluded training frames carry no weight
        fm = _S06_INFO["fixed_mask"][mode]
        assert len(fm["frames"]) >= 2 and FIXED_MASK[mode].any(), (mode, fm["frames"])
        assert all(fm["coverage"][y] >= fm["min_coverage"] for y in fm["frames"])
        valid_grid = np.zeros(ROI_MASK.shape, bool)
        inter = ROI_MASK.copy()
        for y in fm["frames"]:
            valid_grid[:] = False
            sel = (year == y) & np.isfinite(lst)
            valid_grid[DF["row"].to_numpy()[sel], DF["col"].to_numpy()[sel]] = True
            inter &= valid_grid
        assert np.array_equal(inter, FIXED_MASK[mode]), f"FIXED_MASK[{mode}] is not the strict intersection"
        for y in fm["excluded_frames"]:
            assert not np.any(w[year == y] > 0), f"{mode} {y}: excluded training frame has weight"
        cov = FIXED_MASK_COVERAGE[mode]
        assert sorted(cov) == frames and all(0.0 <= cov[y] <= 1.0 for y in frames)
        assert all(cov[y] == 1.0 for y in fm["frames"]), (mode, cov)
        n_fixed = int(FIXED_MASK[mode].sum())
        for y in frames:
            sel = (year == y) & fixed_row & np.isfinite(lst)
            assert abs(cov[y] - int(sel.sum()) / n_fixed) < 1e-9, (mode, y, cov[y])
    if "day" in CFG.MODES:
        n_day = [int(np.isfinite(DF["lst_day"].to_numpy()[year == y]).sum()) for y in frames]
        assert min(n_day) > 0, f"a day frame has no valid target ({dict(zip(frames, n_day))})"
    y_last = frames[-1]
    check = FOCAL_RECOMPUTE({k: FEATURE_GRIDS[y_last][k] for k in BASE_FEATURES})
    for name in _S06_FOCAL_FEATURES + ["dist_water", "dist_urban", "dist_builtup"]:
        assert np.allclose(check[name], FEATURE_GRIDS[y_last][name], equal_nan=True, atol=1e-5), name
    log("06: contract check passed (SPEC §3.3)")


_s06_contract_check()


def FEATURE_MATRIX(grids, rows=None, cols=None):
    """(n, 47) float32 design matrix in ``FEATURES`` order sampled from a dict of (H, W) feature grids.

    ``rows``/``cols`` default to the canonical cell order (``CELL_ROWS``/``CELL_COLS``), so
    ``FEATURE_MATRIX(FEATURE_GRIDS[y])`` equals ``DF.loc[DF.year == y, FEATURES]``.
    """
    r = CELL_ROWS if rows is None else np.asarray(rows)
    c = CELL_COLS if cols is None else np.asarray(cols)
    return np.ascontiguousarray(np.column_stack([np.asarray(grids[f], np.float32)[r, c] for f in FEATURES]),
                                dtype=np.float32)


def _s06_save_features(frame):
    out_dir = Path(CFG.PARQUET_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / "features_long.parquet"
    try:
        frame.to_parquet(path, engine="pyarrow", compression="zstd", index=False)
        log(f"06: saved {path} ({path.stat().st_size / 1e6:.1f} MB)")
    except Exception as exc:  # noqa: BLE001 - the hand-off file is a convenience, not a dependency
        log(f"06: could not write {path}: {exc!r}", "WARNING")
    return path


with timer("06_save_parquet"):
    _s06_save_features(DF)
print(f"LM_VALIDATION: status={LM_VALIDATION.get('status')} n={LM_VALIDATION.get('n')} "
      f"max_rel_diff={LM_VALIDATION.get('max_rel_diff')}")
free_memory()

# 7. Models, validation suite, final models, persistent residual and area of applicability

One pooled model per mode (day, night) predicts the **anomaly** target of section 06 from the 47 features (no year
feature, no coordinates). This section answers four questions and leaves the objects later sections need:

| Question | Answer here | Used by |
|---|---|---|
| How well does the model transfer to unseen **places**? | pooled out-of-fold (OOF) metrics under 20 km-block spatial CV (`block20`, headline), v1's 5 × 5 tiles (`block5`), leave-one-district-out (`district`), the block-size curve and error-vs-distance | metrics.json, Model Lab |
| ... to unseen **times**? | leave-one-epoch-out (`loeo`, raw and **recentred**) and unseen place **and** time (`place_time`) | timeline headline, hindcast credibility |
| What is the final model? | `FINAL_MODELS[m]` (XGBoost, monotone constraints, rounds from block20 early stopping) and `PREDICT_ANOM` | 08 SHAP, 09 hindcast, 11 projections |
| Where can it be trusted, and how wrong is it there? | `PERSISTENT_RESID` (time transfer only), `AOA[m]` + `COMPUTE_DI` (Meyer & Pebesma 2021) and a conformal error model by DI decile | 09 intervals, 10/11 flags, 11 Monte Carlo |

**Validation design (SPEC §4.2).** Every scheme scores *pooled* OOF predictions — $R^2 = 1 - \sum w(y-\hat y)^2 /
\sum w (y - \bar y_w)^2$ over all held-out rows, with the sample weights of section 06 — and reports the per-fold
spread separately. Pooling avoids v1's mistake of averaging per-fold $R^2$, which is dominated by a fold whose
own variance happens to be small. Folds are spatial blocks that keep a cell (and its whole block) in one fold in
every frame, so the same place in another year never helps predict itself.

| Scheme | Split | Models | Question |
|---|---|---|---|
| `block20` | 20 km blocks → `N_FOLDS_BLOCK` folds | all `CFG.BENCHMARKS` | headline spatial transfer (≈ residual correlation range) |
| `block5` | v1 5 × 5 tiles → 5 folds | xgboost, lightgbm, gnn | comparability with v1 |
| `district` | leave one district out (< 200 rows merged into the nearest district) | xgboost | administrative transfer |
| `loeo` | leave one training frame out; recentred over the held-out frame's fixed mask | all | temporal transfer (timeline headline) |
| `place_time` | train frames < last & folds ≠ k → test last frame & fold k | xgboost | unseen place **and** time |
| `random` | 5 random row folds | xgboost, random_forest, ridge | optimistic upper bound |

**Recentring.** The anomaly target has zero mean over the fixed mask *by definition*, so subtracting the mean of a
predicted anomaly map over the same cells uses predictions only and is legitimate. It removes the frame-wide offset
that land-cover product drift or sensor changes induce in temporal transfer (real data: LOEO $R^2$ 0.80 → 0.89).

**Leakage controls** (asserted at run time): test and training rows are disjoint; spatial schemes share no cell
and no block between train and test; LOEO never trains on the held-out frame; early stopping uses whole 10 km blocks
drawn from the *training* rows only; imputation medians and standardisation come from the training rows only.

**Compute layout.** All (scheme × model × fold) fits of a mode are one job list split into lanes that run
concurrently: the **GPU lane** spreads XGBoost-CUDA, GraphSAGE and cuML-RF jobs over the GPUs with `gpu_pool_map`
(one job per T4 at a time); the **CPU lane** runs ridge and scikit-learn RF (and LightGBM on the CPU); when the
LightGBM GPU probe succeeds, LightGBM gets its own serial **lgbm lane** on the last GPU (`gpu_id = n_gpus − 1`; at
most one LightGBM booster alive at a time, its OpenCL learner is not thread safe). Identical fits requested by two
schemes (e.g. the 20 km point of the block-size curve when `N_FOLDS_BLOCK = 5`) run once. Every job is
stage-cached individually, so an interrupted Kaggle session resumes where it stopped; each lane stops scheduling
new jobs when the time budget (section 00) would be exceeded and the section then stops cleanly.

In [ ]:
_lst_enter_section('07')  # section marker (generated by build_notebook.py)

In [ ]:
import hashlib as _s07_hashlib
import math as _s07_math
import os as _s07_os
import threading as _s07_threading
import time as _s07_time
import traceback as _s07_traceback
import warnings as _s07_warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse as _s07_sparse
from scipy.spatial import cKDTree as _s07_cKDTree

_S07_VERSION = "07-v2.1"          # final model / AOA / residual models (07_post)
# CV fits themselves are unchanged since v2.0: their identities (job de-duplication and the 07_job cache) keep
# the job-recipe version, so a scheduling-only change reuses the cached fits.
_S07_JOB_VERSION = "07-v2.0"
_S07_SCHEMES = ["block20", "block5", "district", "loeo", "place_time", "random"]
_S07_SCHEME_MODELS = {
    "block20": list(CFG.BENCHMARKS),
    "block5": [m for m in ("xgboost", "lightgbm", "gnn") if m in CFG.BENCHMARKS],
    "district": ["xgboost"],
    "loeo": list(CFG.BENCHMARKS),
    "place_time": ["xgboost"],
    "random": [m for m in ("xgboost", "random_forest", "ridge") if m in CFG.BENCHMARKS],
}
_S07_CURVE_KM = [5, 10, 20, 40, 66]
_S07_CURVE_FOLDS = 5
_S07_DIST_BINS = [0.0, 2.0, 5.0, 10.0, 20.0, 40.0, float("inf")]
_S07_LAMBDAS = [0.5, 1.0, 2.0, 3.0, 5.0, 10.0, float("inf")]
_S07_DISTRICT_MIN_ROWS = 200
_S07_AOA_SAMPLE = 20000
_S07_CONFORMAL_ROWS = 100000
_S07_RESID_SAMPLES = 5000
_S07_RF_TREES = 60 if CFG.PROFILE == "fast" else 300
# scikit-learn RF is the slowest CPU model (~1-2 s per tree on 400k rows). Each fold's forest is fitted on at most
# this many training rows (seeded subsample; each tree then bootstraps max_samples = 50 % of them).
_S07_RF_MAX_ROWS = int(_s07_os.environ.get("LST_RF_MAX_ROWS", "120000"))
# CPU fallback only: wall-clock budget per GNN fold (the GPU path always runs CFG.GNN_EPOCHS with early stopping).
_S07_GNN_CPU_BUDGET_S = float(_s07_os.environ.get("LST_GNN_CPU_BUDGET_S", "900"))
_S07_RUN_ABLATION = _s07_os.environ.get("LST_ABLATION", "1").strip().lower() not in ("0", "false", "no", "off")
_S07_LGBM_PARAMS = {"learning_rate": 0.03, "num_leaves": 63, "subsample": 0.8, "subsample_freq": 1,
                    "colsample_bytree": 0.8, "min_child_samples": 20, "reg_lambda": 1.0, "verbose": -1}
_S07_V1_REFERENCE = {"v1_metric": "mean of per-fold R² (5×5)", "v1_xgb": [0.623, 0.305], "v1_gnn": [0.823, 0.048]}

# Monotone constraints (SPEC §4.3): physically certain signs only. Day: vegetation, tree cover and water cool the
# mid-morning surface; night: built-up fraction and building volume (heat storage) warm the night surface.
_S07_MONOTONE = {
    "day": {f: -1 for f in ("ndvi", "ndvi_m3", "ndvi_m15", "frac_forest", "frac_forest_m3", "frac_forest_m15",
                            "frac_water", "frac_water_m3", "frac_water_m15", "mndwi")},
    "night": {f: 1 for f in ("built_v", "frac_built", "frac_built_m3", "frac_built_m15")},
}

# Shared read-only design data (row positions everywhere are DF row positions).
_S07_X = np.ascontiguousarray(DF[FEATURES].to_numpy(np.float32))
_S07_N = len(DF)
_S07_YEAR = DF["year"].to_numpy(np.int64)
_S07_CELL = DF["cell_id"].to_numpy(np.int64)
_S07_ROW = DF["row"].to_numpy(np.int64)
_S07_COL = DF["col"].to_numpy(np.int64)
_S07_XY = DF[["x", "y"]].to_numpy(np.float64)
_S07_BLOCK10 = DF["block10_id"].to_numpy(np.int64)
_S07_BLOCK20 = DF["block20_id"].to_numpy(np.int64)
_S07_FOLD20 = DF["fold_block20"].to_numpy(np.int64)
_S07_FRAMES_DF = sorted(int(y) for y in np.unique(_S07_YEAR))
_S07_CELL_POS = np.searchsorted(CELL_IDS, _S07_CELL).astype(np.int64)    # canonical cell index of every row
_S07_BASE_IDX = np.array([FEATURES.index(f) for f in BASE_FEATURES], dtype=np.int64)
_S07_TORCH_LOCK = _s07_threading.Lock()
_S07_PRINT_LOCK = _s07_threading.Lock()
if not np.array_equal(CELL_IDS[_S07_CELL_POS], _S07_CELL):
    raise RuntimeError("07: DF cell ids are not in the canonical cell set")
log(f"07: design matrix {_S07_X.shape} float32 ({_S07_X.nbytes / 2**20:.0f} MiB); benchmarks {CFG.BENCHMARKS}; "
    f"XGBoost on {ENV.get('xgb_device')} ({ENV.get('n_gpus', 0)} GPU(s)); RF trees {_S07_RF_TREES} "
    f"(<= {_S07_RF_MAX_ROWS:,} rows/fold)")


def _s07_hash(*parts):
    """Short SHA-1 of arrays / JSON-able parts (used for stage-cache keys and job de-duplication)."""
    h = _s07_hashlib.sha1()
    for p in parts:
        if isinstance(p, np.ndarray):
            h.update(str(p.dtype).encode())
            h.update(np.ascontiguousarray(p).tobytes())
        else:
            h.update(repr(p).encode())
        h.update(b"|")
    return h.hexdigest()[:20]


def _s07_df_fingerprint():
    """Fingerprint of everything in DF the models see (features, targets, weights, folds)."""
    cols = ["anom_day", "anom_night", "w_day", "w_night", "fold_block5", "fold_block20", "fold_random",
            "block10_id", "block20_id", "fold_district"]
    sums = [float(np.nansum(_S07_X[:, j], dtype=np.float64)) for j in range(_S07_X.shape[1])]
    sums += [float(np.nansum(DF[c].to_numpy(np.float64))) for c in cols]
    counts = [int(np.isfinite(_S07_X).sum())] + [int(np.isfinite(DF[c].to_numpy(np.float64)).sum()) for c in cols]
    return _s07_hash(FEATURES, _S07_N, np.round(np.asarray(sums), 6), counts)


_S07_DF_FP = _s07_df_fingerprint()

## 7.1 Mode context, folds and leakage checks
A mode's training rows are the rows with weight > 0 (day: every observed frame; night: `NIGHT_TRAIN_FRAMES`).
Fold generators return DF row positions; `_s07_check_fold` asserts the leakage rules for every fold before any fit.

In [ ]:
def _s07_context(mode):
    """Per-mode targets, weights, training rows, fixed-mask rows and monotone constraints."""
    y = DF[f"anom_{mode}"].to_numpy(np.float32)
    w = DF[f"w_{mode}"].to_numpy(np.float32)
    train = np.flatnonzero(w > 0).astype(np.int64)
    if train.size and not np.all(np.isfinite(y[train])):
        raise RuntimeError(f"07: {mode} has training rows without a finite anomaly target")
    frames = sorted(int(v) for v in np.unique(_S07_YEAR[train]))
    fixed_row = FIXED_MASK[mode][_S07_ROW, _S07_COL]
    mono = np.array([_S07_MONOTONE.get(mode, {}).get(f, 0) for f in FEATURES], dtype=np.int64)
    return {"mode": mode, "y": y, "w": w, "train": train, "frames": frames, "fixed_row": fixed_row,
            "mono": mono, "gnn": None}


def _s07_greedy(block_ids_of_cells, n_folds):
    """Greedy size-balanced block → fold map (largest block first → smallest fold), on ROI cell counts."""
    uniq, counts = np.unique(block_ids_of_cells, return_counts=True)
    order = sorted(range(uniq.size), key=lambda i: (-counts[i], uniq[i]))
    load = np.zeros(int(n_folds), dtype=np.int64)
    fold_of = {}
    for i in order:
        k = int(np.argmin(load))
        fold_of[int(uniq[i])] = k
        load[k] += counts[i]
    return fold_of


def _s07_block_fold_rows(block_cells, n_folds):
    """(N_DF,) fold of every DF row for square blocks of ``block_cells`` cells (greedy on ROI cells)."""
    b = max(1, int(block_cells))
    ncb = _s07_math.ceil(GRID.width / b)
    cell_block = (CELL_ROWS.astype(np.int64) // b) * ncb + CELL_COLS.astype(np.int64) // b
    fold_of = _s07_greedy(cell_block, n_folds)
    lut = np.vectorize(fold_of.get, otypes=[np.int64])(cell_block)
    return lut[_S07_CELL_POS], (_S07_ROW // b) * ncb + _S07_COL // b


def _s07_district_fold(ctx):
    """Leave-one-district-out folds; districts with < 200 training rows join the nearest district's fold."""
    tr = ctx["train"]
    d_rows = DF["fold_district"].to_numpy(np.int64)[tr]
    n_d = len(DISTRICTS)
    counts = np.bincount(d_rows, minlength=n_d)
    cell_d = DISTRICT_IDX[CELL_ROWS, CELL_COLS].astype(np.int64)
    X, Y = GRID.cell_centers_xy()
    cx = np.bincount(cell_d, weights=np.asarray(X)[CELL_ROWS, CELL_COLS], minlength=n_d)
    cy = np.bincount(cell_d, weights=np.asarray(Y)[CELL_ROWS, CELL_COLS], minlength=n_d)
    nc = np.maximum(np.bincount(cell_d, minlength=n_d), 1)
    cx, cy = cx / nc, cy / nc
    big = [i for i in range(n_d) if counts[i] >= _S07_DISTRICT_MIN_ROWS]
    if len(big) < 2:
        log(f"07: {ctx['mode']}: fewer than 2 districts with >= {_S07_DISTRICT_MIN_ROWS} rows - district CV skipped",
            "WARNING")
        return None, []
    lut = np.arange(n_d, dtype=np.int64)
    merges = []
    for i in range(n_d):
        if 0 < counts[i] < _S07_DISTRICT_MIN_ROWS:
            j = min(big, key=lambda b: (cx[b] - cx[i]) ** 2 + (cy[b] - cy[i]) ** 2)
            lut[i] = j
            merges.append({"district": int(i), "into": int(j), "rows": int(counts[i])})
    if merges:
        log(f"07: {ctx['mode']}: districts merged into their nearest neighbour fold: {merges}")
    return lut[d_rows], merges


def _s07_folds(ctx, scheme):
    """List of ``{"fold", "train", "test"}`` (DF row positions) for one scheme of a mode."""
    tr = ctx["train"]
    out = []
    if tr.size == 0:
        return out
    yr = _S07_YEAR[tr]
    if scheme in ("block20", "block5", "random"):
        col = {"block20": "fold_block20", "block5": "fold_block5", "random": "fold_random"}[scheme]
        f = DF[col].to_numpy(np.int64)[tr]
        for k in np.unique(f):
            out.append({"fold": int(k), "train": tr[f != k], "test": tr[f == k]})
    elif scheme == "district":
        f, merges = _s07_district_fold(ctx)
        ctx["district_merges"] = merges
        if f is not None:
            for k in np.unique(f):
                out.append({"fold": int(k), "train": tr[f != k], "test": tr[f == k]})
    elif scheme == "loeo":
        if len(ctx["frames"]) >= 2:
            for t in ctx["frames"]:
                out.append({"fold": int(t), "train": tr[yr != t], "test": tr[yr == t]})
    elif scheme == "place_time":
        if len(ctx["frames"]) >= 2:
            last = ctx["frames"][-1]
            f = _S07_FOLD20[tr]
            for k in np.unique(f[yr == last]):
                out.append({"fold": int(k), "train": tr[(yr < last) & (f != k)], "test": tr[(yr == last) & (f == k)]})
    else:
        raise ValueError(f"unknown scheme {scheme!r}")
    return [fd for fd in out if fd["train"].size > 0 and fd["test"].size > 0]


def _s07_check_fold(ctx, scheme, fold):
    """Assert the leakage rules of one fold (raises AssertionError)."""
    tr, te = fold["train"], fold["test"]
    mark = np.zeros(_S07_N, dtype=bool)
    mark[tr] = True
    assert not mark[te].any(), f"{scheme} fold {fold['fold']}: train and test rows overlap"
    assert np.all(ctx["w"][tr] > 0) and np.all(ctx["w"][te] > 0), f"{scheme}: non-training rows in a fold"
    if scheme in ("block20", "block5", "district", "place_time") or scheme.startswith("curve_"):
        shared = np.intersect1d(np.unique(_S07_CELL[tr]), np.unique(_S07_CELL[te]), assume_unique=True)
        assert shared.size == 0, f"{scheme} fold {fold['fold']}: {shared.size} cells in both train and test"
    if scheme in ("block20", "place_time"):
        assert np.intersect1d(_S07_BLOCK20[tr], _S07_BLOCK20[te]).size == 0, f"{scheme}: a 20 km block is split"
    if scheme == "loeo":
        test_years = np.unique(_S07_YEAR[te])
        assert test_years.size == 1 and test_years[0] not in set(_S07_YEAR[tr].tolist()), \
            f"loeo fold {fold['fold']} trains on the held-out frame"
    if scheme == "place_time":
        last = ctx["frames"][-1]
        assert np.all(_S07_YEAR[te] == last) and np.all(_S07_YEAR[tr] < last), "place_time frame rule violated"


def _s07_inner_split(train_idx, val_share=0.10):
    """(fit_idx, val_idx) holding out whole 10 km blocks (~``val_share`` of the TRAINING rows) for early stopping.

    Blocks are drawn in a seeded random order until the share is reached (always keeping one block for fitting);
    with a single block the split falls back to a seeded random row split. Only training rows are ever used.
    """
    rng = np.random.default_rng(CFG.SEED)
    groups = _S07_BLOCK10[train_idx]
    uniq, counts = np.unique(groups, return_counts=True)
    if uniq.size < 2:
        is_val = np.zeros(train_idx.size, dtype=bool)
        is_val[rng.permutation(train_idx.size)[: max(1, int(round(val_share * train_idx.size)))]] = True
    else:
        order = rng.permutation(uniq.size)
        cum = np.cumsum(counts[order])
        n_take = min(int(np.searchsorted(cum, val_share * train_idx.size) + 1), uniq.size - 1)
        is_val = np.isin(groups, uniq[order[:n_take]])
    fit_idx, val_idx = train_idx[~is_val], train_idx[is_val]
    assert fit_idx.size and val_idx.size and np.intersect1d(_S07_BLOCK10[fit_idx], _S07_BLOCK10[val_idx]).size == 0 \
        or uniq.size < 2, "inner split must hold out whole 10 km blocks"
    return fit_idx, val_idx


def _s07_wmetrics(y, p, w):
    """Weighted pooled R², RMSE, MAE and bias (mean of prediction − observation)."""
    y, p, w = (np.asarray(a, np.float64) for a in (y, p, w))
    ok = np.isfinite(y) & np.isfinite(p) & np.isfinite(w) & (w > 0)
    if ok.sum() < 2:
        return {"r2": np.nan, "rmse": np.nan, "mae": np.nan, "bias": np.nan, "n": int(ok.sum())}
    y, p, w = y[ok], p[ok], w[ok]
    sw = w.sum()
    r = y - p
    ybar = float((w * y).sum() / sw)
    sse = float((w * r * r).sum())
    sst = float((w * (y - ybar) ** 2).sum())
    return {"r2": 1.0 - sse / sst if sst > 0 else np.nan, "rmse": float(np.sqrt(sse / sw)),
            "mae": float((w * np.abs(r)).sum() / sw), "bias": float((w * (p - y)).sum() / sw), "n": int(ok.sum())}

## 7.2 Model implementations
Each fitter receives the job (mode context, training/test rows, optional feature subset) and a GPU id and returns
test predictions (anomaly °C) plus an info dict (`device`, `best_iteration` = boosting rounds or GNN epoch).
Sample weights enter every loss. XGBoost and the GNN retry a fold on the CPU after a **GPU** out-of-memory error;
LightGBM and cuML fall back to the CPU on any GPU error.

| Model | Details |
|---|---|
| ridge | median imputation + standardisation + `RidgeCV` (α ∈ 10⁻³…10³, efficient GCV) |
| random_forest | scikit-learn, 300 trees (60 in the fast profile), `max_features=0.33`, `min_samples_leaf=5`, `max_samples=0.5`; cuML only with `USE_RAPIDS` |
| lightgbm | v1 parameters (63 leaves, η 0.03), monotone constraints, inner-block early stopping; GPU only if the probe succeeds |
| xgboost | SPEC §1.4 (depth 6, min_child_weight 20, η 0.03, subsample/colsample 0.8, λ 1), `hist` on `cuda:k`, monotone constraints, inner 10 km early stopping |
| gnn | v1 pure-PyTorch GraphSAGE (3 layers, 128 hidden) on all 47 features, k = 8 neighbours within the same frame, weighted MSE, early stopping on inner blocks |

In [ ]:
def _s07_take(idx, feat_idx=None):
    """Rows (and optionally a column subset) of the design matrix as a C-contiguous float32 copy."""
    x = _S07_X[idx]
    return np.ascontiguousarray(x if feat_idx is None else x[:, feat_idx])


def _s07_train_medians(idx, feat_idx=None):
    """Training-row medians (0 for all-NaN columns) used for imputation."""
    with _s07_warnings.catch_warnings():
        _s07_warnings.simplefilter("ignore", RuntimeWarning)
        med = np.nanmedian(_s07_take(idx, feat_idx), axis=0)
    return np.where(np.isfinite(med), med, 0.0).astype(np.float32)


def _s07_impute(x, medians):
    return np.where(np.isnan(x), medians, x).astype(np.float32)


def _s07_fit_ridge(ctx, job, gpu_id):
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import RidgeCV
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    tr, te = job["train"], job["test"]
    model = Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
                      ("scale", StandardScaler()), ("reg", RidgeCV(alphas=np.logspace(-3, 3, 13)))])
    with _s07_warnings.catch_warnings():
        _s07_warnings.simplefilter("ignore", UserWarning)
        model.fit(_s07_take(tr, job["feat_idx"]), ctx["y"][tr], reg__sample_weight=ctx["w"][tr])
        pred = model.predict(_s07_take(te, job["feat_idx"]))
    return pred.astype(np.float32), {"device": "cpu", "best_iteration": None,
                                     "alpha": float(model.named_steps["reg"].alpha_)}


def _s07_fit_random_forest(ctx, job, gpu_id):
    """scikit-learn RF (cuML on the GPU only with RAPIDS) on train-median-imputed features."""
    tr, te = job["train"], job["test"]
    capped = tr
    if tr.size > _S07_RF_MAX_ROWS:
        capped = np.sort(np.random.default_rng(CFG.SEED).choice(tr, size=_S07_RF_MAX_ROWS, replace=False))
    medians = _s07_train_medians(capped, job["feat_idx"])
    x_train = _s07_impute(_s07_take(capped, job["feat_idx"]), medians)
    x_test = _s07_impute(_s07_take(te, job["feat_idx"]), medians)
    info = {"best_iteration": None, "rows_used": int(capped.size)}
    if ENV.get("has_cuml"):
        try:
            import cupy as _cp
            from cuml.ensemble import RandomForestRegressor as _CuRF

            with _cp.cuda.Device(gpu_id):
                model = _CuRF(n_estimators=_S07_RF_TREES, max_features=0.33, min_samples_leaf=5, max_samples=0.5,
                              max_depth=32, n_bins=128, n_streams=1, random_state=CFG.SEED)
                model.fit(x_train, ctx["y"][capped])            # cuML RF has no sample weights
                pred = np.asarray(model.predict(x_test), dtype=np.float32)
                del model
                _cp.get_default_memory_pool().free_all_blocks()
            ENV.setdefault("rapids_used", set()).add("cuml_random_forest")
            return pred, {**info, "device": f"cuda:{gpu_id}", "weights": "unweighted (cuML)"}
        except Exception as exc:  # noqa: BLE001 - any cuML failure -> scikit-learn
            log(f"cuML random forest failed on GPU {gpu_id} ({exc!r}); falling back to scikit-learn", "WARNING")
            free_memory()
    from sklearn.ensemble import RandomForestRegressor

    model = RandomForestRegressor(n_estimators=_S07_RF_TREES, max_features=0.33, min_samples_leaf=5, max_samples=0.5,
                                  n_jobs=int(ENV.get("cpu_count") or -1), random_state=CFG.SEED)
    model.fit(x_train, ctx["y"][capped], sample_weight=ctx["w"][capped])
    pred = model.predict(x_test).astype(np.float32)
    del model
    return pred, {**info, "device": "cpu"}


def _s07_resolve_lgbm_device():
    """Probe LightGBM GPU support once per session (tiny fit) and cache the answer in ENV['lgbm_device']."""
    if globals().get("_S07_LGBM_PROBED"):
        return ENV["lgbm_device"]
    device = "cpu"
    if ENV.get("n_gpus", 0) > 0 and "lightgbm" in CFG.BENCHMARKS:
        try:
            import lightgbm as lgb

            rng = np.random.default_rng(0)
            lgb.LGBMRegressor(n_estimators=2, device="gpu", verbose=-1).fit(rng.normal(size=(256, 4)),
                                                                            rng.normal(size=256))
            device = "gpu"
        except Exception as exc:  # noqa: BLE001
            log(f"LightGBM GPU probe failed ({type(exc).__name__}: {str(exc)[:120]}); LightGBM uses the CPU")
    ENV["lgbm_device"] = device
    globals()["_S07_LGBM_PROBED"] = True
    return device


def _s07_lgbm_eval_kwargs(x_val, y_val, w_val):
    """Validation-set keyword arguments: ``eval_X``/``eval_y`` on LightGBM >= 4.6, ``eval_set`` before."""
    import inspect

    import lightgbm as lgb

    params = inspect.signature(lgb.LGBMRegressor.fit).parameters
    if "eval_X" in params:
        return {"eval_X": (x_val,), "eval_y": (y_val,), "eval_sample_weight": [w_val]}
    return {"eval_set": [(x_val, y_val)], "eval_sample_weight": [w_val]}


def _s07_fit_lightgbm(ctx, job, gpu_id):
    import lightgbm as lgb

    fit_idx, val_idx = _s07_inner_split(job["train"])
    feat = job["feat_idx"]
    mono = ctx["mono"] if feat is None else ctx["mono"][feat]
    device = ENV.get("lgbm_device") or "cpu"
    for attempt in ("primary", "cpu_fallback"):
        try:
            params = dict(_S07_LGBM_PARAMS, n_estimators=int(CFG.XGB_MAX_ROUNDS), random_state=int(CFG.SEED),
                          monotone_constraints=[int(v) for v in mono], n_jobs=int(ENV.get("cpu_count") or -1))
            if device == "gpu":
                params.update(device="gpu", gpu_device_id=int(gpu_id), gpu_platform_id=0)
            model = lgb.LGBMRegressor(**params)
            model.fit(_s07_take(fit_idx, feat), ctx["y"][fit_idx], sample_weight=ctx["w"][fit_idx], eval_metric="l2",
                      callbacks=[lgb.early_stopping(int(CFG.XGB_EARLY_STOP), verbose=False)],
                      **_s07_lgbm_eval_kwargs(_s07_take(val_idx, feat), ctx["y"][val_idx], ctx["w"][val_idx]))
            best = int(model.best_iteration_ or model.n_estimators)          # LightGBM counts from 1
            pred = model.predict(_s07_take(job["test"], feat), num_iteration=best).astype(np.float32)
            return pred, {"device": "cpu" if device == "cpu" else f"gpu:{gpu_id}", "best_iteration": best}
        except Exception as exc:  # noqa: BLE001
            if device == "cpu" or attempt == "cpu_fallback":
                raise
            log(f"LightGBM on GPU {gpu_id} failed ({exc!r}); this and all later LightGBM folds use the CPU",
                "WARNING")
            ENV["lgbm_device"] = "cpu"
            free_memory()
            device = "cpu"


def _s07_xgb_train(ctx, fit_idx, val_idx, device, gpu_id, feat_idx=None, rounds=None):
    """Train one XGBoost booster on ``fit_idx`` (early stopping on ``val_idx`` unless ``rounds`` is fixed).

    Data go to the GPU as CuPy arrays when available; returns (booster, best_iteration 0-based).
    """
    import contextlib

    import xgboost as xgb

    mono = ctx["mono"] if feat_idx is None else ctx["mono"][feat_idx]
    params = CFG.xgb_params(device, seed=int(CFG.SEED),
                            monotone_constraints="(" + ",".join(str(int(v)) for v in mono) + ")")
    use_cupy = device.startswith("cuda") and ENV.get("has_cupy")
    if use_cupy:
        import cupy as _cp

        dev_ctx, to_dev = _cp.cuda.Device(gpu_id), _cp.asarray
    else:
        dev_ctx, to_dev = contextlib.nullcontext(), np.asarray
    with dev_ctx:
        dtrain = xgb.QuantileDMatrix(to_dev(_s07_take(fit_idx, feat_idx)), label=to_dev(ctx["y"][fit_idx]),
                                     weight=to_dev(ctx["w"][fit_idx]), max_bin=int(params.get("max_bin", 256)))
        if rounds is None:
            dval = xgb.QuantileDMatrix(to_dev(_s07_take(val_idx, feat_idx)), label=to_dev(ctx["y"][val_idx]),
                                       weight=to_dev(ctx["w"][val_idx]), ref=dtrain)
            booster = xgb.train(params, dtrain, num_boost_round=int(CFG.XGB_MAX_ROUNDS), evals=[(dval, "val")],
                                early_stopping_rounds=int(CFG.XGB_EARLY_STOP), verbose_eval=False)
            best = int(getattr(booster, "best_iteration", booster.num_boosted_rounds() - 1))
            del dval
        else:
            booster = xgb.train(params, dtrain, num_boost_round=int(rounds), verbose_eval=False)
            best = int(rounds) - 1
        del dtrain
    return booster, best


def _s07_xgb_predict(booster, x, best, device, gpu_id):
    """Predict with the first ``best + 1`` rounds; on the GPU through CuPy when available, else on the host."""
    if device.startswith("cuda") and ENV.get("has_cupy"):
        import cupy as _cp

        with _cp.cuda.Device(gpu_id):
            pred = _cp.asnumpy(booster.inplace_predict(_cp.asarray(x), iteration_range=(0, best + 1)))
            _cp.get_default_memory_pool().free_all_blocks()
        return np.asarray(pred, np.float32)
    booster.set_param({"device": "cpu"})          # host data -> host prediction (no device-mismatch copies)
    return np.asarray(booster.inplace_predict(x, iteration_range=(0, best + 1)), np.float32)


def _s07_fit_xgboost(ctx, job, gpu_id):
    """Inner-block early stopping (or fixed rounds); CUDA out-of-memory -> retry the fold on the CPU."""
    feat = job["feat_idx"]
    fit_idx, val_idx = (job["train"], None) if job.get("rounds") else _s07_inner_split(job["train"])
    device = xgb_device_for(gpu_id)
    for attempt in ("primary", "cpu_fallback"):
        try:
            booster, best = _s07_xgb_train(ctx, fit_idx, val_idx, device, gpu_id, feat, job.get("rounds"))
            pred = _s07_xgb_predict(booster, _s07_take(job["test"], feat), best, device, gpu_id)
            del booster
            return pred, {"device": device, "best_iteration": best}
        except Exception as exc:  # noqa: BLE001
            if device == "cpu" or attempt == "cpu_fallback" or not is_gpu_oom_error(exc):
                raise
            log(f"XGBoost out of GPU memory on {device}; retrying fold on CPU", "WARNING")
            free_memory()
            device = "cpu"

### 7.2.1 Spatial GraphSAGE (pure PyTorch, ported from v1)
Nodes are **all ROI cells of the mode's training frames** (features are known everywhere); edges connect each
cell to its k = 8 nearest cells within the same frame (block-diagonal across frames, row-normalised so $A h$ is
the neighbour mean). Each layer is $h' = \mathrm{LayerNorm}(\mathrm{GELU}(W_s h + W_n A h)) + h$ with dropout 0.1;
three layers reach 3 cells. Training is transductive and full batch: the loss uses only the fitting rows' labels
(weighted MSE), early stopping the inner 10 km validation blocks, and test labels are never touched.

In [ ]:
def _s07_knn_indices(xy, k):
    """(n, k_eff) indices of the k nearest *other* points (exact cKDTree in float64)."""
    n = xy.shape[0]
    k_eff = min(int(k), n - 1)
    if k_eff < 1:
        return None
    _, idx = _s07_cKDTree(np.asarray(xy, dtype=np.float64)).query(xy, k=k_eff + 1)
    idx = np.asarray(idx, dtype=np.int64)
    self_first = idx[:, :1] == np.arange(n)[:, None]
    return np.where(self_first, idx[:, 1:], idx[:, :-1])


def _s07_gnn_graph(ctx, k=8):
    """Node set (DF rows of the mode's training frames), DF→node map and directed kNN edges (cached in ctx)."""
    if ctx.get("gnn") is not None:
        return ctx["gnn"]
    X, Y = GRID.cell_centers_xy()
    cell_xy = np.column_stack([np.asarray(X)[CELL_ROWS, CELL_COLS], np.asarray(Y)[CELL_ROWS, CELL_COLS]])
    neigh = _s07_knn_indices(cell_xy, k)
    n_cells = int(N_CELLS)
    nodes, src, dst = [], [], []
    for i, year in enumerate(ctx["frames"]):
        j = _S07_FRAMES_DF.index(year)
        nodes.append(np.arange(j * n_cells, (j + 1) * n_cells, dtype=np.int64))
        if neigh is not None:
            src.append(i * n_cells + np.repeat(np.arange(n_cells, dtype=np.int64), neigh.shape[1]))
            dst.append(i * n_cells + neigh.ravel())
    nodes = np.concatenate(nodes)
    assert np.all(_S07_YEAR[nodes] == np.repeat(ctx["frames"], n_cells)), "DF frame blocks out of order"
    df_to_node = np.full(_S07_N, -1, dtype=np.int64)
    df_to_node[nodes] = np.arange(nodes.size)
    ctx["gnn"] = {"nodes": nodes, "df_to_node": df_to_node,
                  "src": np.concatenate(src) if src else np.zeros(0, np.int64),
                  "dst": np.concatenate(dst) if dst else np.zeros(0, np.int64)}
    return ctx["gnn"]


def _s07_build_sage(n_in, hidden=128, n_layers=3, dropout=0.1):
    """GraphSAGE network (torch imported lazily so the notebook runs without it)."""
    import torch
    from torch import nn

    class _SageLayer(nn.Module):
        def __init__(self):
            super().__init__()
            self.w_self = nn.Linear(hidden, hidden)
            self.w_neigh = nn.Linear(hidden, hidden, bias=False)
            self.norm = nn.LayerNorm(hidden)
            self.drop = nn.Dropout(dropout)

        def forward(self, h, adj):
            agg = torch.sparse.mm(adj, h)
            return self.drop(self.norm(nn.functional.gelu(self.w_self(h) + self.w_neigh(agg)))) + h

    class SpatialSAGE(nn.Module):
        def __init__(self):
            super().__init__()
            self.inp = nn.Linear(n_in, hidden)
            self.layers = nn.ModuleList([_SageLayer() for _ in range(n_layers)])
            self.head = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.GELU(), nn.Linear(hidden // 2, 1))

        def forward(self, x, adj):
            h = self.inp(x)
            for layer in self.layers:
                h = layer(h, adj)
            return self.head(h).squeeze(-1)

    return SpatialSAGE()


def _s07_torch_cuda_count():
    try:
        import torch

        return torch.cuda.device_count() if torch.cuda.is_available() else 0
    except Exception:  # noqa: BLE001
        return 0


def _s07_gnn_train(ctx, job, device):
    """Full-batch transductive GraphSAGE fit on ``device``; returns (test predictions, best epoch)."""
    import torch

    graph = _s07_gnn_graph(ctx)
    nodes, d2n = graph["nodes"], graph["df_to_node"]
    tr, te = job["train"], job["test"]
    fit_idx, val_idx = _s07_inner_split(tr)
    fit_n, val_n, test_n = d2n[fit_idx], d2n[val_idx], d2n[te]
    if min(fit_n.min(), val_n.min(), test_n.min()) < 0:
        raise RuntimeError("GNN: a fold row is not a graph node")
    medians = _s07_train_medians(tr)
    x = _s07_impute(_S07_X[nodes], medians)
    tr_n = d2n[tr]
    mu, sd = x[tr_n].mean(axis=0), x[tr_n].std(axis=0)
    x = ((x - mu) / np.where(sd > 0, sd, 1.0)).astype(np.float32)
    y_fit = ctx["y"][fit_idx].astype(np.float64)
    y_mu, y_sd = float(y_fit.mean()), float(y_fit.std() or 1.0)
    y_nodes = np.nan_to_num((ctx["y"][nodes] - y_mu) / y_sd, nan=0.0).astype(np.float32)
    w_nodes = ctx["w"][nodes].astype(np.float32)

    n_nodes = nodes.size
    deg = np.bincount(graph["src"], minlength=n_nodes).astype(np.float32)
    weights = 1.0 / np.maximum(deg[graph["src"]], 1.0)
    dev = torch.device(device)
    adj = torch.sparse_coo_tensor(torch.from_numpy(np.vstack([graph["src"], graph["dst"]])),
                                  torch.from_numpy(weights.astype(np.float32)), (n_nodes, n_nodes)).coalesce().to(dev)
    if dev.type == "cpu":  # CSR SpMM is ~2x faster than COO on the CPU
        with _s07_warnings.catch_warnings():
            _s07_warnings.simplefilter("ignore", UserWarning)
            adj = adj.to_sparse_csr()
    x_t = torch.from_numpy(x).to(dev)
    y_t = torch.from_numpy(y_nodes).to(dev)
    w_t = torch.from_numpy(w_nodes).to(dev)
    fit_t, val_t = torch.from_numpy(fit_n).to(dev), torch.from_numpy(val_n).to(dev)
    with _S07_TORCH_LOCK:   # deterministic initialisation even when two folds start concurrently
        torch.manual_seed(int(CFG.SEED))
        if dev.type == "cuda":
            with torch.cuda.device(dev):
                torch.cuda.manual_seed(int(CFG.SEED))
        model = _s07_build_sage(x.shape[1]).to(dev)

    def wmse(pred, idx, target=y_t, weight=w_t):
        ww = weight[idx]
        return (ww * (pred[idx] - target[idx]) ** 2).sum() / ww.sum()

    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10, min_lr=1e-5)
    best_loss, best_epoch, best_state, stale = np.inf, 0, None, 0
    max_epochs, t_start, epoch = int(CFG.GNN_EPOCHS), _s07_time.time(), 0
    while epoch < max_epochs:
        model.train()
        opt.zero_grad(set_to_none=True)
        loss = wmse(model(x_t, adj), fit_t)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = float(wmse(model(x_t, adj), val_t))
        sched.step(val_loss)
        if val_loss < best_loss - 1e-6:
            best_loss, best_epoch, stale = val_loss, epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= 30:
                break
        if epoch == 1 and dev.type == "cpu":
            per_epoch = (_s07_time.time() - t_start) / 2.0
            affordable = max(10, int(_s07_gnn_cpu_budget() / max(per_epoch, 1e-6)))
            if affordable < max_epochs:
                log(f"GNN on CPU: {per_epoch:.2f} s/epoch -> this fold capped at {affordable} of {max_epochs} epochs "
                    f"({_s07_gnn_cpu_budget():.0f} s budget, LST_GNN_CPU_BUDGET_S)", "WARNING")
                max_epochs = affordable
        epoch += 1
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred = model(x_t, adj)[torch.from_numpy(test_n).to(dev)].float().cpu().numpy()
    del model, opt, adj, x_t, y_t, w_t, best_state
    return (pred * y_sd + y_mu).astype(np.float32), int(best_epoch)


def _s07_gnn_cpu_budget():
    """Per-fold CPU budget in seconds (read at call time so it can be changed between runs)."""
    return float(_s07_os.environ.get("LST_GNN_CPU_BUDGET_S", _S07_GNN_CPU_BUDGET_S))


def _s07_fit_gnn(ctx, job, gpu_id):
    """GraphSAGE fold on ``cuda:<gpu_id>`` (or the CPU); CUDA OOM -> CPU retry; CUDA cache emptied afterwards."""
    import torch

    n_cuda = _s07_torch_cuda_count()
    device = f"cuda:{min(int(gpu_id), n_cuda - 1)}" if n_cuda else "cpu"
    try:
        pred, best = _s07_gnn_train(ctx, job, device)
    except Exception as exc:  # noqa: BLE001
        if device == "cpu" or not is_gpu_oom_error(exc):
            raise
        log(f"GNN out of memory on {device}; retrying fold on CPU", "WARNING")
        torch.cuda.empty_cache()
        free_memory()
        device = "cpu"
        pred, best = _s07_gnn_train(ctx, job, device)
    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return pred, {"device": device, "best_iteration": best}


_S07_FITTERS = {"ridge": _s07_fit_ridge, "random_forest": _s07_fit_random_forest, "lightgbm": _s07_fit_lightgbm,
                "xgboost": _s07_fit_xgboost, "gnn": _s07_fit_gnn}


def _s07_model_available(model):
    """Skip models whose library is missing or broken (with a warning) instead of failing the run.

    Uses section 00's optional imports (``torch``, ``xgb``, ``lgb``), where ANY import failure counts as missing -
    e.g. a PyTorch DLL / CUDA initialisation error (``OSError``), not only ``ImportError`` - so a broken optional
    library never stops the pipeline: the GNN is a benchmark only.
    """
    preloaded = {"gnn": ("torch", globals().get("torch")), "xgboost": ("xgboost", globals().get("xgb")),
                 "lightgbm": ("lightgbm", globals().get("lgb"))}
    if model in preloaded:
        name, mod = preloaded[model]
        if mod is None:
            log(f"{name} is not available (missing or failed to import, see section 0.2) - model '{model}' "
                "skipped", "WARNING")
            return False
        return True
    try:
        __import__("sklearn")
        return True
    except Exception as exc:  # noqa: BLE001 - optional dependency, any failure means "unavailable"
        log(f"scikit-learn is not available ({type(exc).__name__}: {str(exc)[:160]}) - model '{model}' skipped",
            "WARNING")
        return False

## 7.3 Job scheduler (GPU lane, CPU lane, LightGBM-GPU lane; per-job stage cache)
A job is one (model, training rows, test rows, feature subset) fit. Its identity is a hash of exactly these inputs
plus the model configuration, so two schemes asking for the same fit share one job and its cache entry.

| Lane | Jobs | Runs on |
|---|---|---|
| `gpu` | XGBoost (CUDA), GraphSAGE, cuML RF | `gpu_pool_map`: one worker per GPU, each GPU one job at a time |
| `cpu` | ridge, scikit-learn RF, LightGBM when its GPU probe failed | thread `s07-cpu-lane`, serial |
| `lgbm` | LightGBM when `ENV["lgbm_device"] == "gpu"` | thread `s07-lgbm-lane`, serial, on GPU `n_gpus − 1` |

The LightGBM lane is serial because LightGBM's OpenCL learner is not thread safe (never two boosters at once).
XGBoost, LightGBM, scikit-learn trees and PyTorch release the GIL inside their native code, so the lanes overlap
for real; all lanes join at the mode barrier. A failing job is logged with its traceback and leaves NaN
predictions; it never stops the suite. **Time budget:** before every job a lane asks `budget_exhausted(1.25 ×
the longest job seen in that lane)` (section 00); on exhaustion it stops scheduling and sets a shared stop event,
the other lanes stop too, and after the barrier `check_budget("07 jobs")` raises `LSTStop` — a mode is never
published with partial CV results (the finished jobs are cached and a resumed run continues from them). The busy
seconds and job counts per lane are logged after each mode (`MODEL_RESULTS[m]["lane_busy_seconds"]`).

In [ ]:
_S07_MODEL_CFG = {
    "xgboost": lambda: {"params": CFG.XGB_PARAMS, "max_rounds": CFG.XGB_MAX_ROUNDS, "es": CFG.XGB_EARLY_STOP},
    "lightgbm": lambda: {"params": _S07_LGBM_PARAMS, "max_rounds": CFG.XGB_MAX_ROUNDS, "es": CFG.XGB_EARLY_STOP},
    "random_forest": lambda: {"trees": _S07_RF_TREES, "max_rows": _S07_RF_MAX_ROWS},
    "ridge": lambda: {"alphas": [-3, 3, 13]},
    "gnn": lambda: {"epochs": CFG.GNN_EPOCHS, "hidden": 128, "layers": 3, "k": 8},
}
_S07_COST = {"ridge": 0.05, "random_forest": 1.5, "lightgbm": 1.0, "xgboost": 1.0, "gnn": 2.5}


def _s07_lane(model):
    """'gpu' (gpu_pool_map), 'lgbm' (serial LightGBM-GPU lane) or 'cpu' for a model in this session."""
    if model == "xgboost":
        return "gpu" if ENV.get("xgb_device") == "cuda" else "cpu"
    if model == "gnn":
        return "gpu" if _s07_torch_cuda_count() > 0 else "cpu"
    if model == "random_forest":
        return "gpu" if ENV.get("has_cuml") else "cpu"
    if model == "lightgbm":
        return "lgbm" if ENV.get("lgbm_device") == "gpu" and int(ENV.get("n_gpus", 0) or 0) >= 1 else "cpu"
    return "cpu"


def _s07_budget_exhausted(expected_s):
    """Section-00 time-budget check (D3) through a defensive lookup; False when the helper is absent."""
    fn = globals().get("budget_exhausted")
    return bool(fn is not None and fn(float(expected_s)))


def _s07_make_job(ctx, label, model, fold, feat="all", rounds=None):
    """Job dict for one fit; ``ident`` de-duplicates identical fits across schemes."""
    feat_idx = None if feat == "all" else _S07_BASE_IDX
    ident = _s07_hash(_S07_JOB_VERSION, ctx["mode"], model, feat, rounds, _S07_MODEL_CFG[model](), fold["train"],
                      fold["test"])
    return {"ident": ident, "mode": ctx["mode"], "model": model, "label": label, "fold": int(fold["fold"]),
            "train": fold["train"], "test": fold["test"], "feat": feat, "feat_idx": feat_idx, "rounds": rounds,
            "cost": float(fold["train"].size) * _S07_COST[model]}


def _s07_execute(job, gpu_id):
    """Run (or load from the stage cache) one job; never raises - failures are returned as ``{"error": ...}``."""
    ctx = _S07_CTX[job["mode"]]
    sc = stage_cache(f"07_job_{job['mode']}_{job['model']}",
                     CFG.key(version=_S07_JOB_VERSION, ident=job["ident"], df=_S07_DF_FP))
    if sc.hit:
        try:
            res = sc.load()
            res["cached"] = True
            return res
        except StageCacheError as exc:
            log(f"{exc}; refitting", "WARNING")
    t0 = _s07_time.perf_counter()
    try:
        pred, info = _S07_FITTERS[job["model"]](ctx, job, gpu_id)
        pred = np.asarray(pred, np.float32).ravel()
        if pred.shape != job["test"].shape or not np.all(np.isfinite(pred)):
            raise ValueError(f"invalid predictions (shape {pred.shape}, finite {np.isfinite(pred).mean():.3f})")
    except Exception:  # noqa: BLE001 - one failing fold must not stop the suite
        return {"error": _s07_traceback.format_exc()}
    res = {"pred": pred, "info": info, "seconds": round(_s07_time.perf_counter() - t0, 2),
           "n_train": int(job["train"].size), "n_test": int(job["test"].size)}
    met = _s07_wmetrics(ctx["y"][job["test"]], pred, ctx["w"][job["test"]])
    with _S07_PRINT_LOCK:
        log(f"07 {job['mode']:<5} {job['label']:<14} {job['model']:<13} fold {job['fold']:>4}: R2 {met['r2']:.3f} "
            f"RMSE {met['rmse']:.3f} | {res['seconds']:.1f}s on {info.get('device')} (n_train {res['n_train']:,})")
    sc.save(res)
    res["cached"] = False
    return res


def _s07_run_jobs(jobs, mode=""):
    """Run unique jobs on the GPU lane, the CPU lane and the LightGBM-GPU lane concurrently (D20).

    Returns ({ident: result}, busy seconds per lane, job counts per lane). Raises ``LSTStop`` (via
    ``check_budget``) when a lane stopped on the time budget, and re-raises any BaseException of a lane after all
    lane threads have joined - never a partial result set.
    """
    unique = {}
    for job in jobs:
        unique.setdefault(job["ident"], job)
    lanes = {name: sorted((j for j in unique.values() if _s07_lane(j["model"]) == name), key=lambda j: -j["cost"])
             for name in ("gpu", "cpu", "lgbm")}
    results = {}
    busy = {"gpu": 0.0, "cpu": 0.0, "lgbm": 0.0}
    counts = {"gpu": 0, "cpu": 0, "lgbm": 0}
    longest = {"gpu": 0.0, "cpu": 0.0, "lgbm": 0.0}
    lock = _s07_threading.Lock()
    stop = _s07_threading.Event()
    stop_info = {"reason": None, "expected_s": 0.0, "errors": []}
    lgbm_gpu = max(int(ENV.get("n_gpus", 0) or 0) - 1, 0)
    log(f"07: {len(jobs)} fits requested -> {len(unique)} unique jobs ({len(lanes['gpu'])} GPU lane, "
        f"{len(lanes['cpu'])} CPU lane, {len(lanes['lgbm'])} LightGBM-GPU lane"
        + (f" on GPU {lgbm_gpu}" if lanes["lgbm"] else "") + ")")

    def record(lane, job, res):
        with lock:
            results[job["ident"]] = res
            counts[lane] += 1
            if not res.get("cached") and "seconds" in res:
                busy[lane] += float(res["seconds"])
                longest[lane] = max(longest[lane], float(res["seconds"]))

    def may_start(lane):
        """False once the run is stopping or the next job would not fit into the time budget."""
        if stop.is_set():
            return False
        with lock:
            expected = 1.25 * longest[lane]
        if _s07_budget_exhausted(expected):
            with lock:
                if stop_info["reason"] is None:
                    stop_info["reason"] = f"time budget reached in the 07 {lane} lane ({mode})"
                stop_info["expected_s"] = max(stop_info["expected_s"], expected)
            stop.set()
            return False
        return True

    def serial_lane(lane, gpu_id):
        try:
            for job in lanes[lane]:
                if not may_start(lane):
                    break
                record(lane, job, _s07_execute(job, gpu_id))
        except BaseException as exc:  # noqa: BLE001 - recorded and re-raised by the main thread after the barrier
            with lock:
                stop_info["errors"].append(exc)
            stop.set()

    def gpu_item(job, gpu_id):
        if not may_start("gpu"):
            return {"error": "not run: the 07 lanes are stopping", "skipped": True}
        return _s07_execute(job, gpu_id)

    threads = []
    for lane, gpu_id in (("cpu", 0), ("lgbm", lgbm_gpu)):
        if lanes[lane]:
            t = _s07_threading.Thread(target=serial_lane, args=(lane, gpu_id), name=f"s07-{lane}-lane", daemon=True)
            t.start()
            threads.append(t)
    try:
        if lanes["gpu"]:
            for job, res in zip(lanes["gpu"], gpu_pool_map(gpu_item, lanes["gpu"])):
                if not res.get("skipped"):
                    record("gpu", job, res)
    except BaseException:
        stop.set()                      # the serial lanes finish their current job and stop
        for t in threads:
            t.join()
        raise
    for t in threads:
        t.join()                        # mode barrier
    if stop.is_set():
        if stop_info["errors"]:
            raise stop_info["errors"][0]
        check = globals().get("check_budget")
        if check is not None:
            check(f"07 jobs ({mode})", stop_info["expected_s"])
        raise RuntimeError(f"07: job lanes stopped ({stop_info['reason']}) - no partial results are published")
    for ident in unique:
        results.setdefault(ident, {"error": "job did not run (lane thread terminated)"})
    log(f"07: {mode} lane busy seconds (GPU summed over devices) "
        + ", ".join(f"{k} {busy[k]:.1f}s / {counts[k]} jobs" for k in ("gpu", "cpu", "lgbm")))
    return results, busy, counts

## 7.4 Collecting a scheme: OOF arrays, recentring, pooled and per-fold metrics
* **LOEO** — the predicted anomaly map of the held-out frame is recentred to zero mean over that frame's fixed-mask
  rows; raw and recentred metrics are both kept (`oof["loeo_raw"]`, `oof["loeo"]`).
* **place_time** — every fold predicts one block group of the last frame; the pooled OOF map of the last frame is
  complete, and it is recentred once over its fixed-mask rows (predictions only).

In [ ]:
def _s07_collect(ctx, scheme, models, folds, jobs_by_key, results):
    """OOF arrays {model: float32[N_DF]}, fold-metric rows and info lists for one scheme."""
    oof, raw_oof, rows, infos = {}, {}, [], {}
    for model in models:
        oof[model] = np.full(_S07_N, np.nan, dtype=np.float32)
        raw = np.full(_S07_N, np.nan, dtype=np.float32)
        infos[model] = []
        per_fold = []
        for fold in folds:
            job = jobs_by_key[(scheme, model, fold["fold"])]
            res = results[job["ident"]]
            if "error" in res:
                log(f"07 {ctx['mode']} {scheme}/{model} fold {fold['fold']} FAILED:\n{res['error']}", "ERROR")
                continue
            te = fold["test"]
            raw[te] = res["pred"]
            infos[model].append({"fold": fold["fold"], **res["info"]})
            per_fold.append((fold, res))
        if scheme == "loeo":
            for fold, _ in per_fold:
                te = fold["test"]
                sel = te[ctx["fixed_row"][te]]
                offset = float(np.mean(raw[sel])) if sel.size else float(np.mean(raw[te]))
                oof[model][te] = raw[te] - offset
        elif scheme == "place_time":
            done = np.flatnonzero(np.isfinite(raw))
            sel = done[ctx["fixed_row"][done]]
            offset = float(np.mean(raw[sel])) if sel.size else (float(np.mean(raw[done])) if done.size else 0.0)
            oof[model] = np.where(np.isfinite(raw), raw - offset, np.nan).astype(np.float32)
        else:
            oof[model] = raw
        raw_oof[model] = raw
        for fold, res in per_fold:
            te = fold["test"]
            met = _s07_wmetrics(ctx["y"][te], oof[model][te], ctx["w"][te])
            rows.append({"scheme": scheme, "model": model, "fold": int(fold["fold"]), "r2": met["r2"],
                         "rmse": met["rmse"], "mae": met["mae"], "bias": met["bias"],
                         "n_train": int(fold["train"].size),
                         "n_test": int(te.size), "fit_seconds": float(res["seconds"]),
                         "device": str(res["info"].get("device"))})
    return oof, raw_oof, rows, infos


def _s07_summary_row(ctx, scheme, model, pred, fold_rows):
    """One cv_summary row: pooled weighted metrics over all OOF rows + per-fold spread."""
    ok = np.flatnonzero(np.isfinite(pred))
    met = _s07_wmetrics(ctx["y"][ok], pred[ok], ctx["w"][ok])
    sub = [r for r in fold_rows if r["scheme"] == scheme and r["model"] == model]
    r2s = np.array([r["r2"] for r in sub], float)
    rmses = np.array([r["rmse"] for r in sub], float)
    sd = lambda a: float(np.nanstd(a, ddof=1)) if np.isfinite(a).sum() > 1 else 0.0  # noqa: E731
    return {"scheme": scheme, "model": model, "r2_pooled": met["r2"], "rmse_pooled": met["rmse"],
            "mae_pooled": met["mae"], "r2_fold_mean": float(np.nanmean(r2s)) if r2s.size else np.nan,
            "r2_fold_sd": sd(r2s), "rmse_fold_mean": float(np.nanmean(rmses)) if rmses.size else np.nan,
            "rmse_fold_sd": sd(rmses), "n_folds": int(len(sub)), "n": int(met["n"])}

## 7.5 Spatial diagnostics: error vs distance, Moran's I, fold geometry
* **Error vs distance** — for every block20 OOF row, the distance (km) to the nearest *training* cell of the same
  frame in its fold's training set; pooled weighted RMSE in distance bins shows how skill decays with distance
  from the data (the honest replacement for a single "map accuracy" number, Wadoux et al. 2021).
* **Moran's I** of block20 OOF residuals per frame (KNN, k = `CFG.MORAN_K`, row-standardised, permutation
  inference: PySAL's `p_sim`/`z_sim`; `esda` when installed, else the vectorised NumPy fallback of v1).
  $I = \frac{n}{S_0}\frac{\sum_{ij} w_{ij} z_i z_j}{\sum_i z_i^2}$; residual structure means spatially structured
  drivers are still missing and naive standard errors would be too small.

In [ ]:
def _s07_error_vs_distance(ctx, folds, oof):
    """[{lo, hi, rmse, n}] of block20 OOF residuals by distance to the nearest same-frame training cell."""
    dist = np.full(_S07_N, np.nan)
    for fold in folds:
        te, tr = fold["test"], fold["train"]
        for year in np.unique(_S07_YEAR[te]):
            te_y, tr_y = te[_S07_YEAR[te] == year], tr[_S07_YEAR[tr] == year]
            if tr_y.size == 0:
                continue
            d, _ = _s07_cKDTree(_S07_XY[tr_y]).query(_S07_XY[te_y], k=1)
            dist[te_y] = d / 1000.0
    out = []
    ok = np.isfinite(oof) & np.isfinite(dist)
    for lo, hi in zip(_S07_DIST_BINS[:-1], _S07_DIST_BINS[1:]):
        sel = ok & (dist >= lo) & (dist < hi)
        met = _s07_wmetrics(ctx["y"][sel], oof[sel], ctx["w"][sel]) if sel.sum() >= 2 else {"rmse": np.nan}
        out.append({"lo": lo, "hi": None if not np.isfinite(hi) else hi, "rmse": met["rmse"], "n": int(sel.sum())})
    return out, dist


def _s07_knn_weights(xy, k):
    """Row-standardised KNN weights (csr, n x n) over the given coordinates."""
    n = xy.shape[0]
    neigh = _s07_knn_indices(xy, k)
    if neigh is None:
        return _s07_sparse.csr_matrix((n, n), dtype=np.float64)
    k_eff = neigh.shape[1]
    rows = np.repeat(np.arange(n), k_eff)
    return _s07_sparse.csr_matrix((np.full(rows.size, 1.0 / k_eff), (rows, neigh.ravel())), shape=(n, n))


def _s07_moran_numpy(y, W, permutations, seed):
    """Moran's I with permutation inference, vectorised in batches of permutations (NumPy/SciPy)."""
    n = y.size
    z = y - y.mean()
    z2 = float(z @ z)
    s0 = float(W.sum())
    if z2 == 0 or s0 == 0:
        return {"I": np.nan, "expected_I": -1.0 / (n - 1), "z": np.nan, "p": np.nan}
    scale = n / s0 / z2
    moran = scale * float(z @ (W @ z))
    out = {"I": moran, "expected_I": -1.0 / (n - 1), "z": np.nan, "p": np.nan}
    if permutations:
        rng = np.random.default_rng(seed)
        batch = max(1, min(int(permutations), 4_000_000 // max(n, 1)))
        sims = []
        for start in range(0, int(permutations), batch):
            perm = np.tile(z, (min(batch, int(permutations) - start), 1))
            rng.permuted(perm, axis=1, out=perm)
            lag = (W @ perm.T).T
            sims.append(scale * np.einsum("bi,bi->b", perm, lag))
        sims = np.concatenate(sims)
        larger = int((sims >= moran).sum())
        larger = min(larger, int(permutations) - larger)
        sd = sims.std()
        out["p"] = (larger + 1.0) / (permutations + 1.0)
        out["z"] = (moran - sims.mean()) / sd if sd > 0 else np.nan
    return out


def _s07_morans_i(values, W, permutations=None):
    """Global Moran's I (``{"I", "expected_I", "z", "p", "backend"}``), esda when installed, else NumPy."""
    y = np.asarray(values, dtype=np.float64).ravel()
    perms = int(CFG.MORAN_PERMS if permutations is None else permutations)
    if y.size < 3 or not np.all(np.isfinite(y)):
        return {"I": np.nan, "expected_I": np.nan, "z": np.nan, "p": np.nan, "backend": "none"}
    try:
        import esda
        from libpysal.weights import W as _PysalW

        state = np.random.get_state()
        np.random.seed(int(CFG.SEED))   # esda permutes with the global NumPy RNG
        try:
            with _s07_warnings.catch_warnings():
                _s07_warnings.simplefilter("ignore")
                mi = esda.Moran(y, _PysalW.from_sparse(_s07_sparse.csr_matrix(W)), transformation="r",
                                permutations=perms)
        finally:
            np.random.set_state(state)
        z, p = (mi.z_sim, mi.p_sim) if perms else (mi.z_norm, mi.p_norm)
        return {"I": float(mi.I), "expected_I": float(mi.EI), "z": float(z), "p": float(p), "backend": "esda"}
    except ImportError:
        res = _s07_moran_numpy(y, W, perms, int(CFG.SEED))
        res["backend"] = "numpy"
        return {k: (float(v) if k != "backend" else v) for k, v in res.items()}


def _s07_moran_tables(ctx, oof_block20):
    """Moran's I of the target and of block20 OOF residuals (xgboost, gnn, ridge) per frame + scatter data."""
    models = [m for m in ("xgboost", "gnn", "ridge") if m in oof_block20]
    res_rows, tgt_rows, scatter = [], [], {}
    tr = ctx["train"]
    rng = np.random.default_rng(CFG.SEED)
    last = ctx["frames"][-1] if ctx["frames"] else None
    for year in ctx["frames"]:
        rows = tr[_S07_YEAR[tr] == year]
        if rows.size < max(10, int(CFG.MORAN_K) + 2):
            continue
        W = _s07_knn_weights(_S07_XY[rows], int(CFG.MORAN_K))
        r = _s07_morans_i(ctx["y"][rows], W)
        tgt_rows.append({"year": int(year), **{k: r[k] for k in ("I", "expected_I", "z", "p")}})
        for model in models:
            pred = oof_block20[model][rows]
            ok = np.isfinite(pred)
            if ok.sum() < max(10, int(CFG.MORAN_K) + 2):
                continue
            resid = ctx["y"][rows].astype(np.float64) - pred.astype(np.float64)
            Wm = W if ok.all() else _s07_knn_weights(_S07_XY[rows[ok]], int(CFG.MORAN_K))
            m_res = _s07_morans_i(resid[ok], Wm)
            res_rows.append({"scheme": "block20", "model": model, "year": int(year),
                             **{k: m_res[k] for k in ("I", "expected_I", "z", "p")}})
            if year == last and ok.all():
                sd = resid.std()
                if sd > 0:
                    zz = (resid - resid.mean()) / sd
                    lag = np.asarray(W @ zz)
                    pick = np.sort(rng.choice(zz.size, size=min(2000, zz.size), replace=False))
                    scatter[model] = {"year": int(year), "z": zz[pick].round(4).tolist(),
                                      "lag": lag[pick].round(4).tolist(), "slope": float(m_res["I"])}
    moran = pd.DataFrame(res_rows, columns=["scheme", "model", "year", "I", "expected_I", "z", "p"])
    moran_target = pd.DataFrame(tgt_rows, columns=["year", "I", "expected_I", "z", "p"])
    head = scatter.get("xgboost") or (next(iter(scatter.values())) if scatter else {})
    moran_scatter = {"model": "xgboost" if "xgboost" in scatter else (next(iter(scatter)) if scatter else None),
                     **head, "by_model": scatter}
    return moran, moran_target, moran_scatter


def _s07_fold_geometry(ctx, oof):
    """Fold map payload: block → fold and pooled block RMSE (row-major blocks from the grid's top-left)."""
    out = {}
    h, w = GRID.height, GRID.width
    cell_rows, cell_cols = _S07_ROW, _S07_COL
    for scheme in ("block20", "block5"):
        if scheme == "block20":
            b = max(1, int(round(float(CFG.BLOCK_KM) * 1000.0 / float(GRID.res))))
            n_r, n_c = _s07_math.ceil(h / b), _s07_math.ceil(w / b)
            block = (cell_rows // b) * n_c + cell_cols // b
            fold_col = DF["fold_block20"].to_numpy(np.int64)
            meta = {"block_km": float(CFG.BLOCK_KM), "block_cells": b, "kind": "square"}
        else:
            nby, nbx = (int(v) for v in CFG.SPATIAL_BLOCKS)
            n_r, n_c = nby, nbx
            block = (cell_rows * nby // h) * nbx + cell_cols * nbx // w
            fold_col = DF["fold_block5"].to_numpy(np.int64)
            meta = {"block_km": round(float(np.sqrt((h / nby) * (w / nbx))) * GRID.res / 1000.0, 1),
                    "tile_km": [round(h / nby * GRID.res / 1000.0, 1), round(w / nbx * GRID.res / 1000.0, 1)],
                    "kind": "tiles", "rule": "tile = (row*5//H)*5 + col*5//W"}
        n_blocks = n_r * n_c
        fold_of_block = np.full(n_blocks, -1, dtype=np.int64)
        fold_of_block[block] = fold_col                       # one fold per block (asserted in section 06)
        rmse = [None] * n_blocks
        pred = oof.get(scheme, {}).get("xgboost")
        if pred is not None:
            ok = np.isfinite(pred) & (ctx["w"] > 0)
            r2w = np.bincount(block[ok], weights=ctx["w"][ok] * (ctx["y"][ok] - pred[ok]) ** 2, minlength=n_blocks)
            sw = np.bincount(block[ok], weights=ctx["w"][ok], minlength=n_blocks)
            rmse = [float(np.sqrt(a / s)) if s > 0 else None for a, s in zip(r2w, sw)]
        out[scheme] = {**meta, "n_rows": int(n_r), "n_cols": int(n_c), "fold_of_block": fold_of_block.tolist(),
                       "rmse_of_block": rmse}
    return out

## 7.6 Final model, persistent residual and area of applicability
**Final model.** XGBoost on *all* training rows of the mode, `round(1.1 × median best rounds of block20)` rounds
(the 10 % accounts for the larger training set; no early stopping, no hold-out). It is stored with
`device="cpu"` so `PREDICT_ANOM` works anywhere; section 08 may switch it to `cuda` for TreeSHAP.

**Persistent residual (SPEC §4.5, time transfer only).** About half of the remaining error is a per-cell offset
that repeats every frame (soil, irrigation practice, emissivity — drivers the features miss). With the recentred
LOEO residuals $\rho(i,s)$, the shrunken mean residual of the *other* frames
$\bar r_i^{(-t)} = \sum_{s\ne t} w_{is}\rho(i,s) / (\sum_{s\ne t} w_{is} + \lambda)$ is added to the LOEO
prediction of frame $t$; $\lambda$ (a pseudo-count of zero-residual frames) is chosen on the LOEO RMSE. The
final `PERSISTENT_RESID` uses all frames. It is never part of a spatial-CV number (an unseen place has no
residual history).

**Area of applicability (Meyer & Pebesma 2021).** Features are standardised with the training mean/std and
weighted by the final model's mean |SHAP|; with a training sample of ≤ 20 000 **complete** training rows (every
weighted feature finite), $DI = d_{min}(x, \text{train}) / \bar d$ where $\bar d$ is the mean pairwise distance of
the sample. For training rows the nearest row must come from a **different block20 fold** (the distance a CV fold
actually had to bridge). Threshold $= Q_{75} + 1.5\,IQR$ of the training DI. A row with a missing weighted feature
(e.g. a Landsat gap in an early frame) has an unknown distance to the training data: `COMPUTE_DI` returns
$DI = +\infty$ for it (outside the AOA, top decile) instead of imputing the training mean, which would make it look
typical. The **conformal error model** stores block20 OOF |residual| quantiles (q50/q90/q95) per DI decile (edges
from the training DI) and signed residual samples per decile.

**Projection-setting error model.** A modelled or projected frame is predicted at a *known* place for *another*
time with the final model plus the cell's persistent residual. The matching residual is
$\rho_{proj}(i,t) = y(i,t) - (\hat a^{LOEO}(i,t) + \bar r_i^{(-t)})$ (recentred LOEO prediction + the shrunken
residual of the other frames with the chosen λ). It is binned by the same different-fold DI deciles:
`AOA[m]["resid_samples_proj"]` (signed samples, the Monte-Carlo model error of section 11) and
`AOA[m]["conformal_proj"]` (|residual| quantiles, the modelled-frame intervals of section 09). Residuals of nearby
cells are correlated: `AOA[m]["resid_rho_block20"]` is the share of the residual variance shared within a 20 km
block, $\rho_t = (\mathrm{Var}(\bar\rho_b) - \overline{s^2_b / n_b}) / \mathrm{Var}(\rho)$ over blocks with ≥ 5 rows,
row-weighted over frames and clipped to [0, 0.95] — section 11 uses it as the block correlation of its draws.

In [ ]:
def _s07_final_rounds(infos_block20, infos_fallback):
    """Median best rounds (best_iteration + 1) of the block20 XGBoost folds (fallback: other schemes)."""
    for source, infos in (("block20", infos_block20), ("fallback", infos_fallback)):
        rounds = [int(i["best_iteration"]) + 1 for i in infos if i.get("best_iteration") is not None]
        if rounds:
            if source != "block20":
                log("07: no block20 XGBoost fold succeeded; final rounds from other schemes", "WARNING")
            return rounds
    fallback = max(1, int(CFG.XGB_MAX_ROUNDS) // 3)
    log(f"07: no XGBoost CV fold succeeded; best_rounds falls back to [{fallback}]", "WARNING")
    return [fallback]


def _s07_train_final(ctx, rounds):
    """Final XGBoost booster on all training rows (GPU 0 when available; CPU retry on GPU OOM)."""
    device = xgb_device_for(0)
    try:
        booster, _ = _s07_xgb_train(ctx, ctx["train"], None, device, 0, None, rounds)
    except Exception as exc:  # noqa: BLE001
        if device == "cpu" or not is_gpu_oom_error(exc):
            raise
        log("07: final XGBoost out of GPU memory; training on the CPU", "WARNING")
        free_memory()
        booster, _ = _s07_xgb_train(ctx, ctx["train"], None, "cpu", 0, None, rounds)
    booster.set_param({"device": "cpu"})
    return booster


def _s07_as_matrix(X):
    """(n, 47) float32 C-contiguous matrix in FEATURES order from a DataFrame or an array."""
    if isinstance(X, pd.DataFrame):
        missing = [f for f in FEATURES if f not in X.columns]
        if missing:
            raise KeyError(f"feature columns missing: {missing}")
        X = X[FEATURES].to_numpy(np.float32)
    X = np.ascontiguousarray(np.asarray(X, dtype=np.float32))
    if X.ndim != 2 or X.shape[1] != len(FEATURES):
        raise ValueError(f"expected an (n, {len(FEATURES)}) feature matrix, got {X.shape}")
    return X


def PREDICT_ANOM(m, X, recentre_mask=None):
    """Final-model anomaly prediction (°C) for mode ``m``; optional recentring over ``recentre_mask`` rows.

    ``X``: DataFrame with the ``FEATURES`` columns or an (n, 47) array in ``FEATURES`` order (NaN allowed).
    ``recentre_mask``: bool array over the rows of ONE frame (typically its fixed-mask cells); when given, the mean
    prediction over those rows is subtracted (SPEC §1.4 recentring).
    """
    if m not in FINAL_MODELS:
        raise KeyError(f"no final model for mode {m!r} (available: {sorted(FINAL_MODELS)})")
    Xa = _s07_as_matrix(X)
    pred = np.asarray(FINAL_MODELS[m].inplace_predict(Xa), dtype=np.float32)
    if recentre_mask is not None:
        mask = np.asarray(recentre_mask, dtype=bool)
        if mask.shape != pred.shape:
            raise ValueError(f"recentre_mask shape {mask.shape} != predictions {pred.shape}")
        if not mask.any():
            raise ValueError("recentre_mask selects no rows")
        pred = pred - np.float32(np.mean(pred[mask], dtype=np.float64))
    return pred


def _s07_persistent_residual(ctx, loeo_pred):
    """λ selection on LOEO (SPEC §4.5) → (H, W) residual grid, λ and the report dict."""
    tr = ctx["train"]
    tr = tr[np.isfinite(loeo_pred[tr])]
    empty_grid = np.where(ROI_MASK, 0.0, np.nan).astype(np.float32)
    rbar_rows = np.full(_S07_N, np.nan, dtype=np.float64)     # rbar^(-t) of every training row with a LOEO prediction
    if tr.size == 0 or len(np.unique(_S07_YEAR[tr])) < 2:
        log(f"07: {ctx['mode']}: no LOEO predictions - persistent residual disabled (lambda = inf)", "WARNING")
        return empty_grid, float("inf"), {"lambda": None, "lambda_is_inf": True, "loeo_rmse_without": None,
                                          "loeo_rmse_with": None, "by_lambda": []}, rbar_rows
    w = ctx["w"][tr].astype(np.float64)
    rho = ctx["y"][tr].astype(np.float64) - loeo_pred[tr].astype(np.float64)
    cell = _S07_CELL_POS[tr]
    s_all = np.bincount(cell, weights=w * rho, minlength=int(N_CELLS))
    w_all = np.bincount(cell, weights=w, minlength=int(N_CELLS))
    s_other = s_all[cell] - w * rho            # sums over the OTHER frames s != t of the row's frame t
    w_other = w_all[cell] - w
    by_lambda = []
    for lam in _S07_LAMBDAS:
        rbar = np.zeros_like(rho) if not np.isfinite(lam) else s_other / (w_other + lam)
        met = _s07_wmetrics(ctx["y"][tr], loeo_pred[tr] + rbar, w)
        by_lambda.append({"lambda": None if not np.isfinite(lam) else lam, "rmse": met["rmse"], "r2": met["r2"]})
    best = min(range(len(_S07_LAMBDAS)), key=lambda i: (by_lambda[i]["rmse"], -i))
    lam = _S07_LAMBDAS[best]
    grid = np.full(ROI_MASK.shape, np.nan, dtype=np.float32)
    rbar_cells = np.zeros(int(N_CELLS)) if not np.isfinite(lam) else s_all / (w_all + lam)
    grid[CELL_ROWS, CELL_COLS] = rbar_cells.astype(np.float32)
    report = {"lambda": None if not np.isfinite(lam) else lam, "lambda_is_inf": not np.isfinite(lam),
              "loeo_rmse_without": by_lambda[-1]["rmse"], "loeo_rmse_with": by_lambda[best]["rmse"],
              "loeo_r2_without": by_lambda[-1]["r2"], "loeo_r2_with": by_lambda[best]["r2"],
              "gain_rmse": by_lambda[-1]["rmse"] - by_lambda[best]["rmse"], "by_lambda": by_lambda,
              "resid_sd": float(np.nanstd(rbar_cells)), "model": "xgboost"}
    rbar_rows[tr] = 0.0 if not np.isfinite(lam) else s_other / (w_other + lam)
    return grid, lam, report, rbar_rows


def _s07_sq_norms(a):
    return np.einsum("ij,ij->i", a, a, dtype=np.float64)


def _s07_nearest_cuda(query, ref, chunk=4096):
    """Plain nearest distances on a CUDA GPU via PyTorch (the COMPUTE_DI hot path); None when unavailable."""
    if _s07_torch_cuda_count() == 0:
        return None
    try:
        import torch

        dev = torch.device("cuda:0")
        with _S07_TORCH_LOCK, torch.no_grad():
            r = torch.from_numpy(ref).to(dev)
            rn = (r * r).sum(dim=1)
            out = np.empty(query.shape[0], dtype=np.float64)
            for s in range(0, query.shape[0], chunk):
                q = torch.from_numpy(query[s:s + chunk]).to(dev)
                j = torch.addmm(rn[None, :], q, r.T, alpha=-2.0).argmin(dim=1)
                # exact distance to the selected neighbour (no |q|^2 + |r|^2 - 2qr cancellation near zero)
                out[s:s + q.shape[0]] = torch.linalg.vector_norm(q.double() - r[j].double(), dim=1).cpu().numpy()
            del r, rn
            torch.cuda.empty_cache()
        return out
    except Exception as exc:  # noqa: BLE001 - any GPU problem -> NumPy path
        log(f"07: CUDA nearest-neighbour search failed ({exc!r}); using NumPy", "WARNING")
        return None


def _s07_nearest(query, ref, ref_groups=None, query_groups=None, chunk=2048, want_mean=False, self_ref=False):
    """Brute-force nearest Euclidean distances from ``query`` rows to ``ref`` rows (float32 BLAS, chunked).

    Uses d² = |q|² + |r|² − 2 q·r: one matrix product per chunk, then a row minimum; only the
    minimum is square-rooted. With groups, reference rows of the query row's own group are excluded (the
    different-fold nearest neighbour of the AOA); the reference rows are sorted by group once so the exclusion is a
    contiguous column slice. ``self_ref`` declares that ``query`` and ``ref`` are the same rows (each row's zero
    distance to itself is excluded). With ``want_mean`` the mean of all pairwise distances (self pairs excluded)
    is also returned - the AOA mean pairwise distance.
    """
    query = np.ascontiguousarray(query, np.float32)
    ref = np.ascontiguousarray(ref, np.float32)
    n_q, n_r = query.shape[0], ref.shape[0]
    if ref_groups is None and not want_mean and not self_ref and n_q * n_r >= 2e8:
        fast = _s07_nearest_cuda(query, ref)
        if fast is not None:
            return fast
    order = np.arange(n_r)
    spans = None
    if ref_groups is not None:
        order = np.argsort(np.asarray(ref_groups), kind="stable")
        g_sorted = np.asarray(ref_groups)[order]
        uniq, first = np.unique(g_sorted, return_index=True)
        last = np.append(first[1:], n_r)
        spans = {int(g): (int(a), int(b)) for g, a, b in zip(uniq, first, last)}
        ref = ref[order]
    rn = _s07_sq_norms(ref).astype(np.float32)
    qn = _s07_sq_norms(query)
    self_pos = np.empty(n_r, dtype=np.int64)
    self_pos[order] = np.arange(n_r)              # column of every original reference row after sorting
    out = np.empty(n_q, dtype=np.float64)
    total, count = 0.0, 0
    for s in range(0, n_q, chunk):
        e = min(n_q, s + chunk)
        g = (query[s:e] * np.float32(-2.0)) @ ref.T   # (c, n_r) float32 = -2 q.r
        g += rn[None, :]
        rows = np.arange(e - s)
        if want_mean:
            d = np.sqrt(np.maximum(g + qn[s:e, None].astype(np.float32), 0.0))
            if self_ref:
                d[rows, self_pos[s:e]] = 0.0
            total += float(d.sum(dtype=np.float64))
            count += d.size - (e - s if self_ref else 0)
            del d
        if self_ref:
            g[rows, self_pos[s:e]] = np.inf
        if spans is not None:
            qg = np.asarray(query_groups[s:e])
            for grp in np.unique(qg):
                if int(grp) in spans:
                    a, b = spans[int(grp)]
                    g[np.flatnonzero(qg == grp), a:b] = np.inf
        j = g.argmin(axis=1)
        best = g[rows, j]
        # exact distance to the selected neighbour (avoids |q|^2 + |r|^2 - 2qr cancellation near zero)
        exact = np.sqrt(((query[s:e].astype(np.float64) - ref[j].astype(np.float64)) ** 2).sum(axis=1))
        out[s:e] = np.where(np.isfinite(best), exact, np.inf)
        del g
    return (out, total / max(count, 1)) if want_mean else out


def _s07_decile_error_model(resid, weight, dec, rng, edges):
    """Per-DI-decile error model of signed residuals: rows, |resid| q50/q90/q95, weighted RMSE, signed samples.

    Deciles with < 5 rows borrow the quantiles and samples of the nearest populated decile (monotone use downstream);
    their ``rmse`` stays NaN and ``n`` is their own count.
    """
    by_decile, q50, q90, q95, samples = [], [], [], [], []
    bounds = [0.0] + list(edges) + [None]
    for k in range(10):
        sel = dec == k
        n_k = int(sel.sum())
        if n_k >= 5:
            a = np.abs(resid[sel])
            rm = float(np.sqrt(np.sum(weight[sel] * resid[sel] ** 2) / np.sum(weight[sel])))
            qq = np.quantile(a, [0.5, 0.9, 0.95])
            pick = rng.choice(np.flatnonzero(sel), size=min(_S07_RESID_SAMPLES, n_k), replace=False)
            smp = resid[pick].astype(np.float32)
        else:
            rm, qq, smp = np.nan, [np.nan] * 3, np.zeros(0, np.float32)
        by_decile.append({"decile": k, "di_lo": bounds[k], "di_hi": bounds[k + 1], "rmse": rm, "n": n_k})
        q50.append(float(qq[0]))
        q90.append(float(qq[1]))
        q95.append(float(qq[2]))
        samples.append(smp)
    filled = [k for k in range(10) if np.isfinite(q90[k])]
    for k in range(10):
        if not np.isfinite(q90[k]) and filled:
            j = min(filled, key=lambda f: abs(f - k))
            q50[k], q90[k], q95[k], samples[k] = q50[j], q90[j], q95[j], samples[j]
    return by_decile, q50, q90, q95, samples


def _s07_block_resid_rho(rows, resid):
    """Pooled intra-block20 correlation of residuals (D13).

    Per frame, over the block20 blocks with >= 5 rows: rho_t = (var(block means) - mean(within-block var / n_b)) /
    var(residuals of the frame) - the between-block variance share (ANOVA estimator). Pooled as the row-weighted mean
    over frames and clipped to [0, 0.95]. Returns (rho, per-frame rows).
    """
    years, blocks = _S07_YEAR[rows], _S07_BLOCK20[rows]
    per_frame, num, den = [], 0.0, 0.0
    for t in np.unique(years):
        sel = years == t
        r_t = resid[sel].astype(np.float64)
        if r_t.size < 10:
            continue
        v_tot = float(np.var(r_t, ddof=1))
        uniq, inv, cnt = np.unique(blocks[sel], return_inverse=True, return_counts=True)
        s1 = np.bincount(inv, weights=r_t, minlength=uniq.size)
        s2 = np.bincount(inv, weights=r_t * r_t, minlength=uniq.size)
        keep = cnt >= 5
        if keep.sum() < 2 or not v_tot > 0:
            continue
        n_b = cnt[keep].astype(np.float64)
        means = s1[keep] / n_b
        wvar = np.maximum(s2[keep] - n_b * means * means, 0.0) / (n_b - 1.0)
        rho_t = (float(np.var(means, ddof=1)) - float(np.mean(wvar / n_b))) / v_tot
        n_t = float(n_b.sum())
        per_frame.append({"year": int(t), "rho": rho_t, "n_rows": int(n_t), "n_blocks": int(keep.sum())})
        num += rho_t * n_t
        den += n_t
    rho = float(np.clip(num / den, 0.0, 0.95)) if den > 0 else 0.0
    return rho, per_frame


def _s07_aoa(ctx, booster, oof_block20, loeo_pred=None, rbar_rows=None):
    """AOA dict (SPEC §3.4 + extras) for one mode.

    DI model (complete training rows only, D12), the conformal block20 error model by DI decile and, given the
    recentred LOEO predictions and the per-row persistent residual of the other frames, the projection-setting error
    model (D13): ``resid_samples_proj``, ``conformal_proj`` and ``resid_rho_block20``.
    """
    import xgboost as xgb

    tr = ctx["train"]
    rng = np.random.default_rng(CFG.SEED)
    shap_rows = np.sort(rng.choice(tr, size=min(_S07_AOA_SAMPLE, tr.size), replace=False))
    # mean |SHAP| of the final model on the sample (TreeSHAP; on the GPU when XGBoost has CUDA)
    xs = _s07_take(shap_rows)
    try:
        if ENV.get("xgb_device") == "cuda":
            booster.set_param({"device": xgb_device_for(0)})
        contrib = booster.predict(xgb.DMatrix(xs), pred_contribs=True)
    finally:
        booster.set_param({"device": "cpu"})
    weights = np.abs(contrib[:, :len(FEATURES)]).mean(axis=0).astype(np.float64)
    with _s07_warnings.catch_warnings():
        _s07_warnings.simplefilter("ignore", RuntimeWarning)
        mean = np.nanmean(_S07_X[tr], axis=0).astype(np.float64)
        std = np.nanstd(_S07_X[tr], axis=0).astype(np.float64)
    mean = np.where(np.isfinite(mean), mean, 0.0)
    std = np.where(np.isfinite(std) & (std > 0), std, 1.0)
    used = weights > 0

    def complete(rows):
        """Rows with every AOA-weighted feature finite (D12: the others get DI = +inf)."""
        return np.isfinite(_S07_X[rows][:, used]).all(axis=1)

    def transform(x):
        z = (np.asarray(x, np.float64) - mean) / std
        return (np.nan_to_num(z, nan=0.0) * weights).astype(np.float32)   # NaN only in weight-0 features here

    # The AOA training sample is drawn from COMPLETE training rows only, so d_bar, the threshold and di_train stay
    # finite and describe real feature vectors (D12).
    tr_complete = tr[complete(tr)]
    if tr_complete.size < 10:
        raise RuntimeError(f"AOA: only {tr_complete.size} training rows have every weighted feature")
    sample = np.sort(np.random.default_rng([int(CFG.SEED), 707]).choice(
        tr_complete, size=min(_S07_AOA_SAMPLE, tr_complete.size), replace=False))
    S = transform(_S07_X[sample])
    folds = _S07_FOLD20[sample]
    di_train, d_bar = _s07_nearest(S, S, folds, folds, want_mean=True, self_ref=True)
    if not np.isfinite(d_bar) or d_bar <= 0:
        raise RuntimeError("AOA: mean pairwise distance of the training sample is not positive")
    di_train = di_train / d_bar
    finite_train = di_train[np.isfinite(di_train)]
    q25, q75 = np.percentile(finite_train, [25, 75])
    threshold = float(q75 + 1.5 * (q75 - q25))

    def di_other_fold(rows):
        """DI with the nearest training-sample row from a DIFFERENT block20 fold; +inf for incomplete rows."""
        out = np.full(rows.size, np.inf)
        ok = complete(rows)
        if ok.any():
            out[ok] = _s07_nearest(transform(_S07_X[rows[ok]]), S, folds, _S07_FOLD20[rows[ok]]) / d_bar
        return out

    # Conformal error model on block20 OOF residuals of up to 100k training rows (DI: different-fold neighbour;
    # rows with a missing weighted feature have DI = inf and land in the top decile).
    cand = tr[np.isfinite(oof_block20[tr])]
    rows = np.sort(rng.choice(cand, size=min(_S07_CONFORMAL_ROWS, cand.size), replace=False)) if cand.size else cand
    edges = np.quantile(finite_train, np.linspace(0.1, 0.9, 9)).tolist()
    by_decile, q50, q90, q95, samples = [], [], [], [], []
    di_cv = np.zeros(0)
    if rows.size:
        di_cv = di_other_fold(rows)
        resid = ctx["y"][rows].astype(np.float64) - oof_block20[rows].astype(np.float64)
        wr = ctx["w"][rows].astype(np.float64)
        dec = np.searchsorted(np.asarray(edges), di_cv, side="right")
        by_decile, q50, q90, q95, samples = _s07_decile_error_model(resid, wr, dec, rng, edges)
    out = {"weights": weights.astype(np.float32), "mean": mean.astype(np.float32), "std": std.astype(np.float32),
           "train_sample": S, "train_rows": sample, "d_bar": float(d_bar), "threshold": threshold,
           "di_train": di_train.astype(np.float32), "rmse_by_di_decile": by_decile,
           "conformal": {"edges": edges, "abs_resid_quantiles": {"q50": q50, "q90": q90, "q95": q95}},
           "resid_samples": samples, "feature_order": list(FEATURES), "n_conformal_rows": int(rows.size),
           "n_conformal_rows_incomplete": int(np.sum(~np.isfinite(di_cv))),
           "resid_rho_block20": 0.0, "resid_rho_by_frame": [],
           "projection_error_model": "unavailable (no LOEO predictions)"}

    # ---- projection-setting error model (D13): a known place at another time --------------------------------
    # rho_proj = y - (recentred LOEO prediction + rbar^(-t)) is the residual of exactly the predictor used for
    # modelled and projected frames (final model + persistent residual of the cell's other frames).
    if loeo_pred is None or rbar_rows is None:
        return out
    lp = np.asarray(loeo_pred, np.float64)
    rb = np.asarray(rbar_rows, np.float64)
    all_p = tr[np.isfinite(lp[tr]) & np.isfinite(rb[tr])]
    if all_p.size < 50:
        log(f"07: {ctx['mode']}: only {all_p.size} rows with LOEO predictions - projection error model unavailable",
            "WARNING")
        return out
    rho_all = ctx["y"][all_p].astype(np.float64) - (lp[all_p] + rb[all_p])
    rho_b20, per_frame = _s07_block_resid_rho(all_p, rho_all)
    out["resid_rho_block20"] = rho_b20
    out["resid_rho_by_frame"] = per_frame
    out["projection_error_model"] = ("recentred LOEO prediction + persistent residual of the other frames; deciles of "
                                     "the different-fold DI with the conformal edges")
    in_p = (np.isfinite(lp[rows]) & np.isfinite(rb[rows])) if rows.size else np.zeros(0, bool)
    if in_p.sum() >= 50:
        rows_p = rows[in_p]
        resid_p = ctx["y"][rows_p].astype(np.float64) - (lp[rows_p] + rb[rows_p])
        dec_p = np.searchsorted(np.asarray(edges), di_cv[in_p], side="right")
        bd_p, p50, p90, p95, smp_p = _s07_decile_error_model(resid_p, ctx["w"][rows_p].astype(np.float64), dec_p,
                                                             rng, edges)
        if all(np.isfinite(p90)):
            out["conformal_proj"] = {"edges": edges, "abs_resid_quantiles": {"q50": p50, "q90": p90, "q95": p95}}
            out["resid_samples_proj"] = smp_p
            out["rmse_by_di_decile_proj"] = bd_p
            out["n_proj_rows"] = int(rows_p.size)
    return out


def COMPUTE_DI(m, X):
    """Dissimilarity index (Meyer & Pebesma 2021) of feature rows for mode ``m`` → float32 array.

    ``X``: DataFrame with the ``FEATURES`` columns or an (n, 47) array. A row with a NaN in any feature of positive
    AOA weight gets DI = +inf (outside the AOA and in the top DI decile, D12): its distance to the training data is
    unknown, and imputing the training mean would make it look typical. DI > ``AOA[m]["threshold"]`` means outside
    the area of applicability.
    """
    a = AOA[m]
    Xa = _s07_as_matrix(X)
    w = a["weights"].astype(np.float64)
    out = np.full(Xa.shape[0], np.inf, dtype=np.float32)
    ok = np.isfinite(Xa[:, w > 0]).all(axis=1)
    if ok.any():
        z = (Xa[ok].astype(np.float64) - a["mean"].astype(np.float64)) / a["std"].astype(np.float64)
        q = (np.nan_to_num(z, nan=0.0) * w).astype(np.float32)
        out[ok] = (_s07_nearest(q, a["train_sample"]) / a["d_bar"]).astype(np.float32)
    return out

## 7.7 Run the suite per mode
Order per mode: (1) all CV jobs in one scheduled pass (six schemes, block-size curve, base-only ablation);
(2) collection, summaries, error-vs-distance, Moran, fold geometry; (3) final model, AOA, persistent residual
(stage-cached together, keyed by the CV results). The block-size curve uses 5 greedy folds per block size
(5/10/20/40/66 km) plus the random folds; the **ablation** refits block20 XGBoost on the 33 base features only, so
the gain from neighbourhood context is measured on identical folds. It isolates the **14 focal means only**: the
distances, TPI and ERA5 features are in both runs, so its gain should be well below the +0.2 R² of the real-data
experiments (0.63 → 0.83), which added focal means together with distance and relative-elevation features.

In [ ]:
MODEL_RESULTS, FINAL_MODELS, PERSISTENT_RESID, PERSISTENT_LAMBDA, AOA = {}, {}, {}, {}, {}
_S07_CTX = {}
_S07_TIMES = {}
_s07_resolve_lgbm_device()


def _s07_run_mode(mode):
    """Full validation suite + final products of one mode → MODEL_RESULTS[mode] etc."""
    t_mode = _s07_time.perf_counter()
    ctx = _s07_context(mode)
    _S07_CTX[mode] = ctx
    if ctx["train"].size == 0:
        log(f"07: mode {mode} has no training rows - skipped", "WARNING")
        return
    log(f"07: === mode {mode}: {ctx['train'].size:,} training rows, frames {ctx['frames']} ===")

    # ---- (1) job list --------------------------------------------------------------------------------------------
    groups, jobs, jobs_by_key = {}, [], {}
    for scheme in _S07_SCHEMES:
        folds = _s07_folds(ctx, scheme)
        models = [m for m in _S07_SCHEME_MODELS[scheme] if _s07_model_available(m)]
        if not folds:
            log(f"07: {mode}: scheme {scheme} has no usable folds - skipped", "WARNING")
            continue
        for fold in folds:
            _s07_check_fold(ctx, scheme, fold)
        groups[scheme] = {"folds": folds, "models": models}
    curve_info = {}
    for km in _S07_CURVE_KM:
        b = max(1, int(round(km * 1000.0 / float(GRID.res))))
        fold_rows, _ = _s07_block_fold_rows(b, _S07_CURVE_FOLDS)
        f = fold_rows[ctx["train"]]
        folds = [{"fold": int(k), "train": ctx["train"][f != k], "test": ctx["train"][f == k]} for k in np.unique(f)]
        folds = [fd for fd in folds if fd["train"].size and fd["test"].size]
        for fold in folds:
            _s07_check_fold(ctx, f"curve_{km}", fold)
        groups[f"curve_{km}"] = {"folds": folds, "models": ["xgboost"]}
        curve_info[km] = {"block_cells": b, "block_km_actual": b * float(GRID.res) / 1000.0}
    if _S07_RUN_ABLATION and "block20" in groups:
        groups["ablation_base"] = {"folds": groups["block20"]["folds"], "models": ["xgboost"], "feat": "base"}
    for label, g in groups.items():
        for model in g["models"]:
            for fold in g["folds"]:
                job = _s07_make_job(ctx, label, model, fold, feat=g.get("feat", "all"))
                jobs.append(job)
                jobs_by_key[(label, model, fold["fold"])] = job
    if "gnn" in {j["model"] for j in jobs}:
        _s07_gnn_graph(ctx)                                    # build once, before the lanes start

    with timer(f"07_{mode}_cv_jobs"):
        results, busy, lane_counts = _s07_run_jobs(jobs, mode)

    # ---- (2) collection ------------------------------------------------------------------------------------------
    oof_all, fold_rows, infos_all, summary_rows = {}, [], {}, []
    for scheme in _S07_SCHEMES:
        if scheme not in groups:
            continue
        g = groups[scheme]
        oof, raw, rows, infos = _s07_collect(ctx, scheme, g["models"], g["folds"], jobs_by_key, results)
        oof_all[scheme] = oof
        if scheme in ("loeo", "place_time"):
            oof_all[f"{scheme}_raw"] = raw
        fold_rows.extend(rows)
        infos_all[scheme] = infos
        for model in g["models"]:
            summary_rows.append(_s07_summary_row(ctx, scheme, model, oof[model], rows))
    cv_fold_metrics = pd.DataFrame(fold_rows, columns=["scheme", "model", "fold", "r2", "rmse", "mae", "bias",
                                                       "n_train", "n_test", "fit_seconds", "device"])
    cv_summary = pd.DataFrame(summary_rows, columns=["scheme", "model", "r2_pooled", "rmse_pooled", "mae_pooled",
                                                     "r2_fold_mean", "r2_fold_sd", "rmse_fold_mean", "rmse_fold_sd",
                                                     "n_folds", "n"])

    def summary_of(scheme, model="xgboost"):
        sub = cv_summary[(cv_summary["scheme"] == scheme) & (cv_summary["model"] == model)]
        return sub.iloc[0].to_dict() if len(sub) else {}

    # block-size curve (5 greedy folds per size + the random folds of the random scheme)
    curve = []
    for km in _S07_CURVE_KM:
        g = groups[f"curve_{km}"]
        oof, _, rows, _ = _s07_collect(ctx, f"curve_{km}", ["xgboost"], g["folds"], jobs_by_key, results)
        row = _s07_summary_row(ctx, f"curve_{km}", "xgboost", oof["xgboost"], rows)
        curve.append({"block_km": float(km), "label": f"{km} km", "block_km_actual": curve_info[km]["block_km_actual"],
                      "r2_pooled": row["r2_pooled"], "rmse_pooled": row["rmse_pooled"],
                      "r2_fold_mean": row["r2_fold_mean"], "n_folds": row["n_folds"]})
    rnd = summary_of("random")
    if rnd:
        curve.append({"block_km": None, "label": "random", "block_km_actual": None, "r2_pooled": rnd["r2_pooled"],
                      "rmse_pooled": rnd["rmse_pooled"], "r2_fold_mean": rnd["r2_fold_mean"],
                      "n_folds": rnd["n_folds"]})

    ablation = None
    if "ablation_base" in groups:
        oof_b, _, rows_b, _ = _s07_collect(ctx, "ablation_base", ["xgboost"], groups["ablation_base"]["folds"],
                                           jobs_by_key, results)
        base_row = _s07_summary_row(ctx, "ablation_base", "xgboost", oof_b["xgboost"], rows_b)
        full_row = summary_of("block20")
        ablation = {"scheme": "block20", "model": "xgboost",
                    "base_only": {k: base_row[k] for k in ("r2_pooled", "rmse_pooled", "r2_fold_mean", "r2_fold_sd")},
                    "all_features": {k: full_row.get(k) for k in ("r2_pooled", "rmse_pooled", "r2_fold_mean",
                                                                  "r2_fold_sd")},
                    "n_features": {"base_only": len(BASE_FEATURES), "all_features": len(FEATURES)}}
        ablation["gain_r2_pooled"] = (ablation["all_features"]["r2_pooled"] or np.nan) - base_row["r2_pooled"]

    # LOEO table (xgboost headline + every model)
    loeo_by_model = {}
    if "loeo" in groups:
        for model in groups["loeo"]["models"]:
            entries = []
            for fold in groups["loeo"]["folds"]:
                te = fold["test"]
                raw_p, rec_p = oof_all["loeo_raw"][model][te], oof_all["loeo"][model][te]
                if not np.isfinite(raw_p).any():
                    continue
                mr = _s07_wmetrics(ctx["y"][te], raw_p, ctx["w"][te])
                mc = _s07_wmetrics(ctx["y"][te], rec_p, ctx["w"][te])
                entries.append({"held_out": int(fold["fold"]), "r2_raw": mr["r2"], "rmse_raw": mr["rmse"],
                                "bias_raw": mr["bias"], "r2_recentred": mc["r2"], "rmse_recentred": mc["rmse"],
                                "bias_recentred": mc["bias"], "n": int(te.size), "model": model})
            loeo_by_model[model] = entries
    place_time = {}
    if "place_time" in groups:
        pr, pc = oof_all["place_time_raw"]["xgboost"], oof_all["place_time"]["xgboost"]
        ok = np.flatnonzero(np.isfinite(pr))
        mr = _s07_wmetrics(ctx["y"][ok], pr[ok], ctx["w"][ok])
        mc = _s07_wmetrics(ctx["y"][ok], pc[ok], ctx["w"][ok])
        place_time = {"model": "xgboost", "test_frame": ctx["frames"][-1], "train_frames": ctx["frames"][:-1],
                      "r2": mc["r2"], "rmse": mc["rmse"], "recentred": True, "r2_raw": mr["r2"],
                      "rmse_raw": mr["rmse"], "bias_raw": mr["bias"], "r2_recentred": mc["r2"],
                      "rmse_recentred": mc["rmse"], "n": int(ok.size),
                      "fold_r2_mean": summary_of("place_time").get("r2_fold_mean"),
                      "fold_r2_sd": summary_of("place_time").get("r2_fold_sd")}

    with timer(f"07_{mode}_diagnostics"):
        evd, _ = (_s07_error_vs_distance(ctx, groups["block20"]["folds"], oof_all["block20"]["xgboost"])
                  if "block20" in groups else ([], None))
        moran, moran_target, moran_scatter = _s07_moran_tables(ctx, oof_all.get("block20", {}))
        geometry = _s07_fold_geometry(ctx, oof_all)

    # ---- (3) final model, AOA, persistent residual (stage-cached on the CV results) ------------------------------
    best_rounds = _s07_final_rounds(infos_all.get("block20", {}).get("xgboost", []),
                                    infos_all.get("random", {}).get("xgboost", [])
                                    + infos_all.get("loeo", {}).get("xgboost", []))
    final_rounds = max(1, int(round(1.1 * float(np.median(best_rounds)))))
    loeo_x = oof_all.get("loeo", {}).get("xgboost", np.full(_S07_N, np.nan, np.float32))
    oof_b20 = oof_all.get("block20", {}).get("xgboost", np.full(_S07_N, np.nan, np.float32))
    post_key = CFG.key("XGB_PARAMS", version=_S07_VERSION, mode=mode, df=_S07_DF_FP, rounds=final_rounds,
                       loeo=_s07_hash(np.nan_to_num(loeo_x, nan=-999.0)),
                       oof=_s07_hash(np.nan_to_num(oof_b20, nan=-999.0)), aoa_sample=_S07_AOA_SAMPLE,
                       aoa_impl=4)

    def compute_post():
        import xgboost as xgb  # noqa: F401 - imported for the booster type

        with timer(f"07_{mode}_final_model"):
            booster = _s07_train_final(ctx, final_rounds)
        with timer(f"07_{mode}_persistent_residual"):
            grid, lam, report, rbar_rows = _s07_persistent_residual(ctx, loeo_x)
        with timer(f"07_{mode}_aoa"):
            aoa = _s07_aoa(ctx, booster, oof_b20, loeo_x, rbar_rows)
        return {"booster_raw": bytes(booster.save_raw("ubj")), "aoa": aoa, "resid": grid, "lambda": lam,
                "resid_report": report}

    post = stage_cache(f"07_post_{mode}", post_key).get_or_compute(compute_post)
    import xgboost as xgb

    booster = xgb.Booster()
    booster.load_model(bytearray(post["booster_raw"]))
    booster.set_param({"device": "cpu"})
    FINAL_MODELS[mode] = booster
    AOA[mode] = post["aoa"]
    PERSISTENT_RESID[mode] = post["resid"]
    PERSISTENT_LAMBDA[mode] = float(post["lambda"])
    try:
        booster.save_model(str(Path(CFG.MODEL_DIR) / f"xgb_final_{mode}.json"))
    except Exception as exc:  # noqa: BLE001
        log(f"07: could not save the final {mode} model: {exc!r}", "WARNING")

    # AOA share inside per observed frame (all ROI cells of the frame)
    with timer(f"07_{mode}_aoa_frames"):
        di_frames, pct_inside = {}, {}
        for j, year in enumerate(_S07_FRAMES_DF):
            di = COMPUTE_DI(mode, _S07_X[j * int(N_CELLS):(j + 1) * int(N_CELLS)])
            di_frames[year] = di                                 # +inf for rows with a missing weighted feature
            pct_inside[year] = float(np.mean(di <= AOA[mode]["threshold"]))
        AOA[mode]["di_frames"] = di_frames
        AOA[mode]["pct_inside"] = pct_inside

    b20, b5, lo = summary_of("block20"), summary_of("block5"), summary_of("loeo")
    MODEL_RESULTS[mode] = {
        "cv_fold_metrics": cv_fold_metrics, "cv_summary": cv_summary, "oof": oof_all,
        "block_size_curve": curve, "error_vs_distance": evd,
        "loeo": loeo_by_model.get("xgboost", []), "loeo_by_model": loeo_by_model, "place_time": place_time,
        "moran": moran, "moran_target": moran_target, "moran_scatter": moran_scatter, "fold_geometry": geometry,
        "best_rounds": [int(r) for r in best_rounds], "final_rounds": final_rounds,
        "ablation": ablation, "persistent_residual": post["resid_report"],
        "aoa": {"di_threshold": AOA[mode]["threshold"], "pct_inside": pct_inside,
                "rmse_by_di_decile": AOA[mode]["rmse_by_di_decile"], "d_bar": AOA[mode]["d_bar"],
                "resid_rho_block20": float(AOA[mode].get("resid_rho_block20", 0.0)),
                **({"proj_resid_q90": [float(v) for v in AOA[mode]["conformal_proj"]["abs_resid_quantiles"]["q90"]]}
                   if AOA[mode].get("conformal_proj") else {})},
        "v1_comparison": {**_S07_V1_REFERENCE, "v2_block5_pooled": b5.get("r2_pooled"),
                          "v2_block5_fold_mean": b5.get("r2_fold_mean"), "v2_block20_pooled": b20.get("r2_pooled"),
                          "label": ("reference values from the real v1 run" if DATA_MODE == "synthetic"
                                    else "v1 real-data run (MODIS Terra day, 17 features)")},
        "headline": {"spatial": {"scheme": "block20", "model": "xgboost", "r2": b20.get("r2_pooled"),
                                 "rmse": b20.get("rmse_pooled")},
                     "temporal": {"scheme": "loeo", "model": "xgboost", "r2": lo.get("r2_pooled"),
                                  "rmse": lo.get("rmse_pooled"), "recentred": True}},
        "district_merges": ctx.get("district_merges", []),
        "frames": ctx["frames"], "n_train": int(ctx["train"].size),
        "monotone": {f: int(v) for f, v in zip(FEATURES, ctx["mono"]) if v},
        "lane_busy_seconds": {k: round(v, 1) for k, v in busy.items()},
        "lane_job_counts": {k: int(v) for k, v in lane_counts.items()},
        "seconds": round(_s07_time.perf_counter() - t_mode, 1),
    }
    _S07_TIMES[mode] = MODEL_RESULTS[mode]["seconds"]
    ctx["gnn"] = None
    free_memory()


for _s07_mode in CFG.MODES:
    with timer(f"07_{_s07_mode}"):
        _s07_run_mode(_s07_mode)

## 7.8 Results
Tables: pooled skill per scheme and model, LOEO raw vs recentred, the base-only ablation, the persistent-residual
λ scan and the AOA summary. Figure per mode: (a) pooled R² by scheme/model with fold spread, (b) block-size
curve, (c) error vs distance, (d) block20 fold map coloured by block RMSE, (e) predicted vs observed anomaly,
(f) RMSE by DI decile with the AOA threshold.

In [ ]:
def _s07_show(obj):
    """display() inside a notebook kernel, full-width print() in plain scripts."""
    import sys

    if "ipykernel" in sys.modules:
        try:
            from IPython.display import display

            display(obj)
            return
        except Exception:  # noqa: BLE001
            pass
    print(obj.to_string() if isinstance(obj, pd.DataFrame) else obj)


def _s07_print_results(mode):
    res = MODEL_RESULTS[mode]
    print(f"\n=== {mode.upper()} === training rows {res['n_train']:,}, frames {res['frames']}, "
          f"final XGBoost rounds {res['final_rounds']} (median best {int(np.median(res['best_rounds']))})")
    print("Pooled OOF skill (weighted; anomaly °C):")
    _s07_show(res["cv_summary"].round(4))
    if res["loeo"]:
        print("Leave-one-epoch-out (xgboost): raw vs recentred over the held-out frame's fixed mask")
        _s07_show(pd.DataFrame(res["loeo"]).drop(columns=["model"]).round(4))
    if res["place_time"]:
        pt = res["place_time"]
        print(f"Unseen place + time (train < {pt['test_frame']}, block20 folds): R2 raw {pt['r2_raw']:.3f} -> "
              f"recentred {pt['r2_recentred']:.3f} (RMSE {pt['rmse_recentred']:.3f})")
    if res["ablation"]:
        ab = res["ablation"]
        print(f"Ablation (block20 xgboost, pooled R2): base-only {ab['base_only']['r2_pooled']:.4f} "
              f"(33 features) vs all {ab['all_features']['r2_pooled']:.4f} (47) -> gain {ab['gain_r2_pooled']:+.4f}")
    print("Block-size curve:", ", ".join(f"{c['label']}: {c['r2_pooled']:.3f}" for c in res["block_size_curve"]))
    print("Error vs distance:", ", ".join(f"[{e['lo']:g},{e['hi'] if e['hi'] is not None else 'inf'}) "
                                          f"{e['rmse']:.3f} (n={e['n']})" for e in res["error_vs_distance"]))
    pr = res["persistent_residual"]
    if pr.get("by_lambda"):
        print(f"Persistent residual: lambda = {pr['lambda'] if pr['lambda'] is not None else 'inf'}; LOEO RMSE "
              f"{pr['loeo_rmse_without']:.4f} -> {pr['loeo_rmse_with']:.4f} (R2 {pr['loeo_r2_without']:.4f} -> "
              f"{pr['loeo_r2_with']:.4f}); scan: "
              + ", ".join(f"{b['lambda'] if b['lambda'] is not None else 'inf'}:{b['rmse']:.4f}"
                          for b in pr["by_lambda"]))
    a = AOA[mode]
    print(f"AOA: d_bar {a['d_bar']:.4f}, DI threshold {a['threshold']:.4f}, inside share by frame "
          + ", ".join(f"{y}:{v:.3f}" for y, v in MODEL_RESULTS[mode]["aoa"]["pct_inside"].items()))
    print("Conformal |resid| q90 by DI decile:", np.round(a["conformal"]["abs_resid_quantiles"]["q90"], 3).tolist())
    if a.get("conformal_proj"):
        print("Projection-setting (LOEO + persistent residual) |resid| q90 by DI decile:",
              np.round(a["conformal_proj"]["abs_resid_quantiles"]["q90"], 3).tolist(),
              f"| sd per decile {[round(float(np.std(v)), 3) for v in a['resid_samples_proj']]}")
    print(f"block20 residual correlation rho = {a.get('resid_rho_block20', 0.0):.3f} (per frame "
          + ", ".join(f"{r['year']}:{r['rho']:.2f}" for r in a.get("resid_rho_by_frame", [])) + ")")
    n_inf = {y: int(np.sum(~np.isfinite(v))) for y, v in a.get("di_frames", {}).items()}
    if any(n_inf.values()):
        print("Rows with DI = inf (a weighted feature missing; outside the AOA):", n_inf)
    if len(res["moran"]):
        print("Moran's I of block20 residuals (model x year):")
        _s07_show(res["moran"].pivot(index="model", columns="year", values="I").round(3))
    print(f"lane busy seconds {res['lane_busy_seconds']} (jobs {res.get('lane_job_counts')}), mode wall time "
          f"{res['seconds']:.0f}s")


def _s07_plot_mode(mode):
    import matplotlib.pyplot as plt

    res = MODEL_RESULTS[mode]
    ctx = _S07_CTX[mode]
    fig, axes = plt.subplots(2, 3, figsize=(19, 10.5))
    summ = res["cv_summary"]
    schemes = [s for s in _S07_SCHEMES if s in set(summ["scheme"])]
    models = [m for m in ["ridge", "random_forest", "lightgbm", "xgboost", "gnn"] if m in set(summ["model"])]
    colors = {"ridge": "#a8a29e", "random_forest": "#16a34a", "lightgbm": "#7c3aed", "xgboost": "#eb6834",
              "gnn": "#2a78d6"}
    ax = axes[0, 0]
    width = 0.8 / max(len(models), 1)
    for i, model in enumerate(models):
        sub = summ[summ["model"] == model].set_index("scheme").reindex(schemes)
        ax.bar(np.arange(len(schemes)) + (i - (len(models) - 1) / 2) * width, sub["r2_pooled"], width * 0.92,
               yerr=sub["r2_fold_sd"], capsize=2, color=colors[model], label=model,
               error_kw={"elinewidth": 0.8, "ecolor": "#52514e"})
    ax.set_xticks(range(len(schemes)))
    ax.set_xticklabels(schemes)
    ax.set_ylabel("pooled OOF R² (anomaly)")
    lo_r2 = float(np.nanmin(summ["r2_pooled"])) if len(summ) else 0.0
    ax.set_ylim(max(-0.5, min(lo_r2 - 0.15, 0.95)), 1.0)
    ax.legend(frameon=False, fontsize=8, ncol=len(models), loc="lower center")
    ax.set_title(f"{mode}: skill by scheme (bars pooled, whiskers fold s.d.)")
    ax = axes[0, 1]
    pts = [c for c in res["block_size_curve"] if c["block_km"] is not None]
    ax.plot([c["block_km_actual"] for c in pts], [c["r2_pooled"] for c in pts], "o-", color="#eb6834", label="blocks")
    rnd = [c for c in res["block_size_curve"] if c["block_km"] is None]
    if rnd:
        ax.axhline(rnd[0]["r2_pooled"], color="#2a78d6", linestyle="--", label="random rows")
    ax.set_xscale("log")
    ax.set_xlabel("block size (km)")
    ax.set_ylabel("pooled R²")
    ax.set_title("Block-size curve (xgboost, 5 folds)")
    ax.legend(frameon=False)
    ax = axes[0, 2]
    evd = [e for e in res["error_vs_distance"] if e["n"] > 0]
    ax.bar(range(len(evd)), [e["rmse"] for e in evd], color="#eb6834")
    ax.set_xticks(range(len(evd)))
    ax.set_xticklabels([f"{e['lo']:g}-{e['hi'] if e['hi'] is not None else '∞'}" for e in evd])
    ax.set_xlabel("distance to nearest training cell (km)")
    ax.set_ylabel("RMSE (°C)")
    ax.set_title("Error vs distance (block20 xgboost)")
    ax = axes[1, 0]
    geo = res["fold_geometry"]["block20"]
    b = geo["block_cells"]
    rm = np.array([np.nan if v is None else v for v in geo["rmse_of_block"]], float).reshape(geo["n_rows"],
                                                                                           geo["n_cols"])
    grid_rmse = np.kron(rm, np.ones((b, b)))[:GRID.height, :GRID.width]
    im = ax.imshow(np.where(ROI_MASK, grid_rmse, np.nan), cmap="magma")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label="block RMSE (°C)")
    ax.set_axis_off()
    ax.set_title("block20 fold map: pooled RMSE per 20 km block")
    ax = axes[1, 1]
    pred = res["oof"].get("block20", {}).get("xgboost")
    if pred is not None:
        ok = np.isfinite(pred)
        yv = ctx["y"][ok]
        lo_, hi_ = np.percentile(np.concatenate([yv, pred[ok]]), [0.5, 99.5])
        hb = ax.hexbin(yv, pred[ok], gridsize=60, extent=(lo_, hi_, lo_, hi_), mincnt=1, cmap="Blues", bins="log")
        ax.plot([lo_, hi_], [lo_, hi_], "--", color="#52514e", linewidth=1)
        fig.colorbar(hb, ax=ax, label="rows (log)")
        hl = res["headline"]["spatial"]
        ax.set_title(f"block20 xgboost OOF: R² {hl['r2']:.3f}, RMSE {hl['rmse']:.2f} °C")
        ax.set_xlabel("observed anomaly (°C)")
        ax.set_ylabel("predicted anomaly (°C)")
    ax = axes[1, 2]
    dec = AOA[mode]["rmse_by_di_decile"]
    ax.bar([d["decile"] for d in dec], [d["rmse"] for d in dec], color="#7c3aed")
    ax.set_xlabel("DI decile (training DI edges)")
    ax.set_ylabel("block20 OOF RMSE (°C)")
    ax.set_title(f"Error by dissimilarity; AOA threshold DI = {AOA[mode]['threshold']:.3f}")
    fig.suptitle(f"Model validation — {mode} ({DATA_MODE} data, {GRID.res} m grid)", fontsize=12)
    fig.tight_layout()
    fig.savefig(Path(CFG.FIG_DIR) / f"07_validation_{mode}.png", dpi=130)
    plt.show()
    plt.close(fig)


def _s07_save_tables(mode):
    out_dir = Path(CFG.PARQUET_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    for name in ("cv_fold_metrics", "cv_summary", "moran"):
        path = out_dir / f"{name}_{mode}.parquet"
        try:
            MODEL_RESULTS[mode][name].to_parquet(path, engine="pyarrow", compression="zstd", index=False)
        except Exception as exc:  # noqa: BLE001
            log(f"07: could not write {path}: {exc!r}", "WARNING")


Path(CFG.FIG_DIR).mkdir(parents=True, exist_ok=True)
for _s07_mode in MODEL_RESULTS:
    _s07_print_results(_s07_mode)
    _s07_save_tables(_s07_mode)
    try:
        _s07_plot_mode(_s07_mode)
    except Exception:  # noqa: BLE001 - a figure must never stop the pipeline
        log(f"07: validation figure for {_s07_mode} failed:\n{_s07_traceback.format_exc()}", "WARNING")

## 7.9 Contract check (SPEC §3.4) and runtime report
The check asserts the interface later sections rely on and re-verifies the leakage-relevant invariants on the
stored results (LOEO predictions of a frame come from a model that did not see it; OOF arrays cover only training
rows). The runtime table lists the wall-clock time of every 06/07 stage (`TIMINGS`), the basis for the Kaggle
budget (target ≤ 60–90 min for the whole suite on 2 × T4 at full resolution).

In [ ]:
def _s07_contract_check():
    for mode in MODEL_RESULTS:
        res = MODEL_RESULTS[mode]
        ctx = _S07_CTX[mode]
        for key in ("cv_fold_metrics", "cv_summary", "oof", "block_size_curve", "error_vs_distance", "loeo",
                    "place_time", "moran", "moran_target", "moran_scatter", "fold_geometry", "best_rounds"):
            assert key in res, f"MODEL_RESULTS[{mode}] lacks {key}"
        assert list(res["cv_fold_metrics"].columns) == ["scheme", "model", "fold", "r2", "rmse", "mae", "bias",
                                                         "n_train", "n_test", "fit_seconds", "device"]
        assert list(res["cv_summary"].columns)[:3] == ["scheme", "model", "r2_pooled"]
        not_train = np.ones(_S07_N, bool)
        not_train[ctx["train"]] = False
        for scheme, by_model in res["oof"].items():
            for model, arr in by_model.items():
                assert arr.dtype == np.float32 and arr.shape == (_S07_N,), (scheme, model)
                assert not np.isfinite(arr[not_train]).any(), f"{scheme}/{model} predicts non-training rows"
        assert isinstance(FINAL_MODELS[mode].num_boosted_rounds(), int)
        assert FINAL_MODELS[mode].num_boosted_rounds() == res["final_rounds"]
        assert PERSISTENT_RESID[mode].shape == ROI_MASK.shape and PERSISTENT_RESID[mode].dtype == np.float32
        assert np.isfinite(PERSISTENT_RESID[mode][ROI_MASK]).all() and not np.isfinite(
            PERSISTENT_RESID[mode][~ROI_MASK]).any()
        a = AOA[mode]
        for key in ("weights", "mean", "std", "train_sample", "d_bar", "threshold", "di_train", "rmse_by_di_decile",
                    "conformal"):
            assert key in a, f"AOA[{mode}] lacks {key}"
        assert np.isfinite(a["threshold"]) and a["threshold"] > 0 and len(a["conformal"]["edges"]) == 9
        assert all(len(a["conformal"]["abs_resid_quantiles"][q]) == 10 for q in ("q50", "q90", "q95"))
        j = len(_S07_FRAMES_DF) - 1
        rows = slice(j * int(N_CELLS), (j + 1) * int(N_CELLS))
        p = PREDICT_ANOM(mode, DF.iloc[rows][FEATURES])
        assert p.dtype == np.float32 and p.shape == (int(N_CELLS),)
        fixed = ctx["fixed_row"][rows]
        if fixed.any():
            assert abs(float(PREDICT_ANOM(mode, _S07_X[rows], fixed)[fixed].mean())) < 1e-3
        xr = _S07_X[rows]
        incomplete = ~np.isfinite(xr[:, a["weights"] > 0]).all(axis=1)
        pick = np.concatenate([np.flatnonzero(~incomplete)[:100], np.flatnonzero(incomplete)[:20]])
        di = COMPUTE_DI(mode, xr[pick])
        assert di.dtype == np.float32 and np.all(di >= 0)
        assert np.array_equal(np.isinf(di), incomplete[pick]) and np.all(np.isfinite(di[~incomplete[pick]]))
        assert np.all(np.isfinite(a["di_train"])) or np.isfinite(a["di_train"]).mean() > 0.99
        rho = a.get("resid_rho_block20")
        assert rho is not None and np.isfinite(rho) and 0.0 <= rho <= 0.95, rho
        if a.get("conformal_proj"):
            assert all(len(a["conformal_proj"]["abs_resid_quantiles"][q]) == 10 for q in ("q50", "q90", "q95"))
            assert all(np.isfinite(a["conformal_proj"]["abs_resid_quantiles"]["q90"]))
            assert len(a["resid_samples_proj"]) == 10 and all(len(v) > 0 for v in a["resid_samples_proj"])
            assert len(res["aoa"]["proj_resid_q90"]) == 10
        assert set(res["lane_busy_seconds"]) == {"gpu", "cpu", "lgbm"}
        if mode == "night":
            assert all(y in set(CFG.NIGHT_TRAIN_FRAMES) for y in res["frames"]), "night trained outside its frames"
    log("07: contract check passed (SPEC §3.4)")


_s07_contract_check()


def _s07_runtime_report():
    rows = [(k, v) for k, v in TIMINGS.items() if k.startswith(("06", "07"))]
    total_07 = sum(v for k, v in rows if k in {f"07_{m}" for m in CFG.MODES})
    print("Runtime of sections 06-07 (wall-clock seconds):")
    for k, v in rows:
        print(f"  {k:<40} {v:9.1f}")
    n_rows = {m: MODEL_RESULTS[m]["n_train"] for m in MODEL_RESULTS}
    print(f"  {'07 total (both modes)':<40} {total_07:9.1f}  ({total_07 / 60:.1f} min; training rows {n_rows}; "
          f"{ENV.get('n_gpus', 0)} GPU(s) {ENV.get('gpu_names')}, {ENV.get('cpu_count')} CPU threads)")
    return total_07


_S07_TOTAL_SECONDS = _s07_runtime_report()
free_memory()

# 8. Explainability per mode: grouped GPU TreeSHAP, interactions, block bootstrap, thresholds, zones, recommendations and the web surrogate

Everything in this section is done **separately for each mode** (`day`, `night`) with that mode's final model
`FINAL_MODELS[m]` (section 07). The products are the explanation layer of the dashboard (Drivers, Zones) and the
zone ids every timeline/projection frame is coloured with.

## 8.1 TreeSHAP and why it is reported per group
For a prediction $f(x)$ the Shapley value of feature $j$ is its average marginal contribution over all orderings
in which features are "switched on":

$$\phi_j(x)=\sum_{S\subseteq F\setminus\{j\}}\frac{|S|!\,(|F|-|S|-1)!}{|F|!}\big[f_x(S\cup\{j\})-f_x(S)\big].$$

It is the unique attribution with *efficiency* ($f(x)=\phi_0+\sum_j\phi_j$, $\phi_0=E[f(X)]$), symmetry, dummy and
linearity. **TreeSHAP** (Lundberg et al. 2020) computes it exactly for tree ensembles in $O(TLD^2)$ using the
trees' own cover statistics as the conditional expectation; XGBoost ships the CUDA implementation (GPUTreeShap,
Mitchell et al. 2022) through `predict(pred_contribs=True)`, ~20× faster than the CPU, which is what makes
explaining every cell × frame and every bootstrap replicate feasible on two T4s.

**Groups (SPEC §4.4, 18).** v2 has 47 features, 14 of which are neighbourhood (focal) means of a base feature
(`ndvi`, `ndvi_m3`, `ndvi_m15`) and some of which are near-duplicates (`dist_urban`/`dist_builtup`). Correlated
features split SHAP credit arbitrarily between them, so per-feature SHAP is unstable and hard to read. Shapley
values are additive, so the SHAP value of a *group* is exactly the sum of its members' values and the efficiency
property survives: $\text{pred} = \phi_0 + \sum_{g=1}^{18}\Phi_g$. All dependence curves, thresholds, zones and the
dashboard use the 18 group values; per-feature importance is kept only as a diagnostic.

**Additivity is checked, not assumed:** for every row, $\phi_0+\sum_g\Phi_g$ must equal the raw margin
(`output_margin=True`) within $10^{-3}$ °C, otherwise the section stops.

**What SHAP is not.** $\Phi_g(x)$ says how far the *current state* of driver $g$ pushes this cell's predicted
anomaly away from the average cell (a level), not how the prediction would respond if the driver changed (a
slope); and it explains the model, not causation. Responses are estimated separately with model counterfactuals
(8.7).

## 8.2 XAI frames
Day: all eight observed frames (Landsat targets 1990–2025). Night: also all eight — the model was trained on
2005–2025 (Aqua/VIIRS), 2000 is the Terra-reconstructed hindcast check, and the 1990/1995 rows explain the model's
**hindcast** of years without any night product (their tier is `modelled`, section 09). Cells whose 33 base features
are not all available (e.g. a Landsat gap) get NaN SHAP rows (nodata in the bundle); predictions are kept for every
ROI cell (trees route missing values).

## 8.3 Governance zones: K-means in grouped-SHAP space ("supervised clustering")
Clustering raw features groups cells that *look* alike; clustering the SHAP matrix groups cells whose predicted
anomaly is **attributed to the same drivers in similar amounts** — every coordinate is already in °C, so no scaling
is needed and irrelevant drivers contribute ~0 distance. K = 4 (fixed by the governance contract) is fitted on the
group-SHAP vectors pooled over the mode's **training frames**; every frame (observed, modelled and, in section 11,
projected) gets zone ids by the nearest centroid (`ASSIGN_ZONES`), so zones are comparable through time.
**Labelling rule (SPEC §1.5, both modes):** (1) **3 Heat Extreme Core** = highest mean total SHAP. (2) Among the
other three, a cluster is *riparian-eligible* if its mean `frac_water` ≥ 0.05, or ≥ 25 % of its rows have
`frac_water` ≥ 0.2, or the `water` group has its largest mean |SHAP| (of 18) with ≥ 0.25 °C. If any is eligible:
**1 Riparian Buffer** = the eligible cluster with the highest mean `frac_water`, **0 Ecological Cool Base** = the
lower mean total SHAP of the remaining two, **2 Transition** = the last one. If none is (e.g. the night zones of a
dry plain), the three are ordered by mean total SHAP (lowest → 0, middle → 1, highest → 2); the canonical ids,
names and colours stay, but zone 1 carries `display_name = "Moderate mosaic (no riparian signal)"` and a
`label_caveat`, so no map calls a dry mosaic a river buffer. K = 2..8 inertia/silhouette diagnostics are reported
for transparency, not as evidence that 4 is optimal.

## 8.4 Spatial block bootstrap
The uncertainty of an explanation comes mostly from *which places* the model learned from, and cells are spatially
autocorrelated, so rows are resampled in square blocks whose size is the distance at which the correlogram of the
block20 out-of-fold residuals of the last training frame drops below 0.1 (rounded up to 5 km, ≥ the inner 10 km
block; shrunk if < 50 blocks remain) unless `CFG.BOOT_BLOCK_KM` is set. Each replicate draws blocks with
replacement, gives every training row its block multiplicity × its sample weight, and refits the final model's
exact recipe (same rounds, same monotone constraints; no early stopping, so every replicate has the same capacity).
Replicates run one per GPU (`gpu_pool_map`), are stage-cached individually (an interrupted Kaggle session resumes)
and are **reduced immediately** (importance, binned dependence curves on a fixed evaluation sample, thresholds,
recommendation counterfactuals), so memory stays bounded.

**Intervals** are *recentred (bias-corrected) percentile intervals* $[\hat\theta + q_{2.5} - \tilde\theta^*,\
\hat\theta + q_{97.5} - \tilde\theta^*]$ ($\tilde\theta^*$ = replicate median), which always contain the full-data
estimate $\hat\theta$; the bootstrap bias $\tilde\theta^*-\hat\theta$ is reported. With fewer than 50 replicates the
interval is labelled "replicate range" instead of "95% CI" (`ci_method = "recentred-percentile"`).

## 8.5 Dependence curves and thresholds (per group)
x = the group's **primary** feature, y = the group SHAP value. Bins are `CFG.DEP_BINS` equal-count quantile bins
fixed on the full data; a value holding ≥ 1/`DEP_BINS` of the rows (`frac_water = 0`, `frac_cropland = 1`,
CONTAG = 100, …) gets a bin of its own ("atom-aware" binning), otherwise point masses collapse the quantiles exactly
where a rare land cover's effect lives. Estimators (v1, unchanged): **zero_crossing** (single or dominant sign
change; else null with `zero_crossing_flag = "multiple"`; zero = the average cell, not a physical threshold),
**breakpoint** (best continuous hinge, count-weighted, withheld when it removes < 10 % of the straight-line SSE, sits
at the edge of the candidate range or leaves < 5 bins / < 5 % of the weight on a side), **saturation** (first x
after the steepest part where |smoothed slope| stays < 10 % of its maximum for ≥ 3 bins, refined to the knee),
**direction** (sign of Spearman ρ, |ρ| < 0.2 → "mixed"). `<k>_support` = share of replicates that found the
threshold; `<k>_ci` only when the point exists and support ≥ 0.5. A replicate supports the zero crossing only when
its own curve has a crossing by the same rule (single or dominant sign change) within max(one median bin width,
10 % of the primary's p05–p95 range) of the full-data crossing — a crossing somewhere else on a wiggly curve is not
support.

## 8.6 Interactions
SHAP interaction values split $\phi_j$ into a main effect $\phi_{jj}$ and pair terms $\phi_{jk}=\phi_{kj}$ with
$\sum_k\phi_{jk}=\phi_j$. Group interactions sum the member pairs: $\Phi_{gh}=\sum_{j\in g}\sum_{k\in h}\phi_{jk}$
(rows still sum to the group SHAP — checked). Mean $|\Phi_{gh}|$ (18 × 18, diagonal = within-group effect) is
stored globally and per zone from ≤ 5000 sampled rows per frame (GPU; fewer on the CPU); the strongest off-diagonal
partner of a group colours its dependence scatter.

## 8.7 Zone recommendations: model counterfactuals with focal recompute
Recommendations are *plausible interventions evaluated with the final model itself*, not read off the pooled SHAP
curve. **Levers** = the primary features of the actionable groups (`ndvi`, `frac_built`, `frac_forest`,
`frac_water`, `frac_cropland`, `frac_barren`, `lm_ed`); directions: built-up and barren only decrease, tree cover and
water only increase, cropland and edge density either way, NDVI only increases and is not offered in zones whose
median water fraction > 0.15. Every applicable cell of the zone in the current frame (`CFG.NOW`) is shifted by the
same step. A fraction change is **compositional** (the other four fractions and the implicit *other vegetation*
share give/receive land in proportion to their current values, capped by the land available) and the spectral
indices follow through the coupling slopes (`SCENARIO_COUPLING`); spring NDVI follows a season-NDVI shift unless it
has its own slope, and `ndvi_drop = ndvi_spring − ndvi` is recomputed — exactly the dashboard's what-if engine.
**New in v2:** after the base features change, the 3- and 15-cell focal means are **recomputed on the grid**
(distances held fixed, SPEC §4.4) inside the w × w footprint of the edited cells (old values elsewhere), so a
greener zone also changes its neighbours' focal context. **Spill-over cells** are the ROI cells outside the zone
whose feature row changed by more than 10⁻⁶ × max(1, |x|) in any feature (NaN ↔ finite counts); the spill-over
mean is the mean ΔLST over all of them, zeros included (the dashboard simulator reports the same definition).
Feasibility: target within the zone's own p10–p90 and |step| ≤ 0.20 (fractions) / 0.15 (NDVI); the
recommended target is the smallest step achieving ≥ 90 % of the best cooling among the breakpoint, saturation and
25/50/75/100 % of the way to the bound. Expected Δ = mean change of the final model's prediction over the zone's
applicable cells (a ceteris-paribus estimate **associated with** the change, not a causal effect); the interval comes
from replaying the identical counterfactual with every bootstrap replicate. Δ > −0.05 °C is dropped; ≤ 4 per zone.

## 8.8 Web surrogate
The dashboard runs what-ifs in the browser with a compact tree ensemble (v1 format `lst-xgb-compact-v1`, depth ≤ 6,
≤ 400 trees) on all 47 features plus `"focal": FOCAL_SPEC`, so the browser recomputes focal means exactly as above.
It is **distilled** from the final model (label = the final model's margin on every ROI cell × frame, monotone
constraints kept) plus NaN-augmented copies of complete rows (15 % with the six per-cell Landsat features missing —
a spectral gap whose neighbourhood means survive — and 5 % with 10 % random NaNs, labelled by the final model on
the masked rows), so the surrogate follows the final model's missing-value routes; fidelity is reported on complete
rows, actual NaN rows and held-out gap copies. The augmentation is a deliberate trade: on the synthetic 2 km run it
costs a little complete-row fidelity (day complete-row RMSE 0.075 → 0.092 °C against a surrogate without it; night
complete-row R² 0.998) for a usable spectral-gap path (held-out gap RMSE about 0.09 day / 0.045 night °C), because
a browser what-if on a cell with missing Landsat features would otherwise follow routes the final model never uses. Its serialised form is verified with an independent NumPy evaluator
against XGBoost (max |Δ| < 10⁻³ °C, missing values included) and carries 50 check rows for the JavaScript
evaluator.

In [ ]:
_lst_enter_section('08')  # section marker (generated by build_notebook.py)

In [ ]:
import hashlib as _s08_hashlib
import math as _s08_math
import time as _s08_time
import warnings as _s08_warnings
from decimal import ROUND_HALF_UP as _S08_ROUND_HALF_UP
from decimal import Decimal as _S08_Decimal
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy import ndimage as _s08_ndi

_S08_VERSION = "08-v2.1"
_S08_GROUPS = list(SHAP_GROUPS)
_S08_NG = len(_S08_GROUPS)
_S08_K = len(FEATURES)
if _S08_NG != 18 or _S08_K != 47:
    raise AssertionError(f"08 expects 47 features in 18 SHAP groups, got {_S08_K} / {_S08_NG}")
# Membership matrix M (47 x 18): group SHAP = phi @ M, group interactions = M^T Phi M.
_S08_MEMBER = np.zeros((_S08_K, _S08_NG), dtype=np.float64)
for _gi, _g in enumerate(_S08_GROUPS):
    for _f in SHAP_GROUPS[_g]:
        _S08_MEMBER[FEATURES.index(_f), _gi] = 1.0
if not np.array_equal(_S08_MEMBER.sum(axis=1), np.ones(_S08_K)):
    raise AssertionError("SHAP_GROUPS must assign every feature to exactly one group")
_S08_PRIMARY = [GROUP_META[g]["primary"] for g in _S08_GROUPS]
_S08_PRIMARY_IDX = np.array([FEATURES.index(p) for p in _S08_PRIMARY], dtype=np.int64)
_S08_BASE_IDX = np.array([FEATURES.index(f) for f in BASE_FEATURES], dtype=np.int64)

# Design matrix in DF row order. DF is sorted by (year, cell_id) over ALL ROI cells, so frame j is the contiguous
# block [j * N_CELLS, (j + 1) * N_CELLS) in canonical cell order - verified here because everything below relies on it.
_S08_X = np.ascontiguousarray(DF[FEATURES].to_numpy(np.float32))
_S08_YEAR = DF["year"].to_numpy(np.int64)
_S08_FRAMES = sorted(int(y) for y in np.unique(_S08_YEAR))
_S08_NC = int(N_CELLS)
if len(DF) != len(_S08_FRAMES) * _S08_NC:
    raise RuntimeError(f"08: DF has {len(DF)} rows, expected {len(_S08_FRAMES)} frames x {_S08_NC} cells")
_s08_cell_col = DF["cell_id"].to_numpy(np.int64)
for _j, _y in enumerate(_S08_FRAMES):
    _sl = slice(_j * _S08_NC, (_j + 1) * _S08_NC)
    if not (np.all(_S08_YEAR[_sl] == _y) and np.array_equal(_s08_cell_col[_sl], np.asarray(CELL_IDS, np.int64))):
        raise RuntimeError(f"08: DF rows of frame {_y} are not the canonical cell block")
del _s08_cell_col
_S08_FRAME_POS = {y: j for j, y in enumerate(_S08_FRAMES)}
# A SHAP row is reported only when all 33 base features exist (focal means and distances derive from them).
_S08_ROW_VALID = np.isfinite(_S08_X[:, _S08_BASE_IDX]).all(axis=1)
_S08_NOW = int(CFG.NOW) if int(CFG.NOW) in _S08_FRAMES else _S08_FRAMES[-1]
_S08_ZONE_META = [
    {"id": 0, "name": "Ecological Cool Base", "color": "#34d399"},
    {"id": 1, "name": "Riparian Buffer", "color": "#22d3ee"},
    {"id": 2, "name": "Transition", "color": "#a78bfa"},
    {"id": 3, "name": "Heat Extreme Core", "color": "#fb7185"},
]
_S08_N_ZONES = 4
_S08_ZONE_NODATA = 255
if int(CFG.K_ZONES) != _S08_N_ZONES:
    log(f"CFG.K_ZONES={CFG.K_ZONES} but the zone contract has 4 canonical zones; using K=4", "WARNING")
# Monotone constraints of the final model (SPEC §4.3) - bootstrap refits and the surrogate use the same signs.
_S08_MONOTONE = {
    "day": {f: -1 for f in ("ndvi", "ndvi_m3", "ndvi_m15", "frac_forest", "frac_forest_m3", "frac_forest_m15",
                            "frac_water", "frac_water_m3", "frac_water_m15", "mndwi")},
    "night": {f: 1 for f in ("built_v", "frac_built", "frac_built_m3", "frac_built_m15")},
}
_S08_DEVICE = xgb_device_for(0)
_S08_ON_GPU = _S08_DEVICE != "cpu"
_S08_EVAL_ROWS = 60000 if _S08_ON_GPU else 20000        # bootstrap evaluation sample (dependence / importance)
_S08_INTER_ROWS = (int(CFG.SHAP_INTERACTION_ROWS_PER_FRAME) if _S08_ON_GPU
                   else min(int(CFG.SHAP_INTERACTION_ROWS_PER_FRAME), int(CFG.SHAP_INTERACTION_MAX_ROWS or 4000), 1500))
_S08_KMEANS_FIT_ROWS = 200000     # K-means centroids from a seeded sample (all rows are then assigned)
_S08_DIAG_ROWS = 60000            # K = 2..8 diagnostics sample
_S08_SIL_ROWS = 20000             # silhouette sample
_S08_BOOT_MIN_BLOCKS = 50
_S08_MIN_PRED_BATCH = 16
_S08_PRED_KWARGS = {"margin": {"output_margin": True}, "contribs": {"pred_contribs": True},
                    "interactions": {"pred_interactions": True}}
_S08_RES_KM2 = float(GRID.cell_area_km2)
_S08_DISTRICT_NAMES = {int(d["id"]): d["name"] for d in DISTRICTS}
log(f"08: {len(_S08_FRAMES)} frames x {_S08_NC:,} cells, {_S08_NG} SHAP groups; TreeSHAP on {_S08_DEVICE}; "
    f"bootstrap {CFG.N_BOOTSTRAP} replicates, eval sample {_S08_EVAL_ROWS:,}; interactions {_S08_INTER_ROWS:,} "
    f"rows/frame; current frame {_S08_NOW}")


def _s08_frame_slice(year):
    """DF row slice of one frame (canonical cell order)."""
    j = _S08_FRAME_POS[int(year)]
    return slice(j * _S08_NC, (j + 1) * _S08_NC)


def _s08_fit_frames(mode):
    """Training frames of a mode (frames with at least one row of positive weight)."""
    w = DF[f"w_{mode}"].to_numpy(np.float64)
    return sorted(int(y) for y in np.unique(_S08_YEAR[w > 0]))


def _s08_code_hash(*sections):
    """Code hash of other sections whose functions a stage calls (D4); None without the section-00 helper."""
    fn = globals().get("section_code_hash")
    return fn(*sections) if fn is not None else None


def _s08_hash(*parts):
    """Short SHA-1 over arrays / JSON-able parts (stage-cache keys)."""
    h = _s08_hashlib.sha1()
    for p in parts:
        if isinstance(p, np.ndarray):
            h.update(str(p.dtype).encode())
            h.update(np.ascontiguousarray(p).tobytes())
        elif isinstance(p, (bytes, bytearray)):
            h.update(bytes(p))
        else:
            h.update(repr(p).encode())
        h.update(b"|")
    return h.hexdigest()[:20]

## 8.9 XGBoost plumbing: training recipe, batched prediction with OOM fallback
`_s08_train_booster` reproduces the final-model recipe of section 07 (`CFG.xgb_params` + the mode's monotone
constraints) for bootstrap refits. Prediction streams row batches; an out-of-memory error halves the batch and, at
the minimum batch, a *GPU* OOM moves the booster to the CPU (a host OOM propagates).

In [ ]:
def _s08_mono_string(mode):
    return "(" + ",".join(str(int(_S08_MONOTONE.get(mode, {}).get(f, 0))) for f in FEATURES) + ")"


def _s08_train_booster(mode, X, y, w, rounds, device, seed):
    """Train one booster with the final-model recipe; a CUDA OOM retries on the CPU. Returns (booster, device)."""
    for dev in dict.fromkeys([device, "cpu"]):
        params = CFG.xgb_params(dev, seed=int(seed), monotone_constraints=_s08_mono_string(mode))
        try:
            dtrain = xgb.QuantileDMatrix(X, label=y, weight=w, max_bin=int(params.get("max_bin", 256)))
            try:
                booster = xgb.train(params, dtrain, num_boost_round=int(rounds), verbose_eval=False)
            finally:
                del dtrain
            return booster, dev
        except Exception as exc:  # noqa: BLE001
            if dev == "cpu" or not is_gpu_oom_error(exc):
                raise
            log(f"08: GPU OOM training on {dev} ({type(exc).__name__}); retrying on the CPU", "WARNING")
            free_memory()
    raise RuntimeError("unreachable: training loop exhausted")


def _s08_iter_predict(booster, X, *, device, batch, kind, state=None):
    """Yield (start, stop, output) over row batches of X ("margin" | "contribs" | "interactions")."""
    booster.set_param({"device": device})
    batch = max(_S08_MIN_PRED_BATCH, int(batch))
    start, n = 0, len(X)
    while start < n:
        stop = min(n, start + batch)
        try:
            dmat = xgb.DMatrix(X[start:stop], missing=np.nan)
            out = booster.predict(dmat, **_S08_PRED_KWARGS[kind])
            del dmat
        except Exception as exc:  # noqa: BLE001
            gpu_oom = device != "cpu" and is_gpu_oom_error(exc)
            if not (gpu_oom or is_oom_error(exc)):
                raise
            if batch > _S08_MIN_PRED_BATCH:
                batch = max(_S08_MIN_PRED_BATCH, batch // 2)
                log(f"08: OOM in {kind} prediction; batch halved to {batch}", "WARNING")
            elif gpu_oom:
                device = "cpu"
                booster.set_param({"device": "cpu"})
                if state is not None:
                    state["cpu_fallback"] = True
                log(f"08: GPU OOM in {kind} prediction at the minimum batch; falling back to the CPU", "WARNING")
            else:
                raise
            continue
        yield start, stop, out
        start = stop


def _s08_predict_all(booster, X, *, device, kind, batch=None):
    """Concatenated float32 output of `_s08_iter_predict` (empty input -> empty array)."""
    if len(X) == 0:
        width = {"margin": (), "contribs": (_S08_K + 1,), "interactions": (_S08_K + 1, _S08_K + 1)}[kind]
        return np.zeros((0,) + width, dtype=np.float32)
    batch = int(batch or CFG.SHAP_BATCH)
    parts = [out.astype(np.float32, copy=False)
             for _, _, out in _s08_iter_predict(booster, X, device=device, batch=batch, kind=kind)]
    return np.concatenate(parts, axis=0)


def _s08_margin(booster, X, device):
    return _s08_predict_all(booster, X, device=device, kind="margin").astype(np.float64)

## 8.10 Dependence curves and threshold estimators (ported unchanged from v1)
Pure functions shared by the final model and every bootstrap replicate.

In [ ]:
def _s08_bin_index(x, edges):
    """Bin id of each value (0..nb-1, the last bin is closed), -1 for NaN."""
    x = np.asarray(x, dtype=np.float64)
    nb = len(edges) - 1
    idx = np.clip(np.searchsorted(edges, x, side="right") - 1, 0, max(nb - 1, 0))
    idx[~np.isfinite(x)] = -1
    return idx


def _s08_quantile_edges(x, n_bins):
    """Atom-aware quantile bin edges fixed on the full data (≥ 2 edges, no empty bin).

    Every value holding ≥ 1/n_bins of the data gets a bin of its own ([a, nextafter(a)) isolates atom a); the other
    bins are equal-count quantiles of the remaining values. Empty bins are merged into a neighbour and, when atoms
    split quantile bins, the smallest non-atom bins are merged until at most ``n_bins`` remain.
    """
    xf = np.asarray(x, dtype=np.float64)
    xf = xf[np.isfinite(xf)]
    if xf.size == 0:
        return np.array([0.0, 0.0])
    n_bins = max(1, int(n_bins))
    values, counts = np.unique(xf, return_counts=True)
    atoms = values[counts * n_bins >= xf.size]
    rest = xf[~np.isin(xf, atoms)] if atoms.size else xf
    parts = [values[[0, -1]], atoms, np.nextafter(atoms, np.inf)]
    if rest.size:
        n_rest = max(1, n_bins - atoms.size)
        parts.append(np.quantile(rest, np.linspace(0.0, 1.0, n_rest + 1)))
    edges = np.unique(np.concatenate(parts))
    if edges.size < 2:
        return np.array([edges[0], edges[0]])
    while edges.size > 2:
        occupied = np.bincount(_s08_bin_index(xf, edges), minlength=edges.size - 1)
        empty = np.flatnonzero(occupied == 0)
        if empty.size == 0:
            break
        i = int(empty[0])
        edges = np.delete(edges, i if i > 0 else 1)
    atom_set = set(atoms.tolist())
    while edges.size - 1 > n_bins:
        occupied = np.bincount(_s08_bin_index(xf, edges), minlength=edges.size - 1)
        is_atom = np.array([float(edges[b]) in atom_set and edges[b + 1] == np.nextafter(edges[b], np.inf)
                            for b in range(edges.size - 1)])
        merged = False
        for b in np.argsort(occupied, kind="stable"):
            if is_atom[b]:
                continue
            nbrs = [n for n in (b - 1, b + 1) if 0 <= n < edges.size - 1 and not is_atom[n]]
            if not nbrs:
                continue
            n = min(nbrs, key=lambda k: occupied[k])
            edges = np.delete(edges, max(b, n))
            merged = True
            break
        if not merged:
            break
    return edges


def _s08_binned_curve(x, s, edges):
    """(centers, mean, counts): mean SHAP per bin (NaN if empty); centers = mean x in the bin (midpoint if empty)."""
    x = np.asarray(x, dtype=np.float64)
    s = np.asarray(s, dtype=np.float64)
    nb = len(edges) - 1
    idx = _s08_bin_index(x, edges)
    ok = (idx >= 0) & np.isfinite(s)
    counts = np.bincount(idx[ok], minlength=nb)
    sum_s = np.bincount(idx[ok], weights=s[ok], minlength=nb)
    sum_x = np.bincount(idx[ok], weights=x[ok], minlength=nb)
    with np.errstate(invalid="ignore", divide="ignore"):
        mean = sum_s / counts
        centers = sum_x / counts
    mids = 0.5 * (np.asarray(edges[:-1], dtype=np.float64) + np.asarray(edges[1:], dtype=np.float64))
    centers = np.where(counts > 0, centers, mids)
    return centers, np.where(counts > 0, mean, np.nan), counts.astype(np.int64)


def _s08_finite_curve(centers, mean, counts=None):
    c = np.asarray(centers, dtype=np.float64)
    m = np.asarray(mean, dtype=np.float64)
    ok = np.isfinite(c) & np.isfinite(m)
    if counts is not None:
        w = np.asarray(counts, dtype=np.float64)
        ok &= w > 0
        return c[ok], m[ok], w[ok]
    return c[ok], m[ok], np.ones(int(ok.sum()))


def _s08_crossings(centers, mean):
    """All sign changes of the curve: (x by linear interpolation, |jump| across each)."""
    c, m, _ = _s08_finite_curve(centers, mean)
    if c.size < 2:
        return np.zeros(0), np.zeros(0)
    m0, m1 = m[:-1], m[1:]
    cross = np.flatnonzero(((m0 < 0) & (m1 >= 0)) | ((m0 > 0) & (m1 <= 0)))
    if cross.size == 0:
        return np.zeros(0), np.zeros(0)
    x = c[cross] - m[cross] * (c[cross + 1] - c[cross]) / (m[cross + 1] - m[cross])
    return x.astype(np.float64), np.abs(m1[cross] - m0[cross]).astype(np.float64)


def _s08_zero_crossing(centers, mean, ref=None, dominance=2.0):
    """Single / dominant sign change (None otherwise); with ``ref`` the crossing nearest to the full-data one."""
    x, jump = _s08_crossings(centers, mean)
    if x.size == 0:
        return None
    if ref is not None and np.isfinite(ref):
        return float(x[int(np.argmin(np.abs(x - float(ref))))])
    if x.size == 1:
        return float(x[0])
    order = np.argsort(jump)[::-1]
    if jump[order[0]] >= dominance * jump[order[1]]:
        return float(x[order[0]])
    return None


def _s08_zero_crossing_is_multiple(centers, mean, dominance=2.0):
    x, jump = _s08_crossings(centers, mean)
    if x.size < 2:
        return False
    top2 = np.sort(jump)[::-1][:2]
    return bool(top2[0] < dominance * top2[1])


def _s08_hinge_fit(c, m, w, candidates):
    """Weighted continuous hinge y = b0 + b1 x + b2 max(0, x - h) for every candidate h (batched 3x3 solves)."""
    h = np.asarray(candidates, dtype=np.float64)[:, None]
    A = np.stack([np.ones((h.shape[0], c.size)), np.broadcast_to(c, (h.shape[0], c.size)),
                  np.maximum(0.0, c[None, :] - h)], axis=2)
    Aw = A * w[None, :, None]
    ata = np.einsum("cni,cnj->cij", Aw, A) + 1e-12 * np.eye(3)[None]
    aty = np.einsum("cni,n->ci", Aw, m)
    try:
        beta = np.linalg.solve(ata, aty[..., None])[..., 0]
    except np.linalg.LinAlgError:
        # Degenerate curves (few distinct bin centres, e.g. a near-constant primary feature, or a candidate whose
        # hinge column is collinear with [1, x]) make some 3x3 systems exactly singular; the minimum-norm
        # least-squares solution is then the right fit (integrator finding on the gee-mock path).
        beta = np.einsum("cij,cj->ci", np.linalg.pinv(ata), aty)
    resid = m[None, :] - np.einsum("cni,ci->cn", A, beta)
    return (w[None, :] * resid ** 2).sum(axis=1), beta


_S08_HINGE_MIN_SIDE_BINS = 5
_S08_HINGE_MIN_SIDE_WEIGHT = 0.05


def _s08_hinge_breakpoint(centers, mean, counts):
    """(breakpoint, slope_before, slope_after) of the best count-weighted hinge, or (None, None, None)."""
    c, m, w = _s08_finite_curve(centers, mean, counts)
    n = c.size
    if n < 5:
        return None, None, None
    lo, hi = max(1, int(np.floor(0.1 * (n - 1)))), min(n - 2, int(np.ceil(0.9 * (n - 1))))
    if hi <= lo:
        return None, None, None
    candidates = np.unique(np.concatenate([c[lo:hi + 1], np.linspace(c[lo], c[hi], 121)]))
    sse, beta = _s08_hinge_fit(c, m, w, candidates)
    lin = np.polyfit(c, m, 1, w=np.sqrt(w))
    sse_lin = float(np.sum(w * (m - np.polyval(lin, c)) ** 2))
    best = int(np.argmin(sse))
    if sse_lin <= 1e-12 or sse[best] > 0.9 * sse_lin:
        return None, None, None
    h = float(candidates[best])
    eps = 1e-9 * max(1.0, float(np.max(np.abs(c))))
    if h <= c[lo] + eps or h >= c[hi] - eps:
        return None, None, None
    left, right = c <= h, c > h
    total_w = float(w.sum())
    if (left.sum() < _S08_HINGE_MIN_SIDE_BINS or right.sum() < _S08_HINGE_MIN_SIDE_BINS
            or w[left].sum() < _S08_HINGE_MIN_SIDE_WEIGHT * total_w
            or w[right].sum() < _S08_HINGE_MIN_SIDE_WEIGHT * total_w):
        return None, None, None
    b1, b2 = float(beta[best, 1]), float(beta[best, 2])
    return h, b1, b1 + b2


def _s08_smooth3(m):
    num = np.convolve(m, np.ones(3), mode="same")
    den = np.convolve(np.ones_like(m), np.ones(3), mode="same")
    return num / den


_S08_SAT_MAX_TAIL_SHARE = 0.25


def _s08_saturation_point(centers, mean):
    """First x after the steepest section where |smoothed slope| < 10 % of its max for ≥ 3 bins (knee-refined).

    A saturation must also be *final*: the smoothed curve may change by at most ``_S08_SAT_MAX_TAIL_SHARE`` of its
    total range between the candidate and the last bin. Without this guard a zero-inflated predictor (e.g.
    ``frac_water``: most cells exactly 0, so the first bin-to-bin step is by far the steepest) reported a
    "saturation" on the first short noisy plateau although the curve kept falling for another 80 % of its range.
    """
    c, m, _ = _s08_finite_curve(centers, mean)
    if c.size < 5 or np.any(np.diff(c) <= 0):
        return None
    ms = _s08_smooth3(m)
    slope = np.abs(np.gradient(ms, c))
    smax = float(np.max(slope))
    if not np.isfinite(smax) or smax <= 0:
        return None
    steep = int(np.argmax(slope))
    flat = slope < 0.1 * smax
    span = float(np.ptp(ms))
    onset = next((j for j in range(steep + 1, c.size - 2)
                  if flat[j:j + 3].all() and abs(float(ms[-1] - ms[j])) <= _S08_SAT_MAX_TAIL_SHARE * span), None)
    if onset is None:
        return None
    lo, hi = max(steep, onset - 4), min(c.size - 1, onset + 2)
    seg_c, seg_m = c[lo:hi + 1], m[lo:hi + 1]
    if seg_c.size < 4:
        return float(c[onset])
    candidates = np.linspace(c[lo], c[onset], 81)
    sse, _ = _s08_hinge_fit(seg_c, seg_m, np.ones(seg_c.size), candidates)
    return float(candidates[int(np.argmin(sse))])


def _s08_effect_direction(x, s):
    """'warming' / 'cooling' from the sign of Spearman rho(x, SHAP); |rho| < 0.2 -> 'mixed'."""
    x = np.asarray(x, dtype=np.float64)
    s = np.asarray(s, dtype=np.float64)
    ok = np.isfinite(x) & np.isfinite(s)
    if ok.sum() < 10 or np.ptp(x[ok]) == 0 or np.ptp(s[ok]) == 0:
        return "mixed"
    try:
        from scipy.stats import spearmanr

        rho = float(spearmanr(x[ok], s[ok]).correlation)
    except ImportError:
        rx = pd.Series(x[ok]).rank().to_numpy()
        rs = pd.Series(s[ok]).rank().to_numpy()
        rho = float(np.corrcoef(rx, rs)[0, 1])
    if not np.isfinite(rho) or abs(rho) < 0.2:
        return "mixed"
    return "warming" if rho > 0 else "cooling"


def _s08_curve_thresholds(centers, mean, counts, zc_ref=None, zc_tol=None):
    """(zero_crossing, breakpoint, saturation) as floats, NaN for 'not found'.

    The zero crossing uses the full-data rule (a single or dominant sign change). Replicates pass the full-data
    crossing ``zc_ref`` and a tolerance ``zc_tol``: a replicate's own crossing counts only when it exists and lies
    within ``zc_tol`` of ``zc_ref`` (a crossing elsewhere on a wiggly curve is not support for the reported one).
    """
    zc = _s08_zero_crossing(centers, mean)
    if zc is not None and zc_ref is not None:
        if not np.isfinite(zc_ref) or zc_tol is None or not abs(zc - float(zc_ref)) <= float(zc_tol):
            zc = None
    bp, _, _ = _s08_hinge_breakpoint(centers, mean, counts)
    sat = _s08_saturation_point(centers, mean)
    return [np.nan if v is None else v for v in (zc, bp, sat)]


def _s08_zc_tolerance(x, edges):
    """Replicate zero-crossing tolerance: max(one median bin width, 10 % of the primary's p05-p95 range)."""
    x = np.asarray(x, np.float64)
    x = x[np.isfinite(x)]
    width = np.diff(np.asarray(edges, np.float64))
    scale = max(1.0, float(np.max(np.abs(edges)))) if len(edges) else 1.0
    width = width[width > 1e-9 * scale]                 # atom bins [a, nextafter(a)) have no width
    med = float(np.median(width)) if width.size else 0.0
    spread = float(np.subtract(*np.percentile(x, [95, 5]))) if x.size else 0.0
    return max(med, 0.1 * spread)


def _s08_recentred_interval(theta_hat, boot, lower=None):
    """Recentred percentile interval elementwise over the leading replicate axis -> (lo, hi, bias)."""
    boot = np.asarray(boot, dtype=np.float64)
    theta_hat = np.asarray(theta_hat, dtype=np.float64)
    if boot.ndim == 0 or boot.shape[0] == 0:
        nan = np.full(theta_hat.shape, np.nan)
        return nan, nan.copy(), nan.copy()
    with _s08_warnings.catch_warnings():
        _s08_warnings.simplefilter("ignore", RuntimeWarning)
        n_ok = np.isfinite(boot).sum(axis=0)
        q_lo, q_med, q_hi = np.nanpercentile(boot, [2.5, 50.0, 97.5], axis=0)
    lo, hi = theta_hat + (q_lo - q_med), theta_hat + (q_hi - q_med)
    bad = n_ok < 2
    lo, hi = np.where(bad, np.nan, lo), np.where(bad, np.nan, hi)
    if lower is not None:
        lo = np.maximum(lo, lower)
    return lo, hi, np.where(bad, np.nan, q_med - theta_hat)


def _s08_boot_ci(values, point):
    """(interval, support) of one threshold; the interval only when the point exists and support ≥ 0.5."""
    v = np.asarray(values, dtype=np.float64)
    support = float(np.isfinite(v).mean()) if v.size else 0.0
    if point is None or support < 0.5 or np.isfinite(v).sum() < 2:
        return None, (None if point is None else support)
    lo, hi, _ = _s08_recentred_interval(np.array([float(point)]), v[:, None])
    return [float(lo[0]), float(hi[0])], support


def _s08_curve_slope(centers, mean, counts):
    """Count-weighted least-squares slope of the binned curve (°C per unit of x) or None."""
    c, m, w = _s08_finite_curve(centers, mean, counts)
    if c.size < 3 or np.ptp(c) <= 0:
        return None
    return float(np.polyfit(c, m, 1, w=np.sqrt(w))[0])

## 8.11 Scenario coupling and the counterfactual engine (focal recompute, distances fixed)
**Coupling** = partial OLS slopes (d index / d fraction, with intercept, all cells × frames) of `ndvi`, `ndvi_spring`,
`ndbi`, `mndwi` and `albedo` on the five fractions; *other vegetation* is the omitted reference class, so no slope is
needed for it. The engine below is the Python twin of the dashboard's `perturbBaseV2` + focal recompute.

In [ ]:
_S08_FRACTIONS = ["frac_built", "frac_forest", "frac_water", "frac_cropland", "frac_barren"]
_S08_COUPLED = ["ndvi", "ndvi_spring", "ndbi", "mndwi", "albedo"]
_S08_FRAC_IDX = [FEATURES.index(f) for f in _S08_FRACTIONS]
_S08_BOUNDS = {"ndvi": (-1.0, 1.0), "ndvi_spring": (-1.0, 1.0), "ndbi": (-1.0, 1.0), "mndwi": (-1.0, 1.0),
               "albedo": (0.0, 1.0), "lm_contag": (0.0, 100.0), "lm_pd": (0.0, np.inf), "lm_ed": (0.0, np.inf),
               **{f: (0.0, 1.0) for f in _S08_FRACTIONS}}


def _s08_scenario_coupling():
    """SCENARIO_COUPLING (mode-independent; v1 format + focal spec, grid shape and the derived-feature rules)."""
    A_all = _S08_X[:, _S08_FRAC_IDX].astype(np.float64)
    coupling = {f: {} for f in _S08_FRACTIONS}
    r2, intercept, n_used = {}, {}, {}
    for index in _S08_COUPLED:
        y = _S08_X[:, FEATURES.index(index)].astype(np.float64)
        ok = np.isfinite(y) & np.isfinite(A_all).all(axis=1)
        if ok.sum() < 10:
            log(f"08: coupling of {index} has only {int(ok.sum())} complete rows; slopes set to 0", "WARNING")
            for f in _S08_FRACTIONS:
                coupling[f][index] = 0.0
            r2[index], intercept[index], n_used[index] = None, None, int(ok.sum())
            continue
        design = np.column_stack([np.ones(int(ok.sum())), A_all[ok]])
        beta, *_ = np.linalg.lstsq(design, y[ok], rcond=None)
        resid = y[ok] - design @ beta
        sst = float(np.sum((y[ok] - y[ok].mean()) ** 2))
        r2[index] = 1.0 - float(np.sum(resid ** 2)) / sst if sst > 0 else None
        intercept[index] = float(beta[0])
        n_used[index] = int(ok.sum())
        for k, f in enumerate(_S08_FRACTIONS):
            coupling[f][index] = float(beta[k + 1])
    return {
        "description": ("Partial OLS slopes (d index / d fraction) of NDVI, spring NDVI, NDBI, MNDWI and albedo on "
                        "the five land-cover fractions (with intercept; other vegetation is the omitted reference "
                        "class), fitted on all cells and frames. A fraction change exchanges land with the other "
                        "fractions and the implicit other-vegetation share in proportion to their current values; "
                        "each index shifts by the sum of slope x actual fraction change (clamped to its range); "
                        "ndvi_drop = ndvi_spring - ndvi is recomputed; the 3- and 15-cell focal means are recomputed "
                        "on the grid and distances are held fixed."),
        "fraction_features": list(_S08_FRACTIONS),
        "coupled_indices": list(_S08_COUPLED),
        "coupling": coupling, "r2": r2, "intercept": intercept, "n": n_used,
        "derived": {"ndvi_drop": "ndvi_spring - ndvi (recomputed after any change)",
                    "ndvi_spring": "follows a direct NDVI shift by the same amount; coupled explicitly for "
                                   "fraction changes"},
        "bounds": {k: [float(v[0]), (None if not np.isfinite(v[1]) else float(v[1]))] for k, v in _S08_BOUNDS.items()},
        "distances": "held fixed (dist_water, dist_urban, dist_builtup)",
        "focal": {"bases": list(FOCAL_SPEC["bases"]), "windows": [int(w) for w in FOCAL_SPEC["windows"]]},
        "grid": {"width": int(GRID.width), "height": int(GRID.height)},
    }


def _s08_rebalance(F, k, delta):
    """Compositional change of fraction column ``k`` by ``delta`` for every row of F (n, 5).

    The implicit other-vegetation share ``1 - sum(F)`` is a sixth donor/recipient: an increase is capped by the land
    the others can give up and taken from them in proportion to their current values; a decrease (capped at the
    current value) gives the freed land to them in the same proportions (equally when they are all 0).
    Returns (F_after, other_after, applied).
    """
    F = np.asarray(F, dtype=np.float64)
    x = F[:, k]
    other = np.clip(1.0 - F.sum(axis=1), 0.0, None)
    donors = np.column_stack([np.delete(F, k, axis=1), other])
    dsum = donors.sum(axis=1)
    if delta > 0:
        applied = np.clip(np.minimum(np.minimum(delta, 1.0 - x), dsum), 0.0, None)
    else:
        applied = np.maximum(delta, -x)
    with np.errstate(invalid="ignore", divide="ignore"):
        share = np.where(dsum[:, None] > 0, donors / dsum[:, None], 1.0 / donors.shape[1])
    donors_after = np.clip(donors - applied[:, None] * share, 0.0, 1.0)
    F_after = F.copy()
    F_after[:, k] = np.clip(x + applied, 0.0, 1.0)
    others = [i for i in range(F.shape[1]) if i != k]
    F_after[:, others] = donors_after[:, :-1]
    return F_after, donors_after[:, -1], applied


def _s08_perturb(vals, feature, step, coupled=True):
    """Shift ``feature`` by ``step`` in the base-feature vectors ``vals`` (dict name -> (n,) float64).

    Returns (changed: dict name -> new (n,) values, applicable bool (n,), applied change (n,), donors dict of mean
    fraction changes). Fractions: compositional + coupled indices; other levers: direct shift clamped to their range.
    Spring NDVI follows a season-NDVI shift unless it was coupled explicitly; ndvi_drop is recomputed.
    """
    n = len(vals[feature])
    changed, donors = {}, {}
    applied = np.zeros(n)
    ndvi_shift = np.zeros(n)
    if feature in _S08_FRACTIONS:
        F = np.column_stack([vals[f] for f in _S08_FRACTIONS])
        ok = np.isfinite(F).all(axis=1)
        F_ok = F[ok]
        other_before = np.clip(1.0 - F_ok.sum(axis=1), 0.0, None)
        F_after, other_after, app = _s08_rebalance(F_ok, _S08_FRACTIONS.index(feature), float(step))
        applied[ok] = app
        d_frac = F_after - F_ok
        for i, f in enumerate(_S08_FRACTIONS):
            col = vals[f].copy()
            col[ok] = F_after[:, i]
            changed[f] = col
            if f != feature:
                donors[f] = float(d_frac[:, i].mean()) if d_frac.size else 0.0
        donors["other_vegetation"] = float((other_after - other_before).mean()) if d_frac.size else 0.0
        if coupled:
            for index in _S08_COUPLED:
                slopes = np.array([SCENARIO_COUPLING["coupling"][f].get(index, 0.0) for f in _S08_FRACTIONS])
                shift = d_frac @ slopes
                lo, hi = _S08_BOUNDS.get(index, (-np.inf, np.inf))
                col = vals[index].copy()
                old = col[ok]
                new = np.where(np.isfinite(old), np.clip(old + shift, lo, hi), old)
                col[ok] = new
                changed[index] = col
                if index == "ndvi":
                    ndvi_shift[ok] = np.where(np.isfinite(old), new - old, 0.0)
    else:
        ok = np.isfinite(vals[feature])
        lo, hi = _S08_BOUNDS.get(feature, (-np.inf, np.inf))
        col = vals[feature].copy()
        col[ok] = np.clip(col[ok] + float(step), lo, hi)
        applied[ok] = col[ok] - vals[feature][ok]
        changed[feature] = col
        if feature == "ndvi":
            ndvi_shift[ok] = applied[ok]
    if "ndvi_spring" not in changed and np.any(ndvi_shift != 0):
        lo, hi = _S08_BOUNDS["ndvi_spring"]
        spring = vals["ndvi_spring"].copy()
        move = np.isfinite(spring) & (ndvi_shift != 0)
        spring[move] = np.clip(spring[move] + ndvi_shift[move], lo, hi)
        changed["ndvi_spring"] = spring
    if "ndvi" in changed or "ndvi_spring" in changed:
        spring = changed.get("ndvi_spring", vals["ndvi_spring"])
        ndvi = changed.get("ndvi", vals["ndvi"])
        drop = vals["ndvi_drop"].copy()
        both = np.isfinite(spring) & np.isfinite(ndvi)
        drop[both] = spring[both] - ndvi[both]
        changed["ndvi_drop"] = drop
    return changed, ok, applied, donors


_S08_PERTURB_INPUTS = sorted(set(_S08_FRACTIONS) | set(_S08_COUPLED) | {"ndvi_drop", "lm_ed", "lm_pd", "lm_contag"})


def _s08_counterfactual(year, cells, feature, step, coupled=True):
    """Counterfactual feature rows of frame ``year`` after shifting ``feature`` by ``step`` in the cells ``cells``.

    The base features are perturbed (`_s08_perturb`), written back into copies of the frame's feature grids, and the
    focal means of every changed focal base are recomputed with ``FOCAL_RECOMPUTE`` (distances held fixed) and kept
    only inside the w x w dilation footprint of the edited cells, so neighbouring cells change too and nothing else
    does. Returns {"pos": canonical positions of the cells evaluated by the model (material changes plus every
    changed cell of the edited region), "X": their counterfactual rows (float32), "zone_ok": applicable zone cells,
    "zone_cells", "applied", "donors", "n_spill" and "spill_pos" (indices into ``pos`` of the spill-over cells)}.
    """
    grids = FEATURE_GRIDS[int(year)]
    cells = np.asarray(cells, dtype=np.int64)
    r, c = CELL_ROWS[cells], CELL_COLS[cells]
    needed = set(_S08_PERTURB_INPUTS) | {feature}
    vals = {f: np.asarray(grids[f], np.float64)[r, c] for f in needed}
    changed, ok, applied, donors = _s08_perturb(vals, feature, step, coupled=coupled)
    new_grids = {}
    for f, v in changed.items():
        if np.array_equal(v, vals[f], equal_nan=True):
            continue
        g = np.array(grids[f], dtype=np.float32, copy=True)
        g[r, c] = v.astype(np.float32)
        new_grids[f] = g
    changed_bases = [b for b in FOCAL_SPEC["bases"] if b in new_grids]
    if changed_bases:
        focal_in = {b: new_grids.get(b, grids[b]) for b in FOCAL_SPEC["bases"]}
        focal = FOCAL_RECOMPUTE(focal_in, recompute_distances=False)
        for b in changed_bases:
            # Only cells whose w x w window contains an edited cell can change; everywhere else the old focal
            # value is kept, so a full-grid recompute cannot leak float residues into untouched cells (D17).
            old_b = np.asarray(grids[b], np.float32)
            new_b = np.asarray(new_grids[b], np.float32)
            edited = ~((old_b == new_b) | (np.isnan(old_b) & np.isnan(new_b)))
            for w in FOCAL_SPEC["windows"]:
                name = f"{b}_m{w}"
                footprint = _s08_ndi.binary_dilation(edited, structure=np.ones((int(w), int(w)), bool))
                new_grids[name] = np.where(footprint, focal[name], np.asarray(grids[name], np.float32)).astype(
                    np.float32)
    X_frame = _S08_X[_s08_frame_slice(year)]
    in_zone = np.zeros(_S08_NC, dtype=bool)
    in_zone[cells] = True
    if not new_grids:
        return {"pos": np.zeros(0, np.int64), "X": np.zeros((0, _S08_K), np.float32), "zone_ok": cells[ok],
                "zone_cells": cells, "applied": applied[ok], "donors": donors, "n_spill": 0,
                "spill_pos": np.zeros(0, np.int64)}
    cols = [FEATURES.index(f) for f in new_grids]
    new_cols = np.column_stack([np.asarray(new_grids[f], np.float32)[CELL_ROWS, CELL_COLS] for f in new_grids])
    old_cols = X_frame[:, cols]
    exact_same = (new_cols == old_cols) | (np.isnan(new_cols) & np.isnan(old_cols))
    # a material change: > 1e-6 x max(1, |x_old|) in any feature, or NaN <-> finite (D17 spill-over definition)
    with np.errstate(invalid="ignore"):
        tol = 1e-6 * np.maximum(1.0, np.abs(old_cols.astype(np.float64)))
        material = (np.abs(new_cols.astype(np.float64) - old_cols.astype(np.float64)) > tol) | (
            np.isnan(new_cols) != np.isnan(old_cols))
    row_material = material.any(axis=1)
    # rows evaluated by the model: every material change plus any exact change inside the edited region
    pos = np.flatnonzero(row_material | (in_zone & ~exact_same.all(axis=1)))
    Xcf = X_frame[pos].copy()
    Xcf[:, cols] = new_cols[pos]
    spill_pos = np.flatnonzero(~in_zone[pos])            # canonical cells are ROI cells by construction
    return {"pos": pos, "X": np.ascontiguousarray(Xcf, dtype=np.float32), "zone_ok": cells[ok],
            "zone_cells": cells, "applied": applied[ok], "donors": donors, "n_spill": int(spill_pos.size),
            "spill_pos": spill_pos}


def _s08_cf_effect(booster, device, year, cf, base_margin=None):
    """(zone mean Δ, spill-over mean Δ, n spill-over cells) of one counterfactual under ``booster``.

    Δ is the change of the prediction (°C) per cell; cells whose feature row did not change have Δ = 0. The zone
    effect averages over the applicable zone cells. Spill-over cells (D17) are the ROI cells outside the edited
    region whose feature row changed materially (> 1e-6 relative in any feature, or NaN <-> finite); the spill-over
    mean is the mean Δ over all of them, zeros included.
    """
    if cf["zone_ok"].size == 0:
        return np.nan, np.nan, 0
    d = np.zeros(_S08_NC)
    if cf["pos"].size:
        pred_cf = _s08_margin(booster, cf["X"], device)
        if base_margin is None:
            pred_base = _s08_margin(booster, _S08_X[_s08_frame_slice(year)][cf["pos"]], device)
        else:
            pred_base = base_margin[cf["pos"]]
        d[cf["pos"]] = pred_cf - pred_base
    spill = cf["pos"][cf["spill_pos"]] if cf["pos"].size else np.zeros(0, np.int64)
    zone_delta = float(np.mean(d[cf["zone_ok"]]))
    spill_delta = float(np.mean(d[spill])) if spill.size else 0.0
    return zone_delta, spill_delta, int(spill.size)


SCENARIO_COUPLING = _s08_scenario_coupling()
print("SCENARIO_COUPLING slopes (d index / d fraction):")
print(pd.DataFrame(SCENARIO_COUPLING["coupling"]).T.round(3).to_string())
print("R2:", {k: (round(v, 3) if v is not None else None) for k, v in SCENARIO_COUPLING["r2"].items()})

## 8.12 Zoning helpers: K-means, the v2 labelling rule and `ASSIGN_ZONES`

In [ ]:
def _s08_to_numpy(arr):
    if hasattr(arr, "to_numpy"):
        arr = arr.to_numpy()
    if hasattr(arr, "get"):
        arr = arr.get()
    return np.asarray(arr)


def _s08_kmeans(X, k, n_init, seed):
    """(labels int32, inertia, centers (k, d)) - cuML on the GPU with USE_RAPIDS, else scikit-learn."""
    if ENV.get("has_cuml") and CFG.USE_RAPIDS:
        try:
            from cuml.cluster import KMeans as _CuKMeans

            model = _CuKMeans(n_clusters=k, n_init=n_init, random_state=seed, max_iter=300)
            model.fit(X)
            ENV.setdefault("rapids_used", set()).add("cuml_kmeans")
            return (_s08_to_numpy(model.labels_).astype(np.int32), float(model.inertia_),
                    _s08_to_numpy(model.cluster_centers_).astype(np.float64))
        except Exception as exc:  # noqa: BLE001
            log(f"08: cuML KMeans failed ({type(exc).__name__}: {exc}); using scikit-learn", "WARNING")
    from sklearn.cluster import KMeans

    model = KMeans(n_clusters=k, n_init=n_init, random_state=seed).fit(X)
    return model.labels_.astype(np.int32), float(model.inertia_), np.asarray(model.cluster_centers_, np.float64)


def _s08_nearest_centroid(M, centroids, chunk=262144):
    """Index of the nearest centroid (squared Euclidean) per row; -1 for rows with NaN."""
    M = np.asarray(M, dtype=np.float64)
    C = np.asarray(centroids, dtype=np.float64)
    out = np.full(len(M), -1, dtype=np.int64)
    ok = np.isfinite(M).all(axis=1)
    idx = np.flatnonzero(ok)
    c2 = (C * C).sum(axis=1)
    for s in range(0, idx.size, chunk):
        part = M[idx[s:s + chunk]]
        d2 = (part * part).sum(axis=1)[:, None] - 2.0 * part @ C.T + c2[None, :]
        out[idx[s:s + chunk]] = np.argmin(d2, axis=1)
    return out


def ASSIGN_ZONES(m, shap_group_matrix):
    """Zone ids (uint8) by nearest centroid of ``ZONE_MODEL[m]`` in the 18-group SHAP space (255 for NaN rows).

    ``shap_group_matrix``: (n, 18) group SHAP values in ``SHAP_GROUPS`` order (°C). Used for observed frames here and
    for projected frames in section 11, so every frame is zoned in the same space.
    """
    if m not in ZONE_MODEL:
        raise KeyError(f"no zone model for mode {m!r} (available: {sorted(ZONE_MODEL)})")
    M = np.asarray(shap_group_matrix, dtype=np.float64)
    if M.ndim != 2 or M.shape[1] != _S08_NG:
        raise ValueError(f"expected an (n, {_S08_NG}) group-SHAP matrix, got {M.shape}")
    nearest = _s08_nearest_centroid(M, ZONE_MODEL[m]["centroids"])
    out = np.full(len(M), _S08_ZONE_NODATA, dtype=np.uint8)
    out[nearest >= 0] = nearest[nearest >= 0].astype(np.uint8)
    return out


# Riparian eligibility (D11, SPEC §1.5): a non-core cluster shows a riparian signature if ANY of these holds.
_S08_RIPARIAN_MEAN_WATER = 0.05          # mean frac_water >= 5 %
_S08_RIPARIAN_WET_FRAC = 0.2             # ... or >= 25 % of its rows have frac_water >= 0.2
_S08_RIPARIAN_WET_SHARE = 0.25
_S08_RIPARIAN_WATER_SHAP_C = 0.25        # ... or 'water' is its top mean-|SHAP| group with >= 0.25 °C
_S08_RIPARIAN_CRITERION = (f"mean frac_water >= {_S08_RIPARIAN_MEAN_WATER:g}, or >= {_S08_RIPARIAN_WET_SHARE:.0%} "
                           f"of the cluster's rows with frac_water >= {_S08_RIPARIAN_WET_FRAC:g}, or the 'water' group "
                           f"has the largest mean |SHAP| of the 18 groups with >= {_S08_RIPARIAN_WATER_SHAP_C:g} °C")
_S08_NO_RIPARIAN_NAME = "Moderate mosaic (no riparian signal)"


def _s08_riparian_reasons(stats):
    """List of the riparian-eligibility conditions a cluster meets (empty = not eligible)."""
    out = []
    fw = stats["mean_frac_water"]
    if np.isfinite(fw) and fw >= _S08_RIPARIAN_MEAN_WATER:
        out.append(f"mean frac_water {fw:.3f} >= {_S08_RIPARIAN_MEAN_WATER:g}")
    if stats["share_water_ge_0_2"] >= _S08_RIPARIAN_WET_SHARE:
        out.append(f"{stats['share_water_ge_0_2']:.0%} of rows with frac_water >= {_S08_RIPARIAN_WET_FRAC:g}")
    if stats["water_group_rank"] == 1 and stats["water_mean_abs_shap"] >= _S08_RIPARIAN_WATER_SHAP_C:
        out.append(f"water is the top mean-|SHAP| group ({stats['water_mean_abs_shap']:.2f} °C)")
    return out


def _s08_label_clusters(raw, group_shap, frac_water):
    """Canonical zone ids from raw cluster labels with the v2 rule (SPEC §1.5, D11). Returns (lut, record).

    3 Heat Extreme Core = highest mean total SHAP. Among the other three a cluster is riparian-eligible by
    ``_S08_RIPARIAN_CRITERION``. If any is eligible: 1 = the eligible cluster with the highest mean frac_water, 0 =
    the lower mean total SHAP of the remaining two, 2 = the last one. If none is: the three are ordered by mean total
    SHAP (lowest -> 0, middle -> 1, highest -> 2) and zone 1 is displayed as a moderate mosaic.
    """
    total = group_shap.sum(axis=1, dtype=np.float64)
    gi_water = _S08_GROUPS.index("water")
    water_g = group_shap[:, gi_water].astype(np.float64)
    cstats = []
    for c in range(_S08_N_ZONES):
        sel = raw == c
        if not sel.any():
            raise RuntimeError(f"08: K-means cluster {c} is empty after assignment")
        fw_c = frac_water[sel]
        fin = np.isfinite(fw_c)
        with _s08_warnings.catch_warnings():
            _s08_warnings.simplefilter("ignore", RuntimeWarning)
            fw = float(np.nanmean(fw_c)) if fin.any() else float("nan")
        mean_abs = np.abs(group_shap[sel].astype(np.float64)).mean(axis=0)
        rank = 1 + int(np.sum(mean_abs > mean_abs[gi_water]))
        cstats.append({"raw_cluster": c, "n": int(sel.sum()), "mean_total_shap": float(total[sel].mean()),
                       "mean_water_shap": float(water_g[sel].mean()), "mean_frac_water": fw,
                       "share_water_ge_0_2": float(np.mean(fw_c[fin] >= _S08_RIPARIAN_WET_FRAC)) if fin.any() else 0.0,
                       "water_mean_abs_shap": float(mean_abs[gi_water]), "water_group_rank": rank})
    hot = int(np.argmax([s["mean_total_shap"] for s in cstats]))
    rest = [c for c in range(_S08_N_ZONES) if c != hot]
    eligible = {c: _s08_riparian_reasons(cstats[c]) for c in rest}
    eligible = {c: r for c, r in eligible.items() if r}
    met = bool(eligible)
    lut = np.empty(_S08_N_ZONES, dtype=np.int64)
    if met:
        riparian = max(eligible, key=lambda c: (cstats[c]["mean_frac_water"]
                                                if np.isfinite(cstats[c]["mean_frac_water"]) else -np.inf))
        rest2 = [c for c in rest if c != riparian]
        cool = min(rest2, key=lambda c: cstats[c]["mean_total_shap"])
        transition = [c for c in rest2 if c != cool][0]
        reasons = {hot: "highest mean total SHAP",
                   riparian: ("riparian-eligible (" + "; ".join(eligible[riparian]) + ") with the highest mean "
                              "frac_water among the eligible non-core clusters"),
                   cool: "lower mean total SHAP of the last two clusters", transition: "remaining cluster"}
    else:
        cool, riparian, transition = sorted(rest, key=lambda c: cstats[c]["mean_total_shap"])
        reasons = {hot: "highest mean total SHAP",
                   cool: "lowest mean total SHAP of the three non-core clusters (no cluster meets the riparian "
                         "criterion)",
                   riparian: "middle mean total SHAP of the three non-core clusters (no riparian signal; named "
                             "Riparian Buffer only by the zone contract)",
                   transition: "highest mean total SHAP of the three non-core clusters"}
    lut[cool], lut[riparian], lut[transition], lut[hot] = 0, 1, 2, 3
    clusters = sorted(({**cstats[c], "zone": int(lut[c]), "name": _S08_ZONE_META[int(lut[c])]["name"],
                        "riparian_eligible": c in eligible, "reason": reasons[c]} for c in range(_S08_N_ZONES)),
                      key=lambda d: d["zone"])
    record = {"rule": ("3 Heat Extreme Core = highest mean total SHAP; among the other three a cluster is "
                       f"riparian-eligible if {_S08_RIPARIAN_CRITERION}; if any is eligible: 1 Riparian Buffer = the "
                       "eligible cluster with the highest mean frac_water, 0 Ecological Cool Base = lower mean total "
                       "SHAP of the remaining two, 2 Transition = the last one; if none is: the three ordered by mean "
                       "total SHAP (lowest 0, middle 1, highest 2) (SPEC §1.5)"),
              "space": "18 grouped SHAP values (°C)", "clusters": clusters,
              "riparian_criterion_met": met, "riparian_criterion": _S08_RIPARIAN_CRITERION}
    return lut, record


def _s08_silhouette(X, labels):
    from sklearn.metrics import silhouette_score

    if len(np.unique(labels)) < 2:
        return float("nan")
    return float(silhouette_score(X, labels))

## 8.13 Spatial block bootstrap helpers (block size from the residual correlogram)

In [ ]:
def _s08_residual_correlogram(mode, max_points=3000, bin_km=5.0, max_km=100.0):
    """Pair correlogram of the standardised block20 OOF residuals of the last training frame.

    Falls back to the anomaly target when fewer than 100 OOF residuals exist. Returns (bin upper edges km,
    correlation per bin, variable name).
    """
    w = DF[f"w_{mode}"].to_numpy(np.float64)
    y = DF[f"anom_{mode}"].to_numpy(np.float64)
    frames = _s08_fit_frames(mode)
    rows = np.flatnonzero((_S08_YEAR == frames[-1]) & (w > 0))
    oof = MODEL_RESULTS.get(mode, {}).get("oof", {}).get("block20", {}).get("xgboost")
    values = (y - np.asarray(oof, np.float64))[rows] if oof is not None and len(oof) == len(DF) else np.full(rows.size,
                                                                                                           np.nan)
    name = "block20 XGBoost OOF residual"
    if np.isfinite(values).sum() < 100:
        values, name = y[rows], "anomaly target"
    ok = np.isfinite(values)
    rows, values = rows[ok], values[ok]
    rng = np.random.default_rng(CFG.SEED + 55)
    if rows.size > max_points:
        pick = np.sort(rng.choice(rows.size, size=max_points, replace=False))
        rows, values = rows[pick], values[pick]
    z = (values - values.mean()) / (values.std() or 1.0)
    xy = DF[["x", "y"]].to_numpy(np.float64)[rows] / 1000.0
    edges = np.arange(0.0, max_km + bin_km, bin_km)
    sums = np.zeros(edges.size - 1)
    counts = np.zeros(edges.size - 1)
    for start in range(0, len(z), 500):
        d = np.sqrt(((xy[start:start + 500, None, :] - xy[None, :, :]) ** 2).sum(axis=2))
        prod = z[start:start + 500, None] * z[None, :]
        valid = (d > 0) & (d < max_km)
        b = np.clip((d[valid] // bin_km).astype(np.int64), 0, edges.size - 2)
        sums += np.bincount(b, weights=prod[valid], minlength=edges.size - 1)
        counts += np.bincount(b, minlength=edges.size - 1)
    with np.errstate(invalid="ignore", divide="ignore"):
        corr = sums / counts
    return edges[1:], corr, name


def _s08_block_ids(block_km, rows):
    """Square-block id (row-major from the grid's top-left) of DF ``rows`` for blocks of ``block_km``."""
    b = max(1, int(round(float(block_km) * 1000.0 / float(GRID.res))))
    n_cols = int(_s08_math.ceil(GRID.width / b))
    r = DF["row"].to_numpy(np.int64)[rows]
    c = DF["col"].to_numpy(np.int64)[rows]
    return (r // b) * n_cols + c // b


def _s08_choose_blocks(mode, train_rows):
    """(block code per training row, n blocks, block size km, info) following 8.4."""
    info = {}
    if CFG.BOOT_BLOCK_KM:
        block_km, source = float(CFG.BOOT_BLOCK_KM), "CFG.BOOT_BLOCK_KM"
    else:
        upper, corr, name = _s08_residual_correlogram(mode)
        below = np.flatnonzero(np.isfinite(corr) & (corr < 0.1))
        range_km = float(upper[below[0]] - 2.5) if below.size else float(upper[-1])
        info = {"residual_corr_range_km": range_km, "correlogram_variable": name,
                "correlogram": {f"{u - 5:.0f}-{u:.0f}": (None if not np.isfinite(c) else round(float(c), 3))
                                for u, c in zip(upper[:12], corr[:12])}}
        block_km = max(float(CFG.INNER_BLOCK_KM), 5.0 * _s08_math.ceil(range_km / 5.0))
        source = f"{name} correlogram (< 0.1 beyond ~{range_km:.0f} km)"
    codes, ids = pd.factorize(_s08_block_ids(block_km, train_rows), sort=True)
    while len(ids) < _S08_BOOT_MIN_BLOCKS and block_km - 5.0 >= float(CFG.INNER_BLOCK_KM) and not CFG.BOOT_BLOCK_KM:
        block_km -= 5.0
        codes, ids = pd.factorize(_s08_block_ids(block_km, train_rows), sort=True)
        if "shrunk" not in source:
            source += f"; shrunk to keep >= {_S08_BOOT_MIN_BLOCKS} blocks"
    if len(ids) < _S08_BOOT_MIN_BLOCKS:
        log(f"08 {mode}: only {len(ids)} bootstrap blocks of {block_km:.0f} km (< {_S08_BOOT_MIN_BLOCKS}); "
            "intervals will be coarse", "WARNING")
    return codes.astype(np.int64), len(ids), block_km, {**info, "block_size_source": source}

## 8.14 Text helpers for the recommendation rationale (v1 formatting rules)
Numbers in the sentence are formatted from the exported 4-significant-digit values with round-half-up, like the
dashboard's Intl formatting, so text and table never disagree on a tie.

In [ ]:
_S08_DONOR_LABELS = {"other_vegetation": "other vegetation"}


def _s08_as_exported(value, scale=1):
    return _S08_Decimal(repr(float(f"{float(value):.4g}"))) * scale


def _s08_fmt_dp(value, dp, scale=1):
    q = _s08_as_exported(value, scale).quantize(_S08_Decimal(1).scaleb(-dp), rounding=_S08_ROUND_HALF_UP)
    return f"{abs(q) if q == 0 else q:f}"


def _s08_fmt_sig(value, sig=3):
    d = _s08_as_exported(value)
    if d == 0:
        return "0"
    q = d.quantize(_S08_Decimal(1).scaleb(d.adjusted() - (sig - 1)), rounding=_S08_ROUND_HALF_UP)
    text = f"{q:,f}"
    return text.rstrip("0").rstrip(".") if "." in text else text


def _s08_fmt_feature_value(feature, value):
    display = FEATURE_META.get(feature, {}).get("display", "number")
    if display == "percent":
        pct = float(_s08_as_exported(value, 100))
        return f"{_s08_fmt_dp(value, 1, 100)}%" if 0 < abs(pct) < 9.95 else f"{_s08_fmt_dp(value, 0, 100)}%"
    if display == "index":
        return _s08_fmt_dp(value, 2)
    return _s08_fmt_sig(value, 3)


def _s08_pp(frac):
    v = float(_s08_as_exported(frac, 100))
    return _s08_fmt_dp(frac, 1, 100) if abs(v) < 9.95 else _s08_fmt_dp(frac, 0, 100)


def _s08_donor_text(plan):
    moves = sorted(((k, v) for k, v in plan["donors"].items() if abs(v) >= 0.005), key=lambda kv: -abs(kv[1]))[:2]
    verb = "taken from" if plan["step"] > 0 else "given to"
    lead = f" The land is {verb} the other land covers (including other vegetation) in proportion to their shares"
    if moves:
        names = [f"{_S08_DONOR_LABELS.get(k, FEATURE_META.get(k, {}).get('label', k)).lower()} "
                 f"({'+' if v > 0 else '-'}{_s08_pp(abs(v))} pp)" for k, v in moves]
        lead += f", mostly {' and '.join(names)}"
    return lead + "; NDVI, spring NDVI, NDBI, MNDWI and albedo follow through the land-cover coupling."


def _s08_rationale(plan, ci, ci_label):
    feature = plan["feature"]
    label = FEATURE_META.get(feature, {}).get("label", feature)

    def fv(v):
        return _s08_fmt_feature_value(feature, v)

    verb = "raising" if plan["step"] > 0 else "lowering"
    step, applied = abs(float(plan["step"])), abs(float(plan.get("mean_applied", plan["step"]) or 0.0))
    capped = applied < 0.95 * step          # many cells hit a bound or have little land to convert
    median = f"(zone median {fv(plan['current'])} → {fv(plan['target'])})"
    if feature in _S08_FRACTIONS:
        if capped:
            change = (f"{verb} {label.lower()} by up to {_s08_pp(step)} percentage points per cell "
                      f"({_s08_pp(applied)} pp on average: cells near the bound or with little land to convert "
                      f"change less) {median}")
        else:
            change = f"{verb} {label.lower()} by {_s08_pp(step)} percentage points in every cell {median}"
    elif feature == "ndvi":
        if capped:
            change = (f"{verb} NDVI by up to {_s08_fmt_dp(step, 2)} per cell ({_s08_fmt_dp(applied, 2)} on average: "
                      f"cells near the bound change less) {median}, i.e. greening within the existing land covers")
        else:
            change = (f"{verb} NDVI by {_s08_fmt_dp(step, 2)} in every cell {median}, i.e. greening within the "
                      "existing land covers")
    else:
        if capped:
            change = (f"{verb} {label.lower()} by up to {_s08_fmt_sig(step, 3)} per cell "
                      f"({_s08_fmt_sig(applied, 3)} on average: cells near the bound change less) {median}")
        else:
            change = f"{verb} {label.lower()} by {_s08_fmt_sig(step, 3)} in every cell {median}"
    donors = _s08_donor_text(plan) if feature in _S08_FRACTIONS else ""
    ci_text = f"{ci_label} {ci[0]:+.2f} to {ci[1]:+.2f} °C" if ci else "no bootstrap interval"
    kind = plan["kind"]
    if kind == "saturation":
        why = (f" The target is where the {label} effect saturates ({fv(plan['target'])} on the pooled dependence "
               "curve); it already achieves ≥ 90% of the cooling available within the bounds.")
    elif kind == "breakpoint":
        why = (f" The target is where the {label} response changes slope ({fv(plan['target'])}); it already "
               "achieves ≥ 90% of the cooling available within the bounds.")
    elif kind == "bound":
        why = " The target is the feasibility bound (the zone's own 90th/10th percentile, or the maximum step)."
    else:
        why = (" The target is the smallest step that achieves ≥ 90% of the cooling available within the "
               "feasibility bounds.")
    spill = ""
    if plan.get("spill_n_cells", 0) > 0:
        spill = (f" The 3- and 15-cell neighbourhood means are recomputed (distances held fixed): "
                 f"{plan['spill_n_cells']:,} cells outside the zone have changed neighbourhood inputs (mean LST "
                 f"change {plan['spill_delta_c']:+.2f} °C over them, cells whose prediction does not move included).")
    return (f"In the final {plan['mode']} model, {change} is associated with a mean LST change of "
            f"{plan['expected_delta_c']:+.2f} °C over the zone's {plan['n_cells']:,} cells ({ci_text}).{donors}"
            f"{spill} This is a model-based ceteris-paribus estimate, not a causal effect.{why}")

## 8.15 Per-mode pipeline
Order per mode: (1) TreeSHAP on every cell × frame (additivity-checked), group sums; (2) K-means zones and
`ASSIGN_ZONES`; (3) grouped interactions; (4) dependence curves and point thresholds; (5) recommendation planning
with the final model; (6) block bootstrap (replicates cached individually); (7) intervals, importance, profiles,
transitions and recommendations. The whole mode payload is stage-cached on the final booster, `DF` and the bootstrap
settings.

In [ ]:
_S08_LEVERS_RULES = {"frac_built": ("decrease",), "frac_barren": ("decrease",), "frac_forest": ("increase",),
                     "frac_water": ("increase",), "frac_cropland": ("increase", "decrease"), "ndvi": ("increase",),
                     "lm_ed": ("increase", "decrease"), "lm_pd": ("increase", "decrease"),
                     "lm_contag": ("increase", "decrease")}
_S08_REC_MAX_STEP = {**{f: 0.20 for f in _S08_FRACTIONS}, "ndvi": 0.15}
_S08_REC_MIN_COOLING_C = 0.05
_S08_REC_EFFICIENCY = 0.90
_S08_REC_WATER_ZONE = 0.15
_S08_REC_MAX_PER_ZONE = 4
_S08_MIN_DRIVER_C = 0.01
# Levers = primary features of the actionable groups (SPEC §4.4).
_S08_LEVERS = [(g, GROUP_META[g]["primary"]) for g in _S08_GROUPS if GROUP_META[g]["actionable"]]
_s08_unknown = [f for _, f in _S08_LEVERS if f not in _S08_LEVERS_RULES]
if _s08_unknown:
    log(f"08: actionable primaries without an intervention rule are not recommended: {_s08_unknown}", "WARNING")
log("08: recommendation levers: " + ", ".join(f"{g}->{f}" for g, f in _S08_LEVERS))


def _s08_shap_pass(mode, booster):
    """Margins for every DF row, TreeSHAP for valid rows -> group SHAP; additivity checked (raises)."""
    rows = np.flatnonzero(_S08_ROW_VALID)
    try:
        margin = _s08_predict_all(booster, _S08_X, device=_S08_DEVICE, kind="margin").astype(np.float64)
        contribs = _s08_predict_all(booster, _S08_X[rows], device=_S08_DEVICE, kind="contribs")
    finally:
        booster.set_param({"device": "cpu"})
    phi = contribs[:, :_S08_K].astype(np.float64)
    bias = contribs[:, _S08_K].astype(np.float64)
    del contribs
    base = float(np.median(bias)) if bias.size else 0.0
    if bias.size and float(np.ptp(bias)) > 1e-4:
        log(f"08 {mode}: SHAP bias column varies by {np.ptp(bias):.2e} across rows (expected constant)", "WARNING")
    feat_err = float(np.max(np.abs(phi.sum(axis=1) + bias - margin[rows]))) if rows.size else 0.0
    group = phi @ _S08_MEMBER
    group_err = float(np.max(np.abs(group.sum(axis=1) + base - margin[rows]))) if rows.size else 0.0
    if not (feat_err < 1e-3 and group_err < 1e-3):
        raise RuntimeError(f"08 {mode}: TreeSHAP additivity violated (feature {feat_err:.2e}, group {group_err:.2e})")
    full = np.full((len(DF), _S08_NG), np.nan, dtype=np.float32)
    full[rows] = group.astype(np.float32)
    feat_mean_abs = {}
    for y in _S08_FRAMES:
        sel = _S08_YEAR[rows] == y
        feat_mean_abs[y] = np.abs(phi[sel]).mean(axis=0) if sel.any() else np.full(_S08_K, np.nan)
    fit = np.isin(_S08_YEAR[rows], _s08_fit_frames(mode))
    feat_global = np.abs(phi[fit]).mean(axis=0) if fit.any() else np.full(_S08_K, np.nan)
    crosscheck = _s08_crosscheck_shap_package(booster, rows, phi)
    del phi, group
    return {"margin": margin.astype(np.float32), "group": full, "base": base, "additivity_feature": feat_err,
            "additivity_group": group_err, "feature_mean_abs": feat_global, "feature_mean_abs_by_frame": feat_mean_abs,
            "crosscheck_max_diff": crosscheck, "n_rows": int(rows.size)}


def _s08_crosscheck_shap_package(booster, rows, phi, n_rows=300):
    """Compare XGBoost's native TreeSHAP with shap.TreeExplainer on a few rows (warn-only; None if unavailable)."""
    try:
        import shap
    except ImportError:
        log("08: shap package not installed; skipping the TreeExplainer cross-check")
        return None
    if rows.size == 0:
        return None
    pick = np.random.default_rng(CFG.SEED).choice(rows.size, size=min(n_rows, rows.size), replace=False)
    try:
        booster.set_param({"device": "cpu"})
        with _s08_warnings.catch_warnings():
            _s08_warnings.simplefilter("ignore")
            explainer = shap.TreeExplainer(booster)
            ref = np.asarray(explainer.shap_values(_S08_X[rows[pick]]), dtype=np.float64)
    except Exception as exc:  # noqa: BLE001 - shap/xgboost version skew must not break the pipeline
        log(f"08: shap.TreeExplainer cross-check skipped: {type(exc).__name__}: {str(exc)[:160]}", "WARNING")
        return None
    diff = float(np.max(np.abs(ref - phi[pick])))
    level = "WARNING" if diff > 1e-3 else "INFO"
    log(f"08: shap.TreeExplainer vs XGBoost TreeSHAP on {pick.size} rows: max |diff| {diff:.2e}", level)
    return diff


def _s08_zone_fit(mode, group_full, fit_rows):
    """K-means (K = 4, 20 inits) on a seeded sample of the fit rows, v2 labelling, diagnostics K = 2..8."""
    rng = np.random.default_rng(CFG.SEED + 31)
    Xg = group_full[fit_rows].astype(np.float64)
    fit_pick = (np.sort(rng.choice(fit_rows.size, size=_S08_KMEANS_FIT_ROWS, replace=False))
                if fit_rows.size > _S08_KMEANS_FIT_ROWS else np.arange(fit_rows.size))
    _, inertia, centers = _s08_kmeans(Xg[fit_pick], _S08_N_ZONES, n_init=20, seed=CFG.SEED)
    raw = _s08_nearest_centroid(Xg, centers)
    frac_water = _S08_X[fit_rows, FEATURES.index("frac_water")].astype(np.float64)
    lut, labelling = _s08_label_clusters(raw, Xg, frac_water)
    order = np.argsort(lut)                       # raw cluster of canonical zone 0..3
    centroids = centers[order]
    zones_fit = lut[raw]
    sil_pick = (np.sort(rng.choice(fit_rows.size, size=_S08_SIL_ROWS, replace=False))
                if fit_rows.size > _S08_SIL_ROWS else np.arange(fit_rows.size))
    silhouette = _s08_silhouette(Xg[sil_pick], zones_fit[sil_pick])
    diag_pick = (np.sort(rng.choice(fit_rows.size, size=_S08_DIAG_ROWS, replace=False))
                 if fit_rows.size > _S08_DIAG_ROWS else np.arange(fit_rows.size))
    diag_sil = diag_pick[:min(diag_pick.size, _S08_SIL_ROWS)]
    diagnostics = {"k": [], "inertia": [], "silhouette": [], "n_init": 10, "n_rows": int(diag_pick.size)}
    Xd = Xg[diag_pick]
    sil_pos = np.searchsorted(diag_pick, diag_sil)
    for k in range(2, 9):
        lab_k, in_k, _ = _s08_kmeans(Xd, k, n_init=10, seed=CFG.SEED)
        diagnostics["k"].append(k)
        diagnostics["inertia"].append(in_k)
        diagnostics["silhouette"].append(_s08_silhouette(Xd[sil_pos], lab_k[sil_pos]))
    best_k = int(diagnostics["k"][int(np.nanargmax(diagnostics["silhouette"]))])
    log(f"08 {mode}: zones fitted on {fit_pick.size:,} of {fit_rows.size:,} rows (inertia {inertia:,.1f}, "
        f"silhouette {silhouette:.3f}); diagnostics silhouette peaks at K={best_k} (K = 4 is the contract)")
    return {"centroids": centroids.astype(np.float32), "labelling": labelling, "k": _S08_N_ZONES,
            "silhouette": silhouette, "inertia": inertia, "diagnostics": diagnostics,
            "groups": list(_S08_GROUPS), "n_fit_rows": int(fit_pick.size), "n_rows": int(fit_rows.size),
            "space": "grouped SHAP (°C), 18 groups in SHAP_GROUPS order", "centroid_order": "canonical zone id",
            "nodata": _S08_ZONE_NODATA}


def _s08_interaction_pass(mode, booster, zones_full):
    """Grouped SHAP interactions (18 x 18 mean |Phi_gh|) globally and per zone from sampled rows of every frame."""
    rng = np.random.default_rng(CFG.SEED + 7)
    batch = max(64, int(CFG.SHAP_BATCH) // 16) if _S08_ON_GPU else 256
    sum_all = np.zeros((_S08_NG, _S08_NG))
    sum_zone = np.zeros((_S08_N_ZONES, _S08_NG, _S08_NG))
    n_zone = np.zeros(_S08_N_ZONES, dtype=np.int64)
    n_total, worst = 0, 0.0
    M = _S08_MEMBER
    group_ref = _S08_STATE[mode]["group"]
    fallback_frames = []
    try:
        for year in _S08_FRAMES:
            rows = np.flatnonzero((_S08_YEAR == year) & _S08_ROW_VALID)
            if rows.size > _S08_INTER_ROWS:
                rows = np.sort(rng.choice(rows, size=_S08_INTER_ROWS, replace=False))
            state = {"cpu_fallback": False}
            for start, stop, inter in _s08_iter_predict(booster, _S08_X[rows], device=_S08_DEVICE, batch=batch,
                                                         kind="interactions", state=state):
                r = rows[start:stop]
                phi = inter[:, :_S08_K, :_S08_K].astype(np.float64)
                # rows of the full matrix (incl. the bias column) sum back to the SHAP values
                recon = inter[:, :_S08_K, :].sum(axis=2, dtype=np.float64) @ M
                worst = max(worst, float(np.nanmax(np.abs(recon - group_ref[r]))))
                G = np.matmul(M.T[None, :, :], np.matmul(phi, M))            # (b, 18, 18)
                mag = np.abs(G)
                sum_all += mag.sum(axis=0)
                zb = zones_full[r]
                for z in range(_S08_N_ZONES):
                    sel = zb == z
                    if sel.any():
                        sum_zone[z] += mag[sel].sum(axis=0)
                        n_zone[z] += int(sel.sum())
                n_total += r.size
                del inter, phi, G, mag
                if state["cpu_fallback"] and stop < rows.size:
                    fallback_frames.append(int(year))
                    log(f"08 {mode}: interactions {year} fell back to the CPU; remaining rows of the frame skipped",
                        "WARNING")
                    break
            booster.set_param({"device": _S08_DEVICE})
    finally:
        booster.set_param({"device": "cpu"})
    if worst > 1e-2:
        log(f"08 {mode}: group interaction rows deviate from group SHAP by up to {worst:.3e}", "WARNING")
    glob = sum_all / max(n_total, 1)
    by_zone = {z: (sum_zone[z] / n_zone[z] if n_zone[z] else np.zeros((_S08_NG, _S08_NG))) for z in range(4)}
    return {"groups": list(_S08_GROUPS), "global": glob, "by_zone": by_zone, "n_rows": int(n_total),
            "rows_per_frame": int(_S08_INTER_ROWS), "max_consistency_error": worst,
            "cpu_fallback_frames": fallback_frames,
            "definition": "mean |sum of member-pair SHAP interaction values| (diagonal = within-group effect)"}


def _s08_point_thresholds(x, s, edges):
    centers, mean, counts = _s08_binned_curve(x, s, edges)
    bp, slope_before, slope_after = _s08_hinge_breakpoint(centers, mean, counts)
    return {"zero_crossing": _s08_zero_crossing(centers, mean),
            "zero_crossing_multiple": _s08_zero_crossing_is_multiple(centers, mean),
            "breakpoint": bp, "slope_before": slope_before, "slope_after": slope_after,
            "saturation": _s08_saturation_point(centers, mean),
            "direction": _s08_effect_direction(x, s), "slope": _s08_curve_slope(centers, mean, counts),
            "curve": (centers, mean, counts)}


def _s08_plan_zone(mode, zone, booster, base_margin_now, thresholds):
    """Candidate actions of one zone evaluated with the final model (focal recompute); best first, ≤ 4."""
    zones_now = _S08_STATE[mode]["zones_obs"][_S08_NOW]
    cells = np.flatnonzero(zones_now == zone)
    if cells.size < 10:
        log(f"08 {mode}: zone {zone} has {cells.size} cells in {_S08_NOW}; no recommendations", "WARNING")
        return []
    Xz = _S08_X[_s08_frame_slice(_S08_NOW)][cells]
    jw = FEATURES.index("frac_water")
    with _s08_warnings.catch_warnings():
        _s08_warnings.simplefilter("ignore", RuntimeWarning)
        water_median = float(np.nanmedian(Xz[:, jw])) if np.isfinite(Xz[:, jw]).any() else 0.0
    plans = []
    for group, feature in _S08_LEVERS:
        if feature not in _S08_LEVERS_RULES:
            continue
        if feature == "ndvi" and water_median > _S08_REC_WATER_ZONE:
            continue
        j = FEATURES.index(feature)
        vals = Xz[:, j][np.isfinite(Xz[:, j])].astype(np.float64)
        if vals.size < 10:
            continue
        current = float(np.median(vals))
        p10, p90 = (float(v) for v in np.percentile(vals, [10, 90]))
        fstat = FEATURE_STATS[feature]
        spread = float(fstat["p99"]) - float(fstat["p01"])
        min_step = max(1e-6, 0.01 * spread) if np.isfinite(spread) and spread > 0 else 1e-3
        lo_phys, hi_phys = _S08_BOUNDS.get(feature, (-np.inf, np.inf))
        cap = _S08_REC_MAX_STEP.get(feature, np.inf)
        th = thresholds[group]
        best_plan = None
        for direction in _S08_LEVERS_RULES[feature]:
            if direction == "increase":
                bound = min(p90, current + cap, hi_phys)
                if bound - current < min_step:
                    continue
                lo_t, hi_t = current + min_step, bound
            else:
                bound = max(p10, current - cap, lo_phys)
                if current - bound < min_step:
                    continue
                lo_t, hi_t = bound, current - min_step
            cands = [(current + (bound - current) * q, "partial") for q in (0.25, 0.5, 0.75)] + [(bound, "bound")]
            cands += [(float(th[k]), k) for k in ("breakpoint", "saturation")
                      if th[k] is not None and lo_t <= float(th[k]) <= hi_t]
            results = []
            for target, kind in cands:
                cf = _s08_counterfactual(_S08_NOW, cells, feature, target - current)
                if cf["zone_ok"].size == 0:
                    continue
                zd, sd, ns = _s08_cf_effect(booster, "cpu", _S08_NOW, cf, base_margin=base_margin_now)
                results.append({"target": float(target), "kind": kind, "delta": zd, "spill_delta": sd,
                                "spill_n": ns, "n_cells": int(cf["zone_ok"].size), "donors": cf["donors"],
                                "mean_applied": float(np.mean(cf["applied"])) if cf["applied"].size else 0.0})
            if not results:
                continue
            best = min(r["delta"] for r in results)
            if not best < -_S08_REC_MIN_COOLING_C:
                continue
            good = [r for r in results if r["delta"] <= _S08_REC_EFFICIENCY * best]
            chosen = min(good, key=lambda r: (abs(r["target"] - current),
                                              r["kind"] not in ("saturation", "breakpoint")))
            plan = {"mode": mode, "zone": int(zone), "group": group, "feature": feature, "action": direction,
                    "current": current, "target": chosen["target"], "step": chosen["target"] - current,
                    "kind": chosen["kind"], "expected_delta_c": chosen["delta"], "best_delta_c": best,
                    "n_cells": chosen["n_cells"], "donors": chosen["donors"], "p10": p10, "p90": p90,
                    "bound": bound, "cap": (None if not np.isfinite(cap) else cap),
                    "spill_delta_c": chosen["spill_delta"], "spill_n_cells": chosen["spill_n"],
                    "mean_applied": chosen["mean_applied"], "cells": cells, "year": _S08_NOW}
            if best_plan is None or plan["expected_delta_c"] < best_plan["expected_delta_c"]:
                best_plan = plan
        if best_plan is not None:
            plans.append(best_plan)
    plans.sort(key=lambda p: p["expected_delta_c"])
    return plans[:_S08_REC_MAX_PER_ZONE]


def _s08_bootstrap_replicate(job, gpu_id):
    """Train one block-bootstrap replicate and reduce it immediately (cached per replicate)."""
    mode, b = job
    st = _S08_STATE[mode]
    sc = stage_cache(f"08_boot_{mode}", {**st["boot_key"], "b": int(b)})
    if sc.hit:
        try:
            return sc.load()
        except StageCacheError as exc:
            log(f"{exc}; recomputing replicate {b}", "WARNING")
    t0 = _s08_time.time()
    device = xgb_device_for(gpu_id) if _S08_ON_GPU else "cpu"
    try:
        rng = np.random.default_rng(CFG.SEED + 1000 + 7919 * int(b) + (0 if mode == "day" else 104729))
        nb = st["n_blocks"]
        mult = np.bincount(rng.integers(0, nb, nb), minlength=nb)[st["block_codes"]].astype(np.float64)
        keep = mult > 0
        rows = st["train_rows"][keep]
        weight = (st["w"][rows].astype(np.float64) * mult[keep]).astype(np.float32)
        booster, used = _s08_train_booster(mode, _S08_X[rows], st["y"][rows], weight, st["rounds"], device,
                                           CFG.SEED + int(b) + 1)
        contribs = _s08_predict_all(booster, _S08_X[st["eval_rows"]], device=used, kind="contribs")
        group = contribs[:, :_S08_K].astype(np.float64) @ _S08_MEMBER
        del contribs
        curves = np.full((_S08_NG, st["max_bins"]), np.nan)
        thresholds = np.full((_S08_NG, 3), np.nan)
        for gi in range(_S08_NG):
            idx = st["eval_bin_idx"][gi]
            ok = idx >= 0
            nbins = len(st["edges"][gi]) - 1
            counts = st["eval_counts"][gi]
            sums = np.bincount(idx[ok], weights=group[ok, gi], minlength=nbins)
            with np.errstate(invalid="ignore", divide="ignore"):
                mean = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
            curves[gi, :nbins] = mean
            thresholds[gi] = _s08_curve_thresholds(st["curves"][gi][0], mean, counts, zc_ref=st["zc_ref"][gi],
                                                   zc_tol=st["zc_tol"][gi])
        rec = np.array([_s08_cf_effect(booster, used, p["year"], cf)[0]
                        for p, cf in zip(st["plans"], st["plan_cf"])], dtype=np.float64)
        booster.set_param({"device": "cpu"})
        del booster
        result = {"ok": True, "b": int(b), "device": used, "mean_abs": np.abs(group).mean(axis=0),
                  "curves": curves, "thresholds": thresholds, "rec_deltas": rec,
                  "seconds": _s08_time.time() - t0, "n_rows": int(rows.size)}
        del group
        log(f"  08 {mode} bootstrap {b + 1}/{CFG.N_BOOTSTRAP} on {used}: {rows.size:,} rows, "
            f"{result['seconds']:.1f}s")
        return sc.save(result)
    except Exception as exc:  # noqa: BLE001 - one failed replicate must not lose the others
        log(f"  08 {mode} bootstrap replicate {b} failed on {device}: {type(exc).__name__}: {exc}", "WARNING")
        return {"ok": False, "b": int(b), "device": device, "error": f"{type(exc).__name__}: {exc}"}


def _s08_scatter(mode, gi, partner_gi, rng, fit_rows, max_points=1500):
    """≤ max_points fit rows with finite x, stratified across frames; colour = partner group's primary feature."""
    j = _S08_PRIMARY_IDX[gi]
    jc = _S08_PRIMARY_IDX[partner_gi]
    frames = sorted(set(_S08_YEAR[fit_rows].tolist()))
    per = max(1, max_points // max(len(frames), 1))
    picks = []
    for year in frames:
        rows = fit_rows[(_S08_YEAR[fit_rows] == year) & np.isfinite(_S08_X[fit_rows, j])]
        if rows.size:
            picks.append(rng.choice(rows, size=min(per, rows.size), replace=False))
    rows = np.sort(np.concatenate(picks)) if picks else np.zeros(0, np.int64)
    st = _S08_STATE[mode]
    return {"x": _S08_X[rows, j].astype(np.float64), "shap": st["group"][rows, gi].astype(np.float64),
            "color": _S08_X[rows, jc].astype(np.float64), "color_feature": FEATURES[jc],
            "color_group": _S08_GROUPS[partner_gi], "zone": st["zones_full"][rows].astype(np.int64),
            "year": _S08_YEAR[rows].astype(np.int64)}


def _s08_build_threshold(gi, pt, boot_thr):
    out = {"primary": _S08_PRIMARY[gi]}
    centers, mean, counts = pt["curve"]
    finite_mean = mean[np.isfinite(mean)]
    for col, key in enumerate(("zero_crossing", "breakpoint", "saturation")):
        values = boot_thr[:, gi, col] if boot_thr.size else np.zeros(0)
        ci, support = _s08_boot_ci(values, pt[key])
        out[key] = pt[key]
        out[f"{key}_ci"] = ci
        out[f"{key}_support"] = support
    out["zero_crossing_flag"] = ("multiple" if pt["zero_crossing_multiple"] and pt["zero_crossing"] is None
                                 else None)
    out["slope_before"] = pt["slope_before"]
    out["slope_after"] = pt["slope_after"]
    out["slope"] = pt["slope"]
    out["effect_range"] = float(np.ptp(finite_mean)) if finite_mean.size else None
    out["direction"] = pt["direction"]
    order = ["primary", "zero_crossing", "zero_crossing_ci", "zero_crossing_support", "zero_crossing_flag",
             "breakpoint", "breakpoint_ci", "breakpoint_support", "slope_before", "slope_after", "saturation",
             "saturation_ci", "saturation_support", "effect_range", "direction", "slope"]
    return {k: out[k] for k in order}


def _s08_top_pairs(matrix, n=3):
    iu = np.triu_indices(_S08_NG, k=1)
    vals = np.asarray(matrix)[iu]
    order = np.argsort(vals)[::-1][:n]
    return [{"a": _S08_GROUPS[iu[0][o]], "b": _S08_GROUPS[iu[1][o]], "value": float(vals[o])} for o in order]


def _s08_top_districts(cells, n=5):
    d = np.asarray(DISTRICT_IDX)[CELL_ROWS[cells], CELL_COLS[cells]].astype(np.int64)
    total = max(int(cells.size), 1)
    ids, counts = np.unique(d[d >= 0], return_counts=True)
    order = np.argsort(counts)[::-1][:n]
    return [{"district": int(ids[o]), "name": _S08_DISTRICT_NAMES.get(int(ids[o]), str(ids[o])),
             "share": float(counts[o] / total)} for o in order]


def _s08_zone_profiles(mode, recs):
    """ZONE_PROFILES[m] (v1 keys; area/lst keyed {"obs": {year: v}}; lst_mean filled by section 09)."""
    st = _S08_STATE[mode]
    zones_full = st["zones_full"]
    fit_rows = st["fit_rows"]
    anom = DF[f"anom_{mode}"].to_numpy(np.float64)
    margin = st["margin"].astype(np.float64)
    profiles = []
    now_sl = _s08_frame_slice(_S08_NOW)
    labelling = (ZONE_MODEL.get(mode) or {}).get("labelling") or {}
    met = bool(labelling.get("riparian_criterion_met", True))
    for meta in _S08_ZONE_META:
        z = meta["id"]
        display, caveat = meta["name"], None
        if z == 1 and not met:
            fw1 = next((c["mean_frac_water"] for c in labelling.get("clusters", []) if c["zone"] == 1), float("nan"))
            display = _S08_NO_RIPARIAN_NAME
            caveat = ("Named Riparian Buffer by the SPEC zone rule, but no cluster shows a riparian signature (mean "
                      f"water {100 * fw1:.1f} %); this is the middle-heat cluster of the three non-core zones.")
        n_cells, area, pred_mean, obs_mean = {}, {}, {}, {}
        for year in _S08_FRAMES:
            sl = _s08_frame_slice(year)
            mz = zones_full[sl] == z
            n = int(mz.sum())
            n_cells[str(year)] = n
            area[str(year)] = n * _S08_RES_KM2
            pred_mean[str(year)] = float(margin[sl][mz].mean()) if n else None
            ov = anom[sl][mz]
            obs_mean[str(year)] = float(np.nanmean(ov)) if n and np.isfinite(ov).any() else None
        mask_fit = zones_full[fit_rows] == z
        shap_means = (st["group"][fit_rows[mask_fit]].astype(np.float64).mean(axis=0) if mask_fit.any()
                      else np.zeros(_S08_NG))
        now_cells = np.flatnonzero(zones_full[now_sl] == z)
        with _s08_warnings.catch_warnings():
            _s08_warnings.simplefilter("ignore", RuntimeWarning)
            fmeans = (np.nanmean(_S08_X[now_sl][now_cells].astype(np.float64), axis=0) if now_cells.size
                      else np.full(_S08_K, np.nan))
        order = np.argsort(shap_means)
        profile = {
            "id": z, "name": meta["name"], "display_name": display, "label_caveat": caveat,
            "color": meta["color"], "mode": mode, "description": "",
            "n_cells": n_cells, "area_km2": {"obs": area}, "lst_mean": {"obs": {}}, "lst_pred_mean": {"obs": {}},
            "anom_pred_mean": {"obs": pred_mean}, "anom_obs_mean": {"obs": obs_mean},
            "feature_means": {f: float(fmeans[FEATURES.index(f)]) for f in BASE_FEATURES},
            "feature_means_frame": _S08_NOW,
            "shap_means": {g: float(shap_means[i]) for i, g in enumerate(_S08_GROUPS)},
            "top_warming": [{"group": _S08_GROUPS[i], "feature": _S08_PRIMARY[i], "shap": float(shap_means[i])}
                            for i in order[::-1][:3] if shap_means[i] >= _S08_MIN_DRIVER_C],
            "top_cooling": [{"group": _S08_GROUPS[i], "feature": _S08_PRIMARY[i], "shap": float(shap_means[i])}
                            for i in order[:3] if shap_means[i] <= -_S08_MIN_DRIVER_C],
            "top_interactions": _s08_top_pairs(st["interactions"]["by_zone"][z]),
            "top_districts": _s08_top_districts(now_cells),
            "recommendations": recs.get(z, []),
        }
        share = n_cells[str(_S08_NOW)] / max(_S08_NC, 1)
        lab = lambda g: GROUP_META[g]["label"]  # noqa: E731
        warm = ", ".join(f"{lab(t['group'])} ({t['shap']:+.2f} °C)" for t in profile["top_warming"]) or "none"
        cool = ", ".join(f"{lab(t['group'])} ({t['shap']:+.2f} °C)" for t in profile["top_cooling"]) or "none"
        dist = ", ".join(f"{t['name']} ({100 * t['share']:.0f}%)" for t in profile["top_districts"][:3]) or "n/a"
        am = obs_mean[str(_S08_NOW)]
        anom_txt = f"mean observed {mode} anomaly {am:+.2f} °C" if am is not None else f"no observed {mode} target"
        lead = f"{display} ({mode}): " + (f"{caveat} " if caveat else "")
        profile["description"] = (f"{lead}{area[str(_S08_NOW)]:,.0f} km² in {_S08_NOW} "
                                  f"({100 * share:.0f}% of the study area), {anom_txt}. Main warming drivers: {warm}. "
                                  f"Main cooling drivers: {cool}. Largest shares in {dist}.")
        profiles.append(profile)
    return profiles


def _s08_transitions(mode):
    """{"obs": {"y0->y1": 4x4}} between consecutive XAI frames (cells zoned in both; rows = earlier zone)."""
    zones_full = _S08_STATE[mode]["zones_full"]
    out = {}
    for y0, y1 in zip(_S08_FRAMES[:-1], _S08_FRAMES[1:]):
        a = zones_full[_s08_frame_slice(y0)].astype(np.int64)
        b = zones_full[_s08_frame_slice(y1)].astype(np.int64)
        ok = (a < _S08_N_ZONES) & (b < _S08_N_ZONES)
        counts = np.bincount(a[ok] * _S08_N_ZONES + b[ok], minlength=_S08_N_ZONES ** 2)
        out[f"{y0}->{y1}"] = counts.reshape(_S08_N_ZONES, _S08_N_ZONES).astype(int).tolist()
    return {"obs": out}


_S08_STATE = {}


def _s08_run_mode(mode, booster_hash):
    """Full explanation pipeline of one mode -> payload dict (stage-cacheable: numpy / plain Python only)."""
    booster = FINAL_MODELS[mode]
    st = {"mode": mode, "booster_hash": booster_hash}
    _S08_STATE[mode] = st
    fit_frames = _s08_fit_frames(mode)
    w = DF[f"w_{mode}"].to_numpy(np.float32)
    y = DF[f"anom_{mode}"].to_numpy(np.float32)
    st.update({"w": w, "y": y, "fit_frames": fit_frames})
    rounds = int(booster.num_boosted_rounds())

    # ---- (1) TreeSHAP ----------------------------------------------------------------------------------------
    with timer(f"08_{mode}_treeshap"):
        shp = _s08_shap_pass(mode, booster)
    st.update({"margin": shp["margin"], "group": shp["group"]})
    log(f"08 {mode}: TreeSHAP on {shp['n_rows']:,} rows, base {shp['base']:+.4f} °C, additivity max error "
        f"feature {shp['additivity_feature']:.2e} / group {shp['additivity_group']:.2e}")
    fit_rows = np.flatnonzero(np.isin(_S08_YEAR, fit_frames) & _S08_ROW_VALID)
    st["fit_rows"] = fit_rows

    # ---- (2) zones ----------------------------------------------------------------------------------------------
    with timer(f"08_{mode}_zones"):
        zm = _s08_zone_fit(mode, shp["group"], fit_rows)
        ZONE_MODEL[mode] = zm
        zones_full = ASSIGN_ZONES(mode, shp["group"])
    st["zones_full"] = zones_full
    st["zones_obs"] = {yr: zones_full[_s08_frame_slice(yr)] for yr in _S08_FRAMES}

    # ---- (3) interactions ---------------------------------------------------------------------------------------
    with timer(f"08_{mode}_interactions"):
        inter = _s08_interaction_pass(mode, booster, zones_full)
    st["interactions"] = inter
    partner = inter["global"].copy()
    np.fill_diagonal(partner, -np.inf)

    # ---- (4) dependence curves and point thresholds --------------------------------------------------------------
    edges, curves, points = [], [], []
    for gi in range(_S08_NG):
        x = _S08_X[fit_rows, _S08_PRIMARY_IDX[gi]]
        s = shp["group"][fit_rows, gi]
        e = _s08_quantile_edges(x, CFG.DEP_BINS)
        pt = _s08_point_thresholds(x, s, e)
        edges.append(e)
        curves.append(pt["curve"])
        points.append(pt)
    thresholds_point = {g: points[gi] for gi, g in enumerate(_S08_GROUPS)}
    st.update({"edges": edges, "curves": curves, "max_bins": max(len(e) - 1 for e in edges),
               "zc_ref": [np.nan if p["zero_crossing"] is None else float(p["zero_crossing"]) for p in points],
               "zc_tol": [_s08_zc_tolerance(_S08_X[fit_rows, _S08_PRIMARY_IDX[gi]], edges[gi])
                          for gi in range(_S08_NG)]})

    # ---- (5) recommendation planning with the final model --------------------------------------------------------
    with timer(f"08_{mode}_recommendation_planning"):
        base_now = st["margin"][_s08_frame_slice(_S08_NOW)].astype(np.float64)
        booster.set_param({"device": "cpu"})
        plans = [p for meta in _S08_ZONE_META for p in _s08_plan_zone(mode, meta["id"], booster, base_now,
                                                                      thresholds_point)]
        plan_cf = [_s08_counterfactual(p["year"], p["cells"], p["feature"], p["step"]) for p in plans]
        for p, cf in zip(plans, plan_cf):
            zd, sd, ns = _s08_cf_effect(booster, "cpu", p["year"], cf)
            if abs(zd - p["expected_delta_c"]) > 1e-4:
                log(f"08 {mode}: replayed delta of {p['zone']}/{p['feature']} {zd:.5f} != planned "
                    f"{p['expected_delta_c']:.5f}", "WARNING")
            p["expected_delta_c"], p["spill_delta_c"], p["spill_n_cells"] = zd, sd, ns
    st.update({"plans": plans, "plan_cf": plan_cf})
    log(f"08 {mode}: {len(plans)} planned action(s) over {_S08_N_ZONES} zones, "
        f"{sum(cf['pos'].size for cf in plan_cf):,} counterfactual rows per model")

    # ---- (6) block bootstrap ------------------------------------------------------------------------------------
    train_rows = np.flatnonzero(w > 0)
    codes, n_blocks, block_km, block_info = _s08_choose_blocks(mode, train_rows)
    rng = np.random.default_rng(CFG.SEED + 2024)
    eval_rows = (np.sort(rng.choice(fit_rows, size=_S08_EVAL_ROWS, replace=False))
                 if fit_rows.size > _S08_EVAL_ROWS else fit_rows.copy())
    eval_bin_idx = [_s08_bin_index(_S08_X[eval_rows, _S08_PRIMARY_IDX[gi]], edges[gi]) for gi in range(_S08_NG)]
    eval_counts = [np.bincount(ix[ix >= 0], minlength=len(edges[gi]) - 1) for gi, ix in enumerate(eval_bin_idx)]
    st.update({"train_rows": train_rows, "block_codes": codes, "n_blocks": n_blocks, "rounds": rounds,
               "eval_rows": eval_rows, "eval_bin_idx": eval_bin_idx, "eval_counts": eval_counts})
    # replicates replay the counterfactuals, which call FOCAL_RECOMPUTE of section 06 (cross-section code, D4)
    st["boot_key"] = CFG.key("XGB_PARAMS", version=_S08_VERSION, mode=mode, booster=st["booster_hash"],
                             code06=_s08_code_hash("06"),
                             df=_S08_DF_FP, rounds=rounds, blocks=_s08_hash(codes, train_rows),
                             eval=_s08_hash(eval_rows), edges=_s08_hash(*edges),
                             plans=_s08_hash([(p["zone"], p["feature"], round(p["step"], 9)) for p in plans]))
    n_boot = int(CFG.N_BOOTSTRAP)
    with timer(f"08_{mode}_bootstrap"):
        log(f"08 {mode}: block bootstrap {n_boot} replicates x {rounds} rounds, {n_blocks} blocks of "
            f"{block_km:.0f} km ({block_info['block_size_source']}), eval sample {eval_rows.size:,} rows")
        results = gpu_pool_map(_s08_bootstrap_replicate, [(mode, b) for b in range(n_boot)]) if n_boot else []
    ok = [r for r in results if r is not None and r.get("ok")]
    failed = [r for r in results if r is None or not r.get("ok")]
    if n_boot and not ok:
        raise RuntimeError(f"08 {mode}: all {n_boot} bootstrap replicates failed: "
                           f"{[r.get('error') for r in failed if r]}")
    boot_mean_abs = np.stack([r["mean_abs"] for r in ok]) if ok else np.zeros((0, _S08_NG))
    boot_curves = np.stack([r["curves"] for r in ok]) if ok else np.zeros((0, _S08_NG, st["max_bins"]))
    boot_thr = np.stack([r["thresholds"] for r in ok]) if ok else np.zeros((0, _S08_NG, 3))
    boot_rec = np.stack([r["rec_deltas"] for r in ok]) if ok else np.zeros((0, len(plans)))
    n_ok = len(ok)
    ci_label = "95% CI" if n_ok >= 50 else ("replicate range" if n_ok else "no bootstrap")
    boot_info = {
        "n": n_ok, "n_requested": n_boot, "mode": "block", "model_mode": mode, "block_size_km": float(block_km),
        "n_blocks": int(n_blocks), "rounds": rounds, **block_info, "ci_level": 0.95,
        "ci_method": "recentred-percentile",
        "ci_method_description": ("bias-corrected percentile interval: the 2.5/97.5 replicate percentiles shifted by "
                                  "(full-data estimate - replicate median); always contains the estimate"),
        "ci_label": ci_label, "replicate_range": bool(n_ok < 50), "eval_rows": int(eval_rows.size),
        "full_data": False, "rows_sampled": True, "devices": sorted({r["device"] for r in ok}),
        "mean_seconds": float(np.mean([r["seconds"] for r in ok])) if ok else None,
        "failures": [{"b": r.get("b"), "error": r.get("error")} for r in failed if r],
    }

    # ---- (7) importance, dependence, thresholds -----------------------------------------------------------------
    group = shp["group"]
    imp_full = np.abs(group[fit_rows].astype(np.float64)).mean(axis=0)
    imp_ref = np.abs(group[eval_rows].astype(np.float64)).mean(axis=0)
    imp_lo, imp_hi, _ = _s08_recentred_interval(imp_full, boot_mean_abs, lower=0.0)
    with _s08_warnings.catch_warnings():
        _s08_warnings.simplefilter("ignore", RuntimeWarning)
        imp_bias = (np.nanmedian(boot_mean_abs, axis=0) - imp_ref) if n_ok else np.full(_S08_NG, np.nan)
    by_frame = {}
    for yr in _S08_FRAMES:
        sl = _s08_frame_slice(yr)
        gv = group[sl].astype(np.float64)
        okr = np.isfinite(gv).all(axis=1)
        by_frame[yr] = {"mean_abs": np.abs(gv[okr]).mean(axis=0) if okr.any() else np.full(_S08_NG, np.nan),
                        "mean": gv[okr].mean(axis=0) if okr.any() else np.full(_S08_NG, np.nan),
                        "n": int(okr.sum())}
    by_zone = {}
    for z in range(_S08_N_ZONES):
        sel = fit_rows[zones_full[fit_rows] == z]
        gv = group[sel].astype(np.float64)
        by_zone[z] = {"mean_abs": np.abs(gv).mean(axis=0) if sel.size else np.full(_S08_NG, np.nan),
                      "mean": gv.mean(axis=0) if sel.size else np.full(_S08_NG, np.nan), "n": int(sel.size)}
    importance = {"groups": list(_S08_GROUPS),
                  "global": {"mean_abs": imp_full, "ci_lo": imp_lo, "ci_hi": imp_hi, "bias": imp_bias},
                  "by_frame": by_frame, "by_zone": by_zone, "order": [int(i) for i in np.argsort(imp_full)[::-1]],
                  "features": {"names": list(FEATURES), "mean_abs": shp["feature_mean_abs"],
                               "by_frame": shp["feature_mean_abs_by_frame"]},
                  "fit_frames": list(fit_frames), "n_bootstrap": n_ok, "ci_method": "recentred-percentile",
                  "ci_label": ci_label}
    dep_rng = np.random.default_rng(CFG.SEED + 11)
    dependence, thresholds = {}, {}
    for gi, g in enumerate(_S08_GROUPS):
        centers, mean, counts = curves[gi]
        nbins = len(centers)
        lo, hi, _ = _s08_recentred_interval(mean, boot_curves[:, gi, :nbins] if n_ok else np.zeros((0, nbins)))
        thresholds[g] = _s08_build_threshold(gi, points[gi], boot_thr)
        dependence[g] = {"primary": _S08_PRIMARY[gi], "bin_centers": centers, "mean": mean, "ci_lo": lo,
                         "ci_hi": hi, "count": counts.astype(np.int64),
                         "scatter": _s08_scatter(mode, gi, int(np.argmax(partner[gi])), dep_rng, fit_rows),
                         "threshold": thresholds[g]}
    bad = [g for g, t in thresholds.items() for k in ("zero_crossing", "breakpoint", "saturation")
           if t[k] is None and t[f"{k}_ci"] is not None]
    if bad:
        raise AssertionError(f"08 {mode}: threshold interval without a point estimate: {bad}")

    # ---- recommendations with bootstrap intervals --------------------------------------------------------------
    recs = {m_["id"]: [] for m_ in _S08_ZONE_META}
    for i, plan in enumerate(plans):
        delta = float(plan["expected_delta_c"])
        if not delta < -_S08_REC_MIN_COOLING_C:
            continue
        boot = boot_rec[:, i] if boot_rec.size else np.zeros(0)
        ci, bias = None, None
        if np.isfinite(boot).sum() >= 2:
            lo, hi, bb = _s08_recentred_interval(np.array([delta]), boot[:, None])
            ci, bias = [float(lo[0]), float(hi[0])], float(bb[0])
        recs[plan["zone"]].append({
            "feature": plan["feature"], "group": plan["group"], "action": plan["action"],
            "current": plan["current"], "target": plan["target"], "expected_delta_c": delta, "ci": ci,
            "priority": "high" if round(delta, 2) <= -1.0 else ("medium" if round(delta, 2) <= -0.3 else "low"),
            "rationale": _s08_rationale(plan, ci, ci_label),
            "kind": plan["kind"], "step": plan["step"], "n_cells": plan["n_cells"], "coupled": True,
            "bias": bias, "focal_recomputed": True, "distances_fixed": True, "frame": plan["year"],
            "bounds": {"p10": plan["p10"], "p90": plan["p90"], "bound": plan["bound"], "max_step": plan["cap"]},
            "spillover": {"n_cells": plan["spill_n_cells"], "mean_delta_c": plan["spill_delta_c"]},
            "mean_applied_step": plan["mean_applied"], "best_delta_c": plan["best_delta_c"],
        })
    recs = {z: sorted(v, key=lambda r: r["expected_delta_c"])[:_S08_REC_MAX_PER_ZONE] for z, v in recs.items()}

    profiles = _s08_zone_profiles(mode, recs)
    transitions = _s08_transitions(mode)
    nan_rows = ~np.isfinite(group).all(axis=1)
    payload = {
        "shap_base": shp["base"],
        "shap_group_values": {yr: group[_s08_frame_slice(yr)].copy() for yr in _S08_FRAMES},
        "pred_anom": {yr: shp["margin"][_s08_frame_slice(yr)].copy() for yr in _S08_FRAMES},
        "importance": importance, "interactions": inter, "dependence": {"groups": dependence},
        "thresholds": thresholds, "bootstrap_info": boot_info, "zone_model": zm,
        "zones_obs": {yr: zones_full[_s08_frame_slice(yr)].copy() for yr in _S08_FRAMES},
        "zone_profiles": profiles, "zone_transitions": transitions, "recommendations": recs,
        "diagnostics": {"additivity_feature": shp["additivity_feature"], "additivity_group": shp["additivity_group"],
                        "crosscheck_max_diff": shp["crosscheck_max_diff"], "n_nan_rows": int(nan_rows.sum()),
                        "fit_frames": list(fit_frames), "n_fit_rows": int(fit_rows.size),
                        "n_plans": len(plans)},
    }
    return payload

## 8.16 Run both modes
The per-mode payload is stage-cached: its key holds the final booster's bytes, a fingerprint of `DF` and the
block20 OOF predictions, and every setting that changes the result.

In [ ]:
def _s08_df_fingerprint():
    cols = ["anom_day", "anom_night", "w_day", "w_night", "row", "col"]
    parts = [np.round(np.nansum(_S08_X, axis=0, dtype=np.float64), 6), int(np.isfinite(_S08_X).sum())]
    parts += [float(np.nansum(DF[c].to_numpy(np.float64))) for c in cols]
    for m in CFG.MODES:
        oof = MODEL_RESULTS.get(m, {}).get("oof", {}).get("block20", {}).get("xgboost")
        parts.append(None if oof is None else float(np.nansum(np.asarray(oof, np.float64))))
    return _s08_hash(FEATURES, len(DF), *parts)


_S08_DF_FP = _s08_df_fingerprint()
XAI_FRAMES, SHAP_BASE, SHAP_GROUP_VALUES, PRED_ANOM, OOF_ANOM = {}, {}, {}, {}, {}
IMPORTANCE, INTERACTIONS, DEPENDENCE, THRESHOLDS, BOOTSTRAP_INFO = {}, {}, {}, {}, {}
ZONE_MODEL, ZONES_OBS, ZONE_PROFILES, ZONE_TRANSITIONS, RECOMMENDATIONS = {}, {}, {}, {}, {}
WEB_MODELS = {}
_S08_PAYLOADS = {}

for _s08_mode in CFG.MODES:
    if _s08_mode not in FINAL_MODELS:
        log(f"08: no final model for mode {_s08_mode}; its explanation layer is skipped", "WARNING")
        continue
    _s08_booster_hash = _s08_hash(bytes(FINAL_MODELS[_s08_mode].save_raw("ubj")))
    _s08_key = CFG.key("N_BOOTSTRAP", "DEP_BINS", "K_ZONES", "BOOT_BLOCK_KM", "INNER_BLOCK_KM", "SEED",
                       "SHAP_INTERACTION_ROWS_PER_FRAME", "NOW", version=_S08_VERSION, mode=_s08_mode,
                       booster=_s08_booster_hash, df=_S08_DF_FP, on_gpu=_S08_ON_GPU,
                       coupling=_s08_hash(repr(SCENARIO_COUPLING["coupling"])),
                       code06=_s08_code_hash("06"))       # counterfactuals call FOCAL_RECOMPUTE (section 06)
    _s08_sc = stage_cache(f"08_explain_{_s08_mode}", _s08_key)
    with timer(f"08_{_s08_mode}"):
        _S08_PAYLOADS[_s08_mode] = _s08_sc.get_or_compute(
            lambda mode=_s08_mode, booster_hash=_s08_booster_hash: _s08_run_mode(mode, booster_hash))
    _S08_STATE.pop(_s08_mode, None)         # per-mode working arrays (SHAP matrices, plan matrices) are released
    FINAL_MODELS[_s08_mode].set_param({"device": "cpu"})
    free_memory()

## 8.17 Publish the §3.5 globals
`SHAP_GROUP_VALUES[m][year]` and `PRED_ANOM[m][year]` are (N_CELLS × 18) / (N_CELLS) float32 in canonical cell
order; `OOF_ANOM[m][year]` is the block20 XGBoost out-of-fold anomaly (NaN where the row was not a training row of
the mode, e.g. night 1990–2000).

In [ ]:
for _s08_mode, _s08_p in _S08_PAYLOADS.items():
    XAI_FRAMES[_s08_mode] = list(_S08_FRAMES)
    SHAP_BASE[_s08_mode] = float(_s08_p["shap_base"])
    SHAP_GROUP_VALUES[_s08_mode] = {int(y): np.asarray(v, np.float32) for y, v in _s08_p["shap_group_values"].items()}
    PRED_ANOM[_s08_mode] = {int(y): np.asarray(v, np.float32) for y, v in _s08_p["pred_anom"].items()}
    _s08_oof = MODEL_RESULTS.get(_s08_mode, {}).get("oof", {}).get("block20", {}).get("xgboost")
    _s08_w = DF[f"w_{_s08_mode}"].to_numpy(np.float64)
    OOF_ANOM[_s08_mode] = {}
    for _y in _S08_FRAMES:
        _sl = _s08_frame_slice(_y)
        if _s08_oof is None or len(_s08_oof) != len(DF):
            OOF_ANOM[_s08_mode][_y] = np.full(_S08_NC, np.nan, np.float32)
        else:
            OOF_ANOM[_s08_mode][_y] = np.where(_s08_w[_sl] > 0, np.asarray(_s08_oof, np.float32)[_sl],
                                               np.nan).astype(np.float32)
    IMPORTANCE[_s08_mode] = _s08_p["importance"]
    INTERACTIONS[_s08_mode] = _s08_p["interactions"]
    DEPENDENCE[_s08_mode] = _s08_p["dependence"]
    THRESHOLDS[_s08_mode] = _s08_p["thresholds"]
    BOOTSTRAP_INFO[_s08_mode] = _s08_p["bootstrap_info"]
    ZONE_MODEL[_s08_mode] = _s08_p["zone_model"]
    ZONES_OBS[_s08_mode] = {int(y): np.asarray(v, np.uint8) for y, v in _s08_p["zones_obs"].items()}
    ZONE_PROFILES[_s08_mode] = _s08_p["zone_profiles"]
    ZONE_TRANSITIONS[_s08_mode] = _s08_p["zone_transitions"]
    RECOMMENDATIONS[_s08_mode] = _s08_p["recommendations"]
# Zone ids must be reproducible from the published SHAP values with the published centroids.
for _s08_mode in ZONES_OBS:
    for _y in _S08_FRAMES:
        if not np.array_equal(ASSIGN_ZONES(_s08_mode, SHAP_GROUP_VALUES[_s08_mode][_y]), ZONES_OBS[_s08_mode][_y]):
            raise AssertionError(f"08: ZONES_OBS[{_s08_mode}][{_y}] differs from ASSIGN_ZONES of its SHAP values")
for _s08_mode in IMPORTANCE:
    _imp = IMPORTANCE[_s08_mode]["global"]
    _info = BOOTSTRAP_INFO[_s08_mode]
    print(f"\n=== {_s08_mode.upper()} === SHAP base {SHAP_BASE[_s08_mode]:+.3f} °C; bootstrap {_info['n']}/"
          f"{_info['n_requested']} replicates ({_info['ci_label']}), {_info['n_blocks']} blocks of "
          f"{_info['block_size_km']:.0f} km")
    for _r, _gi in enumerate(IMPORTANCE[_s08_mode]["order"][:10], start=1):
        print(f"  {_r:2d}. {_S08_GROUPS[_gi]:<17s} mean|SHAP| {_imp['mean_abs'][_gi]:.3f} °C "
              f"[{_imp['ci_lo'][_gi]:.3f}, {_imp['ci_hi'][_gi]:.3f}] bias {_imp['bias'][_gi]:+.3f}")
    print("  zone labelling: " + ZONE_MODEL[_s08_mode]["labelling"]["rule"])
    for _c in ZONE_MODEL[_s08_mode]["labelling"]["clusters"]:
        print(f"    {_c['zone']} {_c['name']:<22s} <- cluster {_c['raw_cluster']}: n={_c['n']:,}, mean total SHAP "
              f"{_c['mean_total_shap']:+.3f}, mean frac_water {_c['mean_frac_water']:.3f}, rows with water >= 0.2 "
              f"{_c['share_water_ge_0_2']:.1%}, water |SHAP| rank {_c['water_group_rank']} [{_c['reason']}]")
    _lab = ZONE_MODEL[_s08_mode]["labelling"]
    if not _lab["riparian_criterion_met"]:
        log(f"08 {_s08_mode}: no cluster meets the riparian criterion ({_lab['riparian_criterion']}); zone 1 is shown "
            f"as '{_S08_NO_RIPARIAN_NAME}'", "WARNING")
    for _p in ZONE_PROFILES[_s08_mode]:
        print(f"  [{_p['id']}] {_p['description']}")
        for _rec in _p["recommendations"]:
            print(f"      - {_rec['priority'].upper():6s} {_rec['rationale']}")
        if not _p["recommendations"]:
            print("      - no feasible action with an expected cooling > 0.05 °C")

## 8.18 Web surrogates (`WEB_MODELS[m]`, v1 compact format + focal spec)
Format `lst-xgb-compact-v1` (SPEC_v1 §4.3): node 0 is the root; node `i` is a leaf iff `l[i] == -1` (value `v[i]`);
otherwise go to `l[i]` if `x[f[i]] < t[i]`, to `r[i]` if `x[f[i]] >= t[i]`, and to `l[i]` if `d[i] == 1` else `r[i]`
when the value is missing. Prediction (anomaly °C) = `base_score + Σ leaves`; absolute °C = prediction + the frame
level (section 09). Thresholds are exact, finite float32 values (`_s08_threshold`). The surrogate is distilled
from the final model on every valid ROI cell × frame with the same monotone constraints, then shrunk (fewer
trees) only if the JSON exceeds 6 MB.

In [ ]:
_S08_WEB_MAX_BYTES = 6 * 1024 * 1024
_S08_WEB_ROUNDS = 400
_S08_WEB_DEPTH = 6


def _s08_parse_base_score(raw):
    """learner_model_param.base_score is '4.1E1' (XGBoost 2.x) or '[4.1E1]' (3.x vector form)."""
    return float(str(raw).strip().strip("[]").split(",")[0].strip())


def _s08_json_list(arr):
    a = np.asarray(arr, dtype=np.float64)
    out = a.tolist()
    for i in np.flatnonzero(~np.isfinite(a)):
        out[i] = None
    return out


def _s08_json_dumps(obj):
    import json as _json

    return _json.dumps(obj, separators=(",", ":"), allow_nan=False).encode("utf-8")


def _s08_threshold(t):
    """A split threshold as a finite float32 value (exact, as the format requires).

    XGBoost 3.4 writes -inf for a split that only separates missing values: every present value goes right (nothing
    is below -inf), missing values take the default side (3.2 used the feature's minimum there). JSON has no infinity,
    and +-FLT_MAX routes every finite float32 feature value identically.
    """
    limit = np.finfo(np.float32).max
    return float(np.clip(np.float32(t), -limit, limit))


def _s08_booster_to_compact(booster, mode, fidelity, check):
    """Convert a gbtree booster's JSON dump into the lst-xgb-compact-v1 dict (+ v2 keys mode/focal/base_features)."""
    import json as _json

    raw = _json.loads(bytes(booster.save_raw(raw_format="json")).decode("utf-8"))
    learner = raw["learner"]
    gb = learner["gradient_booster"]
    if gb.get("name") != "gbtree":
        raise ValueError(f"compact export supports gbtree only, got {gb.get('name')}")
    trees = []
    for tree in gb["model"]["trees"]:
        left = [int(v) for v in tree["left_children"]]
        right = [int(v) for v in tree["right_children"]]
        leaf = [lc == -1 for lc in left]
        cond = tree["split_conditions"]
        trees.append({
            "f": [-1 if is_leaf else int(fi) for fi, is_leaf in zip(tree["split_indices"], leaf)],
            "t": [0 if is_leaf else _s08_threshold(t) for t, is_leaf in zip(cond, leaf)],
            "l": left, "r": right,
            "d": [0 if is_leaf else int(bool(dl)) for dl, is_leaf in zip(tree["default_left"], leaf)],
            "v": [float(f"{float(np.float32(v)):.9g}") if is_leaf else 0 for v, is_leaf in zip(cond, leaf)],
        })
    return {
        "format": "lst-xgb-compact-v1",
        "objective": str(learner.get("objective", {}).get("name", "reg:squarederror")),
        "target_mode": "anomaly", "mode": mode,
        "features": list(FEATURES), "base_features": list(BASE_FEATURES),
        "focal": {"bases": list(FOCAL_SPEC["bases"]), "windows": [int(w) for w in FOCAL_SPEC["windows"]]},
        "base_score": _s08_parse_base_score(learner["learner_model_param"]["base_score"]),
        "n_trees": len(trees), "trees": trees, "fidelity": fidelity, "check": check,
    }


def _s08_compile_compact(model):
    return [tuple(np.asarray(t[k], dtype=dt) for k, dt in
                  (("f", np.int64), ("t", np.float64), ("l", np.int64), ("r", np.int64), ("d", np.int64),
                   ("v", np.float64))) for t in model["trees"]]


def _s08_predict_compact(model, X):
    """Pure-NumPy evaluator of lst-xgb-compact-v1 (NaN/None = missing), independent of XGBoost."""
    X = np.array(X, dtype=np.float64)
    Xf = X.astype(np.float32).astype(np.float64)        # XGBoost compares float32 features with float32 thresholds
    out = np.full(X.shape[0], float(model["base_score"]), dtype=np.float64)
    for f, t, left, right, dflt, v in _s08_compile_compact(model):
        node = np.zeros(X.shape[0], dtype=np.int64)
        while True:
            active = np.flatnonzero(left[node] != -1)
            if active.size == 0:
                break
            nd = node[active]
            xv = Xf[active, f[nd]]
            go_left = np.where(np.isnan(xv), dflt[nd] == 1, xv < t[nd])
            node[active] = np.where(go_left, left[nd], right[nd])
        out += v[node]
    return out


def _s08_compact_depth(model):
    """Maximum depth (edges from the root) over the compact trees."""
    worst = 0
    for t in model["trees"]:
        depth = {0: 0}
        for i, (lc, rc) in enumerate(zip(t["l"], t["r"])):
            if lc != -1:
                depth[lc] = depth[i] + 1
                depth[rc] = depth[i] + 1
        worst = max(worst, max(depth.values()))
    return worst


def _s08_r2(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, np.float64), np.asarray(y_pred, np.float64)
    ok = np.isfinite(y_true) & np.isfinite(y_pred)
    sst = np.sum((y_true[ok] - y_true[ok].mean()) ** 2)
    return float(1.0 - np.sum((y_true[ok] - y_pred[ok]) ** 2) / sst) if sst > 0 else None


def _s08_with_nans(X, rng, frac=0.1):
    Xn = np.array(X, dtype=np.float32, copy=True)
    Xn[rng.random(Xn.shape) < frac] = np.nan
    return Xn


_S08_SPECTRAL_GAP = ["ndvi", "ndbi", "mndwi", "albedo", "ndvi_spring", "ndvi_drop"]   # a Landsat gap of the cell
_S08_WEB_GAP_SHARE = 0.15          # spectral-gap copies, share of the complete rows
_S08_WEB_RANDNAN_SHARE = 0.05      # copies with 10 % random NaNs, share of the complete rows
_S08_WEB_EVAL_ROWS = 5000          # held-out complete rows for the gap / random-NaN fidelity (never copied)


def _s08_final_margin(mode, X):
    """Final-model anomaly prediction (the distillation label) of arbitrary rows, on the CPU."""
    booster = FINAL_MODELS[mode]
    booster.set_param({"device": "cpu"})
    return np.asarray(booster.inplace_predict(np.ascontiguousarray(X, dtype=np.float32)), dtype=np.float64)


def _s08_rmse(a, b):
    a, b = np.asarray(a, np.float64), np.asarray(b, np.float64)
    ok = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[ok] - b[ok]) ** 2))) if ok.any() else None


def _s08_build_web_model(mode):
    """Distilled depth-6 surrogate of FINAL_MODELS[mode] in compact form, verified against XGBoost.

    Training rows: every ROI cell x frame (the final model predicts every row, missing values included) plus
    NaN-augmented copies of complete rows - the six per-cell Landsat features set to NaN (a spectral gap whose
    neighbourhood means survive) and 10 % random NaNs - all labelled by the final model on the masked rows, so the
    surrogate learns the final model's missing-value routes. Fidelity is reported on complete rows, actual NaN rows
    and held-out spectral-gap / random-NaN copies.
    """
    rng = np.random.default_rng(CFG.SEED + 404 + (0 if mode == "day" else 1))
    X_all = _S08_X
    final_all = np.concatenate([PRED_ANOM[mode][y] for y in _S08_FRAMES]).astype(np.float64)
    w = DF[f"w_{mode}"].to_numpy(np.float64)
    y_obs = DF[f"anom_{mode}"].to_numpy(np.float64)
    complete = np.isfinite(X_all).all(axis=1)
    comp_idx = rng.permutation(np.flatnonzero(complete))
    n_eval = min(_S08_WEB_EVAL_ROWS, comp_idx.size // 5)
    eval_idx, pool = np.sort(comp_idx[:n_eval]), comp_idx[n_eval:]
    n_gap = int(round(_S08_WEB_GAP_SHARE * comp_idx.size))
    n_rnd = int(round(_S08_WEB_RANDNAN_SHARE * comp_idx.size))
    gap_cols = [FEATURES.index(f) for f in _S08_SPECTRAL_GAP]

    def spectral_gap(rows):
        Xg = np.array(X_all[rows], dtype=np.float32, copy=True)
        Xg[:, gap_cols] = np.nan
        return Xg

    X_gap = spectral_gap(np.sort(pool[:n_gap]))
    X_rnd = _s08_with_nans(X_all[np.sort(pool[n_gap:n_gap + n_rnd])], rng)
    X_train = np.ascontiguousarray(np.concatenate([X_all, X_gap, X_rnd]), dtype=np.float32)
    y_train = np.concatenate([final_all] + [_s08_final_margin(mode, Xa) for Xa in (X_gap, X_rnd) if len(Xa)])
    del X_gap, X_rnd
    # learning rate 0.2: the label is a noise-free function of the features, and with the NaN-augmented copies the
    # 400-tree budget needs the larger step (synthetic 2 km: complete-row R2 vs final 0.998 in both modes)
    params = {"objective": "reg:squarederror", "max_depth": _S08_WEB_DEPTH, "learning_rate": 0.2,
              "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 5, "tree_method": "hist",
              "max_bin": 256, "seed": int(CFG.SEED), "verbosity": 0,
              "monotone_constraints": _s08_mono_string(mode)}
    full = None
    for device in dict.fromkeys([xgb_device_for(0), "cpu"]):
        try:
            dtrain = xgb.QuantileDMatrix(X_train, label=y_train.astype(np.float32), max_bin=256)
            full = xgb.train({**params, "device": device}, dtrain, num_boost_round=_S08_WEB_ROUNDS)
            del dtrain
            break
        except Exception as exc:  # noqa: BLE001
            if device == "cpu" or not is_gpu_oom_error(exc):
                raise
            log(f"08: GPU OOM training the {mode} surrogate; retrying on the CPU", "WARNING")
            free_memory()
    n_train_rows = int(X_train.shape[0])
    del X_train, y_train
    full.set_param({"device": "cpu"})
    if eval_idx.size == 0:                 # no complete row to hold out: fidelity on the gap copies is undefined
        eval_idx = np.flatnonzero(complete)[:0]
    X_eval_gap = spectral_gap(eval_idx)
    X_eval_rnd = _s08_with_nans(X_all[eval_idx], rng)
    final_gap = _s08_final_margin(mode, X_eval_gap) if eval_idx.size else np.zeros(0)
    final_rnd = _s08_final_margin(mode, X_eval_rnd) if eval_idx.size else np.zeros(0)
    test_rows = rng.choice(X_all.shape[0], size=min(max(2000, min(5000, X_all.shape[0])), X_all.shape[0]),
                           replace=False)
    X_test = _s08_with_nans(X_all[test_rows], rng)
    check_X = _s08_with_nans(X_all[rng.choice(X_all.shape[0], size=min(50, X_all.shape[0]), replace=False)], rng,
                             frac=0.15)
    n_trees = full.num_boosted_rounds()
    trn = w > 0
    nan_rows = ~complete
    while True:
        booster = full[:n_trees] if n_trees < full.num_boosted_rounds() else full
        sur_all = booster.inplace_predict(X_all).astype(np.float64)
        sur_gap = booster.inplace_predict(X_eval_gap).astype(np.float64) if eval_idx.size else np.zeros(0)
        sur_rnd = booster.inplace_predict(X_eval_rnd).astype(np.float64) if eval_idx.size else np.zeros(0)
        fidelity = {"r2_vs_final": _s08_r2(final_all, sur_all),
                    "rmse_vs_final": _s08_rmse(final_all, sur_all),
                    "r2_vs_obs": _s08_r2(y_obs[trn], sur_all[trn]) if trn.any() else None,
                    "max_depth": _S08_WEB_DEPTH, "rounds": int(n_trees), "n_rows": int(X_all.shape[0]),
                    "n_train_rows": n_train_rows,
                    "r2_vs_final_complete": _s08_r2(final_all[complete], sur_all[complete]),
                    "rmse_vs_final_complete": _s08_rmse(final_all[complete], sur_all[complete]),
                    "rmse_vs_final_nan_rows": _s08_rmse(final_all[nan_rows], sur_all[nan_rows]) if nan_rows.any()
                    else None,
                    "n_nan_rows": int(nan_rows.sum()),
                    "rmse_vs_final_spectral_gap": _s08_rmse(final_gap, sur_gap),
                    "rmse_vs_final_random_nan": _s08_rmse(final_rnd, sur_rnd),
                    "n_eval_rows": int(eval_idx.size),
                    "augmentation": {"spectral_gap_features": list(_S08_SPECTRAL_GAP), "spectral_gap_rows": n_gap,
                                     "random_nan_rows": n_rnd, "random_nan_frac": 0.1},
                    "label": ("final-model anomaly prediction (distillation on all ROI cells x frames plus "
                              "NaN-augmented copies of complete rows)"),
                    "monotone": {f: int(v) for f, v in _S08_MONOTONE.get(mode, {}).items()}}
        check = {"rows": [_s08_json_list(r) for r in check_X],
                 "expected": booster.predict(xgb.DMatrix(check_X, missing=np.nan),
                                             output_margin=True).astype(np.float64).tolist()}
        model = _s08_booster_to_compact(booster, mode, fidelity, check)
        size = len(_s08_json_dumps(model))
        if size <= _S08_WEB_MAX_BYTES or n_trees <= 20:
            break
        n_trees = max(20, int(n_trees * _S08_WEB_MAX_BYTES / size * 0.95))
        log(f"08: model_web_{mode}.json would be {size / 1e6:.1f} MB; truncating the surrogate to {n_trees} trees")
    ref = booster.predict(xgb.DMatrix(X_test, missing=np.nan), output_margin=True).astype(np.float64)
    diff = float(np.max(np.abs(_s08_predict_compact(model, X_test) - ref)))
    if not diff < 1e-3:
        raise RuntimeError(f"08: compact {mode} model evaluator disagrees with XGBoost: max |diff| = {diff:.3e}")
    check_rows = np.array([[np.nan if v is None else v for v in r] for r in check["rows"]], dtype=np.float64)
    check_diff = float(np.max(np.abs(_s08_predict_compact(model, check_rows) - np.array(check["expected"]))))
    if not check_diff < 1e-3:
        raise RuntimeError(f"08: compact {mode} model check rows disagree: max |diff| = {check_diff:.3e}")
    depth = _s08_compact_depth(model)
    if depth > _S08_WEB_DEPTH or model["n_trees"] > _S08_WEB_ROUNDS:
        raise RuntimeError(f"08: surrogate exceeds depth {_S08_WEB_DEPTH} / {_S08_WEB_ROUNDS} trees")
    model["fidelity"]["evaluator_max_abs_diff"] = diff
    model["fidelity"]["check_max_abs_diff"] = check_diff
    model["fidelity"]["bytes"] = int(size)
    fd = model["fidelity"]
    f3 = lambda v: "n/a" if v is None else f"{v:.3f}"  # noqa: E731
    log(f"08 {mode}: surrogate {model['n_trees']} trees (depth {depth}), {size / 1e6:.2f} MB; evaluator vs XGBoost "
        f"max diff {diff:.1e} on {len(X_test):,} rows with NaNs; R2 vs final {f3(fd['r2_vs_final'])} (complete "
        f"rows {f3(fd['r2_vs_final_complete'])}); RMSE vs final: all {f3(fd['rmse_vs_final'])}, NaN rows "
        f"{f3(fd['rmse_vs_final_nan_rows'])} (n={fd['n_nan_rows']:,}), spectral gap "
        f"{f3(fd['rmse_vs_final_spectral_gap'])}, 10% random NaN {f3(fd['rmse_vs_final_random_nan'])} °C; R2 vs obs "
        f"{fd['r2_vs_obs']}")
    del full, booster
    return model


for _s08_mode in _S08_PAYLOADS:
    _s08_wkey = CFG.key(version=_S08_VERSION, mode=_s08_mode, df=_S08_DF_FP, rounds=_S08_WEB_ROUNDS,
                        booster=_s08_hash(bytes(FINAL_MODELS[_s08_mode].save_raw("ubj"))))
    with timer(f"08_{_s08_mode}_web_model"):
        WEB_MODELS[_s08_mode] = stage_cache(f"08_web_{_s08_mode}", _s08_wkey).get_or_compute(
            lambda mode=_s08_mode: _s08_build_web_model(mode))
    free_memory()

## 8.19 Synthetic mode: planted vs recovered
The synthetic generator (section 02) plants every driver response in `SYNTHETIC_TRUTH["planted_thresholds"]`
(per mode and group: kind, value, direction). The table compares them with what the grouped SHAP pipeline recovered
from the final model: saturation points, onsets (hinge breakpoints), linear slopes of the binned group curve and the
effect direction. Groups the generator never used should rank low, and the expected day/night sign flips (built-up,
water) should appear (`docs/research/methods_v2.md` §5).

Reading the table:
* **`linear (local contrast)`** rows: the slope of the group SHAP on `x − x_m15` with `x_m15` as a co-regressor.
  The 15-cell neighbourhood member of a group carries *regional context* (the model has no coordinates, so a smooth
  focal mean also absorbs the regional gradients of other drivers); the whole-curve slope can therefore differ in
  sign from the planted *local* per-pixel response, while the local-contrast slope isolates it. Magnitudes are
  attenuated (shared variance with correlated groups goes to whichever group the trees split on).
* **`kind (hinge)`**: a planted saturation that appears as a kink of the group curve because another member keeps
  acting beyond it (day NDVI: per-pixel saturation plus park cooling via the neighbourhood NDVI).
* **`share_above_planted`**: fraction of rows whose primary value lies above the planted threshold. A threshold
  cannot be recovered from quantile-binned curves when few rows lie beyond it (night built-up saturation at 0.69:
  ~0 % of cells), so `NaN` there is a support limit, not a failure.

In [ ]:
def _s08_local_slope(mode, group, primary):
    """Local-contrast slope of a group's SHAP (°C per unit of the primary) or None.

    OLS of group SHAP on ``[1, x - x_m15, x_m15]`` pooled over the XAI frames. The 15-cell neighbourhood mean is a
    member of most groups and acts as *regional context* (with no coordinates in the model it absorbs the regional
    gradients of other drivers, e.g. the bright, dry, hot south-west for ``albedo_m15``), so the slope of the whole
    group curve can have the opposite sign of the planted *local* response. The coefficient of ``x - x_m15`` isolates
    the local response and is what the planted per-pixel slope should be compared with.
    """
    m15 = f"{primary}_m15"
    if m15 not in FEATURES or primary not in FEATURES:
        return None
    gi = _S08_GROUPS.index(group)
    years = DF["year"].to_numpy()
    xs, xm, sv = [], [], []
    for y in XAI_FRAMES.get(mode, []):
        sel = years == int(y)
        xs.append(DF.loc[sel, primary].to_numpy(np.float64))
        xm.append(DF.loc[sel, m15].to_numpy(np.float64))
        sv.append(np.asarray(SHAP_GROUP_VALUES[mode][int(y)][:, gi], np.float64))
    if not xs:
        return None
    x, m, sh = np.concatenate(xs), np.concatenate(xm), np.concatenate(sv)
    ok = np.isfinite(x) & np.isfinite(m) & np.isfinite(sh)
    if ok.sum() < 100 or np.ptp(x[ok] - m[ok]) <= 0:
        return None
    A = np.column_stack([np.ones(int(ok.sum())), x[ok] - m[ok], m[ok]])
    coef, *_ = np.linalg.lstsq(A, sh[ok], rcond=None)
    return float(coef[1])


def _s08_support_above(mode, group, value):
    """Share of dependence-curve rows with a primary value above ``value`` (support of a planted threshold)."""
    dep = globals().get("DEPENDENCE")
    d = ((dep.get(mode) or {}).get("groups") or {}).get(group) if isinstance(dep, dict) else None
    if not d or value is None:
        return None
    c = np.asarray(d["bin_centers"], np.float64)
    n = np.asarray(d["count"], np.float64)
    tot = float(n[np.isfinite(c)].sum())
    return float(n[np.isfinite(c) & (c > float(value))].sum() / tot) if tot > 0 else None


def _s08_planted_rows(mode, truth):
    rows = []
    planted = (truth.get("planted_thresholds") or {}).get(mode) or {}
    for group, spec in planted.items():
        if group not in THRESHOLDS.get(mode, {}):
            continue
        th = THRESHOLDS[mode][group]
        kind = str(spec.get("kind"))
        base = {"mode": mode, "group": group, "primary": spec.get("primary"), "kind": kind,
                "planted_direction": spec.get("direction"), "recovered_direction": th["direction"],
                "direction_match": th["direction"] == spec.get("direction")}
        if kind == "saturation":
            rec, ci, sup, est = th["saturation"], th["saturation_ci"], th["saturation_support"], "saturation"
            # A planted saturation of one member can appear as a *kink* of the group curve when another member keeps
            # acting beyond it (day NDVI: per-pixel saturation at 0.30 plus park cooling via the neighbourhood NDVI).
            # Fall back to the hinge breakpoint when the slope after it keeps the sign but at most halves.
            sb, sa = th.get("slope_before"), th.get("slope_after")
            if (rec is None and th["breakpoint"] is not None and sb is not None and sa is not None
                    and sb * sa >= 0 and abs(sa) <= 0.5 * abs(sb)):
                rec, ci, sup, est = th["breakpoint"], th["breakpoint_ci"], th["breakpoint_support"], "kink (hinge)"
            rows.append({**base, "planted": spec.get("value"), "recovered": rec, "ci": ci,
                         "support": sup, "estimator": est,
                         "abs_error": None if rec is None else abs(rec - float(spec["value"])),
                         "share_above_planted": _s08_support_above(mode, group, spec.get("value"))})
        elif kind == "onset":
            rec = th["breakpoint"]
            rows.append({**base, "planted": spec.get("value"), "recovered": rec, "ci": th["breakpoint_ci"],
                         "support": th["breakpoint_support"], "estimator": "breakpoint",
                         "abs_error": None if rec is None else abs(rec - float(spec["value"]))})
            if spec.get("saturation") is not None:
                rec2 = th["saturation"]
                rows.append({**base, "kind": "onset/saturation", "planted": spec["saturation"], "recovered": rec2,
                             "ci": th["saturation_ci"], "support": th["saturation_support"],
                             "estimator": "saturation",
                             "abs_error": None if rec2 is None else abs(rec2 - float(spec["saturation"]))})
        else:
            slope_key = next((k for k in ("slope_c_per_unit", "slope_c_per_m", "slope_c_per_log1p") if k in spec),
                             None)
            rec = th.get("slope")
            planted_v = spec.get(slope_key) if slope_key else None
            rows.append({**base, "planted": planted_v, "recovered": rec, "ci": None, "support": None,
                         "estimator": "binned-curve slope",
                         "abs_error": None if (rec is None or planted_v is None) else abs(rec - float(planted_v))})
            local = _s08_local_slope(mode, group, spec.get("primary")) if kind == "linear" else None
            if local is not None:
                rows.append({**base, "kind": "linear (local contrast)", "planted": planted_v, "recovered": local,
                             "recovered_direction": "warming" if local > 0 else "cooling",
                             "direction_match": (local > 0) == (spec.get("direction") == "warming"),
                             "ci": None, "support": None, "estimator": "SHAP ~ (x - x_m15) + x_m15",
                             "abs_error": None if planted_v is None else abs(local - float(planted_v))})
    return rows


def _s08_planted_report(truth):
    """Print the planted-vs-recovered tables; returns the row list (also stored in PLANTED_VS_RECOVERED_08)."""
    rows = [r for m in IMPORTANCE for r in _s08_planted_rows(m, truth)]
    if rows:
        tab = pd.DataFrame(rows)
        for col in ("planted", "recovered", "abs_error", "share_above_planted"):
            if col not in tab:
                tab[col] = None
            tab[col] = tab[col].map(lambda v: None if v is None else round(float(v), 4))
        tab["ci"] = tab["ci"].map(lambda v: None if v is None else [round(float(v[0]), 3), round(float(v[1]), 3)])
        print("Planted (synthetic generator) vs recovered (grouped SHAP thresholds):")
        print(tab.to_string(index=False))
        print(f"direction matches: {int(tab['direction_match'].sum())}/{len(tab)}")
    irrelevant = truth.get("irrelevant_by_construction") or {}
    effects = truth.get("planted_mean_abs_effect_c") or {}
    for mode in IMPORTANCE:
        order = IMPORTANCE[mode]["order"]
        ranks = {_S08_GROUPS[gi]: r for r, gi in enumerate(order, start=1)}
        irr = [g for g in irrelevant.get(mode, []) if g in ranks]
        if irr:
            print(f"{mode}: rank (of 18) of groups irrelevant by construction: "
                  + ", ".join(f"{g}={ranks[g]}" for g in sorted(irr, key=ranks.get)))
        eff = {g: v for g, v in (effects.get(mode) or {}).items() if g in ranks}
        if len(eff) >= 3:
            from scipy.stats import spearmanr

            gs = sorted(eff)
            mean_abs = IMPORTANCE[mode]["global"]["mean_abs"]
            rho = spearmanr([eff[g] for g in gs], [mean_abs[_S08_GROUPS.index(g)] for g in gs]).correlation
            print(f"{mode}: Spearman rho(planted mean |effect|, recovered mean |SHAP|) over {len(gs)} planted groups "
                  f"= {rho:.2f}")
    expect = truth.get("monotone_expectations") or {}
    for mode, groups in expect.items():
        if mode not in THRESHOLDS:
            continue
        for g, sign in groups.items():
            d = THRESHOLDS[mode][g]["direction"]
            want = "warming" if sign > 0 else "cooling"
            print(f"{mode}: expected {g} {want}, recovered {d} -> {'OK' if d == want else 'MISMATCH'}")
    if "day" in THRESHOLDS and "night" in THRESHOLDS:
        for g in ("built", "water"):
            print(f"sign flip {g}: day {THRESHOLDS['day'][g]['direction']}, night {THRESHOLDS['night'][g]['direction']}")
    return rows


PLANTED_VS_RECOVERED_08 = (_s08_planted_report(SYNTHETIC_TRUTH)
                           if isinstance(globals().get("SYNTHETIC_TRUTH"), dict) and SYNTHETIC_TRUTH else [])

## 8.20 Figures

In [ ]:
_S08_FIG_DIR = Path(CFG.FIG_DIR)
_S08_FIG_DIR.mkdir(parents=True, exist_ok=True)


def _s08_save_fig(fig, name):
    fig.savefig(_S08_FIG_DIR / name, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def _s08_raster(values):
    grid = np.full(GRID.shape, np.nan)
    grid[CELL_ROWS, CELL_COLS] = np.asarray(values, dtype=np.float64)
    return grid


def _s08_fig_importance():
    modes = list(IMPORTANCE)
    fig, axes = plt.subplots(1, len(modes), figsize=(7 * len(modes), 6.5), squeeze=False)
    for ax, mode in zip(axes[0], modes):
        imp = IMPORTANCE[mode]["global"]
        order = IMPORTANCE[mode]["order"][::-1]
        vals = np.asarray(imp["mean_abs"])[order]
        lo = np.clip(np.nan_to_num(vals - np.asarray(imp["ci_lo"])[order]), 0, None)
        hi = np.clip(np.nan_to_num(np.asarray(imp["ci_hi"])[order] - vals), 0, None)
        ax.barh(range(_S08_NG), vals, color="#f59e0b" if mode == "day" else "#6366f1", height=0.6,
                xerr=np.vstack([lo, hi]), ecolor="#334155", capsize=2)
        ax.set_yticks(range(_S08_NG))
        ax.set_yticklabels([GROUP_META[_S08_GROUPS[i]]["label"] for i in order], fontsize=8)
        ax.set_xlabel(f"mean |group SHAP| (°C); whiskers {BOOTSTRAP_INFO[mode]['ci_label']}")
        ax.set_title(f"{mode}: grouped SHAP importance ({BOOTSTRAP_INFO[mode]['n']} replicates)")
    _s08_save_fig(fig, "08_importance_groups.png")


def _s08_fig_dependence(mode):
    dep = DEPENDENCE[mode]["groups"]
    top = IMPORTANCE[mode]["order"][:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
    styles = {"zero_crossing": (":", "zero crossing"), "breakpoint": ("--", "breakpoint"),
              "saturation": ("-.", "saturation")}
    for ax, gi in zip(axes.ravel(), top):
        g = _S08_GROUPS[gi]
        d = dep[g]
        sc = d["scatter"]
        ax.scatter(sc["x"], sc["shap"], s=4, c=sc["color"], cmap="viridis", alpha=0.5, linewidths=0)
        ax.fill_between(d["bin_centers"], d["ci_lo"], d["ci_hi"], color="#a78bfa", alpha=0.3)
        ax.plot(d["bin_centers"], d["mean"], color="#6d28d9", lw=2)
        for key, (ls, name) in styles.items():
            if d["threshold"][key] is not None:
                ax.axvline(d["threshold"][key], color="#0f172a", ls=ls, lw=1, label=name)
        ax.axhline(0, color="#475569", lw=0.6)
        ax.set_title(f"{GROUP_META[g]['label']} ({d['threshold']['direction']})", fontsize=10)
        ax.set_xlabel(f"{d['primary']} (colour: {sc['color_feature']})", fontsize=8)
        ax.set_ylabel("group SHAP (°C)")
        if any(d["threshold"][k] is not None for k in styles):
            ax.legend(fontsize=7, frameon=False)
    fig.suptitle(f"{mode}: dependence of the six most important groups (band = {BOOTSTRAP_INFO[mode]['ci_label']})")
    fig.tight_layout()
    _s08_save_fig(fig, f"08_dependence_{mode}.png")


def _s08_fig_zones():
    from matplotlib.colors import ListedColormap
    from matplotlib.patches import Patch

    modes = list(ZONES_OBS)
    cmap = ListedColormap([m["color"] for m in _S08_ZONE_META])
    fig, axes = plt.subplots(1, len(modes), figsize=(6.5 * len(modes), 7), squeeze=False)
    for ax, mode in zip(axes[0], modes):
        z = ZONES_OBS[mode][_S08_NOW].astype(np.float64)
        z[z >= _S08_N_ZONES] = np.nan
        ax.imshow(_s08_raster(z), cmap=cmap, vmin=-0.5, vmax=3.5, interpolation="nearest")
        ax.set_axis_off()
        ax.set_title(f"{mode} zones {_S08_NOW} (silhouette {ZONE_MODEL[mode]['silhouette']:.2f})")
    axes[0][0].legend(handles=[Patch(color=m["color"], label=f"{m['id']} {m['name']}") for m in _S08_ZONE_META],
                      loc="lower left", fontsize=8)
    _s08_save_fig(fig, "08_zones_map.png")


def _s08_fig_signatures():
    modes = list(ZONE_PROFILES)
    fig, axes = plt.subplots(len(modes), 1, figsize=(13, 2.6 * len(modes) + 1), squeeze=False)
    for ax, mode in zip(axes[:, 0], modes):
        mat = np.array([[p["shap_means"][g] for g in _S08_GROUPS] for p in ZONE_PROFILES[mode]])
        vmax = float(np.nanmax(np.abs(mat))) or 1.0
        im = ax.imshow(mat, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
        for (r, c), v in np.ndenumerate(mat):
            ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=6,
                    color="white" if abs(v) > 0.6 * vmax else "#0f172a")
        ax.set_xticks(range(_S08_NG))
        ax.set_xticklabels(_S08_GROUPS, rotation=60, ha="right", fontsize=7)
        ax.set_yticks(range(_S08_N_ZONES))
        ax.set_yticklabels([p["name"] for p in ZONE_PROFILES[mode]], fontsize=8)
        ax.set_title(f"{mode}: zone SHAP signatures (mean group SHAP, °C)")
        fig.colorbar(im, ax=ax)
    fig.tight_layout()
    _s08_save_fig(fig, "08_zone_signatures.png")


def _s08_fig_diagnostics():
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    for mode in ZONE_MODEL:
        dg = ZONE_MODEL[mode]["diagnostics"]
        axes[0].plot(dg["k"], dg["inertia"], marker="o", lw=2, label=mode)
        axes[1].plot(dg["k"], dg["silhouette"], marker="o", lw=2, label=mode)
    axes[0].set_title("K-means inertia (group-SHAP space)")
    axes[1].set_title("silhouette (sample)")
    for ax in axes:
        ax.axvline(_S08_N_ZONES, color="#fb7185", ls="--", lw=1)
        ax.set_xlabel("K")
        ax.legend(frameon=False)
    _s08_save_fig(fig, "08_zone_k_diagnostics.png")


def _s08_fig_interactions():
    modes = list(INTERACTIONS)
    fig, axes = plt.subplots(1, len(modes), figsize=(7.5 * len(modes), 6.5), squeeze=False)
    for ax, mode in zip(axes[0], modes):
        off = np.asarray(INTERACTIONS[mode]["global"], dtype=np.float64).copy()
        np.fill_diagonal(off, np.nan)
        im = ax.imshow(off, cmap="magma_r")
        ax.set_xticks(range(_S08_NG))
        ax.set_yticks(range(_S08_NG))
        ax.set_xticklabels(_S08_GROUPS, rotation=70, ha="right", fontsize=7)
        ax.set_yticklabels(_S08_GROUPS, fontsize=7)
        ax.set_title(f"{mode}: mean |group interaction| (off-diagonal, °C)")
        fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
    _s08_save_fig(fig, "08_interactions.png")


with timer("08_figures"):
    _s08_figs = [_s08_fig_importance, _s08_fig_zones, _s08_fig_signatures, _s08_fig_diagnostics,
                 _s08_fig_interactions] + [(lambda m=m: _s08_fig_dependence(m)) for m in DEPENDENCE]
    for _s08_fig_fn in _s08_figs:
        try:
            if IMPORTANCE:
                _s08_fig_fn()
        except Exception as _s08_exc:  # noqa: BLE001 - a broken figure must never lose the computed results
            log(f"08: figure {getattr(_s08_fig_fn, '__name__', 'dependence')} failed: "
                f"{type(_s08_exc).__name__}: {_s08_exc}", "WARNING")
            plt.close("all")

## 8.21 Contract check (SPEC §3.5)

In [ ]:
def _s08_contract_check():
    """Raise AssertionError if a §3.5 global violates the contract."""
    n = _S08_NC
    for mode in _S08_PAYLOADS:
        assert XAI_FRAMES[mode] == list(_S08_FRAMES), (mode, XAI_FRAMES[mode])
        assert isinstance(SHAP_BASE[mode], float) and np.isfinite(SHAP_BASE[mode])
        for y in XAI_FRAMES[mode]:
            sv, pa, oo = SHAP_GROUP_VALUES[mode][y], PRED_ANOM[mode][y], OOF_ANOM[mode][y]
            assert sv.shape == (n, _S08_NG) and sv.dtype == np.float32, (mode, y, sv.shape, sv.dtype)
            assert pa.shape == (n,) and pa.dtype == np.float32 and np.isfinite(pa).all(), (mode, y)
            assert oo.shape == (n,) and oo.dtype == np.float32, (mode, y)
            valid = np.isfinite(sv).all(axis=1)
            assert np.array_equal(valid, _S08_ROW_VALID[_s08_frame_slice(y)]), f"{mode} {y}: NaN-row pattern"
            assert np.isnan(sv[~valid]).all()
            err = np.abs(pa[valid].astype(np.float64) - (SHAP_BASE[mode] + sv[valid].sum(axis=1, dtype=np.float64)))
            assert err.size == 0 or float(err.max()) < 1e-3, f"{mode} {y}: additivity {float(err.max()):.2e}"
            w = DF[f"w_{mode}"].to_numpy()[_s08_frame_slice(y)]
            assert np.isnan(oo[w <= 0]).all(), f"{mode} {y}: OOF defined on non-training rows"
            z = ZONES_OBS[mode][y]
            assert z.shape == (n,) and z.dtype == np.uint8 and set(np.unique(z).tolist()) <= {0, 1, 2, 3, 255}
            assert np.array_equal(z == 255, ~valid), f"{mode} {y}: zone nodata pattern"
        zm = ZONE_MODEL[mode]
        assert np.asarray(zm["centroids"]).shape == (4, _S08_NG)
        # the labelling rule re-derived from the published zones over the fit frames
        fit = _s08_fit_frames(mode)
        G = np.concatenate([SHAP_GROUP_VALUES[mode][y] for y in fit]).astype(np.float64)
        Z = np.concatenate([ZONES_OBS[mode][y] for y in fit])
        FW = np.concatenate([_S08_X[_s08_frame_slice(y), FEATURES.index("frac_water")] for y in fit])
        ok = Z < 4
        tot = np.array([G[ok & (Z == k)].sum(axis=1).mean() for k in range(4)])
        fw = np.array([np.nanmean(FW[ok & (Z == k)]) for k in range(4)])
        assert int(np.argmax(tot)) == 3, f"{mode}: zone 3 is not the highest mean total SHAP ({tot})"
        lab = zm["labelling"]
        assert isinstance(lab.get("riparian_criterion_met"), bool) and isinstance(lab.get("riparian_criterion"), str)
        for c in lab["clusters"]:
            assert isinstance(c["share_water_ge_0_2"], float) and 0.0 <= c["share_water_ge_0_2"] <= 1.0, c
            assert isinstance(c["water_group_rank"], int) and 1 <= c["water_group_rank"] <= _S08_NG, c
        stats_z = []
        for k in range(3):
            sel = ok & (Z == k)
            fwk = FW[sel][np.isfinite(FW[sel])]
            mabs = np.abs(G[sel]).mean(axis=0)
            stats_z.append({"mean_frac_water": float(fw[k]),
                            "share_water_ge_0_2": float(np.mean(fwk >= _S08_RIPARIAN_WET_FRAC)) if fwk.size else 0.0,
                            "water_mean_abs_shap": float(mabs[_S08_GROUPS.index("water")]),
                            "water_group_rank": 1 + int(np.sum(mabs > mabs[_S08_GROUPS.index("water")]))})
        elig = [bool(_s08_riparian_reasons(s_)) for s_ in stats_z]
        assert any(elig) == lab["riparian_criterion_met"], f"{mode}: riparian criterion re-derivation ({elig})"
        if lab["riparian_criterion_met"]:
            assert elig[1] and all(fw[1] >= fw[k] - 1e-12 for k in (0, 2) if elig[k]), \
                f"{mode}: zone 1 is not the wettest riparian-eligible cluster ({fw}, {elig})"
            assert tot[0] <= tot[2] + 1e-12, f"{mode}: zone 0 must have the lower mean total SHAP of zones 0/2 ({tot})"
        else:
            assert tot[0] <= tot[1] + 1e-12 and tot[1] <= tot[2] + 1e-12, \
                f"{mode}: without a riparian signal zones 0/1/2 must be ordered by mean total SHAP ({tot})"
        for p in ZONE_PROFILES[mode]:
            assert isinstance(p["display_name"], str) and p["display_name"], p["id"]
            assert p["label_caveat"] is None or isinstance(p["label_caveat"], str), p["id"]
            assert p["name"] == _S08_ZONE_META[p["id"]]["name"]                   # canonical names unchanged
            if p["id"] != 1 or lab["riparian_criterion_met"]:
                assert p["display_name"] == p["name"] and p["label_caveat"] is None, p["id"]
            else:
                assert p["display_name"] == _S08_NO_RIPARIAN_NAME and p["label_caveat"]
                assert p["description"].startswith(f"{p['display_name']} ({mode}): {p['label_caveat']}")
        it = INTERACTIONS[mode]
        assert np.asarray(it["global"]).shape == (_S08_NG, _S08_NG) and set(it["by_zone"]) == {0, 1, 2, 3}
        assert list(IMPORTANCE[mode]["groups"]) == _S08_GROUPS and len(IMPORTANCE[mode]["global"]["mean_abs"]) == 18
        assert set(DEPENDENCE[mode]["groups"]) == set(_S08_GROUPS) and set(THRESHOLDS[mode]) == set(_S08_GROUPS)
        for g, t in THRESHOLDS[mode].items():
            for k in ("zero_crossing", "breakpoint", "saturation"):
                assert t[k] is not None or t[f"{k}_ci"] is None, (mode, g, k)
                if t[f"{k}_ci"] is not None:
                    assert t[f"{k}_ci"][0] - 1e-9 <= t[k] <= t[f"{k}_ci"][1] + 1e-9, (mode, g, k)
        assert len(ZONE_PROFILES[mode]) == 4 and [p["id"] for p in ZONE_PROFILES[mode]] == [0, 1, 2, 3]
        assert "obs" in ZONE_TRANSITIONS[mode]
        for recs in RECOMMENDATIONS[mode].values():
            assert len(recs) <= _S08_REC_MAX_PER_ZONE
            for r in recs:
                b = r["bounds"]
                assert r["expected_delta_c"] < -_S08_REC_MIN_COOLING_C, r
                assert b["p10"] - 1e-9 <= r["target"] <= b["p90"] + 1e-9, r
                if b["max_step"] is not None:
                    assert abs(r["step"]) <= b["max_step"] + 1e-9, r
                lo, hi = _S08_BOUNDS.get(r["feature"], (-np.inf, np.inf))
                assert lo - 1e-9 <= r["target"] <= hi + 1e-9, r
                assert r["action"] in _S08_LEVERS_RULES[r["feature"]], r
                assert (r["step"] > 0) == (r["action"] == "increase"), r
                if r["ci"] is not None:
                    assert r["ci"][0] - 1e-9 <= r["expected_delta_c"] <= r["ci"][1] + 1e-9, r
        wm = WEB_MODELS[mode]
        assert wm["format"] == "lst-xgb-compact-v1" and wm["features"] == list(FEATURES)
        assert wm["focal"] == {"bases": list(FOCAL_SPEC["bases"]), "windows": list(FOCAL_SPEC["windows"])}
        assert wm["n_trees"] <= _S08_WEB_ROUNDS and _s08_compact_depth(wm) <= _S08_WEB_DEPTH
        assert len(wm["check"]["rows"]) == len(wm["check"]["expected"]) > 0
        bi = BOOTSTRAP_INFO[mode]
        assert bi["ci_method"] == "recentred-percentile"
        assert bi["ci_label"] in ("95% CI", "replicate range", "no bootstrap")
    assert set(SCENARIO_COUPLING["fraction_features"]) == set(_S08_FRACTIONS)
    assert SCENARIO_COUPLING["focal"]["windows"] == list(FOCAL_SPEC["windows"])


_s08_contract_check()
log("08: contract check passed (SPEC §3.5: shapes, dtypes, additivity, zones/labelling, bounds, web models)")
log("Section 08 complete: XAI_FRAMES, SHAP_BASE, SHAP_GROUP_VALUES, PRED_ANOM, OOF_ANOM, IMPORTANCE, INTERACTIONS, "
    "DEPENDENCE, THRESHOLDS, BOOTSTRAP_INFO, ZONE_MODEL, ASSIGN_ZONES, ZONES_OBS, ZONE_PROFILES, ZONE_TRANSITIONS, "
    "RECOMMENDATIONS, SCENARIO_COUPLING, WEB_MODELS ready")
free_memory()

# 9. Timeline 1990–2025: absolute levels, evidence tiers, frame fields, night hindcast, dLST/dTair and trends

Sections 06–08 work in **anomalies** (each frame centred to zero over the mode's fixed mask). This section restores
absolute °C and attaches an evidence tier and an uncertainty to every number the timeline shows.

## 9.1 Levels $L_m(t)$ (SPEC §4.7)
* **Day.** Landsat scenes are acquired between ~09:15 and ~10:50 IST depending on sensor and orbit drift, and the
  pre-monsoon surface warms by ~3 K h⁻¹ at that time, so a raw composite of early scenes is several tenths of a degree
  too cold. The level is corrected to the 10:45 IST reference overpass:
  $L_{day}(t) = \bar T^{raw}_{FIXED}(t) + (10.75 - \bar h_t)\,\dot T_t$ with $\bar h_t$ the mean scene time (IST hours)
  and $\dot T_t$ the ERA5-Land skin-temperature heating rate between 09:30 and 11:30 IST of the frame's window
  (`FRAME_META[t]["day"]`). Interval: $\pm(0.25\,|c_t| + 0.3)$ °C (documented heuristic: a quarter of the correction
  for the heating-rate/overpass uncertainty plus 0.3 °C for the Landsat ST level). The day frame field is the raw
  fine-pixel mean + the same correction (anomalies are unaffected). `DAY_LEVEL_CHECK` compares the corrected level
  with the independent MODIS Terra fixed-mask mean (2000–2020): a constant offset (algorithm/time-of-day difference)
  is expected, a frame-to-frame scatter is not.
* **Night, observed/reconstructed.** $L_{night}(t)$ = the fixed-mask mean of the Aqua-equivalent target (2000 =
  Terra 22:30 calibrated to Aqua). Interval ±0.3 °C (observed Aqua), ±(0.3 + 0.5·RMSE$_{cal}$) for calibrated sensors.
* **Night 1990/1995 (modelled).** No night product exists. The annual NCR-mean night LST (Aqua 2003–2021, VIIRS
  calibrated 2022–2025) is regressed on ERA5-Land's 20 UTC skin temperature (OLS, leave-one-year-out RMSE). The
  window-mean prediction is tied to the fixed-mask level definition by a time-dependent offset
  $o(t) = \bar\delta + \hat d\,(t_{mid} - \bar t_{mid})$: $\bar\delta$ = mean frame-level residual over the observed
  frames and $\hat d$ = the **residual drift**, the OLS slope of the annual regression residuals over the years of
  the *primary* sensor only (night: Aqua 2003–2021, so the VIIRS calibration step cannot pose as drift), applied
  only when $|\hat d| > 2\,se(\hat d)$ on at least 10 primary-sensor years spanning at least 10 years — ERA5 does
  not see the land-driven warming of the night level by
  urbanisation, so a level extrapolated 25 years back would otherwise be too warm. The same $o(t)$ produces the
  `regression_check` of the observed frames. 90 % prediction interval
  $t_{0.95,n-2}\sqrt{s^2/k + s^2 x_0^\top(X^\top X)^{-1}x_0 + \mathrm{Var}(\delta)(1+1/n_f) + (\Delta t)^2 se_{drift}^2}$
  ($k$ window years; $\Delta t$ years from the regression period's mean year,
  $se_{drift} = s/\sqrt{\sum (t-\bar t)^2}$).
  *Known limitation:* the primary record is short (19 Aqua years), so a real residual drift of ~0.015–0.02 °C/yr
  can stay just below the 2-se gate and is then not applied; the 1990/1995 levels then keep a warm bias of about
  $|\hat d|\,\Delta t$ (+0.26/+0.40 °C on the synthetic world, drift 0.0174 ± 0.0092 °C/yr = 1.9 se; both levels
  stay inside their prediction intervals). The WARNING log and the modelled frames' `method` text report
  $\hat d \pm se$ and the shift the drift would have made ($\hat d\,\Delta t$), and the reconstructed 2000 frame's
  `regression_check` error shows how well the offset carries back in time.
  The field is the recentred model hindcast + the persistent residual + the level, with per-cell p05/p95 = field ∓
  √(q90² + level half-width²), q90 = the projection-setting conformal |residual| q90 of the cell's AOA-DI decile
  (section 07: LOEO + persistent residual — a known place at another time; block20 q90 as fallback).
* **Composition correction (all non-modelled frames).** A raw level is the mean over the FIXED_MASK cells valid at
  that frame. When a frame covers < 99.9 % of the mask (`FIXED_MASK_COVERAGE`), the missing cells' anomaly is
  taken from the model, $a$ = final-model anomaly (not recentred) + persistent residual:
  $c_t = \overline{a}_{FIXED} - \overline{a}_{FIXED \wedge valid_t}$, level = raw (+ overpass correction) + $c_t$,
  half-width $\sqrt{hw^2 + (c_t/2)^2}$. All levels then refer to the same cell set; the frame field itself stays the
  observation. A day frame never becomes `modelled`: a day frame without a fixed-mask level stops the section.

## 9.2 Evidence tiers (computed from `FRAME_META`, SPEC §1.2)
Day: `observed-sparse` when any of the five WRS-2 path/rows has < 6 clear scenes in the window, else `observed`
(never `modelled`: a day frame without a fixed-mask level is an acquisition fault and stops the section).
Night: `observed` for Aqua/VIIRS, `reconstructed` for calibrated Terra, `modelled` without a night product.
Projection frames are `projected`.

## 9.3 Hindcast checks
* **Night 2000:** the night model was trained on 2005–2025 only, so its recentred prediction of 2000 vs the
  Terra-reconstructed observation is a genuine out-of-period test; reference = persistence (the 2005 anomaly).
* **Day "train late, predict early":** a model with the final recipe on 2005–2025 predicts 1990/1995/2000
  (recentred), again against persistence of 2005 — this mimics the night hindcast for a mode where the truth exists.

## 9.4 Climate sensitivity β = dLST/dT_air (SPEC §4.7)
Year-to-year variation of the NCR-mean satellite LST against ERA5-Land air temperature (day: MODIS Terra 2003–2020
on Tmax; night: Aqua 2003–2021 + VIIRS 2022–2025 on Tmin), with top-soil moisture as co-regressor so that dry years
do not inflate β. OLS coefficient of temperature; uncertainty from a **moving-block bootstrap** (3-year blocks,
2000 draws — preserves the short-range autocorrelation of annual anomalies); Theil–Sen univariate slope alongside.
The draws feed the Monte-Carlo projections (section 11).

## 9.5 Trends and frame statistics
Theil–Sen slope per decade of the frame levels (observed/reconstructed tiers only) with a Monte-Carlo interval
that propagates the level intervals; Theil–Sen (95 % CI) of the annual ERA5-Land and satellite series. Per scope
(NCR, 25 districts, zones of the mode): mean (NCR: the frame **level** and its CI — fixed mask, composition
consistent; districts and zones: mean of their valid cells with the level's half-widths), p90, hot-core area (LST ≥
the mode's reference-period 2000–2020 p90, fixed; the share of valid scope cells above it × the scope's cell count,
so a cloud gap does not shrink the area — unbiased for scattered gaps, while a spatially contiguous gap over a
cooler or hotter part of the scope moves the area by the difference of the hidden and the visible hot share, e.g.
+11 % to −36 % for 10 % strips in the synthetic tests; every row exports `coverage` and rows below 0.9 are logged),
the **signed** surface UHI (mean over cells with built ≥ 0.5 minus cells
with built ≤ 0.05 and water < 0.2), the number of valid cells, `coverage` (valid / scope cells) and the tier.

In [ ]:
_lst_enter_section('09')  # section marker (generated by build_notebook.py)

In [ ]:
import math as _s09_math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats as _s09_stats

_S09_VERSION = "09-v2.1"
_S09_FRAMES = sorted(int(y) for y in np.unique(DF["year"].to_numpy()))
_S09_NC = int(N_CELLS)
_S09_REF_H = float(CFG.DAY_REFERENCE_OVERPASS_IST_H)
_S09_TIERS = ["observed", "observed-sparse", "reconstructed", "modelled", "projected"]
_S09_MIN_SCENES = 6
_S09_Z90 = float(_s09_stats.norm.ppf(0.95))
_S09_NIGHT_OBS_HW = 0.3
_S09_CAL_HW_FACTOR = 0.5
_S09_MC_DRAWS = 2000
_S09_BETA_DRAWS = 2000
_S09_BETA_BLOCK = 3
_S09_REF_PERIOD = tuple(int(v) for v in CFG.REFERENCE_PERIOD)
_S09_TEMP_VAR = {"day": "era5_tmax", "night": "era5_tmin"}
_S09_SKIN_VAR = {"day": "era5_skin_day", "night": "era5_skin_night"}
_S09_DRIFT_MIN_YEARS = 10           # a residual drift is applied only on at least this many years / this span
# Satellite annual series per mode (variable, first year, last year) - SPEC §4.7 / §5.4.
_S09_BETA_SAT = {"day": [("modis_terra_day", 2003, 2020)],
                 "night": [("modis_aqua_night", 2003, 2021), ("viirs_night_cal", 2022, 2025)]}
_S09_LEVEL_SAT = {"day": [("modis_terra_day", 2003, 2021)],
                  "night": [("modis_aqua_night", 2003, 2021), ("viirs_night_cal", 2022, 2025)]}
_S09_SENSOR_LABEL = {"modis_terra_day": "MODIS Terra", "modis_aqua_night": "MODIS Aqua",
                     "viirs_night_cal": "VIIRS (Aqua-calibrated)"}
_S09_MODES = [m for m in CFG.MODES if m in FINAL_MODELS]
_S09_RES_KM2 = float(GRID.cell_area_km2)
if len(DF) != len(_S09_FRAMES) * _S09_NC:
    raise RuntimeError("09: DF is not the (frame x canonical cell) table of section 06")
log(f"09: modes {_S09_MODES}, frames {_S09_FRAMES}, reference overpass {_S09_REF_H:.2f} h IST, reference period "
    f"{_S09_REF_PERIOD}")


def _s09_slice(year):
    j = _S09_FRAMES.index(int(year))
    return slice(j * _S09_NC, (j + 1) * _S09_NC)


def _s09_col(name, year):
    """DF column ``name`` of one frame (canonical cell order) as float64."""
    return DF[name].to_numpy(np.float64)[_s09_slice(year)]


def _s09_fixed_cells(mode):
    """Bool (N_CELLS,) - canonical cells inside FIXED_MASK[mode]."""
    return np.asarray(FIXED_MASK[mode], bool)[CELL_ROWS, CELL_COLS]


def _s09_annual(variable, scope="ncr"):
    """pd.Series (year -> value) of one ANNUAL_SERIES variable and scope (empty when absent)."""
    sub = ANNUAL_SERIES[(ANNUAL_SERIES["variable"] == variable) & (ANNUAL_SERIES["scope"] == scope)]
    s = pd.Series(sub["value"].to_numpy(np.float64), index=sub["year"].to_numpy(np.int64))
    return s[~s.index.duplicated()].sort_index()


def _s09_annual_flags(variable, scope="ncr"):
    sub = ANNUAL_SERIES[(ANNUAL_SERIES["variable"] == variable) & (ANNUAL_SERIES["scope"] == scope)]
    return dict(zip(sub["year"].astype(int), sub["flag"]))


def _s09_sat_series(mode, ranges, scope="ncr", drop_drift=False):
    """Combined satellite series of a mode over ``ranges`` [(variable, y0, y1)] -> (Series, sensor per year)."""
    vals, sensor = {}, {}
    for var, y0, y1 in ranges:
        s = _s09_annual(var, scope)
        flags = _s09_annual_flags(var, scope)
        for y, v in s.items():
            if y0 <= y <= y1 and np.isfinite(v) and not (drop_drift and flags.get(int(y)) == "drift"):
                vals[int(y)] = float(v)
                sensor[int(y)] = var
    s = pd.Series(vals, dtype=np.float64).sort_index()
    return s, sensor

## 9.6 Evidence tiers (`FRAME_TIERS`)

In [ ]:
def _s09_tier(mode, year):
    meta = FRAME_META.get(int(year)) or {}
    raw_level = RAW_LEVELS.get(mode, {}).get(int(year), float("nan"))
    if mode == "day":
        if not np.isfinite(raw_level):
            # D9: an observed day frame is always observed or observed-sparse; no fixed-mask level means the
            # acquisition (section 03) or the masks (section 06) failed for this frame
            raise RuntimeError(
                f"09: day frame {int(year)} has no finite fixed-mask level (RAW_LEVELS['day'][{int(year)}] = "
                f"{raw_level}); its valid cells do not overlap FIXED_MASK['day']. Day frames are never modelled - "
                "check the section-03 coverage of this frame and the section-06 coverage table.")
        spr = (meta.get("day") or {}).get("scenes_per_pathrow") or {}
        return "observed-sparse" if spr and min(int(v) for v in spr.values()) < _S09_MIN_SCENES else "observed"
    source = (meta.get("night") or {}).get("source")
    if source is None or not np.isfinite(raw_level):
        return "modelled"
    return "reconstructed" if source == "terra_cal" else "observed"


FRAME_TIERS = {m: {int(y): _s09_tier(m, y) for y in _S09_FRAMES} for m in CFG.MODES}
for _m in CFG.MODES:
    FRAME_TIERS[_m].update({int(y): "projected" for y in CFG.FRAMES_PROJ})
log("09: tiers " + "; ".join(f"{m}: " + ", ".join(f"{y}={t}" for y, t in FRAME_TIERS[m].items()) for m in CFG.MODES))

## 9.7 Levels (`LEVELS`)
Every record keeps `raw_level` (the fixed-mask mean over the valid cells), `fixed_mask_coverage` and
`composition_correction` (non-modelled frames; 0 when the frame covers the whole mask), and its `method` says which
corrections were applied. Modelled frames carry `drift_correction_c` (the residual-drift part of the offset, 9.1);
`_S09_LEVEL_REG[m]` (also `HINDCAST_METRICS[m]["level_regression"]`) records `drift_per_year`, `drift_se`,
`drift_applied`, `drift_years` and the Theil–Sen slope of all residuals for reference.

In [ ]:
def _s09_level_regression(mode):
    """OLS of the annual NCR satellite level on ERA5-Land skin temperature (+ LOYO RMSE); None if < 5 years."""
    y_s, sensor = _s09_sat_series(mode, _S09_LEVEL_SAT[mode])
    x_s = _s09_annual(_S09_SKIN_VAR[mode])
    years = sorted(set(y_s.index) & set(x_s.index))
    if len(years) < 5:
        log(f"09: {mode}: only {len(years)} years for the level regression", "WARNING")
        return None
    x = x_s.loc[years].to_numpy(np.float64)
    y = y_s.loc[years].to_numpy(np.float64)
    X = np.column_stack([np.ones(len(x)), x])
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ coef
    n = len(years)
    s2 = float(resid @ resid) / max(n - 2, 1)
    xtx_inv = np.linalg.pinv(X.T @ X)            # pinv: a constant regressor must not crash the notebook
    loyo = []
    for i in range(n):
        keep = np.arange(n) != i
        c_i, *_ = np.linalg.lstsq(X[keep], y[keep], rcond=None)
        loyo.append(y[i] - X[i] @ c_i)
    sst = float(np.sum((y - y.mean()) ** 2))
    yr = np.asarray(years, np.float64)
    drift_se_pi = float(np.sqrt(s2 / np.sum((yr - yr.mean()) ** 2))) if n > 2 else 0.0
    drift_ts = float(_s09_stats.theilslopes(resid, yr)[0]) if n > 2 else 0.0
    # D10: residual drift = OLS slope (+ se) of the annual residuals over the PRIMARY sensor's years only, so a
    # calibration step between sensors (Aqua -> calibrated VIIRS) cannot pose as drift
    prim_var, p0, p1 = _S09_LEVEL_SAT[mode][0]
    prim = np.array([sensor.get(int(v)) == prim_var and p0 <= int(v) <= p1 for v in years], dtype=bool)
    drift, drift_se, drift_years = 0.0, None, [int(v) for v in yr[prim]]
    if prim.sum() >= 4:
        tp, ep = yr[prim], resid[prim]
        tc = tp - tp.mean()
        drift = float((tc @ (ep - ep.mean())) / (tc @ tc))
        e2 = ep - ep.mean() - drift * tc
        drift_se = float(np.sqrt(float(e2 @ e2) / (tp.size - 2) / float(tc @ tc)))
    # A linear drift extrapolated decades back needs a real record: with too few years a slope can pass the 2 se test
    # by chance (first real run: 5 Aqua years gave -0.24 °C/yr, a 1990 night level of 28.6 °C vs 19-21 °C observed).
    enough = bool(prim.any()) and prim.sum() >= _S09_DRIFT_MIN_YEARS and np.ptp(yr[prim]) >= _S09_DRIFT_MIN_YEARS
    if drift_se is not None and abs(drift) > 2.0 * drift_se and not enough:
        log(f"09: {mode}: residual drift {drift:+.4f} +- {drift_se:.4f} °C/yr not applied: it rests on {int(prim.sum())} "
            f"{prim_var} years (needs >= {_S09_DRIFT_MIN_YEARS} years spanning >= {_S09_DRIFT_MIN_YEARS})", "WARNING")
    applied = drift if enough and drift_se is not None and abs(drift) > 2.0 * drift_se else 0.0
    return {"a": float(coef[0]), "b": float(coef[1]), "s": float(np.sqrt(s2)), "xtx_inv": xtx_inv, "n": n,
            "year_mean": float(yr.mean()), "drift_se_per_year": drift_se_pi, "residual_drift_theil_sen": drift_ts,
            "drift_per_year": drift, "drift_se": drift_se, "drift_applied": float(applied),
            "drift_years": drift_years, "drift_sensor": prim_var,
            "years": [int(v) for v in years], "r2": 1.0 - float(resid @ resid) / sst if sst > 0 else None,
            "loyo_rmse": float(np.sqrt(np.mean(np.square(loyo)))), "t90": float(_s09_stats.t.ppf(0.95, max(n - 2, 1))),
            "x_var": _S09_SKIN_VAR[mode], "y_series": sorted(set(sensor.values()))}


def _s09_window_x(mode, year):
    """(mean ERA5 skin temperature over the frame's window years, years used)."""
    x_s = _s09_annual(_S09_SKIN_VAR[mode])
    y0, y1 = CFG.window(int(year))
    yrs = [y for y in range(int(y0), int(y1) + 1) if y in x_s.index and np.isfinite(x_s.loc[y])]
    return (float(x_s.loc[yrs].mean()) if yrs else float("nan")), yrs


def _s09_regression_level(reg, mode, year, offset, offset_var):
    """Window-mean level predicted by the regression (+ mask offset) with its 90 % prediction half-width."""
    xbar, yrs = _s09_window_x(mode, year)
    if reg is None or not np.isfinite(xbar):
        return float("nan"), float("nan"), yrs
    x0 = np.array([1.0, xbar])
    k = max(len(yrs), 1)
    w0, w1 = CFG.window(int(year))
    dt = 0.5 * (int(w0) + int(w1)) - reg["year_mean"]
    se2 = (reg["s"] ** 2 / k + reg["s"] ** 2 * float(x0 @ reg["xtx_inv"] @ x0) + offset_var
           + (dt * reg["drift_se_per_year"]) ** 2)
    return reg["a"] + reg["b"] * xbar + offset, reg["t90"] * _s09_math.sqrt(se2), yrs


def _s09_night_obs_halfwidth(year):
    source = ((FRAME_META.get(int(year)) or {}).get("night") or {}).get("source")
    if source in ("terra_cal", "viirs_cal"):
        kind = "terra_to_aqua" if source == "terra_cal" else "viirs_to_aqua"
        rmse = float((NIGHT_CALIBRATION.get(kind) or {}).get("rmse") or 0.0)
        return _S09_NIGHT_OBS_HW + _S09_CAL_HW_FACTOR * rmse, kind, rmse
    return _S09_NIGHT_OBS_HW, None, None


def _s09_t_mid(year):
    w0, w1 = CFG.window(int(year))
    return 0.5 * (int(w0) + int(w1))


def _s09_composition(mode, year):
    """(composition correction °C, FIXED_MASK coverage) of a frame's raw level (D8).

    The raw level is the mean over the FIXED_MASK cells that are valid at this frame. When they are not all valid,
    the missing cells' anomaly is taken from the model: a = final-model anomaly (not recentred) + persistent residual,
    corr = mean_FIXED(a) - mean_{FIXED & valid}(a), so raw + corr estimates the mean over the whole mask.
    """
    fixed = _s09_fixed_cells(mode)
    valid = np.isfinite(_s09_col(f"lst_{mode}", year))
    both = fixed & valid
    cov = float(both.sum()) / max(int(fixed.sum()), 1)
    if cov >= 0.999 or not both.any():
        return 0.0, cov
    X = DF[FEATURES].to_numpy(np.float32)[_s09_slice(year)]
    a = PREDICT_ANOM(mode, X).astype(np.float64)
    a = a + np.nan_to_num(np.asarray(PERSISTENT_RESID[mode], np.float64)[CELL_ROWS, CELL_COLS], nan=0.0)
    return float(np.mean(a[fixed]) - np.mean(a[both])), cov


def _s09_levels(mode):
    """LEVELS[mode] for every observed frame + the regression record."""
    out = {}
    reg = _s09_level_regression(mode)
    observed = [y for y in _S09_FRAMES if FRAME_TIERS[mode][y] in ("observed", "observed-sparse")
                and np.isfinite(RAW_LEVELS[mode].get(y, np.nan))]
    # composition corrections first: the regression offset is fitted on the corrected levels (D8)
    comp = {}
    for y in _S09_FRAMES:
        if FRAME_TIERS[mode][y] != "modelled" and np.isfinite(RAW_LEVELS[mode].get(y, np.nan)):
            comp[y] = _s09_composition(mode, y)
            if abs(comp[y][0]) > 0.2:
                log(f"09: {mode} {y}: composition correction {comp[y][0]:+.2f} °C (the frame covers "
                    f"{comp[y][1]:.1%} of the fixed mask)", "WARNING")

    def obs_level(y):
        lev = RAW_LEVELS[mode][y] + comp.get(y, (0.0, 1.0))[0]
        return lev + _s09_overpass_correction(y) if mode == "day" else lev

    resid, resid_t, resid_y = [], [], []
    if reg is not None:
        for y in observed:
            pred, _, _ = _s09_regression_level(reg, mode, y, 0.0, 0.0)
            if np.isfinite(pred):
                resid.append(obs_level(y) - pred)
                resid_t.append(_s09_t_mid(y))
                resid_y.append(y)
    offset_mean = float(np.mean(resid)) if resid else 0.0
    t_bar = float(np.mean(resid_t)) if resid_t else 0.0
    drift = float(reg["drift_applied"]) if reg is not None else 0.0

    def offset(y):
        """Mask offset at frame y: mean frame residual + the applied residual drift (D10)."""
        return offset_mean + drift * (_s09_t_mid(y) - t_bar)

    drift_se_txt = f"{reg['drift_se']:.4f}" if reg is not None and reg.get("drift_se") is not None else "n/a"
    dev = [r - offset(y) for r, y in zip(resid, resid_y)]
    # a NEW frame's level residual: Var(delta) * (1 + 1/n_f) (prediction, not confidence, interval)
    offset_var = float(np.var(dev, ddof=1) * (1.0 + 1.0 / len(dev))) if len(dev) >= 2 else 0.0
    for y in _S09_FRAMES:
        tier = FRAME_TIERS[mode][y]
        raw = RAW_LEVELS[mode].get(y, float("nan"))
        corr_c, cov = comp.get(y, (None, None))
        if cov is None:
            fixed = _s09_fixed_cells(mode)
            cov = float((fixed & np.isfinite(_s09_col(f"lst_{mode}", y))).sum()) / max(int(fixed.sum()), 1)
        rec = {"tier": tier, "raw_level": None if not np.isfinite(raw) else float(raw),
               "composition_correction": None if corr_c is None else float(corr_c), "fixed_mask_coverage": float(cov)}
        comp_txt = (f"; + composition correction {corr_c:+.3f} °C for the {1.0 - cov:.1%} of the fixed mask without "
                    "a valid target (model anomaly + persistent residual)" if corr_c else "")
        if tier == "modelled":
            level, hw, yrs = _s09_regression_level(reg, mode, y, offset(y), offset_var)
            if not np.isfinite(level):
                ref = [obs_level(t) for t in observed]
                level, hw = (float(np.mean(ref)) if ref else float("nan")), 1.5
                log(f"09: {mode} {y}: level regression unavailable; mean observed level +- 1.5 °C used", "WARNING")
                method = "fallback: mean of the observed frame levels (+-1.5 °C)"
                drift_c = None
            else:
                drift_c = float(drift * (_s09_t_mid(y) - t_bar)) if drift else 0.0   # no -0.0 in the JSON
                method = (f"OLS of annual NCR {'/'.join(reg['y_series'])} on {reg['x_var']} "
                          f"({reg['years'][0]}-{reg['years'][-1]}, n={reg['n']}), window mean of {yrs}, "
                          f"+ mask offset {offset_mean:+.3f} °C"
                          + (f" + residual drift {drift:+.4f} °C/yr x {(_s09_t_mid(y) - t_bar):+.1f} yr "
                             f"= {drift_c:+.3f} °C" if drift else
                             f" (residual drift {reg['drift_per_year']:+.4f} +- {drift_se_txt} °C/yr below 2 se, "
                             f"not applied; applied, it would move this level by "
                             f"{reg['drift_per_year'] * (_s09_t_mid(y) - t_bar):+.2f} °C)")
                          + ", 90% prediction interval")
            rec.update({"level": float(level), "ci": [float(level - hw), float(level + hw)], "method": method,
                        "overpass_correction": None, "half_width": float(hw), "window_years_used": yrs,
                        "drift_correction_c": drift_c})
        elif mode == "day":
            corr = _s09_overpass_correction(y)
            level = raw + corr + (corr_c or 0.0)
            hw = float(np.hypot(0.25 * abs(corr) + 0.3, 0.5 * (corr_c or 0.0)))
            meta = (FRAME_META.get(y) or {}).get("day") or {}
            rec.update({"level": float(level), "ci": [float(level - hw), float(level + hw)],
                        "method": ("Landsat C2 L2 ST fixed-mask mean, corrected to the 10:45 IST reference overpass"
                                   + comp_txt),
                        "overpass_correction": float(corr), "half_width": float(hw),
                        "mean_overpass_ist_h": meta.get("mean_overpass_ist_h"),
                        "heating_rate_k_per_h": meta.get("heating_rate_k_per_h")})
        else:
            hw0, kind, rmse = _s09_night_obs_halfwidth(y)
            hw = float(np.hypot(hw0, 0.5 * (corr_c or 0.0)))
            level = raw + (corr_c or 0.0)
            source = ((FRAME_META.get(y) or {}).get("night") or {}).get("source")
            rec.update({"level": float(level), "ci": [float(level - hw), float(level + hw)],
                        "method": (f"fixed-mask mean of the Aqua-equivalent night LST (source {source}"
                                   + (f", calibrated {kind}, RMSE {rmse:.2f} °C" if kind else "") + ")" + comp_txt),
                        "overpass_correction": None, "half_width": float(hw), "source": source})
        if reg is not None and tier != "modelled":
            pred, phw, _ = _s09_regression_level(reg, mode, y, offset(y), offset_var)
            rec["regression_check"] = {"predicted": float(pred), "half_width": float(phw),
                                       "error": float(rec["level"] - pred) if np.isfinite(pred) else None}
        out[y] = rec
    record = None
    if reg is not None:
        record = {k: v for k, v in reg.items() if k != "xtx_inv"}
        record.update({"mask_offset": offset_mean, "mask_offset_var": offset_var, "offset_frames": observed,
                       "offset_t_mid_mean": t_bar,
                       "offset_rule": "offset(t) = mean frame residual + drift_applied x (t_mid - offset_t_mid_mean)"})
        checks = {y: r["regression_check"]["error"] for y, r in out.items()
                  if r["tier"] == "reconstructed" and r.get("regression_check")}
        se_txt = f"{reg['drift_se']:.4f}" if reg["drift_se"] is not None else "n/a"
        log(f"09: {mode} level-regression residual drift {reg['drift_per_year']:+.4f} +- {se_txt} °C/yr over "
            f"{reg['drift_sensor']} {reg['drift_years'][:1] + reg['drift_years'][-1:]} -> "
            + (f"applied ({drift:+.4f} °C/yr)" if drift else "not applied (|drift| <= 2 se)")
            + ("; reconstructed-frame regression_check error " + ", ".join(
                f"{y}: {e:+.3f} °C" for y, e in checks.items() if e is not None) if checks else ""), "WARNING")
    return out, record


def _s09_overpass_correction(year):
    """(10.75 - mean overpass IST h) x heating rate (°C, positive for early scenes); 0 with a warning if missing."""
    meta = (FRAME_META.get(int(year)) or {}).get("day") or {}
    h, rate = meta.get("mean_overpass_ist_h"), meta.get("heating_rate_k_per_h")
    if h is None or rate is None or not (np.isfinite(h) and np.isfinite(rate)):
        log(f"09: day {year}: overpass time or heating rate missing; no overpass correction", "WARNING")
        return 0.0
    return float((_S09_REF_H - float(h)) * float(rate))


LEVELS, _S09_LEVEL_REG = {}, {}
for _m in _S09_MODES:
    LEVELS[_m], _S09_LEVEL_REG[_m] = _s09_levels(_m)
    log(f"09: {_m} levels: " + ", ".join(f"{y}={r['level']:.2f} [{r['ci'][0]:.2f}, {r['ci'][1]:.2f}]"
                                         + ("*" if r["tier"] == "modelled" else "") for y, r in LEVELS[_m].items()))
    if _S09_LEVEL_REG[_m]:
        _r = _S09_LEVEL_REG[_m]
        log(f"09: {_m} level regression: {_r['y_series']} = {_r['a']:.2f} + {_r['b']:.3f} x {_r['x_var']} "
            f"(n={_r['n']}, R2 {_r['r2']:.3f}, LOYO RMSE {_r['loyo_rmse']:.3f} °C, mask offset "
            f"{_r['mask_offset']:+.3f})")

## 9.8 Frame fields, uncertainty and AOA flags
Observed and reconstructed frames: the observed 1 km values where the target is valid (NaN elsewhere; day + the
overpass correction). Modelled frames: the full model field. `FRAME_UNC` exists for modelled and reconstructed
frames only (observed frames carry the observation itself).

In [ ]:
def _s09_conformal_q90(mode, di, projection=False):
    """(|residual| q90 of each cell's AOA DI decile, error-model label) from section 07.

    ``projection=True`` (modelled frames: a known place at another time) uses the projection-setting model
    ``AOA[m]["conformal_proj"]`` (LOEO + persistent residual) when present, else the block20 conformal model.
    DI = +inf (a missing weighted feature) falls into the top decile.
    """
    proj = AOA[mode].get("conformal_proj") if projection else None
    conf = proj or AOA[mode]["conformal"]
    label = ("projection-setting (LOEO + persistent residual)" if proj else "block20 spatial-CV")
    edges = np.asarray(conf.get("edges", AOA[mode]["conformal"]["edges"]), dtype=np.float64)
    q90 = np.asarray(conf["abs_resid_quantiles"]["q90"], dtype=np.float64)
    dec = np.clip(np.searchsorted(edges, np.asarray(di, np.float64), side="right"), 0, q90.size - 1)
    return q90[dec], label


def _s09_di(mode, year):
    di = (AOA.get(mode) or {}).get("di_frames", {}).get(int(year))
    if di is None:
        di = COMPUTE_DI(mode, DF[FEATURES].to_numpy(np.float32)[_s09_slice(year)])
    return np.asarray(di, dtype=np.float64)


def _s09_modelled_field(mode, year):
    """Recentred final-model anomaly + persistent residual + level for every ROI cell (float64)."""
    X = DF[FEATURES].to_numpy(np.float32)[_s09_slice(year)]
    fixed = _s09_fixed_cells(mode)
    anom = PREDICT_ANOM(mode, X, recentre_mask=fixed).astype(np.float64)
    resid = np.nan_to_num(np.asarray(PERSISTENT_RESID[mode], np.float64)[CELL_ROWS, CELL_COLS], nan=0.0)
    return anom + resid + LEVELS[mode][year]["level"], anom, resid


def _s09_frame_fields(mode):
    fields, unc, aoa_out = {}, {}, {}
    for y in _S09_FRAMES:
        tier = FRAME_TIERS[mode][y]
        lev = LEVELS[mode][y]
        di = _s09_di(mode, y)
        aoa_out[y] = (di > float(AOA[mode]["threshold"]))         # DI = +inf (missing feature) is outside
        if tier == "modelled":
            field, _, _ = _s09_modelled_field(mode, y)
            q90, label = _s09_conformal_q90(mode, di, projection=True)
            hw = np.sqrt(q90 ** 2 + lev["half_width"] ** 2)
            unc[y] = {"p05": (field - hw).astype(np.float32), "p95": (field + hw).astype(np.float32),
                      "method": f"conformal q90 by AOA-DI decile ({label} residuals) (+) level 90% PI, in "
                                "quadrature; DI = inf (missing feature) uses the top decile"}
        else:
            obs = _s09_col(f"lst_{mode}", y)
            corr = lev["overpass_correction"] or 0.0
            field = np.where(np.isfinite(obs), obs + corr, np.nan)
            if tier == "reconstructed":
                _, kind, rmse = _s09_night_obs_halfwidth(y)
                hw = np.sqrt((_S09_Z90 * float(rmse or 0.0)) ** 2 + lev["half_width"] ** 2)
                unc[y] = {"p05": np.where(np.isfinite(field), field - hw, np.nan).astype(np.float32),
                          "p95": np.where(np.isfinite(field), field + hw, np.nan).astype(np.float32),
                          "method": (f"calibration ({kind}) RMSE x 1.645 (+) level interval, in quadrature; the "
                                     "values are calibrated observations, not model predictions")}
        fields[y] = np.asarray(field, dtype=np.float32)
    return fields, unc, aoa_out


FRAME_FIELDS, FRAME_UNC, FRAME_AOA_OUT = {}, {}, {}
with timer("09_frame_fields"):
    for _m in _S09_MODES:
        FRAME_FIELDS[_m], FRAME_UNC[_m], FRAME_AOA_OUT[_m] = _s09_frame_fields(_m)
        log(f"09: {_m} frame fields: " + ", ".join(
            f"{y}:{FRAME_TIERS[_m][y][:4]} n={int(np.isfinite(v).sum()):,} mean={np.nanmean(v):.2f}"
            for y, v in FRAME_FIELDS[_m].items()))

## 9.9 Hindcast checks (`HINDCAST_METRICS`)

In [ ]:
def _s09_metrics(obs, pred, weight=None):
    obs, pred = np.asarray(obs, np.float64), np.asarray(pred, np.float64)
    w = np.ones_like(obs) if weight is None else np.asarray(weight, np.float64)
    ok = np.isfinite(obs) & np.isfinite(pred) & (w > 0)
    if ok.sum() < 3:
        return {"r2": None, "rmse": None, "bias": None, "n": int(ok.sum())}
    o, p, w = obs[ok], pred[ok], w[ok]
    sw = w.sum()
    ob = (w * o).sum() / sw
    sse = float((w * (o - p) ** 2).sum())
    sst = float((w * (o - ob) ** 2).sum())
    return {"r2": 1.0 - sse / sst if sst > 0 else None, "rmse": float(np.sqrt(sse / sw)),
            "bias": float((w * (p - o)).sum() / sw), "n": int(ok.sum())}


def _s09_night_hindcast():
    """Night 2000 (Terra-reconstructed) vs the recentred night model trained on 2005-2025, vs persistence."""
    mode = "night"
    if mode not in _S09_MODES or 2000 not in _S09_FRAMES or FRAME_TIERS[mode].get(2000) != "reconstructed":
        return {"available": False, "reason": "no reconstructed night 2000 frame"}
    if 2000 in set(int(v) for v in np.unique(DF["year"].to_numpy()[DF["w_night"].to_numpy() > 0])):
        return {"available": False, "reason": "2000 is a night training frame (not an independent test)"}
    _, anom_pred, resid = _s09_modelled_field(mode, 2000)
    obs = _s09_col("anom_night", 2000)
    later = [y for y in _S09_FRAMES if y > 2000 and FRAME_TIERS[mode][y] == "observed"]
    persist = _s09_col("anom_night", later[0]) if later else np.full(_S09_NC, np.nan)
    both = np.isfinite(obs) & np.isfinite(persist)
    m_model = _s09_metrics(obs[both], anom_pred[both])
    m_resid = _s09_metrics(obs[both], (anom_pred + resid)[both])
    m_pers = _s09_metrics(obs[both], persist[both])
    lev = LEVELS[mode][2000].get("regression_check") or {}
    return {"available": True, "frame": 2000, "train_frames": sorted(int(v) for v in CFG.NIGHT_TRAIN_FRAMES),
            "r2": m_model["r2"], "rmse": m_model["rmse"], "bias": m_model["bias"], "n": m_model["n"],
            "r2_with_persistent_residual": m_resid["r2"], "rmse_with_persistent_residual": m_resid["rmse"],
            "persistence_frame": later[0] if later else None, "persistence_r2": m_pers["r2"],
            "persistence_rmse": m_pers["rmse"], "recentred": True,
            "level_check": {"observed_level": LEVELS[mode][2000]["level"], "regression_level": lev.get("predicted"),
                            "error": lev.get("error"), "half_width": lev.get("half_width")}}


def _s09_day_hindcast():
    """Day 'train late, predict early': final recipe on frames >= 2005, scored (recentred) on the earlier frames."""
    mode = "day"
    early = [y for y in _S09_FRAMES if y < 2005]
    if mode not in _S09_MODES or not early:
        return {"available": False, "reason": "no day frames before 2005"}
    w = DF["w_day"].to_numpy(np.float32)
    years = DF["year"].to_numpy(np.int64)
    train = np.flatnonzero((w > 0) & (years >= 2005))
    if train.size < 100:
        return {"available": False, "reason": "too few late training rows"}
    rounds = int(FINAL_MODELS[mode].num_boosted_rounds())
    X = DF[FEATURES].to_numpy(np.float32)
    y = DF["anom_day"].to_numpy(np.float32)
    # cross-section code (D4): the refit uses the final-model recipe of section 07 through _s08_train_booster (08)
    _h = globals().get("section_code_hash")
    key = CFG.key("XGB_PARAMS", version=_S09_VERSION, rounds=rounds, df=_S08_DF_FP, early=early,
                  code07=_h("07") if _h else None, code08=_h("08") if _h else None)

    def compute():
        booster, used = _s08_train_booster(mode, X[train], y[train], w[train], rounds, xgb_device_for(0), CFG.SEED)
        booster.set_param({"device": "cpu"})
        preds = {int(t): np.asarray(booster.inplace_predict(X[_s09_slice(t)]), np.float32) for t in early}
        del booster
        return {"preds": preds, "device": used}

    with timer("09_day_hindcast"):
        res = stage_cache("09_day_hindcast", key).get_or_compute(compute)
    fixed = _s09_fixed_cells(mode)
    later = [t for t in _S09_FRAMES if t >= 2005]
    persist = _s09_col("anom_day", later[0])
    by_frame, pool_o, pool_p, pool_q, pool_w = [], [], [], [], []
    for t in early:
        pred = res["preds"][t].astype(np.float64)
        pred = pred - np.mean(pred[fixed])
        obs = _s09_col("anom_day", t)
        wt = _s09_col("w_day", t)
        if not np.any(wt > 0):          # a frame left out of training (D8): score its valid cells unweighted
            wt = np.where(np.isfinite(obs), 1.0, 0.0)
        ok = np.isfinite(obs) & np.isfinite(persist) & (wt > 0)
        mm = _s09_metrics(obs[ok], pred[ok], wt[ok])
        mp = _s09_metrics(obs[ok], persist[ok], wt[ok])
        by_frame.append({"frame": int(t), "r2": mm["r2"], "rmse": mm["rmse"], "bias": mm["bias"], "n": mm["n"],
                         "persistence_r2": mp["r2"], "persistence_rmse": mp["rmse"],
                         "years_from_training": int(later[0] - t)})
        pool_o.append(obs[ok]), pool_p.append(pred[ok]), pool_q.append(persist[ok]), pool_w.append(wt[ok])
    po, pp, pq, pw = (np.concatenate(v) for v in (pool_o, pool_p, pool_q, pool_w))
    mm, mp = _s09_metrics(po, pp, pw), _s09_metrics(po, pq, pw)
    return {"available": True, "frames": early, "train_frames": later, "r2": mm["r2"], "rmse": mm["rmse"],
            "bias": mm["bias"], "n": mm["n"], "persistence_r2": mp["r2"], "persistence_rmse": mp["rmse"],
            "persistence_frame": later[0], "by_frame": by_frame, "recentred": True, "rounds": rounds,
            "device": res["device"]}


HINDCAST_METRICS = {}
if "night" in _S09_MODES:
    HINDCAST_METRICS["night"] = _s09_night_hindcast()
    HINDCAST_METRICS["night"]["level_regression"] = _S09_LEVEL_REG.get("night")
if "day" in _S09_MODES:
    HINDCAST_METRICS["day"] = _s09_day_hindcast()
    HINDCAST_METRICS["day"]["level_regression"] = _S09_LEVEL_REG.get("day")
for _m, _h in HINDCAST_METRICS.items():
    if _h.get("available"):
        log(f"09: {_m} hindcast: R2 {_h['r2']:.3f}, RMSE {_h['rmse']:.3f} °C (persistence R2 "
            f"{_h['persistence_r2']:.3f}, RMSE {_h['persistence_rmse']:.3f})")
    else:
        log(f"09: {_m} hindcast not available: {_h.get('reason')}")

## 9.10 Climate sensitivity β (`CLIMATE_SENSITIVITY`)

In [ ]:
def _s09_ols(X, y):
    coef, _, rank, _ = np.linalg.lstsq(X, y, rcond=None)
    return coef if rank == X.shape[1] else None


def _s09_moving_block_indices(n, block, rng):
    n_blocks = int(_s09_math.ceil(n / block))
    starts = rng.integers(0, n - block + 1, size=n_blocks)
    return (starts[:, None] + np.arange(block)[None, :]).ravel()[:n]


def _s09_beta(mode):
    """OLS dLST/dT (+ swvl1 co-regressor), moving-block bootstrap draws and Theil-Sen (SPEC §4.7)."""
    y_s, sensor = _s09_sat_series(mode, _S09_BETA_SAT[mode], drop_drift=(mode == "day"))
    t_s, sw_s = _s09_annual(_S09_TEMP_VAR[mode]), _s09_annual("era5_swvl1")
    years = sorted(set(y_s.index) & set(t_s.index) & set(sw_s.index))
    n = len(years)
    base = {"temperature_var": _S09_TEMP_VAR[mode], "regressors": [_S09_TEMP_VAR[mode], "era5_swvl1"],
            "years": [int(v) for v in years], "n_years": n, "series": sorted(set(sensor.values())),
            "method": (f"OLS of the annual NCR satellite LST on {_S09_TEMP_VAR[mode]} + era5_swvl1; moving-block "
                       f"bootstrap ({_S09_BETA_BLOCK}-year blocks, {_S09_BETA_DRAWS} draws, percentile CI); "
                       "Theil-Sen univariate slope alongside")}
    y = y_s.loc[years].to_numpy(np.float64) if n else np.zeros(0)
    T = t_s.loc[years].to_numpy(np.float64) if n else np.zeros(0)
    if n < 6 or not np.ptp(T) > 0:
        why = f"only {n} years" if n < 6 else f"{_S09_TEMP_VAR[mode]} is constant over the fit years"
        log(f"09: {mode}: {why} for beta; falling back to beta = 1 +- 0.5", "WARNING")
        draws = np.random.default_rng(CFG.SEED).normal(1.0, 0.25, _S09_BETA_DRAWS).astype(np.float32)
        return {**base, "beta": 1.0, "ci": [0.5, 1.5], "draws": draws, "r2": None, "theil_sen": None,
                "theil_sen_ci": None, "coef_swvl1": None, "fallback": True}
    S = sw_s.loc[years].to_numpy(np.float64)
    X = np.column_stack([np.ones(n), T, S])
    coef = _s09_ols(X, y)
    if coef is None:
        X = X[:, :2]
        coef = _s09_ols(X, y)
        base["regressors"] = [_S09_TEMP_VAR[mode]]
    resid = y - X @ coef
    sst = float(np.sum((y - y.mean()) ** 2))
    r2 = 1.0 - float(resid @ resid) / sst if sst > 0 else None
    sigma2 = float(resid @ resid) / max(n - X.shape[1], 1)
    se = float(np.sqrt(max(sigma2 * float(np.linalg.pinv(X.T @ X)[1, 1]), 0.0)))
    rng = np.random.default_rng(CFG.SEED + 909 + (0 if mode == "day" else 1))
    draws = []
    attempts = 0
    while len(draws) < _S09_BETA_DRAWS and attempts < 5 * _S09_BETA_DRAWS:
        attempts += 1
        idx = _s09_moving_block_indices(n, min(_S09_BETA_BLOCK, n), rng)
        c = _s09_ols(X[idx], y[idx])
        if c is not None and np.isfinite(c[1]):
            draws.append(c[1])
    draws = np.asarray(draws, dtype=np.float32)
    ts = _s09_stats.theilslopes(y, T, alpha=0.95)
    return {**base, "beta": float(coef[1]), "ci": [float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))],
            "draws": draws, "r2": r2, "se_ols": se, "coef_swvl1": float(coef[2]) if X.shape[1] == 3 else None,
            "intercept": float(coef[0]), "theil_sen": float(ts[0]), "theil_sen_ci": [float(ts[2]), float(ts[3])],
            "bootstrap_median": float(np.median(draws)), "fallback": False}


CLIMATE_SENSITIVITY = {m: _s09_beta(m) for m in CFG.MODES}
for _m, _b in CLIMATE_SENSITIVITY.items():
    log(f"09: beta {_m} = {_b['beta']:.3f} [{_b['ci'][0]:.3f}, {_b['ci'][1]:.3f}] (R2 {_b['r2']}, n {_b['n_years']}, "
        f"Theil-Sen {_b['theil_sen']}, {len(_b['draws'])} block-bootstrap draws)")

## 9.11 Frame statistics per scope (`SERIES_FRAMES`) and the hot-core threshold

In [ ]:
def _s09_scopes():
    scopes = [{"id": "ncr", "label": "Delhi NCR", "kind": "ncr"}]
    scopes += [{"id": f"d:{int(d['id'])}", "label": d["name"], "kind": "district"} for d in DISTRICTS]
    for m in _S09_MODES:
        if m in ZONE_PROFILES:
            scopes += [{"id": f"z:{m}:{p['id']}", "label": p.get("display_name") or p["name"], "kind": "zone",
                        "mode": m} for p in ZONE_PROFILES[m]]
    return scopes


SERIES_SCOPES = _s09_scopes()
_S09_DISTRICT_OF_CELL = np.asarray(DISTRICT_IDX)[CELL_ROWS, CELL_COLS].astype(np.int64)


def _s09_hot_threshold(mode):
    ref = [y for y in _S09_FRAMES if _S09_REF_PERIOD[0] <= y <= _S09_REF_PERIOD[1]] or list(_S09_FRAMES)
    vals = np.concatenate([FRAME_FIELDS[mode][y][np.isfinite(FRAME_FIELDS[mode][y])] for y in ref])
    return float(np.percentile(vals, 90)) if vals.size else float("nan"), ref


def _s09_uhi(field, fb, fw, sel):
    urban = sel & (fb >= 0.5) & np.isfinite(field)
    rural = sel & (fb <= 0.05) & (fw < 0.2) & np.isfinite(field)
    if urban.sum() < 3 or rural.sum() < 3:
        return None
    return float(field[urban].mean() - field[rural].mean())


def _s09_scope_mask(scope_id, mode, year):
    if scope_id == "ncr":
        return np.ones(_S09_NC, dtype=bool)
    if scope_id.startswith("d:"):
        return _S09_DISTRICT_OF_CELL == int(scope_id[2:])
    _, zm, zk = scope_id.split(":")
    if zm != mode:
        return None
    return np.asarray(ZONES_OBS[mode][year]) == int(zk)


def _s09_series_frames(mode):
    thr = HOT_CORE_THRESHOLD[mode]
    out, low = {}, []
    for sc in SERIES_SCOPES:
        if sc["kind"] == "zone" and sc.get("mode") != mode:
            continue
        rows = []
        for y in _S09_FRAMES:
            field = FRAME_FIELDS[mode][y].astype(np.float64)
            sel = _s09_scope_mask(sc["id"], mode, y)
            fb = np.asarray(FEATURE_GRIDS[y]["frac_built"], np.float64)[CELL_ROWS, CELL_COLS]
            fw = np.asarray(FEATURE_GRIDS[y]["frac_water"], np.float64)[CELL_ROWS, CELL_COLS]
            lev = LEVELS[mode][y]
            n_scope = int(sel.sum())                     # ROI cells of the scope at this frame (zones: assigned)
            vals = field[sel & np.isfinite(field)]
            coverage = round(vals.size / n_scope, 4) if n_scope else 0.0
            if vals.size == 0:
                rows.append({"year": y, "mean": None, "ci": None, "p90": None, "hot_core_km2": 0.0, "uhi": None,
                             "n_cells": 0, "tier": FRAME_TIERS[mode][y], "coverage": coverage})
                continue
            if sc["id"] == "ncr":
                # the NCR value is the frame LEVEL (fixed mask, composition-consistent) and its CI (D16)
                mean, ci = float(lev["level"]), [float(lev["ci"][0]), float(lev["ci"][1])]
            else:
                mean = float(vals.mean())
                ci = [mean - (lev["level"] - lev["ci"][0]), mean + (lev["ci"][1] - lev["level"])]
            hot = (float((vals >= thr).sum()) / vals.size * n_scope * _S09_RES_KM2) if np.isfinite(thr) else None
            rows.append({"year": y, "mean": mean, "ci": ci, "p90": float(np.percentile(vals, 90)),
                         "hot_core_km2": hot, "uhi": _s09_uhi(field, fb, fw, sel), "n_cells": int(vals.size),
                         "tier": FRAME_TIERS[mode][y], "coverage": coverage})
            if coverage < 0.9:
                low.append(f"{sc['id']} {y} ({coverage:.0%})")
        out[sc["id"]] = rows
    if low:
        log(f"09: {mode}: {len(low)} scope-frame rows cover < 90 % of their cells (hot-core area scaled to the "
            f"scope): " + ", ".join(low[:12]) + (" ..." if len(low) > 12 else ""))
    return out


HOT_CORE_THRESHOLD, _S09_HOT_FRAMES, SERIES_FRAMES = {}, {}, {}
with timer("09_series_frames"):
    for _m in _S09_MODES:
        HOT_CORE_THRESHOLD[_m], _S09_HOT_FRAMES[_m] = _s09_hot_threshold(_m)
        SERIES_FRAMES[_m] = _s09_series_frames(_m)
        log(f"09: {_m} hot-core threshold {HOT_CORE_THRESHOLD[_m]:.2f} °C (p90 of {_S09_HOT_FRAMES[_m]}); NCR UHI "
            + ", ".join(f"{r['year']}={r['uhi']:+.2f}" if r["uhi"] is not None else f"{r['year']}=n/a"
                        for r in SERIES_FRAMES[_m]["ncr"]))


def _s09_series_annual(mode):
    """Annual series per scope (ncr + districts): ERA5 skin temperature with anomaly, satellite values."""
    out = {}
    for sc in SERIES_SCOPES:
        if sc["kind"] == "zone":
            continue
        skin = _s09_annual(_S09_SKIN_VAR[mode], sc["id"])
        ref = skin[(skin.index >= _S09_REF_PERIOD[0]) & (skin.index <= _S09_REF_PERIOD[1])]
        ref_mean = float(ref.mean()) if len(ref) else (float(skin.mean()) if len(skin) else float("nan"))
        sat_rows = []
        for var, _, _ in _S09_LEVEL_SAT[mode]:
            s = _s09_annual(var, sc["id"])
            flags = _s09_annual_flags(var, sc["id"])
            sat_rows += [{"year": int(y), "value": float(v), "sensor": _S09_SENSOR_LABEL.get(var, var),
                          "variable": var, "flag": flags.get(int(y))} for y, v in s.items() if np.isfinite(v)]
        out[sc["id"]] = {"era5_skin": [{"year": int(y), "value": float(v), "anomaly": float(v - ref_mean)}
                                       for y, v in skin.items() if np.isfinite(v)],
                         "satellite": sorted(sat_rows, key=lambda r: (r["year"], r["variable"]))}
    return out


SERIES_ANNUAL = {m: _s09_series_annual(m) for m in _S09_MODES}

## 9.12 Trends (`TRENDS`)

In [ ]:
def _s09_theil_sen_many(x, Y):
    """Theil-Sen slopes of each row of Y (D, k) against x (k,) - median of all pairwise slopes, vectorised."""
    x = np.asarray(x, np.float64)
    i, j = np.triu_indices(x.size, k=1)
    dx = x[j] - x[i]
    keep = dx != 0
    slopes = (Y[:, j[keep]] - Y[:, i[keep]]) / dx[keep][None, :]
    return np.median(slopes, axis=1)


def _s09_level_trend(years, levels, cis, rng):
    years = np.asarray(years, np.float64)
    lev = np.asarray(levels, np.float64)
    if years.size < 3:
        return None
    sd = np.array([(c[1] - c[0]) / (2.0 * _S09_Z90) for c in cis])
    draws = lev[None, :] + rng.standard_normal((_S09_MC_DRAWS, years.size)) * sd[None, :]
    slope = float(_s09_theil_sen_many(years / 10.0, lev[None, :])[0])
    mc = _s09_theil_sen_many(years / 10.0, draws)
    return {"slope_per_decade": slope, "ci": [float(np.percentile(mc, 2.5)), float(np.percentile(mc, 97.5))],
            "period": [int(years[0]), int(years[-1])], "method": "Theil-Sen + MC", "n_frames": int(years.size),
            "frames": [int(v) for v in years]}


def _s09_annual_trend(series, y0=None, y1=None):
    s = series.dropna()
    if y0 is not None:
        s = s[(s.index >= y0) & (s.index <= y1)]
    if len(s) < 5:
        return None
    ts = _s09_stats.theilslopes(s.to_numpy(np.float64), s.index.to_numpy(np.float64), alpha=0.95)
    return {"slope_per_decade": float(ts[0] * 10.0), "ci": [float(ts[2] * 10.0), float(ts[3] * 10.0)],
            "period": [int(s.index[0]), int(s.index[-1])], "method": "Theil-Sen (95% CI)", "n_years": int(len(s))}


def _s09_trends(mode):
    rng = np.random.default_rng(CFG.SEED + 77 + (0 if mode == "day" else 1))
    obs_frames = [y for y in _S09_FRAMES if FRAME_TIERS[mode][y] in ("observed", "observed-sparse", "reconstructed")]
    obs = _s09_level_trend(obs_frames, [LEVELS[mode][y]["level"] for y in obs_frames],
                           [LEVELS[mode][y]["ci"] for y in obs_frames], rng)
    sat, _ = _s09_sat_series(mode, _S09_LEVEL_SAT[mode], drop_drift=(mode == "day"))
    annual = {"era5_skin": _s09_annual_trend(_s09_annual(_S09_SKIN_VAR[mode])),
              _S09_TEMP_VAR[mode]: _s09_annual_trend(_s09_annual(_S09_TEMP_VAR[mode])),
              "satellite": _s09_annual_trend(sat)}
    by_scope = {}
    for scope, rows in SERIES_FRAMES[mode].items():
        rr = [r for r in rows if r["year"] in obs_frames and r["mean"] is not None]
        tr = _s09_level_trend([r["year"] for r in rr], [r["mean"] for r in rr], [r["ci"] for r in rr], rng)
        entry = {"obs": tr}
        if not scope.startswith("z:"):
            sat_s, _ = _s09_sat_series(mode, _S09_LEVEL_SAT[mode], scope=scope, drop_drift=(mode == "day"))
            entry["annual"] = {"era5_skin": _s09_annual_trend(_s09_annual(_S09_SKIN_VAR[mode], scope)),
                               "satellite": _s09_annual_trend(sat_s)}
        by_scope[scope] = entry
    return {"obs": obs, "annual": annual, "by_scope": by_scope, "frames": obs_frames}


TRENDS = {}
with timer("09_trends"):
    for _m in _S09_MODES:
        TRENDS[_m] = _s09_trends(_m)
        _t = TRENDS[_m]["obs"]
        if _t:
            log(f"09: {_m} level trend {_t['slope_per_decade']:+.3f} °C/decade [{_t['ci'][0]:+.3f}, "
                f"{_t['ci'][1]:+.3f}] over {_t['frames']}")

## 9.13 Day level check (Landsat vs MODIS Terra) and zone LST means

In [ ]:
def _s09_day_level_check():
    if "day" not in _S09_MODES:
        return {"available": False}
    fixed = _s09_fixed_cells("day")
    frames, cross = [], []
    for y in _S09_FRAMES:
        if y < 2000:
            continue
        modis = np.asarray(RAW_OBS[y]["lst_day_modis"], np.float64)[CELL_ROWS, CELL_COLS]
        sel = fixed & np.isfinite(modis)
        if sel.sum() < 10:
            continue
        m_level = float(modis[sel].mean())
        lev = LEVELS["day"][y]
        drift = y >= 2022 or CFG.window(y)[1] >= 2022
        frames.append({"year": y, "landsat_level": lev["level"], "landsat_raw_level": lev["raw_level"],
                       "modis_terra_level": m_level, "diff_corrected": lev["level"] - m_level,
                       "diff_raw": lev["raw_level"] - m_level, "overpass_correction": lev["overpass_correction"],
                       "n_cells": int(sel.sum()), "flag": "drift" if drift else None})
        field = FRAME_FIELDS["day"][y].astype(np.float64)
        both = np.isfinite(field) & np.isfinite(modis)
        if both.sum() >= 10:
            a = field[both] - field[both].mean()
            b = modis[both] - modis[both].mean()
            r = float(np.corrcoef(a, b)[0, 1])
            cross.append({"year": y, "r2": r * r, "rmse": float(np.sqrt(np.mean((a - b) ** 2))), "n": int(both.sum()),
                          "flag": "drift" if drift else None})
    good = [f for f in frames if f["flag"] is None and f["year"] <= _S09_REF_PERIOD[1]]
    summ = {}
    if len(good) >= 2:
        dc = np.array([f["diff_corrected"] for f in good])
        dr = np.array([f["diff_raw"] for f in good])
        summ = {"frames": [f["year"] for f in good], "mean_offset_corrected": float(dc.mean()),
                "sd_corrected": float(dc.std(ddof=1)), "mean_offset_raw": float(dr.mean()),
                "sd_raw": float(dr.std(ddof=1)),
                "note": "Landsat (10:45 IST, corrected) minus MODIS Terra (~10:30-11:30 local solar time) fixed-mask "
                        "means; a constant offset is expected, the frame-to-frame spread measures the residual "
                        "inconsistency"}
    return {"available": bool(frames), "frames": frames, "summary": summ, "cross_sensor": cross}


DAY_LEVEL_CHECK = _s09_day_level_check()
if DAY_LEVEL_CHECK.get("summary"):
    _s = DAY_LEVEL_CHECK["summary"]
    log(f"09: DAY_LEVEL_CHECK Landsat - MODIS Terra: corrected {_s['mean_offset_corrected']:+.2f} +- "
        f"{_s['sd_corrected']:.2f} °C vs raw {_s['mean_offset_raw']:+.2f} +- {_s['sd_raw']:.2f} °C over {_s['frames']}")

# zone profiles of section 08 get their absolute means here (observed field and model prediction + level)
for _m in _S09_MODES:
    for _p in ZONE_PROFILES.get(_m, []):
        for _y in _S09_FRAMES:
            _sel = np.asarray(ZONES_OBS[_m][_y]) == _p["id"]
            _f = FRAME_FIELDS[_m][_y].astype(np.float64)[_sel]
            _p["lst_mean"]["obs"][str(_y)] = float(np.nanmean(_f)) if np.isfinite(_f).any() else None
            _pr = PRED_ANOM[_m][_y].astype(np.float64)[_sel]
            _p["lst_pred_mean"]["obs"][str(_y)] = (float(_pr.mean() + LEVELS[_m][_y]["level"]) if _pr.size
                                                   else None)

## 9.14 Synthetic mode: planted vs recovered (overpass correction, levels, night calibration, β, UHI signs)

In [ ]:
def _s09_planted_report(truth):
    report = {}
    lv = truth.get("levels") or {}
    if "day" in LEVELS:
        bias = (truth.get("overpass") or {}).get("raw_day_level_bias_c") or {}
        rows = []
        for y in _S09_FRAMES:
            planted_corr = -float(bias[y]) if y in bias else None
            rec = LEVELS["day"][y]
            rows.append({"year": y, "tier": rec["tier"], "planted_correction": planted_corr,
                         "recovered_correction": rec["overpass_correction"],
                         "true_level": lv.get("day", {}).get(y), "level": rec["level"],
                         "level_error": (rec["level"] - lv["day"][y]) if y in lv.get("day", {}) else None,
                         "in_ci": (rec["ci"][0] <= lv["day"][y] <= rec["ci"][1]) if y in lv.get("day", {}) else None})
        report["day_levels"] = rows
        print("Day overpass correction and level (planted vs recovered):")
        print(pd.DataFrame(rows).round(3).to_string(index=False))
    if "night" in LEVELS:
        rows = []
        for y in _S09_FRAMES:
            rec = LEVELS["night"][y]
            t = lv.get("night", {}).get(y)
            rows.append({"year": y, "tier": rec["tier"], "true_level": t, "level": rec["level"],
                         "ci": [round(rec["ci"][0], 2), round(rec["ci"][1], 2)],
                         "level_error": None if t is None else rec["level"] - t,
                         "in_ci": None if t is None else bool(rec["ci"][0] <= t <= rec["ci"][1])})
        report["night_levels"] = rows
        print("Night levels (planted vs recovered; modelled 1990/1995 from the ERA5 regression):")
        print(pd.DataFrame(rows).round(3).to_string(index=False))
    cal_rows = []
    for kind, spec in (truth.get("calibration") or {}).items():
        rec = {int(c["class"]): c for c in (NIGHT_CALIBRATION.get(kind) or {}).get("by_class", [])}
        for p in spec.get("planted_by_class", []):
            r = rec.get(int(p["class"]))
            cal_rows.append({"kind": kind, "class": p["class"], "planted_b": p["b"],
                             "expected_ols_b": p.get("expected_ols_b"), "recovered_b": None if r is None else r["b"],
                             "planted_a": p["a"], "recovered_a": None if r is None else r["a"],
                             "n": None if r is None else r.get("n")})
    if cal_rows:
        report["night_calibration"] = cal_rows
        print("Night calibration slopes (planted, expected OLS after attenuation, recovered):")
        print(pd.DataFrame(cal_rows).round(3).to_string(index=False))
    beta_rows = []
    for m, b in CLIMATE_SENSITIVITY.items():
        spec = (truth.get("beta") or {}).get(m) or {}
        planted = spec.get("value")
        exp = (spec.get("expected_ols_noise_free") or {}).get("beta")
        beta_rows.append({"mode": m, "planted": planted, "expected_ols_noise_free": exp, "recovered": b["beta"],
                          "ci": [round(b["ci"][0], 3), round(b["ci"][1], 3)],
                          "planted_in_ci": None if planted is None else bool(b["ci"][0] <= planted <= b["ci"][1]),
                          "expected_in_ci": None if exp is None else bool(b["ci"][0] <= exp <= b["ci"][1]),
                          "theil_sen": b["theil_sen"], "n_years": b["n_years"]})
    report["beta"] = beta_rows
    print("Climate sensitivity beta (planted vs recovered):")
    print(pd.DataFrame(beta_rows).to_string(index=False))
    uhi_rows = []
    realised = truth.get("uhi") or {}
    for m in _S09_MODES:
        for r in SERIES_FRAMES[m]["ncr"]:
            t = (realised.get(m) or {}).get("realised", {}).get(r["year"])
            uhi_rows.append({"mode": m, "year": r["year"], "tier": r["tier"], "uhi": r["uhi"], "planted_realised": t,
                             "sign": (realised.get(m) or {}).get("sign")})
    report["uhi"] = uhi_rows
    tab = pd.DataFrame(uhi_rows)
    print("Surface UHI (urban - rural, signed) by frame:")
    print(tab.round(3).to_string(index=False))
    ok_day = all(r["uhi"] is not None and r["uhi"] < 0 for r in uhi_rows if r["mode"] == "day")
    ok_night = all(r["uhi"] is not None and r["uhi"] > 0 for r in uhi_rows if r["mode"] == "night")
    print(f"UHI signs: day < 0 in every frame: {ok_day}; night > 0 in every frame: {ok_night}")
    report["uhi_signs_ok"] = {"day": ok_day, "night": ok_night}
    return report


PLANTED_VS_RECOVERED_09 = (_s09_planted_report(SYNTHETIC_TRUTH)
                           if isinstance(globals().get("SYNTHETIC_TRUTH"), dict) and SYNTHETIC_TRUTH else {})

## 9.15 Figures

In [ ]:
_S09_FIG_DIR = Path(CFG.FIG_DIR)
_S09_FIG_DIR.mkdir(parents=True, exist_ok=True)
_S09_TIER_MARKER = {"observed": "o", "observed-sparse": "s", "reconstructed": "D", "modelled": "^"}


def _s09_save_fig(fig, name):
    fig.savefig(_S09_FIG_DIR / name, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def _s09_fig_levels():
    fig, axes = plt.subplots(1, len(_S09_MODES), figsize=(7 * len(_S09_MODES), 4.2), squeeze=False)
    for ax, m in zip(axes[0], _S09_MODES):
        for tier, mk in _S09_TIER_MARKER.items():
            ys = [y for y in _S09_FRAMES if LEVELS[m][y]["tier"] == tier]
            if not ys:
                continue
            lv = np.array([LEVELS[m][y]["level"] for y in ys])
            lo = lv - np.array([LEVELS[m][y]["ci"][0] for y in ys])
            hi = np.array([LEVELS[m][y]["ci"][1] for y in ys]) - lv
            ax.errorbar(ys, lv, yerr=np.vstack([lo, hi]), fmt=mk, ms=6, capsize=3, label=f"frame level ({tier})")
        sat, _ = _s09_sat_series(m, _S09_LEVEL_SAT[m])
        if len(sat):
            ax.plot(sat.index, sat.values, color="#64748b", lw=1, label="annual satellite (NCR)")
        skin = _s09_annual(_S09_SKIN_VAR[m])
        if len(skin):
            ax.plot(skin.index, skin.values, color="#f59e0b", lw=1, ls="--", label=f"{_S09_SKIN_VAR[m]} (NCR)")
        if TRENDS.get(m, {}).get("obs"):
            t = TRENDS[m]["obs"]
            ax.set_title(f"{m}: levels; trend {t['slope_per_decade']:+.2f} °C/decade [{t['ci'][0]:+.2f}, "
                         f"{t['ci'][1]:+.2f}]", fontsize=10)
        ax.set_ylabel("°C")
        ax.legend(fontsize=7, frameon=False)
    _s09_save_fig(fig, "09_levels.png")


def _s09_fig_beta():
    fig, axes = plt.subplots(1, len(CLIMATE_SENSITIVITY), figsize=(6 * len(CLIMATE_SENSITIVITY), 3.8), squeeze=False)
    for ax, (m, b) in zip(axes[0], CLIMATE_SENSITIVITY.items()):
        ax.hist(b["draws"], bins=40, color="#22d3ee", alpha=0.8)
        ax.axvline(b["beta"], color="#0f172a", lw=2, label=f"OLS {b['beta']:.2f}")
        ax.axvspan(b["ci"][0], b["ci"][1], color="#a78bfa", alpha=0.2, label="95% block-bootstrap CI")
        if b.get("theil_sen") is not None:
            ax.axvline(b["theil_sen"], color="#fb7185", ls="--", label=f"Theil-Sen {b['theil_sen']:.2f}")
        planted = ((SYNTHETIC_TRUTH or {}).get("beta") or {}).get(m, {}).get("value") if isinstance(
            globals().get("SYNTHETIC_TRUTH"), dict) else None
        if planted is not None:
            ax.axvline(planted, color="#16a34a", ls=":", lw=2, label=f"planted {planted:.2f}")
        ax.set_title(f"{m}: dLST/d{b['temperature_var']} (n={b['n_years']})")
        ax.legend(fontsize=7, frameon=False)
    _s09_save_fig(fig, "09_beta.png")


def _s09_fig_modelled():
    m = "night"
    frames = [y for y in _S09_FRAMES if m in FRAME_TIERS and FRAME_TIERS[m].get(y) == "modelled"]
    if m not in FRAME_FIELDS or not frames:
        return
    y = frames[0]
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
    for ax, (title, vals) in zip(axes, [(f"night {y} modelled field (°C)", FRAME_FIELDS[m][y]),
                                        ("p95 - p05 (°C)", FRAME_UNC[m][y]["p95"] - FRAME_UNC[m][y]["p05"]),
                                        ("outside AOA", FRAME_AOA_OUT[m][y].astype(np.float64))]):
        grid = np.full(GRID.shape, np.nan)
        grid[CELL_ROWS, CELL_COLS] = vals
        im = ax.imshow(grid, cmap="inferno" if "field" in title else "viridis")
        ax.set_axis_off()
        ax.set_title(title)
        fig.colorbar(im, ax=ax, shrink=0.7)
    _s09_save_fig(fig, f"09_night_{y}_modelled.png")


def _s09_fig_uhi():
    fig, ax = plt.subplots(figsize=(7, 3.6))
    for m in _S09_MODES:
        rows = [r for r in SERIES_FRAMES[m]["ncr"] if r["uhi"] is not None]
        ax.plot([r["year"] for r in rows], [r["uhi"] for r in rows], marker="o", lw=2, label=m)
    ax.axhline(0, color="#475569", lw=0.8)
    ax.set_ylabel("surface UHI, urban - rural (°C)")
    ax.set_title("Signed surface UHI by frame (NCR)")
    ax.legend(frameon=False)
    _s09_save_fig(fig, "09_uhi.png")


def _s09_fig_day_check():
    if not DAY_LEVEL_CHECK.get("frames"):
        return
    fr = DAY_LEVEL_CHECK["frames"]
    fig, ax = plt.subplots(figsize=(7, 3.6))
    ax.plot([f["year"] for f in fr], [f["diff_raw"] for f in fr], marker="s", label="raw Landsat - MODIS Terra")
    ax.plot([f["year"] for f in fr], [f["diff_corrected"] for f in fr], marker="o",
            label="overpass-corrected Landsat - MODIS Terra")
    ax.set_ylabel("°C")
    ax.set_title("Day level check (fixed-mask means)")
    ax.legend(frameon=False, fontsize=8)
    _s09_save_fig(fig, "09_day_level_check.png")


with timer("09_figures"):
    for _s09_fn in (_s09_fig_levels, _s09_fig_beta, _s09_fig_modelled, _s09_fig_uhi, _s09_fig_day_check):
        try:
            _s09_fn()
        except Exception as _s09_exc:  # noqa: BLE001 - a broken figure must never lose the computed results
            log(f"09: figure {_s09_fn.__name__} failed: {type(_s09_exc).__name__}: {_s09_exc}", "WARNING")
            plt.close("all")

## 9.16 Contract check (SPEC §3.6)

In [ ]:
def _s09_contract_check():
    n = _S09_NC
    for m in _S09_MODES:
        assert set(LEVELS[m]) == set(_S09_FRAMES), (m, sorted(LEVELS[m]))
        for y, rec in LEVELS[m].items():
            assert {"level", "ci", "method", "overpass_correction"} <= set(rec), (m, y)
            assert np.isfinite(rec["level"]) and rec["ci"][0] <= rec["level"] <= rec["ci"][1], (m, y, rec)
            assert (rec["overpass_correction"] is None) == (m != "day" or rec["tier"] == "modelled"), (m, y)
        for y in _S09_FRAMES:
            tier = FRAME_TIERS[m][y]
            assert tier in _S09_TIERS, (m, y, tier)
            f = FRAME_FIELDS[m][y]
            assert f.shape == (n,) and f.dtype == np.float32, (m, y, f.shape, f.dtype)
            if tier == "modelled":
                assert np.isfinite(f).all(), f"{m} {y}: modelled field must be complete"
            else:
                obs = _s09_col(f"lst_{m}", y)
                assert np.array_equal(np.isfinite(f), np.isfinite(obs)), f"{m} {y}: observed NaN pattern"
            a = FRAME_AOA_OUT[m][y]
            assert a.shape == (n,) and a.dtype == bool
            if tier in ("modelled", "reconstructed"):
                u = FRAME_UNC[m][y]
                assert u["p05"].shape == (n,) and u["p95"].shape == (n,) and u["p05"].dtype == np.float32
                okc = np.isfinite(f)
                assert np.all(u["p05"][okc] <= f[okc] + 1e-4) and np.all(f[okc] <= u["p95"][okc] + 1e-4)
            else:
                assert y not in FRAME_UNC[m], (m, y)
        if m == "day" and LEVELS[m]:
            tiers = {FRAME_TIERS[m][y] for y in _S09_FRAMES}
            assert tiers <= {"observed", "observed-sparse"}, f"day tiers must be observed(-sparse): {tiers}"
        for y, rec in LEVELS[m].items():
            assert 0.0 <= rec["fixed_mask_coverage"] <= 1.0, (m, y)
            if rec["tier"] != "modelled":
                cc = rec["composition_correction"]
                assert cc is not None and np.isfinite(cc), (m, y)
                assert cc == 0.0 or rec["fixed_mask_coverage"] < 0.999, (m, y, cc)
            else:
                assert "drift_correction_c" in rec, (m, y)
        for y in CFG.FRAMES_PROJ:
            assert FRAME_TIERS[m][int(y)] == "projected"
        for scope, rows in SERIES_FRAMES[m].items():
            assert [r["year"] for r in rows] == list(_S09_FRAMES), (m, scope)
            for r in rows:
                assert set(r) == {"year", "mean", "ci", "p90", "hot_core_km2", "uhi", "n_cells", "tier", "coverage"}
                assert 0.0 <= r["coverage"] <= 1.0, (m, scope, r)
            for r in (rows if scope == "ncr" else []):
                lev = LEVELS[m][r["year"]]
                assert r["mean"] is None or (abs(r["mean"] - lev["level"]) < 1e-9 and r["ci"] == lev["ci"]), r
        assert np.isfinite(HOT_CORE_THRESHOLD[m])
        assert "ncr" in SERIES_FRAMES[m] and all(f"d:{d['id']}" in SERIES_FRAMES[m] for d in DISTRICTS)
    for m, b in CLIMATE_SENSITIVITY.items():
        assert {"beta", "ci", "draws", "r2", "n_years", "regressors", "method", "temperature_var"} <= set(b)
        assert len(b["draws"]) >= 1000 and b["temperature_var"] == _S09_TEMP_VAR[m]
        assert b["ci"][0] <= b["ci"][1]
    assert isinstance(HINDCAST_METRICS, dict) and isinstance(DAY_LEVEL_CHECK, dict)


_s09_contract_check()
log("09: contract check passed (SPEC §3.6: levels, tiers, frame fields, uncertainty, AOA flags, beta, series)")
log("Section 09 complete: LEVELS, FRAME_TIERS, FRAME_FIELDS, FRAME_UNC, FRAME_AOA_OUT, CLIMATE_SENSITIVITY, TRENDS, "
    "SERIES_FRAMES, SERIES_SCOPES, SERIES_ANNUAL, HOT_CORE_THRESHOLD, HINDCAST_METRICS, DAY_LEVEL_CHECK ready")
free_memory()

# 10. Fine-resolution fields (250 m): day Landsat, night DMS downscaling, validation and EPSG:4326 rasters

The 1 km grid is the analysis grid of sections 06–09. The dashboard additionally shows every observed frame on
the **fine grid** `GRID_FINE` (250 m at the 1 km grid; fine pixel (r, c) lies in cell (r // 4, c // 4)).

**Day** is natively fine: the harmonised Landsat C2 L2 composite `lst_day_f` exists at 250 m (mean of 30 m
pixels), so the day fine field is *observed*: `lst_day_f` + the frame's overpass correction (SPEC §4.7, the same
correction section 09 applies to the 1 km level), with fine pixels resting on fewer than two clear scenes masked.

**Night** exists only at 1 km (MODIS Aqua / calibrated VIIRS / calibrated Terra; modelled for 1990/1995), so it is
**downscaled** with a Data-Mining-Sharpener-style regression (Gao, Kustas & Anderson 2012; Hutengs & Vohland 2016):

1. Fine predictors $P_f$ (10): `ndvi_f, ndbi_f, mndwi_f, albedo_f, built_f, water_occ_f, elevation_f`, the fine
   topographic position `tpi_f = elevation_f − box7(elevation_f)` (cold-air pooling) and two neighbourhood terms,
   `box5(ndvi_f)` and `box9(built_f)` (≈ 1.25 km / 2.25 km context at 250 m). Coarse predictors $P_c$ are the
   exact 4 × 4 block means of $P_f$, so the regression learnt at 1 km is applied to the same quantities at 250 m.
2. XGBoost (depth 6, 600 rounds, η 0.05) learns $T_c = g(P_c)$ on the 1 km cells with a valid night value,
   **excluding the 20 % most heterogeneous cells** (sum of the z-scored within-cell standard deviations of NDVI and
   built-up fraction): inside a mixed cell the 1 km mean is a poor label for the sub-pixel predictor mixture.
3. $\hat T_f = g(P_f)$; residual $r_c = T_c − \overline{\hat T_f}^{(4×4)}$ is interpolated **bilinearly** at the
   fine-pixel centres (a constant per-cell redistribution would leave block artefacts) and added.
4. Three iterations of **block conservation** $e_c = T_c − \overline{T_f}^{(4×4)}$, $T_f \mathrel{+}= e_c$
   (nearest-upsampled) make every 4 × 4 block average back to the 1 km value: $|e_c| < 0.01$ °C is asserted. The
   downscaled field therefore *adds* sub-pixel structure without changing a single 1 km number.

Observed night frames are downscaled where the 1 km value exists (NaN elsewhere); the modelled 1990/1995 frames
downscale the complete modelled 1 km field of section 09.

**Validation (SPEC §4.8).**
* *Scale invariance* (Bartkowiak et al. 2019): aggregate the 1 km night anomalies and the 1 km predictors to
  4 × 4 coarse cells (4 km at the 1 km grid), train at the coarse scale on the block20 folds ≠ k (strictly: coarse
  cells containing any valid fold-k cell are excluded from training), downscale coarse → 1 km and score the held-out
  fold-k 1 km cells against the true 1 km values — against the bilinear interpolation of the coarse field as the
  baseline. The test assumes the predictor–LST relation is similar across the 4 km → 1 km and 1 km → 250 m steps.
* *Day demonstration*: MODIS Terra day LST (1 km, 2010/2015/2020) is downscaled to the fine grid with the same
  algorithm and compared, as anomalies (each product minus its own mean over the common pixels, which removes the
  overpass/algorithm offset), with the Landsat fine composite — pooled and by dominant GLC class, vs bilinear.

**Rasters.** Each fine field is reprojected from UTM 43N to a regular **EPSG:4326** lon/lat grid (0.0025° ≈ 250 m
and a 0.005° "500 m" preview at the 1 km analysis grid; both scale with the fine-pixel size) by **inverse
mapping**: every output pixel centre is transformed with pyproj (in memory-bounded row chunks) and takes the
nearest fine pixel. Placing UTM imagery on Mercator bounds would shift pixels by up to ~4 km at the NCR's edges.
PNG (8-bit RGB, Pillow): R = 0 nodata, else $1 + \mathrm{round}((v − lo)/(hi − lo) · 254)$ clipped to 1…255;
G = flags (`FINE_FLAGS`: bit 0 outside the AOA, bits 1–2 tier code 0 observed / 1 observed-sparse or
reconstructed / 2 modelled / 3 downscaled); B = 0. The value domain is **fixed per mode**: p0.5–p99.5 of all
observed fine fields of the mode, padded by 3 °C (room for the projected warming, so a projected raster never
clips) and rounded outward to 1 °C.

Outputs (SPEC §3.7): `FINE_FIELDS[m][year]` (Hf, Wf) float32 °C, `FINE_FLAGS[m][year]` uint8,
`DOWNSCALING_METRICS`, `RASTER_INDEX` and the PNGs under `CFG.WEB_DIR/raster/` (+ `raster_index.json`).

In [ ]:
_lst_enter_section('10')  # section marker (generated by build_notebook.py)

In [ ]:
import hashlib as _s10_hashlib
import math as _s10_math
import time as _s10_time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xgboost as xgb
from scipy import ndimage as _s10_ndi

_S10_VERSION = "10-v2.0"
_S10_F = int(CFG.FINE_FACTOR)
_S10_NC = int(N_CELLS)
_S10_H, _S10_W = (int(v) for v in ROI_MASK.shape)
_S10_HF, _S10_WF = _S10_H * _S10_F, _S10_W * _S10_F
_S10_FRAMES = sorted(int(y) for y in np.unique(DF["year"].to_numpy()))
_S10_MODES = [m for m in CFG.MODES if m in FRAME_FIELDS]
_S10_FINE_RES_M = float(GRID_FINE.res)
_S10_PRED_NAMES = ["ndvi_f", "ndbi_f", "mndwi_f", "albedo_f", "built_f", "water_occ_f", "elevation_f", "tpi_f",
                   "ndvi_f_box5", "built_f_box9"]
_S10_HET_IDX = (_S10_PRED_NAMES.index("ndvi_f"), _S10_PRED_NAMES.index("built_f"))
# SPEC §4.8 regression: XGBoost depth 6, 600 rounds, eta 0.05 (remaining settings = robust defaults for ~10^4 rows)
_S10_XGB_PARAMS = {"objective": "reg:squarederror", "tree_method": "hist", "max_depth": 6, "learning_rate": 0.05,
                   "subsample": 0.8, "colsample_bytree": 0.9, "min_child_weight": 5, "reg_lambda": 1.0,
                   "max_bin": 256}
_S10_ROUNDS = 600
_S10_HET_DROP = 0.20                  # share of most heterogeneous cells excluded from training
_S10_CONSERVE_ITERS = 3
_S10_CONSERVE_TOL_C = 0.01            # asserted max |block mean - 1 km value|
_S10_MIN_TRAIN = 30                   # fewer training cells -> no regression (bilinear + conservation only)
_S10_SCALE_FACTOR = 4                 # scale-invariance test: 4 x 4 cells (4 km at the 1 km grid)
_S10_SCALE_MIN_VALID = 0.5            # a coarse test cell needs >= 50 % valid 1 km cells
_S10_DEMO_YEARS = [2010, 2015, 2020]  # MODIS Terra day demonstration (SPEC §4.8 b)
_S10_DAY_MIN_SCENES = 2.0             # fine day pixels resting on < 2 clear scenes are NaN (SPEC §4.8)
_S10_TIER_CODE = {"observed": 0, "observed-sparse": 1, "reconstructed": 1, "modelled": 2, "downscaled": 3}
_S10_RES_SCALE = _S10_FINE_RES_M / 250.0
_S10_RASTER_RES_DEG = round(0.0025 * _S10_RES_SCALE, 10)   # 0.0025 deg at the 250 m fine grid (SPEC §4.8)
_S10_PREVIEW_RES_DEG = round(0.005 * _S10_RES_SCALE, 10)   # "500 m" preview
_S10_DOMAIN_PAD_C = 3.0
_S10_DOMAIN_Q = (0.5, 99.5)
_S10_MAP_CHUNK_PX = 1 << 21           # output pixels per pyproj inverse-mapping chunk (~2 M -> ~50 MB peak)
_S10_PRED_CHUNK = 1 << 20             # fine pixels per XGBoost prediction call
_S10_RASTER_DIR = Path(CFG.WEB_DIR) / "raster"
_S10_FIG_DIR = Path(CFG.FIG_DIR)
_S10_GLC_KEYS = ["glc_built", "glc_forest", "glc_water", "glc_cropland", "glc_barren", "glc_otherveg"]
_S10_CLASS_LABEL = {k + 1: LC_CLASS_NAMES.get(k + 1, f"class {k + 1}") for k in range(6)}
if len(DF) != len(_S10_FRAMES) * _S10_NC:
    raise RuntimeError("10: DF is not the (frame x canonical cell) table of section 06")
if GRID_FINE.shape != (_S10_HF, _S10_WF) or ROI_MASK_FINE.shape != (_S10_HF, _S10_WF):
    raise RuntimeError(f"10: GRID_FINE {GRID_FINE.shape} is not the {_S10_F}x refinement of GRID {GRID.shape}")
log(f"10: fine grid {_S10_WF}x{_S10_HF} @ {_S10_FINE_RES_M:g} m (factor {_S10_F}); modes {_S10_MODES}; frames "
    f"{_S10_FRAMES}; rasters {_S10_RASTER_RES_DEG:g} deg (+ {_S10_PREVIEW_RES_DEG:g} deg preview)")

## 10.1 Grid operators
Block statistics reshape a fine raster to `(h, f, w, f)` and reduce over the two `f` axes, optionally over a
validity mask — the block mean "over the pixels that carry the field" is what block conservation must preserve.
Bilinear upsampling is separable: coarse cell centres sit at fine coordinate $(R + ½)f$, so a fine pixel centre
$i + ½$ has coarse coordinate $u = (i + ½)/f − ½$ and weights $(1 − \{u\}, \{u\})$ on its two neighbours
(clamped at the edges). NaN-aware box means use the ratio of filtered value·mask and filtered mask (zero padding),
i.e. the mean of the valid pixels inside the window.

In [ ]:
def _s10_cell_grid(values, fill=np.nan, dtype=np.float32):
    """(N_CELLS,) canonical vector -> (H, W) grid (``fill`` outside the ROI)."""
    out = np.full((_S10_H, _S10_W), fill, dtype=dtype)
    out[CELL_ROWS, CELL_COLS] = np.asarray(values).astype(dtype, copy=False)
    return out


def _s10_block_mean(fine, f, valid=None):
    """NaN-aware mean of every f x f block (optionally only over ``valid`` pixels) -> (mean float64, count int)."""
    fine = np.asarray(fine)
    h, w = fine.shape[0] // f, fine.shape[1] // f
    if fine.shape != (h * f, w * f):
        raise ValueError(f"raster {fine.shape} is not a multiple of {f}")
    blocks = fine.reshape(h, f, w, f)
    ok = np.isfinite(blocks)
    if valid is not None:
        ok &= np.asarray(valid, bool).reshape(h, f, w, f)
    n = ok.sum(axis=(1, 3))
    s = np.where(ok, blocks, 0.0).sum(axis=(1, 3), dtype=np.float64)
    with np.errstate(invalid="ignore", divide="ignore"):
        mean = s / n
    mean[n == 0] = np.nan
    return mean, n


def _s10_block_std(fine, f):
    """NaN-aware population standard deviation of every f x f block (NaN with < 2 valid pixels)."""
    fine = np.asarray(fine, np.float64)
    h, w = fine.shape[0] // f, fine.shape[1] // f
    blocks = fine.reshape(h, f, w, f)
    ok = np.isfinite(blocks)
    n = ok.sum(axis=(1, 3))
    x = np.where(ok, blocks, 0.0)
    with np.errstate(invalid="ignore", divide="ignore"):
        m1 = x.sum(axis=(1, 3)) / n
        m2 = (x * x).sum(axis=(1, 3)) / n
    sd = np.sqrt(np.maximum(m2 - m1 * m1, 0.0))
    sd[n < 2] = np.nan
    return sd


def _s10_upsample(coarse, f):
    """Nearest (block-repeat) upsampling by an integer factor."""
    return np.repeat(np.repeat(np.asarray(coarse), f, axis=0), f, axis=1)


def _s10_lin_axis(n_coarse, f):
    u = (np.arange(n_coarse * f, dtype=np.float64) + 0.5) / f - 0.5
    i0 = np.clip(np.floor(u).astype(np.int64), 0, n_coarse - 1)
    i1 = np.clip(i0 + 1, 0, n_coarse - 1)
    w = np.clip(u - i0, 0.0, 1.0)
    return i0, i1, w


def _s10_bilinear_up(coarse, f):
    """Separable bilinear interpolation of a COMPLETE (finite) coarse raster at the fine-pixel centres (float32)."""
    c = np.asarray(coarse, np.float32)
    r0, r1, wr = _s10_lin_axis(c.shape[0], f)
    c0, c1, wc = _s10_lin_axis(c.shape[1], f)
    wr, wc = wr.astype(np.float32), wc.astype(np.float32)
    tmp = c[r0] * (1.0 - wr)[:, None] + c[r1] * wr[:, None]
    out = tmp[:, c0] * (1.0 - wc)[None, :] + tmp[:, c1] * wc[None, :]
    return out.astype(np.float32, copy=False)


def _s10_fill_nearest(a):
    """Replace NaN by the nearest finite value (Euclidean); all-NaN -> zeros."""
    a = np.asarray(a, np.float64)
    ok = np.isfinite(a)
    if ok.all():
        return a.copy()
    if not ok.any():
        return np.zeros_like(a)
    idx = _s10_ndi.distance_transform_edt(~ok, return_distances=False, return_indices=True)
    return a[tuple(idx)]


def _s10_box_mean(x, size, valid):
    """NaN-aware box mean over ``valid`` pixels (float32 output, NaN where the window has no valid pixel)."""
    x = np.asarray(x, np.float64)
    m = np.asarray(valid, bool) & np.isfinite(x)
    num = _s10_ndi.uniform_filter(np.where(m, x, 0.0), size=int(size), mode="constant", cval=0.0)
    den = _s10_ndi.uniform_filter(m.astype(np.float64), size=int(size), mode="constant", cval=0.0)
    with np.errstate(invalid="ignore", divide="ignore"):
        out = num / den
    out[den < 0.5 / (size * size)] = np.nan
    return out.astype(np.float32)


def _s10_pad_to(a, f, fill):
    """Pad a 2-D (or leading-axis-stacked 3-D) raster at the bottom/right to multiples of f."""
    a = np.asarray(a)
    h, w = a.shape[-2:]
    ph, pw = (-h) % f, (-w) % f
    if not (ph or pw):
        return a
    pad = [(0, 0)] * (a.ndim - 2) + [(0, ph), (0, pw)]
    return np.pad(a, pad, mode="constant", constant_values=fill)


def _s10_dominant_class(year):
    """(H, W) uint8 dominant GLC class (1 built … 6 other vegetation; 0 outside the ROI / no data)."""
    raw = RAW_OBS[year]
    stack = np.stack([np.nan_to_num(np.asarray(raw[k], np.float64), nan=-1.0) for k in _S10_GLC_KEYS])
    dom = (np.argmax(stack, axis=0) + 1).astype(np.uint8)
    dom[~ROI_MASK | (stack.max(axis=0) < 0)] = 0
    return dom

## 10.2 Fine predictors and the generic DMS routine
`_s10_dms` is written for *any* pair of nested grids (factor f), so the operational 1 km → 250 m step, the
4 km → 1 km scale-invariance test and the MODIS-day demonstration run the very same code.

In [ ]:
_S10_STATIC = {}


def _s10_static_fine():
    """Static fine predictors (computed once): water occurrence, elevation, fine TPI (elevation - box7)."""
    if not _S10_STATIC:
        roi_f = ROI_MASK_FINE
        elev = np.where(roi_f, np.asarray(RAW_STATIC["elevation_f"], np.float64), np.nan)
        _S10_STATIC["water_occ_f"] = np.where(roi_f, RAW_STATIC["water_occ_f"], np.nan).astype(np.float32)
        _S10_STATIC["elevation_f"] = elev.astype(np.float32)
        _S10_STATIC["tpi_f"] = (elev - _s10_box_mean(elev, 7, roi_f)).astype(np.float32)
    return _S10_STATIC


def _s10_fine_predictors(year):
    """(10, Hf, Wf) float32 fine predictor stack of one frame (NaN outside the ROI / where Landsat has no data)."""
    raw = RAW_OBS[year]
    roi_f = ROI_MASK_FINE
    st = _s10_static_fine()
    out = np.empty((len(_S10_PRED_NAMES), _S10_HF, _S10_WF), dtype=np.float32)
    for k, name in enumerate(_S10_PRED_NAMES[:5]):
        out[k] = np.where(roi_f, raw[name], np.nan)
    out[5] = st["water_occ_f"]
    out[6] = st["elevation_f"]
    out[7] = st["tpi_f"]
    out[8] = _s10_box_mean(out[0], 5, roi_f)
    out[9] = _s10_box_mean(out[4], 9, roi_f)
    out[:, ~roi_f] = np.nan
    return out


def _s10_coarse_predictors(pred_f, f):
    """Block means of every fine predictor -> (k, h, w) float32."""
    return np.stack([_s10_block_mean(p, f)[0] for p in pred_f]).astype(np.float32)


def _s10_heterogeneity(pred_f, f, cand):
    """Sum of z-scored within-cell std of NDVI and built-up fraction (over candidate cells; NaN std -> 0)."""
    score = np.zeros(cand.shape, np.float64)
    for k in _S10_HET_IDX:
        sd = _s10_block_std(pred_f[k], f)
        v = sd[cand & np.isfinite(sd)]
        if v.size >= 2 and np.std(v) > 0:
            score += np.nan_to_num((sd - v.mean()) / v.std(), nan=0.0)
    return score


def _s10_xgb_fit(X, y, device, seed):
    """Train the SPEC §4.8 regressor; a CUDA OOM retries on the CPU. Returns (booster, device used)."""
    for dev in dict.fromkeys([device, "cpu"]):
        params = dict(_S10_XGB_PARAMS, device=dev, seed=int(seed))
        try:
            dtrain = xgb.DMatrix(X, label=y, missing=np.nan)
            booster = xgb.train(params, dtrain, num_boost_round=_S10_ROUNDS, verbose_eval=False)
            del dtrain
            return booster, dev
        except Exception as exc:  # noqa: BLE001
            if dev == "cpu" or not is_gpu_oom_error(exc):
                raise
            log(f"10: GPU OOM in the DMS regression on {dev}; retrying on the CPU", "WARNING")
            free_memory()
    raise RuntimeError("unreachable")


def _s10_xgb_predict(booster, X):
    """Chunked in-place prediction (float32)."""
    out = np.empty(X.shape[0], dtype=np.float32)
    for s in range(0, X.shape[0], _S10_PRED_CHUNK):
        out[s:s + _S10_PRED_CHUNK] = booster.inplace_predict(np.ascontiguousarray(X[s:s + _S10_PRED_CHUNK]))
    return out


def _s10_dms(target_c, pred_f, f, train_c, fine_valid, *, device="cpu", seed=0, label="", keep_unobserved=False):
    """Data-Mining-Sharpener downscaling of ``target_c`` (h, w) by factor ``f`` with fine predictors ``pred_f``.

    Parameters
    ----------
    target_c : (h, w) coarse field (NaN = no observation).
    pred_f : (k, h*f, w*f) float32 fine predictors (NaN allowed; XGBoost routes missing values).
    train_c : (h, w) bool, coarse cells allowed as training rows (further restricted to finite targets and to the
        80 % least heterogeneous of them).
    fine_valid : (h*f, w*f) bool, fine pixels that carry the field; block means (residuals, conservation) run over
        these pixels only.
    keep_unobserved : keep the regression value (without residual correction) in coarse cells without a target;
        otherwise those fine pixels are NaN.

    Returns ``(field (h*f, w*f) float32, model-only field float32, info dict)``.
    """
    target_c = np.asarray(target_c, np.float64)
    fine_valid = np.asarray(fine_valid, bool)
    _, n_valid_c = _s10_block_mean(np.where(fine_valid, 0.0, np.nan), f)
    obs_c = np.isfinite(target_c) & (n_valid_c > 0)
    pred_c = _s10_coarse_predictors(pred_f, f)
    cand = np.asarray(train_c, bool) & obs_c
    het = _s10_heterogeneity(pred_f, f, cand)
    n_cand = int(cand.sum())
    if n_cand >= _S10_MIN_TRAIN:
        cut = float(np.quantile(het[cand], 1.0 - _S10_HET_DROP))
        train = cand & (het <= cut)
        if train.sum() < _S10_MIN_TRAIN:
            train = cand
    else:
        train = cand
    info = {"label": label, "n_candidates": n_cand, "n_train": int(train.sum()),
            "n_excluded_heterogeneous": int(n_cand - train.sum()), "device": None, "train_r2": None,
            "train_rmse": None}
    fine_pix = np.flatnonzero(fine_valid.ravel())
    t_hat = np.full(fine_valid.size, np.nan, dtype=np.float32)
    if int(train.sum()) >= _S10_MIN_TRAIN:
        X = pred_c[:, train].T.astype(np.float32)
        y = target_c[train].astype(np.float32)
        booster, used = _s10_xgb_fit(X, y, device, seed)
        booster.set_param({"device": "cpu"})          # host arrays -> CPU prediction (no device-mismatch copy)
        fitted = _s10_xgb_predict(booster, X).astype(np.float64)
        sse = float(np.sum((y - fitted) ** 2))
        sst = float(np.sum((y - y.mean()) ** 2))
        info.update({"device": used, "train_r2": 1.0 - sse / sst if sst > 0 else None,
                     "train_rmse": float(np.sqrt(sse / y.size))})
        Xf = pred_f.reshape(pred_f.shape[0], -1)[:, fine_pix].T
        t_hat[fine_pix] = _s10_xgb_predict(booster, Xf)
        del X, Xf, booster
    else:
        # too few cells for a regression: the coarse field itself (bilinear, then conserved) is the best estimate
        log(f"10: {label}: only {int(train.sum())} training cells - no regression, bilinear + conservation only",
            "WARNING")
        t_hat[fine_pix] = 0.0
    t_hat = t_hat.reshape(fine_valid.shape)
    model_only = t_hat.copy()
    # residual correction: bilinear r_c at fine centres, then block conservation over the valid fine pixels
    bm, _ = _s10_block_mean(t_hat, f, fine_valid)
    r_c = np.where(obs_c, target_c - bm, np.nan)
    info["resid_rmse_c"] = float(np.sqrt(np.nanmean(r_c[obs_c] ** 2))) if obs_c.any() else None
    field = t_hat + _s10_bilinear_up(_s10_fill_nearest(r_c), f)
    field[~fine_valid] = np.nan
    for _ in range(_S10_CONSERVE_ITERS):
        bm, _ = _s10_block_mean(field, f, fine_valid)
        e_c = np.where(obs_c, target_c - bm, 0.0)
        field += _s10_upsample(e_c, f).astype(np.float32)
    bm, _ = _s10_block_mean(field, f, fine_valid)
    err = np.abs(target_c - bm)[obs_c]
    info["max_block_error_c"] = float(err.max()) if err.size else 0.0
    info["n_coarse_observed"] = int(obs_c.sum())
    if not keep_unobserved:
        unobs = _s10_upsample(~obs_c, f)
        field[unobs] = np.nan
        model_only[unobs] = np.nan
    field[~fine_valid] = np.nan
    return field.astype(np.float32), model_only.astype(np.float32), info

## 10.3 Fine fields per mode and frame (`FINE_FIELDS`, `FINE_FLAGS`)
Day: observed Landsat + overpass correction. Night: one DMS regression per frame (DMS is calibrated per scene /
epoch because the predictor–LST relation changes with season, moisture and sensor), spread over the GPUs with
`gpu_pool_map`.

In [ ]:
def _s10_day_fine(year):
    """Observed day fine field (°C, 10:45 IST reference) + info."""
    raw = RAW_OBS[year]
    lev = LEVELS["day"][year]
    corr = float(lev.get("overpass_correction") or 0.0)
    nobs = np.nan_to_num(np.asarray(raw["lst_day_nobs_f"], np.float64), nan=0.0)
    lst = np.asarray(raw["lst_day_f"], np.float64)
    ok = ROI_MASK_FINE & np.isfinite(lst) & (nobs >= _S10_DAY_MIN_SCENES)
    field = np.where(ok, lst + corr, np.nan).astype(np.float32)
    return field, {"overpass_correction_c": corr, "coverage": float(ok[ROI_MASK_FINE].mean()),
                   "n_valid": int(ok.sum()), "tier": FRAME_TIERS["day"][year]}


def _s10_night_job(year, gpu_id):
    """Downscale the night 1 km frame field of ``year`` (DMS) -> dict(field, info)."""
    t0 = _s10_time.perf_counter()
    tier = FRAME_TIERS["night"][year]
    coarse = _s10_cell_grid(FRAME_FIELDS["night"][year], dtype=np.float64)
    pred_f = _s10_fine_predictors(year)
    field, _, info = _s10_dms(coarse, pred_f, _S10_F, ROI_MASK, ROI_MASK_FINE, device=xgb_device_for(gpu_id),
                              seed=int(CFG.SEED) + 1000 + int(year), label=f"night {year}")
    del pred_f
    info.update({"tier": tier, "coverage": float(np.isfinite(field[ROI_MASK_FINE]).mean()),
                 "seconds": round(_s10_time.perf_counter() - t0, 2)})
    return {"field": field, "info": info}


def _s10_flags(mode, year, field):
    """FINE_FLAGS of one frame: bit 0 outside AOA (parent 1 km cell), bits 1-2 tier code; 0 on nodata pixels."""
    tier = FRAME_TIERS[mode][year]
    code = _S10_TIER_CODE["downscaled"] if (mode == "night" and tier != "modelled") else _S10_TIER_CODE.get(tier, 2)
    aoa = _s10_cell_grid(np.asarray(FRAME_AOA_OUT[mode][year], bool), fill=False, dtype=bool)
    flags = (np.uint8(code) << np.uint8(1)) | _s10_upsample(aoa, _S10_F).astype(np.uint8)
    flags = flags.astype(np.uint8)
    flags[~np.isfinite(field)] = 0
    return flags, code

## 10.4 Validation: scale invariance (night) and the MODIS-day demonstration

In [ ]:
def _s10_fold_grid():
    """(H, W) int block20 fold of every ROI cell (-1 outside), from DF (folds are constant across frames)."""
    sl = slice(0, _S10_NC)
    return _s10_cell_grid(DF["fold_block20"].to_numpy(np.int64)[sl], fill=-1, dtype=np.int64)


def _s10_scale_job(year, gpu_id):
    """Scale-invariance test of one night frame: coarse (4x) -> 1 km on every held-out block20 fold."""
    sl = slice(_S10_FRAMES.index(year) * _S10_NC, (_S10_FRAMES.index(year) + 1) * _S10_NC)
    anom = _s10_cell_grid(DF["anom_night"].to_numpy(np.float64)[sl], dtype=np.float64)
    pred_f = _s10_fine_predictors(year)
    pred_1km = _s10_coarse_predictors(pred_f, _S10_F)           # the "fine" level of the test = 1 km
    del pred_f
    k = _S10_SCALE_FACTOR
    folds = _s10_pad_to(_s10_fold_grid(), k, -1)
    a1 = _s10_pad_to(anom, k, np.nan)
    p1 = _s10_pad_to(pred_1km, k, np.nan).astype(np.float32)
    dom = _s10_pad_to(_s10_dominant_class(year), k, 0)
    valid1 = np.isfinite(a1)
    a4, n4 = _s10_block_mean(a1, k)
    a4[n4 < _S10_SCALE_MIN_VALID * k * k] = np.nan
    bil = _s10_bilinear_up(_s10_fill_nearest(a4), k)
    rows = []
    for fold in sorted(int(v) for v in np.unique(folds[folds >= 0])):
        in_fold = valid1 & (folds == fold)
        _, n_fold4 = _s10_block_mean(np.where(in_fold, 1.0, np.nan), k)
        train4 = np.isfinite(a4) & (n_fold4 == 0)
        field, model_only, info = _s10_dms(a4, p1, k, train4, valid1, device=xgb_device_for(gpu_id),
                                           seed=int(CFG.SEED) + 2000 + int(year) * 16 + fold,
                                           label=f"scale-invariance night {year} fold {fold}")
        test = in_fold & np.isfinite(field) & np.isfinite(bil)
        if test.sum() == 0:
            continue
        rows.append({"year": int(year), "fold": fold, "obs": a1[test].astype(np.float32),
                     "pred": field[test], "bil": bil[test], "model": model_only[test],
                     "cls": dom[test], "max_block_error_c": info["max_block_error_c"], "n_train": info["n_train"]})
    return rows


def _s10_metric(obs, pred):
    obs, pred = np.asarray(obs, np.float64), np.asarray(pred, np.float64)
    ok = np.isfinite(obs) & np.isfinite(pred)
    if ok.sum() < 3:
        return {"r2": None, "rmse": None, "bias": None, "n": int(ok.sum())}
    o, p = obs[ok], pred[ok]
    sst = float(np.sum((o - o.mean()) ** 2))
    sse = float(np.sum((o - p) ** 2))
    return {"r2": 1.0 - sse / sst if sst > 0 else None, "rmse": float(np.sqrt(sse / o.size)),
            "bias": float(np.mean(p - o)), "n": int(o.size)}


def _s10_by_class(obs, pred, base, cls):
    out = []
    for c in sorted(int(v) for v in np.unique(cls) if v > 0):
        sel = cls == c
        if sel.sum() < 20:
            continue
        m, b = _s10_metric(obs[sel], pred[sel]), _s10_metric(obs[sel], base[sel])
        out.append({"class": c, "label": _S10_CLASS_LABEL.get(c, str(c)), "n": m["n"], "r2": m["r2"],
                    "rmse": m["rmse"], "bias": m["bias"], "bilinear_r2": b["r2"], "bilinear_rmse": b["rmse"]})
    return out


def _s10_scale_summary(rows):
    if not rows:
        return {"available": False, "reason": "no night frame with valid anomalies"}, []
    obs = np.concatenate([r["obs"] for r in rows])
    pred = np.concatenate([r["pred"] for r in rows])
    bil = np.concatenate([r["bil"] for r in rows])
    mod = np.concatenate([r["model"] for r in rows])
    cls = np.concatenate([r["cls"] for r in rows])
    m, b, mo = _s10_metric(obs, pred), _s10_metric(obs, bil), _s10_metric(obs, mod)
    by_frame = []
    for y in sorted({r["year"] for r in rows}):
        sel = [r for r in rows if r["year"] == y]
        o = np.concatenate([r["obs"] for r in sel])
        mm = _s10_metric(o, np.concatenate([r["pred"] for r in sel]))
        bb = _s10_metric(o, np.concatenate([r["bil"] for r in sel]))
        by_frame.append({"year": int(y), "r2": mm["r2"], "rmse": mm["rmse"], "bias": mm["bias"], "n": mm["n"],
                         "bilinear_r2": bb["r2"], "bilinear_rmse": bb["rmse"], "n_folds": len(sel)})
    gain = (1.0 - m["rmse"] / b["rmse"]) if (m["rmse"] is not None and b["rmse"]) else None
    summary = {"available": True, "r2": m["r2"], "rmse": m["rmse"], "bias": m["bias"], "n": m["n"],
               "bilinear_r2": b["r2"], "bilinear_rmse": b["rmse"], "bilinear_bias": b["bias"],
               "model_only_r2": mo["r2"], "model_only_rmse": mo["rmse"], "rmse_gain_vs_bilinear": gain,
               "factor": _S10_SCALE_FACTOR, "coarse_res_m": int(GRID.res) * _S10_SCALE_FACTOR,
               "fine_res_m": int(GRID.res), "folds": "block20 (coarse training cells strictly outside the held-out "
                                                    "fold)", "target": "night anomaly (fixed-mask centred)",
               "frames": sorted({r["year"] for r in rows}), "n_fits": len(rows),
               "max_block_error_c": float(max(r["max_block_error_c"] for r in rows)), "by_frame": by_frame}
    return summary, _s10_by_class(obs, pred, bil, cls)


def _s10_day_demo_job(year, gpu_id):
    """Downscale MODIS Terra day (1 km) -> fine and compare anomalies with the Landsat fine composite."""
    raw = RAW_OBS[year]
    modis = np.where(ROI_MASK, np.asarray(raw["lst_day_modis"], np.float64), np.nan)
    if np.isfinite(modis).sum() < 100:
        return None
    pred_f = _s10_fine_predictors(year)
    field, _, info = _s10_dms(modis, pred_f, _S10_F, ROI_MASK, ROI_MASK_FINE, device=xgb_device_for(gpu_id),
                              seed=int(CFG.SEED) + 3000 + int(year), label=f"MODIS day {year}")
    del pred_f
    bil = _s10_bilinear_up(_s10_fill_nearest(modis), _S10_F)
    blk = _s10_upsample(modis, _S10_F).astype(np.float32)
    nobs = np.nan_to_num(np.asarray(raw["lst_day_nobs_f"], np.float64), nan=0.0)
    landsat = np.where(nobs >= _S10_DAY_MIN_SCENES, np.asarray(raw["lst_day_f"], np.float64), np.nan)
    ok = ROI_MASK_FINE & np.isfinite(landsat) & np.isfinite(field) & np.isfinite(blk)
    if ok.sum() < 100:
        return None
    dom = _s10_upsample(_s10_dominant_class(year), _S10_F)[ok]

    def anom(a):
        v = np.asarray(a, np.float64)[ok]
        return v - v.mean()

    return {"year": int(year), "obs": anom(landsat).astype(np.float32), "pred": anom(field).astype(np.float32),
            "bil": anom(bil).astype(np.float32), "blk": anom(blk).astype(np.float32), "cls": dom,
            "max_block_error_c": info["max_block_error_c"], "n_train": info["n_train"]}


def _s10_day_demo_summary(results):
    results = [r for r in results if r is not None]
    if not results:
        return {"available": False, "reason": "no frame of 2010/2015/2020 with MODIS Terra day data",
                "years": [], "r2": None, "rmse": None, "bilinear_rmse": None}
    obs = np.concatenate([r["obs"] for r in results])
    pred = np.concatenate([r["pred"] for r in results])
    bil = np.concatenate([r["bil"] for r in results])
    blk = np.concatenate([r["blk"] for r in results])
    cls = np.concatenate([r["cls"] for r in results])
    m, b, u = _s10_metric(obs, pred), _s10_metric(obs, bil), _s10_metric(obs, blk)
    by_year = []
    for r in results:
        mm, bb = _s10_metric(r["obs"], r["pred"]), _s10_metric(r["obs"], r["bil"])
        by_year.append({"year": r["year"], "r2": mm["r2"], "rmse": mm["rmse"], "bilinear_r2": bb["r2"],
                        "bilinear_rmse": bb["rmse"], "n": mm["n"], "max_block_error_c": r["max_block_error_c"]})
    return {"available": True, "r2": m["r2"], "rmse": m["rmse"], "n": m["n"], "bilinear_r2": b["r2"],
            "bilinear_rmse": b["rmse"], "uniform_r2": u["r2"], "uniform_rmse": u["rmse"],
            "years": [r["year"] for r in results], "by_year": by_year, "by_lc": _s10_by_class(obs, pred, bil, cls),
            "comparison": "anomalies (each product minus its mean over the common fine pixels) vs the Landsat C2 L2 "
                          "fine composite (>= 2 clear scenes)",
            "note": "MODIS Terra (~10:30 local solar time) and Landsat (~10:30 IST) differ in overpass and algorithm; "
                    "anomalies remove the frame-wide offset"}

## 10.5 Run (stage-cached)
The cache key holds the version, the grid/profile, and fingerprints of every input (1 km frame fields, levels,
AOA flags, fine Landsat/GHSL layers), so a changed upstream result can never reuse stale downscaled fields.

In [ ]:
def _s10_fingerprint():
    h = _s10_hashlib.sha1()
    for m in _S10_MODES:
        for y in _S10_FRAMES:
            v = np.asarray(FRAME_FIELDS[m][y], np.float64)
            h.update(np.array([np.nansum(v), np.isfinite(v).sum(), LEVELS[m][y]["level"],
                               float(np.sum(FRAME_AOA_OUT[m][y]))], np.float64).tobytes())
            h.update(FRAME_TIERS[m][y].encode())
    for y in _S10_FRAMES:
        for key in _S10_PRED_NAMES[:5] + ["lst_day_f", "lst_day_nobs_f", "lst_day_modis"]:
            v = np.asarray(RAW_OBS[y][key], np.float64)
            h.update(np.array([np.nansum(v), np.isfinite(v).sum()], np.float64).tobytes())
    for key in ("elevation_f", "water_occ_f"):
        v = np.asarray(RAW_STATIC[key], np.float64)
        h.update(np.array([np.nansum(v), np.isfinite(v).sum()], np.float64).tobytes())
    h.update(np.asarray(DF["anom_night"].to_numpy(np.float64)[np.isfinite(DF["anom_night"].to_numpy())]
                        .sum()).tobytes())
    return h.hexdigest()[:16]


def _s10_compute():
    t0 = _s10_time.perf_counter()
    fields, infos = {m: {} for m in _S10_MODES}, {m: {} for m in _S10_MODES}
    if "day" in _S10_MODES:
        for y in _S10_FRAMES:
            fields["day"][y], infos["day"][y] = _s10_day_fine(y)
    if "night" in _S10_MODES:
        with timer("10_night_dms"):
            res = gpu_pool_map(_s10_night_job, _S10_FRAMES)
        for y, r in zip(_S10_FRAMES, res):
            fields["night"][y], infos["night"][y] = r["field"], r["info"]
        del res
        free_memory()
    scale_rows = []
    scale_frames = [y for y in _S10_FRAMES if "night" in _S10_MODES
                    and FRAME_TIERS["night"][y] in ("observed", "reconstructed")
                    and np.isfinite(DF["anom_night"].to_numpy()[_S10_FRAMES.index(y) * _S10_NC:
                                                                (_S10_FRAMES.index(y) + 1) * _S10_NC]).sum() > 100]
    with timer("10_scale_invariance"):
        for rows in gpu_pool_map(_s10_scale_job, scale_frames):
            scale_rows.extend(rows)
    scale, by_lc = _s10_scale_summary(scale_rows)
    del scale_rows
    demo_years = [y for y in _S10_DEMO_YEARS if y in _S10_FRAMES]
    with timer("10_day_demo"):
        demo = _s10_day_demo_summary(gpu_pool_map(_s10_day_demo_job, demo_years))
    free_memory()
    return {"fields": fields, "infos": infos, "scale": scale, "scale_by_lc": by_lc, "day_demo": demo,
            "seconds": round(_s10_time.perf_counter() - t0, 1)}


_s10_key = CFG.key("SEED", "FINE_FACTOR", version=_S10_VERSION, inputs=_s10_fingerprint(),
                   params=_S10_XGB_PARAMS, rounds=_S10_ROUNDS, het=_S10_HET_DROP, scale_factor=_S10_SCALE_FACTOR,
                   demo=_S10_DEMO_YEARS)
with timer("10_downscaling"):
    _S10_OUT = stage_cache("10_downscaling", _s10_key).get_or_compute(_s10_compute)

FINE_FIELDS = {m: {int(y): np.asarray(v, np.float32) for y, v in _S10_OUT["fields"][m].items()} for m in _S10_MODES}
FINE_FLAGS, _S10_TIER_CODES = {m: {} for m in _S10_MODES}, {m: {} for m in _S10_MODES}
for _m in _S10_MODES:
    for _y, _v in FINE_FIELDS[_m].items():
        FINE_FLAGS[_m][_y], _S10_TIER_CODES[_m][_y] = _s10_flags(_m, _y, _v)

# Block conservation (SPEC §4.8): every downscaled night 4 x 4 block must average back to its 1 km value.
_S10_CONSERVATION = {}
for _y, _v in FINE_FIELDS.get("night", {}).items():
    _c = _s10_cell_grid(FRAME_FIELDS["night"][_y], dtype=np.float64)
    _bm, _ = _s10_block_mean(_v, _S10_F, ROI_MASK_FINE)
    _ok = np.isfinite(_c) & ROI_MASK
    _err = float(np.nanmax(np.abs(_bm[_ok] - _c[_ok]))) if _ok.any() else 0.0
    _S10_CONSERVATION[_y] = _err
    if not _err < _S10_CONSERVE_TOL_C:
        raise AssertionError(f"10: night {_y}: block conservation violated (max |e_c| = {_err:.4f} °C)")
    if not np.array_equal(np.isfinite(_v), _s10_upsample(_ok, _S10_F)):
        raise AssertionError(f"10: night {_y}: fine NaN pattern differs from the 1 km field")
log("10: night block conservation max |e_c| per frame: "
    + ", ".join(f"{y}={e:.1e}" for y, e in _S10_CONSERVATION.items()))

DOWNSCALING_METRICS = {
    "night": {"scale_invariance": _S10_OUT["scale"], "by_lc": _S10_OUT["scale_by_lc"],
              "conservation": {"max_abs_block_error_c": max(_S10_CONSERVATION.values()) if _S10_CONSERVATION
                               else None, "tolerance_c": _S10_CONSERVE_TOL_C,
                               "by_frame": {str(y): e for y, e in _S10_CONSERVATION.items()}},
              "frames": {str(y): dict(i) for y, i in _S10_OUT["infos"].get("night", {}).items()},
              "method": ("DMS: XGBoost (depth 6, 600 rounds, eta 0.05) on 10 fine predictors aggregated to 1 km, "
                         "20 % most heterogeneous cells excluded; bilinear residual correction + 3 iterations of "
                         "block conservation (|e_c| < 0.01 °C)"),
              "predictors": list(_S10_PRED_NAMES)},
    "day_demo": _S10_OUT["day_demo"],
    "day": {"method": "Landsat C2 L2 ST fine composite (observed) + overpass correction to 10:45 IST; pixels with "
                      "< 2 clear scenes masked (not downscaled)",
            "frames": {str(y): i for y, i in _S10_OUT["infos"].get("day", {}).items()}},
    "fine_res_m": _S10_FINE_RES_M, "seconds": _S10_OUT["seconds"],
}
_s10_si = DOWNSCALING_METRICS["night"]["scale_invariance"]
if _s10_si.get("available"):
    log(f"10: scale invariance ({_s10_si['coarse_res_m'] / 1000:g} km -> {_s10_si['fine_res_m'] / 1000:g} km, "
        f"{_s10_si['n_fits']} fits, n={_s10_si['n']:,}): DMS R2 {_s10_si['r2']:.3f} RMSE {_s10_si['rmse']:.3f} °C "
        f"bias {_s10_si['bias']:+.3f} | bilinear R2 {_s10_si['bilinear_r2']:.3f} RMSE {_s10_si['bilinear_rmse']:.3f} "
        f"| regression only RMSE {_s10_si['model_only_rmse']:.3f}")
_s10_dd = DOWNSCALING_METRICS["day_demo"]
if _s10_dd.get("available"):
    log(f"10: day demonstration (MODIS Terra {_s10_dd['years']} -> {_S10_FINE_RES_M:g} m vs Landsat): DMS R2 "
        f"{_s10_dd['r2']:.3f} RMSE {_s10_dd['rmse']:.3f} °C | bilinear R2 {_s10_dd['bilinear_r2']:.3f} RMSE "
        f"{_s10_dd['bilinear_rmse']:.3f} | block-constant RMSE {_s10_dd['uniform_rmse']:.3f}")
else:
    log(f"10: day demonstration not available: {_s10_dd.get('reason')}")

## 10.6 EPSG:4326 rasters (`RASTER_INDEX`, PNGs)
The lon/lat raster covers the bounding box of the ROI (its UTM rectangle densified along the edges before
transforming, because straight UTM edges are curved in lon/lat), snapped outward to multiples of the preview
resolution so the full-resolution and preview rasters share identical bounds (preview = exactly half the size).
The inverse map (output pixel → flat fine-pixel index, −1 outside the ROI) is computed **once** per resolution in
row chunks and reused for every frame; then each PNG is a gather. An alignment self-test transforms random output
pixel centres again and asserts that the selected fine pixel's centre lies within half a fine pixel (in metres) of
the output pixel centre on both axes.

In [ ]:
def _s10_lonlat_to_xy(lon, lat):
    if pyproj is not None:
        tr = pyproj.Transformer.from_crs("EPSG:4326", CFG.CRS, always_xy=True)
        return tr.transform(lon, lat)
    return _grid_lonlat_to_xy(lon, lat, CFG.CRS)


def _s10_xy_to_lonlat(x, y):
    if pyproj is not None:
        tr = pyproj.Transformer.from_crs(CFG.CRS, "EPSG:4326", always_xy=True)
        return tr.transform(x, y)
    return _grid_xy_to_lonlat(x, y, CFG.CRS)


def _s10_raster_bounds():
    """[W, S, E, N] of the ROI's fine-pixel bounding box in lon/lat, snapped outward to the preview resolution."""
    rows, cols = np.nonzero(ROI_MASK)
    res = float(GRID.res)
    xl, xr = GRID.x0 + cols.min() * res, GRID.x0 + (cols.max() + 1) * res
    yt, yb = GRID.y0 - rows.min() * res, GRID.y0 - (rows.max() + 1) * res
    t = np.linspace(0.0, 1.0, 200)
    ring_x = np.concatenate([xl + (xr - xl) * t, np.full_like(t, xr), xr - (xr - xl) * t, np.full_like(t, xl)])
    ring_y = np.concatenate([np.full_like(t, yt), yt - (yt - yb) * t, np.full_like(t, yb), yb + (yt - yb) * t])
    lon, lat = _s10_xy_to_lonlat(ring_x, ring_y)
    p = _S10_PREVIEW_RES_DEG
    w_i = _s10_math.floor(float(np.min(lon)) / p - 1e-9)
    e_i = _s10_math.ceil(float(np.max(lon)) / p + 1e-9)
    s_i = _s10_math.floor(float(np.min(lat)) / p - 1e-9)
    n_i = _s10_math.ceil(float(np.max(lat)) / p + 1e-9)
    return [w_i * p, s_i * p, e_i * p, n_i * p], (e_i - w_i, n_i - s_i)


def _s10_inverse_map(bounds, width, height):
    """(height, width) int32 flat fine-pixel index of every output pixel centre (nearest), -1 outside the ROI."""
    west, _, east, north = bounds
    res_lon = (east - west) / width
    res_lat = (north - bounds[1]) / height
    lon_c = west + (np.arange(width, dtype=np.float64) + 0.5) * res_lon
    out = np.full((height, width), -1, dtype=np.int32)
    roi_flat = ROI_MASK_FINE.ravel()
    fres = float(GRID_FINE.res)
    rows_per_chunk = max(1, _S10_MAP_CHUNK_PX // max(width, 1))
    for r0 in range(0, height, rows_per_chunk):
        r1 = min(height, r0 + rows_per_chunk)
        lat_c = north - (np.arange(r0, r1, dtype=np.float64) + 0.5) * res_lat
        lon_g, lat_g = np.meshgrid(lon_c, lat_c)
        x, y = _s10_lonlat_to_xy(lon_g.ravel(), lat_g.ravel())
        col = np.floor((np.asarray(x) - GRID_FINE.x0) / fres).astype(np.int64)
        row = np.floor((GRID_FINE.y0 - np.asarray(y)) / fres).astype(np.int64)
        inside = (col >= 0) & (col < _S10_WF) & (row >= 0) & (row < _S10_HF)
        flat = np.where(inside, row * _S10_WF + col, 0)
        keep = inside & roi_flat[flat]
        out[r0:r1] = np.where(keep, flat, -1).reshape(r1 - r0, width).astype(np.int32)
        del lon_g, lat_g, x, y, col, row, inside, flat, keep
    return out


def _s10_alignment_check(mapping, bounds, n=4000, seed=0):
    """Max |dx|, |dy| (m) between output pixel centres and their fine pixel centres (must be <= fres/2)."""
    height, width = mapping.shape
    valid = np.flatnonzero(mapping.ravel() >= 0)
    if valid.size == 0:
        raise AssertionError("10: the raster inverse map selects no ROI pixel")
    pick = np.random.default_rng(seed).choice(valid, size=min(n, valid.size), replace=False)
    i, j = pick // width, pick % width
    res_lon = (bounds[2] - bounds[0]) / width
    res_lat = (bounds[3] - bounds[1]) / height
    lon = bounds[0] + (j + 0.5) * res_lon
    lat = bounds[3] - (i + 0.5) * res_lat
    x, y = _s10_lonlat_to_xy(lon, lat)
    flat = mapping.ravel()[pick].astype(np.int64)
    fres = float(GRID_FINE.res)
    xc = GRID_FINE.x0 + (flat % _S10_WF + 0.5) * fres
    yc = GRID_FINE.y0 - (flat // _S10_WF + 0.5) * fres
    dx, dy = float(np.max(np.abs(np.asarray(x) - xc))), float(np.max(np.abs(np.asarray(y) - yc)))
    if not (dx <= fres / 2 + 1e-6 and dy <= fres / 2 + 1e-6):
        raise AssertionError(f"10: raster alignment off: max |dx| {dx:.2f} m, |dy| {dy:.2f} m > {fres / 2} m")
    return {"n": int(pick.size), "max_dx_m": dx, "max_dy_m": dy, "limit_m": fres / 2}


def _s10_domain(mode):
    """Fixed raster domain of a mode: p0.5-p99.5 of all observed fine fields, padded by 3 °C, rounded outward."""
    vals = np.concatenate([v[np.isfinite(v)] for v in FINE_FIELDS[mode].values()])
    if vals.size == 0:
        raise RuntimeError(f"10: {mode}: no finite fine values for the raster domain")
    lo, hi = np.percentile(vals.astype(np.float64), _S10_DOMAIN_Q)
    return [float(_s10_math.floor(lo - _S10_DOMAIN_PAD_C)), float(_s10_math.ceil(hi + _S10_DOMAIN_PAD_C))]


def _s10_encode(values, lo, hi):
    """R channel: 0 nodata, else 1 + round((v - lo) / (hi - lo) * 254) clipped to 1..255."""
    v = np.asarray(values, np.float64)
    ok = np.isfinite(v)
    code = np.zeros(v.shape, dtype=np.uint8)
    code[ok] = np.clip(1.0 + np.rint((v[ok] - lo) / (hi - lo) * 254.0), 1, 255).astype(np.uint8)
    return code


def _s10_decode(code, lo, hi):
    c = np.asarray(code, np.float64)
    return np.where(c > 0, lo + (c - 1.0) * (hi - lo) / 254.0, np.nan)


def _s10_write_png(path, field, flags, mapping, lo, hi):
    """Gather the fine field through the inverse map and write an 8-bit RGB PNG (Pillow); returns pixel count."""
    from PIL import Image

    flat = mapping.ravel()
    ok = flat >= 0
    vals = np.full(flat.size, np.nan, dtype=np.float32)
    vals[ok] = field.ravel()[flat[ok]]
    rgb = np.zeros((flat.size, 3), dtype=np.uint8)
    rgb[:, 0] = _s10_encode(vals, lo, hi)
    g = np.zeros(flat.size, dtype=np.uint8)
    g[ok] = flags.ravel()[flat[ok]]
    rgb[:, 1] = np.where(rgb[:, 0] > 0, g, 0)
    img = Image.fromarray(rgb.reshape(mapping.shape + (3,)), mode="RGB")
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    img.save(tmp, format="PNG", optimize=True)
    tmp.replace(path)
    n = int((rgb[:, 0] > 0).sum())
    del vals, rgb, g, img
    return n


def _s10_build_rasters():
    try:
        import PIL  # noqa: F401  (Pillow is the PNG writer)
    except ImportError:
        log("10: Pillow is not installed - rasters are NOT written (pip install pillow)", "ERROR")
        return {"crs": "EPSG:4326", "available": False, "reason": "Pillow missing", "frames": {}}, None
    bounds, (pw, ph) = _s10_raster_bounds()
    width, height = 2 * pw, 2 * ph
    with timer("10_raster_inverse_map"):
        full_map = _s10_inverse_map(bounds, width, height)
        prev_map = _s10_inverse_map(bounds, pw, ph)
    align = {"full": _s10_alignment_check(full_map, bounds, seed=int(CFG.SEED)),
             "preview": _s10_alignment_check(prev_map, bounds, seed=int(CFG.SEED) + 1)}
    log(f"10: raster grid {width}x{height} @ {_S10_RASTER_RES_DEG:g} deg (+ {pw}x{ph} preview), bounds "
        f"{np.round(bounds, 4).tolist()}; alignment max |dx| {align['full']['max_dx_m']:.1f} m, |dy| "
        f"{align['full']['max_dy_m']:.1f} m (limit {align['full']['limit_m']:.0f} m)")
    domains = {m: _s10_domain(m) for m in _S10_MODES}
    frames = {}
    with timer("10_raster_png"):
        for m in _S10_MODES:
            lo, hi = domains[m]
            for y in sorted(FINE_FIELDS[m]):
                key = f"{m}_{y}"
                rel, rel_prev = f"raster/{key}.png", f"raster/{key}_500m.png"
                n_full = _s10_write_png(Path(CFG.WEB_DIR) / rel, FINE_FIELDS[m][y], FINE_FLAGS[m][y], full_map,
                                        lo, hi)
                n_prev = _s10_write_png(Path(CFG.WEB_DIR) / rel_prev, FINE_FIELDS[m][y], FINE_FLAGS[m][y],
                                        prev_map, lo, hi)
                tier = FRAME_TIERS[m][y]
                frames[key] = {"file": rel, "preview": rel_prev, "lo": lo, "hi": hi, "tier": tier,
                               "downscaled": bool(m == "night"), "mode": m, "year": int(y),
                               "tier_code": int(_S10_TIER_CODES[m][y]), "valid_pixels": n_full,
                               "valid_pixels_preview": n_prev}
    index = {"crs": "EPSG:4326", "bounds": [round(float(b), 6) for b in bounds], "width": int(width),
             "height": int(height), "res_deg": _S10_RASTER_RES_DEG,
             "preview": {"width": int(pw), "height": int(ph), "res_deg": _S10_PREVIEW_RES_DEG},
             "domains": domains, "fine_res_m": _S10_FINE_RES_M, "source_crs": CFG.CRS,
             "encoding": {"R": "0 = nodata, else value = lo + (R - 1) * (hi - lo) / 254 (°C)",
                          "G": "flags: bit 0 outside AOA; bits 1-2 tier code (0 observed, 1 observed-sparse or "
                               "reconstructed, 2 modelled, 3 downscaled)", "B": "0"},
             "resampling": "inverse mapping to the nearest fine pixel (pyproj)", "alignment": align,
             "frames": frames}
    return index, (full_map, prev_map)


RASTER_INDEX, _S10_MAPS = _s10_build_rasters()
if RASTER_INDEX.get("frames"):
    save_json(RASTER_INDEX, Path(CFG.WEB_DIR) / "raster_index.json")
    log(f"10: wrote {2 * len(RASTER_INDEX['frames'])} PNGs to {_S10_RASTER_DIR} and raster_index.json; domains "
        + ", ".join(f"{m} {d}" for m, d in RASTER_INDEX["domains"].items()))
    DOWNSCALING_METRICS["rasters"] = {"width": RASTER_INDEX["width"], "height": RASTER_INDEX["height"],
                                      "res_deg": RASTER_INDEX["res_deg"], "domains": RASTER_INDEX["domains"],
                                      "alignment": RASTER_INDEX["alignment"]}

## 10.7 Figures
(1) Night frame at 1 km vs downscaled vs bilinear (zoom on the urban core); (2) scale-invariance error by
dominant land cover, DMS vs bilinear; (3) the day demonstration by land cover.

In [ ]:
def _s10_save_fig(fig, name):
    fig.savefig(_S10_FIG_DIR / name, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def _s10_fig_night():
    if "night" not in FINE_FIELDS:
        return
    obs_frames = [y for y in _S10_FRAMES if FRAME_TIERS["night"][y] == "observed"] or _S10_FRAMES
    y = obs_frames[-1]
    fine = FINE_FIELDS["night"][y]
    coarse = _s10_cell_grid(FRAME_FIELDS["night"][y], dtype=np.float64)
    bil = np.where(np.isfinite(fine), _s10_bilinear_up(_s10_fill_nearest(coarse), _S10_F), np.nan)
    blk = _s10_upsample(coarse, _S10_F)
    built = np.asarray(RAW_OBS[y]["built_f"], np.float64)
    # zoom window centred on the most built-up 1 km cell
    r, c = np.unravel_index(np.nanargmax(np.where(ROI_MASK, _s10_block_mean(built, _S10_F)[0], -1)),
                            (_S10_H, _S10_W))
    half = max(8, min(_S10_H, _S10_W) // 6) * _S10_F
    r0, c0 = max(0, r * _S10_F - half), max(0, c * _S10_F - half)
    win = (slice(r0, r0 + 2 * half), slice(c0, c0 + 2 * half))
    vals = fine[win][np.isfinite(fine[win])]
    if vals.size == 0:
        return
    vmin, vmax = np.percentile(vals, [2, 98])
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, arr, title in zip(axes, (blk, bil, fine), (f"1 km observed ({y})", "bilinear 1 km -> fine",
                                                         f"DMS downscaled ({_S10_FINE_RES_M:g} m)")):
        im = ax.imshow(arr[win], cmap="inferno", vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.colorbar(im, ax=axes, shrink=0.8, label="night LST (°C)")
    fig.suptitle("Night LST downscaling around the most built-up cell (block-conserving)")
    _s10_save_fig(fig, "10_night_downscaling.png")


def _s10_fig_validation():
    si_lc = DOWNSCALING_METRICS["night"]["by_lc"]
    dd = DOWNSCALING_METRICS["day_demo"]
    panels = [(si_lc, "Night scale invariance (coarse -> 1 km)")]
    if dd.get("available"):
        panels.append((dd["by_lc"], "Day demonstration (MODIS -> fine vs Landsat)"))
    panels = [(rows, t) for rows, t in panels if rows]
    if not panels:
        return
    fig, axes = plt.subplots(1, len(panels), figsize=(7 * len(panels), 4.5), squeeze=False)
    for ax, (rows, title) in zip(axes[0], panels):
        labels = [r["label"] for r in rows]
        x = np.arange(len(rows))
        ax.bar(x - 0.2, [r["rmse"] or 0 for r in rows], 0.4, label="DMS", color="#f97316")
        ax.bar(x + 0.2, [r["bilinear_rmse"] or 0 for r in rows], 0.4, label="bilinear", color="#64748b")
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=30, ha="right")
        ax.set_ylabel("RMSE (°C)")
        ax.set_title(title)
        ax.legend()
    _s10_save_fig(fig, "10_downscaling_validation.png")


for _s10_fn in (_s10_fig_night, _s10_fig_validation):
    try:
        _s10_fn()
    except Exception as _s10_exc:  # noqa: BLE001 - a broken figure must never lose the computed results
        log(f"10: figure {_s10_fn.__name__} failed: {type(_s10_exc).__name__}: {_s10_exc}", "WARNING")
        plt.close("all")

## 10.8 Contract check (SPEC §3.7)

In [ ]:
def _s10_contract_check():
    for m in _S10_MODES:
        assert sorted(FINE_FIELDS[m]) == _S10_FRAMES, (m, sorted(FINE_FIELDS[m]))
        for y in _S10_FRAMES:
            f, g = FINE_FIELDS[m][y], FINE_FLAGS[m][y]
            assert f.shape == (_S10_HF, _S10_WF) and f.dtype == np.float32, (m, y, f.shape, f.dtype)
            assert g.shape == f.shape and g.dtype == np.uint8, (m, y, g.shape, g.dtype)
            assert not np.isfinite(f[~ROI_MASK_FINE]).any(), f"{m} {y}: values outside the ROI"
            assert np.all(g[~np.isfinite(f)] == 0), f"{m} {y}: flags on nodata pixels"
            assert np.all((g >> 1) <= 3)
            if m == "night" and FRAME_TIERS[m][y] == "modelled":
                assert np.isfinite(f[ROI_MASK_FINE]).all(), f"night {y}: modelled field must be complete"
    if RASTER_INDEX.get("frames"):
        for fr in RASTER_INDEX["frames"].values():
            for rel in (fr["file"], fr["preview"]):
                assert (Path(CFG.WEB_DIR) / rel).is_file(), rel
            assert fr["lo"] < fr["hi"] and [fr["lo"], fr["hi"]] == RASTER_INDEX["domains"][fr["mode"]]
        assert RASTER_INDEX["width"] == 2 * RASTER_INDEX["preview"]["width"]
        assert RASTER_INDEX["height"] == 2 * RASTER_INDEX["preview"]["height"]
        # quantisation round trip
        lo, hi = RASTER_INDEX["domains"][_S10_MODES[0]]
        probe = np.linspace(lo, hi, 1001)
        err = np.abs(_s10_decode(_s10_encode(probe, lo, hi), lo, hi) - probe)
        assert float(err.max()) <= (hi - lo) / 254.0 / 2.0 + 1e-9, float(err.max())
    for y, e in DOWNSCALING_METRICS["night"]["conservation"]["by_frame"].items():
        assert e < _S10_CONSERVE_TOL_C, (y, e)
    log("10: contract check OK (FINE_FIELDS, FINE_FLAGS, DOWNSCALING_METRICS, RASTER_INDEX)")


_s10_contract_check()
del _S10_MAPS
free_memory()
log("Section 10 complete: FINE_FIELDS, FINE_FLAGS, DOWNSCALING_METRICS, RASTER_INDEX ready")

# 11. Projections 2030–2050: land change, derived predictors, CMIP6 deltas and Monte-Carlo uncertainty

**Scenario projections, not forecasts.** Each SSP is a coherent "what if" for land and climate; scenarios are
never mixed. The chain (SPEC §4.9–§4.10, `methods_v2.md` §4):

1. **Land change** — a fractional cellular automaton (CA) on the 1 km built-up fraction $u$ (GHSL). District
   demand follows the Chen et al. (2020) SSP urban-area ratios: $U_{d,s}(y) = U_d(2025)\,g_{d,s}(y)/g_{d,s}(2025)$.
   Each 5-year step allocates the demand increment as $\Delta u_i = \min(cap_i,\; k_d\,S_i\,N_i\,\varepsilon_i)$:
   $S_i$ = transition suitability (XGBoost classifier of $\Delta u_{2005→2015} > 0.05$ on proximity, terrain,
   population and land-cover fractions), $N_i$ = 3 × 3 neighbourhood mean of $u$ (urban growth is contagious),
   $\varepsilon_i \sim \mathrm{LogNormal}(0, 0.3)$ = allocation stochasticity, $cap_i = \max(0, 1 − u_i − forest_i −
   water_i)$ (**forest and water are protected**), and $k_d$ found by bisection so that the district total meets
   the demand exactly. New built-up land comes from cropland, barren and other vegetation in proportion to their
   current shares. `N_LAND_REAL` seeded realisations per scenario.
   **Validation:** a 2015 → 2025 hindcast with the *observed* district demand, scored with Pontius' figure of
   merit on cells with $\Delta u > 0.05$ (hits / (hits + misses + false alarms + wrong hits); a single change
   category has no wrong hits), quantity and allocation disagreement (Pontius & Millones 2011) and RMSE($\Delta u$),
   against the **null model** $\Delta u \propto u_{2015}$ (proportional growth) meeting the same demand.
2. **Derived predictors** — spectral indices, built volume and landscape metrics cannot be observed in the
   future. Delta mapping keeps every cell's observed 2025 idiosyncrasy and adds only the land-driven change:
   $X_y = X_{2025} + h_X(LC_y) − h_X(LC_{2025})$, $h_X$ = XGBoost (depth 4, 300 rounds) of $X$ on the five fractions,
   their 3/15-cell focal means and terrain, fitted on the 2015–2025 frames. `ndvi_drop = ndvi_spring − ndvi` is
   recomputed from the mapped pair (identical to delta-mapping it with $h_{drop} = h_{spring} − h_{ndvi}$).
   Population: $pop_y = pop_{2025}\,F_s(y)\,(u_y + 0.02)/(u_{2025} + 0.02)$, renormalised to the NCR total
   $\times F_s(y)$ (IIASA SSP urban population). Focal features and distances are recomputed with
   `FOCAL_RECOMPUTE`. ERA5 features = mean of the 2015/2020/2025 frames (climate change enters **only** through
   the delta, never twice); the 2025 reference prediction uses the same ERA5 climatology so that
   $\hat A_y − \hat A_{2025}$ isolates the land effect.
3. **Climate delta** — $\Delta T_{s,g}(y)$ = GCM $g$'s Apr–Jun 20-year-window mean (tasmax for day, tasmin for
   night) minus its 2016–2035 window (the 2025 baseline). SSP1-2.6 is not in GEE's NEX-GDDP-CMIP6, so it is
   **pattern-scaled** from SSP2-4.5 per GCM with the AR6 assessed GSAT increments:
   $\Delta T_{126,g} = \Delta T_{245,g} · \Delta GSAT_{126}/\Delta GSAT_{245}$. The spatial pattern (ensemble-median
   2050 delta / its ROI mean) distributes it over the cells.
4. **Baseline (weather-normalised)** — $L_{base,m} = L_m(2025) − β_m\,a$ removes the 2025 window's weather
   anomaly from the observed level. $a = \bar T_{2024–26} − T_{trend}(2025)$, where $T_{trend}$ is the OLS line of the
   ERA5-Land NCR annual series of the mode's temperature variable (Tmax day, Tmin night; section 09) over 2001–2025,
   evaluated at the mean year of the window. Using the trend line rather than a flat 2016–2025 mean keeps the
   warming that has already happened inside the baseline (a flat mean would call part of the trend a "weather
   anomaly"). With fewer than 10 annual values the flat 2016–2025-mean rule is used (WARNING).
5. **Monte Carlo** (`N_MC` draws per scenario × mode): GCM $g$ (uniform), $β$ from its block-bootstrap draws
   (section 09), land realisation $r$ (uniform), per-cell model error $e_i$ from the **projection-setting**
   residuals of the cell's AOA-DI decile (section 07: $y − (\hat y_{LOEO} + \bar r)$, i.e. a known place at an
   unseen time, the setting of a projection; the block20 OOF residuals are the fallback):
   $$LST_k(i, y) = L_{base,m} + β_k\,\Delta T_{s,g}(y)\,P_m(i)
     + [\hat A_{m,s,r}(y, i) − \overline{\hat A_m(2025)}_{FIXED}] + \bar r_m(i) + e_i .$$
   Model errors are spatially correlated, so independent per-cell draws would average away in a district mean. They
   are drawn through a **block20 Gaussian copula**: $u_{ki} = Φ(\sqrt{ρ}\,z_{kb(i)} + \sqrt{1−ρ}\,z_{ki})$ with
   one $z_{kb} \sim N(0, 1)$ per draw and 20 km block (shared by all scenarios, modes and cells) and
   $ρ$ = `AOA[m]['resid_rho_block20']` (section 07; ρ = 0 reproduces independent draws exactly).
   The anomaly prediction is recentred **relative to the 2025 prediction mean**, so the land-driven shift of the
   domain mean is kept. Draws use common random numbers across scenarios (the same GCM rank, β, realisation index
   and error quantiles), so scenario differences are not blurred by sampling noise.

**Outputs** (SPEC §3.8): per-cell P05/P25/P50/P75/P95, scope series quantiles from **per-draw scope means** (a
scope mean's uncertainty is not the mean of per-cell quantiles), the decomposition {climate, land, total,
baseline_c} with total = P50 − baseline_c relative to the scope's own weather-normalised 2025 baseline
baseline_c = L_base + scope mean of ($\hat A_{2025} − \overline{\hat A_{2025}}_{FIXED} + \bar r$), so that
total ≈ climate + land + mean model error for every scope (checked in 11.11), the
uncertainty budget (main effects of GCM, sensitivity β, land realisation and model error — computed analytically
from the additive model and normalised to 1), threshold years (+1/+2/+3 °C above the scope's weather-normalised
baseline), AOA flags (DI of the median land realisation's features) with VSUP tiers from (P95 − P05)/2
(< 0.5, < 1, < 2, ≥ 2 °C), and projected zones (nearest centroid of the median realisation's grouped SHAP).

Omitted feedbacks (stated on every projection panel): irrigation change, aerosol trends, heat-wave frequency,
monsoon-onset timing; the LST model is a pre-monsoon statistical model, not a land-surface model.

Memory: everything is processed scenario by scenario; the Monte-Carlo field is never held for all cells at
once (cell chunks × draws), and each scenario's land products are stage-cached. The **pilot** profile (no
projection frames) skips this section cleanly with empty outputs.

In [ ]:
_lst_enter_section('11')  # section marker (generated by build_notebook.py)

In [ ]:
import hashlib as _s11_hashlib
import time as _s11_time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy import ndimage as _s11_ndi
from scipy import special as _s11_special

# 11-v2.1: decomposition total vs the scope's own baseline (+ baseline_c), projection-setting residual samples with a
# block20 Gaussian copula, L_base anomaly against the ERA5 trend, land-model plausibility checks (D13-D15, D18).
_S11_VERSION = "11-v2.1"
_S11_FRAMES_OBS = sorted(int(y) for y in np.unique(DF["year"].to_numpy()))
_S11_NOW = int(CFG.NOW) if int(CFG.NOW) in _S11_FRAMES_OBS else _S11_FRAMES_OBS[-1]
_S11_PROJ = [int(y) for y in CFG.FRAMES_PROJ]
_S11_SCEN = [str(s) for s in CFG.SCENARIOS]
_S11_MODES = [m for m in CFG.MODES if m in FINAL_MODELS and m in LEVELS and m in AOA]
_S11_NC = int(N_CELLS)
_S11_H, _S11_W = (int(v) for v in ROI_MASK.shape)
_S11_CELL_KM2 = float(GRID.cell_area_km2)
_S11_QK = ["p05", "p25", "p50", "p75", "p95"]
_S11_Q = np.array([0.05, 0.25, 0.50, 0.75, 0.95])
_S11_DU_THR = 0.05                    # Pontius change threshold and suitability label (SPEC §4.9)
_S11_EPS_SIGMA = 0.3                  # allocation noise LogNormal(0, 0.3)
_S11_POP_EPS = 0.02
_S11_FRACS = ["frac_built", "frac_forest", "frac_water", "frac_cropland", "frac_barren"]
_S11_SUIT_FEATURES = ["frac_built", "frac_built_m3", "frac_built_m15", "dist_urban", "dist_builtup", "slope",
                      "elevation", "log_pop", "frac_forest", "frac_water", "frac_cropland", "frac_barren"]
_S11_SUIT_PARAMS = {"objective": "binary:logistic", "eval_metric": "auc", "tree_method": "hist", "max_depth": 5,
                    "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 5,
                    "reg_lambda": 1.0, "max_bin": 256}
_S11_SUIT_ROUNDS = 400
_S11_H_TARGETS = ["ndvi", "ndbi", "mndwi", "albedo", "ndvi_spring", "built_v", "lm_pd", "lm_ed", "lm_contag"]
_S11_H_PARAMS = {"objective": "reg:squarederror", "tree_method": "hist", "max_depth": 4, "learning_rate": 0.1,
                 "subsample": 0.8, "colsample_bytree": 0.9, "min_child_weight": 10, "reg_lambda": 1.0,
                 "max_bin": 256}
_S11_H_ROUNDS = 300
_S11_TERRAIN = ["elevation", "slope", "aspect_sin", "aspect_cos", "elev_tpi7", "elev_tpi21", "elev_std7"]
_S11_ERA5 = ["era5_t2m", "era5_vpd", "era5_swvl1", "era5_evap", "era5_ssr", "era5_wind", "era5_precip_jfm"]
_S11_BOUNDS = {"ndvi": (-1.0, 1.0), "ndbi": (-1.0, 1.0), "mndwi": (-1.0, 1.0), "albedo": (0.0, 1.0),
               "ndvi_spring": (-1.0, 1.0), "built_v": (0.0, np.inf), "lm_pd": (0.0, np.inf),
               "lm_ed": (0.0, np.inf), "lm_contag": (0.0, 100.0)}
_S11_TEMP_VAR = {"day": "tasmax", "night": "tasmin"}
_S11_VSUP_EDGES = np.array([0.5, 1.0, 2.0])
_S11_TREND_YEARS = (2001, 2025)       # ERA5 OLS trend of the weather-normalised baseline (D15)
_S11_TREND_MIN_YEARS = 10
_S11_RHO_MAX = 0.95                   # block20 copula correlation is clipped to [0, 0.95] (D13)
_S11_U_CLIP = 1e-6
_S11_DECOMP_TOL_C = 0.3               # |total - climate - land - model_error_mean| contract tolerance (D14)
_S11_CELL_CHUNK = 8192
_S11_SHAP_BATCH = 8192
_S11_ZONE_NODATA = 255
_S11_NOTE = "scenario projection, not a forecast"
_S11_OMITTED = ["irrigation change", "aerosol trends", "heat-wave frequency", "monsoon-onset timing"]
_S11_FIG_DIR = Path(CFG.FIG_DIR)
_S11_ACTIVE = bool(_S11_PROJ) and int(CFG.N_MC) > 0 and int(CFG.N_LAND_REAL) > 0 and bool(_S11_MODES)

# Outputs (SPEC §3.8) - defined up front so that a skipped section still publishes the complete (empty) interface.
LAND_MODEL_METRICS, CMIP6_DELTAS, PROJ_DRIVER_SUMMARY = {}, {}, {}
PROJ_QUANTILES, PROJ_FLAGS, PROJ_ZONES, PROJ_SERIES = {}, {}, {}, {}
PROJ_DECOMPOSITION, PROJ_UNCERTAINTY, PROJ_THRESHOLD_YEARS, PROJ_META = {}, {}, {}, {}
if not _S11_ACTIVE:
    _s11_reason = ("no projection frames (pilot profile)" if not _S11_PROJ else
                   "N_MC / N_LAND_REAL < 1" if (int(CFG.N_MC) < 1 or int(CFG.N_LAND_REAL) < 1) else
                   "no final model")
    LAND_MODEL_METRICS = {"available": False, "reason": _s11_reason}
    CMIP6_DELTAS = {"available": False, "reason": _s11_reason, "scenarios": {}}
    log(f"11: projections skipped - {_s11_reason}; all §3.8 outputs are empty", "WARNING")
else:
    log(f"11: projections {_S11_PROJ} x {_S11_SCEN} for modes {_S11_MODES}; baseline frame {_S11_NOW}; "
        f"{CFG.N_LAND_REAL} land realisations, {CFG.N_MC} Monte-Carlo draws per scenario and mode")

## 11.1 Grid helpers, land state and the population rule
A **land state** is a dict of canonical-cell vectors (float64): `u` (built-up), `forest`, `water`, `crop`,
`barren` (GLC fractions rescaled to the non-built share, section 06) and `pop` (persons km⁻²). Other vegetation is
the implicit remainder. Cells whose fractions are missing are frozen (`lc_ok = False`); `pop_ok` marks cells with
a population value (missing ones keep a missing `log_pop`, exactly as in section 06).

In [ ]:
def _s11_grid(values, fill=np.nan, dtype=np.float32):
    out = np.full((_S11_H, _S11_W), fill, dtype=dtype)
    out[CELL_ROWS, CELL_COLS] = np.asarray(values).astype(dtype, copy=False)
    return out


def _s11_cells(grid):
    return np.asarray(grid)[CELL_ROWS, CELL_COLS]


def _s11_box_mean(x_cells, size):
    """NaN-aware box mean of a canonical-cell vector on the 1 km grid over ROI cells (same rule as section 06)."""
    g = _s11_grid(x_cells, dtype=np.float64)
    m = ROI_MASK & np.isfinite(g)
    num = _s11_ndi.uniform_filter(np.where(m, g, 0.0), size=int(size), mode="constant", cval=0.0)
    den = _s11_ndi.uniform_filter(m.astype(np.float64), size=int(size), mode="constant", cval=0.0)
    with np.errstate(invalid="ignore", divide="ignore"):
        out = num / den
    out[den < 0.5 / (size * size)] = np.nan
    return _s11_cells(out)


_S11_DISTRICT = np.asarray(DISTRICT_IDX)[CELL_ROWS, CELL_COLS].astype(np.int64)
_S11_DIST_IDS = sorted(int(d) for d in np.unique(_S11_DISTRICT) if d >= 0)
_S11_DIST_CELLS = {d: np.flatnonzero(_S11_DISTRICT == d) for d in _S11_DIST_IDS}
_S11_DIST_NAME = {int(d["id"]): d["name"] for d in DISTRICTS}
_S11_FIXED = {m: np.asarray(FIXED_MASK[m], bool)[CELL_ROWS, CELL_COLS] for m in CFG.MODES if m in FIXED_MASK}
_S11_FOLD20 = DF["fold_block20"].to_numpy(np.int64)[:_S11_NC]


def _s11_state(year):
    """Observed land state of a frame (canonical cells)."""
    g = FEATURE_GRIDS[int(year)]
    st = {}
    for key, feat in (("u", "frac_built"), ("forest", "frac_forest"), ("water", "frac_water"),
                      ("crop", "frac_cropland"), ("barren", "frac_barren")):
        st[key] = _s11_cells(g[feat]).astype(np.float64)
    ok = np.ones(_S11_NC, dtype=bool)
    for key in ("u", "forest", "water", "crop", "barren"):
        ok &= np.isfinite(st[key])
        st[key] = np.clip(np.nan_to_num(st[key], nan=0.0), 0.0, 1.0)
    st["lc_ok"] = ok
    pop = np.asarray(RAW_OBS[int(year)]["pop"], np.float64)[CELL_ROWS, CELL_COLS]
    st["pop_ok"] = np.isfinite(pop)
    st["pop"] = np.clip(np.nan_to_num(pop, nan=0.0), 0.0, None)
    return st


def _s11_copy_state(st):
    return {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in st.items()}


def _s11_other(st):
    return np.clip(1.0 - st["u"] - st["forest"] - st["water"] - st["crop"] - st["barren"], 0.0, None)


def _s11_pop_rule(u_new, pop_ref, u_ref, factor):
    """pop_y = pop_ref * F * (u_y + 0.02) / (u_ref + 0.02), renormalised to the reference total x F (SPEC §4.9)."""
    raw = pop_ref * factor * (u_new + _S11_POP_EPS) / (u_ref + _S11_POP_EPS)
    total_target = float(pop_ref.sum()) * factor
    s = float(raw.sum())
    return raw * (total_target / s) if s > 0 else raw


def _s11_district_totals(u):
    """Built-up totals per district (in cell units: sum of fractions)."""
    return {d: float(u[idx].sum()) for d, idx in _S11_DIST_CELLS.items()}

## 11.2 Transition suitability (XGBoost classifier)
Label: $\Delta u > 0.05$ between the calibration frames (2005 → 2015), features at the start frame, rows = ROI
cells that can still convert (cap > 0). Spatial skill is reported as the block20-fold out-of-fold AUC; the
**hindcast AUC** applies the model to the 2015 state and scores the 2015 → 2025 transitions it never saw.

In [ ]:
def _s11_pick_pair(want0, want1, latest_end):
    """Calibration frame pair: (want0, want1) if observed, else the best 10-year pair ending <= latest_end."""
    frames = _S11_FRAMES_OBS
    if want0 in frames and want1 in frames and want1 <= latest_end:
        return want0, want1, True
    pairs = [(a, b) for a in frames for b in frames if a < b <= latest_end]
    if pairs:
        a, b = min(pairs, key=lambda p: (abs((p[1] - p[0]) - 10), -p[1]))
        return a, b, True
    pairs = [(a, b) for a in frames for b in frames if a < b]
    if not pairs:
        return None, None, False
    a, b = min(pairs, key=lambda p: (abs((p[1] - p[0]) - 10), -p[1]))
    return a, b, False


def _s11_suit_matrix(st, ref_year):
    """(N_CELLS, 12) float32 suitability features of a land state (focal/distances via FOCAL_RECOMPUTE)."""
    base = FEATURE_GRIDS[int(ref_year)]
    grids = {b: base[b] for b in FOCAL_SPEC["bases"]}
    grids["frac_built"] = _s11_grid(st["u"])
    grids["frac_forest"] = _s11_grid(st["forest"])
    grids["frac_water"] = _s11_grid(st["water"])
    grids["frac_barren"] = _s11_grid(st["barren"])
    full = FOCAL_RECOMPUTE(grids, recompute_distances=True)
    cols = {"frac_built": st["u"], "frac_built_m3": _s11_cells(full["frac_built_m3"]),
            "frac_built_m15": _s11_cells(full["frac_built_m15"]), "dist_urban": _s11_cells(full["dist_urban"]),
            "dist_builtup": _s11_cells(full["dist_builtup"]), "slope": _s11_cells(base["slope"]),
            "elevation": _s11_cells(base["elevation"]), "log_pop": np.log1p(st["pop"]),
            "frac_forest": st["forest"], "frac_water": st["water"], "frac_cropland": st["crop"],
            "frac_barren": st["barren"]}
    return np.column_stack([np.asarray(cols[f], np.float32) for f in _S11_SUIT_FEATURES]).astype(np.float32)


def _s11_auc(y, p):
    from sklearn.metrics import roc_auc_score

    y = np.asarray(y)
    if y.size == 0 or y.min() == y.max():
        return None
    return float(roc_auc_score(y, p))


def _s11_xgb_train(params, X, y, rounds, seed, w=None):
    """XGBoost on GPU 0 when available (CPU retry on GPU OOM); the returned booster predicts on the CPU."""
    device = xgb_device_for(0)
    for dev in dict.fromkeys([device, "cpu"]):
        try:
            dtrain = xgb.DMatrix(X, label=y, weight=w, missing=np.nan)
            booster = xgb.train(dict(params, device=dev, seed=int(seed)), dtrain, num_boost_round=int(rounds),
                                verbose_eval=False)
            del dtrain
            booster.set_param({"device": "cpu"})
            return booster
        except Exception as exc:  # noqa: BLE001
            if dev == "cpu" or not is_gpu_oom_error(exc):
                raise
            log(f"11: GPU OOM training on {dev}; retrying on the CPU", "WARNING")
            free_memory()
    raise RuntimeError("unreachable")


def _s11_fit_suitability(y0, y1, hind0, hind1):
    st0, st1 = _s11_state(y0), _s11_state(y1)
    X = _s11_suit_matrix(st0, y0)
    du = st1["u"] - st0["u"]
    cap0 = np.clip(1.0 - st0["u"] - st0["forest"] - st0["water"], 0.0, None)
    rows = np.flatnonzero(st0["lc_ok"] & st1["lc_ok"] & (cap0 > 0.0))
    lab = (du[rows] > _S11_DU_THR).astype(np.float32)
    n_pos = int(lab.sum())
    if n_pos < 10 or n_pos > rows.size - 10:
        raise RuntimeError(f"11: suitability needs both classes ({n_pos} positives of {rows.size} rows, "
                           f"{y0}->{y1})")
    oof = np.full(rows.size, np.nan)
    folds = _S11_FOLD20[rows]
    for k in sorted(int(v) for v in np.unique(folds)):
        tr, te = folds != k, folds == k
        if lab[tr].min() == lab[tr].max() or not te.any():
            continue
        b = _s11_xgb_train(_S11_SUIT_PARAMS, X[rows[tr]], lab[tr], _S11_SUIT_ROUNDS, int(CFG.SEED) + 1100 + k)
        oof[te] = b.inplace_predict(X[rows[te]])
        del b
    ok = np.isfinite(oof)
    booster = _s11_xgb_train(_S11_SUIT_PARAMS, X[rows], lab, _S11_SUIT_ROUNDS, int(CFG.SEED) + 1100)
    info = {"calibration": [int(y0), int(y1)], "n": int(rows.size), "n_positive": n_pos,
            "label": f"du > {_S11_DU_THR}", "features": list(_S11_SUIT_FEATURES),
            "auc_train": _s11_auc(lab, booster.inplace_predict(X[rows])),
            "auc_block20_oof": _s11_auc(lab[ok], oof[ok]), "auc_hindcast": None}
    if hind0 is not None and hind1 is not None:
        sh0, sh1 = _s11_state(hind0), _s11_state(hind1)
        Xh = _s11_suit_matrix(sh0, hind0)
        caph = np.clip(1.0 - sh0["u"] - sh0["forest"] - sh0["water"], 0.0, None)
        rh = np.flatnonzero(sh0["lc_ok"] & sh1["lc_ok"] & (caph > 0.0))
        info["auc_hindcast"] = _s11_auc((sh1["u"] - sh0["u"])[rh] > _S11_DU_THR, booster.inplace_predict(Xh[rh]))
        info["hindcast_period"] = [int(hind0), int(hind1)]
    imp = booster.get_score(importance_type="gain")
    info["gain_importance"] = {_S11_SUIT_FEATURES[int(k[1:])]: float(v) for k, v in imp.items()
                               if k.startswith("f") and k[1:].isdigit()}
    return booster, info

## 11.3 Allocation (bisection on $k_d$) and the CA run
For a district with demand increment $D$, $f(k) = \sum_i \min(cap_i, k\,a_i)$ is continuous and non-decreasing in
$k$, so bisection finds $k_d$ with $f(k_d) = D$; the last bracket is closed exactly by rescaling the uncapped
cells. If the eligible capacity ($a_i > 0$, cap > 0) is smaller than $D$, every eligible cell is filled and the
shortfall is recorded (it is carried into the next step's demand). Built-up land never decreases (GHSL rule).

In [ ]:
def _s11_allocate(st, demand_inc, attract):
    """Allocate district demand increments; returns (new state, du, shortfall dict) - donors in proportion."""
    u, forest, water = st["u"], st["forest"], st["water"]
    cap = np.where(st["lc_ok"], np.clip(1.0 - u - forest - water, 0.0, None), 0.0)
    a = np.clip(np.nan_to_num(attract, nan=0.0), 0.0, None)
    du = np.zeros(_S11_NC)
    shortfall = {}
    for d, demand in demand_inc.items():
        if demand <= 1e-12 or d not in _S11_DIST_CELLS:
            continue
        idx = _S11_DIST_CELLS[d]
        ai, ci = a[idx], cap[idx]
        elig = (ai > 0) & (ci > 0)
        total = float(ci[elig].sum())
        if total <= demand:
            du[idx[elig]] = ci[elig]
            if demand - total > 1e-9:
                shortfall[d] = float(demand - total)
            continue
        ae, ce = ai[elig], ci[elig]

        def f(k, ce=ce, ae=ae):
            return float(np.minimum(ce, k * ae).sum())

        lo, hi = 0.0, demand / float(ae.sum())
        for _ in range(200):
            if f(hi) >= demand:
                break
            lo, hi = hi, hi * 2.0
        for _ in range(100):
            mid = 0.5 * (lo + hi)
            if f(mid) < demand:
                lo = mid
            else:
                hi = mid
            if hi - lo <= 1e-12 * hi:
                break
        alloc = np.minimum(ce, hi * ae)
        unc = alloc < ce
        excess = float(alloc.sum()) - demand
        s_unc = float(alloc[unc].sum())
        if excess > 0 and s_unc > 0:
            alloc[unc] *= max(0.0, (s_unc - excess) / s_unc)
        du[idx[elig]] = alloc
    new = _s11_copy_state(st)
    other = _s11_other(st)
    donors = st["crop"] + st["barren"] + other
    with np.errstate(invalid="ignore", divide="ignore"):
        take_c = np.where(donors > 0, du * st["crop"] / donors, 0.0)
        take_b = np.where(donors > 0, du * st["barren"] / donors, 0.0)
    new["u"] = np.clip(u + du, 0.0, 1.0)
    new["crop"] = np.clip(st["crop"] - take_c, 0.0, 1.0)
    new["barren"] = np.clip(st["barren"] - take_b, 0.0, 1.0)
    donor = {"cropland": float(take_c.sum()), "barren": float(take_b.sum()),
             "other_vegetation": float(du.sum() - take_c.sum() - take_b.sum())}
    return new, du, shortfall, donor


def _s11_run_ca(st0, path, rng, suit, ref_year, pop_fn, null=False):
    """Run the CA through ``path`` = [(year, {district: target total})]; returns [(year, state, info)]."""
    st = _s11_copy_state(st0)
    out = []
    for year, targets in path:
        cur = _s11_district_totals(st["u"])
        inc = {d: max(0.0, float(targets.get(d, cur[d])) - cur[d]) for d in _S11_DIST_IDS}
        if null:
            attract = st["u"].copy()                     # proportional growth: du ~ u (Pontius null model)
        else:
            S = suit.inplace_predict(_s11_suit_matrix(st, ref_year)).astype(np.float64)
            N = np.nan_to_num(_s11_box_mean(st["u"], 3), nan=0.0)
            eps = rng.lognormal(0.0, _S11_EPS_SIGMA, _S11_NC)
            attract = S * N * eps
        st, du, short, donor = _s11_allocate(st, inc, attract)
        st["pop"] = pop_fn(st, year)
        out.append((int(year), _s11_copy_state(st), {"shortfall": short, "donor": donor,
                                                       "allocated": float(du.sum())}))
    return out


def _s11_change_metrics(du_obs, du_sim, valid):
    """Pontius change agreement on cells with du > threshold + RMSE(du)."""
    o = (du_obs > _S11_DU_THR) & valid
    s = (du_sim > _S11_DU_THR) & valid
    hits, misses, fa = int((o & s).sum()), int((o & ~s).sum()), int((~o & s).sum())
    n = int(valid.sum())
    denom = hits + misses + fa
    corr = None
    if n > 2:
        a, b = np.asarray(du_sim[valid], np.float64), np.asarray(du_obs[valid], np.float64)
        if float(a.std()) > 0 and float(b.std()) > 0:
            corr = float(np.corrcoef(a, b)[0, 1])
    return {"fom": hits / denom if denom else None, "hits": hits, "misses": misses, "false_alarms": fa,
            "corr_du": corr,
            "wrong_hits": 0, "quantity_disagreement": abs(int(s.sum()) - int(o.sum())) / n if n else None,
            "allocation_disagreement": 2.0 * min(misses, fa) / n if n else None,
            "rmse_du": float(np.sqrt(np.mean((du_sim[valid] - du_obs[valid]) ** 2))) if n else None,
            "n_obs_change": int(o.sum()), "n_sim_change": int(s.sum())}


def _s11_hindcast(suit, h0, h1, independent):
    """2015 -> 2025 hindcast with the observed district demand: CA realisations vs the proportional-growth null."""
    st0, st1 = _s11_state(h0), _s11_state(h1)
    mids = [y for y in _S11_FRAMES_OBS if h0 < y < h1]
    path = [(y, _s11_district_totals(_s11_state(y)["u"])) for y in mids] + [(h1, _s11_district_totals(st1["u"]))]
    totals = {y: float(_s11_state(y)["pop"].sum()) for y in [h0] + [p[0] for p in path]}

    def pop_fn(st, year):
        return _s11_pop_rule(st["u"], st0["pop"], st0["u"], totals[year] / max(totals[h0], 1.0))

    valid = st0["lc_ok"] & st1["lc_ok"]
    du_obs = st1["u"] - st0["u"]
    real = []
    sims = []
    for r in range(int(CFG.N_LAND_REAL)):
        rng = np.random.default_rng([int(CFG.SEED), 1150, r])
        res = _s11_run_ca(st0, path, rng, suit, h0, pop_fn)
        du_sim = res[-1][1]["u"] - st0["u"]
        sims.append(du_sim.astype(np.float32))
        real.append(_s11_change_metrics(du_obs, du_sim, valid))
    null_res = _s11_run_ca(st0, path, None, suit, h0, pop_fn, null=True)
    du_null = null_res[-1][1]["u"] - st0["u"]
    nm = _s11_change_metrics(du_obs, du_null, valid)

    def mean_of(key):
        vals = [m[key] for m in real if m[key] is not None]
        return float(np.mean(vals)) if vals else None

    # observed donor shares of the converted land (cells with du > threshold)
    sel = valid & (du_obs > _S11_DU_THR)
    d_crop = float(np.clip(st0["crop"][sel] - st1["crop"][sel], 0.0, None).sum())
    d_bar = float(np.clip(st0["barren"][sel] - st1["barren"][sel], 0.0, None).sum())
    d_oth = float(np.clip(_s11_other(st0)[sel] - _s11_other(st1)[sel], 0.0, None).sum())
    tot = d_crop + d_bar + d_oth
    fom_vals = [m["fom"] for m in real if m["fom"] is not None]
    metrics = {"fom": mean_of("fom"), "fom_sd": float(np.std(fom_vals)) if fom_vals else None,
               "fom_null": nm["fom"], "quantity_disagreement": mean_of("quantity_disagreement"),
               "allocation_disagreement": mean_of("allocation_disagreement"), "rmse_du": mean_of("rmse_du"),
               "rmse_du_null": nm["rmse_du"], "quantity_disagreement_null": nm["quantity_disagreement"],
               "allocation_disagreement_null": nm["allocation_disagreement"],
               "corr_du": mean_of("corr_du"), "corr_du_null": nm["corr_du"],
               "hits": mean_of("hits"), "misses": mean_of("misses"), "false_alarms": mean_of("false_alarms"),
               "n_obs_change": real[0]["n_obs_change"] if real else None, "n_realisations": len(real),
               "period": [int(h0), int(h1)], "steps": [int(p[0]) for p in path], "independent": bool(independent),
               "threshold_du": _S11_DU_THR, "null_model": "du proportional to u_2015 (same district demand)",
               "observed_donor_shares": ({"cropland": d_crop / tot, "barren": d_bar / tot,
                                          "other_vegetation": d_oth / tot} if tot > 0 else None)}
    maps = {"du_obs": du_obs.astype(np.float32), "du_sim": sims[0] if sims else None,
            "du_null": du_null.astype(np.float32)}
    metrics.update(_s11_plausibility(metrics))
    return metrics, maps


def _s11_plausibility(hm):
    """Multi-metric comparison of the CA hindcast with the proportional-growth null (D18).

    The CA output still drives the projections; the verdict only tells the reader how much allocation skill the
    hindcast demonstrated. A check with a missing value has ok = None and does not enter the verdict."""
    specs = (("fom > null", "fom", "fom_null", lambda v, n: v > n),
             ("rmse_du <= null", "rmse_du", "rmse_du_null", lambda v, n: v <= n),
             ("quantity_disagreement <= null", "quantity_disagreement", "quantity_disagreement_null",
              lambda v, n: v <= n),
             ("corr_du >= null", "corr_du", "corr_du_null", lambda v, n: v >= n))
    checks = []
    for name, key, key0, rule in specs:
        v, n0 = hm.get(key), hm.get(key0)
        ok = bool(rule(float(v), float(n0))) if v is not None and n0 is not None else None
        checks.append({"check": name, "value": v, "null": n0, "ok": ok})
    oks = [c["ok"] for c in checks if c["ok"] is not None]
    verdict = (None if not oks else "beats_null" if all(oks) else "worse_than_null" if not any(oks) else "mixed")
    end = int((hm.get("period") or [0, 0])[1])
    truth = f"GHSL P2023A {end} epoch" + (" (an extrapolated GHSL epoch, not an observation)" if end == 2025 else "")
    return {"checks": checks, "verdict": verdict, "hindcast_truth": truth}

## 11.4 Derived predictors $h_X$ (delta mapping) and projected feature matrices

In [ ]:
def _s11_lc_design(fr):
    """(N_CELLS, 22) design of h_X: 5 fractions, their 3/15-cell focal means, 7 terrain features."""
    cols = []
    for f in _S11_FRACS:
        cols.append(np.asarray(fr[f], np.float64))
    for f in _S11_FRACS:
        for w in FOCAL_SPEC["windows"]:
            cols.append(_s11_box_mean(fr[f], int(w)))
    for t in _S11_TERRAIN:
        cols.append(_s11_cells(FEATURE_GRIDS[_S11_NOW][t]).astype(np.float64))
    return np.column_stack(cols).astype(np.float32)


def _s11_state_fracs(st):
    return {"frac_built": st["u"], "frac_forest": st["forest"], "frac_water": st["water"],
            "frac_cropland": st["crop"], "frac_barren": st["barren"]}


def _s11_fit_h():
    frames = [y for y in (2015, 2020, 2025) if y in _S11_FRAMES_OBS] or _S11_FRAMES_OBS[-3:]
    designs = {y: _s11_lc_design({f: _s11_cells(FEATURE_GRIDS[y][f]).astype(np.float64) for f in _S11_FRACS})
               for y in frames}
    models, report = {}, {}
    for j, target in enumerate(_S11_H_TARGETS):
        Xs, ys, folds = [], [], []
        for y in frames:
            t = _s11_cells(FEATURE_GRIDS[y][target]).astype(np.float64)
            ok = np.isfinite(t)
            Xs.append(designs[y][ok])
            ys.append(t[ok])
            folds.append(_S11_FOLD20[ok])
        X, yv, fo = np.vstack(Xs), np.concatenate(ys).astype(np.float32), np.concatenate(folds)
        if yv.size < 50:
            log(f"11: h_{target}: only {yv.size} rows - delta mapping disabled (X held at 2025)", "WARNING")
            models[target], report[target] = None, {"n": int(yv.size), "r2_holdout": None}
            continue
        hold = fo == 0
        r2 = None
        if hold.any() and (~hold).sum() > 50:
            b = _s11_xgb_train(_S11_H_PARAMS, X[~hold], yv[~hold], _S11_H_ROUNDS, int(CFG.SEED) + 1200 + j)
            p = b.inplace_predict(X[hold]).astype(np.float64)
            sst = float(np.sum((yv[hold] - yv[hold].mean()) ** 2))
            r2 = 1.0 - float(np.sum((yv[hold] - p) ** 2)) / sst if sst > 0 else None
            del b
        models[target] = _s11_xgb_train(_S11_H_PARAMS, X, yv, _S11_H_ROUNDS, int(CFG.SEED) + 1200 + j)
        report[target] = {"n": int(yv.size), "r2_holdout": r2, "holdout": "block20 fold 0", "frames": frames}
    return models, report, frames


def _s11_era5_climatology():
    frames = [y for y in (2015, 2020, 2025) if y in _S11_FRAMES_OBS] or _S11_FRAMES_OBS[-3:]
    clim = {}
    for f in _S11_ERA5:
        stack = np.stack([np.asarray(FEATURE_GRIDS[y][f], np.float64) for y in frames])
        ok = np.isfinite(stack)
        n = ok.sum(axis=0)
        with np.errstate(invalid="ignore", divide="ignore"):
            mean = np.where(ok, stack, 0.0).sum(axis=0) / n
        clim[f] = np.where(ROI_MASK & (n > 0), mean, np.nan).astype(np.float32)
    return clim, frames


def _s11_features(st, ctx):
    """(N_CELLS, 47) float32 projected feature matrix of a land state (delta mapping, pop, ERA5 climatology)."""
    base = FEATURE_GRIDS[_S11_NOW]
    grids = {f: base[f] for f in BASE_FEATURES}
    now_fr = ctx["now_fracs"]
    fr = _s11_state_fracs(st)
    lc_ok = st["lc_ok"]
    for f in _S11_FRACS:
        vals = np.where(lc_ok, fr[f], now_fr[f])
        grids[f] = _s11_grid(vals)
    design = _s11_lc_design(fr)
    for target in _S11_H_TARGETS:
        x_now = ctx["x_now"][target]
        model = ctx["h_models"].get(target)
        if model is None:
            x = x_now
        else:
            x = x_now + (model.inplace_predict(design).astype(np.float64) - ctx["h_now"][target])
            lo, hi = _S11_BOUNDS[target]
            x = np.where(np.isfinite(x_now), np.clip(x, lo, hi), np.nan)
        grids[target] = _s11_grid(x)
    grids["ndvi_drop"] = grids["ndvi_spring"] - grids["ndvi"]
    grids["log_pop"] = _s11_grid(np.where(st["pop_ok"], np.log1p(np.clip(st["pop"], 0.0, None)), np.nan))
    for f in _S11_ERA5:
        grids[f] = ctx["era5"][f]
    full = FOCAL_RECOMPUTE(grids, recompute_distances=True)
    return FEATURE_MATRIX(full)


def _s11_predict(m, X):
    """Raw final-model anomaly prediction (°C), GPU when available."""
    booster = FINAL_MODELS[m]
    device = xgb_device_for(0)
    try:
        if device != "cpu":
            booster.set_param({"device": device})
            try:
                return np.asarray(booster.predict(xgb.DMatrix(X, missing=np.nan)), np.float32)
            except Exception as exc:  # noqa: BLE001
                if not is_gpu_oom_error(exc):
                    raise
                log("11: GPU OOM in prediction; using the CPU", "WARNING")
                booster.set_param({"device": "cpu"})
        return np.asarray(booster.inplace_predict(np.ascontiguousarray(X, np.float32)), np.float32)
    finally:
        booster.set_param({"device": "cpu"})


_S11_MEMBER = np.zeros((len(FEATURES), len(SHAP_GROUPS)), dtype=np.float64)
for _s11_gi, _s11_g in enumerate(SHAP_GROUPS):
    for _s11_f in SHAP_GROUPS[_s11_g]:
        _S11_MEMBER[FEATURES.index(_s11_f), _s11_gi] = 1.0
_S11_BASE_IDX = np.array([FEATURES.index(f) for f in BASE_FEATURES])


def _s11_group_shap(m, X):
    """(n, 18) grouped TreeSHAP of the final model (NaN rows where a base feature is missing, as in section 08)."""
    booster = FINAL_MODELS[m]
    out = np.empty((X.shape[0], len(SHAP_GROUPS)), dtype=np.float32)
    device = xgb_device_for(0)
    batch = _S11_SHAP_BATCH
    try:
        booster.set_param({"device": device})
        s = 0
        while s < X.shape[0]:
            e = min(X.shape[0], s + batch)
            try:
                phi = booster.predict(xgb.DMatrix(X[s:e], missing=np.nan), pred_contribs=True)
            except Exception as exc:  # noqa: BLE001
                if device != "cpu" and is_gpu_oom_error(exc):
                    device = "cpu"
                    booster.set_param({"device": "cpu"})
                    log("11: GPU OOM in TreeSHAP; continuing on the CPU", "WARNING")
                    continue
                raise
            out[s:e] = (phi[:, :len(FEATURES)].astype(np.float64) @ _S11_MEMBER).astype(np.float32)
            s = e
    finally:
        booster.set_param({"device": "cpu"})
    out[~np.isfinite(X[:, _S11_BASE_IDX]).all(axis=1)] = np.nan
    return out

## 11.5 Demand, climate deltas and the weather-normalised baseline

In [ ]:
def _s11_ratio_table():
    ug = FUTURE_INPUTS["urban_growth"]
    tab = {}
    for (s, d), grp in ug.groupby(["scenario", "district"]):
        tab[(str(s), int(d))] = dict(zip(grp["frame"].astype(int), grp["ratio"].astype(float)))
    return tab


def _s11_interp_ratio(series, year):
    """Geometric interpolation of an urban-area ratio series (frame -> ratio) at ``year``."""
    if year in series:
        return float(series[year])
    xs = sorted(series)
    if not xs:
        return float("nan")
    if year <= xs[0]:
        return float(series[xs[0]])
    if year >= xs[-1]:
        return float(series[xs[-1]])
    lo = max(x for x in xs if x <= year)
    hi = min(x for x in xs if x >= year)
    w = (year - lo) / (hi - lo)
    return float(series[lo] ** (1.0 - w) * series[hi] ** w)


def _s11_demand(s, u_now):
    """Target district totals per projection frame: U_d(2025) x g_d,s(y) / g_d,s(2025)."""
    tab = _s11_ratio_table()
    base = _s11_district_totals(u_now)
    out, missing = {}, []
    for y in _S11_PROJ:
        targets = {}
        med = [(_s11_interp_ratio(v, y) / _s11_interp_ratio(v, _S11_NOW)) for (sc, _), v in tab.items() if sc == s]
        fallback = float(np.median(med)) if med else 1.0
        for d in _S11_DIST_IDS:
            series = tab.get((s, d))
            if series is None:
                g = fallback
                if y == _S11_PROJ[0]:
                    missing.append(d)
            else:
                g = _s11_interp_ratio(series, y) / _s11_interp_ratio(series, _S11_NOW)
            targets[d] = base[d] * max(float(g), 1.0)       # built-up never shrinks
        out[y] = targets
    if missing:
        log(f"11: {s}: no urban-growth ratio for districts {missing}; the scenario's median ratio is used", "WARNING")
    return out


def _s11_pop_factor(s, y):
    pg = FUTURE_INPUTS["pop_growth"]
    sub = pg[pg["scenario"] == s]
    if sub.empty:
        log(f"11: no population factor for {s}; held at 1", "WARNING")
        return 1.0
    series = dict(zip(sub["frame"].astype(int), sub["factor"].astype(float)))
    return _s11_interp_ratio(series, int(y)) / _s11_interp_ratio(series, _S11_NOW)


def _s11_cmip6_deltas():
    df = FUTURE_INPUTS["cmip6"]
    gsat = FUTURE_INPUTS.get("gsat_increment") or {}
    # baseline_window = the 20-year GCM window of the baseline frame (2016-2035); observation_window = the frame's
    # observation window (2024-2026) whose weather anomaly L_base removes (D15).
    gcm_window = CFG.PROJ_WINDOWS.get(_S11_NOW, (_S11_NOW - 9, _S11_NOW + 10))
    out = {"available": True, "baseline_frame": _S11_NOW, "baseline_window": [int(v) for v in gcm_window],
           "observation_window": [int(v) for v in CFG.window(_S11_NOW)], "frames": list(_S11_PROJ),
           "variables": {"day": "tasmax", "night": "tasmin"}, "scenarios": {},
           "definition": "delta = GCM Apr-Jun 20-year-window mean at the frame minus its 2016-2035 (2025) window"}

    def gcm_deltas(scen, var):
        sub = df[(df["scenario"] == scen) & (df["var"] == var)]
        piv = sub.pivot_table(index="model", columns="frame", values="value_k", aggfunc="mean")
        need = [_S11_NOW] + _S11_PROJ
        if any(c not in piv.columns for c in need):
            return None
        piv = piv[need].dropna()
        return {"models": [str(mm) for mm in piv.index],
                "delta": {y: (piv[y] - piv[_S11_NOW]).to_numpy(np.float64) for y in _S11_PROJ}}

    for s in _S11_SCEN:
        rec = {"variables": {}}
        direct = all(gcm_deltas(s, v) is not None for v in ("tasmax", "tasmin")) if s in set(df["scenario"]) \
            else False
        for var in ("tasmax", "tasmin"):
            if direct:
                d = gcm_deltas(s, var)
                rec["method"] = "gcm"
            else:
                d = gcm_deltas("ssp245", var)
                if d is None:
                    raise RuntimeError(f"11: no CMIP6 data for {s} and no ssp245 to pattern-scale from")
                ratios = {}
                for y in _S11_PROJ:
                    g245 = float((gsat.get("ssp245") or {}).get(y, float("nan")))
                    gs = float((gsat.get(s) or {}).get(y, float("nan")))
                    ratios[y] = gs / g245 if (np.isfinite(g245) and g245 > 0 and np.isfinite(gs)) else 1.0
                d = {"models": d["models"], "delta": {y: d["delta"][y] * ratios[y] for y in _S11_PROJ}}
                rec["method"] = "pattern-scaled"
                rec["gsat_ratio"] = ratios
                rec["source_scenario"] = "ssp245"
            summary = {}
            for y in _S11_PROJ:
                v = d["delta"][y]
                summary[y] = {"mean": float(v.mean()), "sd": float(v.std()), "p05": float(np.percentile(v, 5)),
                              "p50": float(np.median(v)), "p95": float(np.percentile(v, 95)), "n": int(v.size)}
            rec["variables"][var] = {"models": d["models"], "delta": d["delta"], "summary": summary}
        rec["n_gcm"] = min(len(rec["variables"][v]["models"]) for v in ("tasmax", "tasmin"))
        out["scenarios"][s] = rec
    pat = FUTURE_INPUTS.get("cmip6_pattern") or {}
    out["pattern"] = {}
    for var in ("tasmax", "tasmin"):
        p = np.asarray(pat.get(var, np.ones((_S11_H, _S11_W))), np.float64)[CELL_ROWS, CELL_COLS]
        p = np.where(np.isfinite(p), p, 1.0)
        out["pattern"][var] = {"cells": p.astype(np.float32), "min": float(p.min()), "max": float(p.max()),
                               "mean": float(p.mean())}
    return out


def _s11_l_base(m):
    """Weather-normalised baseline level L_base,m (SPEC §4.10)."""
    cs = CLIMATE_SENSITIVITY[m]
    var = cs.get("temperature_var") or ("era5_tmax" if m == "day" else "era5_tmin")
    sub = ANNUAL_SERIES[(ANNUAL_SERIES["variable"] == var) & (ANNUAL_SERIES["scope"] == "ncr")]
    s = pd.Series(sub["value"].to_numpy(np.float64), index=sub["year"].to_numpy(np.int64)).groupby(level=0).mean()
    w0, w1 = CFG.window(_S11_NOW)
    now_years = [y for y in range(int(w0), int(w1) + 1) if y in s.index and np.isfinite(s[y])]
    t0, t1 = _S11_TREND_YEARS
    trend_years = [y for y in range(int(t0), int(t1) + 1) if y in s.index and np.isfinite(s[y])]
    ref_years = [y for y in range(_S11_NOW - 9, _S11_NOW + 1) if y in s.index and np.isfinite(s[y])]
    level = float(LEVELS[m][_S11_NOW]["level"])
    slope_dec = None
    if not now_years:
        log(f"11: {m}: {var} annual series has no value in the {_S11_NOW} window; baseline = observed level "
            "(no weather normalisation)", "WARNING")
        anomaly, method = 0.0, "none (no annual value in the observation window)"
    elif len(trend_years) >= _S11_TREND_MIN_YEARS:
        # anomaly = window mean - OLS trend line at the window's mean year (D15)
        ty = np.asarray(trend_years, np.float64)
        tv = s[trend_years].to_numpy(np.float64)
        slope, icpt = np.polyfit(ty, tv, 1)
        t_mid = float(np.mean(now_years))
        anomaly = float(s[now_years].mean() - (icpt + slope * t_mid))
        slope_dec = float(slope * 10.0)
        method = (f"window mean {now_years[0]}-{now_years[-1]} minus the {trend_years[0]}-{trend_years[-1]} OLS "
                  f"trend of the ERA5-Land NCR annual {var} at {t_mid:g}")
    elif ref_years:
        log(f"11: {m}: only {len(trend_years)} annual {var} values in {t0}-{t1} (< {_S11_TREND_MIN_YEARS}); the "
            f"weather anomaly falls back to the window mean minus the {ref_years[0]}-{ref_years[-1]} mean", "WARNING")
        anomaly = float(s[now_years].mean() - s[ref_years].mean())
        method = f"window mean minus the {ref_years[0]}-{ref_years[-1]} mean (fallback: too few years for a trend)"
    else:
        log(f"11: {m}: {var} annual series incomplete; baseline = observed level (no weather normalisation)",
            "WARNING")
        anomaly, method = 0.0, "none (annual series incomplete)"
    beta = float(cs["beta"])
    return {"l_base": level - beta * anomaly, "level_now": level, "beta": beta, "weather_anomaly_c": anomaly,
            "temperature_var": var, "window_years": now_years, "reference_years": ref_years,
            "trend_slope_per_decade": slope_dec, "trend_years": trend_years, "method": method}

## 11.6 Land realisations per scenario (stage-cached)
Each realisation is driven through 2030 … 2050; at every frame the projected feature matrix is built and the
final models of both modes predict the raw anomaly $\hat A$. The **median realisation** (the medoid — the
realisation closest to the per-cell median of $u$ over all frames) provides the features for the AOA flags and
the projected zones.

In [ ]:
def _s11_land_scenario(s, ctx):
    t0 = _s11_time.perf_counter()
    st_now = ctx["state_now"]
    demand = _s11_demand(s, st_now["u"])
    path = [(y, demand[y]) for y in _S11_PROJ]
    n_r, n_y = int(CFG.N_LAND_REAL), len(_S11_PROJ)
    A = {m: np.empty((n_r, n_y, _S11_NC), dtype=np.float32) for m in _S11_MODES}
    U = np.empty((n_r, n_y, _S11_NC), dtype=np.float32)
    states_crop = np.empty((n_r, n_y, _S11_NC), dtype=np.float32)
    states_barren = np.empty((n_r, n_y, _S11_NC), dtype=np.float32)
    pops = np.empty((n_r, n_y, _S11_NC), dtype=np.float32)
    short_tot = np.zeros((n_r, n_y))
    donors = []

    def pop_fn(st, year):
        return _s11_pop_rule(st["u"], st_now["pop"], st_now["u"], _s11_pop_factor(s, year))

    for r in range(n_r):
        rng = np.random.default_rng([int(CFG.SEED), 1160, r])     # common random numbers across scenarios
        res = _s11_run_ca(st_now, path, rng, ctx["suit"], _S11_NOW, pop_fn)
        for j, (_y, st, info) in enumerate(res):
            U[r, j], states_crop[r, j], states_barren[r, j], pops[r, j] = st["u"], st["crop"], st["barren"], st["pop"]
            short_tot[r, j] = sum(info["shortfall"].values())
            donors.append(info["donor"])
            X = _s11_features(st, ctx)
            for m in _S11_MODES:
                A[m][r, j] = _s11_predict(m, X)
            del X
    med = np.median(U, axis=0)
    medoid = int(np.argmin(((U - med[None]) ** 2).sum(axis=(1, 2))))
    di, zones, shap_mean = {m: [] for m in _S11_MODES}, {m: [] for m in _S11_MODES}, {m: [] for m in _S11_MODES}
    for j in range(n_y):
        st = {"u": U[medoid, j].astype(np.float64), "forest": st_now["forest"], "water": st_now["water"],
              "crop": states_crop[medoid, j].astype(np.float64), "barren": states_barren[medoid, j].astype(np.float64),
              "pop": pops[medoid, j].astype(np.float64), "lc_ok": st_now["lc_ok"], "pop_ok": st_now["pop_ok"]}
        X = _s11_features(st, ctx)
        for m in _S11_MODES:
            di[m].append(COMPUTE_DI(m, X).astype(np.float32))
            if m in ZONE_MODEL:
                G = _s11_group_shap(m, X)
                zones[m].append(ASSIGN_ZONES(m, G))
                shap_mean[m].append(np.nanmean(G, axis=0).astype(np.float32))
                del G
            else:
                zones[m].append(np.full(_S11_NC, _S11_ZONE_NODATA, np.uint8))
                shap_mean[m].append(np.full(len(SHAP_GROUPS), np.nan, np.float32))
        del X
    tot = {k: float(sum(dn[k] for dn in donors)) for k in ("cropland", "barren", "other_vegetation")}
    tsum = sum(tot.values())
    driver = {}
    for j, y in enumerate(_S11_PROJ):
        rows = {}
        for d, idx in _S11_DIST_CELLS.items():
            rows[d] = {"name": _S11_DIST_NAME.get(d, str(d)),
                       "built_km2": float(U[:, j, idx].sum(axis=1).mean() * _S11_CELL_KM2),
                       "built_frac": float(U[:, j, idx].mean()),
                       "pop": float(pops[:, j, idx].sum(axis=1).mean() * _S11_CELL_KM2),
                       "demand_km2": float(demand[y][d] * _S11_CELL_KM2)}
        rows["ncr"] = {"name": "Delhi NCR", "built_km2": float(U[:, j].sum(axis=1).mean() * _S11_CELL_KM2),
                       "built_frac": float(U[:, j].mean()),
                       "pop": float(pops[:, j].sum(axis=1).mean() * _S11_CELL_KM2),
                       "demand_km2": float(sum(demand[y].values()) * _S11_CELL_KM2),
                       "shortfall_km2": float(short_tot[:, j].mean() * _S11_CELL_KM2),
                       "pop_factor": _s11_pop_factor(s, y)}
        driver[y] = rows
    out = {"A": A, "U": U, "medoid": medoid, "di": {m: np.stack(v) for m, v in di.items()},
           "zones": {m: np.stack(v) for m, v in zones.items()},
           "shap_mean": {m: np.stack(v) for m, v in shap_mean.items()}, "driver": driver,
           "donor_shares": {k: v / tsum for k, v in tot.items()} if tsum > 0 else None,
           "shortfall_km2": {y: float(short_tot[:, j].mean() * _S11_CELL_KM2) for j, y in enumerate(_S11_PROJ)},
           "seconds": round(_s11_time.perf_counter() - t0, 1)}
    del states_crop, states_barren, pops
    return out

## 11.7 Monte Carlo per scenario × mode
Draw parameters (GCM rank, β index, realisation index) are shared by all scenarios and modes; the per-cell model
error is generated chunk by chunk from a seeded uniform $U_{k,i}$ mapped through the **sorted** residual sample of
the cell's DI decile (a quantile mapping, so a cell keeps its error rank when its DI decile changes between
frames). The residual sample is the **projection-setting** one of section 07 (`resid_samples_proj`: signed
$y − (\hat y_{LOEO} + \bar r)$ by DI decile) when it is complete, else the block20 OOF sample (`resid_samples`).
Spatial correlation enters through a block20 Gaussian copula: $z_{kb} \sim N(0, 1)$ per draw $k$ and 20 km block
$b$ (drawn once per run from `default_rng([SEED, 1181])`, shared by all scenarios, modes and chunks),
$z_{ki} = Φ^{-1}(U_{k,i})$ and $u_{ki} = Φ(\sqrt{ρ}\,z_{kb(i)} + \sqrt{1 − ρ}\,z_{ki})$ with
ρ = `AOA[m]['resid_rho_block20']` ∈ [0, 0.95]; ρ = 0 uses $U_{k,i}$ directly (bit-identical to independent
draws). A DI of +inf (row with a missing weighted feature, section 07) falls into the top decile and is flagged
outside the AOA. Scope sums of every draw are accumulated with a dense (cells × scopes) membership matrix per chunk.

In [ ]:
def _s11_scopes(m):
    out = [("ncr", None)]
    out += [(f"d:{d}", d) for d in _S11_DIST_IDS]
    if m in ZONE_MODEL:
        out += [(f"z:{m}:{k}", ("zone", k)) for k in range(4)]
    return out


def _s11_membership(scopes, zones_y, sl):
    n = sl.stop - sl.start
    M = np.zeros((n, len(scopes)), dtype=np.float32)
    dist = _S11_DISTRICT[sl]
    for j, (_sid, key) in enumerate(scopes):
        if key is None:
            M[:, j] = 1.0
        elif isinstance(key, tuple):
            M[:, j] = (zones_y[sl] == key[1])
        else:
            M[:, j] = (dist == key)
    return M


def _s11_draws():
    rng = np.random.default_rng([int(CFG.SEED), 1170])
    n = int(CFG.N_MC)
    out = {"g_u": rng.random(n), "beta_u": rng.random(n), "r_idx": rng.integers(0, int(CFG.N_LAND_REAL), n)}
    # block20 copula factors (D13): one N(0, 1) per draw and 20 km block, shared by every scenario, mode and chunk
    blk = DF["block20_id"].to_numpy()[:_S11_NC]
    _ids, inv = np.unique(blk, return_inverse=True)
    out["block_idx"] = inv.astype(np.int64).reshape(-1)
    out["z_block"] = np.random.default_rng([int(CFG.SEED), 1181]).standard_normal((n, int(_ids.size)))
    return out


def _s11_error_model(m, n_dec):
    """(sorted residual samples per DI decile, source label, rho) of the Monte-Carlo model error (D13)."""
    proj = AOA[m].get("resid_samples_proj")
    samples, source = None, "block20"
    if proj is not None:
        if len(proj) == n_dec and all(np.asarray(v).size > 0 for v in proj):
            samples = [np.sort(np.asarray(v, np.float32)) for v in proj]
            source = "loeo+rbar"
        else:
            log(f"11: {m}: projection-setting residual samples incomplete ({len(proj)} deciles, "
                f"{sum(int(np.asarray(v).size == 0) for v in proj)} empty); using the block20 OOF samples", "WARNING")
    if samples is None:
        samples = [np.sort(np.asarray(v, np.float32)) for v in AOA[m].get("resid_samples", [])]
        if len(samples) != n_dec or any(v.size == 0 for v in samples):
            log(f"11: {m}: conformal residual samples incomplete; empty deciles contribute zero model error",
                "WARNING")
            samples = [v if v.size else np.zeros(1, np.float32) for v in samples] or [np.zeros(1, np.float32)]
            samples += [samples[-1]] * (n_dec - len(samples))
    rho = AOA[m].get("resid_rho_block20", 0.0)
    rho = float(rho) if rho is not None and np.isfinite(float(rho)) else 0.0
    return samples, source, float(np.clip(rho, 0.0, _S11_RHO_MAX))


def _s11_monte_carlo(s, m, land, ctx):
    t0 = _s11_time.perf_counter()
    draws = ctx["draws"]
    n_mc, n_y = int(CFG.N_MC), len(_S11_PROJ)
    cm = CMIP6_DELTAS["scenarios"][s]["variables"][_S11_TEMP_VAR[m]]
    n_g = len(cm["models"])
    g_idx = np.minimum((draws["g_u"] * n_g).astype(np.int64), n_g - 1)
    beta_all = np.sort(np.asarray(CLIMATE_SENSITIVITY[m]["draws"], np.float64))
    beta_k = beta_all[np.minimum((draws["beta_u"] * beta_all.size).astype(np.int64), beta_all.size - 1)]
    r_idx = draws["r_idx"]
    lb = ctx["l_base"][m]["l_base"]
    pat = CMIP6_DELTAS["pattern"][_S11_TEMP_VAR[m]]["cells"].astype(np.float32)
    rbar = np.nan_to_num(np.asarray(PERSISTENT_RESID[m], np.float64)[CELL_ROWS, CELL_COLS], nan=0.0)
    a_base = ctx["a_base"][m]
    a_mean = float(a_base[_S11_FIXED[m]].mean())
    edges = np.asarray(AOA[m]["conformal"]["edges"], np.float64)
    samples, err_source, rho = _s11_error_model(m, edges.size + 1)
    blk_idx, z_block = draws["block_idx"], draws["z_block"]
    sq_rho, sq_1rho = np.sqrt(rho), np.sqrt(1.0 - rho)
    thr = float(AOA[m]["threshold"])
    scopes = _s11_scopes(m)
    n_sc = len(scopes)
    quant = {k: np.empty((n_y, _S11_NC), dtype=np.float32) for k in _S11_QK}
    flags = np.zeros((n_y, _S11_NC), dtype=np.uint8)
    scope_sum = np.zeros((n_y, n_mc, n_sc))
    e_sum = np.zeros((n_y, n_mc, n_sc))
    land_sc = np.zeros((n_y, int(CFG.N_LAND_REAL), n_sc))
    pat_sc = np.zeros((n_y, n_sc))
    base_sc = np.zeros((n_y, n_sc))
    count_sc = np.zeros((n_y, n_sc))
    adev_base = (a_base - a_mean).astype(np.float32)
    for j, y in enumerate(_S11_PROJ):
        clim_k = (beta_k * cm["delta"][y][g_idx]).astype(np.float32)       # (n_mc,)
        adev = land["A"][m][:, j, :] - np.float32(a_mean)                    # (R, N)
        dec = np.clip(np.searchsorted(edges, land["di"][m][j].astype(np.float64), side="right"), 0, len(samples) - 1)
        zones_y = land["zones"][m][j]
        for ci, s0 in enumerate(range(0, _S11_NC, _S11_CELL_CHUNK)):
            sl = slice(s0, min(_S11_NC, s0 + _S11_CELL_CHUNK))
            nc = sl.stop - sl.start
            uni = np.random.default_rng([int(CFG.SEED), 1180, ci]).random((n_mc, nc), dtype=np.float32)
            if rho > 0.0:
                # block20 Gaussian copula: u = Phi(sqrt(rho) z_kb + sqrt(1 - rho) z_ki), z_ki = Phi^-1(uni)
                z = _s11_special.ndtri(np.clip(uni.astype(np.float64), _S11_U_CLIP, 1.0 - _S11_U_CLIP))
                z *= sq_1rho
                z += sq_rho * z_block[:, blk_idx[sl]]
                uni = _s11_special.ndtr(z)
                del z
            e = np.empty((n_mc, nc), dtype=np.float32)
            dec_c = dec[sl]
            for d in np.unique(dec_c):
                cols = np.flatnonzero(dec_c == d)
                smp = samples[int(d)]
                e[:, cols] = smp[np.minimum((uni[:, cols] * smp.size).astype(np.int64), smp.size - 1)]
            del uni
            lst = (np.float32(lb) + clim_k[:, None] * pat[None, sl] + adev[:, sl][r_idx]
                   + rbar[sl].astype(np.float32)[None, :] + e)
            q = np.quantile(lst, _S11_Q, axis=0)
            for k, key in enumerate(_S11_QK):
                quant[key][j, sl] = q[k]
            M = _s11_membership(scopes, zones_y, sl)
            scope_sum[j] += lst.astype(np.float64) @ M
            e_sum[j] += e.astype(np.float64) @ M
            land_sc[j] += adev[:, sl].astype(np.float64) @ M - (adev_base[sl].astype(np.float64) @ M)[None, :]
            pat_sc[j] += pat[sl].astype(np.float64) @ M
            base_sc[j] += (adev_base[sl].astype(np.float64) + rbar[sl]) @ M
            count_sc[j] += M.sum(axis=0)
            del lst, e, q, M
        half = 0.5 * (quant["p95"][j] - quant["p05"][j])
        tier = np.searchsorted(_S11_VSUP_EDGES, half, side="right").astype(np.uint8)
        flags[j] = (tier << np.uint8(1)) | (land["di"][m][j] > thr).astype(np.uint8)
    # ---- scope products -------------------------------------------------------------------------------------------
    beta_bar, beta_var = float(beta_all.mean()), float(beta_all.var())
    series, decomp, budget, thresholds = {}, {y: {} for y in _S11_PROJ}, {y: {} for y in _S11_PROJ}, {}
    for c, (sid, _) in enumerate(scopes):
        rows, thr_years = [], {"+1": None, "+2": None, "+3": None}
        for j, y in enumerate(_S11_PROJ):
            n = count_sc[j, c]
            if n <= 0:
                rows.append({"year": int(y), **{k: None for k in _S11_QK}})
                continue
            means = scope_sum[j, :, c] / n
            qv = np.quantile(means, _S11_Q)
            rows.append({"year": int(y), **{k: float(v) for k, v in zip(_S11_QK, qv)}})
            dT = cm["delta"][y]
            p_sc = pat_sc[j, c] / n
            land_r = land_sc[j, :, c] / n
            # the scope's own weather-normalised 2025 baseline: every draw mean = baseline + climate + land + error,
            # so total = p50 - baseline ~ climate + land + model_error_mean for districts and zones too (D14)
            baseline = lb + base_sc[j, c] / n
            decomp[y][sid] = {"climate": beta_bar * float(dT.mean()) * p_sc, "land": float(land_r.mean()),
                              "total": float(qv[2] - baseline), "baseline_c": float(baseline),
                              "baseline_offset": float(base_sc[j, c] / n),
                              "model_error_mean": float((e_sum[j, :, c] / n).mean())}
            parts = {"gcm": (beta_bar * p_sc) ** 2 * float(dT.var()),
                     "sensitivity": beta_var * (float(dT.mean()) * p_sc) ** 2,
                     "land": float(land_r.var()), "model": float((e_sum[j, :, c] / n).var())}
            tot = sum(parts.values())
            budget[y][sid] = {**{k: (v / tot if tot > 0 else 0.0) for k, v in parts.items()},
                              "total_variance": tot, "draw_sd_c": float(means.std())}
            for key, inc in (("+1", 1.0), ("+2", 2.0), ("+3", 3.0)):
                if thr_years[key] is None and qv[2] > baseline + inc:
                    thr_years[key] = int(y)
        series[sid] = rows
        thresholds[sid] = thr_years
    ncr = {y: decomp[y].get("ncr") for y in _S11_PROJ}
    meta = {"seconds": round(_s11_time.perf_counter() - t0, 1), "n_mc": n_mc, "n_gcm": n_g,
            "n_land_real": int(CFG.N_LAND_REAL), "beta_mean": beta_bar, "beta_sd": float(np.sqrt(beta_var)),
            "l_base": lb, "a2025_fixed_mean": a_mean, "scopes": [sid for sid, _ in scopes],
            "model_error": {"samples": err_source, "rho": rho}}
    out_decomp = {y: {**(ncr[y] or {}), "by_scope": decomp[y]} for y in _S11_PROJ}
    out_budget = {y: {**{k: v for k, v in (budget[y].get("ncr") or {}).items()}, "by_scope": budget[y]}
                  for y in _S11_PROJ}
    return quant, flags, series, out_decomp, out_budget, thresholds, meta

## 11.8 Run: suitability, hindcast, derived predictors, CMIP6 deltas, baseline, scenarios

In [ ]:
def _s11_hash(*parts):
    h = _s11_hashlib.sha1()
    for p in parts:
        if isinstance(p, np.ndarray):
            h.update(str(p.dtype).encode())
            h.update(np.ascontiguousarray(p).tobytes())
        elif isinstance(p, (bytes, bytearray)):
            h.update(bytes(p))
        elif isinstance(p, pd.DataFrame):
            h.update(pd.util.hash_pandas_object(p, index=False).to_numpy().tobytes())
        else:
            h.update(repr(p).encode())
        h.update(b"|")
    return h.hexdigest()[:16]


def _s11_land_fingerprint():
    parts = [_S11_VERSION, _S11_PROJ, _S11_SCEN, _S11_NOW]
    for y in _S11_FRAMES_OBS:
        for f in _S11_FRACS + _S11_H_TARGETS + ["log_pop", "dist_urban"]:
            v = np.asarray(FEATURE_GRIDS[y][f], np.float64)
            parts.append(np.array([np.nansum(v), np.isfinite(v).sum()]))
        parts.append(np.array([np.nansum(np.asarray(RAW_OBS[y]["pop"], np.float64))]))
    for m in _S11_MODES:
        parts.append(bytes(FINAL_MODELS[m].save_raw("ubj")))
        parts += [float(AOA[m]["threshold"]), float(AOA[m]["d_bar"])]
        if m in ZONE_MODEL:
            parts.append(np.asarray(ZONE_MODEL[m]["centroids"], np.float64))
    parts += [FUTURE_INPUTS["urban_growth"], FUTURE_INPUTS["pop_growth"]]
    return _s11_hash(*parts)


if _S11_ACTIVE:
    _S11_T0 = _s11_time.perf_counter()
    _s11_h0, _s11_h1, _s11_hind_ok = _s11_pick_pair(2015, _S11_NOW, _S11_NOW)
    _s11_c0, _s11_c1, _s11_cal_ok = _s11_pick_pair(2005, 2015, _s11_h0 if _s11_h0 is not None else _S11_NOW)
    if _s11_c0 is None:
        raise RuntimeError("11: projections need at least two observed frames for the land-change model")
    _s11_indep = bool(_s11_cal_ok and _s11_h0 is not None and _s11_c1 <= _s11_h0)
    with timer("11_suitability"):
        _S11_SUIT, _S11_SUIT_INFO = _s11_fit_suitability(_s11_c0, _s11_c1, _s11_h0, _s11_h1)
    log(f"11: suitability {_s11_c0}->{_s11_c1}: n={_S11_SUIT_INFO['n']:,} ({_S11_SUIT_INFO['n_positive']} positive), "
        f"AUC train {_S11_SUIT_INFO['auc_train']:.3f}, block20 OOF {_S11_SUIT_INFO['auc_block20_oof']}, "
        f"hindcast {_S11_SUIT_INFO['auc_hindcast']}")
    if _s11_h0 is not None and _s11_h1 is not None and _s11_h0 < _s11_h1:
        with timer("11_land_hindcast"):
            _S11_HIND, _S11_HIND_MAPS = _s11_hindcast(_S11_SUIT, _s11_h0, _s11_h1, _s11_indep)
        log(f"11: land hindcast {_s11_h0}->{_s11_h1}: FoM {_S11_HIND['fom']:.3f} (sd {_S11_HIND['fom_sd']:.3f}) vs "
            f"null {_S11_HIND['fom_null']:.3f}; quantity {_S11_HIND['quantity_disagreement']:.4f}, allocation "
            f"{_S11_HIND['allocation_disagreement']:.4f}; RMSE du {_S11_HIND['rmse_du']:.4f} vs null "
            f"{_S11_HIND['rmse_du_null']:.4f}; corr du {_S11_HIND['corr_du']} vs null {_S11_HIND['corr_du_null']}; "
            f"verdict {_S11_HIND['verdict']} (truth: {_S11_HIND['hindcast_truth']})")
    else:
        _S11_HIND, _S11_HIND_MAPS = {"available": False, "reason": "no hindcast frame pair"}, None
    with timer("11_derived_predictors"):
        _S11_H_MODELS, _S11_H_REPORT, _S11_H_FRAMES = _s11_fit_h()
    log("11: delta-mapping models h_X (R2 on block20 fold 0): "
        + ", ".join(f"{k}={v['r2_holdout']:.2f}" if v["r2_holdout"] is not None else f"{k}=n/a"
                    for k, v in _S11_H_REPORT.items()))
    CMIP6_DELTAS = _s11_cmip6_deltas()
    for _s11_s, _s11_r in CMIP6_DELTAS["scenarios"].items():
        log(f"11: CMIP6 {_s11_s} ({_s11_r['method']}, {_s11_r['n_gcm']} GCMs): delta {_S11_PROJ[-1]} tasmax "
            f"{_s11_r['variables']['tasmax']['summary'][_S11_PROJ[-1]]['mean']:+.2f} K, tasmin "
            f"{_s11_r['variables']['tasmin']['summary'][_S11_PROJ[-1]]['mean']:+.2f} K")

    # shared context of all scenarios
    _s11_state_now = _s11_state(_S11_NOW)
    _s11_era5, _s11_era5_frames = _s11_era5_climatology()
    _s11_now_fracs = {f: _s11_cells(FEATURE_GRIDS[_S11_NOW][f]).astype(np.float64) for f in _S11_FRACS}
    _s11_design_now = _s11_lc_design(_s11_state_fracs(_s11_state_now))
    _S11_CTX = {"state_now": _s11_state_now, "suit": _S11_SUIT, "era5": _s11_era5, "now_fracs": _s11_now_fracs,
                "h_models": _S11_H_MODELS,
                "h_now": {t: (_S11_H_MODELS[t].inplace_predict(_s11_design_now).astype(np.float64)
                              if _S11_H_MODELS.get(t) is not None else None) for t in _S11_H_TARGETS},
                "x_now": {t: _s11_cells(FEATURE_GRIDS[_S11_NOW][t]).astype(np.float64) for t in _S11_H_TARGETS},
                "draws": _s11_draws()}
    _s11_x_base = _s11_features(_s11_state_now, _S11_CTX)
    _S11_CTX["a_base"] = {m: _s11_predict(m, _s11_x_base) for m in _S11_MODES}
    # Consistency check: the 2025 baseline features reproduce the observed 2025 features except ERA5 (climatology).
    _s11_obs_now = FEATURE_MATRIX(FEATURE_GRIDS[_S11_NOW])
    _s11_non_era5 = [i for i, f in enumerate(FEATURES) if f not in _S11_ERA5]
    _s11_dev = np.nanmax(np.abs(_s11_x_base[:, _s11_non_era5] - _s11_obs_now[:, _s11_non_era5]))
    if not _s11_dev < 1e-3:
        raise AssertionError(f"11: the 2025 projected features differ from the observed ones by {_s11_dev:.3g}")
    del _s11_obs_now, _s11_x_base
    _S11_CTX["l_base"] = {m: _s11_l_base(m) for m in _S11_MODES}
    for _m, _b in _S11_CTX["l_base"].items():
        _s11_tr = ("" if _b["trend_slope_per_decade"] is None else
                   f"; ERA5 trend {_b['trend_slope_per_decade']:+.3f} °C/decade over {len(_b['trend_years'])} years")
        log(f"11: {_m} baseline L_base = {_b['l_base']:.2f} °C (observed {_b['level_now']:.2f}, weather anomaly "
            f"{_b['weather_anomaly_c']:+.3f} °C x beta {_b['beta']:.2f}{_s11_tr}; {_b['method']})")

    _s11_land_fp = _s11_land_fingerprint()
    # The land stage executes COMPUTE_DI (section 07) and ASSIGN_ZONES / grouped SHAP inputs (section 08), so their
    # code identity is part of its key (D4); without section_code_hash (older 00) both are None.
    _s11_sch = globals().get("section_code_hash")
    _s11_code07 = _s11_sch("07") if _s11_sch is not None else None
    _s11_code08 = _s11_sch("08") if _s11_sch is not None else None
    _S11_LAND_INFO = {}
    for _s11_s in _S11_SCEN:
        _s11_key = CFG.key("N_LAND_REAL", "SEED", "FRAMES_PROJ", version=_S11_VERSION, scenario=_s11_s,
                           land=_s11_land_fp, suit=_S11_SUIT_INFO["auc_train"], eps=_S11_EPS_SIGMA,
                           code07=_s11_code07, code08=_s11_code08)
        with timer(f"11_land_{_s11_s}"):
            _s11_land = stage_cache(f"11_land_{_s11_s}", _s11_key).get_or_compute(
                lambda s=_s11_s: _s11_land_scenario(s, _S11_CTX))
        _S11_LAND_INFO[_s11_s] = {"medoid": _s11_land["medoid"], "donor_shares": _s11_land["donor_shares"],
                                  "shortfall_km2": _s11_land["shortfall_km2"], "seconds": _s11_land["seconds"],
                                  "u_mean": {int(y): float(_s11_land["U"][:, j].mean())
                                             for j, y in enumerate(_S11_PROJ)}}
        PROJ_DRIVER_SUMMARY[_s11_s] = _s11_land["driver"]
        for _m in _S11_MODES:
            with timer(f"11_mc_{_s11_s}_{_m}"):
                _q, _fl, _ser, _dec, _bud, _thr, _meta = _s11_monte_carlo(_s11_s, _m, _s11_land, _S11_CTX)
            PROJ_QUANTILES.setdefault(_m, {})[_s11_s] = _q
            PROJ_FLAGS.setdefault(_m, {})[_s11_s] = _fl
            PROJ_ZONES.setdefault(_m, {})[_s11_s] = np.asarray(_s11_land["zones"][_m], np.uint8)
            PROJ_SERIES.setdefault(_m, {})[_s11_s] = _ser
            PROJ_DECOMPOSITION.setdefault(_m, {})[_s11_s] = _dec
            PROJ_UNCERTAINTY.setdefault(_m, {})[_s11_s] = _bud
            PROJ_THRESHOLD_YEARS.setdefault(_m, {})[_s11_s] = _thr
            PROJ_META.setdefault(_m, {})[_s11_s] = {
                **_meta, "frames": list(_S11_PROJ), "method": CMIP6_DELTAS["scenarios"][_s11_s]["method"],
                "pct_outside_aoa": {int(y): float(np.mean(_s11_land["di"][_m][j] > AOA[_m]["threshold"]))
                                    for j, y in enumerate(_S11_PROJ)},
                "shap_group_mean": {int(y): _s11_land["shap_mean"][_m][j] for j, y in enumerate(_S11_PROJ)},
                "note": _S11_NOTE, "omitted_feedbacks": list(_S11_OMITTED)}
            _ncr, _dl = _ser["ncr"][-1], _dec[_S11_PROJ[-1]]
            log(f"11: {_s11_s} {_m}: NCR {_ncr['year']} p50 {_ncr['p50']:.2f} °C [{_ncr['p05']:.2f}, {_ncr['p95']:.2f}]"
                f"; vs baseline {_dl['baseline_c']:.2f} °C: climate {_dl['climate']:+.2f}, land {_dl['land']:+.2f}, "
                f"total {_dl['total']:+.2f} °C; budget "
                + ", ".join(f"{k} {_bud[_S11_PROJ[-1]][k]:.2f}" for k in ("gcm", "sensitivity", "land", "model"))
                + f"; model error {_meta['model_error']['samples']} rho {_meta['model_error']['rho']:.2f}"
                + f"; thresholds {_thr['ncr']}")
            del _q, _fl
        del _s11_land
        free_memory()

    # GHSL-2030 demand comparison (SPEC §4.9: report the GHSL-2030 demand alongside the SSP demand)
    _s11_ghsl = (FUTURE_INPUTS.get("ghsl_2030") or {}).get("built")
    _s11_ghsl_cmp = None
    if _s11_ghsl is not None and 2030 in _S11_PROJ:
        _s11_gc = np.nan_to_num(np.asarray(_s11_ghsl, np.float64)[CELL_ROWS, CELL_COLS], nan=0.0)
        _s11_ghsl_cmp = {"ncr_ghsl_2030_km2": float(_s11_gc.sum() * _S11_CELL_KM2),
                         "ncr_2025_km2": float(_s11_state_now["u"].sum() * _S11_CELL_KM2),
                         "ncr_demand_2030_km2": {s: PROJ_DRIVER_SUMMARY[s][2030]["ncr"]["demand_km2"]
                                                 for s in _S11_SCEN}}
        for _s11_s in _S11_SCEN:
            for _d, _idx in _S11_DIST_CELLS.items():
                PROJ_DRIVER_SUMMARY[_s11_s][2030][_d]["ghsl_2030_km2"] = float(_s11_gc[_idx].sum() * _S11_CELL_KM2)
            PROJ_DRIVER_SUMMARY[_s11_s][2030]["ncr"]["ghsl_2030_km2"] = _s11_ghsl_cmp["ncr_ghsl_2030_km2"]
    for _s11_s in _S11_SCEN:
        PROJ_DRIVER_SUMMARY[_s11_s]["baseline"] = {
            **{d: {"name": _S11_DIST_NAME.get(d, str(d)),
                   "built_km2": float(_s11_state_now["u"][idx].sum() * _S11_CELL_KM2),
                   "built_frac": float(_s11_state_now["u"][idx].mean()),
                   "pop": float(_s11_state_now["pop"][idx].sum() * _S11_CELL_KM2)}
               for d, idx in _S11_DIST_CELLS.items()},
            "ncr": {"name": "Delhi NCR", "year": _S11_NOW,
                    "built_km2": float(_s11_state_now["u"].sum() * _S11_CELL_KM2),
                    "built_frac": float(_s11_state_now["u"].mean()),
                    "pop": float(_s11_state_now["pop"].sum() * _S11_CELL_KM2)}}

    LAND_MODEL_METRICS = {
        "available": True, "fom": None, "fom_null": None, "quantity_disagreement": None,
        "allocation_disagreement": None, "rmse_du": None, "rmse_du_null": None, "corr_du": None,
        "corr_du_null": None, "checks": [], "verdict": None, "hindcast_truth": None, **_S11_HIND,
        "suitability": _S11_SUIT_INFO, "derived_predictors": _S11_H_REPORT, "derived_frames": _S11_H_FRAMES,
        "era5_climatology_frames": _s11_era5_frames,
        "allocation": {"rule": "du_i = min(cap_i, k_d * S_i * N_i * eps_i), k_d by bisection; cap = 1 - u - forest "
                               "- water (forest and water protected); donors cropland, barren, other vegetation in "
                               "proportion to their current shares", "eps": f"LogNormal(0, {_S11_EPS_SIGMA})",
                       "neighbourhood": "3x3 mean of u", "n_realisations": int(CFG.N_LAND_REAL)},
        "projected_donor_shares": {s: i["donor_shares"] for s, i in _S11_LAND_INFO.items()},
        "shortfall_km2": {s: i["shortfall_km2"] for s, i in _S11_LAND_INFO.items()},
        "medoid_realisation": {s: i["medoid"] for s, i in _S11_LAND_INFO.items()},
        "ghsl_2030_comparison": _s11_ghsl_cmp, "seconds": round(_s11_time.perf_counter() - _S11_T0, 1)}
    CMIP6_DELTAS["l_base"] = _S11_CTX["l_base"]
    free_memory()

## 11.9 Planted vs recovered (synthetic) and the scenario-ordering check

In [ ]:
def _s11_report():
    rows = []
    last = _S11_PROJ[-1]
    for m in _S11_MODES:
        vals = {s: PROJ_SERIES[m][s]["ncr"][-1]["p50"] for s in _S11_SCEN if s in PROJ_SERIES.get(m, {})}
        ordered = all(vals[a] < vals[b] for a, b in zip(_S11_SCEN, _S11_SCEN[1:]) if a in vals and b in vals)
        rows.append({"check": f"{m}: NCR p50 {last} ordered {' < '.join(_S11_SCEN)}",
                     "value": ", ".join(f"{s} {v:.2f}" for s, v in vals.items()), "ok": ordered})
        if not ordered:
            log(f"11: {m}: NCR p50 {last} is not ordered by scenario forcing: {vals}", "WARNING")
    # land-model plausibility vs the proportional-growth null, one row per metric (D18)
    for c in LAND_MODEL_METRICS.get("checks") or []:
        v, n0 = c.get("value"), c.get("null")
        rows.append({"check": f"land hindcast {c['check']} (proportional growth)",
                     "value": (f"{v:.4f} vs {n0:.4f}" if v is not None and n0 is not None else "n/a"),
                     "ok": c.get("ok")})
    if LAND_MODEL_METRICS.get("verdict") is not None:
        rows.append({"check": "land hindcast verdict", "value": f"{LAND_MODEL_METRICS['verdict']} (truth: "
                                                                f"{LAND_MODEL_METRICS.get('hindcast_truth')})",
                     "ok": LAND_MODEL_METRICS["verdict"] == "beats_null"})
    truth = SYNTHETIC_TRUTH if DATA_MODE == "synthetic" and isinstance(SYNTHETIC_TRUTH, dict) else None
    if truth:
        planted = ((truth.get("future") or {}).get("cmip6_delta_2050") or {})
        for s, by_var in planted.items():
            rec = CMIP6_DELTAS["scenarios"].get(s)
            if rec is None:
                continue
            for var, st in by_var.items():
                got = rec["variables"][var]["summary"][last]["mean"]
                # the planted summary is rounded to 3 dp (section 02), so agreement means |diff| <= 0.0005 K
                rows.append({"check": f"CMIP6 {s} {var} delta {last} (planted ensemble mean)",
                             "value": f"{got:+.3f} vs planted {st['mean']:+.3f}",
                             "ok": bool(abs(got - float(st["mean"])) <= 5e-4 + 1e-9)})
        for m in _S11_MODES:
            b = (truth.get("beta") or {}).get(m, {}).get("value")
            if b is not None:
                rows.append({"check": f"beta {m} used (planted {b})",
                             "value": f"{CLIMATE_SENSITIVITY[m]['beta']:.3f}", "ok": None})
    print("Section 11 checks (planted vs recovered / plausibility):")
    print(pd.DataFrame(rows).to_string(index=False))
    return rows


if _S11_ACTIVE:
    _S11_REPORT = _s11_report()

## 11.10 Figures: land hindcast, NCR fan charts, uncertainty budget

In [ ]:
def _s11_save_fig(fig, name):
    fig.savefig(_S11_FIG_DIR / name, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def _s11_fig_hindcast():
    if not _S11_HIND_MAPS:
        return
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    titles = (f"observed du {LAND_MODEL_METRICS['period'][0]}-{LAND_MODEL_METRICS['period'][1]}",
              f"CA realisation 0 (FoM {LAND_MODEL_METRICS['fom']:.2f})",
              f"null: du ~ u (FoM {LAND_MODEL_METRICS['fom_null']:.2f})")
    vmax = max(0.1, float(np.nanpercentile(_S11_HIND_MAPS["du_obs"], 99.5)))
    for ax, key, title in zip(axes, ("du_obs", "du_sim", "du_null"), titles):
        im = ax.imshow(_s11_grid(_S11_HIND_MAPS[key]), cmap="magma", vmin=0, vmax=vmax)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.colorbar(im, ax=axes, shrink=0.8, label="built-up fraction change")
    _s11_save_fig(fig, "11_land_hindcast.png")


def _s11_fig_fans():
    colors = {"ssp126": "#22c55e", "ssp245": "#f59e0b", "ssp585": "#ef4444"}
    fig, axes = plt.subplots(1, len(_S11_MODES), figsize=(7 * len(_S11_MODES), 4.5), squeeze=False)
    for ax, m in zip(axes[0], _S11_MODES):
        obs_y = sorted(LEVELS[m])
        ax.plot(obs_y, [LEVELS[m][y]["level"] for y in obs_y], "ko-", ms=4, label="observed level")
        lb = _S11_CTX["l_base"][m]["l_base"]
        ax.axhline(lb, color="0.5", lw=0.8, ls=":")
        for s in _S11_SCEN:
            rows = PROJ_SERIES[m][s]["ncr"]
            yrs = [_S11_NOW] + [r["year"] for r in rows]
            get = {k: [lb] + [r[k] for r in rows] for k in _S11_QK}
            c = colors.get(s, None)
            ax.fill_between(yrs, get["p05"], get["p95"], color=c, alpha=0.15)
            ax.fill_between(yrs, get["p25"], get["p75"], color=c, alpha=0.3)
            ax.plot(yrs, get["p50"], color=c, lw=2, label=s)
        ax.set_title(f"{m}: NCR mean LST ({_S11_NOTE})")
        ax.set_ylabel("°C")
        ax.legend(fontsize=8)
    _s11_save_fig(fig, "11_projection_fans.png")


def _s11_fig_budget():
    last = _S11_PROJ[-1]
    keys = ["gcm", "sensitivity", "land", "model"]
    colors = ["#60a5fa", "#f472b6", "#a3e635", "#94a3b8"]
    labels = [f"{m}\n{s}" for m in _S11_MODES for s in _S11_SCEN]
    vals = np.array([[PROJ_UNCERTAINTY[m][s][last][k] for k in keys] for m in _S11_MODES for s in _S11_SCEN])
    fig, ax = plt.subplots(figsize=(1.4 * len(labels) + 2, 4))
    bottom = np.zeros(len(labels))
    for i, k in enumerate(keys):
        ax.bar(labels, vals[:, i], bottom=bottom, color=colors[i], label=k)
        bottom += vals[:, i]
    ax.set_ylabel(f"share of variance of the NCR mean ({last})")
    ax.legend(ncol=4, fontsize=8)
    _s11_save_fig(fig, "11_uncertainty_budget.png")


if _S11_ACTIVE:
    for _s11_fn in (_s11_fig_hindcast, _s11_fig_fans, _s11_fig_budget):
        try:
            _s11_fn()
        except Exception as _s11_exc:  # noqa: BLE001 - a broken figure must never lose the computed results
            log(f"11: figure {_s11_fn.__name__} failed: {type(_s11_exc).__name__}: {_s11_exc}", "WARNING")
            plt.close("all")

## 11.11 Tables and contract check (SPEC §3.8)

In [ ]:
def _s11_save_tables():
    rows = []
    for m, by_s in PROJ_SERIES.items():
        for s, by_scope in by_s.items():
            for sid, recs in by_scope.items():
                for r in recs:
                    rows.append({"mode": m, "scenario": s, "scope": sid, **r})
    if not rows:
        return
    out = Path(CFG.PARQUET_DIR)
    out.mkdir(parents=True, exist_ok=True)
    try:
        pd.DataFrame(rows).to_parquet(out / "projection_series.parquet", index=False)
    except Exception as exc:  # noqa: BLE001 - parquet engine missing -> CSV
        log(f"11: parquet unavailable ({exc!r}); writing CSV", "WARNING")
        pd.DataFrame(rows).to_csv(out / "projection_series.csv", index=False)


def _s11_contract_check():
    n_y = len(_S11_PROJ)
    for m in _S11_MODES:
        assert set(PROJ_QUANTILES[m]) == set(_S11_SCEN), (m, sorted(PROJ_QUANTILES[m]))
        for s in _S11_SCEN:
            q = PROJ_QUANTILES[m][s]
            assert list(q) == _S11_QK
            for k in _S11_QK:
                assert q[k].shape == (n_y, _S11_NC) and q[k].dtype == np.float32, (m, s, k, q[k].shape)
                assert np.isfinite(q[k]).all(), f"{m} {s} {k}: non-finite projected quantiles"
            for a, b in zip(_S11_QK, _S11_QK[1:]):
                assert np.all(q[a] <= q[b] + 1e-5), f"{m} {s}: quantiles not monotone ({a} > {b})"
            fl, zn = PROJ_FLAGS[m][s], PROJ_ZONES[m][s]
            assert fl.shape == (n_y, _S11_NC) and fl.dtype == np.uint8 and int(fl.max()) <= 7
            assert zn.shape == (n_y, _S11_NC) and zn.dtype == np.uint8
            assert set(np.unique(zn).tolist()) <= {0, 1, 2, 3, _S11_ZONE_NODATA}
            for sid, recs in PROJ_SERIES[m][s].items():
                assert [r["year"] for r in recs] == _S11_PROJ, (m, s, sid)
                for r in recs:
                    if r["p50"] is not None:
                        assert r["p05"] <= r["p25"] <= r["p50"] <= r["p75"] <= r["p95"], (m, s, sid, r)
                assert set(PROJ_THRESHOLD_YEARS[m][s][sid]) == {"+1", "+2", "+3"}
            for y in _S11_PROJ:
                dec, bud = PROJ_DECOMPOSITION[m][s][y], PROJ_UNCERTAINTY[m][s][y]
                assert {"climate", "land", "total", "baseline_c"} <= set(dec), (m, s, y)
                parts = [bud[k] for k in ("gcm", "sensitivity", "land", "model")]
                assert abs(sum(parts) - 1.0) < 1e-6 or sum(parts) == 0.0, (m, s, y, parts)
    # Decomposition identity (D14): every draw's scope mean = baseline_c + climate + land + error, so the median's
    # distance from the scope's own baseline must be close to the sum of the parts for EVERY scope.
    gaps = []
    for m in _S11_MODES:
        for s in _S11_SCEN:
            for y in _S11_PROJ:
                for sid, d in (PROJ_DECOMPOSITION[m][s][y].get("by_scope") or {}).items():
                    assert "baseline_c" in d, (m, s, y, sid)
                    gap = abs(d["total"] - d["climate"] - d["land"] - d["model_error_mean"])
                    gaps.append((gap, m, s, y, sid))
    gaps.sort(reverse=True)
    bad = [g for g in gaps if not g[0] < _S11_DECOMP_TOL_C]
    if bad:
        msg = (f"11: decomposition total deviates from climate + land + model error by >= {_S11_DECOMP_TOL_C} °C "
               f"for {len(bad)} scope-frames; worst: "
               + "; ".join(f"{m} {s} {y} {sid} {g:.3f}" for g, m, s, y, sid in bad[:8]))
        if DATA_MODE == "synthetic":
            raise AssertionError(msg)
        log(msg, "WARNING")
    elif gaps:
        log(f"11: decomposition identity OK: max |total - climate - land - model error| {gaps[0][0]:.3f} °C "
            f"({gaps[0][1]} {gaps[0][2]} {gaps[0][3]} {gaps[0][4]})")
    for s in _S11_SCEN:
        assert CMIP6_DELTAS["scenarios"][s]["method"] in ("gcm", "pattern-scaled")
        assert set(PROJ_DRIVER_SUMMARY[s]) >= set(_S11_PROJ)
    assert "fom" in LAND_MODEL_METRICS and "fom_null" in LAND_MODEL_METRICS
    # Section 10 fixed the per-mode raster/colour domain from the observed fields padded by 3 °C, precisely so that
    # projected fields never clip; verify that promise on the projected medians (p0.5-p99.5, as for the domain).
    domains = (RASTER_INDEX or {}).get("domains") or {}
    for m in _S11_MODES:
        if m not in domains:
            continue
        vals = np.concatenate([PROJ_QUANTILES[m][s]["p50"].ravel() for s in _S11_SCEN]).astype(np.float64)
        lo, hi = (float(v) for v in np.percentile(vals, [0.5, 99.5]))
        d_lo, d_hi = (float(v) for v in domains[m])
        inside = d_lo <= lo and hi <= d_hi
        log(f"11: {m}: projected p50 p0.5-p99.5 [{lo:.2f}, {hi:.2f}] °C vs raster domain [{d_lo:g}, {d_hi:g}] "
            + ("(inside)" if inside else "- OUTSIDE: projected rasters would clip"), "INFO" if inside else "WARNING")
    log("11: contract check OK (PROJ_QUANTILES, PROJ_FLAGS, PROJ_ZONES, PROJ_SERIES, decomposition, budget, "
        "thresholds, CMIP6_DELTAS, LAND_MODEL_METRICS, PROJ_DRIVER_SUMMARY)")


if _S11_ACTIVE:
    _s11_save_tables()
    _s11_contract_check()
    del _S11_CTX["era5"]
    free_memory()
log("Section 11 complete: LAND_MODEL_METRICS, PROJ_QUANTILES, PROJ_FLAGS, PROJ_ZONES, PROJ_SERIES, "
    "PROJ_DECOMPOSITION, PROJ_UNCERTAINTY, PROJ_THRESHOLD_YEARS, CMIP6_DELTAS, PROJ_DRIVER_SUMMARY, PROJ_META ready"
    + ("" if _S11_ACTIVE else " (empty: projections skipped)"))

# 12. Export: the v2 web bundle, Parquet / GeoJSON / NPZ / native models, read-back validation and the zip

This section turns the in-memory products of sections 06–11 into files. Nothing is re-estimated here: every number
written is a global of an earlier section (SPEC §3), only **re-shaped, quantised and rounded** to the data contract of
SPEC §5. The section therefore has two jobs: (1) write exactly the files, names, keys, dtypes and quantisation of
§5, and (2) prove by reading the files back from disk that they decode to what the pipeline computed.

```
outputs/
├── web/                      ← copy the CONTENTS to dashboard/public/data/ (real run) or /demo/ (synthetic)
│   ├── manifest.json                     schema 2.0 (§5.3): modes, groups, features, zones, timeline, files
│   ├── cells.lstb.gz                     canonical cell order: cell_id, row, col, lon, lat, district
│   ├── timeline_series.json.gz           §5.4: per scope × mode frame stats, annual series, SSP fans, trends
│   ├── cube_obs_<mode>.lstb.gz           observed/reconstructed/modelled fields [F_obs, n] + AOA + p05/p95 offsets
│   ├── cube_<ssp>_<mode>.lstb.gz         projected p50 + quantile offsets + flags [F_proj, n]
│   ├── cube_zones_<mode>.lstb.gz         zone ids, observed and per scenario
│   ├── xai_<year>_features.lstb.gz       the 33 base features of a frame
│   ├── xai_<year>_<mode>.lstb.gz         18 group SHAP + pred / oof / obs / resid (absolute °C)
│   ├── raster_index.json, raster/*.png   written by section 10 (EPSG:4326 PNGs), referenced here
│   ├── zones.json, shap_global.json, dependence.json.gz, interactions.json, metrics.json, provenance.json
│   ├── model_web_day.json, model_web_night.json, scenario_coupling.json, districts.geojson
├── parquet/   predictions_<mode>, projections_<mode>, levels, series_frames, annual_series, importance (+ 06/07/11)
├── geo/       ncr_districts.geojson, zones_<mode>.geojson (cell squares with per-frame zone / LST properties)
├── npz/       fine_fields_<mode>.npz (250 m fields + flags, UTM grid metadata)
├── models/    xgb_final_<mode>.json (native XGBoost, written by section 07; re-saved if missing)
└── web_bundle.zip
```

## 12.1 Binary container and quantisation (SPEC §5.1)
Per-cell arrays go into **LSTB** containers (`lstb_write`, section 00): a small JSON header followed by 8-byte aligned
little-endian typed columns, gzip level 9 with a fixed header (re-runs are byte-identical). A value is stored as
`raw = round((v − offset) / scale)` and decoded as `raw · scale + offset`, so the rounding error is ≤ scale / 2:

| column | dtype | scale | worst-case error | why this resolution is enough |
|---|---|---|---|---|
| LST (°C, absolute) | int16 | 0.01 °C | 0.005 °C | 10× finer than the Landsat ST accuracy (~1 K) and the 2-dp JSON rounding |
| uncertainty offsets | uint8 | 0.05 °C | 0.025 °C | intervals are ≥ 0.3 °C wide; uint8 keeps the cubes small (max 12.75 °C) |
| group SHAP | int16 | 0.001 °C | 0.0005 °C | 18 groups sum to ≤ 0.009 °C error, inside the 0.05 °C waterfall tolerance |
| base features | int16 | per column (p0–p100 + 2 % headroom over 65 534 steps) | span / 131 068 | far below the measurement noise of every predictor |

NaN is stored as the nodata code (−32768 for int16, 255 for zone ids); offsets of cells without an interval are 0.
Uncertainty offsets are taken relative to the **quantised** central value, so decoded quantiles stay ordered.

## 12.2 Fixed colour domains (SPEC §5.3)
The dashboard colours every frame of a mode with one fixed domain, so a colour means the same temperature in 1990
and in 2050. Section 10 already fixed it for the PNG rasters (p0.5–p99.5 of all observed fine fields, padded by
3 °C so that projections never clip; section 11 verified that). The manifest uses **that** domain for
`timeline.domains.<mode>.abs` (the PNGs are encoded with it, SPEC §4.8) and widens it outward only if the 1 km
p0.5–p99.5 of the observed fields and projected medians (the SPEC §5.3 definition) were to fall outside — a widening
is logged. Anomaly, SUHI, diurnal-range (DTR) and spread domains are computed here from the same arrays.

## 12.3 What "validated" means
After writing, every file is **re-read from disk** (gzip detected by magic bytes, JSON parsed strictly with NaN /
Infinity rejected) and checked: LSTB headers (dtypes, scales, nodata codes, shapes = frames × n, 8-byte alignment),
the same `n` everywhere, frame and tier lists equal to the manifest, the SHAP waterfall invariant
`|pred − (base_value_c + Σ group SHAP)| ≤ 0.05 °C` on **every** non-NaN row of every XAI file, decoded values equal to
the in-memory arrays within the quantisation step, zone ids in {0..3, 255}, and every referenced file present.
**Cross-file alignment** catches row- or frame-order bugs that every single file would hide: the XAI `obs` of an
observed / observed-sparse / reconstructed frame equals the cube field (+ the frame's composition correction of
section 09) within 0.011 °C with the same NaN pattern, `corr(pred, obs) ≥ 0.5` (≥ 0.25 for a reconstructed frame,
which the model never trained on; a value under 0.5 there is logged), and the nearest zone centroid of
every complete SHAP row equals the zone cube on ≥ 99 % of rows. Zone entries carry `display_name` / `label_caveat`
(SPEC §1.5) and series frame rows an optional `coverage` in [0, 1]. Finally the cache is packed (`pack_cache`,
Kaggle's output-file cap) and a stale `RUN_STOPPED.json` of an earlier clean stop is removed at the start.
When the repository's standalone checker (`notebook/validate_bundle.py`, numpy only) is reachable it is run as
well and must report VALID; on Kaggle, attach it as a dataset or run it locally on the downloaded bundle.

In [ ]:
_lst_enter_section('12')  # section marker (generated by build_notebook.py)

In [ ]:
import datetime as _s12_dt
import gzip as _s12_gzip
import json as _s12_json
import math as _s12_math
import os as _s12_os
import subprocess as _s12_subprocess
import sys as _s12_sys
import time as _s12_time
import zipfile as _s12_zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 12-v2.1: zone display_name / label_caveat, frame coverage, decomposition baseline_c, night-calibration diag,
# cross-file read-back checks, cache packing at the end (D3, D11, D14, D16, D19).
_S12_VERSION = "12-v2.1"
_S12_T0 = _s12_time.perf_counter()
_S12_OUT = Path(CFG.OUTPUT_DIR)
# A completed export supersedes an earlier clean stop in the same output directory (D3).
_s12_stopped = _S12_OUT / "RUN_STOPPED.json"
if _s12_stopped.is_file():
    try:
        _s12_stopped.unlink()
        log("12: removed a stale RUN_STOPPED.json (this export completes the run)")
    except OSError as _s12_exc:
        log(f"12: could not remove stale {_s12_stopped}: {_s12_exc}", "WARNING")
_S12_WEB = Path(CFG.WEB_DIR)
_S12_PARQUET = Path(CFG.PARQUET_DIR)
_S12_GEO = Path(CFG.GEO_DIR)
_S12_MODEL_DIR = Path(CFG.MODEL_DIR)
_S12_NPZ = Path(CFG.NPZ_DIR)
_S12_FIG_DIR = Path(CFG.FIG_DIR)
for _s12_d in (_S12_WEB, _S12_PARQUET, _S12_GEO, _S12_MODEL_DIR, _S12_NPZ, _S12_FIG_DIR):
    _s12_d.mkdir(parents=True, exist_ok=True)

_S12_NC = int(N_CELLS)
_S12_GROUPS = list(SHAP_GROUPS)
_S12_MODES = [m for m in CFG.MODES if m in FRAME_FIELDS]
if not _S12_MODES:
    raise RuntimeError("12: no mode has frame fields (section 09 did not run?) - nothing to export")
_S12_OBS = [int(y) for y in CFG.FRAMES_OBS]
_S12_NOW = int(CFG.NOW) if int(CFG.NOW) in _S12_OBS else _S12_OBS[-1]
# Projections exist only when section 11 ran (pilot profile: none). A scenario is exported only when every mode has it.
_S12_PROJ = [int(y) for y in CFG.FRAMES_PROJ] if globals().get("PROJ_QUANTILES") else []
_S12_SCEN = [s for s in CFG.SCENARIOS if _S12_PROJ and all(s in PROJ_QUANTILES.get(m, {}) for m in _S12_MODES)]
if _S12_PROJ and not _S12_SCEN:
    log("12: projection frames configured but no scenario has quantiles for every mode; exporting observed frames "
        "only", "WARNING")
    _S12_PROJ = []
_S12_ZONE_META = [
    {"id": 0, "name": "Ecological Cool Base", "color": "#34d399"},
    {"id": 1, "name": "Riparian Buffer", "color": "#22d3ee"},
    {"id": 2, "name": "Transition", "color": "#a78bfa"},
    {"id": 3, "name": "Heat Extreme Core", "color": "#fb7185"},
]
_S12_ZONE_NODATA = 255
_S12_SCEN_LABEL = {"ssp126": "SSP1-2.6", "ssp245": "SSP2-4.5", "ssp585": "SSP5-8.5"}
_S12_MODE_INFO = {
    "day": {"label": "Day · ~10:30 IST", "target": "Landsat C2 L2 ST (harmonised, 10:45 IST reference)",
            "ramp": "solar", "reference_overpass_ist": "10:45", "native_m": 30},
    "night": {"label": "Night · ~01:30 IST",
              "target": "MODIS Aqua night (Aqua-equivalent; VIIRS/Terra calibrated)", "ramp": "nocturne",
              "reference_overpass_ist": "01:30", "native_m": 1000},
}
_S12_TIERS = ["observed", "observed-sparse", "reconstructed", "modelled", "projected"]
_S12_PROJ_NOTE = "scenario projection, not a forecast"
# Quantisation constants of SPEC §5.2 (shared by the writer and the read-back check).
_S12_LST_SCALE = 0.01
_S12_OFF_SCALE = 0.05
_S12_SHAP_SCALE = 0.001
_S12_I16_NODATA = -32768
_S12_WATERFALL_TOL_C = 0.05
# Floor of the cross-file check corr(pred, obs): rows in the wrong order give ~0. Frames the model trained on agree at
# 0.95-0.99 on real data. A reconstructed frame (2000 night: sparse Terra nights, never trained on) agrees at only
# ~0.5 (0.497-0.524 in real pilot runs), so it gets a lower floor plus a logged note instead of a failed run.
_S12_CORR_MIN = {"reconstructed": 0.25}
_S12_CELL_KM2 = float(GRID.cell_area_km2)
_S12_SIZES = {}            # published path -> bytes on disk (filled by the writers)

log(f"12: exporting modes {_S12_MODES}, {len(_S12_OBS)} observed frames, "
    f"{len(_S12_PROJ)} projected frames x {_S12_SCEN or 'no scenarios'}, {_S12_NC:,} cells -> {_S12_WEB}")

## 12.4 Rounding, JSON and LSTB writers

In [ ]:
def _s12_num(v, dp=None):
    """Finite float (optionally rounded) or None; ints stay ints, bools stay bools."""
    if v is None or isinstance(v, (bool, np.bool_)):
        return None if v is None else bool(v)
    if isinstance(v, (int, np.integer)):
        return int(v)
    try:
        f = float(v)
    except (TypeError, ValueError):
        return None
    if not _s12_math.isfinite(f):
        return None
    return round(f, dp) if dp is not None else f


def _s12_sig(v, sig=4):
    """Round to ``sig`` significant digits so that the JSON text is the shortest decimal (None for non-finite)."""
    f = _s12_num(v)
    if f is None or isinstance(f, bool):
        return f
    if f == 0:
        return 0.0
    k = sig - 1 - int(_s12_math.floor(_s12_math.log10(abs(f))))
    return round(f, k) if k >= 0 else float(round(f / 10.0 ** (-k)) * 10.0 ** (-k))


def _s12_sig_arr(arr, sig=4):
    """Vectorised significant-digit rounding (NaN stays NaN) - used for feature values and bin centres."""
    a = np.asarray(arr, dtype=np.float64)
    out = np.full(a.shape, np.nan)
    fin = np.isfinite(a)
    zero = fin & (a == 0)
    out[zero] = 0.0
    ok = fin & ~zero
    if ok.any():
        k = (sig - 1 - np.floor(np.log10(np.abs(a[ok])))).astype(np.int64)
        v = a[ok]
        res = np.empty_like(v)
        pos = k >= 0
        scale_pos = 10.0 ** k[pos]
        res[pos] = np.round(v[pos] * scale_pos) / scale_pos
        scale_neg = 10.0 ** (-k[~pos])
        res[~pos] = np.round(v[~pos] / scale_neg) * scale_neg
        out[ok] = res
    return out


def _s12_sig_increasing(arr, sig=4, max_sig=12):
    """Significant-digit rounding that keeps a strictly increasing sequence strictly increasing.

    Dependence bin centres of a feature with a small range relative to its magnitude (smooth ERA5 fields, e.g. wind
    2.3521 / 2.3524 m/s) collapse to duplicates at 4 significant digits, which breaks the bundle invariant
    "bin_centers increasing" (validate_bundle) and the dashboard's interpolation. Digits are added until the finite
    values stay strictly increasing (or ``max_sig`` is reached)."""
    a = np.asarray(arr, dtype=np.float64)
    fin = np.isfinite(a)
    increasing = bool(np.all(np.diff(a[fin]) > 0))
    for digits in range(int(sig), int(max_sig) + 1):
        out = _s12_sig_arr(a, digits)
        if not increasing or np.all(np.diff(out[fin]) > 0):
            return out
    return out


def _s12_tree(obj, dp=4):
    """Recursively convert numpy / pandas containers to JSON-able Python and round every float to ``dp`` decimals.

    dict keys become strings; DataFrames become record lists; NaN / Inf become None (strict JSON, SPEC §5)."""
    if isinstance(obj, dict):
        return {str(k): _s12_tree(v, dp) for k, v in obj.items()}
    if isinstance(obj, pd.DataFrame):
        return [_s12_tree(r, dp) for r in obj.to_dict(orient="records")]
    if isinstance(obj, (pd.Series, pd.Index)):
        return [_s12_tree(v, dp) for v in obj.tolist()]
    if isinstance(obj, np.ndarray):
        if obj.dtype.kind in "biu":
            return obj.tolist()
        return [_s12_tree(v, dp) for v in obj.tolist()]
    if isinstance(obj, (list, tuple, set, frozenset)):
        return [_s12_tree(v, dp) for v in (sorted(obj) if isinstance(obj, (set, frozenset)) else obj)]
    if isinstance(obj, (bool, np.bool_)):
        return bool(obj)
    if isinstance(obj, (int, np.integer)):
        return int(obj)
    if isinstance(obj, (float, np.floating)):
        return _s12_num(obj, dp)
    if obj is None or isinstance(obj, str):
        return obj
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (pd.Timestamp, _s12_dt.datetime, _s12_dt.date)):
        return obj.isoformat()
    if hasattr(obj, "item") and callable(obj.item):
        return _s12_tree(obj.item(), dp)
    return str(obj)


def _s12_write_json(obj, name):
    """Strict compact UTF-8 JSON (gzip level 9 when ``name`` ends in .gz) under the web dir; records the size."""
    path = save_json(obj, _S12_WEB / name)
    _S12_SIZES[name] = path.stat().st_size
    return path


def _s12_write_lstb(name, columns, meta, kind):
    """LSTB container under the web dir (n = N_CELLS for every bundle file); records the size."""
    path = lstb_write(_S12_WEB / name, columns, meta=meta, kind=kind, n=_S12_NC)
    _S12_SIZES[name] = path.stat().st_size
    return path


def _s12_quantised(values, scale, offset=0.0, lo=None, hi=None):
    """What a decoder will see for ``values`` stored with (scale, offset): round-half-even like numpy/lstb_write."""
    q = np.rint((np.asarray(values, np.float64) - offset) / scale)
    if lo is not None:
        q = np.clip(q, lo, hi)
    return q * scale + offset


def _s12_offsets(centre_q, other, sign):
    """Non-negative uint8 offsets (scale 0.05 °C) of a quantile from the quantised centre.

    ``sign`` = −1 for quantiles below the centre (p05, p25), +1 above. NaN (no interval) → 0.
    Returns (array, n_clipped)."""
    d = sign * (np.asarray(other, np.float64) - np.asarray(centre_q, np.float64))
    d = np.where(np.isfinite(d), np.maximum(d, 0.0), 0.0)
    clipped = int(np.count_nonzero(np.rint(d / _S12_OFF_SCALE) > 255))
    return np.minimum(d, 255 * _S12_OFF_SCALE), clipped


def _s12_hhmm(hours):
    """Decimal IST hours → "HH:MM" (None when unknown)."""
    h = _s12_num(hours)
    if h is None:
        return None
    minutes = int(round(h * 60.0)) % (24 * 60)
    return f"{minutes // 60:02d}:{minutes % 60:02d}"


def _s12_cells(grid2d):
    """(H, W) grid → canonical-order cell vector (float64)."""
    return np.asarray(grid2d, np.float64)[CELL_ROWS, CELL_COLS]


_S12_DISTRICT_OF_CELL = np.asarray(DISTRICT_IDX)[CELL_ROWS, CELL_COLS].astype(np.int16)

## 12.5 Levels, frame catalogue and colour domains

In [ ]:
def _s12_level(mode, year):
    """(level °C, [ci_lo, ci_hi]) of an observed frame from section 09's LEVELS (None when absent)."""
    rec = (LEVELS.get(mode) or {}).get(int(year))
    if not rec:
        return None, None
    lev = _s12_num(rec.get("level"))
    ci = rec.get("ci")
    ci = [_s12_num(ci[0]), _s12_num(ci[1])] if ci is not None and len(ci) == 2 else None
    return lev, ci


def _s12_tier(mode, year):
    tier = (FRAME_TIERS.get(mode) or {}).get(int(year))
    if tier is None:
        tier = "projected" if int(year) in _S12_PROJ else "modelled"
    if tier not in _S12_TIERS:
        raise ValueError(f"12: unknown evidence tier {tier!r} for {mode} {year} (vocabulary {_S12_TIERS})")
    return tier


def _s12_frame_field(mode, year):
    """Absolute °C field of an observed frame (canonical order, float64, NaN = no value)."""
    v = (FRAME_FIELDS.get(mode) or {}).get(int(year))
    return np.full(_S12_NC, np.nan) if v is None else np.asarray(v, np.float64)


def _s12_ncr_proj(mode, scen, year):
    """NCR scope quantiles {p05..p95} of a projected frame (from section 11's per-draw scope means)."""
    for row in PROJ_SERIES[mode][scen].get("ncr", []):
        if int(row["year"]) == int(year):
            return row
    return None


def _s12_round_out(lo, hi, step=1.0):
    return [float(_s12_math.floor(lo / step) * step), float(_s12_math.ceil(hi / step) * step)]


def _s12_domains():
    """timeline.domains (SPEC §5.3): abs per mode (fixed; = the raster domain of section 10, widened if needed),
    symmetric anomaly and SUHI domains, the diurnal range domain and the projection spread domain."""
    out, notes = {}, []
    raster_dom = (globals().get("RASTER_INDEX") or {}).get("domains") or {}
    spreads = []
    for m in _S12_MODES:
        abs_vals, anom_vals, suhi_vals = [], [], []
        for y in _S12_OBS:
            f = _s12_frame_field(m, y)
            ok = np.isfinite(f)
            if not ok.any():
                continue
            abs_vals.append(f[ok])
            lev, _ = _s12_level(m, y)
            if lev is not None:
                anom_vals.append(np.abs(f[ok] - lev))
            fb = _s12_cells(FEATURE_GRIDS[y]["frac_built"])
            fw = _s12_cells(FEATURE_GRIDS[y]["frac_water"])
            rural = ok & (fb <= 0.05) & (fw < 0.2)
            if rural.sum() >= 3:
                suhi_vals.append(np.abs(f[ok] - f[rural].mean()))
            unc = (globals().get("FRAME_UNC") or {}).get(m, {}).get(y)
            if unc is not None:
                sp = np.asarray(unc["p95"], np.float64) - np.asarray(unc["p05"], np.float64)
                spreads.append(sp[np.isfinite(sp)])
        for s in _S12_SCEN:
            q = PROJ_QUANTILES[m][s]
            p50 = np.asarray(q["p50"], np.float64)
            abs_vals.append(p50[np.isfinite(p50)])
            sp = np.asarray(q["p95"], np.float64) - np.asarray(q["p05"], np.float64)
            spreads.append(sp[np.isfinite(sp)])
        allv = np.concatenate(abs_vals) if abs_vals else np.zeros(0)
        if allv.size == 0:
            raise RuntimeError(f"12: mode {m} has no finite LST values to define its colour domain")
        spec_dom = _s12_round_out(*np.percentile(allv, [0.5, 99.5]))
        if m in raster_dom:
            dom = [float(raster_dom[m][0]), float(raster_dom[m][1])]
            if spec_dom[0] < dom[0] or spec_dom[1] > dom[1]:
                widened = [min(dom[0], spec_dom[0]), max(dom[1], spec_dom[1])]
                notes.append(f"{m}: raster domain {dom} widened to {widened} to cover 1 km p0.5-p99.5 {spec_dom}")
                log(f"12: {notes[-1]} (rasters keep their own lo/hi in raster_index.json)", "WARNING")
                dom = widened
        else:
            dom = spec_dom
        a = float(np.ceil(np.percentile(np.concatenate(anom_vals), 99.5))) if anom_vals else 5.0
        b = float(np.ceil(np.percentile(np.concatenate(suhi_vals), 99.5))) if suhi_vals else 5.0
        out[m] = {"abs": dom, "anom": [-max(a, 1.0), max(a, 1.0)], "suhi": [-max(b, 1.0), max(b, 1.0)],
                  "abs_spec_p005_p995": spec_dom}
    if "day" in _S12_MODES and "night" in _S12_MODES:
        dtr = []
        for y in _S12_OBS:
            d = _s12_frame_field("day", y) - _s12_frame_field("night", y)
            dtr.append(d[np.isfinite(d)])
        for s in _S12_SCEN:
            d = (np.asarray(PROJ_QUANTILES["day"][s]["p50"], np.float64)
                 - np.asarray(PROJ_QUANTILES["night"][s]["p50"], np.float64))
            dtr.append(d[np.isfinite(d)])
        dv = np.concatenate(dtr) if dtr else np.zeros(0)
        out["dtr"] = {"abs": _s12_round_out(*np.percentile(dv, [0.5, 99.5])) if dv.size else [0.0, 25.0]}
    sv = np.concatenate(spreads) if spreads else np.zeros(0)
    s_hi = float(np.ceil(np.percentile(sv, 99.5) * 2.0) / 2.0) if sv.size else 2.0
    out["spread"] = [0.0, max(s_hi, 0.5)]
    return out, notes


with timer("12_domains"):
    _S12_DOMAINS, _S12_DOMAIN_NOTES = _s12_domains()
for _s12_m in _S12_MODES:
    log(f"12: {_s12_m} domains abs {_S12_DOMAINS[_s12_m]['abs']} (SPEC p0.5-p99.5 "
        f"{_S12_DOMAINS[_s12_m]['abs_spec_p005_p995']}), anom {_S12_DOMAINS[_s12_m]['anom']}, "
        f"suhi {_S12_DOMAINS[_s12_m]['suhi']}")
log(f"12: dtr {(_S12_DOMAINS.get('dtr') or {}).get('abs')}, spread {_S12_DOMAINS['spread']}")


def _s12_level_extras(mode, year):
    """Optional level diagnostics of section 09 (D8, D10) for the manifest frame entry; {} when absent."""
    lev = LEVELS.get(mode, {}).get(int(year)) or {}
    out = {}
    for k, dp in (("raw_level", 3), ("composition_correction", 3), ("fixed_mask_coverage", 4),
                  ("drift_correction_c", 3)):
        if lev.get(k) is not None:
            out[k] = _s12_num(lev.get(k), dp)
    return out


def _s12_obs_mode_entry(mode, year):
    """manifest.timeline.frames[].<mode> for an observed frame (SPEC §5.3)."""
    fm = FRAME_META[int(year)]
    tier = _s12_tier(mode, year)
    lev, ci = _s12_level(mode, year)
    field = _s12_frame_field(mode, year)
    if mode == "day":
        d = fm.get("day") or {}
        sensors = d.get("sensors") or []
        return {**_s12_level_extras(mode, year), "tier": tier, "sensor": " + ".join(sensors) if sensors else "Landsat",
                "overpass_ist": _s12_hhmm(d.get("mean_overpass_ist_h")), "scenes": int(d.get("scenes") or 0),
                "coverage": _s12_num(d.get("coverage"), 4), "native_m": 30,
                "available": bool(np.isfinite(field).any()), "level": _s12_num(lev, 2),
                "level_ci": None if ci is None else [_s12_num(ci[0], 2), _s12_num(ci[1], 2)],
                "overpass_correction": _s12_num((LEVELS.get(mode, {}).get(int(year)) or {})
                                                .get("overpass_correction"), 3),
                "heating_rate_k_per_h": _s12_num(d.get("heating_rate_k_per_h"), 3),
                "scenes_per_pathrow": {str(k): int(v) for k, v in (d.get("scenes_per_pathrow") or {}).items()},
                "path_offsets": {str(k): _s12_num(v, 4) for k, v in (d.get("path_offsets") or {}).items()}}
    n = fm.get("night") or {}
    modelled = tier == "modelled"
    return {**_s12_level_extras(mode, year), "tier": tier,
            "sensor": "model hindcast (ERA5-Land level)" if modelled else (n.get("sensor") or n.get("source")),
            "overpass_ist": "01:30" if modelled or n.get("overpass_ist_h") is None
            else _s12_hhmm(n.get("overpass_ist_h")),
            "scenes": int(n.get("composites") or 0),
            "coverage": 1.0 if modelled else _s12_num(n.get("coverage"), 4), "native_m": 1000,
            "available": bool(np.isfinite(field).any()), "level": _s12_num(lev, 2),
            "level_ci": None if ci is None else [_s12_num(ci[0], 2), _s12_num(ci[1], 2)],
            "source": n.get("source"), "method": (LEVELS.get(mode, {}).get(int(year)) or {}).get("method")}


def _s12_proj_mode_entry(mode, year):
    """manifest.timeline.frames[].<mode> for a projected frame: headline level = SSP2-4.5 NCR median."""
    by_s = {}
    for s in _S12_SCEN:
        r = _s12_ncr_proj(mode, s, year) or {}
        by_s[s] = {k: _s12_num(r.get(k), 2) for k in ("p05", "p25", "p50", "p75", "p95")}
    head = by_s.get("ssp245") or (by_s[_S12_SCEN[0]] if _S12_SCEN else {})
    p50 = PROJ_QUANTILES[mode][_S12_SCEN[0]]["p50"][_S12_PROJ.index(int(year))] if _S12_SCEN else None
    return {"tier": "projected", "sensor": "CMIP6 delta x dLST/dTair + land-change model (Monte Carlo)",
            "overpass_ist": _S12_MODE_INFO[mode]["reference_overpass_ist"], "scenes": 0,
            "coverage": _s12_num(np.isfinite(p50).mean(), 4) if p50 is not None else 0.0,
            "native_m": int(CFG.GRID_RES_M), "available": True, "level": head.get("p50"),
            "level_ci": ([head.get("p05"), head.get("p95")] if head.get("p05") is not None else None),
            "level_scenario": "ssp245" if "ssp245" in by_s else (_S12_SCEN[0] if _S12_SCEN else None),
            "by_scenario": by_s}


def _s12_timeline_frames():
    raster_frames = ((globals().get("RASTER_INDEX") or {}).get("frames")) or {}
    frames = []
    for y in _S12_OBS:
        entry = {"year": y, "window": list(CFG.window(y)), "kind": "observed"}
        for m in CFG.MODES:
            entry[m] = (_s12_obs_mode_entry(m, y) if m in _S12_MODES
                        else {"tier": "modelled", "available": False, "level": None, "level_ci": None})
        entry["xai"] = {m: bool(m in XAI_FRAMES and y in XAI_FRAMES[m]) for m in CFG.MODES}
        entry["raster"] = {m: f"{m}_{y}" in raster_frames for m in CFG.MODES}
        frames.append(entry)
    for y in _S12_PROJ:
        entry = {"year": y, "window": list(CFG.window(y)), "kind": "projected"}
        for m in CFG.MODES:
            entry[m] = (_s12_proj_mode_entry(m, y) if m in _S12_MODES
                        else {"tier": "projected", "available": False, "level": None, "level_ci": None})
        entry["xai"] = {m: False for m in CFG.MODES}
        entry["raster"] = {m: f"{m}_{y}" in raster_frames for m in CFG.MODES}
        frames.append(entry)
    return frames


_S12_TIMELINE_FRAMES = _s12_timeline_frames()

## 12.6 Per-cell binary files: cells, cubes, zones, XAI frames (SPEC §5.1–§5.2)

In [ ]:
def _s12_write_cells():
    lon = np.asarray(CELL_LON, np.float64)
    lat = np.asarray(CELL_LAT, np.float64)
    cols = [{"name": "cell_id", "array": np.asarray(CELL_IDS, np.int32), "dtype": "int32"},
            {"name": "row", "array": np.asarray(CELL_ROWS, np.int16), "dtype": "int16"},
            {"name": "col", "array": np.asarray(CELL_COLS, np.int16), "dtype": "int16"},
            {"name": "lon", "array": lon.astype(np.float32), "dtype": "float32"},
            {"name": "lat", "array": lat.astype(np.float32), "dtype": "float32"},
            {"name": "district", "array": _S12_DISTRICT_OF_CELL, "dtype": "int16"}]
    if int(np.max(CELL_ROWS)) > 32767 or int(np.max(CELL_COLS)) > 32767:
        raise ValueError("12: grid larger than int16 row/col range")
    meta = {"grid": {"crs": GRID.crs, "res": int(GRID.res), "x0": float(GRID.x0), "y0": float(GRID.y0),
                     "width": int(GRID.width), "height": int(GRID.height)},
            "order": "ROI cells sorted by cell_id = row * width + col", "cell_area_km2": _S12_CELL_KM2}
    return _s12_write_lstb("cells.lstb.gz", cols, meta, "cells")


def _s12_write_obs_cube(mode):
    """cube_obs_<mode>: lst int16 [F_obs, n] (0.01 °C), aoa_out uint8, p05_off / p95_off uint8 (0.05 °C)."""
    F = len(_S12_OBS)
    lst = np.full((F, _S12_NC), np.nan)
    aoa = np.zeros((F, _S12_NC), np.uint8)
    lo_off = np.zeros((F, _S12_NC))
    hi_off = np.zeros((F, _S12_NC))
    tiers, levels, level_ci, clipped = [], [], [], 0
    for j, y in enumerate(_S12_OBS):
        lst[j] = _s12_frame_field(mode, y)
        a = (globals().get("FRAME_AOA_OUT") or {}).get(mode, {}).get(y)
        if a is not None:
            aoa[j] = np.asarray(a, bool).astype(np.uint8)
        tier = _s12_tier(mode, y)
        tiers.append(tier)
        lev, ci = _s12_level(mode, y)
        levels.append(_s12_num(lev, 4))
        level_ci.append(None if ci is None else [_s12_num(ci[0], 4), _s12_num(ci[1], 4)])
        unc = (globals().get("FRAME_UNC") or {}).get(mode, {}).get(y)
        if unc is not None and tier != "observed":
            centre = _s12_quantised(lst[j], _S12_LST_SCALE)
            lo_off[j], c1 = _s12_offsets(centre, unc["p05"], -1)
            hi_off[j], c2 = _s12_offsets(centre, unc["p95"], +1)
            clipped += c1 + c2
    if clipped:
        log(f"12: cube_obs_{mode}: {clipped} uncertainty offsets clipped at 12.75 °C", "WARNING")
    cols = [{"name": "lst", "array": lst, "dtype": "int16", "scale": _S12_LST_SCALE, "offset": 0.0,
             "nodata": _S12_I16_NODATA},
            {"name": "aoa_out", "array": aoa, "dtype": "uint8"},
            {"name": "p05_off", "array": lo_off, "dtype": "uint8", "scale": _S12_OFF_SCALE},
            {"name": "p95_off", "array": hi_off, "dtype": "uint8", "scale": _S12_OFF_SCALE}]
    meta = {"mode": mode, "frames": list(_S12_OBS), "tiers": tiers, "levels": levels, "level_ci": level_ci,
            "units": "degC", "series": "obs",
            "uncertainty": "p05 = lst - p05_off, p95 = lst + p95_off (modelled / reconstructed frames; 0 = none)"}
    return _s12_write_lstb(f"cube_obs_{mode}.lstb.gz", cols, meta, "cube")


def _s12_write_ssp_cube(mode, scen):
    """cube_<ssp>_<mode>: p50 int16 [F_proj, n]; p05/p25/p75/p95 offsets uint8 (0.05 °C); flags uint8."""
    q = PROJ_QUANTILES[mode][scen]
    p50 = np.asarray(q["p50"], np.float64)
    centre = _s12_quantised(p50, _S12_LST_SCALE)
    cols = [{"name": "p50", "array": p50, "dtype": "int16", "scale": _S12_LST_SCALE, "offset": 0.0,
             "nodata": _S12_I16_NODATA}]
    clipped = 0
    for key, sign in (("p05", -1), ("p25", -1), ("p75", +1), ("p95", +1)):
        off, c = _s12_offsets(centre, q[key], sign)
        clipped += c
        cols.append({"name": f"{key}_off", "array": off, "dtype": "uint8", "scale": _S12_OFF_SCALE})
    if clipped:
        log(f"12: cube_{scen}_{mode}: {clipped} quantile offsets clipped at 12.75 °C", "WARNING")
    flags = np.asarray(PROJ_FLAGS[mode][scen], np.uint8)
    cols.append({"name": "flags", "array": flags, "dtype": "uint8"})
    pm = (globals().get("PROJ_META") or {}).get(mode, {}).get(scen, {})
    cd = CMIP6_DELTAS["scenarios"].get(scen, {})
    meta = {"mode": mode, "scenario": scen, "frames": list(_S12_PROJ), "method": cd.get("method", "gcm"),
            "units": "degC", "series": scen, "n_gcm": cd.get("n_gcm"), "n_mc": pm.get("n_mc"),
            "n_land_real": pm.get("n_land_real"), "l_base": _s12_num(pm.get("l_base"), 4),
            "flags": "bit0 outside AOA; bits1-2 VSUP tier of (p95 - p05) / 2 (<0.5, <1, <2, >=2 degC)",
            "note": _S12_PROJ_NOTE}
    return _s12_write_lstb(f"cube_{scen}_{mode}.lstb.gz", cols, meta, "cube")


def _s12_zone_float(z):
    """uint8 zone ids → float with NaN for nodata (lstb_write then stores the 255 nodata code)."""
    a = np.asarray(z).astype(np.float64)
    a[(a > 3) | ~np.isfinite(a)] = np.nan
    return a


def _s12_write_zone_cube(mode):
    obs = np.full((len(_S12_OBS), _S12_NC), np.nan)
    for j, y in enumerate(_S12_OBS):
        z = (ZONES_OBS.get(mode) or {}).get(y)
        if z is not None:
            obs[j] = _s12_zone_float(z)
    cols = [{"name": "obs", "array": obs, "dtype": "uint8", "nodata": _S12_ZONE_NODATA}]
    frames = {"obs": list(_S12_OBS)}
    for s in _S12_SCEN:
        cols.append({"name": s, "array": _s12_zone_float(PROJ_ZONES[mode][s]), "dtype": "uint8",
                     "nodata": _S12_ZONE_NODATA})
        frames[s] = list(_S12_PROJ)
    meta = {"mode": mode, "frames": frames, "frames_obs": list(_S12_OBS), "frames_proj": list(_S12_PROJ),
            "zones": _S12_ZONE_META, "nodata": _S12_ZONE_NODATA,
            "assignment": "nearest centroid in the 18-group SHAP space of the mode (section 08 ZONE_MODEL)"}
    return _s12_write_lstb(f"cube_zones_{mode}.lstb.gz", cols, meta, "zones")


_S12_XAI_YEARS = sorted({int(y) for m in _S12_MODES for y in (XAI_FRAMES.get(m) or [])})


def _s12_write_features(year):
    """xai_<year>_features: the 33 base features, int16 with per-column (scale, offset) from quant_params."""
    cols = []
    for f in BASE_FEATURES:
        v = _s12_cells(FEATURE_GRIDS[year][f])
        scale, offset = quant_params(v, "int16", nodata=_S12_I16_NODATA)
        cols.append({"name": f, "array": v, "dtype": "int16", "scale": scale, "offset": offset,
                     "nodata": _S12_I16_NODATA})
    meta = {"year": int(year), "features": list(BASE_FEATURES),
            "note": "focal (_m3/_m15) features are recomputed from these on the grid (FOCAL_SPEC)"}
    return _s12_write_lstb(f"xai_{year}_features.lstb.gz", cols, meta, "features")


_S12_XAI_STATS = {}


def _s12_comp_corr(mode, year):
    """Section 09's composition correction of a frame level (°C; 0 when absent or not finite, D8)."""
    v = (LEVELS.get(mode, {}).get(int(year)) or {}).get("composition_correction")
    try:
        v = float(v)
    except (TypeError, ValueError):
        return 0.0
    return v if np.isfinite(v) else 0.0


def _s12_write_xai(mode, year):
    """xai_<year>_<mode>: 18 group SHAP int16 (0.001 °C) + pred / oof / obs / resid int16 (0.01 °C, absolute).

    base_value_c = SHAP_BASE + level, so pred = base_value_c + Σ SHAP holds in absolute °C (verified on the
    quantised values before writing)."""
    lev, _ = _s12_level(mode, year)
    if lev is None:
        raise RuntimeError(f"12: no level for {mode} {year}; cannot express XAI predictions in °C")
    lev = round(float(lev), 4)
    base_c = round(float(SHAP_BASE[mode]) + lev, 4)
    shap = np.asarray(SHAP_GROUP_VALUES[mode][year], np.float64)
    if shap.shape != (_S12_NC, len(_S12_GROUPS)):
        raise ValueError(f"12: SHAP_GROUP_VALUES[{mode}][{year}] has shape {shap.shape}")
    pred = np.asarray(PRED_ANOM[mode][year], np.float64) + lev
    oof = np.asarray(OOF_ANOM[mode][year], np.float64) + lev
    anom = DF[f"anom_{mode}"].to_numpy(np.float64)
    j = _S12_OBS.index(year)
    obs = anom[j * _S12_NC:(j + 1) * _S12_NC] + lev
    if _s12_tier(mode, year) == "modelled":
        obs = np.full(_S12_NC, np.nan)
    resid = obs - oof
    if np.nanmax(np.abs(shap)) > 32.7:
        log(f"12: {mode} {year}: |group SHAP| up to {np.nanmax(np.abs(shap)):.1f} °C exceeds the int16 range at "
            "0.001 °C and is clipped", "WARNING")
    # Waterfall invariant on exactly what the browser will decode.
    shap_q = _s12_quantised(shap, _S12_SHAP_SCALE, lo=-32767, hi=32767)
    pred_q = _s12_quantised(pred, _S12_LST_SCALE, lo=-32767, hi=32767)
    rows = np.isfinite(shap_q).all(axis=1) & np.isfinite(pred_q)
    err = np.abs(pred_q[rows] - (base_c + shap_q[rows].sum(axis=1)))
    max_err = float(err.max()) if err.size else 0.0
    if not max_err <= _S12_WATERFALL_TOL_C:
        raise AssertionError(f"12: {mode} {year}: waterfall invariant violated after quantisation "
                             f"(max err {max_err:.4f} °C on {int((err > _S12_WATERFALL_TOL_C).sum())} rows)")
    field = _s12_frame_field(mode, year)
    # obs = anomaly + level; the level of a partially covered frame carries section 09's composition correction
    # (D8) while the frame field keeps the observed values, so obs = field + composition_correction.
    cc = _s12_comp_corr(mode, year)
    both = np.isfinite(field) & np.isfinite(obs)
    obs_vs_field = float(np.max(np.abs(field[both] + cc - obs[both]))) if both.any() else None
    _S12_XAI_STATS[(mode, year)] = {"waterfall_max_err": max_err, "n_rows": int(rows.sum()),
                                    "obs_vs_frame_field_max_diff": obs_vs_field}
    cols = [{"name": g, "array": shap[:, gi], "dtype": "int16", "scale": _S12_SHAP_SCALE, "offset": 0.0,
             "nodata": _S12_I16_NODATA} for gi, g in enumerate(_S12_GROUPS)]
    for name, arr in (("pred", pred), ("oof", oof), ("obs", obs), ("resid", resid)):
        cols.append({"name": name, "array": arr, "dtype": "int16", "scale": _S12_LST_SCALE, "offset": 0.0,
                     "nodata": _S12_I16_NODATA})
    meta = {"year": int(year), "mode": mode, "tier": _s12_tier(mode, year), "base_value_c": base_c,
            "level": lev, "shap_base": round(float(SHAP_BASE[mode]), 4), "groups": list(_S12_GROUPS),
            "units": "degC", "oof_scheme": "block20 XGBoost out-of-fold (NaN where the row was not a training row)",
            "invariant": "pred = base_value_c + sum(group SHAP) within 0.05 degC on rows without NaN",
            "composition_correction": round(cc, 4)}
    return _s12_write_lstb(f"xai_{year}_{mode}.lstb.gz", cols, meta, "xai")


with timer("12_binary"):
    _s12_write_cells()
    for _s12_m in _S12_MODES:
        _s12_write_obs_cube(_s12_m)
        for _s12_s in _S12_SCEN:
            _s12_write_ssp_cube(_s12_m, _s12_s)
        _s12_write_zone_cube(_s12_m)
    for _s12_y in _S12_XAI_YEARS:
        _s12_write_features(_s12_y)
    for _s12_m in _S12_MODES:
        for _s12_y in XAI_FRAMES.get(_s12_m, []):
            _s12_write_xai(_s12_m, int(_s12_y))
free_memory()
log("12: XAI waterfall max error after quantisation: " + ", ".join(
    f"{m}/{y}={v['waterfall_max_err']:.3f}" for (m, y), v in sorted(_S12_XAI_STATS.items())))
_s12_obs_diff = [v["obs_vs_frame_field_max_diff"] for v in _S12_XAI_STATS.values()
                 if v["obs_vs_frame_field_max_diff"] is not None]
if _s12_obs_diff:
    log(f"12: XAI 'obs' (anomaly + level) vs cube frame field: max |diff| {max(_s12_obs_diff):.2e} °C")
    # XAI obs = anomaly + level must be the same field as the cube frame (row order and level identity, C-01)
    if max(_s12_obs_diff) > 1e-3:
        raise ValueError(f"12: XAI 'obs' differs from the cube frame field by up to {max(_s12_obs_diff):.4f} °C "
                         "(row order or level mismatch between sections 06/08/09): "
                         + ", ".join(f"{m}/{y}={v['obs_vs_frame_field_max_diff']:.4f}"
                                     for (m, y), v in sorted(_S12_XAI_STATS.items())
                                     if (v["obs_vs_frame_field_max_diff"] or 0.0) > 1e-3))

## 12.7 Timeline series (SPEC §5.4)
One entry per scope (NCR, the 25 districts and the four zones of each mode) and mode: frame statistics of section
09 (mean, level CI, p90, hot-core area, signed SUHI, cell count, tier), the annual ERA5-Land and satellite series,
the SSP fans of section 11 (quantiles of per-draw scope means), trends, the decomposition of the projected change
into climate and land parts, the uncertainty budget (shares of the variance of the scope mean) and the years at
which the median first exceeds the weather-normalised baseline by +1/+2/+3 °C. Zone shares per frame close the file.

In [ ]:
def _s12_round_rows(rows, dp=2, keep_int=("year", "n_cells")):
    out = []
    for r in rows or []:
        rr = {}
        for k, v in r.items():
            if k in keep_int:
                rr[k] = None if v is None else int(v)
            elif k == "hot_core_km2":
                rr[k] = _s12_num(v, 1)
            elif k == "coverage":                 # finite cells / ROI cells of the scope (D16)
                rr[k] = _s12_num(v, 4)
            elif isinstance(v, (list, tuple)):
                rr[k] = [_s12_num(x, dp) for x in v]
            elif isinstance(v, str) or v is None or isinstance(v, bool):
                rr[k] = v
            else:
                rr[k] = _s12_num(v, dp)
        out.append(rr)
    return out


def _s12_trend(t):
    if not t:
        return None
    return {k: (_s12_num(v, 3) if isinstance(v, (float, np.floating)) else _s12_tree(v, 3)) for k, v in t.items()}


def _s12_series_entry(mode, scope):
    frames = _s12_round_rows(SERIES_FRAMES[mode].get(scope, []))
    ann = (SERIES_ANNUAL.get(mode) or {}).get(scope) or {}
    annual = {"era5_skin": [{"year": int(r["year"]), "value": _s12_num(r["value"], 2),
                             "anomaly": _s12_num(r.get("anomaly"), 2)} for r in ann.get("era5_skin", [])],
              "satellite": [{"year": int(r["year"]), "value": _s12_num(r["value"], 2), "sensor": r.get("sensor"),
                             "variable": r.get("variable"), "flag": r.get("flag")}
                            for r in ann.get("satellite", [])]}
    tr = (TRENDS.get(mode) or {})
    if scope == "ncr":
        trend = {"obs": _s12_trend(tr.get("obs")),
                 "annual": {k: _s12_trend(v) for k, v in (tr.get("annual") or {}).items()}}
    else:
        bs = (tr.get("by_scope") or {}).get(scope) or {}
        trend = {"obs": _s12_trend(bs.get("obs")),
                 "annual": {k: _s12_trend(v) for k, v in (bs.get("annual") or {}).items()}}
    ssp, decomp, budget, thr = {}, {}, {}, {}
    for s in _S12_SCEN:
        # A scope without cells (e.g. a district merged away in the boundary layer) has no section-11 rows: export
        # every projection year with null quantiles, as its observed frames carry null means (SPEC §5.4).
        ssp[s] = _s12_round_rows(PROJ_SERIES[mode][s].get(scope)
                                 or [{"year": int(y), **{k: None for k in ("p05", "p25", "p50", "p75", "p95")}}
                                     for y in _S12_PROJ])
        decomp[s], budget[s] = {}, {}
        for y in _S12_PROJ:
            d = ((PROJ_DECOMPOSITION[mode][s].get(y) or {}).get("by_scope") or {}).get(scope)
            if d:
                # total = p50 - baseline_c (the scope's weather-normalised 2025 baseline, D14)
                decomp[s][str(y)] = {k: _s12_num(d.get(k), 2) for k in ("climate", "land", "total", "baseline_c")}
            b = ((PROJ_UNCERTAINTY[mode][s].get(y) or {}).get("by_scope") or {}).get(scope)
            if b:
                budget[s][str(y)] = {k: _s12_num(b.get(k), 4) for k in ("gcm", "sensitivity", "land", "model")}
                budget[s][str(y)]["draw_sd_c"] = _s12_num(b.get("draw_sd_c"), 3)
        t = PROJ_THRESHOLD_YEARS[mode][s].get(scope) or {}
        thr[s] = {k: (None if t.get(k) is None else int(t[k])) for k in ("+1", "+2", "+3")}
    return {"frames": frames, "annual": annual, "ssp": ssp, "trend": trend, "decomposition": decomp,
            "uncertainty_budget": budget, "threshold_years": thr}


def _s12_zone_share(z):
    z = np.asarray(z)
    ok = z < 4
    n = int(ok.sum())
    if n == 0:
        return [None] * 4
    return [round(float(np.count_nonzero(z[ok] == k)) / n, 4) for k in range(4)]


def _s12_timeline_series():
    scopes = []
    for sc in SERIES_SCOPES:
        if sc["kind"] == "zone" and sc.get("mode") not in _S12_MODES:
            continue
        scopes.append({k: sc[k] for k in ("id", "label", "kind", "mode") if k in sc})
    series = {}
    for sc in scopes:
        modes = [sc["mode"]] if sc["kind"] == "zone" else _S12_MODES
        series[sc["id"]] = {m: _s12_series_entry(m, sc["id"]) for m in modes}
    zone_share = {}
    for m in _S12_MODES:
        zone_share[m] = {"obs": {str(y): _s12_zone_share(ZONES_OBS[m][y]) for y in _S12_OBS if y in ZONES_OBS[m]}}
        for s in _S12_SCEN:
            zone_share[m][s] = {str(y): _s12_zone_share(PROJ_ZONES[m][s][j]) for j, y in enumerate(_S12_PROJ)}
    lb = (globals().get("CMIP6_DELTAS") or {}).get("l_base") or {}
    return {"reference_period": list(CFG.REFERENCE_PERIOD),
            "hot_core_threshold_c": {m: _s12_num(HOT_CORE_THRESHOLD.get(m), 2) for m in _S12_MODES},
            "hot_core_frames": {m: [int(y) for y in (globals().get("_S09_HOT_FRAMES") or {}).get(m, [])]
                                for m in _S12_MODES},
            "baseline_level_c": {m: _s12_num((lb.get(m) or {}).get("l_base"), 2) for m in _S12_MODES},
            "frames_obs": list(_S12_OBS), "frames_proj": list(_S12_PROJ), "scenarios": list(_S12_SCEN),
            "scopes": scopes, "series": series, "zone_share": zone_share,
            "notes": [_S12_PROJ_NOTE] if _S12_PROJ else []}


with timer("12_timeline_series"):
    _S12_SERIES = _s12_timeline_series()
    _s12_write_json(_S12_SERIES, "timeline_series.json.gz")
log(f"12: timeline_series.json.gz: {len(_S12_SERIES['scopes'])} scopes, "
    f"{_S12_SIZES['timeline_series.json.gz'] / 1e6:.2f} MB")

## 12.8 Explanation files: shap_global, dependence, interactions, zones (SPEC §5.5)
All are `{"modes": {"day": {...}, "night": {...}}}` with the v1 content per mode, keyed by the 18 SHAP groups.
Zones additionally carry their projected area / LST per scenario and transition matrices from the current frame
into the projections (`"<ssp>": {"2025->2030": 4×4, ...}`).

In [ ]:
def _s12_threshold(th):
    s4 = lambda v: _s12_sig(v, 4)  # noqa: E731
    out = {"primary": th.get("primary")}
    for k in ("zero_crossing", "breakpoint", "saturation"):
        point = s4(th.get(k))
        ci = th.get(f"{k}_ci")
        out[k] = point
        # an interval is only meaningful around an existing point estimate (SPEC_v1 §4.4)
        out[f"{k}_ci"] = None if point is None or ci is None else [s4(ci[0]), s4(ci[1])]
        sup = th.get(f"{k}_support")
        out[f"{k}_support"] = None if point is None or sup is None else round(float(sup), 3)
    out["zero_crossing_flag"] = "multiple" if th.get("zero_crossing_flag") == "multiple" else None
    out["slope_before"] = s4(th.get("slope_before"))
    out["slope_after"] = s4(th.get("slope_after"))
    out["slope"] = s4(th.get("slope"))
    out["effect_range"] = _s12_num(th.get("effect_range"), 3)
    out["direction"] = th.get("direction")
    order = ["primary", "zero_crossing", "zero_crossing_ci", "zero_crossing_support", "zero_crossing_flag",
             "breakpoint", "breakpoint_ci", "breakpoint_support", "slope_before", "slope_after", "saturation",
             "saturation_ci", "saturation_support", "effect_range", "direction", "slope"]
    return {k: out[k] for k in order}


def _s12_r3(a):
    return [_s12_num(v, 3) for v in np.asarray(a, np.float64).tolist()]


def _s12_shap_global():
    modes = {}
    for m in _S12_MODES:
        imp = IMPORTANCE[m]
        g = imp["global"]
        info = BOOTSTRAP_INFO.get(m, {})
        modes[m] = {
            "groups": list(imp["groups"]),
            "global": {k: _s12_r3(g[k]) for k in ("mean_abs", "ci_lo", "ci_hi", "bias") if k in g},
            "by_frame": {str(y): {"mean_abs": _s12_r3(v["mean_abs"]), "mean": _s12_r3(v["mean"]),
                                  "n": int(v.get("n", 0))} for y, v in imp["by_frame"].items()},
            "by_zone": {str(z): {"mean_abs": _s12_r3(v["mean_abs"]), "mean": _s12_r3(v["mean"]),
                                 "n": int(v.get("n", 0))} for z, v in imp["by_zone"].items()},
            "order": [int(i) for i in imp.get("order", [])],
            "features": {"names": list(imp["features"]["names"]),
                         "mean_abs": _s12_r3(imp["features"]["mean_abs"])} if imp.get("features") else None,
            "fit_frames": [int(y) for y in imp.get("fit_frames", [])],
            "n_bootstrap": int(imp.get("n_bootstrap", info.get("n", 0))),
            "bootstrap_mode": info.get("mode", "block"),
            "ci_method": imp.get("ci_method", "recentred-percentile"),
            "ci_label": imp.get("ci_label", info.get("ci_label")),
        }
    return {"modes": modes}


def _s12_dependence():
    modes = {}
    for m in _S12_MODES:
        groups = {}
        for g, d in DEPENDENCE[m]["groups"].items():
            sc = d["scatter"]
            groups[g] = {
                "primary": d["primary"],
                "bin_centers": _s12_sig_increasing(d["bin_centers"]), "mean": np.round(np.asarray(d["mean"], float), 3),
                "ci_lo": np.round(np.asarray(d["ci_lo"], float), 3),
                "ci_hi": np.round(np.asarray(d["ci_hi"], float), 3),
                "count": np.asarray(d["count"], np.int64),
                "scatter": {"x": _s12_sig_arr(sc["x"]), "shap": np.round(np.asarray(sc["shap"], float), 3),
                            "color": _s12_sig_arr(sc["color"]), "color_feature": sc.get("color_feature"),
                            "color_group": sc.get("color_group"), "zone": np.asarray(sc["zone"], np.int64),
                            "year": np.asarray(sc["year"], np.int64)},
                "threshold": _s12_threshold(d["threshold"]),
            }
        modes[m] = {"groups": groups, "bins": int(CFG.DEP_BINS),
                    "x": "primary feature value", "y": "group SHAP (degC)"}
    return {"modes": modes}


def _s12_interactions():
    modes = {}
    for m in _S12_MODES:
        it = INTERACTIONS[m]
        r4 = lambda a: np.round(np.asarray(a, np.float64), 4)  # noqa: E731
        modes[m] = {"groups": list(it["groups"]), "global": r4(it["global"]),
                    "by_zone": {str(z): r4(v) for z, v in it["by_zone"].items()},
                    "n_rows": int(it.get("n_rows", 0)), "rows_per_frame": int(it.get("rows_per_frame", 0)),
                    "definition": it.get("definition")}
    return {"modes": modes}


def _s12_transition(a, b):
    a = np.asarray(a, np.int64)
    b = np.asarray(b, np.int64)
    ok = (a < 4) & (b < 4)
    return np.bincount(a[ok] * 4 + b[ok], minlength=16).reshape(4, 4).astype(int).tolist()


def _s12_recommendation(r):
    out = dict(r)
    out["current"] = _s12_sig(r.get("current"))
    out["target"] = _s12_sig(r.get("target"))
    out["step"] = _s12_sig(r.get("step"))
    out["expected_delta_c"] = _s12_num(r.get("expected_delta_c"), 2)
    out["ci"] = None if r.get("ci") is None else [_s12_num(r["ci"][0], 2), _s12_num(r["ci"][1], 2)]
    out["bias"] = _s12_num(r.get("bias"), 3)
    for k in ("mean_applied_step", "best_delta_c"):
        if k in r:
            out[k] = _s12_num(r[k], 4)
    if isinstance(r.get("bounds"), dict):
        out["bounds"] = {k: _s12_sig(v) for k, v in r["bounds"].items()}
    if isinstance(r.get("spillover"), dict):
        out["spillover"] = {"n_cells": int(r["spillover"].get("n_cells") or 0),
                            "mean_delta_c": _s12_num(r["spillover"].get("mean_delta_c"), 3)}
    return _s12_tree(out, 4)


def _s12_zones():
    modes = {}
    for m in _S12_MODES:
        zm = ZONE_MODEL[m]
        zones = []
        for p in ZONE_PROFILES[m]:
            z = int(p["id"])
            area = {"obs": {k: _s12_num(v, 1) for k, v in p["area_km2"]["obs"].items()}}
            lst_mean = {"obs": {k: _s12_num(v, 2) for k, v in p["lst_mean"]["obs"].items()}}
            lst_pred = {"obs": {k: _s12_num(v, 2) for k, v in p["lst_pred_mean"]["obs"].items()}}
            n_proj = {}
            for s in _S12_SCEN:
                zs = np.asarray(PROJ_ZONES[m][s])
                p50 = np.asarray(PROJ_QUANTILES[m][s]["p50"], np.float64)
                area[s], lst_mean[s], n_proj[s] = {}, {}, {}
                for j, y in enumerate(_S12_PROJ):
                    sel = zs[j] == z
                    n = int(sel.sum())
                    vals = p50[j][sel]
                    vals = vals[np.isfinite(vals)]
                    area[s][str(y)] = round(n * _S12_CELL_KM2, 1)
                    lst_mean[s][str(y)] = round(float(vals.mean()), 2) if vals.size else None
                    n_proj[s][str(y)] = n
            zones.append({
                "id": z, "name": p["name"], "color": p["color"], "mode": m, "description": p["description"],
                # D11: canonical name stays; display_name / label_caveat qualify a label the data do not support
                "display_name": str(p.get("display_name") or p["name"]), "label_caveat": p.get("label_caveat"),
                "n_cells": {str(k): int(v) for k, v in p["n_cells"].items()}, "n_cells_proj": n_proj,
                "area_km2": area, "lst_mean": lst_mean, "lst_pred_mean": lst_pred,
                "anom_pred_mean": _s12_tree(p.get("anom_pred_mean"), 3),
                "anom_obs_mean": _s12_tree(p.get("anom_obs_mean"), 3),
                "feature_means": {f: _s12_sig(v) for f, v in p["feature_means"].items()},
                "feature_means_frame": int(p.get("feature_means_frame", _S12_NOW)),
                "shap_means": {g: _s12_num(v, 3) for g, v in p["shap_means"].items()},
                "top_warming": _s12_tree(p["top_warming"], 3), "top_cooling": _s12_tree(p["top_cooling"], 3),
                "top_interactions": _s12_tree(p["top_interactions"], 4),
                "top_districts": _s12_tree(p["top_districts"], 4),
                "recommendations": [_s12_recommendation(r) for r in p["recommendations"]],
            })
        transitions = {"obs": ZONE_TRANSITIONS[m]["obs"]}
        for s in _S12_SCEN:
            tr, prev_y, prev = {}, _S12_NOW, ZONES_OBS[m][_S12_NOW]
            for j, y in enumerate(_S12_PROJ):
                cur = PROJ_ZONES[m][s][j]
                tr[f"{prev_y}->{y}"] = _s12_transition(prev, cur)
                prev_y, prev = y, cur
            transitions[s] = tr
        diag = zm.get("diagnostics", {})
        modes[m] = {"k": int(zm.get("k", 4)), "silhouette": _s12_num(zm.get("silhouette"), 4),
                    "diagnostics": {"k": [int(k) for k in diag.get("k", [])],
                                    "inertia": [_s12_sig(v, 6) for v in diag.get("inertia", [])],
                                    "silhouette": [_s12_num(v, 4) for v in diag.get("silhouette", [])],
                                    "n_init": int(diag.get("n_init", 10)), "n_rows": int(diag.get("n_rows", 0))},
                    "labelling": _s12_tree(zm.get("labelling"), 4), "groups": list(_S12_GROUPS),
                    "centroids": np.round(np.asarray(zm["centroids"], np.float64), 4),
                    "space": zm.get("space"), "zones": zones, "transitions": transitions}
    return {"modes": modes}


with timer("12_explain_files"):
    _s12_write_json(_s12_shap_global(), "shap_global.json")
    _s12_write_json(_s12_dependence(), "dependence.json.gz")
    _s12_write_json(_s12_interactions(), "interactions.json")
    _s12_write_json(_s12_zones(), "zones.json")
free_memory()

## 12.9 Metrics, provenance, web models, coupling and districts (SPEC §5.6–§5.8)

In [ ]:
_S12_SCHEME_ORDER = ["block20", "block5", "district", "loeo", "place_time", "random"]


def _s12_metrics_mode(m):
    mr = MODEL_RESULTS[m]
    schemes_present = list(dict.fromkeys(mr["cv_summary"]["scheme"].tolist())) if len(mr["cv_summary"]) else []
    schemes = [s for s in _S12_SCHEME_ORDER if s in schemes_present] + [
        s for s in schemes_present if s not in _S12_SCHEME_ORDER]
    out = {"schemes": schemes, "models": list(CFG.BENCHMARKS),
           "summary": _s12_tree(mr["cv_summary"], 4), "folds": _s12_tree(mr["cv_fold_metrics"], 4)}
    for key in ("block_size_curve", "error_vs_distance", "loeo", "loeo_by_model", "place_time", "moran",
                "moran_target", "fold_geometry", "aoa", "persistent_residual", "v1_comparison", "headline",
                "ablation", "best_rounds", "final_rounds", "frames", "n_train", "district_merges", "monotone"):
        if key in mr:
            out[key] = _s12_tree(mr[key], 4)
    ms = mr.get("moran_scatter") or {}
    out["moran_scatter"] = _s12_tree(ms, 3)
    out["hindcast"] = _s12_tree((globals().get("HINDCAST_METRICS") or {}).get(m), 4)
    if m == "day":
        dlc = globals().get("DAY_LEVEL_CHECK") or {}
        out["cross_sensor"] = {"landsat_vs_modis_terra": _s12_tree(dlc.get("cross_sensor", []), 4)}
        out["day_level_check"] = _s12_tree({k: v for k, v in dlc.items() if k != "cross_sensor"}, 4)
    else:
        nc = globals().get("NIGHT_CALIBRATION") or {}
        out["cross_sensor"] = {k: _s12_tree({kk: vv for kk, vv in v.items() if kk != "by_class"}, 4)
                               for k, v in nc.items() if isinstance(v, dict)}
    return out


def _s12_night_calibration():
    """metrics.night_calibration: SPEC §3.2 keys per kind, plus 'diag' (section 04, gee mode) when available (D19)."""
    nc = globals().get("NIGHT_CALIBRATION")
    diag = globals().get("NIGHT_CALIBRATION_DIAG")
    if not isinstance(nc, dict) or not isinstance(diag, dict):
        return _s12_tree(nc, 4)
    out = {}
    for kind, rec in nc.items():
        if isinstance(rec, dict) and isinstance(diag.get(kind), dict):
            out[kind] = {**_s12_tree(rec, 4), "diag": _s12_tree(diag[kind], 4)}
        else:
            out[kind] = _s12_tree(rec, 4)
    return out


def _s12_metrics():
    cs = {}
    for m in _S12_MODES:
        c = CLIMATE_SENSITIVITY.get(m) or {}
        cs[m] = _s12_tree({k: v for k, v in c.items() if k != "draws"}, 4)
    cmip = globals().get("CMIP6_DELTAS") or {}
    cmip_out = None
    if cmip.get("available"):
        cmip_out = {"baseline_frame": cmip.get("baseline_frame"), "baseline_window": cmip.get("baseline_window"),
                    "observation_window": cmip.get("observation_window"), "frames": cmip.get("frames"),
                    "definition": cmip.get("definition"),
                    "l_base": _s12_tree(cmip.get("l_base"), 4), "scenarios": {}}
        for s, rec in (cmip.get("scenarios") or {}).items():
            cmip_out["scenarios"][s] = {
                "method": rec.get("method"), "n_gcm": rec.get("n_gcm"), "source_scenario": rec.get("source_scenario"),
                "gsat_ratio": _s12_tree(rec.get("gsat_ratio"), 4),
                "variables": {v: {"models": list(d.get("models", [])), "summary": _s12_tree(d.get("summary"), 4)}
                              for v, d in (rec.get("variables") or {}).items()}}
    dm = globals().get("DOWNSCALING_METRICS") or {}
    hw = {"gpus": list(ENV.get("gpu_names") or []), "n_gpus": int(ENV.get("n_gpus") or 0),
          "xgb_device": ENV.get("xgb_device"), "lgbm_device": ENV.get("lgbm_device"),
          "rapids": bool(ENV.get("rapids_used")), "rapids_used": sorted(ENV.get("rapids_used") or []),
          "xgb_version": ENV.get("xgb_version"), "torch_version": str(ENV.get("torch_version") or ""),
          "python": ENV.get("python"), "platform": ENV.get("platform"), "cpu_count": ENV.get("cpu_count"),
          "is_kaggle": bool(ENV.get("is_kaggle"))}
    return {
        "modes": {m: _s12_metrics_mode(m) for m in _S12_MODES},
        "downscaling": _s12_tree(dm, 4),
        "land_model": _s12_tree(globals().get("LAND_MODEL_METRICS"), 4),
        "climate_sensitivity": cs,
        "night_calibration": _s12_night_calibration(),
        "cmip6": cmip_out,
        "projection_meta": _s12_tree({m: {s: {k: v for k, v in pm.items() if k != "shap_group_mean"}
                                              for s, pm in (globals().get("PROJ_META") or {}).get(m, {}).items()}
                                      for m in _S12_MODES}, 4),
        "bootstrap": {m: _s12_tree({k: v for k, v in BOOTSTRAP_INFO.get(m, {}).items() if k != "failures"}, 4)
                      for m in _S12_MODES},
        "hardware": hw,
        "timings": {str(k): round(float(v), 2) for k, v in TIMINGS.items()},
        "profile": CFG.PROFILE, "grid_res_m": int(CFG.GRID_RES_M), "data_mode": DATA_MODE,
    }


def _s12_provenance():
    syn = DATA_MODE == "synthetic"
    tag = " (SYNTHETIC stand-in generated by section 02; the gee run uses this product)" if syn else ""
    src = [
        ("landsat_c2_l2", "Landsat 5/7/8/9 Collection 2 Level-2 (SR + ST)", "LANDSAT/LT05|LE07|LC08|LC09/C02/T1_L2",
         1989, 2026, 30, "day target", ["lst_day", "ndvi", "ndbi", "mndwi", "albedo", "ndvi_spring"],
         "per-path median composites, path-offset harmonised, corrected to 10:45 IST"),
        ("modis_terra_day", "MODIS Terra LST 8-day (MOD11A2 v061)", "MODIS/061/MOD11A2", 2000, 2026, 1000,
         "day validation", ["lst_day_modis", "annual satellite series", "beta (day)"],
         "QC + view-time filtered; drift flagged from 2022"),
        ("modis_aqua_night", "MODIS Aqua LST 8-day (MYD11A2 v061)", "MODIS/061/MYD11A2", 2002, 2021, 1000,
         "night target", ["lst_night 2005-2020", "beta (night)"], "QC bits 0-1 and 6-7 <= 1"),
        ("modis_terra_night", "MODIS Terra LST night (22:30, MOD11A2)", "MODIS/061/MOD11A2", 1999, 2001, 1000,
         "night reconstruction", ["lst_night 2000"], "calibrated to Aqua (NIGHT_CALIBRATION.terra_to_aqua)"),
        ("viirs_night", "VIIRS VNP21A1N daily night LST", "NASA/VIIRS/002/VNP21A1N", 2012, 2026, 1000,
         "night target", ["lst_night 2025"], "calibrated to Aqua (NIGHT_CALIBRATION.viirs_to_aqua)"),
        ("ghsl_built_s", "GHSL built-up surface P2023A", "JRC/GHSL/P2023A/GHS_BUILT_S", 1975, 2030, 100,
         "predictor", ["frac_built"], "fraction of built surface"),
        ("ghsl_built_v", "GHSL built-up volume P2023A", "JRC/GHSL/P2023A/GHS_BUILT_V", 1975, 2030, 100,
         "predictor", ["built_v"], "m3 per m2 ground"),
        ("ghsl_pop", "GHSL population P2023A", "JRC/GHSL/P2023A/GHS_POP", 1975, 2030, 100, "predictor",
         ["log_pop", "projected population"], "persons per km2"),
        ("glc_fcs30d", "GLC_FCS30D land cover", "projects/sat-io/open-datasets/GLC-FCS30D", 1985, 2022, 30,
         "predictor", ["frac_forest", "frac_water", "frac_cropland", "frac_barren", "lm_pd", "lm_ed", "lm_contag"],
         "one product for every frame; 2025 uses the 2022 map"),
        ("srtm", "SRTM 1 arc-second DEM", "USGS/SRTMGL1_003", 2000, 2000, 30, "predictor",
         ["elevation", "slope", "aspect", "elev_tpi7", "elev_tpi21", "elev_std7"], "static"),
        ("jrc_gsw", "JRC Global Surface Water occurrence", "JRC/GSW1_4/GlobalSurfaceWater", 1984, 2021, 30,
         "predictor", ["dist_water", "water_occ"], "static"),
        ("era5_land", "ERA5-Land monthly / daily / hourly", "ECMWF/ERA5_LAND/MONTHLY_AGGR|DAILY_AGGR|HOURLY",
         1989, 2026, 11132, "predictor + annual series",
         ["era5_t2m", "era5_vpd", "era5_swvl1", "era5_evap", "era5_ssr", "era5_wind", "era5_precip_jfm",
          "heating rate", "night level 1990/1995"], "bilinear to the analysis grid"),
        ("gddp_cmip6", "NASA NEX-GDDP-CMIP6", "NASA/GDDP-CMIP6", 1950, 2060, 27830, "projection climate",
         ["tasmax delta (day)", "tasmin delta (night)"], "historical + ssp245/ssp585; ssp126 pattern-scaled"),
        ("chen2020", "Chen et al. 2020 global SSP urban land", "projects/sat-io/open-datasets/FUTURE-URBAN-LAND/"
         "CHEN_2020_2100", 2020, 2100, 1000, "projection land demand", ["urban growth ratios"],
         "SSP1/2/5 mapped to ssp126/245/585"),
        ("iiasa_ssp", "IIASA SSP database v3 (India urban population)", "external table", 2020, 2050, None,
         "projection population", ["population factor"], "national urban population growth factor"),
        ("ar6_gsat", "IPCC AR6 WG1 Table 4.5 assessed GSAT", "publication", 2021, 2100, None,
         "ssp126 pattern scaling", ["gsat_increment"], "interpolated between assessed windows"),
    ]
    sources = [{"id": i, "label": lab, "asset": asset, "start": s0, "end": s1, "resolution_m": res, "role": role,
                "used_for": used, "notes": note + tag} for i, lab, asset, s0, s1, res, role, used, note in src]
    frames = []
    for fr in _S12_TIMELINE_FRAMES:
        for m in _S12_MODES:
            e = fr[m]
            frames.append({"year": fr["year"], "mode": m, "tier": e.get("tier"), "sensor": e.get("sensor"),
                           "window": fr["window"], "scenes": e.get("scenes"), "coverage": e.get("coverage"),
                           "overpass_ist": e.get("overpass_ist")})
    return {"data_mode": DATA_MODE, "sources": sources, "frames": frames, "annotations": _S12_ANNOTATIONS,
            "data_sources": _S12_DATA_SOURCES, "caveats": [str(c) for c in (globals().get("DATA_CAVEATS") or [])]}


def _s12_data_sources():
    out = {}
    for var, vals in DATA_SOURCES.items():
        if isinstance(vals, dict):
            out[str(var)] = {str(k): str(v) for k, v in vals.items()}
        else:
            out[str(var)] = {"all": str(vals)}
    return out


_S12_DATA_SOURCES = _s12_data_sources()
_S12_ANNOTATIONS = [
    {"year": 1999, "label": "Landsat 7 ETM+ joins Landsat 5", "kind": "sensor"},
    {"year": 2003, "label": "Landsat 7 SLC-off", "kind": "sensor"},
    {"year": 2013, "label": "Landsat 8 OLI/TIRS", "kind": "sensor"},
    {"year": 2022, "label": "Terra/Aqua drift; VIIRS bridge", "kind": "sensor"},
]
if _S12_PROJ:
    _S12_ANNOTATIONS.append({"year": _S12_NOW, "label": f"now ({_S12_NOW}); projections beyond are scenario "
                                                       "projections, not forecasts", "kind": "method"})


def _s12_web_model(m):
    model = dict(WEB_MODELS[m])
    if model.get("format") != "lst-xgb-compact-v1":
        raise ValueError(f"12: WEB_MODELS[{m}] has format {model.get('format')!r}")
    if list(model.get("features", [])) != list(FEATURES):
        raise ValueError(f"12: WEB_MODELS[{m}] feature order differs from FEATURES")
    model.setdefault("focal", {"bases": list(FOCAL_SPEC["bases"]), "windows": list(FOCAL_SPEC["windows"])})
    model.setdefault("mode", m)
    return model


def _s12_coupling():
    c = SCENARIO_COUPLING
    out = {k: _s12_tree(v, 6) for k, v in c.items()}
    out["coupling"] = {f: {k: _s12_sig(v, 4) for k, v in d.items()} for f, d in c["coupling"].items()}
    out["intercept"] = {k: _s12_sig(v, 4) for k, v in c.get("intercept", {}).items()}
    out["r2"] = {k: _s12_num(v, 4) for k, v in c.get("r2", {}).items()}
    out["focal"] = {"bases": list(FOCAL_SPEC["bases"]), "windows": [int(w) for w in FOCAL_SPEC["windows"]]}
    out["grid"] = {"width": int(GRID.width), "height": int(GRID.height), "res": int(GRID.res), "crs": GRID.crs,
                   "x0": float(GRID.x0), "y0": float(GRID.y0)}
    return out


def _s12_round_coords(obj):
    if isinstance(obj, dict):
        return {k: _s12_round_coords(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_s12_round_coords(v) for v in obj]
    if isinstance(obj, (float, np.floating)):
        return round(float(obj), 5)
    return obj


def _s12_districts_geojson():
    """BOUNDARY_GEOJSON with 5-dp coordinates and exactly the contract properties (SPEC_v1 §4.8)."""
    feats = []
    for feat in BOUNDARY_GEOJSON["features"]:
        props = feat.get("properties", {})
        feats.append({"type": "Feature", "geometry": _s12_round_coords(feat["geometry"]),
                      "properties": {"id": int(props["id"]), "name": str(props["name"]),
                                     "state": str(props["state"])}})
    return {"type": "FeatureCollection", "features": feats}


with timer("12_small_files"):
    _S12_METRICS = _s12_metrics()
    for _s12_m in _S12_MODES:
        _s12_write_json(_s12_web_model(_s12_m), f"model_web_{_s12_m}.json")
    _s12_write_json(_s12_coupling(), "scenario_coupling.json")
    _S12_DISTRICTS_FC = _s12_districts_geojson()
    _s12_write_json(_S12_DISTRICTS_FC, "districts.geojson")
    _s12_write_json(_s12_provenance(), "provenance.json")

## 12.10 Manifest (SPEC §5.3)
The manifest is the dashboard's table of contents: it declares the study area and grid, both modes, the 47 features
(with statistics, family, SHAP group and base flag), the 18 groups, the zones of each mode, the districts, every file,
and the **timeline**: one record per frame (1990–2050) and mode with tier, sensor, overpass, scene count, coverage,
native resolution, level and level interval, the scenarios, the fixed colour domains, the annotations and the cube
file names. `files.xai` lists one features file and one SHAP file per mode and frame.

In [ ]:
def _s12_bbox():
    """Lon/lat bounds of the ROI cell squares (corner lattice transformed once; 5 dp)."""
    try:
        from pyproj import Transformer
    except ImportError:
        pip_install(["pyproj"])
        from pyproj import Transformer
    tr = Transformer.from_crs(GRID.crs, "EPSG:4326", always_xy=True)
    r = np.asarray(CELL_ROWS, np.float64)
    c = np.asarray(CELL_COLS, np.float64)
    xs = np.concatenate([GRID.x0 + c * GRID.res, GRID.x0 + (c + 1) * GRID.res] * 2)
    ys = np.concatenate([GRID.y0 - r * GRID.res, GRID.y0 - r * GRID.res, GRID.y0 - (r + 1) * GRID.res,
                         GRID.y0 - (r + 1) * GRID.res])
    lon, lat = tr.transform(xs, ys)
    return [round(float(np.min(lon)), 5), round(float(np.min(lat)), 5), round(float(np.max(lon)), 5),
            round(float(np.max(lat)), 5)]


_S12_BBOX = _s12_bbox()
_S12_MODEL_NOTES = [
    "SHAP values explain the model, not causal effects. Zone recommendations are model-based counterfactuals (the "
    "final model re-evaluated on the zone's cells after a feasible land-cover / NDVI / landscape change, with the "
    "neighbourhood (focal) features recomputed); the cooling they report is associated with the change in the "
    "model, not a causal effect.",
    "Targets are per-frame anomalies (fixed-mask mean removed); absolute degC = anomaly + the frame level of section "
    "09. Day levels are corrected to a 10:45 IST reference overpass; night is in Aqua-equivalent units.",
    "Headline accuracy is pooled out-of-fold R2/RMSE under 20 km block cross-validation (spatial) and leave-one-"
    "epoch-out with recentred predictions (temporal); random CV is an optimistic upper bound.",
    "Night 1990 and 1995 are model hindcasts (tier 'modelled', no satellite product exists); night 2000 is Terra "
    "22:30 calibrated to Aqua (tier 'reconstructed'). Their intervals combine conformal model error and the level "
    "uncertainty.",
    "In-browser scenarios use depth-6 surrogates of the final models (model_web_<mode>.json fidelity) and recompute "
    "the focal features on the grid (FOCAL_SPEC).",
]


def _s12_notes():
    notes = [str(c) for c in (globals().get("DATA_CAVEATS") or [])]
    if DATA_MODE == "synthetic" and not any("SYNTHETIC" in n.upper() for n in notes):
        notes.insert(0, "SYNTHETIC DEMO DATA: generated by the notebook's synthetic world (section 02), not satellite "
                        "observations. Use for software demonstration only.")
    if BOUNDARY_APPROXIMATE and DATA_MODE == "gee":
        notes.append("District boundaries are APPROXIMATE (fallback outline / HQ-point Voronoi); areas per district "
                     "are indicative.")
    notes += _S12_MODEL_NOTES
    for m in _S12_MODES:
        info = BOOTSTRAP_INFO.get(m, {})
        if info.get("replicate_range"):
            notes.append(f"{m}: only {info.get('n')} bootstrap replicates - intervals are a replicate range "
                         "(bias-corrected 2.5-97.5 percentiles), not calibrated 95% confidence intervals.")
    if _S12_PROJ:
        notes.append(f"Frames {_S12_PROJ[0]}-{_S12_PROJ[-1]} are a {_S12_PROJ_NOTE}: CMIP6 temperature deltas x an "
                     "empirical dLST/dTair sensitivity + simulated land change, Monte-Carlo quantiles (GCM, "
                     "sensitivity, land realisation, model error). SSP1-2.6 climate is pattern-scaled from SSP2-4.5.")
    notes += [f"Colour domain: {n}" for n in _S12_DOMAIN_NOTES]
    return notes


def _s12_features_meta():
    out = []
    for f in FEATURES:
        meta = FEATURE_META[f]
        st = FEATURE_STATS.get(f, {})
        out.append({"name": f, "label": meta["label"], "unit": meta["unit"], "group": meta["group"],
                    "family": meta["family"], "base": bool(meta["base"]), "description": meta["description"],
                    "source": meta["source"], "actionable": bool(meta["actionable"]), "display": meta["display"],
                    "stats": {k: _s12_sig(st.get(k)) for k in ("min", "max", "p01", "p99", "median", "mean", "std")}
                    | {"missing": _s12_num(st.get("missing"), 5)}})
    return out


def _s12_manifest_zones(m):
    """manifest.zones[m]: the canonical zones (_S12_ZONE_META order, ids/names/colours) + the D11 display fields."""
    prof = {int(p["id"]): p for p in (globals().get("ZONE_PROFILES") or {}).get(m, [])}
    out = []
    for z in _S12_ZONE_META:
        p = prof.get(int(z["id"])) or {}
        out.append({**z, "display_name": str(p.get("display_name") or z["name"]),
                    "label_caveat": p.get("label_caveat")})
    return out


def _s12_manifest():
    files_xai = {m: {str(y): {"features": f"xai_{y}_features.lstb.gz", "shap": f"xai_{y}_{m}.lstb.gz"}
                     for y in XAI_FRAMES.get(m, [])} for m in _S12_MODES}
    cubes = {f"obs_{m}": f"cube_obs_{m}.lstb.gz" for m in _S12_MODES}
    for m in _S12_MODES:
        for s in _S12_SCEN:
            cubes[f"{s}_{m}"] = f"cube_{s}_{m}.lstb.gz"
        cubes[f"zones_{m}"] = f"cube_zones_{m}.lstb.gz"
    has_raster = ((_S12_WEB / "raster_index.json").is_file()
                  and bool((globals().get("RASTER_INDEX") or {}).get("frames")))
    scenarios = []
    for s in _S12_SCEN:
        rec = CMIP6_DELTAS["scenarios"].get(s, {})
        scenarios.append({"id": s, "label": _S12_SCEN_LABEL.get(s, s), "method": rec.get("method", "gcm"),
                          "n_gcm": int(rec.get("n_gcm") or 0),
                          "source_scenario": rec.get("source_scenario")})
    roi_prov = DATA_SOURCES.get("roi")
    roi_sources = sorted({str(v) for v in roi_prov.values()}) if isinstance(roi_prov, dict) else []
    shap_base = {m: round(float(SHAP_BASE[m]), 4) for m in _S12_MODES}
    return {
        "schema_version": "2.0",
        "generated_at": _s12_dt.datetime.now(_s12_dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "generator": {"notebook": "delhi_ncr_lst_pipeline", "export": _S12_VERSION, "profile": CFG.PROFILE},
        "data_mode": DATA_MODE,
        "data_mode_reason": str(globals().get("DATA_MODE_REASON") or "") or None,
        "study_area": {"name": "Delhi NCR", "area_km2": round(_S12_NC * _S12_CELL_KM2, 1), "crs": CFG.CRS,
                       "grid_res_m": int(CFG.GRID_RES_M), "fine_res_m": int(CFG.grid_fine_res()),
                       "grid": {"crs": GRID.crs, "res": int(GRID.res), "x0": float(GRID.x0), "y0": float(GRID.y0),
                                "width": int(GRID.width), "height": int(GRID.height)},
                       "n_cells": _S12_NC, "bbox": _S12_BBOX,
                       "center": [round(0.5 * (_S12_BBOX[0] + _S12_BBOX[2]), 5),
                                  round(0.5 * (_S12_BBOX[1] + _S12_BBOX[3]), 5)],
                       "boundary_approximate": bool(BOUNDARY_APPROXIMATE),
                       "boundary_source": "; ".join(roi_sources) or None},
        "season": list(CFG.SEASON),
        "target_mode": "anomaly",
        "modes": [{"id": m, **{k: _S12_MODE_INFO[m][k] for k in ("label", "target", "ramp")},
                   "reference_overpass_ist": _S12_MODE_INFO[m]["reference_overpass_ist"]} for m in _S12_MODES],
        "features": _s12_features_meta(),
        "focal": {"bases": list(FOCAL_SPEC["bases"]), "windows": [int(w) for w in FOCAL_SPEC["windows"]]},
        "groups": [{"id": g, "label": GROUP_META[g]["label"], "primary": GROUP_META[g]["primary"],
                    "family": GROUP_META[g]["family"], "actionable": bool(GROUP_META[g]["actionable"]),
                    "members": list(SHAP_GROUPS[g])} for g in _S12_GROUPS],
        "zones": {m: _s12_manifest_zones(m) for m in _S12_MODES},
        "districts": [{"id": int(d["id"]), "name": d["name"], "state": d["state"]} for d in DISTRICTS],
        "shap_base": shap_base,
        "timeline": {
            "now": int(CFG.NOW), "harmonised": True, "reference_period": list(CFG.REFERENCE_PERIOD),
            "frames": _S12_TIMELINE_FRAMES, "scenarios": scenarios,
            "domains": {**{m: {k: _S12_DOMAINS[m][k] for k in ("abs", "anom", "suhi")} for m in _S12_MODES},
                        **({"dtr": _S12_DOMAINS["dtr"]} if "dtr" in _S12_DOMAINS else {}),
                        "spread": _S12_DOMAINS["spread"]},
            "annotations": _S12_ANNOTATIONS, "tiers": list(_S12_TIERS), "cubes": cubes,
            "raster_index": "raster_index.json" if has_raster else None,
        },
        "files": {"cells": "cells.lstb.gz", "series": "timeline_series.json.gz", "xai": files_xai,
                  "model": {m: f"model_web_{m}.json" for m in _S12_MODES}, "shap_global": "shap_global.json",
                  "dependence": "dependence.json.gz", "zones": "zones.json", "metrics": "metrics.json",
                  "interactions": "interactions.json", "coupling": "scenario_coupling.json",
                  "districts": "districts.geojson", "provenance": "provenance.json",
                  **({"raster_index": "raster_index.json"} if has_raster else {})},
        "data_sources": _S12_DATA_SOURCES,
        "notes": _s12_notes(),
        "bootstrap": {m: _s12_tree({k: v for k, v in BOOTSTRAP_INFO.get(m, {}).items()
                                    if k in ("n", "n_requested", "mode", "block_size_km", "n_blocks",
                                             "block_size_source", "ci_level", "ci_method", "ci_method_description",
                                             "ci_label", "replicate_range", "full_data", "rounds")}, 4)
                      for m in _S12_MODES},
        "synthetic_truth": _s12_tree(SYNTHETIC_TRUTH, 6) if DATA_MODE == "synthetic" else None,
    }


with timer("12_manifest"):
    EXPORT_MANIFEST = _s12_manifest()
    _s12_write_json(_S12_METRICS, "metrics.json")
    _s12_write_json(EXPORT_MANIFEST, "manifest.json")
    if EXPORT_MANIFEST["timeline"]["raster_index"]:
        _S12_SIZES["raster_index.json"] = (_S12_WEB / "raster_index.json").stat().st_size
        for _s12_fr in RASTER_INDEX["frames"].values():
            for _s12_k in ("file", "preview"):
                _s12_p = _S12_WEB / _s12_fr[_s12_k]
                if not _s12_p.is_file():
                    raise FileNotFoundError(f"12: raster {_s12_p} referenced by raster_index.json is missing")
                _S12_SIZES[_s12_fr[_s12_k]] = _s12_p.stat().st_size


def _s12_clear_stale():
    """Remove bundle files of earlier exports that this export did not write (never touches the cache)."""
    keep = set(_S12_SIZES)
    removed = []
    for p in sorted(_S12_WEB.rglob("*")):
        if not p.is_file():
            continue
        rel = p.relative_to(_S12_WEB).as_posix()
        if rel in keep:
            continue
        if rel.endswith((".json", ".json.gz", ".lstb.gz", ".lstb", ".geojson", ".png", ".tmp")):
            p.unlink()
            removed.append(rel)
    if removed:
        log(f"12: removed {len(removed)} stale bundle file(s): {removed[:8]}{' ...' if len(removed) > 8 else ''}")
    return removed


_s12_clear_stale()
_S12_BUNDLE_BYTES = sum(_S12_SIZES.values())
log(f"12: web bundle written: {len(_S12_SIZES)} files, {_S12_BUNDLE_BYTES / 1e6:.2f} MB on disk")

## 12.11 Read-back validation of the written bundle
Everything below re-reads the files from disk (no in-memory shortcut) and fails the section on any contract
violation. `lstb_parse` gives the raw integers so dtypes, nodata codes and scales are checked exactly.

In [ ]:
def _s12_reject_constant(token):
    raise ValueError(f"non-JSON constant {token!r}")


def _s12_load(name):
    raw = (_S12_WEB / name).read_bytes()
    if raw[:2] == b"\x1f\x8b":
        raw = _s12_gzip.decompress(raw)
    return _s12_json.loads(raw.decode("utf-8"), parse_constant=_s12_reject_constant)


def _s12_decode(col, raw):
    v = raw.astype(np.float64) * float(col["scale"]) + float(col["offset"])
    if col.get("nodata") is not None:
        v[raw == col["nodata"]] = np.nan
    return v


def _s12_check_bundle():
    errors = []

    def req(cond, msg):
        if not cond:
            errors.append(msg)
        return bool(cond)

    man = _s12_load("manifest.json")
    req(man["schema_version"] == "2.0", "manifest.schema_version != 2.0")
    tl = man["timeline"]
    obs_years = [f["year"] for f in tl["frames"] if f["kind"] == "observed"]
    proj_years = [f["year"] for f in tl["frames"] if f["kind"] == "projected"]
    req(obs_years == _S12_OBS and proj_years == _S12_PROJ, "manifest frame lists differ from the pipeline frames")
    tiers = {m: [f[m]["tier"] for f in tl["frames"] if f["kind"] == "observed"] for m in _S12_MODES}
    levels = {m: [f[m]["level"] for f in tl["frames"] if f["kind"] == "observed"] for m in _S12_MODES}
    # every referenced file exists
    names = [man["files"][k] for k in ("cells", "series", "shap_global", "dependence", "zones", "metrics",
                                       "interactions", "coupling", "districts", "provenance")]
    names += list(man["files"]["model"].values()) + list(tl["cubes"].values())
    names += [v for m in man["files"]["xai"].values() for d in m.values() for v in d.values()]
    for nm in names:
        req((_S12_WEB / nm).is_file(), f"missing file {nm}")
    if errors:
        return errors
    for nm in names:
        if nm.endswith((".json", ".json.gz", ".geojson")):
            try:
                _s12_load(nm)
            except ValueError as exc:
                errors.append(f"{nm}: not strict JSON ({exc})")
    # cells
    h, raws = lstb_parse((_S12_WEB / "cells.lstb.gz").read_bytes())
    req(h["kind"] == "cells" and h["n"] == _S12_NC, "cells: kind/n")
    dt = {c["name"]: c["dtype"] for c in h["columns"]}
    req(dt == {"cell_id": "int32", "row": "int16", "col": "int16", "lon": "float32", "lat": "float32",
               "district": "int16"}, f"cells: dtypes {dt}")
    req(np.array_equal(raws["cell_id"], np.asarray(CELL_IDS, np.int32)), "cells: cell_id differs from CELL_IDS")
    W = man["study_area"]["grid"]["width"]
    req(np.array_equal(raws["row"].astype(np.int64) * W + raws["col"], raws["cell_id"].astype(np.int64)),
        "cells: cell_id != row * W + col")
    # cubes
    cube_lst, cube_zones = {}, {}
    for key, nm in tl["cubes"].items():
        h, raws = lstb_parse((_S12_WEB / nm).read_bytes())
        req(h["n"] == _S12_NC, f"{nm}: n {h['n']} != {_S12_NC}")
        cols = {c["name"]: c for c in h["columns"]}
        if key.startswith("zones_"):
            req(h["kind"] == "zones", f"{nm}: kind")
            cube_zones[h["meta"]["mode"]] = raws["obs"]
            req([{k: z.get(k) for k in ("id", "name", "color")} for z in h["meta"]["zones"]] == _S12_ZONE_META,
                f"{nm}: meta.zones must be the canonical zones")
            for name, c in cols.items():
                F = len(_S12_OBS) if name == "obs" else len(_S12_PROJ)
                req(c["dtype"] == "uint8" and c["shape"] == [F, _S12_NC] and c["nodata"] == 255,
                    f"{nm}.{name}: dtype/shape/nodata")
                bad = ~np.isin(raws[name], [0, 1, 2, 3, 255])
                req(not bad.any(), f"{nm}.{name}: {int(bad.sum())} values outside {{0..3, 255}}")
            continue
        req(h["kind"] == "cube", f"{nm}: kind")
        mode = h["meta"]["mode"]
        main = "lst" if key.startswith("obs_") else "p50"
        F = len(_S12_OBS) if main == "lst" else len(_S12_PROJ)
        req(h["meta"]["frames"] == (_S12_OBS if main == "lst" else _S12_PROJ), f"{nm}: meta.frames")
        c = cols[main]
        req(c["dtype"] == "int16" and abs(c["scale"] - _S12_LST_SCALE) < 1e-12 and c["nodata"] == _S12_I16_NODATA
            and c["shape"] == [F, _S12_NC], f"{nm}.{main}: dtype/scale/nodata/shape")
        for name, cc in cols.items():
            if name.endswith("_off"):
                req(cc["dtype"] == "uint8" and abs(cc["scale"] - _S12_OFF_SCALE) < 1e-12
                    and cc["shape"] == [F, _S12_NC], f"{nm}.{name}: dtype/scale/shape")
        vals = _s12_decode(c, raws[main])
        if main == "lst":
            cube_lst[mode] = vals
            req(h["meta"]["tiers"] == tiers[mode], f"{nm}: meta.tiers != manifest")
            req(all(a is None or b is None or abs(a - b) <= 0.011
                    for a, b in zip(h["meta"]["levels"], levels[mode])), f"{nm}: meta.levels != manifest")
            for j, y in enumerate(_S12_OBS):
                ref = _s12_frame_field(mode, y)
                same_nan = np.array_equal(np.isnan(vals[j]), ~np.isfinite(ref))
                fin = np.isfinite(ref)
                diff = float(np.max(np.abs(vals[j][fin] - ref[fin]))) if fin.any() else 0.0
                req(same_nan and diff <= _S12_LST_SCALE / 2 + 1e-6, f"{nm} frame {y}: decoded field differs "
                                                                    f"(max {diff:.4f} °C, NaN pattern {same_nan})")
                if tiers[mode][j] == "observed":
                    req(not raws["p05_off"][j].any() and not raws["p95_off"][j].any(),
                        f"{nm} frame {y}: observed frame with non-zero uncertainty offsets")
            req(set(np.unique(raws["aoa_out"]).tolist()) <= {0, 1}, f"{nm}: aoa_out not 0/1")
        else:
            scen = h["meta"]["scenario"]
            ref = np.asarray(PROJ_QUANTILES[mode][scen]["p50"], np.float64)
            fin = np.isfinite(ref)
            req(float(np.max(np.abs(vals[fin] - ref[fin]))) <= _S12_LST_SCALE / 2 + 1e-6, f"{nm}: p50 differs")
            req(np.array_equal(raws["flags"], np.asarray(PROJ_FLAGS[mode][scen], np.uint8)), f"{nm}: flags differ")
            for qk in ("p05", "p95"):
                sign = -1 if qk == "p05" else 1
                q = vals + sign * raws[f"{qk}_off"].astype(np.float64) * _S12_OFF_SCALE
                refq = np.asarray(PROJ_QUANTILES[mode][scen][qk], np.float64)
                ok = fin & np.isfinite(refq) & (np.abs(refq - ref) < 12.7)
                d = float(np.max(np.abs(q[ok] - refq[ok]))) if ok.any() else 0.0
                req(d <= _S12_OFF_SCALE / 2 + _S12_LST_SCALE / 2 + 1e-6, f"{nm}: {qk} decodes {d:.3f} °C off")
    # XAI frames: waterfall invariant on every row
    for m, by_year in man["files"]["xai"].items():
        for y, fn in by_year.items():
            h, raws = lstb_parse((_S12_WEB / fn["shap"]).read_bytes())
            cols = {c["name"]: c for c in h["columns"]}
            req(h["kind"] == "xai" and h["n"] == _S12_NC and h["meta"]["groups"] == _S12_GROUPS,
                f"{fn['shap']}: kind/n/groups")
            shap = np.stack([_s12_decode(cols[g], raws[g]) for g in _S12_GROUPS], axis=1)
            pred = _s12_decode(cols["pred"], raws["pred"])
            ok = np.isfinite(shap).all(axis=1) & np.isfinite(pred)
            err = np.abs(pred[ok] - (h["meta"]["base_value_c"] + shap[ok].sum(axis=1)))
            req(ok.any() and float(err.max()) <= _S12_WATERFALL_TOL_C,
                f"{fn['shap']}: waterfall invariant violated (max {float(err.max()) if err.size else -1:.3f})")
            req(h["meta"]["tier"] == tiers[m][_S12_OBS.index(int(y))], f"{fn['shap']}: meta.tier != manifest")
            # cross-file alignment (C-01): XAI obs vs cube field, pred vs obs, zone cube vs nearest centroid
            j = _S12_OBS.index(int(y))
            obs = _s12_decode(cols["obs"], raws["obs"])
            if m in cube_lst and tiers[m][j] in ("observed", "observed-sparse", "reconstructed"):
                f = cube_lst[m][j]
                cc = float(h["meta"].get("composition_correction") or 0.0)
                both = np.isfinite(obs) & np.isfinite(f)
                dmax = float(np.max(np.abs(obs[both] - cc - f[both]))) if both.any() else 0.0
                same_nan = bool(np.array_equal(np.isfinite(obs), np.isfinite(f)))
                req(dmax <= 0.011 and same_nan, f"{fn['shap']}: obs vs cube_obs_{m} frame {y}: max {dmax:.3f} °C "
                                                f"(composition correction {cc:+.3f}), same NaN {same_nan}")
            okp = np.isfinite(pred) & np.isfinite(obs)
            if okp.sum() >= 100 and pred[okp].std() > 0 and obs[okp].std() > 0:
                rho = float(np.corrcoef(pred[okp], obs[okp])[0, 1])
                floor = _S12_CORR_MIN.get(tiers[m][j], 0.5)
                req(rho >= floor, f"{fn['shap']}: corr(pred, obs) {rho:.3f} < {floor} (row misalignment?)")
                if floor <= rho < 0.5:
                    log(f"12: note: {m} {y} is a {tiers[m][j]} frame the model never trained on; its prediction "
                        f"agrees only weakly with those observations (corr {rho:.3f})")
            if m in cube_zones and m in ZONE_MODEL:
                rows = np.isfinite(shap).all(axis=1)
                if rows.sum() >= 100:
                    cen = np.round(np.asarray(ZONE_MODEL[m]["centroids"], np.float64), 4)
                    near = np.argmin(((shap[rows][:, None, :] - cen[None]) ** 2).sum(axis=-1), axis=1)
                    agree = float(np.mean(near == cube_zones[m][j][rows].astype(np.int64)))
                    req(agree >= 0.99, f"{fn['shap']}: nearest-centroid zones agree with cube_zones_{m} frame {y} "
                                       f"on {100 * agree:.1f} % of rows (< 99 %)")
            hf, _ = lstb_parse((_S12_WEB / fn["features"]).read_bytes())
            req(hf["kind"] == "features" and [c["name"] for c in hf["columns"]] == list(BASE_FEATURES)
                and all(c["dtype"] == "int16" for c in hf["columns"]), f"{fn['features']}: columns")
    # series
    ser = _s12_load(man["files"]["series"])
    req("ncr" in ser["series"] and all(m in ser["series"]["ncr"] for m in _S12_MODES), "series: ncr scope")
    for m in _S12_MODES:
        fr = ser["series"]["ncr"][m]["frames"]
        req([r["year"] for r in fr] == _S12_OBS and [r["tier"] for r in fr] == tiers[m],
            f"series ncr.{m}: frame years/tiers differ from the manifest")
    bad_cov = [(sid, m, r["year"]) for sid, by_m in ser["series"].items() for m, e in by_m.items()
               for r in e["frames"] if r.get("coverage") is not None and not 0.0 <= r["coverage"] <= 1.0]
    req(not bad_cov, f"series: frame coverage outside [0, 1] for {bad_cov[:5]}")
    # zones: canonical id/name/colour + D11 display fields, in zones.json and in the manifest
    zj = _s12_load(man["files"]["zones"])
    for m in _S12_MODES:
        for where, zl in (("zones.json", zj["modes"][m]["zones"]), ("manifest", man["zones"][m])):
            req([{k: z.get(k) for k in ("id", "name", "color")} for z in zl] == _S12_ZONE_META,
                f"{where} {m}: zones must be the canonical ids/names/colours")
            req(all(isinstance(z.get("display_name"), str) and z["display_name"].strip() for z in zl)
                and all(z.get("label_caveat") is None or isinstance(z["label_caveat"], str) for z in zl),
                f"{where} {m}: display_name must be a non-empty string and label_caveat a string or null")
    # raster index
    if tl.get("raster_index"):
        ri = _s12_load(tl["raster_index"])
        expected = {f"{m}_{f['year']}" for f in tl["frames"] for m in _S12_MODES if f["raster"][m]}
        req(set(ri["frames"]) == expected, "raster_index frames != manifest raster flags")
        for fr in ri["frames"].values():
            for k in ("file", "preview"):
                req((_S12_WEB / fr[k]).is_file(), f"raster file {fr[k]} missing")
    return errors


def _s12_find_validator():
    """Locate the repository's standalone validate_bundle.py (local checkout or an attached Kaggle dataset)."""
    cands = []
    env = _s12_os.environ.get("LST_VALIDATOR")
    if env:
        cands.append(Path(env))
    cwd = Path.cwd()
    cands += [cwd / "notebook" / "validate_bundle.py", cwd / "validate_bundle.py",
              cwd.parent / "notebook" / "validate_bundle.py"]
    kin = Path("/kaggle/input")
    if kin.is_dir():               # /kaggle/input/<slug>/... or /kaggle/input/datasets/<owner>/<slug>/lst_code/...
        for depth in range(1, 6):
            cands += sorted(kin.glob("*/" * depth + "validate_bundle.py"))
    for c in cands:
        if c.is_file():
            return c
    return None


with timer("12_validate"):
    _S12_ERRORS = _s12_check_bundle()
    if _S12_ERRORS:
        raise ValueError(f"12: web bundle invalid ({len(_S12_ERRORS)} problems):\n  - " + "\n  - ".join(_S12_ERRORS))
    log(f"12: read-back validation OK ({len(_S12_SIZES)} files)")
    _S12_VALIDATOR = _s12_find_validator()
    _S12_VALIDATOR_RESULT = None
    if _S12_VALIDATOR is not None:
        try:
            # The validator always ends with a verdict line ("VALID: ...", "INVALID (n errors): ..." or "BUNDLE
            # UNREADABLE: ..."). A non-zero exit WITHOUT one means the child process never got to judge the bundle
            # (e.g. it could not start under memory pressure at the end of a long run): retry once, then warn as for
            # a timeout - the bundle already passed the in-notebook checks. Only a real verdict can fail the run.
            for _s12_attempt in (1, 2):
                _s12_proc = _s12_subprocess.run([_s12_sys.executable, str(_S12_VALIDATOR), str(_S12_WEB), "--quiet"],
                                                capture_output=True, text=True, timeout=1800,
                                                env={**_s12_os.environ, "PYTHONIOENCODING": "utf-8"})
                _s12_out, _s12_err = _s12_proc.stdout or "", _s12_proc.stderr or ""
                _s12_verdict = any(ln.startswith(("VALID:", "INVALID (", "BUNDLE UNREADABLE:"))
                                   for ln in _s12_out.splitlines())
                if _s12_proc.returncode == 0 or _s12_verdict:
                    break
                log(f"12: standalone validator exited {_s12_proc.returncode} ({_s12_proc.returncode & 0xFFFFFFFF:#x}) "
                    f"without a verdict (attempt {_s12_attempt}/2); stderr: {_s12_err[-1000:].strip() or '-'}",
                    "WARNING")
            _S12_VALIDATOR_RESULT = {"path": str(_S12_VALIDATOR), "returncode": _s12_proc.returncode,
                                     "verdict": _s12_verdict, "output": _s12_out[-4000:]}
            print(_s12_out[-4000:])
            if _s12_proc.returncode != 0 and _s12_verdict:
                raise ValueError(f"12: standalone validator reported the bundle INVALID (exit {_s12_proc.returncode}):"
                                 f"\n{_s12_out[-4000:]}\n{_s12_err[-2000:]}")
            if _s12_proc.returncode != 0:
                log(f"12: standalone validator could not run (exit {_s12_proc.returncode}, no verdict twice); the "
                    "bundle passed the in-notebook checks - run notebook/validate_bundle.py on it", "WARNING")
            else:
                log(f"12: standalone validator {_S12_VALIDATOR.name}: VALID")
        except _s12_subprocess.TimeoutExpired:
            log("12: standalone validator timed out (bundle already passed the in-notebook checks)", "WARNING")
    else:
        log("12: standalone validate_bundle.py not found (attach it or run it on the downloaded bundle); the "
            "in-notebook read-back checks passed")

## 12.12 Parquet tables, GeoJSON cells, fine-field NPZ and native models
Analysis-ready copies of the per-cell products (zstd Parquet, one row per cell × frame), polygons of the analysis
cells with their zone and LST per observed frame (RFC 7946, counter-clockwise rings, shared vertices from one
transformed corner lattice), the 250 m fields as compressed NPZ with their UTM grid, and the native final boosters.

In [ ]:
def _s12_to_parquet(df, name):
    path = _S12_PARQUET / name
    df.to_parquet(path, engine="pyarrow", compression="zstd", index=False)
    log(f"  parquet/{name}: {len(df):,} rows, {path.stat().st_size / 1e6:.2f} MB")
    return path


def _s12_parquet():
    if not (_S12_PARQUET / "features_long.parquet").is_file():
        _s12_to_parquet(DF, "features_long.parquet")
    base = pd.DataFrame({"cell_id": np.asarray(CELL_IDS, np.int32), "row": np.asarray(CELL_ROWS, np.int32),
                         "col": np.asarray(CELL_COLS, np.int32), "lon": np.asarray(CELL_LON, np.float64),
                         "lat": np.asarray(CELL_LAT, np.float64), "district": _S12_DISTRICT_OF_CELL})
    for m in _S12_MODES:
        parts = []
        for y in _S12_OBS:
            lev, _ = _s12_level(m, y)
            lev = float(lev) if lev is not None else np.nan
            d = base.copy()
            d["year"] = np.int16(y)
            d["tier"] = _s12_tier(m, y)
            d["lst"] = _s12_frame_field(m, y).astype(np.float32)
            unc = (globals().get("FRAME_UNC") or {}).get(m, {}).get(y)
            for qk in ("p05", "p95"):
                d[f"lst_{qk}"] = (np.asarray(unc[qk], np.float32) if unc is not None
                                  else np.full(_S12_NC, np.nan, np.float32))
            a = (globals().get("FRAME_AOA_OUT") or {}).get(m, {}).get(y)
            d["aoa_out"] = np.asarray(a, bool) if a is not None else False
            if y in (PRED_ANOM.get(m) or {}):
                d["lst_pred"] = (np.asarray(PRED_ANOM[m][y], np.float64) + lev).astype(np.float32)
                d["lst_pred_oof"] = (np.asarray(OOF_ANOM[m][y], np.float64) + lev).astype(np.float32)
                d["zone"] = np.asarray(ZONES_OBS[m][y], np.uint8)
                g = np.asarray(SHAP_GROUP_VALUES[m][y], np.float32)
                for gi, gname in enumerate(_S12_GROUPS):
                    d[f"shap_{gname}"] = g[:, gi]
            parts.append(d)
        _s12_to_parquet(pd.concat(parts, ignore_index=True), f"predictions_{m}.parquet")
        del parts
        if _S12_SCEN:
            pparts = []
            for s in _S12_SCEN:
                q = PROJ_QUANTILES[m][s]
                for j, y in enumerate(_S12_PROJ):
                    d = base[["cell_id", "district"]].copy()
                    d["scenario"] = s
                    d["year"] = np.int16(y)
                    for k in ("p05", "p25", "p50", "p75", "p95"):
                        d[k] = np.asarray(q[k][j], np.float32)
                    d["flags"] = np.asarray(PROJ_FLAGS[m][s][j], np.uint8)
                    d["zone"] = np.asarray(PROJ_ZONES[m][s][j], np.uint8)
                    pparts.append(d)
            _s12_to_parquet(pd.concat(pparts, ignore_index=True), f"projections_{m}.parquet")
            del pparts
    lev_rows = []
    for m in _S12_MODES:
        for y in _S12_OBS:
            rec = LEVELS[m].get(y) or {}
            ci = rec.get("ci") or [None, None]
            lev_rows.append({"mode": m, "year": y, "tier": _s12_tier(m, y), "level": rec.get("level"),
                             "ci_lo": ci[0], "ci_hi": ci[1], "raw_level": rec.get("raw_level"),
                             "overpass_correction": rec.get("overpass_correction"), "method": rec.get("method")})
    _s12_to_parquet(pd.DataFrame(lev_rows), "levels.parquet")
    sf_rows = []
    for m in _S12_MODES:
        for scope, rows in SERIES_FRAMES[m].items():
            for r in rows:
                ci = r.get("ci") or [None, None]
                sf_rows.append({"mode": m, "scope": scope, "year": r["year"], "mean": r.get("mean"),
                                "ci_lo": ci[0], "ci_hi": ci[1], "p90": r.get("p90"),
                                "hot_core_km2": r.get("hot_core_km2"), "uhi": r.get("uhi"),
                                "n_cells": r.get("n_cells"), "tier": r.get("tier")})
    _s12_to_parquet(pd.DataFrame(sf_rows), "series_frames.parquet")
    _s12_to_parquet(ANNUAL_SERIES.astype({"flag": "object"}), "annual_series.parquet")
    imp_rows = []
    for m in _S12_MODES:
        g = IMPORTANCE[m]["global"]
        for gi, gname in enumerate(IMPORTANCE[m]["groups"]):
            imp_rows.append({"mode": m, "group": gname, "mean_abs": float(g["mean_abs"][gi]),
                             "ci_lo": float(g["ci_lo"][gi]), "ci_hi": float(g["ci_hi"][gi]),
                             "bias": float(g["bias"][gi])})
    _s12_to_parquet(pd.DataFrame(imp_rows), "importance.parquet")


def _s12_corner_lonlat():
    from pyproj import Transformer
    xs = GRID.x0 + np.arange(GRID.width + 1) * GRID.res
    ys = GRID.y0 - np.arange(GRID.height + 1) * GRID.res
    XX, YY = np.meshgrid(xs, ys)
    lon, lat = Transformer.from_crs(GRID.crs, "EPSG:4326", always_xy=True).transform(XX, YY)
    return np.round(lon, 5), np.round(lat, 5)


def _s12_zone_geojson(mode, lon_c, lat_c):
    """One FeatureCollection of all ROI cell squares with zone_<year> / lst_<year> properties of the mode."""
    r = np.asarray(CELL_ROWS, np.int64)
    c = np.asarray(CELL_COLS, np.int64)
    zones = {y: np.asarray(ZONES_OBS[mode][y]) for y in _S12_OBS if y in ZONES_OBS.get(mode, {})}
    lst = {y: np.round(_s12_frame_field(mode, y), 2) for y in _S12_OBS}
    path = _S12_GEO / f"zones_{mode}.geojson"
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as fh:
        fh.write('{"type":"FeatureCollection","features":[')
        for i in range(_S12_NC):
            # bottom-left -> bottom-right -> top-right -> top-left -> close (counter-clockwise; row 0 is north)
            corners = [(r[i] + 1, c[i]), (r[i] + 1, c[i] + 1), (r[i], c[i] + 1), (r[i], c[i]), (r[i] + 1, c[i])]
            ring = [[float(lon_c[a, b]), float(lat_c[a, b])] for a, b in corners]
            props = {"cell_id": int(CELL_IDS[i]), "district": int(_S12_DISTRICT_OF_CELL[i])}
            for y in _S12_OBS:
                if y in zones:
                    props[f"zone_{y}"] = None if zones[y][i] > 3 else int(zones[y][i])
                v = lst[y][i]
                props[f"lst_{y}"] = float(v) if np.isfinite(v) else None
            feat = {"type": "Feature", "geometry": {"type": "Polygon", "coordinates": [ring]}, "properties": props}
            fh.write(("," if i else "") + _s12_json.dumps(feat, separators=(",", ":"), allow_nan=False))
        fh.write("]}")
    _s12_os.replace(tmp, path)
    log(f"  geo/{path.name}: {_S12_NC:,} cells, {path.stat().st_size / 1e6:.1f} MB")
    return path


def _s12_npz():
    for m in _S12_MODES:
        ff = (globals().get("FINE_FIELDS") or {}).get(m) or {}
        if not ff:
            continue
        years = sorted(int(y) for y in ff)
        arrays = {f"lst_{y}": np.asarray(ff[y], np.float32) for y in years}
        arrays.update({f"flags_{y}": np.asarray(FINE_FLAGS[m][y], np.uint8) for y in years
                       if y in (globals().get("FINE_FLAGS") or {}).get(m, {})})
        arrays["grid"] = np.array([GRID_FINE.x0, GRID_FINE.y0, GRID_FINE.res, GRID_FINE.width, GRID_FINE.height],
                                  np.float64)
        arrays["years"] = np.asarray(years, np.int32)
        path = _S12_NPZ / f"fine_fields_{m}.npz"
        np.savez_compressed(path, **arrays)
        log(f"  npz/{path.name}: {len(years)} frames {GRID_FINE.height}x{GRID_FINE.width} "
            f"({GRID_FINE.res} m, {GRID_FINE.crs}; grid = [x0, y0, res, width, height]), "
            f"{path.stat().st_size / 1e6:.1f} MB")


def _s12_models():
    for m in _S12_MODES:
        path = _S12_MODEL_DIR / f"xgb_final_{m}.json"
        if not path.is_file() and m in FINAL_MODELS:
            FINAL_MODELS[m].save_model(str(path))
        if path.is_file():
            log(f"  models/{path.name}: {path.stat().st_size / 1e6:.2f} MB")


with timer("12_parquet"):
    _s12_parquet()
free_memory()
with timer("12_geojson"):
    save_json(_S12_DISTRICTS_FC, _S12_GEO / "ncr_districts.geojson")
    _S12_LON_C, _S12_LAT_C = _s12_corner_lonlat()
    for _s12_m in _S12_MODES:
        _s12_zone_geojson(_s12_m, _S12_LON_C, _S12_LAT_C)
    del _S12_LON_C, _S12_LAT_C
with timer("12_npz_models"):
    _s12_npz()
    _s12_models()
free_memory()

## 12.13 Overview figure (decoded from the written files), zip, output tree and run summary
The figure is drawn from the **files on disk** (cube and series decoded again), so it doubles as a visual check that
the dashboard will see what the pipeline computed: the current-frame day and night maps and the NCR timeline with
the observed levels and the SSP fans.

In [ ]:
def _s12_fig_overview():
    man = EXPORT_MANIFEST
    ser = _s12_load(man["files"]["series"])
    fig, axes = plt.subplots(1, len(_S12_MODES) + 1, figsize=(6 * (len(_S12_MODES) + 1), 5))
    for k, m in enumerate(_S12_MODES):
        h, raws = lstb_parse((_S12_WEB / man["timeline"]["cubes"][f"obs_{m}"]).read_bytes())
        col = next(c for c in h["columns"] if c["name"] == "lst")
        v = _s12_decode(col, raws["lst"])[_S12_OBS.index(_S12_NOW)]
        img = np.full((GRID.height, GRID.width), np.nan)
        img[CELL_ROWS, CELL_COLS] = v
        lo, hi = man["timeline"]["domains"][m]["abs"]
        im = axes[k].imshow(img, cmap="inferno" if m == "day" else "magma", vmin=lo, vmax=hi)
        axes[k].set_title(f"{m} {_S12_NOW} (decoded cube, fixed domain {lo:g}-{hi:g} °C)")
        axes[k].axis("off")
        fig.colorbar(im, ax=axes[k], shrink=0.75, label="LST °C")
    ax = axes[-1]
    for m, color in zip(_S12_MODES, ("#f97316", "#6366f1")):
        e = ser["series"]["ncr"][m]
        yrs = [r["year"] for r in e["frames"] if r["mean"] is not None]
        ax.plot(yrs, [r["mean"] for r in e["frames"] if r["mean"] is not None], "o-", color=color, label=f"{m} obs")
        for s, ls in zip(_S12_SCEN, (":", "--", "-.")):
            fan = e["ssp"].get(s, [])
            if not fan:
                continue
            fy = [r["year"] for r in fan]
            ax.fill_between(fy, [r["p05"] for r in fan], [r["p95"] for r in fan], color=color, alpha=0.08)
            ax.plot(fy, [r["p50"] for r in fan], ls, color=color, label=f"{m} {s} p50")
    ax.set_title("NCR mean LST (timeline_series.json.gz)")
    ax.set_xlabel("year")
    ax.set_ylabel("°C")
    ax.legend(fontsize=7, ncol=2)
    if DATA_MODE == "synthetic":
        fig.suptitle("SYNTHETIC DEMO DATA", color="crimson")
    fig.tight_layout()
    fig.savefig(_S12_FIG_DIR / "12_export_overview.png", dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def _s12_zip(zip_path):
    with _s12_zipfile.ZipFile(zip_path, "w", compression=_s12_zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for p in sorted(_S12_WEB.rglob("*")):
            if p.is_file():
                rel = p.relative_to(_S12_WEB).as_posix()
                # gzip / PNG members are already compressed: store them as they are
                stored = rel.endswith((".gz", ".png"))
                zf.write(p, arcname=rel, compress_type=_s12_zipfile.ZIP_STORED if stored else _s12_zipfile.ZIP_DEFLATED)
    return zip_path


def _s12_print_tree(root, max_files=30):
    """Outputs tree with sizes; directories with many files (caches, rasters) are summarised."""
    root = Path(root)
    print(f"{root}/")
    for d in sorted([root] + [p for p in root.rglob("*") if p.is_dir()]):
        files = sorted(p for p in d.iterdir() if p.is_file())
        depth = len(d.relative_to(root).parts)
        if d != root:
            print(f"{'  ' * depth}{d.name}/")
        if len(files) > max_files or "cache" in d.relative_to(root).parts:
            total = sum(p.stat().st_size for p in files)
            if files:
                print(f"{'  ' * (depth + 1)}[{len(files)} files, {total / 1e6:.1f} MB]")
            continue
        for p in files:
            print(f"{'  ' * (depth + 1)}{p.name:<44s} {p.stat().st_size / 1e6:9.2f} MB")


try:
    _s12_fig_overview()
except Exception as _s12_exc:  # noqa: BLE001 - the figure is a convenience, never a reason to lose the export
    log(f"12: overview figure failed: {type(_s12_exc).__name__}: {_s12_exc}", "WARNING")
_S12_ZIP = _s12_zip(_S12_OUT / "web_bundle.zip")
log(f"12: web bundle zipped: {_S12_ZIP} ({_S12_ZIP.stat().st_size / 1e6:.1f} MB)")
_s12_print_tree(_S12_OUT)

print("\n" + "=" * 96)
print(f"RUN SUMMARY  (data mode: {DATA_MODE.upper()}{' - SYNTHETIC DEMO DATA' if DATA_MODE == 'synthetic' else ''}"
      f", profile {CFG.PROFILE}, grid {CFG.GRID_RES_M} m, {_S12_NC:,} cells)")
print("=" * 96)
for _s12_m in _S12_MODES:
    _s12_h = (MODEL_RESULTS.get(_s12_m) or {}).get("headline") or {}
    _s12_sp, _s12_tp = _s12_h.get("spatial") or {}, _s12_h.get("temporal") or {}
    _s12_lv = [f"{y}:{_s12_level(_s12_m, y)[0]:.1f}" for y in _S12_OBS if _s12_level(_s12_m, y)[0] is not None]
    print(f"{_s12_m.upper():>5}: spatial {_s12_sp.get('scheme')} R2 {_s12_num(_s12_sp.get('r2'), 3)} RMSE "
          f"{_s12_num(_s12_sp.get('rmse'), 3)} °C | temporal {_s12_tp.get('scheme')} R2 "
          f"{_s12_num(_s12_tp.get('r2'), 3)}"
          f" RMSE {_s12_num(_s12_tp.get('rmse'), 3)} °C | beta {_s12_num(CLIMATE_SENSITIVITY[_s12_m].get('beta'), 2)}")
    print(f"       levels {' '.join(_s12_lv)}")
    for _s12_s in _S12_SCEN:
        _s12_r = _s12_ncr_proj(_s12_m, _s12_s, _S12_PROJ[-1]) or {}
        print(f"       {_s12_s} {_S12_PROJ[-1]}: NCR p50 {_s12_num(_s12_r.get('p50'), 2)} °C "
              f"[{_s12_num(_s12_r.get('p05'), 2)}, {_s12_num(_s12_r.get('p95'), 2)}]")
_s12_probes = (globals().get("FUTURE_INPUTS_DIAG") or {}).get("probes") if isinstance(
    globals().get("FUTURE_INPUTS_DIAG"), dict) else None
if _s12_probes:
    # pilot probes of section 05 (D2): Earth Engine cost of the CMIP6 pattern, for extrapolating the full run
    print("Pilot probes (section 05):")
    _s12_rows = [{"probe": k, **{c: (v or {}).get(c) for c in ("seconds", "requests", "cache_hits",
                                                               "roi_mean_delta_k", "error")}}
                 for k, v in _s12_probes.items()]
    print(pd.DataFrame(_s12_rows).to_string(index=False))
print(f"Web bundle: {len(_S12_SIZES)} files, {_S12_BUNDLE_BYTES / 1e6:.1f} MB on disk -> {_S12_WEB}")
print(f"Zip: {_S12_ZIP} ({_S12_ZIP.stat().st_size / 1e6:.1f} MB)")
_s12_target = "dashboard/public/demo/" if DATA_MODE == "synthetic" else "dashboard/public/data/"
print(f"\nNEXT STEP: copy the contents of {_S12_WEB}/ (or unzip {_S12_ZIP.name}) into {_s12_target}, then run "
      "`npm run dev` in dashboard/ (optionally `python notebook/validate_bundle.py <dir>` first).")
TIMINGS["12_export_total"] = round(_s12_time.perf_counter() - _S12_T0, 2)
free_memory()
log(f"Section 12 complete in {TIMINGS['12_export_total']:.1f}s: EXPORT_MANIFEST ready")
# Last statement of the run: pack the cache into a few tar files (Kaggle output file cap; pack_cache honours
# LST_PACK_CACHE and never touches web/ or web_bundle.zip). Without the helper (older 00) only report the count.
_s12_pk = globals().get("pack_cache")
if _s12_pk is not None:
    _s12_pk()
else:
    _s12_nfiles = sum(1 for _p in _S12_OUT.rglob("*") if _p.is_file())
    log(f"12: {_s12_nfiles} files under {_S12_OUT} (pack_cache unavailable; Kaggle keeps about 500 output files)",
        "WARNING" if _s12_nfiles > 450 else "INFO")